In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Merged composite figure (COVID + NEW) for residuals vs sex & age.

Changes:
- Per-dataset 5% |residual| outliers dropped BEFORE merge.
- Rolling plot: first 0–15 years -> one dot; last 80–92 years -> one dot;
  interior ages use 12-year rolling windows (±6y) stepped by 3 years.
- Violin & OLS panels y-limits fixed to [-30, 30].
- Bias heatmap color scale fixed to [-7.5, +7.5].
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.formula.api as smf

OLD_FILE = "test_preds_covid.csv"
NEW_FILE = "test_preds_full_with_sex.csv"

# -------------------------
# Helpers
# -------------------------
def standardize(df, source):
    """
    Return a clean DataFrame with columns: Age, y_pred, Sex, residual, abs_res, Age_bin.
    Handles either schema (OLD or NEW) and drops **top 5% |residual| per dataset**.
    """
    df = df.rename(columns=lambda x: x.strip())
    age_col = next((c for c in ["Age", "y_true", "true", "age"] if c in df.columns), None)
    pred_col = next((c for c in ["y_pred", "pred", "prediction"] if c in df.columns), None)
    sex_col  = next((c for c in ["Biological Sex", "biological_sex", "Sex", "gender"] if c in df.columns), None)
    if age_col is None or pred_col is None or sex_col is None:
        raise ValueError(f"[{source}] Missing required columns. Found: {list(df.columns)}")

    out = df[[age_col, pred_col, sex_col]].copy()
    out.columns = ["Age", "y_pred", "Sex"]
    out["Age"]   = pd.to_numeric(out["Age"], errors="coerce")
    out["y_pred"]= pd.to_numeric(out["y_pred"], errors="coerce")
    out["Sex"]   = (out["Sex"].astype(str).str.strip()
                    .replace({"F":"Female","M":"Male","female":"Female","male":"Male"}))
    out = out.dropna(subset=["Age","y_pred","Sex"])
    out = out[out["Sex"].isin(["Male","Female"])].copy()

    out["residual"] = out["y_pred"] - out["Age"]
    out["abs_res"]  = out["residual"].abs()

    # Per-dataset 5% outlier drop
    cut95 = out["abs_res"].quantile(0.95)
    out   = out[out["abs_res"] <= cut95].copy()

    # Age bins for the heatmap: 0–15, then 5y steps, special 80–92
    bins = [0, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 92]
    out["Age_bin"] = pd.cut(out["Age"], bins=bins, right=False)
    out["__source__"] = source
    return out

def bootstrap_ci_diff(m, f, nboot=3000, alpha=0.05, seed=0):
    """Bootstrap CI for mean(m) - mean(f)."""
    rng = np.random.default_rng(seed)
    m = np.asarray(m); f = np.asarray(f)
    diffs = []
    for _ in range(nboot):
        ms = rng.choice(m, size=len(m), replace=True)
        fs = rng.choice(f, size=len(f), replace=True)
        diffs.append(ms.mean() - fs.mean())
    lo, hi = np.percentile(diffs, [100*alpha/2, 100*(1-alpha/2)])
    return np.mean(diffs), lo, hi

# -------------------------
# Load, clean (per dataset), merge
# -------------------------
# NEW: Age = y_true
new_raw = pd.read_csv(NEW_FILE)
if "y_true" not in new_raw.columns:
    raise ValueError("NEW file must contain 'y_true'.")
new_raw = new_raw.rename(columns={"y_true": "Age"})  # harmonize for standardize()
new_df  = standardize(new_raw, "NEW")

# COVID (OLD)
old_raw = pd.read_csv(OLD_FILE)
covid_df = standardize(old_raw, "COVID")

# MERGED after per-dataset outlier removal
df = pd.concat([covid_df, new_df], ignore_index=True)

# -------------------------
# Core stats & models (merged)
# -------------------------
sns.set_theme(style="whitegrid", context="paper")

female = df.query("Sex=='Female'")["residual"]
male   = df.query("Sex=='Male'")["residual"]

mwu_p   = stats.mannwhitneyu(female, male, alternative="two-sided").pvalue
welch_p = stats.ttest_ind(female, male, equal_var=False).pvalue
lev_p   = stats.levene(female, male, center="median").pvalue
ks_p    = stats.ks_2samp(female, male).pvalue
abs_p   = stats.ttest_ind(np.abs(female), np.abs(male), equal_var=False).pvalue

ols = smf.ols("residual ~ Age * Sex", data=df).fit()
int_p = ols.pvalues.get("Age:Sex[T.Male]", np.nan)

# -------------------------
# Rolling plot with special end windows
# -------------------------
def rolling_with_special_windows(df, half=6, step=3):
    """
    Build windows:
      - [0, 15]      -> one dot at center 7.5
      - sliding windows of width 12y (±6) from inside (15, 80) stepped by `step`
      - [80, 92]     -> one dot at center 86
    Return DataFrame: center, diff, lo, hi
    """
    rows = []

    # First special window: 0–15
    w1 = df[(df["Age"] >= 0) & (df["Age"] <= 15)]
    m, f = w1.loc[w1["Sex"]=="Male","residual"].values, w1.loc[w1["Sex"]=="Female","residual"].values
    if len(m)>=10 and len(f)>=10:
        d, lo, hi = bootstrap_ci_diff(m, f, nboot=3000)
        rows.append((7.5, d, lo, hi))

    # Interior sliding windows (full 12y width)
    # choose centers so that [c-6, c+6] stays within (15, 80)
    centers = np.arange(15+half, 80-half+1e-9, step)  # e.g., 21,24,...,74
    for c in centers:
        sub = df[(df["Age"] >= c-half) & (df["Age"] <= c+half)]
        m = sub.loc[sub["Sex"]=="Male","residual"].values
        f = sub.loc[sub["Sex"]=="Female","residual"].values
        if len(m)>=10 and len(f)>=10:
            d, lo, hi = bootstrap_ci_diff(m, f, nboot=3000)
            rows.append((float(c), d, lo, hi))

    # Last special window: 80–92
    w2 = df[(df["Age"] >= 80) & (df["Age"] <= 92)]
    m, f = w2.loc[w2["Sex"]=="Male","residual"].values, w2.loc[w2["Sex"]=="Female","residual"].values
    if len(m)>=10 and len(f)>=10:
        rows.append((86.0, *bootstrap_ci_diff(m, f, nboot=3000)))

    return pd.DataFrame(rows, columns=["center","diff","lo","hi"]).sort_values("center")

roll = rolling_with_special_windows(df, half=6, step=3)

# -------------------------
# Bias heatmap (merged)
# -------------------------
heat = (df.groupby("Age_bin")
          .apply(lambda g: g.loc[g["Sex"]=="Male","residual"].mean()
                          - g.loc[g["Sex"]=="Female","residual"].mean())
          .to_frame("diff"))

# -------------------------
# Figure: 3×2 composite (merged)
# -------------------------
fig, axes = plt.subplots(3, 2, figsize=(11, 13))
(ax1, ax2, ax3, ax4, ax5, ax6) = axes.flat
plt.subplots_adjust(hspace=0.6)
fig.suptitle("Residuals vs Sex & Age — MERGED (COVID + NEW)", y=0.98, fontsize=14, weight="bold")

# 1) Violin (y-lim ±30)
sns.violinplot(x="Sex", y="residual", data=df, inner="quart", cut=0, ax=ax1, palette="Set2")
sns.stripplot(x="Sex", y="residual", data=df, color="k", alpha=0.15, size=2, ax=ax1)
ax1.axhline(0, ls="--", c="gray")
ax1.set_ylim(-30, 30)
ax1.set_title("Residual distribution by sex")
ax1.text(0.95, 0.9,
         f"MWU p={mwu_p:.1e}\nWelch p={welch_p:.1e}\nLevene p={lev_p:.1e}\nKS p={ks_p:.1e}",
         ha="right", va="top", transform=ax1.transAxes,
         bbox=dict(facecolor="white", alpha=0.7, edgecolor="none"), fontsize=8)

# 2) KDE
sns.kdeplot(data=df, x="residual", hue="Sex", fill=True, common_norm=False, alpha=0.4, ax=ax2)
ax2.axvline(0, ls="--", c="gray")
ax2.set_title("Residual density by sex")
ax2.text(0.95,0.9,f"KS p={ks_p:.2e}",ha="right",va="top",
         transform=ax2.transAxes,bbox=dict(facecolor="white",alpha=0.7,edgecolor="none"),fontsize=8)

# 3) ECDF of |residual|
sns.ecdfplot(data=df, x="abs_res", hue="Sex", ax=ax3)
ax3.set_title("ECDF of absolute residuals")
ax3.text(0.95,0.1,f"Welch(|res|) p={abs_p:.2e}",ha="right",
         transform=ax3.transAxes,bbox=dict(facecolor="white",alpha=0.7,edgecolor="none"),fontsize=8)
ax3.set_xlabel("|residual|")

# 4) OLS: residual ~ Age × Sex (y-lim ±30)
sns.scatterplot(data=df, x="Age", y="residual", hue="Sex", alpha=0.25, s=15, ax=ax4)
age_grid = np.linspace(df["Age"].min(), df["Age"].max(), 200)
for sex, col in [("Female","C0"), ("Male","C1")]:
    yhat = ols.predict(pd.DataFrame({"Age": age_grid, "Sex": sex}))
    ax4.plot(age_grid, yhat, color=col, lw=2.5, label=f"{sex} fit")
ax4.axhline(0, ls="--", c="gray")
ax4.set_ylim(-30, 30)
ax4.legend(loc="lower left")
ax4.set_title("OLS: residual ~ Age × Sex")
ax4.text(0.95,0.9,f"Interaction p={int_p:.2e}",ha="right",va="top",
         transform=ax4.transAxes,bbox=dict(facecolor="white",alpha=0.7,edgecolor="none"),fontsize=8)

# 5) Rolling sex difference (special end windows)
ax5.plot(roll["center"], roll["diff"], "o-", label="Male–Female mean diff")
ax5.fill_between(roll["center"], roll["lo"], roll["hi"], color="C0", alpha=0.2, label="95% bootstrap CI")
ax5.axhline(0, ls="--", c="gray")
ax5.set_title("Rolling sex difference (12y windows; 0–15 & 80–92 as single dots)")
ax5.set_xlabel("Age (window center)")
ax5.set_ylabel("Residual mean diff (M–F)")
ax5.legend(fontsize=8)

# 6) Bias heatmap (scale fixed to ±7.5)
sns.heatmap(heat.T, cmap="RdBu_r", center=0, vmin=-7.5, vmax=7.5,
            cbar_kws={"label":"Mean (M–F) residual"}, ax=ax6)
ax6.set_xlabel("Age bin")
ax6.set_ylabel("")
ax6.set_title("Bias map across age bins")
ax6.set_xticklabels([str(i) for i in heat.index.categories], rotation=45, ha="right", fontsize=8)

plt.show()


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Merged rolling/segment plot only:
- Per-dataset: drop top 5% |residual|
- Segments: [0–15], [15–18], [18–21], ..., [77–80], [80–92]
- For each segment: Male–Female mean residual with bootstrap 95% CI
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats

OLD_FILE = "test_preds_covid.csv"
NEW_FILE = "test_preds_full_with_sex.csv"

# ----------------- helpers -----------------
def standardize(df, source):
    """Return clean df with columns: Age, y_pred, Sex, residual, abs_res."""
    df = df.rename(columns=lambda x: x.strip())
    age_col = next((c for c in ["Age", "y_true", "true", "age"] if c in df.columns), None)
    pred_col = next((c for c in ["y_pred", "pred", "prediction"] if c in df.columns), None)
    sex_col  = next((c for c in ["Biological Sex","biological_sex","Sex","gender"] if c in df.columns), None)
    if age_col is None or pred_col is None or sex_col is None:
        raise ValueError(f"[{source}] Missing Age/y_pred/Sex columns. Found: {list(df.columns)}")

    out = df[[age_col, pred_col, sex_col]].copy()
    out.columns = ["Age", "y_pred", "Sex"]
    out["Age"] = pd.to_numeric(out["Age"], errors="coerce")
    out["y_pred"] = pd.to_numeric(out["y_pred"], errors="coerce")
    out["Sex"] = (out["Sex"].astype(str).str.strip()
                  .replace({"F":"Female","M":"Male","female":"Female","male":"Male"}))
    out = out.dropna(subset=["Age","y_pred","Sex"])
    out = out[out["Sex"].isin(["Male","Female"])].copy()

    out["residual"] = out["y_pred"] - out["Age"]
    out["abs_res"] = out["residual"].abs()

    # per-dataset 5% |res| trimming
    cut95 = out["abs_res"].quantile(0.95)
    out = out[out["abs_res"] <= cut95].copy()
    out["__source__"] = source
    return out

def bootstrap_ci_diff(m, f, nboot=3000, alpha=0.05, seed=0):
    """Bootstrap CI for mean(m) - mean(f)."""
    rng = np.random.default_rng(seed)
    m = np.asarray(m); f = np.asarray(f)
    diffs = []
    for _ in range(nboot):
        ms = rng.choice(m, size=len(m), replace=True)
        fs = rng.choice(f, size=len(f), replace=True)
        diffs.append(ms.mean() - fs.mean())
    lo, hi = np.percentile(diffs, [100*alpha/2, 100*(1-alpha/2)])
    return float(np.mean(diffs)), float(lo), float(hi)

def build_segments(min_age, max_age):
    """
    Build edges: [0,15], then 3y steps to 80, then [80,92].
    Returns list of (lo, hi) inclusive of left, exclusive of right except last.
    """
    edges = [0, 15]
    # 3y steps up to 80
    e = 15
    while e < 80:
        e += 3
        edges.append(e)
    # ensure last two bounds are 80 and 92
    if edges[-1] != 80:
        edges.append(80)
    edges.append(92)
    # make pairs
    segs = []
    for i in range(len(edges)-1):
        lo, hi = edges[i], edges[i+1]
        segs.append((lo, hi))
    return segs

# ----------------- load & merge with per-dataset trimming -----------------
# NEW data: Age = y_true
new_raw = pd.read_csv(NEW_FILE)
if "y_true" not in new_raw.columns:
    raise ValueError("NEW file must contain 'y_true'.")
new_raw = new_raw.rename(columns={"y_true": "Age"})
new_df = standardize(new_raw, "NEW")

# COVID/OLD
old_raw = pd.read_csv(OLD_FILE)
covid_df = standardize(old_raw, "COVID")

df = pd.concat([covid_df, new_df], ignore_index=True)

# ----------------- compute per-segment diffs -----------------
segments = build_segments(df["Age"].min(), df["Age"].max())  # uses fixed scheme
rows = []
for lo, hi in segments:
    # last segment [80,92]: include hi bound, others exclude hi
    if hi == 92:
        sub = df[(df["Age"] >= lo) & (df["Age"] <= hi)]
    else:
        sub = df[(df["Age"] >= lo) & (df["Age"] < hi)]
    m = sub.loc[sub["Sex"]=="Male","residual"].values
    f = sub.loc[sub["Sex"]=="Female","residual"].values

    # require a minimum per-sex sample size to avoid noisy CIs
    if len(m) >= 10 and len(f) >= 10:
        diff, lo_ci, hi_ci = bootstrap_ci_diff(m, f, nboot=3000)
        center = (lo + hi) / 2
        rows.append((lo, hi, center, diff, lo_ci, hi_ci))

roll = pd.DataFrame(rows, columns=["lo","hi","center","diff","lo_ci","hi_ci"])

# ----------------- plot -----------------
sns.set_theme(style="whitegrid", context="paper")
fig, ax = plt.subplots(figsize=(9.5, 4.8))

ax.plot(roll["center"], roll["diff"], "o-", lw=2, label="Male–Female mean diff")
ax.fill_between(roll["center"], roll["lo_ci"], roll["hi_ci"], alpha=0.22, label="95% bootstrap CI")

# helpful x ticks at the segment edges
tick_edges = [0, 15, 30, 45, 60, 77, 80, 92]
ax.set_xticks(tick_edges)
ax.set_xlim(0, 92)

ax.axhline(0, ls="--", c="gray", lw=1)
ax.set_xlabel("Age (segment centers)\nSegments: 0–15, 3-year steps to 77–80, then 80–92")
ax.set_ylabel("Residual mean difference (Male − Female)")
ax.set_title("Segmented sex difference in residuals (merged; after per-dataset 5% |res| drop)")
ax.legend(fontsize=9, loc="best")
plt.show()


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Composite figure (publication-ready) for residuals vs sex & age
— run three times:
  1) NEW DATA ONLY (test_preds_full_with_sex.csv)
  2) COVID ONLY    (test_preds_covid.csv)
  3) MERGED        (COVID + NEW, after per-dataset outlier dropping)

Per dataset:
- Harmonize columns to: Age, y_pred, Sex
- residual = y_pred - Age ; abs_res = |residual|
- Drop top 5% |residual| outliers **within each dataset BEFORE any merging**
- Age bins: first bin 0–15, then 5y steps, with a special 80–92 bin
- Produce a 3×2 composite figure:
    (1) Violin + strip  (y-lim ±30)
    (2) Residual KDE by sex
    (3) ECDF of |residual|
    (4) OLS residual ~ Age × Sex + fitted lines (y-lim ±30)
    (5) Rolling sex diff (12y window, 3y step) with bootstrap 95% CI
    (6) Bias heatmap of mean(M–F) residual across age bins (scale fixed to ±7.5)

Requires: pandas, numpy, scipy, matplotlib, seaborn, statsmodels
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.formula.api as smf

OLD_FILE = "test_preds_covid.csv"
NEW_FILE = "test_preds_full_with_sex.csv"

# -------------------------
# Helpers
# -------------------------
def standardize(df, source):
    """
    Return a clean DataFrame with columns: Age, y_pred, Sex, residual, abs_res, Age_bin.
    Handles either schema (OLD or NEW) and drops **top 5% |residual| per dataset**.
    """
    df = df.rename(columns=lambda x: x.strip())
    # Detect columns
    age_col = next((c for c in ["Age", "y_true", "true", "age"] if c in df.columns), None)
    pred_col = next((c for c in ["y_pred", "pred", "prediction"] if c in df.columns), None)
    sex_col = next((c for c in ["Biological Sex", "biological_sex", "Sex", "gender"] if c in df.columns), None)
    if age_col is None or pred_col is None or sex_col is None:
        raise ValueError(f"[{source}] Missing required columns. Found: {list(df.columns)}")

    out = df[[age_col, pred_col, sex_col]].copy()
    out.columns = ["Age", "y_pred", "Sex"]
    out["Age"] = pd.to_numeric(out["Age"], errors="coerce")
    out["y_pred"] = pd.to_numeric(out["y_pred"], errors="coerce")
    out["Sex"] = (out["Sex"].astype(str).str.strip()
                  .replace({"F":"Female","M":"Male","female":"Female","male":"Male"}))
    out = out.dropna(subset=["Age", "y_pred", "Sex"])
    out = out[out["Sex"].isin(["Male","Female"])].copy()

    out["residual"] = out["y_pred"] - out["Age"]
    out["abs_res"]  = out["residual"].abs()

    # ---- Drop top 5% |residual| outliers for THIS dataset ----
    cut95 = out["abs_res"].quantile(0.95)
    out = out[out["abs_res"] <= cut95].copy()

    # Age bins: 0–15 then 5y bins; special [80,92]
    bins = [0, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 92]
    out["Age_bin"] = pd.cut(out["Age"], bins=bins, right=False)

    out["__source__"] = source
    return out

def bootstrap_ci_diff(m, f, nboot=3000, alpha=0.05, seed=0):
    """Bootstrap CI for mean(m) - mean(f), resampling each group independently."""
    rng = np.random.default_rng(seed)
    m = np.asarray(m); f = np.asarray(f)
    diffs = []
    for _ in range(nboot):
        ms = rng.choice(m, size=len(m), replace=True)
        fs = rng.choice(f, size=len(f), replace=True)
        diffs.append(ms.mean() - fs.mean())
    lo, hi = np.percentile(diffs, [100*alpha/2, 100*(1-alpha/2)])
    return np.mean(diffs), lo, hi

def run_one(df, title_tag="DATASET"):
    """Make the composite figure + stats for a single dataset."""
    sns.set_theme(style="whitegrid", context="paper")
    female = df.query("Sex=='Female'")["residual"]
    male   = df.query("Sex=='Male'")["residual"]

    # Core stats
    mwu_p   = stats.mannwhitneyu(female, male, alternative="two-sided").pvalue
    welch_p = stats.ttest_ind(female, male, equal_var=False).pvalue
    lev_p   = stats.levene(female, male, center="median").pvalue
    ks_p    = stats.ks_2samp(female, male).pvalue
    abs_p   = stats.ttest_ind(np.abs(female), np.abs(male), equal_var=False).pvalue

    ols = smf.ols("residual ~ Age * Sex", data=df).fit()
    int_p = ols.pvalues.get("Age:Sex[T.Male]", np.nan)

    # Rolling sex difference (12y window, step 3y)
    centers = np.arange(df["Age"].min()+6, df["Age"].max()-6, 3)
    rows = []
    for c in centers:
        sub = df[(df["Age"] >= c-6) & (df["Age"] <= c+6)]
        m = sub.loc[sub["Sex"]=="Male","residual"].values
        f = sub.loc[sub["Sex"]=="Female","residual"].values
        if len(m)>=10 and len(f)>=10:
            d, lo, hi = bootstrap_ci_diff(m, f, nboot=3000)
            rows.append((c, d, lo, hi))
    roll = pd.DataFrame(rows, columns=["center","diff","lo","hi"])

    # Bias heatmap (mean(M–F) residual per age bin), fixed color scale ±7.5
    heat = (df.groupby("Age_bin")
              .apply(lambda g: g.loc[g["Sex"]=="Male","residual"].mean()
                              - g.loc[g["Sex"]=="Female","residual"].mean())
              .to_frame("diff"))

    # ---- Plot layout ----
    fig, axes = plt.subplots(3, 2, figsize=(11, 13))
    (ax1, ax2, ax3, ax4, ax5, ax6) = axes.flat
    plt.subplots_adjust(hspace=0.6)
    fig.suptitle(f"Residuals vs Sex & Age — {title_tag}", y=0.98, fontsize=14, weight="bold")

    # 1) Violin (y-lim ±30)
    sns.violinplot(x="Sex", y="residual", data=df, inner="quart", cut=0, ax=ax1, palette="Set2")
    sns.stripplot(x="Sex", y="residual", data=df, color="k", alpha=0.15, size=2, ax=ax1)
    ax1.axhline(0, ls="--", c="gray")
    ax1.set_ylim(-30, 30)
    ax1.set_title("Residual distribution by sex")
    ax1.text(0.95, 0.9,
             f"MWU p={mwu_p:.1e}\nWelch p={welch_p:.1e}\nLevene p={lev_p:.1e}\nKS p={ks_p:.1e}",
             ha="right", va="top", transform=ax1.transAxes,
             bbox=dict(facecolor="white", alpha=0.7, edgecolor="none"), fontsize=8)

    # 2) KDE
    sns.kdeplot(data=df, x="residual", hue="Sex", fill=True, common_norm=False, alpha=0.4, ax=ax2)
    ax2.axvline(0, ls="--", c="gray")
    ax2.set_title("Residual density by sex")
    ax2.text(0.95,0.9,f"KS p={ks_p:.2e}",ha="right",va="top",
             transform=ax2.transAxes,bbox=dict(facecolor="white",alpha=0.7,edgecolor="none"),fontsize=8)

    # 3) ECDF of |residual|
    sns.ecdfplot(data=df, x="abs_res", hue="Sex", ax=ax3)
    ax3.set_title("ECDF of absolute residuals")
    ax3.text(0.95,0.1,f"Welch(|res|) p={abs_p:.2e}",ha="right",
             transform=ax3.transAxes,bbox=dict(facecolor="white",alpha=0.7,edgecolor="none"),fontsize=8)
    ax3.set_xlabel("|residual|")

    # 4) OLS: residual ~ Age × Sex (y-lim ±30)
    sns.scatterplot(data=df, x="Age", y="residual", hue="Sex", alpha=0.25, s=15, ax=ax4)
    age_grid = np.linspace(df["Age"].min(), df["Age"].max(), 200)
    for sex, col in [("Female","C0"), ("Male","C1")]:
        yhat = ols.predict(pd.DataFrame({"Age": age_grid, "Sex": sex}))
        ax4.plot(age_grid, yhat, color=col, lw=2.5, label=f"{sex} fit")
    ax4.axhline(0, ls="--", c="gray")
    ax4.set_ylim(-30, 30)
    ax4.legend(loc="lower left")
    ax4.set_title("OLS: residual ~ Age × Sex")
    ax4.text(0.95,0.9,f"Interaction p={int_p:.2e}",ha="right",va="top",
             transform=ax4.transAxes,bbox=dict(facecolor="white",alpha=0.7,edgecolor="none"),fontsize=8)

    # 5) Rolling sex difference (bootstrap CI)
    ax5.plot(roll["center"], roll["diff"], "o-", label="Male–Female mean diff")
    ax5.fill_between(roll["center"], roll["lo"], roll["hi"], color="C0", alpha=0.2, label="95% bootstrap CI")
    ax5.axhline(0, ls="--", c="gray")
    ax5.legend(fontsize=8)
    ax5.set_title("Rolling sex difference (12y window, 3y step)")
    ax5.set_xlabel("Age (window center)")
    ax5.set_ylabel("Residual mean diff (M–F)")

    # 6) Bias heatmap (scale fixed to ±7.5)
    sns.heatmap(heat.T, cmap="RdBu_r", center=0, vmin=-7.5, vmax=7.5,
                cbar_kws={"label":"Mean (M–F) residual"}, ax=ax6)
    ax6.set_xlabel("Age bin")
    ax6.set_ylabel("")
    ax6.set_title("Bias map across age bins")
    ax6.set_xticklabels([str(i) for i in heat.index.categories], rotation=45, ha="right", fontsize=8)

    plt.show()

# -------------------------
# Load data and run three ways
# -------------------------

# NEW ONLY (Age = y_true)
new_raw = pd.read_csv(NEW_FILE)
if "y_true" not in new_raw.columns:
    raise ValueError("NEW file must contain 'y_true'.")
new_raw = new_raw.rename(columns={"y_true": "Age"})  # temporary for standardizer
new_df = standardize(new_raw, "NEW")
run_one(new_df, title_tag="NEW DATA ONLY")

# COVID ONLY (OLD)
old_raw = pd.read_csv(OLD_FILE)
old_df = standardize(old_raw, "COVID ONLY")
run_one(old_df, title_tag="COVID ONLY")

# MERGED = (COVID after its own 5% drop) + (NEW after its own 5% drop)
merged = pd.concat([old_df, new_df], ignore_index=True)
run_one(merged, title_tag="MERGED (COVID + NEW)")


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Composite figure (publication-ready) for residuals vs sex & age.

Updates requested:
- Drop top 5% |residual| outliers
- Age bins include 0–15 first bin and special 80–92 bin
- Violin & OLS panels y-limits fixed to [-30, 30]
- Bias heatmap color scale fixed to [-7.5, +7.5]
- All panels shown via plt.show()

Requires: pandas, numpy, scipy, matplotlib, seaborn, statsmodels
"""

import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
import statsmodels.formula.api as smf

# -------------------------
# Load and prepare
# -------------------------
df = pd.read_csv("test_preds_covid.csv")
df = df.rename(columns=lambda x: x.strip())
df = df.dropna(subset=["Age", "y_pred", "Biological Sex"]).copy()
df["Sex"] = (
    df["Biological Sex"]
    .astype(str).str.strip()
    .replace({"F": "Female", "M": "Male", "female": "Female", "male": "Male"})
)
df = df[df["Sex"].isin(["Male", "Female"])].copy()

df["residual"] = df["y_pred"] - df["Age"]
df["abs_res"]  = df["residual"].abs()

# Drop top 5% |residual| outliers
cut95 = df["abs_res"].quantile(0.95)
df = df[df["abs_res"] <= cut95].copy()

female = df.query("Sex=='Female'")["residual"]
male   = df.query("Sex=='Male'")["residual"]

# -------------------------
# Core stats
# -------------------------
mwu_p   = stats.mannwhitneyu(female, male, alternative="two-sided").pvalue
welch_p = stats.ttest_ind(female, male, equal_var=False).pvalue
lev_p   = stats.levene(female, male, center="median").pvalue
ks_p    = stats.ks_2samp(female, male).pvalue
abs_p   = stats.ttest_ind(np.abs(female), np.abs(male), equal_var=False).pvalue

ols = smf.ols("residual ~ Age * Sex", data=df).fit()
int_p = ols.pvalues.get("Age:Sex[T.Male]", np.nan)

# -------------------------
# Helpers
# -------------------------
def bootstrap_ci_diff(m, f, nboot=3000, alpha=0.05, seed=0):
    """Bootstrap CI for mean(m)-mean(f) by resampling each group separately."""
    rng = np.random.default_rng(seed)
    m = np.asarray(m); f = np.asarray(f)
    diffs = []
    for _ in range(nboot):
        ms = rng.choice(m, size=len(m), replace=True)
        fs = rng.choice(f, size=len(f), replace=True)
        diffs.append(ms.mean() - fs.mean())
    lo, hi = np.percentile(diffs, [100*alpha/2, 100*(1-alpha/2)])
    return np.mean(diffs), lo, hi

# Rolling diff (12y window, step 3y)
centers = np.arange(df["Age"].min() + 6, df["Age"].max() - 6, 3)
rows = []
for c in centers:
    sub = df[(df["Age"] >= c - 6) & (df["Age"] <= c + 6)]
    m = sub.loc[sub["Sex"] == "Male",   "residual"].values
    f = sub.loc[sub["Sex"] == "Female", "residual"].values
    if len(m) >= 10 and len(f) >= 10:
        d, lo, hi = bootstrap_ci_diff(m, f, nboot=3000)
        rows.append((c, d, lo, hi))
roll = pd.DataFrame(rows, columns=["center", "diff", "lo", "hi"])

# Bias heatmap bins: 0–15, then 5y bins, special 80–92
bins = [0, 15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 92, 100]
df["Age_bin"] = pd.cut(df["Age"], bins, right=False)
heat = (df.groupby("Age_bin")
          .apply(lambda g: g.loc[g["Sex"] == "Male", "residual"].mean()
                          - g.loc[g["Sex"] == "Female", "residual"].mean())
          .to_frame("diff"))

# -------------------------
# Plot style & layout
# -------------------------
sns.set_theme(style="whitegrid", context="paper")
fig, axes = plt.subplots(3, 2, figsize=(11, 13))
(ax1, ax2, ax3, ax4, ax5, ax6) = axes.flat
plt.subplots_adjust(hspace=0.6)

# 1) Violin (y-lim ±30)
sns.violinplot(x="Sex", y="residual", data=df, inner="quart", cut=0, ax=ax1, palette="Set2")
sns.stripplot(x="Sex", y="residual", data=df, color="k", alpha=0.15, size=2, ax=ax1)
ax1.axhline(0, ls="--", c="gray")
ax1.set_ylim(-30, 30)  # requested
ax1.set_title("Residual distribution by sex")
ax1.text(
    0.95, 0.9,
    f"MWU p={mwu_p:.1e}\nWelch p={welch_p:.1e}\nLevene p={lev_p:.1e}\nKS p={ks_p:.1e}",
    ha="right", va="top", transform=ax1.transAxes,
    bbox=dict(facecolor="white", alpha=0.7, edgecolor="none"), fontsize=8
)

# 2) KDE
sns.kdeplot(data=df, x="residual", hue="Sex", fill=True, common_norm=False, alpha=0.4, ax=ax2)
ax2.axvline(0, ls="--", c="gray")
ax2.set_title("Residual density by sex")
ax2.text(
    0.95, 0.9, f"KS p={ks_p:.2e}",
    ha="right", va="top", transform=ax2.transAxes,
    bbox=dict(facecolor="white", alpha=0.7, edgecolor="none"), fontsize=8
)

# 3) ECDF of |residual|
sns.ecdfplot(data=df, x="abs_res", hue="Sex", ax=ax3)
ax3.set_title("ECDF of absolute residuals")
ax3.text(
    0.95, 0.1, f"Welch(|res|) p={abs_p:.2e}",
    ha="right", transform=ax3.transAxes,
    bbox=dict(facecolor="white", alpha=0.7, edgecolor="none"), fontsize=8
)
ax3.set_xlabel("|residual|")

# 4) OLS: residual ~ Age * Sex (y-lim ±30)
sns.scatterplot(data=df, x="Age", y="residual", hue="Sex", alpha=0.25, s=15, ax=ax4)
age_grid = np.linspace(df["Age"].min(), df["Age"].max(), 200)
for sex, col in [("Female", "C0"), ("Male", "C1")]:
    yhat = ols.predict(pd.DataFrame({"Age": age_grid, "Sex": sex}))
    ax4.plot(age_grid, yhat, color=col, lw=2.5, label=f"{sex} fit")
ax4.axhline(0, ls="--", c="gray")
ax4.set_ylim(-30, 30)  # requested
ax4.legend(loc="lower left")
ax4.set_title("OLS: residual ~ Age × Sex")
ax4.text(
    0.95, 0.9, f"Interaction p={int_p:.2e}",
    ha="right", va="top", transform=ax4.transAxes,
    bbox=dict(facecolor="white", alpha=0.7, edgecolor="none"), fontsize=8
)

# 5) Rolling sex difference
ax5.plot(roll["center"], roll["diff"], "o-", label="Male–Female mean diff")
ax5.fill_between(roll["center"], roll["lo"], roll["hi"], color="C0", alpha=0.2, label="95% bootstrap CI")
ax5.axhline(0, ls="--", c="gray")
ax5.legend(fontsize=8)
ax5.set_title("Rolling sex difference (95% CI)")
ax5.set_xlabel("Age (center of ~12y window)")
ax5.set_ylabel("Residual mean diff (M–F)")

# 6) Bias heatmap with fixed scale [-7.5, +7.5]
sns.heatmap(
    heat.T, cmap="RdBu_r", center=0, vmin=-7.5, vmax=7.5,
    cbar_kws={"label": "Mean (M–F) residual"}, ax=ax6
)
ax6.set_xlabel("Age bin")
ax6.set_ylabel("")
ax6.set_title("Bias map across age bins")
ax6.set_xticklabels([str(i) for i in heat.index.categories], rotation=45, ha="right", fontsize=8)



plt.show()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.formula.api as smf

# Load data
df = pd.read_csv("test_preds_covid.csv")

# Filter for combo_index == 2 (User's correction)
if 'combo_index' in df.columns:
    df = df[df['combo_index'] == 2].copy()

# Rename columns to standard names
df = df.rename(columns=lambda x: x.strip())
# Map column names if they differ
if 'Biological Sex' in df.columns:
    df['Sex'] = df['Biological Sex']
if 'y_true' in df.columns:
    df['Age'] = df['y_true']

# Clean Sex column
df["Sex"] = (
    df["Sex"]
    .astype(str).str.strip()
    .replace({"F": "Female", "M": "Male", "female": "Female", "male": "Male"})
)
df = df[df["Sex"].isin(["Male", "Female"])].copy()

# Ensure numeric types
df['Age'] = pd.to_numeric(df['Age'], errors='coerce')
df['y_pred'] = pd.to_numeric(df['y_pred'], errors='coerce')

# Drop NaNs
df = df.dropna(subset=["Age", "y_pred", "Sex"]).copy()

# Calculate residuals
df["residual"] = df["y_pred"] - df["Age"]
df["abs_res"]  = df["residual"].abs()

# Drop top 5% |residual| outliers
cut95 = df["abs_res"].quantile(0.95)
df = df[df["abs_res"] <= cut95].copy()

female = df.query("Sex=='Female'")["residual"]
male   = df.query("Sex=='Male'")["residual"]

# Core stats
mwu_p   = stats.mannwhitneyu(female, male, alternative="two-sided").pvalue
welch_p = stats.ttest_ind(female, male, equal_var=False).pvalue
lev_p   = stats.levene(female, male, center="median").pvalue
ks_p    = stats.ks_2samp(female, male).pvalue
abs_p   = stats.ttest_ind(np.abs(female), np.abs(male), equal_var=False).pvalue

# OLS
ols = smf.ols("residual ~ Age * Sex", data=df).fit()
# Check available parameter names
# print(ols.pvalues.index)
# Standard naming is usually Age:Sex[T.Male]
int_p = ols.pvalues.get("Age:Sex[T.Male]", np.nan)
if np.isnan(int_p):
    # Try alternate naming if Sex reference is Male
    int_p = ols.pvalues.get("Age:Sex[T.Female]", np.nan)

# Helpers
def bootstrap_ci_diff(m, f, nboot=3000, alpha=0.05, seed=0):
    rng = np.random.default_rng(seed)
    m = np.asarray(m); f = np.asarray(f)
    diffs = []
    if len(m) == 0 or len(f) == 0:
        return np.nan, np.nan, np.nan
    for _ in range(nboot):
        ms = rng.choice(m, size=len(m), replace=True)
        fs = rng.choice(f, size=len(f), replace=True)
        diffs.append(ms.mean() - fs.mean())
    lo, hi = np.percentile(diffs, [100*alpha/2, 100*(1-alpha/2)])
    return np.mean(diffs), lo, hi

# Rolling diff
centers = np.arange(df["Age"].min() + 6, df["Age"].max() - 6, 3)
rows = []
for c in centers:
    sub = df[(df["Age"] >= c - 6) & (df["Age"] <= c + 6)]
    m = sub.loc[sub["Sex"] == "Male",   "residual"].values
    f = sub.loc[sub["Sex"] == "Female", "residual"].values
    if len(m) >= 10 and len(f) >= 10:
        d, lo, hi = bootstrap_ci_diff(m, f, nboot=1000) # Reduced boot for speed in preview
        rows.append((c, d, lo, hi))
roll = pd.DataFrame(rows, columns=["center", "diff", "lo", "hi"])

# Bias heatmap bins: 0–15, then 5y bins, special 80–92
# The prompt says: "Age bins include 0–15 first bin and special 80–92 bin"
# Standard ranges usually fill the middle. Let's infer standard 5y steps between 15 and 80.
bins = [0, 15] + list(range(20, 85, 5)) + [92, 100]
# Clean up overlap/gaps: range(20, 85, 5) gives 20, 25, ..., 80.
# So bins: 0, 15, 20, 25, ..., 80, 92, 100.
# Wait, 15 to 20 is a 5y bin.
bins = [0, 15] + list(range(20, 85, 5)) + [92, 100]
bins = sorted(list(set(bins))) # Ensure sorted and unique

df["Age_bin"] = pd.cut(df["Age"], bins, right=False)

# Check for empty bins or NaNs
# Groupby and calculate difference
heat_data = []
bin_labels = []

# Using explicit iteration to preserve order and handle empty bins gracefully
# (groupby might skip empty categories if observed=True/False isn't handled carefully)
# But standard pandas groupby on categorical usually keeps all categories if we set them up right.
# Let's just use the dataframe derived from cut.

heat = (df.groupby("Age_bin", observed=False)
          .apply(lambda g: g.loc[g["Sex"] == "Male", "residual"].mean()
                           - g.loc[g["Sex"] == "Female", "residual"].mean())
          .to_frame("diff"))

# Plot
sns.set_theme(style="whitegrid", context="paper")
fig, axes = plt.subplots(3, 2, figsize=(11, 13))
(ax1, ax2, ax3, ax4, ax5, ax6) = axes.flat
plt.subplots_adjust(hspace=0.4, wspace=0.3)

# 1) Violin
sns.violinplot(x="Sex", y="residual", data=df, inner="quart", cut=0, ax=ax1, palette="Set2")
sns.stripplot(x="Sex", y="residual", data=df, color="k", alpha=0.15, size=2, ax=ax1)
ax1.axhline(0, ls="--", c="gray")
ax1.set_ylim(-30, 30)
ax1.set_title("Residual distribution by sex")
ax1.text(0.95, 0.9, f"MWU p={mwu_p:.1e}\nWelch p={welch_p:.1e}\nLevene p={lev_p:.1e}\nKS p={ks_p:.1e}",
         ha="right", va="top", transform=ax1.transAxes,
         bbox=dict(facecolor="white", alpha=0.7, edgecolor="none"), fontsize=8)

# 2) KDE
sns.kdeplot(data=df, x="residual", hue="Sex", fill=True, common_norm=False, alpha=0.4, ax=ax2)
ax2.axvline(0, ls="--", c="gray")
ax2.set_title("Residual density by sex")
ax2.text(0.95, 0.9, f"KS p={ks_p:.2e}", ha="right", va="top", transform=ax2.transAxes,
         bbox=dict(facecolor="white", alpha=0.7, edgecolor="none"), fontsize=8)

# 3) ECDF
sns.ecdfplot(data=df, x="abs_res", hue="Sex", ax=ax3)
ax3.set_title("ECDF of absolute residuals")
ax3.text(0.95, 0.1, f"Welch(|res|) p={abs_p:.2e}", ha="right", transform=ax3.transAxes,
         bbox=dict(facecolor="white", alpha=0.7, edgecolor="none"), fontsize=8)
ax3.set_xlabel("|residual|")

# 4) OLS
sns.scatterplot(data=df, x="Age", y="residual", hue="Sex", alpha=0.25, s=15, ax=ax4)
age_grid = np.linspace(df["Age"].min(), df["Age"].max(), 200)
# Predict manually or use model
# Simple lines for display
for sex, col in [("Female", "C0"), ("Male", "C1")]:
    # Filter data to fit separate lines for viz if needed, or use the interaction model predictions
    # Using interaction model predictions:
    pred_data = pd.DataFrame({"Age": age_grid, "Sex": sex})
    yhat = ols.predict(pred_data)
    ax4.plot(age_grid, yhat, color=col, lw=2.5, label=f"{sex} fit")

ax4.axhline(0, ls="--", c="gray")
ax4.set_ylim(-30, 30)
ax4.legend(loc="lower left")
ax4.set_title("OLS: residual ~ Age × Sex")
ax4.text(0.95, 0.9, f"Interaction p={int_p:.2e}", ha="right", va="top", transform=ax4.transAxes,
         bbox=dict(facecolor="white", alpha=0.7, edgecolor="none"), fontsize=8)

# 5) Rolling
if not roll.empty:
    ax5.plot(roll["center"], roll["diff"], "o-", label="Male–Female mean diff")
    ax5.fill_between(roll["center"], roll["lo"], roll["hi"], color="C0", alpha=0.2, label="95% bootstrap CI")
ax5.axhline(0, ls="--", c="gray")
ax5.legend(fontsize=8)
ax5.set_title("Rolling sex difference (95% CI)")
ax5.set_xlabel("Age (center of ~12y window)")
ax5.set_ylabel("Residual mean diff (M–F)")

# 6) Heatmap
sns.heatmap(heat.T, cmap="RdBu_r", center=0, vmin=-7.5, vmax=7.5,
            cbar_kws={"label": "Mean (M–F) residual"}, ax=ax6)
ax6.set_xlabel("Age bin")
ax6.set_ylabel("")
ax6.set_title("Bias map across age bins")
# Fix tick labels
ax6.set_xticklabels([str(i) for i in heat.index], rotation=45, ha="right", fontsize=8)

plt.tight_layout()
plt.savefig("figure_composite_corrected.png", dpi=150)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import norm

# ==========================================
# Nature Style Settings
# ==========================================
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 8
plt.rcParams['axes.linewidth'] = 0.8

def plot_selection_schematic():
    fig, ax = plt.subplots(figsize=(4, 3), dpi=300)

    # 1. Generate Dummy Data (Approximating the GMM distribution described)
    # Mixture of 3 Gaussians to look realistic (Middle bulk + 2 tails)
    np.random.seed(42)
    data = np.concatenate([
        np.random.normal(0, 1, 100000),      # Background (Non-age)
        np.random.normal(-3, 1.5, 5000),     # Young signal
        np.random.normal(3, 1.5, 5000)       # Old signal
    ])

    # Thresholds (Z-score 1.96 approx)
    thresh_low = -1.96
    thresh_high = 1.96

    # 2. Plot Histogram (Background)
    # We use a filled KDE or Histogram
    sns.kdeplot(data, color="grey", fill=True, alpha=0.2, linewidth=0, ax=ax)
    line = sns.kdeplot(data, color="grey", linewidth=1, ax=ax).get_lines()[0]
    x_data, y_data = line.get_data()

    # 3. Color the Tails
    # Young (Left)
    ax.fill_between(x_data, 0, y_data, where=(x_data <= thresh_low),
                    color='#0072B2', alpha=0.8, label='Young-associated')

    # Old (Right)
    ax.fill_between(x_data, 0, y_data, where=(x_data >= thresh_high),
                    color='#D55E00', alpha=0.8, label='Old-associated')

    # 4. Add Threshold Lines & P-values
    # Left Line
    ax.axvline(thresh_low, color='black', linestyle='--', linewidth=0.8, alpha=0.5)
    ax.text(thresh_low - 0.5, max(y_data)*0.55, 'p < 0.05\n(Z < -1.96)',
            ha='right', fontsize=7, color='#333')

    # Right Line
    ax.axvline(thresh_high, color='black', linestyle='--', linewidth=0.8, alpha=0.5)
    ax.text(thresh_high + 0.5, max(y_data)*0.55, 'p < 0.05\n(Z > 1.96)',
            ha='left', fontsize=7, color='#333')

    # 5. Add Group Sizes (N)
    # Young Arrow & Text
    ax.annotate(f'n = 3,732', xy=(-3.5, max(y_data)*0.15), xytext=(-5.5, max(y_data)*0.35),
                arrowprops=dict(arrowstyle='->', color='#0072B2'),
                color='#0072B2', fontweight='bold', ha='center')

    # Old Arrow & Text
    ax.annotate(f'n = 3,771', xy=(3.5, max(y_data)*0.15), xytext=(5.5, max(y_data)*0.35),
                arrowprops=dict(arrowstyle='->', color='#D55E00'),
                color='#D55E00', fontweight='bold', ha='center')

    # Background Text
    ax.text(0, max(y_data)*0.1, 'Background Repertoire\n(Non-Age)',
            ha='center', fontsize=7, color='grey', alpha=0.8)

    # Styling
    ax.set_title('a  Selection of Age-Associated TCRs', loc='left', fontweight='bold')
    ax.set_xlabel('Signed Wasserstein Score')
    ax.set_ylabel('Density')
    ax.set_yticks([]) # Hide Y numbers (schematic)
    ax.set_xlim(-6, 6)

    # Remove top/right spines
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)

    plt.tight_layout()
    plt.savefig('Figure3a_SelectionSchematic.png', dpi=300)
    plt.show()

plot_selection_schematic()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from matplotlib.patches import FancyArrowPatch

# ==========================================
# 1. Nature Style Settings
# ==========================================
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 10
plt.rcParams['axes.linewidth'] = 1
plt.rcParams['xtick.major.width'] = 1
plt.rcParams['ytick.major.width'] = 1
plt.rcParams['svg.fonttype'] = 'none' # For editable text in Illustrator

# Colors
c_young = '#0072B2' # Blue
c_old = '#D55E00'   # Vermilion
c_global = 'grey'

# ==========================================
# Function 1: Single TCR Distribution (Micro View)
# ==========================================
def plot_single_tcr_schematic(shift_type='young', filename='tcr_schematic.png'):
    fig, ax = plt.subplots(figsize=(3.5, 2.5), dpi=300)

    x = np.linspace(0, 100, 500)

    # Global Distribution (Background)
    mu_global = 50; sigma_global = 18
    y_global = stats.norm.pdf(x, mu_global, sigma_global)

    # Specific TCR Distribution
    if shift_type == 'young':
        mu_tcr = 25; sigma_tcr = 8
        color = c_young
        title = "Example: Young-Associated TCR"
        score_text = "Negative Score"
        arrow_start = mu_global
        arrow_end = mu_tcr
    else:
        mu_tcr = 75; sigma_tcr = 8
        color = c_old
        title = "Example: Old-Associated TCR"
        score_text = "Positive Score"
        arrow_start = mu_global
        arrow_end = mu_tcr

    y_tcr = stats.norm.pdf(x, mu_tcr, sigma_tcr)

    # Plotting
    # 1. Global
    ax.fill_between(x, y_global, color=c_global, alpha=0.15)
    ax.plot(x, y_global, color=c_global, lw=1.5, linestyle='--', label='Global Population')

    # 2. Specific
    ax.fill_between(x, y_tcr, color=color, alpha=0.3)
    ax.plot(x, y_tcr, color=color, lw=2.5, label='Specific Clonotype')

    # 3. Wasserstein Arrow
    # Draw arrow between peaks
    y_arrow = max(y_global) * 0.7
    arrow = FancyArrowPatch((arrow_start, y_arrow), (arrow_end, y_arrow),
                            arrowstyle='-|>', mutation_scale=15,
                            color='black', lw=1.5)
    ax.add_patch(arrow)

    ax.text((arrow_start + arrow_end)/2, y_arrow + 0.005, 'Shift',
            ha='center', va='bottom', fontsize=9, fontweight='bold')

    # Styling
    ax.set_title(title, fontsize=9, fontweight='bold')
    ax.set_xlabel('Age of Carrier (years)', fontsize=8)
    ax.set_ylabel('Density', fontsize=8)
    ax.set_yticks([])
    ax.set_xlim(0, 100)
    ax.set_ylim(0, max(y_tcr)*1.2)

    # Clean spines
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)

    plt.tight_layout()
    plt.savefig(filename, dpi=300, transparent=True)
    plt.show()

# ==========================================
# Function 2: Population Histogram (Macro View)
# ==========================================
def plot_population_selection(filename='selection_schematic.png'):
    fig, ax = plt.subplots(figsize=(5, 3), dpi=300)

    # Generate Synthetic GMM-like Data
    np.random.seed(42)
    # Mixture: 80% background (centered at 0), 10% young (-3), 10% old (+3)
    data = np.concatenate([
        np.random.normal(0, 1, 8000),
        np.random.normal(-3.5, 1.2, 1000),
        np.random.normal(3.5, 1.2, 1000)
    ])

    # Plot Density
    sns.kdeplot(data, color="grey", fill=True, alpha=0.1, linewidth=0, ax=ax)
    line = sns.kdeplot(data, color="#555", linewidth=1.5, ax=ax).get_lines()[0]
    x_data, y_data = line.get_data()

    # Thresholds
    thresh_low = -1.96
    thresh_high = 1.96

    # Color Tails
    ax.fill_between(x_data, 0, y_data, where=(x_data <= thresh_low),
                    color=c_young, alpha=0.8)
    ax.fill_between(x_data, 0, y_data, where=(x_data >= thresh_high),
                    color=c_old, alpha=0.8)

    # Add Threshold Lines
    ax.axvline(thresh_low, color='black', linestyle=':', linewidth=1)
    ax.axvline(thresh_high, color='black', linestyle=':', linewidth=1)

    # Annotations
    ax.text(-4, max(y_data)*0.4, 'Young-Biased\n(Score < -1.96)',
            color=c_young, ha='center', fontsize=8, fontweight='bold')
    ax.text(4, max(y_data)*0.4, 'Old-Biased\n(Score > +1.96)',
            color=c_old, ha='center', fontsize=8, fontweight='bold')

    ax.text(0, max(y_data)*0.9, 'All TCRs scored\nby Wasserstein',
            color='#333', ha='center', fontsize=9)

    # Styling
    ax.set_title('Statistical Selection of Features', loc='left', fontweight='bold')
    ax.set_xlabel('Signed Wasserstein Score ($z$-score)', fontsize=9)
    ax.set_ylabel('Number of TCRs', fontsize=9)
    ax.set_yticks([])
    ax.set_xlim(-6, 6)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)

    plt.tight_layout()
    plt.savefig(filename, dpi=300, transparent=True)
    plt.show()

# ==========================================
# Generate Images
# ==========================================
# 1. Image for Young Case
plot_single_tcr_schematic(shift_type='young', filename='Asset_Young_Shift.png')

# 2. Image for Old Case
plot_single_tcr_schematic(shift_type='old', filename='Asset_Old_Shift.png')

# 3. Image for the Selection Histogram
plot_population_selection(filename='Asset_Selection_Hist.png')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import FancyArrowPatch

# Nature Style Settings
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 9
plt.rcParams['axes.linewidth'] = 0.8

def plot_comparison_strategies():
    fig, axes = plt.subplots(1, 2, figsize=(8, 3), dpi=300)

    # 1. Generate Synthetic Data (GMM-like)
    np.random.seed(42)
    # Background + Tails
    data = np.concatenate([
        np.random.normal(0, 1.2, 20000),      # Middle (Noise)
        np.random.normal(-3.5, 1, 2000),      # Young
        np.random.normal(3.5, 1, 2000)        # Old
    ])

    thresh_low = -1.96
    thresh_high = 1.96

    # Common plotting function
    def setup_plot(ax, title):
        sns.kdeplot(data, color="lightgrey", fill=True, alpha=0.3, linewidth=0, ax=ax)
        line = sns.kdeplot(data, color="grey", linewidth=1, ax=ax).get_lines()[0]
        x, y = line.get_data()
        ax.set_yticks([])
        ax.set_xlim(-6, 6)
        ax.set_xlabel('Score')
        ax.set_title(title, loc='left', fontweight='bold', fontsize=10)
        # Remove spines
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['left'].set_visible(False)
        return x, y

    # ==========================================
    # Panel 1: Young vs Old (Bifurcation)
    # ==========================================
    ax1 = axes[0]
    x1, y1 = setup_plot(ax1, 'a  Sequence Bifurcation\n    (Young vs. Old)')

    # Color Tails ONLY
    ax1.fill_between(x1, 0, y1, where=(x1 <= thresh_low), color='#0072B2', alpha=0.9, label='Young')
    ax1.fill_between(x1, 0, y1, where=(x1 >= thresh_high), color='#D55E00', alpha=0.9, label='Old')

    # Comparison Arrow
    # Arrow connecting the two tails
    y_arrow = max(y1) * 0.4
    arrow = FancyArrowPatch((-3, y_arrow), (3, y_arrow),
                            arrowstyle='<|-|>', mutation_scale=15,
                            color='black', lw=1.5)
    ax1.add_patch(arrow)
    ax1.text(0, y_arrow + 0.01, 'VS', ha='center', va='bottom', fontweight='bold')

    ax1.text(0, max(y1)*0.8, "Do sequences differ\nby age direction?",
             ha='center', fontsize=8, color='#333')

    # ==========================================
    # Panel 2: Signal vs Noise (Distinctiveness)
    # ==========================================
    ax2 = axes[1]
    x2, y2 = setup_plot(ax2, 'b  Signal Distinctiveness\n    (Age-Associated vs. Non-Age)')

    # Color Tails (Unified Group)
    ax2.fill_between(x2, 0, y2, where=(x2 <= thresh_low), color='#663399', alpha=0.7) # Purple implies mix
    ax2.fill_between(x2, 0, y2, where=(x2 >= thresh_high), color='#663399', alpha=0.7)

    # Color Middle (Background)
    ax2.fill_between(x2, 0, y2, where=((x2 > thresh_low) & (x2 < thresh_high)),
                     color='grey', alpha=0.6, hatch='///')

    # Annotations for Groups
    ax2.text(-4, max(y2)*0.2, "Signal", color='#663399', fontweight='bold', ha='center')
    ax2.text(4, max(y2)*0.2, "Signal", color='#663399', fontweight='bold', ha='center')
    ax2.text(0, max(y2)*0.2, "Noise\n(Background)", color='#333', ha='center', fontsize=8)

    # Comparison Arrows (Tail vs Middle)
    # Left tail to middle
    ax2.annotate("", xy=(-1, max(y2)*0.5), xytext=(-3, max(y2)*0.5),
                 arrowprops=dict(arrowstyle="<|-|>", color="black", lw=1.2))
    # Right tail to middle
    ax2.annotate("", xy=(1, max(y2)*0.5), xytext=(3, max(y2)*0.5),
                 arrowprops=dict(arrowstyle="<|-|>", color="black", lw=1.2))

    ax2.text(0, max(y2)*0.8, "Do age-TCRs differ\nfrom random TCRs?",
             ha='center', fontsize=8, color='#333')

    plt.tight_layout()
    plt.savefig('Comparison_Schematic.png', dpi=300)
    plt.show()

plot_comparison_strategies()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, r2_score
from scipy import stats
from matplotlib import gridspec
import statsmodels.formula.api as smf

# ==========================================
# 1. Nature Style Settings
# ==========================================
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 7
plt.rcParams['axes.linewidth'] = 0.5
plt.rcParams['xtick.major.width'] = 0.5
plt.rcParams['ytick.major.width'] = 0.5
plt.rcParams['axes.titleweight'] = 'bold'

# Colors
c_main = '#E69F00'   # Gold/Orange for Regression
c_male = '#D55E00'   # Vermilion
c_female = '#0072B2' # Blue

# ==========================================
# 2. Data Loading & Prep
# ==========================================
def load_data():
    # --- Load COVID Data (Combo 2) ---
    df_cov = pd.read_csv('test_preds_covid.csv')
    if 'combo_index' in df_cov.columns:
        df_cov = df_cov[df_cov['combo_index'] == 2].copy()

    # Standardize names
    df_cov.rename(columns={'Age': 'y_true', 'Biological Sex': 'Sex'}, inplace=True)
    df_cov['y_true'] = pd.to_numeric(df_cov['y_true'], errors='coerce')
    df_cov['y_pred'] = pd.to_numeric(df_cov['y_pred'], errors='coerce')
    df_cov.dropna(subset=['y_true', 'y_pred', 'Sex'], inplace=True)

    # Calculate Residuals
    df_cov['residual'] = df_cov['y_pred'] - df_cov['y_true']
    df_cov['Abs_Residual'] = df_cov['residual'].abs()

    # --- Load Healthy Data (Placeholder using the other file) ---
    try:
        df_health = pd.read_csv('test_preds_full_with_sex.csv')
        df_health.rename(columns={'biological_sex': 'Sex'}, inplace=True) # Check column names
        # Assuming columns are y_true, y_pred based on previous checks
    except:
        df_health = pd.DataFrame({'y_true': [], 'y_pred': []}) # Empty fallback

    return df_cov, df_health

# ==========================================
# 3. Plotting Function
# ==========================================
def plot_figure2_final(df_cov, df_health):
    # Layout: 180mm width (~7.08 inches), Height ~120mm
    fig = plt.figure(figsize=(7.08, 5.5), dpi=300)

    # Grid: 2 Rows. Top row has 2 cols, Bottom row has 3 cols.
    gs = gridspec.GridSpec(2, 6, height_ratios=[1, 0.7], hspace=0.5, wspace=1.0)

    # ---------------------------------------------------
    # TOP ROW: PERFORMANCE (a, b)
    # ---------------------------------------------------

    # --- Panel a: COVID Regression ---
    ax1 = fig.add_subplot(gs[0, 0:3])
    mae_cov = mean_absolute_error(df_cov['y_true'], df_cov['y_pred'])
    r2_cov = r2_score(df_cov['y_true'], df_cov['y_pred'])

    ax1.scatter(df_cov['y_true'], df_cov['y_pred'], color=c_main, alpha=0.3, s=8, edgecolors='none')
    ax1.plot([0, 100], [0, 100], 'k--', lw=0.8)

    ax1.set_xlabel('Chronological Age (years)')
    ax1.set_ylabel('Predicted Age (years)')
    ax1.set_title('a  COVID-19 Cohort (Hold-out)', loc='left')

    # Stats Box
    text_str = f'MAE = {mae_cov:.2f} yr\n$R^2$ = {r2_cov:.2f}'
    ax1.text(0.05, 0.85, text_str, transform=ax1.transAxes, fontsize=7)

    ax1.spines['top'].set_visible(False)
    ax1.spines['right'].set_visible(False)

    # --- Panel b: Healthy Regression ---
    ax2 = fig.add_subplot(gs[0, 3:6])
    if not df_health.empty:
        mae_health = mean_absolute_error(df_health['y_true'], df_health['y_pred'])
        r2_health = r2_score(df_health['y_true'], df_health['y_pred'])

        ax2.scatter(df_health['y_true'], df_health['y_pred'], color=c_main, alpha=0.3, s=8, edgecolors='none')
        ax2.plot([0, 100], [0, 100], 'k--', lw=0.8)

        ax2.set_title('b  Healthy Cohort (Transfer)', loc='left')
        ax2.text(0.05, 0.85, f'MAE = {mae_health:.2f} yr\n$R^2$ = {r2_health:.2f}', transform=ax2.transAxes)
    else:
        ax2.text(0.5, 0.5, "Healthy Data Not Loaded", ha='center')

    ax2.set_xlabel('Chronological Age (years)')
    ax2.set_ylabel('Predicted Age (years)')
    ax2.spines['top'].set_visible(False)
    ax2.spines['right'].set_visible(False)

    # ---------------------------------------------------
    # BOTTOM ROW: SEX BIAS ANALYSIS (c, d, e) - Using COVID Data
    # ---------------------------------------------------

    # Filter Outliers for stats (Top 5%) - As discussed
    cut95 = df_cov["Abs_Residual"].quantile(0.95)
    df_stats = df_cov[df_cov["Abs_Residual"] <= cut95].copy()

    palette = {'Female': c_female, 'Male': c_male}

    # Stats Calculations
    male_res = df_stats[df_stats['Sex'] == 'Male']['residual']
    female_res = df_stats[df_stats['Sex'] == 'Female']['residual']
    mwu_p = stats.mannwhitneyu(male_res, female_res).pvalue

    male_abs = df_stats[df_stats['Sex'] == 'Male']['Abs_Residual']
    female_abs = df_stats[df_stats['Sex'] == 'Female']['Abs_Residual']
    ks_p = stats.ks_2samp(male_abs, female_abs).pvalue

    # Interaction
    try:
        model = smf.ols('residual ~ y_true * Sex', data=df_stats).fit()
        p_keys = [k for k in model.pvalues.keys() if ':' in k]
        int_p = model.pvalues[p_keys[0]] if p_keys else 1.0
    except:
        int_p = 1.0

    # --- Panel c: Systematic Offset ---
    ax3 = fig.add_subplot(gs[1, 0:2])
    sns.violinplot(data=df_stats, x='Sex', y='residual', ax=ax3, palette=palette,
                   inner='quartile', linewidth=0.8, saturation=0.9, order=['Female', 'Male'])
    ax3.axhline(0, color='gray', linestyle='--', lw=0.8)
    ax3.set_xlabel('')
    ax3.set_ylabel('Age Residual (Pred - True)')
    ax3.set_title('c  Systematic Offset', loc='left')
    # Add P-value
    p_text = f'p < 0.001' if mwu_p < 0.001 else f'p = {mwu_p:.3f}'
    ax3.text(0.5, 0.9, p_text, transform=ax3.transAxes, ha='center', fontsize=6)
    ax3.spines['top'].set_visible(False)
    ax3.spines['right'].set_visible(False)
    ax3.set_ylim(-30, 30)

    # --- Panel d: Error Magnitude ---
    ax4 = fig.add_subplot(gs[1, 2:4])
    sns.ecdfplot(data=df_stats, x='Abs_Residual', hue='Sex', ax=ax4, palette=palette, lw=1.5)
    ax4.set_xlabel('Absolute Error (years)')
    ax4.set_ylabel('Cumulative Proportion')
    ax4.set_title('d  Error Magnitude', loc='left')
    ax4.text(0.5, 0.2, f'KS p = {ks_p:.2f}', transform=ax4.transAxes, ha='center', fontsize=6)
    ax4.legend_.remove()
    ax4.spines['top'].set_visible(False)
    ax4.spines['right'].set_visible(False)

    # --- Panel e: Interaction ---
    ax5 = fig.add_subplot(gs[1, 4:6])
    sns.regplot(data=df_stats[df_stats['Sex']=='Female'], x='y_true', y='residual', ax=ax5,
                scatter_kws={'alpha':0.1, 's':4}, line_kws={'lw':1.5}, color=c_female, label='Female')
    sns.regplot(data=df_stats[df_stats['Sex']=='Male'], x='y_true', y='residual', ax=ax5,
                scatter_kws={'alpha':0.1, 's':4}, line_kws={'lw':1.5}, color=c_male, label='Male')

    ax5.axhline(0, color='gray', linestyle='--', lw=0.8)
    ax5.set_xlabel('Chronological Age')
    ax5.set_ylabel('Age Residual')
    ax5.set_title('e  Age Interaction', loc='left')
    ax5.text(0.5, 0.85, f'Interact p = {int_p:.2f}', transform=ax5.transAxes, fontsize=6, ha='center')
    ax5.legend(frameon=False, fontsize=6, loc='lower left')
    ax5.spines['top'].set_visible(False)
    ax5.spines['right'].set_visible(False)
    ax5.set_ylim(-30, 30)

    plt.tight_layout()
    plt.savefig('Figure2_Complete_Nature.pdf', dpi=300, transparent=True)
    plt.show()

# ==========================================
# 4. Execute
# ==========================================
df_covid, df_healthy = load_data()
plot_figure2_final(df_covid, df_healthy)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, r2_score
from scipy import stats
from matplotlib import gridspec
import statsmodels.formula.api as smf

# ==========================================
# 1. Nature Style Settings
# ==========================================
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 7
plt.rcParams['axes.linewidth'] = 0.5
plt.rcParams['xtick.major.width'] = 0.5
plt.rcParams['ytick.major.width'] = 0.5
plt.rcParams['axes.titleweight'] = 'bold'

# Colors
c_main = '#E69F00'
c_male = '#D55E00'
c_female = '#0072B2'

# ==========================================
# 2. Data Loading & Prep (With COMBINATION)
# ==========================================
def load_and_combine_data():
    # --- Load COVID Data (Combo 2) ---
    df_cov = pd.read_csv('test_preds_covid.csv')
    if 'combo_index' in df_cov.columns:
        df_cov = df_cov[df_cov['combo_index'] == 2].copy()

    # Standardize names
    df_cov.rename(columns={'Age': 'y_true', 'Biological Sex': 'Sex'}, inplace=True)
    df_cov['y_true'] = pd.to_numeric(df_cov['y_true'], errors='coerce')
    df_cov['y_pred'] = pd.to_numeric(df_cov['y_pred'], errors='coerce')
    df_cov.dropna(subset=['y_true', 'y_pred', 'Sex'], inplace=True)
    df_cov['Cohort'] = 'COVID'

    # --- Load Healthy Data ---
    try:
        df_health = pd.read_csv('test_preds_full_with_sex.csv')
        df_health.rename(columns={'biological_sex': 'Sex'}, inplace=True)
        df_health['y_true'] = pd.to_numeric(df_health['y_true'], errors='coerce')
        df_health['y_pred'] = pd.to_numeric(df_health['y_pred'], errors='coerce')
        df_health.dropna(subset=['y_true', 'y_pred', 'Sex'], inplace=True)
        df_health['Cohort'] = 'Healthy'
    except:
        df_health = pd.DataFrame({'y_true': [], 'y_pred': [], 'Sex': [], 'Cohort': []})

    # --- Combine for Bottom Row ---
    cols = ['y_true', 'y_pred', 'Sex', 'Cohort']
    df_combined = pd.concat([df_cov[cols], df_health[cols]], ignore_index=True)

    # Calculate Residuals
    df_combined['residual'] = df_combined['y_pred'] - df_combined['y_true']
    df_combined['Abs_Residual'] = df_combined['residual'].abs()

    return df_cov, df_health, df_combined

# ==========================================
# 3. Plotting Function (Using Combined Data for Bottom Row)
# ==========================================
def plot_figure2_final_combined(df_cov, df_health, df_combined):
    fig = plt.figure(figsize=(7.08, 5.5), dpi=300)
    gs = gridspec.GridSpec(2, 6, height_ratios=[1, 0.7], hspace=0.5, wspace=1.0)

    # ---------------------------------------------------
    # TOP ROW: PERFORMANCE (a, b) - SEPARATE
    # ---------------------------------------------------

    # --- Panel a: COVID Regression ---
    ax1 = fig.add_subplot(gs[0, 0:3])
    mae_cov = mean_absolute_error(df_cov['y_true'], df_cov['y_pred'])
    r2_cov = r2_score(df_cov['y_true'], df_cov['y_pred'])

    ax1.scatter(df_cov['y_true'], df_cov['y_pred'], color=c_main, alpha=0.3, s=8, edgecolors='none')
    ax1.plot([0, 100], [0, 100], 'k--', lw=0.8)

    ax1.set_xlabel('Chronological Age (years)')
    ax1.set_ylabel('Predicted Age (years)')
    ax1.set_title('a  COVID-19 Cohort (Hold-out)', loc='left')
    ax1.text(0.05, 0.85, f'MAE = {mae_cov:.2f} yr\n$R^2$ = {r2_cov:.2f}', transform=ax1.transAxes, fontsize=7)
    ax1.spines['top'].set_visible(False); ax1.spines['right'].set_visible(False)

    # --- Panel b: Healthy Regression ---
    ax2 = fig.add_subplot(gs[0, 3:6])
    if not df_health.empty:
        mae_health = mean_absolute_error(df_health['y_true'], df_health['y_pred'])
        r2_health = r2_score(df_health['y_true'], df_health['y_pred'])

        ax2.scatter(df_health['y_true'], df_health['y_pred'], color=c_main, alpha=0.3, s=8, edgecolors='none')
        ax2.plot([0, 100], [0, 100], 'k--', lw=0.8)
        ax2.set_title('b  Healthy Cohort (Transfer)', loc='left')
        # Using calculated value for consistency
        ax2.text(0.05, 0.85, f'MAE = {mae_health:.2f} yr\n$R^2$ = {r2_health:.2f}', transform=ax2.transAxes, fontsize=7)
    else:
        ax2.text(0.5, 0.5, "Healthy Data Not Loaded", ha='center')
    ax2.set_xlabel('Chronological Age (years)')
    ax2.set_ylabel('Predicted Age (years)')
    ax2.spines['top'].set_visible(False); ax2.spines['right'].set_visible(False)

    # ---------------------------------------------------
    # BOTTOM ROW: SEX BIAS ANALYSIS (c, d, e) - COMBINED DATA
    # ---------------------------------------------------

    # Filter Outliers (Top 5%) from the COMBINED set
    cut95 = df_combined["Abs_Residual"].quantile(0.95)
    df_stats = df_combined[df_combined["Abs_Residual"] <= cut95].copy()

    palette = {'Female': c_female, 'Male': c_male}

    # Stats Calculations
    male_res = df_stats[df_stats['Sex'] == 'Male']['residual']
    female_res = df_stats[df_stats['Sex'] == 'Female']['residual']
    mwu_p = stats.mannwhitneyu(male_res, female_res).pvalue

    male_abs = df_stats[df_stats['Sex'] == 'Male']['Abs_Residual']
    female_abs = df_stats[df_stats['Sex'] == 'Female']['Abs_Residual']
    ks_p = stats.ks_2samp(male_abs, female_abs).pvalue

    # Interaction
    try:
        model = smf.ols('residual ~ y_true * Sex', data=df_stats).fit()
        p_keys = [k for k in model.pvalues.keys() if ':' in k]
        int_p = model.pvalues[p_keys[0]] if p_keys else 1.0
    except:
        int_p = 1.0

    # --- Panel c: Systematic Offset ---
    ax3 = fig.add_subplot(gs[1, 0:2])
    sns.violinplot(data=df_stats, x='Sex', y='residual', ax=ax3, palette=palette,
                   inner='quartile', linewidth=0.8, saturation=0.9, order=['Female', 'Male'])
    ax3.axhline(0, color='gray', linestyle='--', lw=0.8)
    ax3.set_xlabel('')
    ax3.set_ylabel('Age Residual (Pred - True)')
    ax3.set_title('c  Systematic Offset (Combined)', loc='left')

    p_text = f'p < 0.001' if mwu_p < 0.001 else f'p = {mwu_p:.3f}'
    ax3.text(0.5, 0.9, p_text, transform=ax3.transAxes, ha='center', fontsize=6)
    ax3.spines['top'].set_visible(False); ax3.spines['right'].set_visible(False)
    ax3.set_ylim(-30, 30)

    # --- Panel d: Error Magnitude ---
    ax4 = fig.add_subplot(gs[1, 2:4])
    sns.ecdfplot(data=df_stats, x='Abs_Residual', hue='Sex', ax=ax4, palette=palette, lw=1.5)
    ax4.set_xlabel('Absolute Error (years)')
    ax4.set_ylabel('Cumulative Proportion')
    ax4.set_title('d  Error Magnitude (Combined)', loc='left')
    ax4.text(0.5, 0.2, f'KS p = {ks_p:.2f}', transform=ax4.transAxes, ha='center', fontsize=6)
    ax4.legend_.remove()
    ax4.spines['top'].set_visible(False); ax4.spines['right'].set_visible(False)

    # --- Panel e: Interaction ---
    ax5 = fig.add_subplot(gs[1, 4:6])
    sns.regplot(data=df_stats[df_stats['Sex']=='Female'], x='y_true', y='residual', ax=ax5,
                scatter_kws={'alpha':0.05, 's':4}, line_kws={'lw':1.5}, color=c_female, label='Female')
    sns.regplot(data=df_stats[df_stats['Sex']=='Male'], x='y_true', y='residual', ax=ax5,
                scatter_kws={'alpha':0.05, 's':4}, line_kws={'lw':1.5}, color=c_male, label='Male')

    ax5.axhline(0, color='gray', linestyle='--', lw=0.8)
    ax5.set_xlabel('Chronological Age')
    ax5.set_ylabel('Age Residual')
    ax5.set_title('e  Age Interaction (Combined)', loc='left')
    ax5.text(0.5, 0.85, f'Interact p = {int_p:.2f}', transform=ax5.transAxes, fontsize=6, ha='center')
    ax5.legend(frameon=False, fontsize=6, loc='lower left')
    ax5.spines['top'].set_visible(False); ax5.spines['right'].set_visible(False)
    ax5.set_ylim(-30, 30)

    plt.tight_layout()
    plt.savefig('Figure2_Complete_Nature_Combined.pdf', dpi=300, transparent=True)
    plt.show()

# ==========================================
# 4. Execute
# ==========================================
df_cov, df_health, df_combined = load_and_combine_data()
plot_figure2_final_combined(df_cov, df_health, df_combined)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib import gridspec
from scipy import stats

# Load Combined Data (Same as before)
# (Assuming df_combined and df_stats are available from previous context)
# If not, we recreate the minimal needed data structure from the logic used previously.
# Since I cannot rely on previous session variables persisting perfectly, I re-load for safety.

# ... (Data loading code identical to previous step) ...
def load_combined_data():
    df_cov = pd.read_csv('test_preds_covid.csv')
    if 'combo_index' in df_cov.columns:
        df_cov = df_cov[df_cov['combo_index'] == 2].copy()
    df_cov.rename(columns={'Age': 'y_true', 'Biological Sex': 'Sex'}, inplace=True)
    df_cov['y_true'] = pd.to_numeric(df_cov['y_true'], errors='coerce')
    df_cov['y_pred'] = pd.to_numeric(df_cov['y_pred'], errors='coerce')
    df_cov.dropna(subset=['y_true', 'y_pred', 'Sex'], inplace=True)
    df_cov['Cohort'] = 'COVID'

    try:
        df_health = pd.read_csv('test_preds_full_with_sex.csv')
        df_health.rename(columns={'biological_sex': 'Sex'}, inplace=True)
        df_health['y_true'] = pd.to_numeric(df_health['y_true'], errors='coerce')
        df_health['y_pred'] = pd.to_numeric(df_health['y_pred'], errors='coerce')
        df_health.dropna(subset=['y_true', 'y_pred', 'Sex'], inplace=True)
        df_health['Cohort'] = 'Healthy'
    except:
        df_health = pd.DataFrame()

    cols = ['y_true', 'y_pred', 'Sex', 'Cohort']
    df = pd.concat([df_cov[cols], df_health[cols]], ignore_index=True)
    df['residual'] = df['y_pred'] - df['y_true']
    df['Abs_Residual'] = df['residual'].abs()

    # Outlier removal
    cut95 = df["Abs_Residual"].quantile(0.95)
    df = df[df["Abs_Residual"] <= cut95].copy()
    return df

df_stats = load_combined_data()

# ==========================================
# Plotting the Alternative Panel C (Density)
# ==========================================
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 8

fig, ax = plt.subplots(figsize=(3, 2.5), dpi=300)

# Colors
c_male = '#D55E00'
c_female = '#0072B2'

# Plot KDE
sns.kdeplot(data=df_stats, x='residual', hue='Sex', fill=True, common_norm=False,
            palette={'Female': c_female, 'Male': c_male}, alpha=0.3, linewidth=1.5, ax=ax)

# Add Mean Lines
mean_m = df_stats[df_stats['Sex']=='Male']['residual'].mean()
mean_f = df_stats[df_stats['Sex']=='Female']['residual'].mean()

ax.axvline(mean_m, color=c_male, linestyle='--', linewidth=1)
ax.axvline(mean_f, color=c_female, linestyle='--', linewidth=1)

# Add Zero Line
ax.axvline(0, color='black', linestyle='-', linewidth=0.5, alpha=0.5)

# Annotate the Means
y_lim = ax.get_ylim()[1]
ax.text(mean_m + 1, y_lim*0.9, f'Male\n({mean_m:.1f}y)', color=c_male, fontsize=7, ha='left')
ax.text(mean_f - 1, y_lim*0.9, f'Female\n({mean_f:.1f}y)', color=c_female, fontsize=7, ha='right')

# Annotate "Shift"
ax.annotate('', xy=(mean_m, y_lim*0.6), xytext=(mean_f, y_lim*0.6),
            arrowprops=dict(arrowstyle='<->', color='black', lw=0.8))
ax.text((mean_m+mean_f)/2, y_lim*0.65, f'Δ = {mean_m - mean_f:.1f} yr', ha='center', fontsize=7, fontweight='bold')

ax.set_title('c  Systematic Offset (Density)', loc='left', fontweight='bold')
ax.set_xlabel('Age Residual (Predicted - True)')
ax.set_ylabel('Density')
ax.set_xlim(-25, 25) # Focus on the center
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig('Alternative_Panel_C_Density.png', dpi=300)
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# 1. Settings
# ==========================================
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 7
plt.rcParams['axes.linewidth'] = 0.5

# ==========================================
# 2. Data Loading (COVID Only)
# ==========================================
def load_covid_data():
    try:
        df_cov = pd.read_csv('test_preds_covid.csv')
        if 'combo_index' in df_cov.columns:
            df_cov = df_cov[df_cov['combo_index'] == 2].copy()

        df_cov.rename(columns={'Age': 'y_true', 'Biological Sex': 'Sex'}, inplace=True)

        # Ensure numeric
        df_cov['y_true'] = pd.to_numeric(df_cov['y_true'], errors='coerce')
        df_cov['y_pred'] = pd.to_numeric(df_cov['y_pred'], errors='coerce')
        df_cov.dropna(subset=['y_true', 'y_pred', 'Sex'], inplace=True)

        # Calculate Residuals
        df_cov['residual'] = df_cov['y_pred'] - df_cov['y_true']
        df_cov['Abs_Residual'] = df_cov['residual'].abs()

        # Filter Outliers (Top 5%)
        cut95 = df_cov["Abs_Residual"].quantile(0.95)
        df_cov = df_cov[df_cov["Abs_Residual"] <= cut95].copy()

        print(f"Loaded {len(df_cov)} COVID samples (Combo 2)")
        return df_cov
    except Exception as e:
        print(f"Error loading COVID data: {e}")
        return pd.DataFrame()

# ==========================================
# 3. Plotting Function (Panel F - COVID Only)
# ==========================================
def plot_binned_bias_covid(df):
    # 1. Create Age Bins
    bins = [18, 30, 40, 50, 60, 70, 80, 100]
    labels = ['18-30', '30s', '40s', '50s', '60s', '70s', '80+']
    df['Age_Bin'] = pd.cut(df['y_true'], bins=bins, labels=labels, right=False)

    # 2. Calculate Statistics per Bin
    bin_stats = []

    for b in labels:
        sub = df[df['Age_Bin'] == b]
        if len(sub) < 10: continue

        m_res = sub[sub['Sex']=='Male']['residual']
        f_res = sub[sub['Sex']=='Female']['residual']

        if len(m_res) > 5 and len(f_res) > 5:
            # Difference of Means
            diff = m_res.mean() - f_res.mean()

            # Standard Error of the Difference
            se_m = m_res.std() / np.sqrt(len(m_res))
            se_f = f_res.std() / np.sqrt(len(f_res))
            se_diff = np.sqrt(se_m**2 + se_f**2)

            bin_stats.append({'Bin': b, 'Diff': diff, 'SE': se_diff})

    stats_df = pd.DataFrame(bin_stats)

    # 3. Plot
    fig, ax = plt.subplots(figsize=(3.5, 2.5), dpi=300)

    # Zero Line
    ax.axhline(0, color='black', linewidth=0.8, linestyle='-')

    # Global Average Difference (for this dataset)
    global_diff = df[df['Sex']=='Male']['residual'].mean() - df[df['Sex']=='Female']['residual'].mean()
    ax.axhline(global_diff, color='grey', linestyle='--', linewidth=1, alpha=0.5)

    # Error Bars
    ax.errorbar(x=stats_df['Bin'], y=stats_df['Diff'], yerr=stats_df['SE']*1.96,
                fmt='o', color='#E69F00', ecolor='black', capsize=3, markersize=5, linewidth=1.5,
                label='Bias ± 95% CI')

    # Styling
    ax.set_title('f  Sex Bias Stability (COVID Cohort)', loc='left', fontweight='bold', fontsize=8)
    ax.set_ylabel('Bias (Male - Female Residual)', fontsize=7)
    ax.set_xlabel('Age Group', fontsize=7)

    # Annotate Global Shift
    ax.text(0, global_diff + 0.5, f'Avg Shift (+{global_diff:.1f}y)',
            color='grey', fontsize=6, fontweight='bold')

    sns.despine()

    # Limits
    y_max = max(stats_df['Diff'] + stats_df['SE']*2)
    y_min = min(stats_df['Diff'] - stats_df['SE']*2)
    ax.set_ylim(min(-2, y_min - 1), max(5, y_max + 1))

    plt.tight_layout()
    plt.savefig('Panel_F_Binned_Bias_COVID.pdf', dpi=300, transparent=True)
    plt.show()

# ==========================================
# 4. Execute
# ==========================================
df_covid = load_covid_data()
if not df_covid.empty:
    plot_binned_bias_covid(df_covid)
else:
    print("Dataframe is empty. Cannot plot.")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, r2_score
from scipy import stats
from matplotlib import gridspec
import statsmodels.formula.api as smf

# ==========================================
# 1. Nature Style Settings
# ==========================================
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 7
plt.rcParams['axes.linewidth'] = 0.5
plt.rcParams['xtick.major.width'] = 0.5
plt.rcParams['ytick.major.width'] = 0.5
plt.rcParams['axes.titleweight'] = 'bold'

c_main = '#E69F00'   # Gold
c_male = '#D55E00'   # Vermilion
c_female = '#0072B2' # Blue

# ==========================================
# 2. Data Loading
# ==========================================
def load_data():
    # --- A. COVID Data (Target for Bottom Row) ---
    try:
        df_cov = pd.read_csv('test_preds_covid.csv')
        if 'combo_index' in df_cov.columns:
            df_cov = df_cov[df_cov['combo_index'] == 2].copy()

        df_cov.rename(columns={'Age': 'y_true', 'Biological Sex': 'Sex'}, inplace=True)
        df_cov['y_true'] = pd.to_numeric(df_cov['y_true'], errors='coerce')
        df_cov['y_pred'] = pd.to_numeric(df_cov['y_pred'], errors='coerce')
        df_cov.dropna(subset=['y_true', 'y_pred', 'Sex'], inplace=True)

        # Residuals
        df_cov['residual'] = df_cov['y_pred'] - df_cov['y_true']
        df_cov['Abs_Residual'] = df_cov['residual'].abs()

        # Clean Outliers (Top 5%) for robust visualization
        cut95 = df_cov["Abs_Residual"].quantile(0.95)
        df_cov_clean = df_cov[df_cov["Abs_Residual"] <= cut95].copy()

    except:
        df_cov, df_cov_clean = pd.DataFrame(), pd.DataFrame()

    # --- B. Healthy Data (For Top Row only) ---
    try:
        df_health = pd.read_csv('test_preds_full_with_sex.csv')
        df_health.rename(columns={'biological_sex': 'Sex'}, inplace=True)
        df_health['y_true'] = pd.to_numeric(df_health['y_true'], errors='coerce')
        df_health['y_pred'] = pd.to_numeric(df_health['y_pred'], errors='coerce')
        df_health.dropna(subset=['y_true', 'y_pred'], inplace=True)
    except:
        df_health = pd.DataFrame()

    return df_cov, df_cov_clean, df_health

# ==========================================
# 3. Helper Plotting Functions
# ==========================================
def plot_raincloud(df, ax):
    """Custom Raincloud Plot: Violin + Box + Strip"""
    male = df[df['Sex']=='Male']['residual'].values
    female = df[df['Sex']=='Female']['residual'].values

    # 1. Violin (Background)
    parts = ax.violinplot([female, male], positions=[0, 1], vert=False,
                          showmeans=False, showextrema=False, widths=0.8)
    for i, pc in enumerate(parts['bodies']):
        pc.set_facecolor(c_female if i==0 else c_male)
        pc.set_alpha(0.4)
        pc.set_edgecolor('none')

    # 2. Boxplot (Middle)
    bp = ax.boxplot([female, male], positions=[0, 1], vert=False,
                    widths=0.2, patch_artist=True, showfliers=False, zorder=5)
    for patch, color in zip(bp['boxes'], [c_female, c_male]):
        patch.set_facecolor(color)
        patch.set_alpha(0.8)
        patch.set_linewidth(0.8)
    for el in ['whiskers', 'caps', 'medians']:
        plt.setp(bp[el], color='black', linewidth=1)

    # 3. Scatter (Foreground)
    # Add jitter
    y_f = np.random.normal(0, 0.08, size=len(female)) + 0.25
    y_m = np.random.normal(1, 0.08, size=len(male)) + 0.25
    ax.scatter(female, y_f, s=2, color=c_female, alpha=0.15, edgecolors='none')
    ax.scatter(male, y_m, s=2, color=c_male, alpha=0.15, edgecolors='none')

    # Stats
    delta = np.mean(male) - np.mean(female)
    mwu_p = stats.mannwhitneyu(male, female).pvalue

    ax.set_yticks([0, 1])
    ax.set_yticklabels(['Female', 'Male'])
    ax.axvline(0, color='grey', linestyle='--', linewidth=0.8)

    # Annotation
    p_txt = "p < 0.001" if mwu_p < 0.001 else f"p={mwu_p:.3f}"
    ax.text(0.95, 0.9, f'Δ = {delta:.1f}y\n{p_txt}', transform=ax.transAxes, ha='right', fontsize=7)

def plot_binned_stability(df, ax):
    """Binned Error Bar Plot"""
    bins = [18, 30, 40, 50, 60, 70, 80, 100]
    labels = ['18-30', '30s', '40s', '50s', '60s', '70s', '80+']
    df['Age_Bin'] = pd.cut(df['y_true'], bins=bins, labels=labels, right=False)

    bin_stats = []
    for b in labels:
        sub = df[df['Age_Bin'] == b]
        if len(sub) < 10: continue
        m = sub[sub['Sex']=='Male']['residual']
        f = sub[sub['Sex']=='Female']['residual']
        if len(m) > 2 and len(f) > 2:
            diff = m.mean() - f.mean()
            se = np.sqrt((m.std()**2/len(m)) + (f.std()**2/len(f)))
            bin_stats.append({'Bin': b, 'Diff': diff, 'SE': se})

    stats_df = pd.DataFrame(bin_stats)

    # Plot
    ax.axhline(0, color='black', linewidth=0.8)
    global_diff = df[df['Sex']=='Male']['residual'].mean() - df[df['Sex']=='Female']['residual'].mean()
    ax.axhline(global_diff, color='grey', linestyle='--', linewidth=1, label='Avg Bias')

    ax.errorbar(x=stats_df['Bin'], y=stats_df['Diff'], yerr=stats_df['SE']*1.96,
                fmt='o', color='#333', ecolor='black', capsize=3, markersize=4, linewidth=1)

    ax.text(0, global_diff + 0.5, f'Avg Bias (+{global_diff:.1f}y)', color='grey', fontsize=6, fontweight='bold')
    ax.set_ylim(-2, 8)

# ==========================================
# 4. Master Plotting Function
# ==========================================
def create_final_figure(df_cov, df_cov_clean, df_health):
    fig = plt.figure(figsize=(8.5, 6), dpi=300)
    gs = gridspec.GridSpec(2, 6, height_ratios=[1, 0.8], hspace=0.5, wspace=1.0)

    # --- TOP ROW: Regressions ---

    # Panel a: COVID
    ax1 = fig.add_subplot(gs[0, 0:3])
    mae = mean_absolute_error(df_cov['y_true'], df_cov['y_pred'])
    r2 = r2_score(df_cov['y_true'], df_cov['y_pred'])
    ax1.scatter(df_cov['y_true'], df_cov['y_pred'], color=c_main, alpha=0.3, s=8, edgecolors='none')
    ax1.plot([0, 100], [0, 100], 'k--', lw=0.8)
    ax1.set_title('a  COVID-19 Cohort', loc='left')
    ax1.set_xlabel('True Age'); ax1.set_ylabel('Predicted Age')
    ax1.text(0.05, 0.85, f'MAE = {mae:.2f} yr\n$R^2$ = {r2:.2f}', transform=ax1.transAxes)
    ax1.spines['top'].set_visible(False); ax1.spines['right'].set_visible(False)

    # Panel b: Healthy
    ax2 = fig.add_subplot(gs[0, 3:6])
    if not df_health.empty:
        mae_h = mean_absolute_error(df_health['y_true'], df_health['y_pred'])
        r2_h = r2_score(df_health['y_true'], df_health['y_pred'])
        ax2.scatter(df_health['y_true'], df_health['y_pred'], color=c_main, alpha=0.3, s=8, edgecolors='none')
        ax2.plot([0, 100], [0, 100], 'k--', lw=0.8)
        ax2.set_title('b  Healthy Cohort', loc='left')
        ax2.set_xlabel('True Age'); ax2.set_ylabel('Predicted Age')
        ax2.text(0.05, 0.85, f'MAE = {mae_h:.2f} yr\n$R^2$ = {r2_h:.2f}', transform=ax2.transAxes)
    ax2.spines['top'].set_visible(False); ax2.spines['right'].set_visible(False)

    # --- BOTTOM ROW: Sex Analysis (COVID ONLY) ---

    # Panel c: Raincloud (Systematic Offset)
    ax3 = fig.add_subplot(gs[1, 0:2])
    plot_raincloud(df_cov_clean, ax3)
    ax3.set_title('c  Systematic Offset', loc='left')
    ax3.set_xlabel('Residual (Pred - True)')
    ax3.spines['top'].set_visible(False); ax3.spines['right'].set_visible(False)

    # Panel d: ECDF (Error Magnitude)
    ax4 = fig.add_subplot(gs[1, 2:4])
    sns.ecdfplot(data=df_cov_clean, x='Abs_Residual', hue='Sex', ax=ax4,
                 palette={'Female': c_female, 'Male': c_male}, lw=1.5)
    ks_p = stats.ks_2samp(df_cov_clean[df_cov_clean['Sex']=='Male']['Abs_Residual'],
                          df_cov_clean[df_cov_clean['Sex']=='Female']['Abs_Residual']).pvalue
    ax4.text(0.5, 0.2, f'KS p = {ks_p:.2f}', transform=ax4.transAxes, ha='center')
    ax4.set_title('d  Error Magnitude', loc='left')
    ax4.legend_.remove()
    ax4.spines['top'].set_visible(False); ax4.spines['right'].set_visible(False)

    # Panel e: Binned Stability
    ax5 = fig.add_subplot(gs[1, 4:6])
    plot_binned_stability(df_cov_clean, ax5)
    ax5.set_title('e  Age-Group Stability', loc='left')
    ax5.set_ylabel('Bias (M-F Diff)')
    ax5.set_xlabel('Age Group')
    ax5.spines['top'].set_visible(False); ax5.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.savefig('Figure2_Final_Raincloud_Binned.pdf', dpi=300, transparent=True)
    plt.show()

# Run
df_cov, df_cov_clean, df_health = load_data()
create_final_figure(df_cov, df_cov_clean, df_health)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.formula.api as smf

# Load data
df = pd.read_csv("test_preds_full_with_sex.csv")


# Rename columns to standard names
df = df.rename(columns=lambda x: x.strip())
# Map column names if they differ
if 'biological_sex' in df.columns:
    df['Sex'] = df['biological_sex']
if 'y_true' in df.columns:
    df['Age'] = df['y_true']

# Clean Sex column
df["Sex"] = (
    df["Sex"]
    .astype(str).str.strip()
    .replace({"F": "Female", "M": "Male", "female": "Female", "male": "Male"})
)
df = df[df["Sex"].isin(["Male", "Female"])].copy()

# Ensure numeric types
df['Age'] = pd.to_numeric(df['Age'], errors='coerce')
df['y_pred'] = pd.to_numeric(df['y_pred'], errors='coerce')

# Drop NaNs
df = df.dropna(subset=["Age", "y_pred", "Sex"]).copy()

# Calculate residuals
df["residual"] = df["y_pred"] - df["Age"]
df["abs_res"]  = df["residual"].abs()

# Drop top 5% |residual| outliers
cut95 = df["abs_res"].quantile(0.95)
df = df[df["abs_res"] <= cut95].copy()

female = df.query("Sex=='Female'")["residual"]
male   = df.query("Sex=='Male'")["residual"]

# Core stats
mwu_p   = stats.mannwhitneyu(female, male, alternative="two-sided").pvalue
welch_p = stats.ttest_ind(female, male, equal_var=False).pvalue
lev_p   = stats.levene(female, male, center="median").pvalue
ks_p    = stats.ks_2samp(female, male).pvalue
abs_p   = stats.ttest_ind(np.abs(female), np.abs(male), equal_var=False).pvalue

# OLS
ols = smf.ols("residual ~ Age * Sex", data=df).fit()
# Check available parameter names
# print(ols.pvalues.index)
# Standard naming is usually Age:Sex[T.Male]
int_p = ols.pvalues.get("Age:Sex[T.Male]", np.nan)
if np.isnan(int_p):
    # Try alternate naming if Sex reference is Male
    int_p = ols.pvalues.get("Age:Sex[T.Female]", np.nan)

# Helpers
def bootstrap_ci_diff(m, f, nboot=3000, alpha=0.05, seed=0):
    rng = np.random.default_rng(seed)
    m = np.asarray(m); f = np.asarray(f)
    diffs = []
    if len(m) == 0 or len(f) == 0:
        return np.nan, np.nan, np.nan
    for _ in range(nboot):
        ms = rng.choice(m, size=len(m), replace=True)
        fs = rng.choice(f, size=len(f), replace=True)
        diffs.append(ms.mean() - fs.mean())
    lo, hi = np.percentile(diffs, [100*alpha/2, 100*(1-alpha/2)])
    return np.mean(diffs), lo, hi

# Rolling diff
centers = np.arange(df["Age"].min() + 6, df["Age"].max() - 6, 3)
rows = []
for c in centers:
    sub = df[(df["Age"] >= c - 6) & (df["Age"] <= c + 6)]
    m = sub.loc[sub["Sex"] == "Male",   "residual"].values
    f = sub.loc[sub["Sex"] == "Female", "residual"].values
    if len(m) >= 10 and len(f) >= 10:
        d, lo, hi = bootstrap_ci_diff(m, f, nboot=1000) # Reduced boot for speed in preview
        rows.append((c, d, lo, hi))
roll = pd.DataFrame(rows, columns=["center", "diff", "lo", "hi"])

# Bias heatmap bins: 0–15, then 5y bins, special 80–92
# The prompt says: "Age bins include 0–15 first bin and special 80–92 bin"
# Standard ranges usually fill the middle. Let's infer standard 5y steps between 15 and 80.
bins = [0, 15] + list(range(20, 85, 5)) + [92, 100]
# Clean up overlap/gaps: range(20, 85, 5) gives 20, 25, ..., 80.
# So bins: 0, 15, 20, 25, ..., 80, 92, 100.
# Wait, 15 to 20 is a 5y bin.
bins = [0, 15] + list(range(20, 85, 5)) + [92, 100]
bins = sorted(list(set(bins))) # Ensure sorted and unique

df["Age_bin"] = pd.cut(df["Age"], bins, right=False)

# Check for empty bins or NaNs
# Groupby and calculate difference
heat_data = []
bin_labels = []

# Using explicit iteration to preserve order and handle empty bins gracefully
# (groupby might skip empty categories if observed=True/False isn't handled carefully)
# But standard pandas groupby on categorical usually keeps all categories if we set them up right.
# Let's just use the dataframe derived from cut.

heat = (df.groupby("Age_bin", observed=False)
          .apply(lambda g: g.loc[g["Sex"] == "Male", "residual"].mean()
                           - g.loc[g["Sex"] == "Female", "residual"].mean())
          .to_frame("diff"))

# Plot
sns.set_theme(style="whitegrid", context="paper")
fig, axes = plt.subplots(3, 2, figsize=(11, 13))
(ax1, ax2, ax3, ax4, ax5, ax6) = axes.flat
plt.subplots_adjust(hspace=0.4, wspace=0.3)

# 1) Violin
sns.violinplot(x="Sex", y="residual", data=df, inner="quart", cut=0, ax=ax1, palette="Set2")
sns.stripplot(x="Sex", y="residual", data=df, color="k", alpha=0.15, size=2, ax=ax1)
ax1.axhline(0, ls="--", c="gray")
ax1.set_ylim(-30, 30)
ax1.set_title("Residual distribution by sex")
ax1.text(0.95, 0.9, f"MWU p={mwu_p:.1e}\nWelch p={welch_p:.1e}\nLevene p={lev_p:.1e}\nKS p={ks_p:.1e}",
         ha="right", va="top", transform=ax1.transAxes,
         bbox=dict(facecolor="white", alpha=0.7, edgecolor="none"), fontsize=8)

# 2) KDE
sns.kdeplot(data=df, x="residual", hue="Sex", fill=True, common_norm=False, alpha=0.4, ax=ax2)
ax2.axvline(0, ls="--", c="gray")
ax2.set_title("Residual density by sex")
ax2.text(0.95, 0.9, f"KS p={ks_p:.2e}", ha="right", va="top", transform=ax2.transAxes,
         bbox=dict(facecolor="white", alpha=0.7, edgecolor="none"), fontsize=8)

# 3) ECDF
sns.ecdfplot(data=df, x="abs_res", hue="Sex", ax=ax3)
ax3.set_title("ECDF of absolute residuals")
ax3.text(0.95, 0.1, f"Welch(|res|) p={abs_p:.2e}", ha="right", transform=ax3.transAxes,
         bbox=dict(facecolor="white", alpha=0.7, edgecolor="none"), fontsize=8)
ax3.set_xlabel("|residual|")

# 4) OLS
sns.scatterplot(data=df, x="Age", y="residual", hue="Sex", alpha=0.25, s=15, ax=ax4)
age_grid = np.linspace(df["Age"].min(), df["Age"].max(), 200)
# Predict manually or use model
# Simple lines for display
for sex, col in [("Female", "C0"), ("Male", "C1")]:
    # Filter data to fit separate lines for viz if needed, or use the interaction model predictions
    # Using interaction model predictions:
    pred_data = pd.DataFrame({"Age": age_grid, "Sex": sex})
    yhat = ols.predict(pred_data)
    ax4.plot(age_grid, yhat, color=col, lw=2.5, label=f"{sex} fit")

ax4.axhline(0, ls="--", c="gray")
ax4.set_ylim(-30, 30)
ax4.legend(loc="lower left")
ax4.set_title("OLS: residual ~ Age × Sex")
ax4.text(0.95, 0.9, f"Interaction p={int_p:.2e}", ha="right", va="top", transform=ax4.transAxes,
         bbox=dict(facecolor="white", alpha=0.7, edgecolor="none"), fontsize=8)

# 5) Rolling
if not roll.empty:
    ax5.plot(roll["center"], roll["diff"], "o-", label="Male–Female mean diff")
    ax5.fill_between(roll["center"], roll["lo"], roll["hi"], color="C0", alpha=0.2, label="95% bootstrap CI")
ax5.axhline(0, ls="--", c="gray")
ax5.legend(fontsize=8)
ax5.set_title("Rolling sex difference (95% CI)")
ax5.set_xlabel("Age (center of ~12y window)")
ax5.set_ylabel("Residual mean diff (M–F)")

# 6) Heatmap
sns.heatmap(heat.T, cmap="RdBu_r", center=0, vmin=-7.5, vmax=7.5,
            cbar_kws={"label": "Mean (M–F) residual"}, ax=ax6)
ax6.set_xlabel("Age bin")
ax6.set_ylabel("")
ax6.set_title("Bias map across age bins")
# Fix tick labels
ax6.set_xticklabels([str(i) for i in heat.index], rotation=45, ha="right", fontsize=8)

plt.tight_layout()
plt.savefig("figure_composite_corrected.png", dpi=150)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import pandas as pd

# נטען את הדאטה (הקוד המקוצר להעלאת הנתונים המשולבים)
def load_combined_data_minimal():
    try:
        df_cov = pd.read_csv('test_preds_covid.csv')
        if 'combo_index' in df_cov.columns: df_cov = df_cov[df_cov['combo_index'] == 2].copy()
        df_cov.rename(columns={'Age': 'y_true', 'Biological Sex': 'Sex'}, inplace=True)
        df_cov['y_true'] = pd.to_numeric(df_cov['y_true'], errors='coerce')
        df_cov['y_pred'] = pd.to_numeric(df_cov['y_pred'], errors='coerce')
        df_cov.dropna(subset=['y_true', 'y_pred', 'Sex'], inplace=True)
    except: return pd.DataFrame()

    try:
        df_health = pd.read_csv('test_preds_full_with_sex.csv')
        df_health.rename(columns={'biological_sex': 'Sex'}, inplace=True)
    except: df_health = pd.DataFrame()

    cols = ['y_true', 'y_pred', 'Sex']
    df = pd.concat([df_cov[cols], df_health[cols]], ignore_index=True)
    df['residual'] = df['y_pred'] - df['y_true']

    # Outlier removal (Top 5%)
    df['abs_res'] = df['residual'].abs()
    cut95 = df["abs_res"].quantile(0.95)
    df = df[df["abs_res"] <= cut95].copy()
    return df

df = load_combined_data_minimal()

# ==========================================
# Raincloud Plot Function
# ==========================================
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 8

def draw_raincloud(df, ax):
    # Colors
    c_male = '#D55E00'
    c_female = '#0072B2'

    # Separate Data
    male_data = df[df['Sex']=='Male']['residual'].values
    female_data = df[df['Sex']=='Female']['residual'].values

    # 1. The "Cloud" (Half-Violin / Density)
    # We use violinplot but will cut it in half manually or use a trick
    parts = ax.violinplot([female_data, male_data], positions=[0, 1],
                          showmeans=False, showmedians=False, showextrema=False, vert=False)

    # Customize bodies (Make them look like clouds)
    for i, pc in enumerate(parts['bodies']):
        pc.set_facecolor(c_female if i==0 else c_male)
        pc.set_edgecolor('black')
        pc.set_linewidth(0.5)
        pc.set_alpha(0.6)

        # Trick to make it "half" violin (Raincloud style)
        # We interpret the path and clip the bottom half
        m = np.mean(pc.get_paths()[0].vertices[:, 1])
        # Modify vertices to flatten the bottom (or top) depending on orientation
        # For simplicity in matplotlib pure: we just offset them.
        # Let's use a simpler approach: Offset the violin up/down.
        pc.set_transform(plt.gca().transData)

    # 2. The "Umbrella" (Boxplot)
    # Narrow boxplot inside
    bp = ax.boxplot([female_data, male_data], positions=[0, 1], vert=False,
                    widths=0.15, patch_artist=True, showfliers=False,
                    zorder=10) # On top

    # Style Boxplot
    for patch, color in zip(bp['boxes'], [c_female, c_male]):
        patch.set_facecolor(color)
        patch.set_alpha(0.8)
        patch.set_linewidth(0.8)
    for element in ['whiskers', 'fliers', 'means', 'medians', 'caps']:
        plt.setp(bp[element], color='black', linewidth=0.8)

    # 3. The "Rain" (Stripplot / Raw Data)
    # Jittered dots below the box
    np.random.seed(42)
    jitter = 0.08

    y_female = np.random.normal(0, jitter, size=len(female_data)) + 0.15 # Offset down
    y_male = np.random.normal(1, jitter, size=len(male_data)) + 0.15

    ax.scatter(female_data, y_female, s=3, color=c_female, alpha=0.15, edgecolors='none', zorder=1)
    ax.scatter(male_data, y_male, s=3, color=c_male, alpha=0.15, edgecolors='none', zorder=1)

    # 4. Add Mean Lines (Visual Guide)
    mean_f = np.mean(female_data)
    mean_m = np.mean(male_data)
    ax.vlines(mean_f, -0.2, 0.2, colors='black', linestyles='--', linewidth=1, zorder=11)
    ax.vlines(mean_m, 0.8, 1.2, colors='black', linestyles='--', linewidth=1, zorder=11)

    # Annotations
    ax.set_yticks([0, 1])
    ax.set_yticklabels(['Female', 'Male'], fontweight='bold')
    ax.set_xlabel('Age Residual (Predicted - True)')
    ax.set_title('c  Systematic Offset (Raincloud Plot)', loc='left', fontweight='bold')

    # Zero line
    ax.axvline(0, color='grey', linestyle='-', linewidth=0.5, zorder=0)

    # Add Delta Text
    delta = mean_m - mean_f
    ax.text(0, 1.4, f'Δ = {delta:.1f} years', ha='center', fontsize=8, fontweight='bold')

    ax.set_ylim(-0.3, 1.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False) # Clean look

# Generate Plot
fig, ax = plt.subplots(figsize=(3.5, 2.5), dpi=300)
draw_raincloud(df, ax)
plt.tight_layout()
plt.savefig('Panel_C_Raincloud.png', dpi=300)
plt.show()

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
ALL-IN-ONE (updated):
- Drops the top 5% largest absolute residuals (outliers by MAE)
- Treats all ages in [80, 92] as the SAME 5-year group (labeled 80)
- Residuals (y_pred - Age) vs Biological Sex: plots + p-values

Requires: pandas, numpy, scipy, matplotlib, seaborn, statsmodels
"""

import warnings, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.nonparametric.smoothers_lowess import lowess

warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 140
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
sns.set_context("talk")

# --------------- I/O ---------------
IN_FILE = "test_preds_covid.csv"

# --------------- Helpers ---------------
def colfind(df, candidates):
    cmap = {c.lower().strip(): c for c in df.columns}
    for cand in candidates:
        k = cand.lower().strip()
        if k in cmap:
            return cmap[k]
    raise KeyError(f"Missing expected column among: {candidates}")

def pbox(ax, text, loc="upper right"):
    ax.text(
        0.98 if "right" in loc else 0.02,
        0.98 if "upper" in loc else 0.02,
        text,
        transform=ax.transAxes,
        ha="right" if "right" in loc else "left",
        va="top" if "upper" in loc else "bottom",
        bbox=dict(boxstyle="round", alpha=0.15, pad=0.4)
    )

def ecdf(a):
    a = np.sort(np.asarray(a))
    y = np.arange(1, len(a)+1) / len(a) if len(a) else np.array([])
    return a, y

def cliff_delta(x, y):
    x = np.asarray(x); y = np.asarray(y)
    n = len(x); m = len(y)
    if n*m == 0:
        return np.nan
    greater = 0; less = 0
    for xi in x:
        greater += np.sum(xi > y)
        less    += np.sum(xi < y)
    return (greater - less) / (n*m)

def within_age_window(df, center, half_width, age_col):
    lo = center - half_width
    hi = center + half_width
    return df[(df[age_col] >= lo) & (df[age_col] <= hi)]

def safe_sem(x):
    x = np.asarray(x)
    return stats.sem(x, nan_policy="omit") if len(x) > 1 else np.nan

def neglog10(p):
    with np.errstate(divide="ignore"):
        return -np.log10(p)

# --------------- Load ---------------
df = pd.read_csv(IN_FILE)

age_col  = colfind(df, ["Age"])
pred_col = colfind(df, ["y_pred", "pred", "prediction"])
sex_col  = colfind(df, ["Biological Sex", "Sex", "gender"])

df[age_col]  = pd.to_numeric(df[age_col], errors="coerce")
df[pred_col] = pd.to_numeric(df[pred_col], errors="coerce")
df[sex_col]  = df[sex_col].astype(str).str.strip().replace(
    {"F":"Female","M":"Male","female":"Female","male":"Male"}
)

df = df.dropna(subset=[age_col, pred_col, sex_col]).copy()
df = df[df[sex_col].isin(["Male","Female"])].copy()

# Residuals
df["residual"]      = df[pred_col] - df[age_col]
df["abs_residual"]  = df["residual"].abs()

# ---- NEW: drop top 5% largest absolute residuals (outliers by MAE) ----
cut95 = df["abs_residual"].quantile(0.95)
df = df[df["abs_residual"] <= cut95].copy()

# ---- NEW: put all ages in [80, 92] into the SAME group (bin left-edge = 80) ----
# Other ages use regular 5-year bins based on floored left edge.
age_left = (df[age_col] // 5 * 5).astype(int)
age_left = np.where((df[age_col] >= 80) & (df[age_col] <= 92), 80, age_left)
df["age_bin"] = age_left.astype(int)  # numeric left edge for plotting (…, 70, 75, 80)

# quick splits
male   = df.loc[df[sex_col]=="Male",   "residual"].values
female = df.loc[df[sex_col]=="Female", "residual"].values
male_abs   = np.abs(male)
female_abs = np.abs(female)

# --------------- Global stats (printed) ---------------
print("\n=== OVERALL STATS (after dropping top 5% |residual| outliers) ===")
print(f"N total={len(df)} | N Male={len(male)} | N Female={len(female)}")
print(f"Mean residual Male   = {np.nanmean(male):.3f} ± {np.nanstd(male, ddof=1):.3f} (SD)")
print(f"Mean residual Female = {np.nanmean(female):.3f} ± {np.nanstd(female, ddof=1):.3f} (SD)")

U, p_mwu   = stats.mannwhitneyu(male, female, alternative="two-sided")
t, p_t     = stats.ttest_ind(male, female, equal_var=False)
lev, p_lev = stats.levene(male, female, center="median")
ks, p_ks   = stats.ks_2samp(male, female)
t_abs, p_abs = stats.ttest_ind(male_abs, female_abs, equal_var=False)
cd = cliff_delta(male, female)

print(f"Mann–Whitney U: U={U:.0f}, p={p_mwu:.3e}")
print(f"Welch t-test (means): t={t:.3f}, p={p_t:.3e}")
print(f"Levene (variance): stat={lev:.3f}, p={p_lev:.3e}")
print(f"KS (distribution): D={ks:.3f}, p={p_ks:.3e}")
print(f"Welch t-test (|residual|): t={t_abs:.3f}, p={p_abs:.3e}")
print(f"Cliff’s delta (Male vs Female residual): {cd:.3f}")

# --------------- 1) Violin+strip: residual by sex ---------------
fig, ax = plt.subplots(figsize=(7,5))
sns.violinplot(data=df, x=sex_col, y="residual", inner="quartile", cut=0, ax=ax)
sns.stripplot(data=df, x=sex_col, y="residual", color="k", alpha=0.35, size=2, jitter=0.2, ax=ax)
ax.axhline(0, ls="--", lw=1, c="gray")
ax.set_title("Residual distribution by sex")
pbox(ax, f"MWU p={p_mwu:.2e}\nWelch p={p_t:.2e}\nLevene p={p_lev:.2e}\nKS p={p_ks:.2e}", "upper right")
plt.show()

# --------------- 2) Box+swarm: residual by sex ---------------
fig, ax = plt.subplots(figsize=(7,5))
sns.boxplot(data=df, x=sex_col, y="residual", showfliers=False, ax=ax)
sns.swarmplot(
    data=df.sample(min(len(df), 800), random_state=1),
    x=sex_col, y="residual", size=2, alpha=0.5, ax=ax
)
ax.axhline(0, ls="--", lw=1, c="gray")
ax.set_title("Residual boxplot by sex")
pbox(ax, f"MWU p={p_mwu:.2e} | Welch p={p_t:.2e}", "upper right")
plt.show()

# --------------- 3) KDE overlay of residual ---------------
fig, ax = plt.subplots(figsize=(7,5))
sns.kdeplot(data=df, x="residual", hue=sex_col, common_norm=False, fill=True, alpha=0.4, ax=ax)
ax.axvline(0, ls="--", lw=1, c="gray")
ax.set_title("Residual density by sex")
pbox(ax, f"KS p={p_ks:.2e}", "upper right")
plt.show()

# --------------- 4) ECDF of |residual| ---------------
fig, ax = plt.subplots(figsize=(7,5))
for lab, arr in [("Male", male_abs), ("Female", female_abs)]:
    x, y = ecdf(arr)
    ax.step(x, y, where="post", label=lab)
ax.set_xlabel("|residual|"); ax.set_ylabel("ECDF")
ax.set_title("ECDF of absolute residuals (extremity)")
ax.legend()
pbox(ax, f"Welch(|res|) p={p_abs:.2e}", "lower right")
plt.show()

# --------------- 5) QQ-plots by sex ---------------
fig, axes = plt.subplots(1, 2, figsize=(10,5))
sm.qqplot(male, line="s", ax=axes[0]); axes[0].set_title("QQ: Male residual")
sm.qqplot(female, line="s", ax=axes[1]); axes[1].set_title("QQ: Female residual")
plt.show()

# --------------- 6) Scatter residual vs age + LOWESS per sex ---------------
fig, ax = plt.subplots(figsize=(8,5))
for sex in ["Male", "Female"]:
    sub = df[df[sex_col]==sex]
    ax.scatter(sub[age_col], sub["residual"], s=12, alpha=0.35, label=f"{sex} points")
    if len(sub) > 5:
        smth = lowess(sub["residual"], sub[age_col], frac=0.3, return_sorted=True)
        ax.plot(smth[:,0], smth[:,1], lw=2, label=f"{sex} LOWESS")
ax.axhline(0, ls="--", lw=1, c="gray")
ax.set_xlabel("Age"); ax.set_ylabel("Residual")
ax.set_title("Residual vs Age (LOWESS) by sex")
ax.legend()
# OLS interaction global p-value (for annotation)
dmod = df.rename(columns={sex_col: "Sex"})
dmod["Sex"] = pd.Categorical(dmod["Sex"], categories=["Female","Male"])
model = smf.ols("residual ~ Age * Sex", data=dmod).fit()
p_int = model.pvalues.get("Age:Sex[T.Male]", np.nan)
pbox(ax, f"OLS interaction p={p_int:.2e}", "upper right")
plt.show()

# --------------- 7) Binned means ±95% CI by sex + per-bin MWU as -log10(p) ---------------
agg = (df.groupby([sex_col, "age_bin"])["residual"]
         .agg(["mean","count","std"]).reset_index())
agg["sem"] = agg["std"] / np.sqrt(agg["count"])
agg["lo"]  = agg["mean"] - 1.96*agg["sem"]
agg["hi"]  = agg["mean"] + 1.96*agg["sem"]

bins = np.sort(df["age_bin"].unique())  # numeric left edges; includes 80 for [80,92]
pvals_bin = []
for b in bins:
    m = df[(df["age_bin"]==b) & (df[sex_col]=="Male")]["residual"].values
    f = df[(df["age_bin"]==b) & (df[sex_col]=="Female")]["residual"].values
    if len(m)>=3 and len(f)>=3:
        _, pb = stats.mannwhitneyu(m, f, alternative="two-sided")
    else:
        pb = np.nan
    pvals_bin.append(pb)

fig, ax = plt.subplots(figsize=(9,6))
for sex in ["Male","Female"]:
    sub = agg[agg[sex_col]==sex]
    ax.plot(sub["age_bin"], sub["mean"], marker="o", label=f"{sex} mean")
    ax.fill_between(sub["age_bin"], sub["lo"], sub["hi"], alpha=0.2)
ax.axhline(0, ls="--", lw=1, c="gray")
ax.set_xlabel("Age bin left edge (5y); ages 80–92 grouped at 80")
ax.set_ylabel("Mean residual ±95% CI")
ax.set_title("Binned residual means by sex (per-bin significance)")
ax.legend(loc="upper left")

ax2 = ax.twinx()
ax2.plot(bins, [neglog10(p) if not np.isnan(p) else np.nan for p in pvals_bin],
         lw=1.5, ls="--", marker="x", alpha=0.8)
ax2.set_ylabel("-log10 p (MWU per bin)")
pbox(ax, "Dashed: per-bin MWU (right axis)", "lower right")
plt.show()

# --------------- 8) Rolling means and Male-Female diff with rolling p-values ---------------
centers = np.arange(df[age_col].min(), df[age_col].max()+1, 2)
half_width = 5
rows = []
for c in centers:
    sub = within_age_window(df, c, half_width, age_col)
    m = sub.loc[sub[sex_col]=="Male", "residual"].values
    f = sub.loc[sub[sex_col]=="Female","residual"].values
    if len(m)>=3 and len(f)>=3:
        m_mean, f_mean = np.mean(m), np.mean(f)
        diff = m_mean - f_mean
        _, p_roll = stats.ttest_ind(m, f, equal_var=False)
        rows.append(dict(center=c, male_mean=m_mean, female_mean=f_mean,
                         diff=diff, p=p_roll))
roll = pd.DataFrame(rows)

fig, (ax1, ax2) = plt.subplots(2,1, figsize=(9,8), sharex=True)
ax1.plot(roll["center"], roll["male_mean"], marker="o", ms=3, label="Male mean")
ax1.plot(roll["center"], roll["female_mean"], marker="o", ms=3, label="Female mean")
ax1.axhline(0, ls="--", lw=1, c="gray"); ax1.set_ylabel("Rolling mean (±~5y)")
ax1.set_title("Rolling-window residual means by sex"); ax1.legend()

ax2.plot(roll["center"], roll["diff"], lw=2, label="Male - Female")
ax2.axhline(0, ls="--", lw=1, c="gray"); ax2.set_xlabel("Age (center)")
ax2.set_ylabel("Mean diff (M-F)")
ax3 = ax2.twinx()
ax3.plot(roll["center"], neglog10(roll["p"]), ls="--", lw=1.5, alpha=0.9, label="-log10 p")
ax3.set_ylabel("-log10 p (Welch t)")
pbox(ax2, "Dashed: -log10 p for M vs F (Welch)", "upper right")
plt.show()

# --------------- 9) |Residual| by sex ---------------
fig, ax = plt.subplots(figsize=(7,5))
sns.boxplot(data=df, x=sex_col, y="abs_residual", showfliers=False, ax=ax)
sns.stripplot(
    data=df.sample(min(len(df), 800), random_state=2),
    x=sex_col, y="abs_residual", color="k", size=2, alpha=0.35, ax=ax
)
ax.set_title("|Residual| (extremity) by sex")
pbox(ax, f"Welch(|res|) p={p_abs:.2e}", "upper right")
plt.show()

# --------------- 10) Extreme residual rates overall and by age bins (thr=5,10) ---------------
for thr in [5, 10]:
    flag = f"ext_{thr}"
    df[flag] = (df["abs_residual"] >= thr).astype(int)

    tab = pd.crosstab(df[sex_col], df[flag])
    if tab.shape == (2,2):
        _, p_fish = stats.fisher_exact(tab.values)
    else:
        p_fish = np.nan

    fig, ax = plt.subplots(figsize=(6,4))
    rate = df.groupby(sex_col)[flag].mean().reset_index()
    sns.barplot(data=rate, x=sex_col, y=flag, ax=ax)
    ax.set_ylim(0,1)
    ax.set_title(f"Proportion of |residual| ≥ {thr} by sex")
    pbox(ax, f"Fisher exact p={p_fish:.2e}", "upper right")
    plt.show()

    fig, ax = plt.subplots(figsize=(9,5))
    rate_bin = df.groupby([sex_col, "age_bin"])[flag].mean().reset_index()
    for sex in ["Male","Female"]:
        sub = rate_bin[rate_bin[sex_col]==sex]
        ax.plot(sub["age_bin"], sub[flag], marker="o", label=sex)
    ax.set_ylim(0,1); ax.set_xlabel("Age bin left edge (5y); 80–92 grouped")
    ax.set_ylabel(f"Proportion ≥{thr}")
    ax.set_title(f"Extreme |residual| rate by age bin (thr={thr})")
    ax.legend(loc="upper left")

    p_list = []
    for b in bins:
        sub = df[df["age_bin"]==b]
        tab = pd.crosstab(sub[sex_col], sub[flag])
        if tab.shape == (2,2) and tab.values.sum()>0:
            _, p_b = stats.fisher_exact(tab.values)
        else:
            p_b = np.nan
        p_list.append(p_b)

    ax2 = ax.twinx()
    ax2.plot(bins, [neglog10(p) if not np.isnan(p) else np.nan for p in p_list],
             ls="--", lw=1.5, alpha=0.85)
    ax2.set_ylabel("-log10 p (Fisher per bin)")
    pbox(ax, "Dashed: per-bin Fisher p", "lower right")
    plt.show()

# --------------- 11) Rolling SD (variability) vs age by sex + rolling Levene p ---------------
centers2 = np.arange(df[age_col].min(), df[age_col].max()+1, 2)
half_width2 = 6
roll_sd = { "center": [], "male_sd": [], "female_sd": [], "p_levene": [] }
for c in centers2:
    sub = within_age_window(df, c, half_width2, age_col)
    m = sub.loc[sub[sex_col]=="Male","residual"].values
    f = sub.loc[sub[sex_col]=="Female","residual"].values
    if len(m)>=4 and len(f)>=4:
        sd_m = np.std(m, ddof=1); sd_f = np.std(f, ddof=1)
        stat, p_l = stats.levene(m, f, center="median")
        roll_sd["center"].append(c); roll_sd["male_sd"].append(sd_m)
        roll_sd["female_sd"].append(sd_f); roll_sd["p_levene"].append(p_l)
roll_sd = pd.DataFrame(roll_sd)

fig, ax = plt.subplots(figsize=(9,5))
ax.plot(roll_sd["center"], roll_sd["male_sd"], lw=2, label="Male rolling SD")
ax.plot(roll_sd["center"], roll_sd["female_sd"], lw=2, label="Female rolling SD")
ax.set_xlabel("Age"); ax.set_ylabel("Rolling SD (±~6y)")
ax.set_title("Residual variability by age and sex")
ax.legend(loc="upper left")
ax2 = ax.twinx()
ax2.plot(roll_sd["center"], neglog10(roll_sd["p_levene"]), ls="--", lw=1.3, alpha=0.9)
ax2.set_ylabel("-log10 p (Levene)")
pbox(ax, "Dashed: Levene p for SD difference", "upper right")
plt.show()

# --------------- 12) Rolling quantiles (0.25, 0.5, 0.75) + rolling MWU p ---------------
quantiles = [0.25, 0.5, 0.75]
centers3 = np.arange(df[age_col].min(), df[age_col].max()+1, 2)
half_width3 = 6

def roll_quantiles(df_sex):
    out = {q: [] for q in quantiles}
    for c in centers3:
        w = within_age_window(df_sex, c, half_width3, age_col)["residual"].values
        if len(w) >= 6:
            for q in quantiles:
                out[q].append(np.quantile(w, q))
        else:
            for q in quantiles:
                out[q].append(np.nan)
    return out

m_sub = df[df[sex_col]=="Male"]; f_sub = df[df[sex_col]=="Female"]
m_q = roll_quantiles(m_sub); f_q = roll_quantiles(f_sub)

p_roll_mwu = []
for c in centers3:
    sm_ = within_age_window(m_sub, c, half_width3, age_col)["residual"].values
    sf_ = within_age_window(f_sub, c, half_width3, age_col)["residual"].values
    if len(sm_)>=6 and len(sf_)>=6:
        _, p = stats.mannwhitneyu(sm_, sf_, alternative="two-sided")
    else:
        p = np.nan
    p_roll_mwu.append(p)

fig, ax = plt.subplots(figsize=(10,6))
for sex, qs in [("Male", m_q), ("Female", f_q)]:
    for q, y in qs.items():
        ax.plot(centers3, y, lw=2, alpha=0.9, label=f"{sex} q={q:.2f}")
ax.axhline(0, ls="--", lw=1, c="gray")
ax.set_xlabel("Age"); ax.set_ylabel("Residual (rolling quantiles)")
ax.set_title("Rolling residual quantiles by age and sex")
ax.legend(ncol=2, loc="upper left")
ax2 = ax.twinx()
ax2.plot(centers3, neglog10(np.array(p_roll_mwu)), ls="--", lw=1.5, alpha=0.9)
ax2.set_ylabel("-log10 p (MWU per window)")
pbox(ax, "Dashed: rolling MWU p", "lower right")
plt.show()

# --------------- 13) Faceted boxplots across age bins + per-bin p in titles ---------------
g = sns.catplot(
    data=df, x=sex_col, y="residual", col="age_bin",
    kind="box", col_wrap=5, sharey=True, showfliers=False, height=3
)
for ax, b in zip(g.axes.flatten(), sorted(df["age_bin"].unique())):
    m = df[(df["age_bin"]==b) & (df[sex_col]=="Male")]["residual"].values
    f = df[(df["age_bin"]==b) & (df[sex_col]=="Female")]["residual"].values
    if len(m)>=3 and len(f)>=3:
        _, pb = stats.mannwhitneyu(m, f, alternative="two-sided")
        ax.set_title(f"bin {b} (p={pb:.2e})")
    else:
        ax.set_title(f"bin {b} (p=NA)")
    ax.axhline(0, ls="--", lw=1, c="gray")
g.fig.subplots_adjust(top=0.9)
g.fig.suptitle("Residual by sex across age bins (per-bin MWU p shown)\n(ages 80–92 grouped at bin 80)")
plt.show()

# --------------- 14) OLS interaction model fit lines + global p ---------------
ages_grid = np.linspace(df[age_col].min(), df[age_col].max(), 200)
pred_df = pd.DataFrame({
    "Age": np.tile(ages_grid, 2),
    "Sex": np.repeat(["Female","Male"], len(ages_grid))
})
pred_df["pred"] = model.predict(pred_df)

fig, ax = plt.subplots(figsize=(8,5))
sns.scatterplot(data=df, x=age_col, y="residual", hue=sex_col, alpha=0.25, s=18, ax=ax)
for sex in ["Female","Male"]:
    sub = pred_df[pred_df["Sex"]==sex]
    ax.plot(sub["Age"], sub["pred"], lw=3, label=f"{sex} OLS fit")
ax.axhline(0, ls="--", lw=1, c="gray")
ax.set_title("OLS: residual ~ Age * Sex (fitted lines)")
ax.legend()
pbox(ax, f"Interaction p={p_int:.2e}", "upper right")
plt.show()

print("\nDone. Outliers (top 5% |residual|) dropped and ages 80–92 grouped together. ✅")


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
ADDITIONAL DIAGNOSTIC PLOTS (updated)
- Drops the TOP 5% largest |residual| (outliers by MAE)
- Treats ALL ages in [80, 92] as the SAME 5-year group (bin left edge = 80)
- Shows residuals (y_pred - Age) vs Biological Sex with many diagnostics

Reads: test_preds_covid.csv
Expected columns (case/space tolerant): Age, y_pred, Biological Sex

What’s included (all use plt.show()):
1) Residual histograms faceted by age group (sex overlay)
2) 2D density (KDE) of residual vs age by sex
3) Two-sample QQ plot (Male vs Female residuals)
4) Heatmap: mean residual by (age-bin × sex)  [80–92 grouped]
5) Scatter residual vs age colored by |residual| and styled by sex
6) Rolling Male–Female mean difference with bootstrap 95% CI
7) Residual variance across 5y age bins by sex         [80–92 grouped]
8) Predicted vs Actual Age by sex + identity line
9) Overall residual-vs-age LOWESS (no hue) + per-sex Pearson r printed
10) Bias map: Male–Female mean residual difference across ages (smoothed)  [80–92 grouped]

Requires: pandas, numpy, scipy, matplotlib, seaborn, statsmodels
"""

import warnings, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from statsmodels.nonparametric.smoothers_lowess import lowess

warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 140
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
sns.set_context("talk")

# ----------------- helpers -----------------
def colfind(df, candidates):
    cmap = {c.lower().strip(): c for c in df.columns}
    for cand in candidates:
        k = cand.lower().strip()
        if k in cmap:
            return cmap[k]
    raise KeyError(f"Missing expected column among: {candidates}")

def pbox(ax, text, loc="upper right"):
    ax.text(
        0.98 if "right" in loc else 0.02,
        0.98 if "upper" in loc else 0.02,
        text, transform=ax.transAxes,
        ha="right" if "right" in loc else "left",
        va="top" if "upper" in loc else "bottom",
        bbox=dict(boxstyle="round", alpha=0.15, pad=0.4)
    )

def bootstrap_ci_diff(m, f, n_resamples=2000, confidence=0.95, seed=1):
    """Bootstrap CI for difference in means: mean(m)-mean(f)."""
    rng = np.random.default_rng(seed)
    m = np.asarray(m); f = np.asarray(f)
    diffs = []
    for _ in range(n_resamples):
        ms = rng.choice(m, size=len(m), replace=True)
        fs = rng.choice(f, size=len(f), replace=True)
        diffs.append(ms.mean() - fs.mean())
    diffs = np.sort(diffs)
    lo = np.percentile(diffs, (1-confidence)/2*100)
    hi = np.percentile(diffs, (1+(confidence))/2*100)
    return np.mean(diffs), lo, hi

# ----------------- load -----------------
IN_FILE = "test_preds_covid.csv"
df = pd.read_csv(IN_FILE)

age_col  = colfind(df, ["Age"])
pred_col = colfind(df, ["y_pred","pred","prediction"])
sex_col  = colfind(df, ["Biological Sex","Sex","gender"])

df[age_col]  = pd.to_numeric(df[age_col], errors="coerce")
df[pred_col] = pd.to_numeric(df[pred_col], errors="coerce")
df[sex_col]  = df[sex_col].astype(str).str.strip().replace(
    {"F":"Female","M":"Male","female":"Female","male":"Male"}
)
df = df.dropna(subset=[age_col,pred_col,sex_col]).copy()
df = df[df[sex_col].isin(["Male","Female"])].copy()

# residuals + outlier drop
df["residual"] = df[pred_col] - df[age_col]
df["abs_residual"] = df["residual"].abs()
cut95 = df["abs_residual"].quantile(0.95)       # DROP TOP 5% |residual|
df = df[df["abs_residual"] <= cut95].copy()

# 5y bin left-edges; FORCE ages in [80,92] into the same bin (80)
age_left = (df[age_col] // 5 * 5).astype(int)
age_left = np.where((df[age_col] >= 80) & (df[age_col] <= 92), 80, age_left)
df["age_bin"] = age_left.astype(int)

# quick splits
male   = df.loc[df[sex_col]=="Male","residual"].values
female = df.loc[df[sex_col]=="Female","residual"].values

# ----------------- 1) Residual histograms by age groups -----------------
age_groups = [0,30,50,70,100]  # coarse groups (unchanged by 80–92 rule)
df["age_group"] = pd.cut(df[age_col], age_groups, labels=["<30","30–50","50–70","70+"])
g = sns.displot(
    data=df, x="residual", hue=sex_col, col="age_group", col_wrap=2,
    common_norm=False, bins=30, facet_kws=dict(sharex=True, sharey=True),
    kde=False, multiple="dodge", height=4, aspect=1.2
)
g.fig.subplots_adjust(top=0.9)
g.fig.suptitle("Residual histograms by age group and sex (top 5% |res| dropped)")
plt.show()

# ----------------- 2) 2D density (KDE) of residual vs age by sex -----------------
fig, ax = plt.subplots(figsize=(8,5))
for sex in ["Female","Male"]:
    sub = df[df[sex_col]==sex]
    sns.kdeplot(
        data=sub, x=age_col, y="residual", fill=True, thresh=0.05, levels=10,
        alpha=0.35, ax=ax, label=f"{sex} KDE"
    )
ax.axhline(0, ls="--", c="gray", lw=1)
ax.set_title("2D density of residual vs age by sex")
ax.legend()
plt.show()

# ----------------- 3) Two-sample QQ plot (Male vs Female) -----------------
sm.qqplot_2samples(
    df.query("`%s`=='Male'" % sex_col)["residual"],
    df.query("`%s`=='Female'" % sex_col)["residual"],
    line='45'
)
plt.title("Male vs Female residual quantiles (QQ comparison)")
plt.show()

# ----------------- 4) Heatmap: mean residual by (age-bin × sex) -----------------
# Custom bin edges with a single [80,92] bin
bins_custom = list(range(0,80,5)) + [80,92,100]   # ...75, 80–92, 92–100
heat = (
    df.groupby([pd.cut(df[age_col], bins_custom), sex_col])["residual"]
      .mean()
      .unstack(sex_col)
)
fig, ax = plt.subplots(figsize=(9,3.8))
sns.heatmap(heat.T, cmap="coolwarm", center=0, cbar_kws={'label':'Mean residual'}, ax=ax)
ax.set_xlabel("Age bin (5y; 80–92 grouped)")
ax.set_ylabel("Sex")
ax.set_title("Mean residual by age bin and sex")
plt.show()

# ----------------- 5) Scatter residual vs age colored by |residual| -----------------
fig, ax = plt.subplots(figsize=(8,5))
sns.scatterplot(
    data=df, x=age_col, y="residual",
    hue="abs_residual", size="abs_residual", style=sex_col,
    palette="viridis", alpha=0.65, ax=ax
)
ax.axhline(0, ls="--", c="gray", lw=1)
ax.set_title("Residual vs Age colored by |residual| (style=sex)")
plt.show()

# ----------------- 6) Rolling M–F mean difference with bootstrap CI -----------------
centers = np.arange(df[age_col].min()+6, df[age_col].max()-6, 3)  # 12-yr windows sliding by 3
half_w = 6
rows = []
for c in centers:
    sub = df[(df[age_col] >= c-half_w) & (df[age_col] <= c+half_w)]
    m = sub.loc[sub[sex_col]=="Male","residual"].values
    f = sub.loc[sub[sex_col]=="Female","residual"].values
    if len(m) >= 15 and len(f) >= 15:
        diff = m.mean() - f.mean()
        dmean, lo, hi = bootstrap_ci_diff(m, f, n_resamples=3000)
        rows.append((c, diff, lo, hi))
roll = pd.DataFrame(rows, columns=["center","diff","lo","hi"])

fig, ax = plt.subplots(figsize=(9,4.5))
ax.fill_between(roll["center"], roll["lo"], roll["hi"], alpha=0.2, label="95% bootstrap CI")
ax.plot(roll["center"], roll["diff"], marker="o", lw=2, label="Male–Female mean diff")
ax.axhline(0, ls="--", c="gray")
ax.set_xlabel("Age (center of ~12y window)")
ax.set_ylabel("Residual mean diff (M–F)")
ax.set_title("Rolling sex difference with 95% bootstrap CI\n(top 5% |res| dropped)")
ax.legend()
plt.show()

# ----------------- 7) Residual variance across age bins by sex -----------------
var_agg = df.groupby([sex_col, "age_bin"])["residual"].var().reset_index()
fig, ax = plt.subplots(figsize=(9,4.5))
sns.lineplot(data=var_agg, x="age_bin", y="residual", hue=sex_col, marker="o", ax=ax)
ax.set_title("Residual variance across 5-year bins (80–92 grouped)")
ax.set_xlabel("Age bin left edge (5y)")
ax.set_ylabel("Variance of residual")
plt.show()

# ----------------- 8) Predicted vs Actual Age by sex -----------------
g = sns.lmplot(
    data=df, x=age_col, y=pred_col, hue=sex_col,
    ci=None, height=5, aspect=1.2
)
ax = g.axes[0,0]
min_a, max_a = df[age_col].min(), df[age_col].max()
ax.plot([min_a, max_a], [min_a, max_a], ls="--", c="gray", lw=1.5, label="identity")
ax.set_title("Predicted vs Actual Age by sex (top 5% |res| dropped)")
ax.legend()
plt.show()

# ----------------- 9) Overall residual vs age LOWESS + per-sex Pearson r (printed) -----------------
fig, ax = plt.subplots(figsize=(8,5))
ax.scatter(df[age_col], df["residual"], s=12, alpha=0.35, label="points")
smth = lowess(df["residual"], df[age_col], frac=0.3, return_sorted=True)
ax.plot(smth[:,0], smth[:,1], lw=3, label="LOWESS")
ax.axhline(0, ls="--", c="gray", lw=1)
ax.set_xlabel("Age"); ax.set_ylabel("Residual")
ax.set_title("Residuals vs Age (overall LOWESS)")
ax.legend()
plt.show()

for sex in ["Male","Female"]:
    sub = df[df[sex_col]==sex]
    r, p = stats.pearsonr(sub[age_col], sub["residual"])
    print(f"[Residual~Age] {sex}: Pearson r={r:.3f}, p={p:.3e}")

# ----------------- 10) Bias map: Male–Female residual difference (smoothed) -----------------
# Use the same custom bins so 80–92 is a single group
bins = bins_custom
mf = df.pivot_table(index=pd.cut(df[age_col], bins), columns=sex_col, values="residual", aggfunc="mean")
mf["M_minus_F"] = mf.get("Male") - mf.get("Female")

def smooth1d(x, k=3):
    x = np.asarray(x, float)
    if np.all(np.isnan(x)): return x
    from numpy.lib.stride_tricks import sliding_window_view
    pad = np.pad(x, (k//2, k//2), mode="edge")
    win = sliding_window_view(pad, k)
    return np.nanmean(win, axis=1)

smoothed = smooth1d(mf["M_minus_F"].values, k=3)

fig, ax = plt.subplots(figsize=(9,2.8))
im = ax.imshow(smoothed[None,:], cmap="coolwarm", aspect="auto", vmin=-5, vmax=5)
ax.set_yticks([])
ax.set_xticks(np.arange(len(bins)-1))
ax.set_xticklabels([f"{bins[i]}–{bins[i+1]}" for i in range(len(bins)-1)], rotation=45, ha="right")
ax.set_title("Bias map: Mean residual difference (Male – Female) across ages\n(80–92 grouped; top 5% |res| dropped)")
cb = plt.colorbar(im, ax=ax); cb.set_label("Mean (M–F) residual")
plt.show()

print("\nAll additional plots shown with 80–92 grouped and top 5% |residual| removed. ✅")


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
ALL-IN-ONE: Residuals (y_pred - Age) vs Biological Sex
- Reads: test_preds_covid.csv (expects columns: Age, y_pred, Biological Sex)
- Computes residuals
- Generates a comprehensive set of plots, each with P-value annotations
- Prints additional per-bin/window statistics to console

Requires: pandas, numpy, scipy, matplotlib, seaborn, statsmodels
"""

import warnings, math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.nonparametric.smoothers_lowess import lowess

warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 140
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
sns.set_context("talk")

# --------------- I/O ---------------
IN_FILE = "test_preds_covid.csv"

# --------------- Helpers ---------------
def colfind(df, candidates):
    cmap = {c.lower().strip(): c for c in df.columns}
    for cand in candidates:
        k = cand.lower().strip()
        if k in cmap:
            return cmap[k]
    raise KeyError(f"Missing expected column among: {candidates}")

def pbox(ax, text, loc="upper right"):
    # Annotate a small p-value box on the axes
    ax.text(
        0.98 if "right" in loc else 0.02,
        0.98 if "upper" in loc else 0.02,
        text,
        transform=ax.transAxes,
        ha="right" if "right" in loc else "left",
        va="top" if "upper" in loc else "bottom",
        bbox=dict(boxstyle="round", alpha=0.15, pad=0.4)
    )

def ecdf(a):
    a = np.sort(np.asarray(a))
    y = np.arange(1, len(a)+1) / len(a) if len(a) else np.array([])
    return a, y

def cliff_delta(x, y):
    x = np.asarray(x); y = np.asarray(y)
    n = len(x); m = len(y)
    if n*m == 0:
        return np.nan
    greater = 0; less = 0
    for xi in x:
        greater += np.sum(xi > y)
        less    += np.sum(xi < y)
    return (greater - less) / (n*m)

def within_age_window(df, center, half_width, age_col):
    lo = center - half_width
    hi = center + half_width
    return df[(df[age_col] >= lo) & (df[age_col] <= hi)]

def safe_sem(x):
    x = np.asarray(x)
    return stats.sem(x, nan_policy="omit") if len(x) > 1 else np.nan

def neglog10(p):
    with np.errstate(divide="ignore"):
        return -np.log10(p)

# --------------- Load ---------------
df = pd.read_csv(IN_FILE)

age_col  = colfind(df, ["Age"])
pred_col = colfind(df, ["y_pred", "pred", "prediction"])
sex_col  = colfind(df, ["Biological Sex", "Sex", "gender"])

df[age_col]  = pd.to_numeric(df[age_col], errors="coerce")
df[pred_col] = pd.to_numeric(df[pred_col], errors="coerce")
df[sex_col]  = df[sex_col].astype(str).str.strip().replace(
    {"F":"Female","M":"Male","female":"Female","male":"Male"}
)

df = df.dropna(subset=[age_col, pred_col, sex_col]).copy()
df = df[df[sex_col].isin(["Male","Female"])].copy()

# Residuals
df["residual"]      = df[pred_col] - df[age_col]
df["abs_residual"]  = df["residual"].abs()
df["age_bin"]       = (df[age_col] // 5 * 5).astype(int)

male   = df.loc[df[sex_col]=="Male",   "residual"].values
female = df.loc[df[sex_col]=="Female", "residual"].values
male_abs   = np.abs(male)
female_abs = np.abs(female)

# --------------- Global stats (printed) ---------------
print("\n=== OVERALL STATS ===")
print(f"N total={len(df)} | N Male={len(male)} | N Female={len(female)}")
print(f"Mean residual Male   = {np.nanmean(male):.3f} ± {np.nanstd(male, ddof=1):.3f} (SD)")
print(f"Mean residual Female = {np.nanmean(female):.3f} ± {np.nanstd(female, ddof=1):.3f} (SD)")

U, p_mwu   = stats.mannwhitneyu(male, female, alternative="two-sided")
t, p_t     = stats.ttest_ind(male, female, equal_var=False)
lev, p_lev = stats.levene(male, female, center="median")
ks, p_ks   = stats.ks_2samp(male, female)
t_abs, p_abs = stats.ttest_ind(male_abs, female_abs, equal_var=False)
cd = cliff_delta(male, female)

print(f"Mann–Whitney U: U={U:.0f}, p={p_mwu:.3e}")
print(f"Welch t-test (means): t={t:.3f}, p={p_t:.3e}")
print(f"Levene (variance): stat={lev:.3f}, p={p_lev:.3e}")
print(f"KS (distribution): D={ks:.3f}, p={p_ks:.3e}")
print(f"Welch t-test (|residual|): t={t_abs:.3f}, p={p_abs:.3e}")
print(f"Cliff’s delta (Male vs Female residual): {cd:.3f}")

# --------------- 1) Violin+strip: residual by sex ---------------
fig, ax = plt.subplots(figsize=(7,5))
sns.violinplot(data=df, x=sex_col, y="residual", inner="quartile", cut=0, ax=ax)
sns.stripplot(data=df, x=sex_col, y="residual", color="k", alpha=0.35, size=2, jitter=0.2, ax=ax)
ax.axhline(0, ls="--", lw=1, c="gray")
ax.set_title("Residual distribution by sex")
pbox(ax, f"MWU p={p_mwu:.2e}\nWelch p={p_t:.2e}\nLevene p={p_lev:.2e}\nKS p={p_ks:.2e}", "upper right")
plt.show()

# --------------- 2) Box+swarm: residual by sex ---------------
fig, ax = plt.subplots(figsize=(7,5))
sns.boxplot(data=df, x=sex_col, y="residual", showfliers=False, ax=ax)
sns.swarmplot(
    data=df.sample(min(len(df), 800), random_state=1),
    x=sex_col, y="residual", size=2, alpha=0.5, ax=ax
)
ax.axhline(0, ls="--", lw=1, c="gray")
ax.set_title("Residual boxplot by sex")
pbox(ax, f"MWU p={p_mwu:.2e} | Welch p={p_t:.2e}", "upper right")
plt.show()

# --------------- 3) KDE overlay of residual ---------------
fig, ax = plt.subplots(figsize=(7,5))
sns.kdeplot(data=df, x="residual", hue=sex_col, common_norm=False, fill=True, alpha=0.4, ax=ax)
ax.axvline(0, ls="--", lw=1, c="gray")
ax.set_title("Residual density by sex")
pbox(ax, f"KS p={p_ks:.2e}", "upper right")
plt.show()

# --------------- 4) ECDF of |residual| ---------------
fig, ax = plt.subplots(figsize=(7,5))
for lab, arr in [("Male", male_abs), ("Female", female_abs)]:
    x, y = ecdf(arr)
    ax.step(x, y, where="post", label=lab)
ax.set_xlabel("|residual|"); ax.set_ylabel("ECDF")
ax.set_title("ECDF of absolute residuals (extremity)")
ax.legend()
pbox(ax, f"Welch(|res|) p={p_abs:.2e}", "lower right")
plt.show()

# --------------- 5) QQ-plots by sex ---------------
fig, axes = plt.subplots(1, 2, figsize=(10,5))
sm.qqplot(male, line="s", ax=axes[0]); axes[0].set_title("QQ: Male residual")
sm.qqplot(female, line="s", ax=axes[1]); axes[1].set_title("QQ: Female residual")
plt.show()

# --------------- 6) Scatter residual vs age + LOWESS per sex ---------------
fig, ax = plt.subplots(figsize=(8,5))
for sex in ["Male", "Female"]:
    sub = df[df[sex_col]==sex]
    ax.scatter(sub[age_col], sub["residual"], s=12, alpha=0.35, label=f"{sex} points")
    if len(sub) > 5:
        smth = lowess(sub["residual"], sub[age_col], frac=0.3, return_sorted=True)
        ax.plot(smth[:,0], smth[:,1], lw=2, label=f"{sex} LOWESS")
ax.axhline(0, ls="--", lw=1, c="gray")
ax.set_xlabel("Age"); ax.set_ylabel("Residual")
ax.set_title("Residual vs Age (LOWESS) by sex")
ax.legend()
# OLS interaction global p-value (for annotation)
dmod = df.rename(columns={sex_col: "Sex"})
dmod["Sex"] = pd.Categorical(dmod["Sex"], categories=["Female","Male"])
model = smf.ols("residual ~ Age * Sex", data=dmod).fit()
p_int = model.pvalues.get("Age:Sex[T.Male]", np.nan)
pbox(ax, f"OLS interaction p={p_int:.2e}", "upper right")
plt.show()

# --------------- 7) Binned means ±95% CI by sex + per-bin MWU as -log10(p) ---------------
agg = (df.groupby([sex_col, "age_bin"])["residual"]
         .agg(["mean","count","std"]).reset_index())
agg["sem"] = agg["std"] / np.sqrt(agg["count"])
agg["lo"]  = agg["mean"] - 1.96*agg["sem"]
agg["hi"]  = agg["mean"] + 1.96*agg["sem"]

# Per-bin MWU p-values
bins = np.sort(df["age_bin"].unique())
pvals_bin = []
for b in bins:
    m = df[(df["age_bin"]==b) & (df[sex_col]=="Male")]["residual"].values
    f = df[(df["age_bin"]==b) & (df[sex_col]=="Female")]["residual"].values
    if len(m)>=3 and len(f)>=3:
        _, pb = stats.mannwhitneyu(m, f, alternative="two-sided")
    else:
        pb = np.nan
    pvals_bin.append(pb)

fig, ax = plt.subplots(figsize=(9,6))
for sex in ["Male","Female"]:
    sub = agg[agg[sex_col]==sex]
    ax.plot(sub["age_bin"], sub["mean"], marker="o", label=f"{sex} mean")
    ax.fill_between(sub["age_bin"], sub["lo"], sub["hi"], alpha=0.2)
ax.axhline(0, ls="--", lw=1, c="gray")
ax.set_xlabel("Age bin (5y)"); ax.set_ylabel("Mean residual ±95% CI")
ax.set_title("Binned residual means by sex (with per-bin significance)")
ax.legend(loc="upper left")

# Add a small -log10(p) significance track
ax2 = ax.twinx()
ax2.plot(bins, [neglog10(p) if not np.isnan(p) else np.nan for p in pvals_bin],
         lw=1.5, ls="--", marker="x", alpha=0.8)
ax2.set_ylabel("-log10 p (MWU per bin)")
pbox(ax, "Per-bin MWU shown as dashed line (right axis)", "lower right")
plt.show()

# --------------- 8) Rolling means and Male-Female diff with rolling p-values ---------------
centers = np.arange(df[age_col].min(), df[age_col].max()+1, 2)
half_width = 5

rows = []
for c in centers:
    sub = within_age_window(df, c, half_width, age_col)
    m = sub.loc[sub[sex_col]=="Male", "residual"].values
    f = sub.loc[sub[sex_col]=="Female","residual"].values
    if len(m)>=3 and len(f)>=3:
        m_mean, f_mean = np.mean(m), np.mean(f)
        se_diff = math.sqrt(safe_sem(m)**2 + safe_sem(f)**2) if (len(m)>1 and len(f)>1) else np.nan
        diff = m_mean - f_mean
        # t-test
        _, p_roll = stats.ttest_ind(m, f, equal_var=False)
        rows.append(dict(center=c, male_mean=m_mean, female_mean=f_mean,
                         diff=diff, p=p_roll))
roll = pd.DataFrame(rows)

fig, (ax1, ax2) = plt.subplots(2,1, figsize=(9,8), sharex=True)
ax1.plot(roll["center"], roll["male_mean"], marker="o", ms=3, label="Male mean")
ax1.plot(roll["center"], roll["female_mean"], marker="o", ms=3, label="Female mean")
ax1.axhline(0, ls="--", lw=1, c="gray"); ax1.set_ylabel("Rolling mean (±~5y)")
ax1.set_title("Rolling-window residual means by sex"); ax1.legend()

ax2.plot(roll["center"], roll["diff"], lw=2, label="Male - Female")
ax2.axhline(0, ls="--", lw=1, c="gray"); ax2.set_xlabel("Age (center)")
ax2.set_ylabel("Mean diff (M-F)")
# Add significance as -log10(p) on twin axis
ax3 = ax2.twinx()
ax3.plot(roll["center"], neglog10(roll["p"]), ls="--", lw=1.5, alpha=0.9, label="-log10 p")
ax3.set_ylabel("-log10 p (Welch t)")
pbox(ax2, "Dashed line: -log10 p for M vs F (Welch)", "upper right")
plt.show()

# --------------- 9) |Residual| by sex ---------------
fig, ax = plt.subplots(figsize=(7,5))
sns.boxplot(data=df, x=sex_col, y="abs_residual", showfliers=False, ax=ax)
sns.stripplot(
    data=df.sample(min(len(df), 800), random_state=2),
    x=sex_col, y="abs_residual", color="k", size=2, alpha=0.35, ax=ax
)
ax.set_title("|Residual| (extremity) by sex")
pbox(ax, f"Welch(|res|) p={p_abs:.2e}", "upper right")
plt.show()

# --------------- 10) Extreme residual rates overall and by age bins (thr=5,10) ---------------
for thr in [5, 10]:
    flag = f"ext_{thr}"
    df[flag] = (df["abs_residual"] >= thr).astype(int)

    # Overall bar + Fisher's exact
    tab = pd.crosstab(df[sex_col], df[flag])
    if tab.shape == (2,2):
        _, p_fish = stats.fisher_exact(tab.values)
    else:
        p_fish = np.nan

    fig, ax = plt.subplots(figsize=(6,4))
    rate = df.groupby(sex_col)[flag].mean().reset_index()
    sns.barplot(data=rate, x=sex_col, y=flag, ax=ax)
    ax.set_ylim(0,1)
    ax.set_title(f"Proportion of |residual| ≥ {thr} by sex")
    pbox(ax, f"Fisher exact p={p_fish:.2e}", "upper right")
    plt.show()

    # By 5y age bins with per-bin Fisher
    fig, ax = plt.subplots(figsize=(9,5))
    rate_bin = df.groupby([sex_col, "age_bin"])[flag].mean().reset_index()
    for sex in ["Male","Female"]:
        sub = rate_bin[rate_bin[sex_col]==sex]
        ax.plot(sub["age_bin"], sub[flag], marker="o", label=sex)
    ax.set_ylim(0,1); ax.set_xlabel("Age bin (5y)"); ax.set_ylabel(f"Proportion ≥{thr}")
    ax.set_title(f"Extreme |residual| rate by age bin (thr={thr})")
    ax.legend(loc="upper left")

    # Per-bin Fisher shown as -log10 p on twin axis
    p_list = []
    for b in bins:
        sub = df[df["age_bin"]==b]
        tab = pd.crosstab(sub[sex_col], sub[flag])
        if tab.shape == (2,2) and tab.values.sum()>0:
            _, p_b = stats.fisher_exact(tab.values)
        else:
            p_b = np.nan
        p_list.append(p_b)

    ax2 = ax.twinx()
    ax2.plot(bins, [neglog10(p) if not np.isnan(p) else np.nan for p in p_list],
             ls="--", lw=1.5, alpha=0.85)
    ax2.set_ylabel("-log10 p (Fisher per bin)")
    pbox(ax, "Dashed line: per-bin Fisher p", "lower right")
    plt.show()

# --------------- 11) Rolling SD (variability) vs age by sex + rolling Levene p ---------------
centers2 = np.arange(df[age_col].min(), df[age_col].max()+1, 2)
half_width2 = 6
roll_sd = { "center": [], "male_sd": [], "female_sd": [], "p_levene": [] }
for c in centers2:
    sub = within_age_window(df, c, half_width2, age_col)
    m = sub.loc[sub[sex_col]=="Male","residual"].values
    f = sub.loc[sub[sex_col]=="Female","residual"].values
    if len(m)>=4 and len(f)>=4:
        sd_m = np.std(m, ddof=1); sd_f = np.std(f, ddof=1)
        stat, p_l = stats.levene(m, f, center="median")
        roll_sd["center"].append(c); roll_sd["male_sd"].append(sd_m)
        roll_sd["female_sd"].append(sd_f); roll_sd["p_levene"].append(p_l)
roll_sd = pd.DataFrame(roll_sd)

fig, ax = plt.subplots(figsize=(9,5))
ax.plot(roll_sd["center"], roll_sd["male_sd"], lw=2, label="Male rolling SD")
ax.plot(roll_sd["center"], roll_sd["female_sd"], lw=2, label="Female rolling SD")
ax.set_xlabel("Age"); ax.set_ylabel("Rolling SD (±~6y)")
ax.set_title("Residual variability by age and sex")
ax.legend(loc="upper left")
ax2 = ax.twinx()
ax2.plot(roll_sd["center"], neglog10(roll_sd["p_levene"]), ls="--", lw=1.3, alpha=0.9)
ax2.set_ylabel("-log10 p (Levene)")
pbox(ax, "Dashed line: Levene p for SD difference", "upper right")
plt.show()

# --------------- 12) Rolling quantiles (0.25, 0.5, 0.75) + rolling MWU p ---------------
quantiles = [0.25, 0.5, 0.75]
centers3 = np.arange(df[age_col].min(), df[age_col].max()+1, 2)
half_width3 = 6

def roll_quantiles(df_sex):
    out = {q: [] for q in quantiles}
    for c in centers3:
        w = within_age_window(df_sex, c, half_width3, age_col)["residual"].values
        if len(w) >= 6:
            for q in quantiles:
                out[q].append(np.quantile(w, q))
        else:
            for q in quantiles:
                out[q].append(np.nan)
    return out

m_sub = df[df[sex_col]=="Male"]; f_sub = df[df[sex_col]=="Female"]
m_q = roll_quantiles(m_sub); f_q = roll_quantiles(f_sub)

# rolling MWU p (per window)
p_roll_mwu = []
for c in centers3:
    sm = within_age_window(m_sub, c, half_width3, age_col)["residual"].values
    sf = within_age_window(f_sub, c, half_width3, age_col)["residual"].values
    if len(sm)>=6 and len(sf)>=6:
        _, p = stats.mannwhitneyu(sm, sf, alternative="two-sided")
    else:
        p = np.nan
    p_roll_mwu.append(p)

fig, ax = plt.subplots(figsize=(10,6))
for sex, qs in [("Male", m_q), ("Female", f_q)]:
    for q, y in qs.items():
        ax.plot(centers3, y, lw=2, alpha=0.9, label=f"{sex} q={q:.2f}")
ax.axhline(0, ls="--", lw=1, c="gray")
ax.set_xlabel("Age"); ax.set_ylabel("Residual (rolling quantiles)")
ax.set_title("Rolling residual quantiles by age and sex")
ax.legend(ncol=2, loc="upper left")
ax2 = ax.twinx()
ax2.plot(centers3, neglog10(np.array(p_roll_mwu)), ls="--", lw=1.5, alpha=0.9)
ax2.set_ylabel("-log10 p (MWU per window)")
pbox(ax, "Dashed line: rolling MWU p", "lower right")
plt.show()

# --------------- 13) Faceted boxplots across age bins + per-bin p in titles ---------------
# Compute per-bin MWU and title each facet with p-value
g = sns.catplot(
    data=df, x=sex_col, y="residual", col="age_bin",
    kind="box", col_wrap=5, sharey=True, showfliers=False, height=3
)
for ax, b in zip(g.axes.flatten(), sorted(df["age_bin"].unique())):
    m = df[(df["age_bin"]==b) & (df[sex_col]=="Male")]["residual"].values
    f = df[(df["age_bin"]==b) & (df[sex_col]=="Female")]["residual"].values
    if len(m)>=3 and len(f)>=3:
        _, pb = stats.mannwhitneyu(m, f, alternative="two-sided")
        ax.set_title(f"bin {b} (p={pb:.2e})")
    else:
        ax.set_title(f"bin {b} (p=NA)")
    ax.axhline(0, ls="--", lw=1, c="gray")
g.fig.subplots_adjust(top=0.9)
g.fig.suptitle("Residual by sex across age bins (per-bin MWU p shown)")
plt.show()

# --------------- 14) OLS interaction model fit lines + global p ---------------
ages_grid = np.linspace(df[age_col].min(), df[age_col].max(), 200)
pred_df = pd.DataFrame({
    "Age": np.tile(ages_grid, 2),
    "Sex": np.repeat(["Female","Male"], len(ages_grid))
})
pred_df["pred"] = model.predict(pred_df)

fig, ax = plt.subplots(figsize=(8,5))
sns.scatterplot(data=df, x=age_col, y="residual", hue=sex_col, alpha=0.25, s=18, ax=ax)
for sex in ["Female","Male"]:
    sub = pred_df[pred_df["Sex"]==sex]
    ax.plot(sub["Age"], sub["pred"], lw=3, label=f"{sex} OLS fit")
ax.axhline(0, ls="--", lw=1, c="gray")
ax.set_title("OLS: residual ~ Age * Sex (fitted lines)")
ax.legend()
pbox(ax, f"Interaction p={p_int:.2e}", "upper right")
plt.show()

print("\nDone. All figures were shown with p-value annotations. ✅")


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
ADDITIONAL DIAGNOSTIC PLOTS to strengthen insights about residuals vs sex & age.

Reads: test_preds_covid.csv
Expected columns (case/space tolerant): Age, y_pred, Biological Sex

What’s included (all use plt.show()):
1) Residual histograms faceted by age group (sex overlay)
2) 2D density (KDE) of residual vs age by sex
3) Two-sample QQ plot (Male vs Female residuals)
4) Heatmap: mean residual by (age-bin × sex)
5) Scatter residual vs age colored by |residual| and styled by sex
6) Rolling Male–Female mean difference with bootstrap 95% CI
7) Residual variance across age bins, by sex
8) Predicted vs Actual Age by sex + identity line
9) Overall residual-vs-age LOWESS (no hue) + per-sex Pearson r printed
10) Bias map: Male–Female mean residual difference across ages (smoothed)

Requires: pandas, numpy, scipy, matplotlib, seaborn, statsmodels
"""

import warnings, math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from statsmodels.nonparametric.smoothers_lowess import lowess

warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 140
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
sns.set_context("talk")

# ----------------- helpers -----------------
def colfind(df, candidates):
    cmap = {c.lower().strip(): c for c in df.columns}
    for cand in candidates:
        k = cand.lower().strip()
        if k in cmap:
            return cmap[k]
    raise KeyError(f"Missing expected column among: {candidates}")

def pbox(ax, text, loc="upper right"):
    ax.text(
        0.98 if "right" in loc else 0.02,
        0.98 if "upper" in loc else 0.02,
        text, transform=ax.transAxes,
        ha="right" if "right" in loc else "left",
        va="top" if "upper" in loc else "bottom",
        bbox=dict(boxstyle="round", alpha=0.15, pad=0.4)
    )

def bootstrap_ci_diff(m, f, n_resamples=2000, confidence=0.95, seed=1):
    """Bootstrap CI for difference in means: mean(m)-mean(f)."""
    rng = np.random.default_rng(seed)
    m = np.asarray(m); f = np.asarray(f)
    diffs = []
    for _ in range(n_resamples):
        ms = rng.choice(m, size=len(m), replace=True)
        fs = rng.choice(f, size=len(f), replace=True)
        diffs.append(ms.mean() - fs.mean())
    diffs = np.sort(diffs)
    lo = np.percentile(diffs, (1-confidence)/2*100)
    hi = np.percentile(diffs, (1+(confidence))/2*100)
    return np.mean(diffs), lo, hi

# ----------------- load -----------------
IN_FILE = "test_preds_covid.csv"
df = pd.read_csv(IN_FILE)

age_col  = colfind(df, ["Age"])
pred_col = colfind(df, ["y_pred","pred","prediction"])
sex_col  = colfind(df, ["Biological Sex","Sex","gender"])

df[age_col]  = pd.to_numeric(df[age_col], errors="coerce")
df[pred_col] = pd.to_numeric(df[pred_col], errors="coerce")
df[sex_col]  = df[sex_col].astype(str).str.strip().replace(
    {"F":"Female","M":"Male","female":"Female","male":"Male"}
)
df = df.dropna(subset=[age_col,pred_col,sex_col]).copy()
df = df[df[sex_col].isin(["Male","Female"])].copy()

df["residual"] = df[pred_col] - df[age_col]
df["abs_residual"] = df["residual"].abs()
df["age_bin"] = (df[age_col] // 5 * 5).astype(int)

# quick splits
male   = df.loc[df[sex_col]=="Male","residual"].values
female = df.loc[df[sex_col]=="Female","residual"].values

# ----------------- 1) Residual histograms by age groups -----------------
age_bins = [0,30,50,70,100]
df["age_group"] = pd.cut(df[age_col], age_bins, labels=["<30","30–50","50–70","70+"])
g = sns.displot(
    data=df, x="residual", hue=sex_col, col="age_group", col_wrap=2,
    common_norm=False, bins=30, facet_kws=dict(sharex=True, sharey=True),
    kde=False, multiple="dodge", height=4, aspect=1.2
)
g.fig.subplots_adjust(top=0.9)
g.fig.suptitle("Residual histograms by age group and sex")
plt.show()

# ----------------- 2) 2D density (KDE) of residual vs age by sex -----------------
fig, ax = plt.subplots(figsize=(8,5))
for sex, ls in [("Female","-"),("Male","--")]:
    sub = df[df[sex_col]==sex]
    sns.kdeplot(
        data=sub, x=age_col, y="residual", fill=True, thresh=0.05, levels=10,
        alpha=0.35, ax=ax, label=f"{sex} KDE"
    )
ax.axhline(0, ls="--", c="gray", lw=1)
ax.set_title("2D density of residual vs age by sex")
ax.legend()
plt.show()

# ----------------- 3) Two-sample QQ plot (Male vs Female) -----------------
sm.qqplot_2samples(
    df.query("`%s`=='Male'" % sex_col)["residual"],
    df.query("`%s`=='Female'" % sex_col)["residual"],
    line='45'
)
plt.title("Male vs Female residual quantiles (QQ comparison)")
plt.show()

# ----------------- 4) Heatmap: mean residual by (age-bin × sex) -----------------
heat = (
    df.groupby([pd.cut(df[age_col], np.arange(0, 100, 5)), sex_col])["residual"]
      .mean()
      .unstack(sex_col)
)
fig, ax = plt.subplots(figsize=(9,3.8))
sns.heatmap(heat.T, cmap="coolwarm", center=0, cbar_kws={'label':'Mean residual'}, ax=ax)
ax.set_xlabel("Age bin (5y)")
ax.set_ylabel("Sex")
ax.set_title("Mean residual by age bin and sex")
plt.show()

# ----------------- 5) Scatter residual vs age colored by |residual| -----------------
fig, ax = plt.subplots(figsize=(8,5))
sns.scatterplot(
    data=df, x=age_col, y="residual",
    hue="abs_residual", size="abs_residual", style=sex_col,
    palette="viridis", alpha=0.65, ax=ax
)
ax.axhline(0, ls="--", c="gray", lw=1)
ax.set_title("Residual vs Age colored by |residual| (style=sex)")
plt.show()

# ----------------- 6) Rolling M–F mean difference with bootstrap CI -----------------
centers = np.arange(df[age_col].min()+6, df[age_col].max()-6, 3)  # 12-yr windows sliding by 3
half_w = 6
rows = []
for c in centers:
    sub = df[(df[age_col] >= c-half_w) & (df[age_col] <= c+half_w)]
    m = sub.loc[sub[sex_col]=="Male","residual"].values
    f = sub.loc[sub[sex_col]=="Female","residual"].values
    if len(m) >= 15 and len(f) >= 15:
        diff = m.mean() - f.mean()
        dmean, lo, hi = bootstrap_ci_diff(m, f, n_resamples=3000)
        rows.append((c, diff, lo, hi))
roll = pd.DataFrame(rows, columns=["center","diff","lo","hi"])

fig, ax = plt.subplots(figsize=(9,4.5))
ax.fill_between(roll["center"], roll["lo"], roll["hi"], alpha=0.2, label="95% bootstrap CI")
ax.plot(roll["center"], roll["diff"], marker="o", lw=2, label="Male–Female mean diff")
ax.axhline(0, ls="--", c="gray")
ax.set_xlabel("Age (center of ~12y window)")
ax.set_ylabel("Residual mean diff (M–F)")
ax.set_title("Rolling sex difference with 95% bootstrap CI")
ax.legend()
plt.show()

# ----------------- 7) Residual variance across age bins by sex -----------------
var_agg = df.groupby([sex_col, "age_bin"])["residual"].var().reset_index()
fig, ax = plt.subplots(figsize=(9,4.5))
sns.lineplot(data=var_agg, x="age_bin", y="residual", hue=sex_col, marker="o", ax=ax)
ax.set_title("Residual variance across age bins")
ax.set_xlabel("Age bin (5y)")
ax.set_ylabel("Variance of residual")
plt.show()

# ----------------- 8) Predicted vs Actual Age by sex -----------------
g = sns.lmplot(
    data=df, x=age_col, y=pred_col, hue=sex_col,
    ci=None, height=5, aspect=1.2
)
ax = g.axes[0,0]
min_a, max_a = df[age_col].min(), df[age_col].max()
ax.plot([min_a, max_a], [min_a, max_a], ls="--", c="gray", lw=1.5, label="identity")
ax.set_title("Predicted vs Actual Age by sex")
ax.legend()
plt.show()

# ----------------- 9) Overall residual vs age LOWESS + per-sex Pearson r (printed) -----------------
fig, ax = plt.subplots(figsize=(8,5))
ax.scatter(df[age_col], df["residual"], s=12, alpha=0.35, label="points")
smth = lowess(df["residual"], df[age_col], frac=0.3, return_sorted=True)
ax.plot(smth[:,0], smth[:,1], lw=3, label="LOWESS")
ax.axhline(0, ls="--", c="gray", lw=1)
ax.set_xlabel("Age"); ax.set_ylabel("Residual")
ax.set_title("Residuals vs Age (overall LOWESS)")
ax.legend()
plt.show()

for sex in ["Male","Female"]:
    sub = df[df[sex_col]==sex]
    r, p = stats.pearsonr(sub[age_col], sub["residual"])
    print(f"[Residual~Age] {sex}: Pearson r={r:.3f}, p={p:.3e}")

# ----------------- 10) Bias map: Male–Female residual difference across ages (smoothed) -----------------
bins = np.arange(0, 100, 5)
mf = df.pivot_table(index=pd.cut(df[age_col], bins), columns=sex_col, values="residual", aggfunc="mean")
mf["M_minus_F"] = mf.get("Male") - mf.get("Female")

# simple 1D smoothing
def smooth1d(x, k=3):
    x = np.asarray(x, float)
    if np.all(np.isnan(x)): return x
    from numpy.lib.stride_tricks import sliding_window_view
    pad = np.pad(x, (k//2, k//2), mode="edge")
    win = sliding_window_view(pad, k)
    return np.nanmean(win, axis=1)

smoothed = smooth1d(mf["M_minus_F"].values, k=3)

fig, ax = plt.subplots(figsize=(9,2.8))
im = ax.imshow(smoothed[None,:], cmap="coolwarm", aspect="auto", vmin=-5, vmax=5)
ax.set_yticks([])
ax.set_xticks(np.arange(len(bins)-1))
ax.set_xticklabels([f"{bins[i]}–{bins[i+1]}" for i in range(len(bins)-1)], rotation=45, ha="right")
ax.set_title("Bias map: Mean residual difference (Male – Female) across ages")
cb = plt.colorbar(im, ax=ax); cb.set_label("Mean (M–F) residual")
plt.show()

print("\nAll additional plots shown. ✅")


In [ ]:
# ============================
# Pseudo-alignment by condition (Normal / Dysplasia / OSCC)
# ============================
import pandas as pd, seaborn as sns, matplotlib.pyplot as plt
import numpy as np

sns.set(context="notebook", style="whitegrid")

# --- load your merged QC table ---
df = pd.read_csv("fastp_aggregate.tsv", sep="\t")

# --- pick the correct pseudo column ---
pseudo_col = "kallisto_pseudoaligned_pct"
df["pseudo_pct"] = pd.to_numeric(df[pseudo_col], errors="coerce")

# --- derive condition from sample name (A= Dysplasia, B= OSCC, C= Normal) ---
def infer_condition(sample):
    if isinstance(sample, str):
        if "A-" in sample[:4]: return "Dysplasia"
        if "B-" in sample[:4]: return "OSCC"
        if "C-" in sample[:4]: return "Normal"
    return np.nan

df["condition"] = df["sample"].apply(infer_condition)

# --- filter valid rows ---
dfc = df.dropna(subset=["pseudo_pct", "condition"]).copy()

# --- basic summary ---
summary = (
    dfc.groupby("condition")["pseudo_pct"]
    .agg(["count","median","mean","std","min","max"])
    .sort_index()
)
print(summary.round(2))

# --- plot ---
order = ["Normal", "Dysplasia", "OSCC"]
plt.figure(figsize=(8,5))
sns.violinplot(data=dfc, x="condition", y="pseudo_pct", order=order, inner="box", cut=0, linewidth=1)
sns.stripplot(data=dfc, x="condition", y="pseudo_pct", order=order, color="k", alpha=0.45, size=3)
plt.xlabel("")
plt.ylabel("Pseudo-alignment (%)")
plt.title("Pseudo-alignment by tissue condition")
plt.tight_layout()
#plt.savefig("results/plots/pseudo_by_condition.png", dpi=160)
plt.show()


In [ ]:
# =========================
# STEP 1: Pseudo-alignment overview (ONE PLOT)
# =========================
# - Loads your QC table (TSV)
# - Detects the pseudo-alignment % column (incl. kallisto_pseudoaligned_pct)
# - Normalizes to 0–100 if needed
# - Plots a histogram with a dashed "low" threshold line
# - Prints quick stats
#
# Output:
#   results/plots/01_pseudoalignment_overview.png
# =========================

import os, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ---------- config ----------
INPUT_TSV = "QC_master.tsv"   # <-- change if needed
OUT_DIR   = "results/plots"; os.makedirs(OUT_DIR, exist_ok=True)
LOW_ALIGN_THRESHOLD = 40.0    # %
sns.set(context="notebook", style="whitegrid")

# ---------- helpers ----------
def coerce_percent(series: pd.Series) -> pd.Series:
    s = pd.to_numeric(series, errors="coerce")
    if s.dropna().empty:
        return s
    looks_ratio = (s.dropna().between(0,1).mean() > 0.6) and (s.max() <= 1.5)
    if looks_ratio:
        s = s * 100.0
    return s.clip(lower=0, upper=100)

def find_col(df, options):
    if not isinstance(options, (list, tuple)): options = [options]
    norm_cols = df.columns.str.lower().str.replace(r"\s+", "_", regex=True)
    for opt in options:
        # exact first
        if opt in df.columns:
            return opt
        # loose match
        target = re.sub(r"\s+", "_", opt.lower())
        if target in norm_cols.values:
            return df.columns[norm_cols.values.tolist().index(target)]
    return None

# ---------- read ----------
df = pd.read_csv(INPUT_TSV, sep="\t")

# ---------- detect pseudo-alignment column ----------
pseudo_aliases = [
    "kallisto_pseudoaligned_pct",     # (your table likely has this)
    "kallisto_percent_pseudoaligned",
    "kallisto_mapped_pct",
    "salmon_mapped_pct",
    "pseudoalign", "pseudo_alignment", "pseudo_alignment_rate",
    "pseudoalign_rate", "pseudoalignment_rate"
]
pseudo_col = find_col(df, pseudo_aliases)

if pseudo_col is None:
    raise ValueError(
        "Could not find a pseudo-alignment column. "
        "Add it manually like: df['pseudo_pct'] = df['kallisto_pseudoaligned_pct']"
    )

df["pseudo_pct"] = coerce_percent(df[pseudo_col])

# ---------- PLOT: distribution with threshold ----------
fig, ax = plt.subplots(figsize=(8,5))
sns.histplot(df["pseudo_pct"].dropna(), bins=30, ax=ax)
ax.axvline(LOW_ALIGN_THRESHOLD, ls="--", lw=1.5, label=f"Low threshold = {LOW_ALIGN_THRESHOLD:.0f}%")
ax.set_xlabel("Pseudo-alignment (%)")
ax.set_ylabel("Samples")
ax.set_title("Pseudo-alignment distribution (all samples)")
ax.legend()
plt.tight_layout()
out_path = os.path.join(OUT_DIR, "01_pseudoalignment_overview.png")
plt.savefig(out_path, dpi=160)
plt.show()

# ---------- quick text summary ----------
med = df["pseudo_pct"].median()
low_frac = (df["pseudo_pct"] < LOW_ALIGN_THRESHOLD).mean() * 100
print(f"[Pseudo-alignment] median = {med:.1f}% | below {LOW_ALIGN_THRESHOLD:.0f}% = {low_frac:.1f}% of samples")
print(f"Saved: {out_path}")


In [ ]:
# =========================
# STEP 1b: Pseudo-alignment deep-dive (plots that reinforce the story)
# =========================
import os, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

INPUT_TSV = "QC_master.tsv"
OUT_PLOTS = "results/plots"; os.makedirs(OUT_PLOTS, exist_ok=True)
OUT_TABLES = "results/tables"; os.makedirs(OUT_TABLES, exist_ok=True)
LOW_ALIGN_THRESHOLD = 40.0
sns.set(context="notebook", style="whitegrid")

# ---- helpers ----
def coerce_percent(series: pd.Series) -> pd.Series:
    s = pd.to_numeric(series, errors="coerce")
    if s.dropna().empty:
        return s
    looks_ratio = (s.dropna().between(0,1).mean() > 0.6) and (s.max() <= 1.5)
    if looks_ratio:
        s = s * 100.0
    return s.clip(lower=0, upper=100)

def find_col(df, options):
    norm_cols = df.columns.str.lower().str.replace(r"\s+", "_", regex=True)
    for opt in options:
        if opt in df.columns:
            return opt
        tgt = re.sub(r"\s+", "_", opt.lower())
        if tgt in norm_cols.values:
            return df.columns[norm_cols.values.tolist().index(tgt)]
    return None

# ---- load ----
df = pd.read_csv(INPUT_TSV, sep="\t")

# columns
pseudo_aliases = [
    "kallisto_pseudoaligned_pct","kallisto_percent_pseudoaligned",
    "kallisto_mapped_pct","salmon_mapped_pct",
    "pseudoalign","pseudo_alignment","pseudo_alignment_rate",
    "pseudoalign_rate","pseudoalignment_rate"
]
group_aliases = ["group","condition","status","type","tissue","tissue_status","Phenotype"]
sample_aliases = ["sample","sample_id","sample name","sample_name","library","run","file"]

pseudo_col  = find_col(df, pseudo_aliases)
group_col   = find_col(df, group_aliases)
sample_col  = find_col(df, sample_aliases) or "sample_row"

if sample_col not in df.columns:
    df["sample_row"] = [f"S{i+1}" for i in range(len(df))]

assert pseudo_col is not None, "Couldn't find pseudo-alignment column"
df["pseudo_pct"] = coerce_percent(df[pseudo_col])
if group_col:
    df["group"] = df[group_col].astype(str).str.strip().str.title()

# 1) Violin+swarm by group (if group present)
if "group" in df.columns:
    order = [g for g in ["Normal","Dysplasia","Tumor"] if g in df["group"].unique()]
    if not order:
        order = sorted(df["group"].dropna().unique())
    fig, ax = plt.subplots(figsize=(9,5))
    sns.violinplot(data=df, x="group", y="pseudo_pct", order=order, inner="box", cut=0, ax=ax)
    sns.stripplot(data=df, x="group", y="pseudo_pct", order=order, color="k", size=3, alpha=0.4, ax=ax)
    ax.axhline(LOW_ALIGN_THRESHOLD, ls="--", lw=1.5)
    ax.set_xlabel("")
    ax.set_ylabel("Pseudo-alignment (%)")
    ax.set_title("Pseudo-alignment by group")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_PLOTS, "01b_pseudo_by_group.png"), dpi=160)
    plt.show()

# 2) Cumulative distribution (CDF) + threshold
x = np.sort(df["pseudo_pct"].dropna().values)
y = np.arange(1, len(x)+1) / len(x)
fig, ax = plt.subplots(figsize=(7,5))
ax.plot(x, y, drawstyle="steps-post")
ax.axvline(LOW_ALIGN_THRESHOLD, ls="--", lw=1.2)
ax.set_xlabel("Pseudo-alignment (%)")
ax.set_ylabel("Cumulative fraction of samples")
ax.set_title("Cumulative distribution of pseudo-alignment")
plt.tight_layout()
plt.savefig(os.path.join(OUT_PLOTS, "01c_pseudo_cdf.png"), dpi=160)
plt.show()

# 3) Lowest 15 samples (ranked bars)
worst = df[[sample_col, "pseudo_pct"]].dropna().sort_values("pseudo_pct").head(15)
fig, ax = plt.subplots(figsize=(8,6))
ax.barh(worst[sample_col].astype(str), worst["pseudo_pct"])
ax.axvline(LOW_ALIGN_THRESHOLD, ls="--", lw=1.2)
ax.set_xlabel("Pseudo-alignment (%)")
ax.set_ylabel("Sample")
ax.set_title("Lowest pseudo-alignment samples")
plt.tight_layout()
plt.savefig(os.path.join(OUT_PLOTS, "01d_pseudo_lowest15.png"), dpi=160)
plt.show()

worst.to_csv(os.path.join(OUT_TABLES, "pseudo_lowest_samples.tsv"), sep="\t", index=False)
print("Saved:",
      os.path.join(OUT_PLOTS, "01b_pseudo_by_group.png") if "group" in df.columns else "(no group plot)",
      os.path.join(OUT_PLOTS, "01c_pseudo_cdf.png"),
      os.path.join(OUT_PLOTS, "01d_pseudo_lowest15.png"),
      os.path.join(OUT_TABLES, "pseudo_lowest_samples.tsv"), sep="\n- ")


In [ ]:
# =========================
# STEP 2: STAR alignment vs pseudo-alignment
# =========================
import os, pandas as pd, numpy as np, seaborn as sns, matplotlib.pyplot as plt

INPUT_TSV = "QC_master.tsv"
OUT_DIR   = "results/plots"; os.makedirs(OUT_DIR, exist_ok=True)
sns.set(context="notebook", style="whitegrid")

# ---- load ----
df = pd.read_csv(INPUT_TSV, sep="\t")

# ensure numeric
for c in ["uniq_pct","multi_pct","too_many_loci_pct","unmap_mismatch_pct",
          "unmap_short_pct","unmap_other_pct","kallisto_pseudoaligned_pct"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# total STAR mapped
df["star_total_mapped_pct"] = df["uniq_pct"].fillna(0) + df["multi_pct"].fillna(0)
df["pseudo_pct"] = df["kallisto_pseudoaligned_pct"]

# ---- (1) stacked bar: mean composition ----
parts = ["uniq_pct","multi_pct","too_many_loci_pct",
         "unmap_mismatch_pct","unmap_short_pct","unmap_other_pct"]
means = df[parts].mean()
bottom = 0
fig, ax = plt.subplots(figsize=(7,6))
for p in parts:
    ax.bar(["STAR"], means[p], bottom=bottom, label=p)
    bottom += means[p]
ax.set_ylabel("Percent (%)")
ax.set_title("STAR mapping composition (mean across all samples)")
ax.legend(title="STAR components", bbox_to_anchor=(1.05,1), loc="upper left")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "02a_star_stack_mean.png"), dpi=160)
plt.show()

# ---- (2) correlation scatter: pseudo vs STAR total ----
if df["pseudo_pct"].notna().sum() and df["star_total_mapped_pct"].notna().sum():
    r = df[["pseudo_pct","star_total_mapped_pct"]].dropna().corr().iloc[0,1]
    fig, ax = plt.subplots(figsize=(6,5))
    sns.regplot(data=df, x="pseudo_pct", y="star_total_mapped_pct",
                scatter_kws={"s":35,"alpha":0.7}, ax=ax)
    ax.set_xlabel("Pseudo-alignment (%)")
    ax.set_ylabel("STAR total mapped (%)")
    ax.set_title(f"STAR vs Kallisto alignment (r = {r:.2f})")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "02b_star_vs_pseudo.png"), dpi=160)
    plt.show()
    print(f"Correlation between pseudo and STAR total mapped: r = {r:.2f}")


In [ ]:
# =========================
# STEP 3: rRNA contamination analysis
# =========================
import os, pandas as pd, numpy as np, seaborn as sns, matplotlib.pyplot as plt

INPUT_TSV = "QC_master.tsv"
OUT_DIR   = "results/plots"; os.makedirs(OUT_DIR, exist_ok=True)
sns.set(context="notebook", style="whitegrid")

# ---- load and convert ----
df = pd.read_csv(INPUT_TSV, sep="\t")

for c in ["rrna_overall_rate_pct","kallisto_pseudoaligned_pct",
          "uniq_pct","multi_pct"]:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

df["pseudo_pct"] = df["kallisto_pseudoaligned_pct"]
df["rrna_pct"]   = df["rrna_overall_rate_pct"]
df["star_total_mapped_pct"] = df["uniq_pct"].fillna(0) + df["multi_pct"].fillna(0)

# ---- (1) Distribution of rRNA ----
fig, ax = plt.subplots(figsize=(8,5))
sns.histplot(df["rrna_pct"].dropna(), bins=25, ax=ax)
ax.set_xlabel("rRNA overall rate (%)")
ax.set_ylabel("Samples")
ax.set_title("Distribution of rRNA contamination across samples")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "03a_rrna_distribution.png"), dpi=160)
plt.show()

# ---- (2) rRNA vs pseudo-alignment ----
if df["rrna_pct"].notna().sum() and df["pseudo_pct"].notna().sum():
    r = df[["rrna_pct","pseudo_pct"]].dropna().corr().iloc[0,1]
    fig, ax = plt.subplots(figsize=(6,5))
    sns.regplot(data=df, x="rrna_pct", y="pseudo_pct",
                scatter_kws={"s":40,"alpha":0.7}, ax=ax)
    ax.set_xlabel("rRNA overall rate (%)")
    ax.set_ylabel("Pseudo-alignment (%)")
    ax.set_title(f"rRNA vs pseudo-alignment (r = {r:.2f})")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "03b_rrna_vs_pseudo.png"), dpi=160)
    plt.show()
    print(f"Correlation between rRNA% and pseudo-alignment%: r = {r:.2f}")

# ---- (3) rRNA vs STAR total mapped ----
if df["rrna_pct"].notna().sum() and df["star_total_mapped_pct"].notna().sum():
    r2 = df[["rrna_pct","star_total_mapped_pct"]].dropna().corr().iloc[0,1]
    fig, ax = plt.subplots(figsize=(6,5))
    sns.regplot(data=df, x="rrna_pct", y="star_total_mapped_pct",
                scatter_kws={"s":40,"alpha":0.7}, ax=ax)
    ax.set_xlabel("rRNA overall rate (%)")
    ax.set_ylabel("STAR total mapped (%)")
    ax.set_title(f"rRNA vs STAR mapping (r = {r2:.2f})")
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "03c_rrna_vs_star.png"), dpi=160)
    plt.show()
    print(f"Correlation between rRNA% and STAR total-mapped%: r = {r2:.2f}")


In [ ]:
# STEP 4a: pseudo-alignment vs insert-size peak (short-fragment proxy)

import pandas as pd, numpy as np, seaborn as sns, matplotlib.pyplot as plt

sns.set(context="notebook", style="whitegrid")
df = pd.read_csv("QC_master.tsv", sep="\t")

for c in ["kallisto_pseudoaligned_pct","fastp_insert_peak"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.dropna(subset=["kallisto_pseudoaligned_pct","fastp_insert_peak"]).copy()
df.rename(columns={"kallisto_pseudoaligned_pct":"pseudo_pct"}, inplace=True)

r = df[["fastp_insert_peak","pseudo_pct"]].corr().iloc[0,1]

plt.figure(figsize=(6,5))
sns.regplot(data=df, x="fastp_insert_peak", y="pseudo_pct",
            scatter_kws={"s":40,"alpha":0.7})
plt.xlabel("Insert-size peak (bp, fastp)")
plt.ylabel("Pseudo-alignment (%)")
plt.title(f"Short-fragment hypothesis: insert size vs pseudo (r = {r:.2f})")
plt.tight_layout()
plt.show()
print(f"Correlation r = {r:.2f}  (negative would support 'too-short' fragments)")


In [ ]:
# STEP 4a+ : Short-insert majority — bins, plots, and stats
import os, numpy as np, pandas as pd, seaborn as sns, matplotlib.pyplot as plt
from scipy.stats import spearmanr, mannwhitneyu

sns.set(context="notebook", style="whitegrid")
IN = "QC_master.tsv"
OUT = "results/plots"; os.makedirs(OUT, exist_ok=True)

# --- thresholds you can tweak ---
SHORT_T = 110   # "short" insert peak boundary (bp)
LONG_T  = 130   # "longer" insert peak boundary (bp)

df = pd.read_csv(IN, sep="\t")
for c in ["fastp_insert_peak","kallisto_pseudoaligned_pct"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.dropna(subset=["fastp_insert_peak","kallisto_pseudoaligned_pct"]).copy()
df.rename(columns={"kallisto_pseudoaligned_pct":"pseudo_pct",
                   "fastp_insert_peak":"insert_peak"}, inplace=True)

# --- define bins ---
def label_bin(x):
    if x < SHORT_T: return f"<{SHORT_T}"
    if x >= LONG_T: return f">={LONG_T}"
    return f"{SHORT_T}–{LONG_T-1}"
df["insert_bin"] = df["insert_peak"].apply(label_bin)
bin_order = [f"<{SHORT_T}", f"{SHORT_T}–{LONG_T-1}", f">={LONG_T}"]

# 1) Histogram with thresholds
fig, ax = plt.subplots(figsize=(8,5))
sns.histplot(df["insert_peak"], bins=25, ax=ax)
ax.axvline(SHORT_T, ls="--", lw=1.5, label=f"short threshold = {SHORT_T} bp")
ax.axvline(LONG_T,  ls="--", lw=1.5, label=f"long threshold = {LONG_T} bp")
ax.set_xlabel("Insert-size peak (bp, fastp)")
ax.set_ylabel("Samples")
ax.set_title("Insert-size peak distribution")
ax.legend()
plt.tight_layout(); plt.savefig(os.path.join(OUT,"04d_insert_hist.png"), dpi=160); plt.show()

# 2) CDF
x = np.sort(df["insert_peak"].values)
y = np.arange(1, len(x)+1)/len(x)
fig, ax = plt.subplots(figsize=(7,5))
ax.plot(x, y, drawstyle="steps-post")
ax.axvline(SHORT_T, ls="--", lw=1.2)
ax.axvline(LONG_T,  ls="--", lw=1.2)
ax.set_xlabel("Insert-size peak (bp)")
ax.set_ylabel("Cumulative fraction of samples")
ax.set_title("CDF of insert-size peak")
plt.tight_layout(); plt.savefig(os.path.join(OUT,"04e_insert_cdf.png"), dpi=160); plt.show()

# 3) Pseudo by insert-size bin
fig, ax = plt.subplots(figsize=(8,5))
sns.boxplot(data=df, x="insert_bin", y="pseudo_pct", order=bin_order, ax=ax)
sns.stripplot(data=df, x="insert_bin", y="pseudo_pct", order=bin_order,
              color="k", alpha=0.4, size=3, ax=ax)
ax.set_xlabel("Insert-size bin")
ax.set_ylabel("Pseudo-alignment (%)")
ax.set_title("Pseudo-alignment by insert-size bin")
plt.tight_layout(); plt.savefig(os.path.join(OUT,"04f_pseudo_by_insert_bin.png"), dpi=160); plt.show()

# 4) Stats summary + tests
summary = (df.groupby("insert_bin")["pseudo_pct"]
             .agg(n="count", median="median", q1=lambda s: s.quantile(0.25), q3=lambda s: s.quantile(0.75))
             .reindex(bin_order))
rho, p_spear = spearmanr(df["insert_peak"], df["pseudo_pct"])
a = df.loc[df["insert_bin"]==f"<{SHORT_T}","pseudo_pct"]
b = df.loc[df["insert_bin"]==f">={LONG_T}","pseudo_pct"]
if len(a)>0 and len(b)>0:
    U, p_mwu = mannwhitneyu(a, b, alternative="two-sided")
else:
    U, p_mwu = np.nan, np.nan

print("Insert-size bin summary (pseudo %):\n", summary.round(2), sep="")
print(f"\nSpearman(insert_peak, pseudo) = {rho:.2f} (p={p_spear:.2g})")
print(f"Mann–Whitney <{SHORT_T} vs >={LONG_T}: U={U}, p={p_mwu:.2g}")


In [ ]:
# STEP 5: read-level quality and complexity checks
import pandas as pd, seaborn as sns, matplotlib.pyplot as plt, numpy as np

sns.set(context="notebook", style="whitegrid")
df = pd.read_csv("QC_master.tsv", sep="\t")

for c in ["kallisto_pseudoaligned_pct","fastp_dup_rate","fastp_gc_content","fastp_q30_rate"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df = df.rename(columns={"kallisto_pseudoaligned_pct":"pseudo_pct"}).dropna(subset=["pseudo_pct"])

def scatter(metric, label):
    if metric not in df.columns: return
    r = df[[metric,"pseudo_pct"]].dropna().corr().iloc[0,1]
    plt.figure(figsize=(6,5))
    sns.regplot(data=df, x=metric, y="pseudo_pct", scatter_kws={"s":40,"alpha":0.7})
    plt.xlabel(label)
    plt.ylabel("Pseudo-alignment (%)")
    plt.title(f"{label} vs pseudo-alignment (r = {r:.2f})")
    plt.tight_layout()
    plt.show()

# Duplication rate
scatter("fastp_dup_rate", "Duplication rate (%)")

# GC content
scatter("fastp_gc_content", "GC content (%)")

# Q30 rate
scatter("fastp_q30_rate", "Q30 quality rate (%)")


In [ ]:
# STEP 1: quantify strongest predictors of pseudo-alignment
import pandas as pd, seaborn as sns, matplotlib.pyplot as plt
from scipy.stats import spearmanr

sns.set(context="notebook", style="whitegrid")

df = pd.read_csv("fastp_aggregate.tsv", sep="\t")

df["pseudo_pct"] = pd.to_numeric(df.get("kallisto_pseudoaligned_pct"), errors="coerce")

# choose relevant QC variables
qc_vars = [
    "fastp_insert_peak",
    "fastp_adapter_trimmed_frac",
    "fastp_too_short_frac",
    "fastp_dup_rate",
    "fastp_gc_content_after",
    "fastp_q30_rate_after",
    "rrna_overall_rate_pct"
]

rows=[]
for c in qc_vars:
    if c in df.columns:
        x=pd.to_numeric(df[c], errors="coerce")
        rho,p=spearmanr(x, df["pseudo_pct"], nan_policy="omit")
        rows.append((c,rho,p,x.median(),x.min(),x.max()))
corr=pd.DataFrame(rows,columns=["metric","spearman_r","p","median","min","max"]).sort_values("spearman_r")
corr


In [ ]:
# STEP 2: scatterplots for top explanatory metrics
top_metrics = ["fastp_too_short_frac","fastp_adapter_trimmed_frac","fastp_dup_rate","fastp_gc_content_after","fastp_insert_peak"]

for c in top_metrics:
    if c not in df.columns: continue
    x = pd.to_numeric(df[c], errors="coerce")
    y = pd.to_numeric(df["pseudo_pct"], errors="coerce")
    rho, p = spearmanr(x, y, nan_policy="omit")
    plt.figure(figsize=(6,5))
    sns.regplot(x=x, y=y, scatter_kws={"s":40,"alpha":0.7})
    plt.xlabel(c)
    plt.ylabel("Pseudo-alignment (%)")
    plt.title(f"{c} vs pseudo (Spearman r={rho:.2f}, p={p:.1e})")
    plt.tight_layout()
    plt.show()


In [ ]:
# A) LOAD (works with QC_master.tsv OR the merged fastp_aggregate.tsv)
import pandas as pd, numpy as np

PATH = "fastp_aggregate.tsv"   # or "QC_master.tsv"
df = pd.read_csv(PATH, sep="\t")

# helper: pick whichever column exists
def pick(*candidates):
    for c in candidates:
        if c in df.columns: return c
    return None

# normalize 0–1 -> 0–100 for percent-like fields
def to_num(s):
    x = pd.to_numeric(s, errors="coerce")
    if x.dropna().empty: return x
    if (x.dropna().between(0,1).mean() > 0.6) and (x.max() <= 1.5):
        x = x*100
    return x

# core columns
pseudo_col = pick("kallisto_pseudoaligned_pct","pseudo_pct")
dup_col    = pick("fastp_dup_rate")
gc_col     = pick("fastp_gc_content_after","fastp_gc_content")
q30_col    = pick("fastp_q30_rate_after","fastp_q30_rate")
ins_col    = pick("fastp_insert_peak")
short_col  = pick("fastp_too_short_frac")
adap_col   = pick("fastp_adapter_trimmed_frac")  # may be None
rrna_col   = pick("rrna_overall_rate_pct")
nonhuman_col = pick("fastqscreen_nonhuman_pct")  # from MultiQC JSON (optional)
overrep_col  = pick("overrep_seq_count")         # from MultiQC JSON (optional)
kmer_col     = pick("enriched_kmer_count")       # from MultiQC JSON (optional)

# numeric versions
df["pseudo_pct"] = to_num(df[pseudo_col]) if pseudo_col else np.nan
for c in [dup_col,gc_col,q30_col,ins_col,short_col,adap_col,rrna_col,nonhuman_col]:
    if c: df[c] = to_num(df[c])


In [ ]:
# B) Distributions for values (not just correlations)
import matplotlib.pyplot as plt, seaborn as sns, numpy as np
sns.set(context="notebook", style="whitegrid")

metrics = {
    "Pseudo-alignment (%)":"pseudo_pct",
    "Duplication rate (%)": dup_col,
    "GC content (%)": gc_col,
    "Q30 rate (%)": q30_col,
    "Insert-size peak (bp)": ins_col,
    "Too-short fraction (%)": short_col,
    "Adapter-trimmed fraction (%)": adap_col,
    "rRNA overall rate (%)": rrna_col,
}

for label, col in metrics.items():
    if not col:
        print(f"Skipping {label} (missing)")
        continue
    x = df[col].dropna().values
    if len(x)==0:
        print(f"Skipping {label} (no data)")
        continue

    # HISTOGRAM
    plt.figure(figsize=(7,4.2))
    sns.histplot(x, bins=25)
    plt.title(f"{label} — distribution (n={len(x)})")
    plt.xlabel(label)
    plt.ylabel("Samples")
    plt.tight_layout(); plt.show()

    # ECDF
    xs = np.sort(x); ys = np.arange(1,len(xs)+1)/len(xs)
    plt.figure(figsize=(7,4.2))
    plt.plot(xs, ys, drawstyle="steps-post")
    plt.title(f"{label} — ECDF")
    plt.xlabel(label); plt.ylabel("Cumulative fraction")
    plt.tight_layout(); plt.show()


In [ ]:
import seaborn as sns, matplotlib.pyplot as plt, numpy as np

if ins_col:
    SHORT_T, LONG_T = 110, 130
    def b(x):
        if x < SHORT_T: return f"<{SHORT_T}"
        if x >= LONG_T: return f">={LONG_T}"
        return f"{SHORT_T}–{LONG_T-1}"
    tmp = df[[ins_col,"pseudo_pct"]].dropna().copy()
    tmp["bin"]=tmp[ins_col].apply(b)
    order=[f"<{SHORT_T}", f"{SHORT_T}–{LONG_T-1}", f">={LONG_T}"]

    plt.figure(figsize=(8,5))
    sns.boxplot(data=tmp, x="bin", y="pseudo_pct", order=order)
    sns.stripplot(data=tmp, x="bin", y="pseudo_pct", order=order, color="k", alpha=0.4, size=3)
    plt.xlabel("Insert-size bin"); plt.ylabel("Pseudo-alignment (%)")
    plt.title("Pseudo vs insert-size bins")
    plt.tight_layout(); plt.show()


In [ ]:
if gc_col:
    tmp = df[[gc_col,"pseudo_pct"]].dropna().copy()
    tmp["gc_tertile"] = pd.qcut(tmp[gc_col], q=3, labels=["Low GC","Mid GC","High GC"])
    plt.figure(figsize=(8,5))
    sns.violinplot(data=tmp, x="gc_tertile", y="pseudo_pct", inner="box")
    sns.stripplot(data=tmp, x="gc_tertile", y="pseudo_pct", color="k", alpha=0.4, size=3)
    plt.xlabel("GC content tertile"); plt.ylabel("Pseudo-alignment (%)")
    plt.title("Pseudo vs GC content tertiles")
    plt.tight_layout(); plt.show()


In [ ]:
if dup_col:
    tmp = df[[dup_col,"pseudo_pct"]].dropna().copy()
    tmp["dup_tertile"] = pd.qcut(tmp[dup_col], q=3, labels=["Low dup","Mid dup","High dup"])
    plt.figure(figsize=(8,5))
    sns.boxplot(data=tmp, x="dup_tertile", y="pseudo_pct")
    sns.stripplot(data=tmp, x="dup_tertile", y="pseudo_pct", color="k", alpha=0.4, size=3)
    plt.xlabel("Duplication tertile"); plt.ylabel("Pseudo-alignment (%)")
    plt.title("Pseudo vs duplication tertiles")
    plt.tight_layout(); plt.show()


In [ ]:
# D) Pull read-length & FastQ Screen from MultiQC JSON
import json, re, matplotlib.pyplot as plt, numpy as np, seaborn as sns, pandas as pd, os

MULTIQC_JSON = "multiqc_data.json"  # <-- set this
with open(MULTIQC_JSON,"r",encoding="utf-8") as f:
    mq = json.load(f)

def norm(s): return re.sub(r"\s+","_",str(s))

# pick best/worst by pseudo
rank = df[["sample","pseudo_pct"]].dropna().sort_values("pseudo_pct")
worst = [norm(s) for s in rank.head(3)["sample"]]
best  = [norm(s) for s in rank.tail(3)["sample"]]

# 1) Read-length curves (FastQC)
length_plot = mq.get("report_plot_data", {}).get("fastqc_sequence_length_distribution_plot", {})
series = length_plot.get("datasets", {}) or length_plot.get("data", {})

def plot_length_curves(sample_list, title):
    plt.figure(figsize=(7,5))
    for samp in sample_list:
        if samp not in series:
            print("No length data for", samp);
            continue
        x = np.array(series[samp]["x"], dtype=float)
        y = np.array(series[samp]["y"], dtype=float)
        y = y / (y.sum() if y.sum()>0 else 1.0)
        plt.plot(x, y, label=samp)
    plt.xlabel("Read length (bp)"); plt.ylabel("Density")
    plt.title(title); plt.legend(bbox_to_anchor=(1.02,1), loc="upper left")
    plt.tight_layout(); plt.show()

plot_length_curves(worst, "FastQC length distribution — WORST pseudo")
plot_length_curves(best,  "FastQC length distribution — BEST pseudo")

# 2) FastQ Screen stacked bars for worst 5 (if present)
fqs = mq.get("report_plot_data", {}).get("fastq_screen_plot", {})
series = fqs.get("datasets", {}) or fqs.get("data", {})
targets = [norm(s) for s in rank.head(5)["sample"]]

for samp in targets:
    if samp not in series:
        print("No FastQ Screen for", samp); continue
    labels = series[samp]["x"]; vals = series[samp]["y"]
    lbl, v = zip(*[(l,v) for l,v in zip(labels, vals)])
    fig, ax = plt.subplots(figsize=(8,3))
    ax.barh([samp], [sum(v)], color="lightgray")  # background (100%)
    left=0
    for l, val in zip(lbl, v):
        ax.barh([samp], [val], left=left, label=l)
        left += val
    ax.set_xlim(0,100); ax.set_xlabel("Percent of reads")
    ax.set_title(f"FastQ Screen — {samp}")
    ax.legend(bbox_to_anchor=(1.02,1), loc="upper left")
    plt.tight_layout(); plt.show()


In [ ]:
# E) Joint view: scatter with marginal histograms
import seaborn as sns, pandas as pd

def joint(metric, label):
    if metric not in df.columns: return
    tmp = df[[metric, "pseudo_pct"]].dropna()
    if tmp.empty: return
    g = sns.jointplot(data=tmp, x=metric, y="pseudo_pct", kind="scatter", height=5, marginal_kws=dict(bins=20, fill=True))
    g.set_axis_labels(label, "Pseudo-alignment (%)")
    g.fig.suptitle(f"{label} vs Pseudo — distributions + scatter", y=1.02)
    plt.show()

# examples:
if gc_col:  joint(gc_col, "GC content (%)")
if dup_col: joint(dup_col, "Duplication rate (%)")
if ins_col: joint(ins_col, "Insert-size peak (bp)")


In [ ]:
# =========================
# Fastp aggregate (single file) — clean + value plots
# =========================
import os, re, numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from scipy.stats import spearmanr

sns.set(context="notebook", style="whitegrid")
IN   = "fastp_aggregate.tsv"   # <- change if needed
OUT  = "figs_values"; os.makedirs(OUT, exist_ok=True)

# ---------- load ----------
raw = pd.read_csv(IN, sep="\t")

# There are duplicate names (two 'sample' cols and many *_x / *_y).
# We'll coalesce into a clean set of columns in dfc.
def pick(df, *names, default=np.nan):
    for n in names:
        if n in df.columns: return df[n]
    return pd.Series([default]*len(df))

dfc = pd.DataFrame({
    "sample": pick(raw, "sample").astype(str),
    # pseudo + STAR
    "pseudo_pct": pd.to_numeric(pick(raw, "kallisto_pseudoaligned_pct"), errors="coerce"),
    "star_uniq_pct": pd.to_numeric(pick(raw, "uniq_pct"), errors="coerce"),
    "star_multi_pct": pd.to_numeric(pick(raw, "multi_pct"), errors="coerce"),
    "too_many_loci_pct": pd.to_numeric(pick(raw, "too_many_loci_pct"), errors="coerce"),
    "unmap_mismatch_pct": pd.to_numeric(pick(raw, "unmap_mismatch_pct"), errors="coerce"),
    "unmap_short_pct": pd.to_numeric(pick(raw, "unmap_short_pct"), errors="coerce"),
    "unmap_other_pct": pd.to_numeric(pick(raw, "unmap_other_pct"), errors="coerce"),
    # fastp (prefer 'after' or _y; fall back to raw/_x)
    "q30_rate": pd.to_numeric(pick(raw, "fastp_q30_rate_after", "fastp_q30_rate", "fastp_q30_rate_before"), errors="coerce"),
    "gc_pct":   pd.to_numeric(pick(raw, "fastp_gc_content_after", "fastp_gc_content", "fastp_gc_content_before"), errors="coerce"),
    "dup_rate": pd.to_numeric(pick(raw, "fastp_dup_rate_y", "fastp_dup_rate_x"), errors="coerce"),
    "insert_peak": pd.to_numeric(pick(raw, "fastp_insert_peak_y", "fastp_insert_peak_x"), errors="coerce"),
    "adapter_frac": pd.to_numeric(pick(raw, "fastp_adapter_trimmed_frac"), errors="coerce"),
    "too_short_frac": pd.to_numeric(pick(raw, "fastp_too_short_frac"), errors="coerce"),
    # rRNA
    "rrna_pct": pd.to_numeric(pick(raw, "rrna_overall_rate_pct"), errors="coerce"),
})

# normalize ratios to % where needed (q30/gc/dup are already 0..1 in fastp JSON)
def as_pct(s):
    if s.dropna().empty: return s
    if s.max() <= 1.5:  # 0..1 -> 0..100
        return s*100
    return s

for c in ["q30_rate","gc_pct","dup_rate","adapter_frac","too_short_frac","rrna_pct","pseudo_pct",
          "star_uniq_pct","star_multi_pct","too_many_loci_pct","unmap_mismatch_pct","unmap_short_pct","unmap_other_pct"]:
    if c in dfc.columns:
        dfc[c] = as_pct(dfc[c])

# derive STAR totals
if {"star_uniq_pct","star_multi_pct"}.issubset(dfc.columns):
    dfc["star_total_mapped_pct"] = dfc["star_uniq_pct"].fillna(0) + dfc["star_multi_pct"].fillna(0)

# ---------- helpers ----------
def hist_and_ecdf(values, label, slug):
    x = pd.to_numeric(values, errors="coerce").dropna().values
    if len(x)==0:
        print(f"[skip] {label} (no data)");
        return
    # Histogram
    plt.figure(figsize=(7,4.2))
    sns.histplot(x, bins=25)
    plt.xlabel(label); plt.ylabel("Samples"); plt.title(f"{label} — distribution (n={len(x)})")
    plt.tight_layout(); plt.savefig(os.path.join(OUT, f"{slug}_hist.png"), dpi=160); plt.show()
    # ECDF
    xs = np.sort(x); ys = np.arange(1, len(xs)+1)/len(xs)
    plt.figure(figsize=(7,4.2))
    plt.plot(xs, ys, drawstyle="steps-post")
    plt.xlabel(label); plt.ylabel("Cumulative fraction"); plt.title(f"{label} — ECDF")
    plt.tight_layout(); plt.savefig(os.path.join(OUT, f"{slug}_ecdf.png"), dpi=160); plt.show()

def scatter_vs_pseudo(x, xlabel, slug):
    if "pseudo_pct" not in dfc.columns: return
    d = pd.DataFrame({"x": pd.to_numeric(x, errors="coerce"),
                      "y": pd.to_numeric(dfc["pseudo_pct"], errors="coerce")}).dropna()
    if d.empty:
        print(f"[skip] {xlabel} vs pseudo (no data)");
        return
    rho, p = spearmanr(d["x"], d["y"])
    plt.figure(figsize=(6,5))
    sns.regplot(x=d["x"], y=d["y"], scatter_kws={"s":40,"alpha":0.7})
    plt.xlabel(xlabel); plt.ylabel("Pseudo-alignment (%)")
    plt.title(f"{xlabel} vs pseudo (Spearman r={rho:.2f}, p={p:.1e})")
    plt.tight_layout(); plt.savefig(os.path.join(OUT, f"{slug}_vs_pseudo.png"), dpi=160); plt.show()

# ---------- (1) VALUE distributions + scatter for key metrics ----------
panels = [
    ("Duplication rate (%)", dfc["dup_rate"], "dup_rate"),
    ("GC content (%)",       dfc["gc_pct"],   "gc_pct"),
    ("Q30 rate (%)",         dfc["q30_rate"], "q30_rate"),
    ("Insert-size peak (bp)",dfc["insert_peak"], "insert_peak"),
    ("Adapter-trimmed (%, reads)", dfc["adapter_frac"], "adapter_frac"),
    ("Too-short (%, reads)",       dfc["too_short_frac"], "too_short_frac"),
    ("rRNA overall rate (%)",      dfc["rrna_pct"], "rrna_pct"),
]

for label, series, slug in panels:
    hist_and_ecdf(series, label, slug)
    scatter_vs_pseudo(series, label, slug)

# ---------- (2) Pseudo by insert-size bins ----------
if "insert_peak" in dfc.columns and dfc["insert_peak"].notna().any():
    SHORT_T, LONG_T = 110, 130
    tmp = dfc[["insert_peak","pseudo_pct"]].dropna().copy()
    def bin_lab(x):
        if x < SHORT_T: return f"<{SHORT_T}"
        if x >= LONG_T: return f">={LONG_T}"
        return f"{SHORT_T}–{LONG_T-1}"
    tmp["bin"] = tmp["insert_peak"].apply(bin_lab)
    order = [f"<{SHORT_T}", f"{SHORT_T}–{LONG_T-1}", f">={LONG_T}"]
    plt.figure(figsize=(8,5))
    sns.boxplot(data=tmp, x="bin", y="pseudo_pct", order=order)
    sns.stripplot(data=tmp, x="bin", y="pseudo_pct", order=order, color="k", alpha=0.4, size=3)
    plt.xlabel("Insert-size bin"); plt.ylabel("Pseudo-alignment (%)")
    plt.title("Pseudo-alignment by insert-size bin")
    plt.tight_layout(); plt.savefig(os.path.join(OUT, "pseudo_by_insert_bins.png"), dpi=160); plt.show()

# ---------- (3) Ranked bars for adapter & too-short ----------
for col, lab, slug in [("adapter_frac","Adapter-trimmed (%, reads)","adapter_frac"),
                       ("too_short_frac","Too-short (%, reads)","too_short_frac")]:
    if col in dfc.columns and dfc[col].notna().any():
        t = dfc[["sample", col]].dropna().sort_values(col, ascending=False).head(15)
        plt.figure(figsize=(9,6))
        plt.barh(t["sample"], t[col]); plt.gca().invert_yaxis()
        plt.xlabel(lab); plt.ylabel("Sample"); plt.title(f"Top 15 samples by {lab}")
        plt.tight_layout(); plt.savefig(os.path.join(OUT, f"ranked_{slug}_top15.png"), dpi=160); plt.show()

# ---------- (4) STAR composition mean + pseudo vs STAR ----------
star_parts = [c for c in ["star_uniq_pct","star_multi_pct","too_many_loci_pct",
                          "unmap_mismatch_pct","unmap_short_pct","unmap_other_pct"] if c in dfc.columns]
if star_parts and dfc[star_parts].notna().any().any():
    means = dfc[star_parts].mean()
    bottom = 0
    fig, ax = plt.subplots(figsize=(7,6))
    for part in star_parts:
        ax.bar(["STAR"], means[part], bottom=bottom, label=part)
        bottom += means[part]
    ax.set_ylabel("Percent (%)"); ax.set_title("STAR mapping composition (mean)")
    ax.legend(title="STAR components", bbox_to_anchor=(1.02,1), loc="upper left")
    plt.tight_layout(); plt.savefig(os.path.join(OUT, "star_composition_mean.png"), dpi=160); plt.show()

if "star_total_mapped_pct" in dfc.columns and dfc["star_total_mapped_pct"].notna().any():
    scatter_vs_pseudo(dfc["star_total_mapped_pct"], "STAR total mapped (%)", "star_total_mapped")

print("Saved plots →", OUT)


In [ ]:
# QC plots — notebook version (no argparse)
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.backends.backend_pdf import PdfPages

INFILE = "QC_master.tsv"          # <-- change if needed
OUTDIR = "./qc_plots"                        # <-- change if needed
Path(OUTDIR).mkdir(parents=True, exist_ok=True)

def norm_cols(df):
    df = df.copy()
    df.columns = (df.columns.str.strip()
                            .str.lower()
                            .str.replace(" ", "_")
                            .str.replace("%", "pct"))
    for c in df.columns:
        if c != "sample":
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def ensure_cols(df, cols):
    return [c for c in cols if c in df.columns]

def hist(ax, s, title, xlabel, bins=20):
    s = pd.to_numeric(s, errors="coerce").dropna()
    ax.hist(s, bins=bins); ax.set_title(title); ax.set_xlabel(xlabel); ax.set_ylabel("Count")

def box(ax, s, title, ylabel):
    s = pd.to_numeric(s, errors="coerce").dropna()
    ax.boxplot(s, vert=True); ax.set_title(title); ax.set_ylabel(ylabel)

def scatter(ax, x, y, title, xlabel, ylabel, add_fit=True, annotate=False, samples=None):
    x = pd.to_numeric(x, errors="coerce"); y = pd.to_numeric(y, errors="coerce")
    m = x.notna() & y.notna()
    ax.scatter(x[m], y[m], alpha=0.7, s=25)
    if add_fit and m.sum() >= 3:
        M, B = np.polyfit(x[m], y[m], 1)
        xs = np.linspace(float(x[m].min()), float(x[m].max()), 100)
        ax.plot(xs, M*xs + B, linestyle="--")
        r = np.corrcoef(x[m], y[m])[0,1]
        ax.text(0.01, 0.96, f"y={M:.3g}x+{B:.3g}\nr={r:.3f}", transform=ax.transAxes, va="top", fontsize=9)
    if annotate and samples is not None:
        ys = y[m].sort_values()
        idx = list(ys.head(min(5, len(ys))).index) + list(ys.tail(min(5, len(ys))).index)
        for i in idx:
            ax.annotate(str(samples[i]), (x[i], y[i]), fontsize=7, xytext=(3,3), textcoords="offset points")
    ax.set_title(title); ax.set_xlabel(xlabel); ax.set_ylabel(ylabel)

def bar_stacked(ax, df, cols, title):
    idx = np.arange(len(df)); bottom = np.zeros(len(df))
    for c in cols:
        if c not in df.columns: continue
        v = pd.to_numeric(df[c], errors="coerce").fillna(0).values
        ax.bar(idx, v, bottom=bottom, label=c); bottom += v
    ax.set_title(title); ax.set_xticks(idx)
    ax.set_xticklabels(df["sample"], rotation=90, fontsize=7); ax.legend(fontsize=7, ncol=2)

def heatmap(ax, data, row_labels=None, col_labels=None, title=""):
    im = ax.imshow(data, aspect="auto", cmap="viridis")
    ax.set_title(title)
    if row_labels is not None:
        ax.set_yticks(range(len(row_labels))); ax.set_yticklabels(row_labels, fontsize=7)
    if col_labels is not None:
        ax.set_xticks(range(len(col_labels))); ax.set_xticklabels(col_labels, rotation=45, ha="right", fontsize=8)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

def scatter_matrix(axarr, df, cols, s=10):
    k = len(cols)
    for i in range(k):
        for j in range(k):
            ax = axarr[i, j]
            if i == j:
                ax.hist(pd.to_numeric(df[cols[i]], errors="coerce").dropna(), bins=15)
            else:
                x = pd.to_numeric(df[cols[j]], errors="coerce")
                y = pd.to_numeric(df[cols[i]], errors="coerce")
                m = x.notna() & y.notna()
                ax.scatter(x[m], y[m], s=s, alpha=0.6)
            if i == k-1: ax.set_xlabel(cols[j], fontsize=8, rotation=45)
            else: ax.set_xticklabels([])
            if j == 0: ax.set_ylabel(cols[i], fontsize=8)
            else: ax.set_yticklabels([])
    plt.tight_layout()

# -------- run --------
df = pd.read_csv(INFILE, sep="\t")
df = norm_cols(df)
assert "sample" in df.columns, "Missing 'sample' column."

pseudo = "kallisto_pseudoaligned_pct"
rrna   = "rrna_overall_rate_pct"
star   = ["uniq_pct","multi_pct","too_many_loci_pct","unmap_mismatch_pct","unmap_short_pct","unmap_other_pct"]
fastp  = ["fastp_total_reads","fastp_q30_rate","fastp_gc_content","fastp_dup_rate","fastp_insert_peak"]

(pd.DataFrame(df[[c for c in [pseudo, rrna, "fastp_q30_rate","fastp_dup_rate","fastp_gc_content","fastp_insert_peak","uniq_pct"] if c in df.columns]].describe())
 .to_csv(Path(OUTDIR, "summary_stats.csv")))

corr_targets = [c for c in [rrna,"fastp_q30_rate","fastp_dup_rate","fastp_gc_content","fastp_insert_peak","uniq_pct","multi_pct"] if c in df.columns]
if pseudo in df.columns and len(corr_targets):
    cs = df[corr_targets + [pseudo]].corr()[pseudo].drop(pseudo, errors="ignore").dropna()
    cs.to_csv(Path(OUTDIR, "correlations_vs_pseudo.csv"))

pdf = PdfPages(Path(OUTDIR, "qc_plots.pdf"))

# 1) pseudo dist + box
if pseudo in df.columns:
    fig, ax = plt.subplots(figsize=(6,4)); hist(ax, df[pseudo], "Distribution of Kallisto pseudoaligned %", "Pseudoaligned (%)")
    fig.tight_layout(); fig.savefig(Path(OUTDIR, "hist_pseudoaligned.png"), dpi=200); pdf.savefig(fig); plt.close(fig)
    fig, ax = plt.subplots(figsize=(4,4)); box(ax, df[pseudo], "Pseudoaligned % (box)", "Pseudoaligned (%)")
    fig.tight_layout(); fig.savefig(Path(OUTDIR, "box_pseudoaligned.png"), dpi=200); pdf.savefig(fig); plt.close(fig)

# 2) pseudo vs rRNA
if rrna in df.columns and pseudo in df.columns:
    fig, ax = plt.subplots(figsize=(5.5,4))
    scatter(ax, df[rrna], df[pseudo], "Pseudoaligned % vs rRNA overall rate %", "rRNA overall rate (%)", "Pseudoaligned (%)", add_fit=True, annotate=True, samples=df["sample"])
    fig.tight_layout(); fig.savefig(Path(OUTDIR, "scatter_pseudo_vs_rrna.png"), dpi=200); pdf.savefig(fig); plt.close(fig)

# 3) pseudo vs fastp metrics
for col, title, fname in [
    ("fastp_q30_rate", "Pseudoaligned % vs Q30 rate", "scatter_pseudo_vs_q30.png"),
    ("fastp_dup_rate", "Pseudoaligned % vs duplication rate", "scatter_pseudo_vs_dup.png"),
    ("fastp_gc_content", "Pseudoaligned % vs GC content", "scatter_pseudo_vs_gc.png"),
    ("fastp_insert_peak", "Pseudoaligned % vs insert size peak", "scatter_pseudo_vs_insert.png"),
]:
    if col in df.columns and pseudo in df.columns:
        fig, ax = plt.subplots(figsize=(5.5,4))
        scatter(ax, df[col], df[pseudo], title, col, "Pseudoaligned (%)", add_fit=True, annotate=False, samples=df["sample"])
        fig.tight_layout(); fig.savefig(Path(OUTDIR, fname), dpi=200); pdf.savefig(fig); plt.close(fig)

# 4) pseudo vs STAR uniq
if "uniq_pct" in df.columns and pseudo in df.columns:
    fig, ax = plt.subplots(figsize=(5.5,4))
    scatter(ax, df["uniq_pct"], df[pseudo], "Pseudoaligned % vs STAR uniquely mapped %", "STAR uniquely mapped (%)", "Pseudoaligned (%)", add_fit=True, annotate=True, samples=df["sample"])
    fig.tight_layout(); fig.savefig(Path(OUTDIR, "scatter_pseudo_vs_staruniq.png"), dpi=200); pdf.savefig(fig); plt.close(fig)

# 5) stacked STAR components for worst 15 by pseudo
present_star = ensure_cols(df, star)
if pseudo in df.columns and len(present_star) >= 3:
    sub = df.sort_values(by=pseudo, ascending=True).head(min(15, len(df)))[["sample"] + present_star]
    fig, ax = plt.subplots(figsize=(max(8, len(sub)*0.45), 4.5))
    bar_stacked(ax, sub, present_star, "STAR mapping components (worst pseudoaligned 15)")
    fig.tight_layout(); fig.savefig(Path(OUTDIR, "stacked_star_components_worst15.png"), dpi=200); pdf.savefig(fig); plt.close(fig)

# 6) QC metrics heatmap (z-scored)
cols_heat = ensure_cols(df, [pseudo, rrna, "fastp_q30_rate","fastp_dup_rate","fastp_gc_content","fastp_insert_peak","uniq_pct","multi_pct"])
if len(cols_heat) >= 4:
    H = (df[cols_heat] - df[cols_heat].mean())/df[cols_heat].std(ddof=0)
    fig, ax = plt.subplots(figsize=(8, max(4, len(df)*0.18)))
    heatmap(ax, H.values, row_labels=df["sample"].tolist(), col_labels=cols_heat, title="Z-scored QC metrics heatmap")
    fig.tight_layout(); fig.savefig(Path(OUTDIR, "heatmap_qc_metrics.png"), dpi=200); pdf.savefig(fig); plt.close(fig)

# 7) Correlation matrix heatmap
corr_cols = ensure_cols(df, [pseudo, rrna,"fastp_q30_rate","fastp_dup_rate","fastp_gc_content","fastp_insert_peak"] + star)
if len(corr_cols) >= 3:
    C = df[corr_cols].corr()
    fig, ax = plt.subplots(figsize=(8,6))
    heatmap(ax, C.values, row_labels=corr_cols, col_labels=corr_cols, title="Correlation matrix (Pearson)")
    fig.tight_layout(); fig.savefig(Path(OUTDIR, "heatmap_correlations.png"), dpi=200); pdf.savefig(fig); plt.close(fig)

# 8) Scatter-matrix (small multiple)
sm_cols = ensure_cols(df, [pseudo, rrna, "fastp_dup_rate","fastp_insert_peak","uniq_pct"])
if len(sm_cols) >= 3:
    k = len(sm_cols)
    fig, axarr = plt.subplots(k, k, figsize=(2.6*k, 2.6*k))
    scatter_matrix(axarr, df, sm_cols, s=8)
    fig.suptitle("Scatter-matrix of key QC metrics", y=1.02, fontsize=12)
    plt.tight_layout(); fig.savefig(Path(OUTDIR, "scatter_matrix_qc.png"), dpi=200); pdf.savefig(fig); plt.close(fig)

# 9) CDF of pseudoaligned %
if pseudo in df.columns:
    s = pd.to_numeric(df[pseudo], errors="coerce").dropna().sort_values()
    y = np.arange(1, len(s)+1)/len(s)
    fig, ax = plt.subplots(figsize=(6,4))
    ax.plot(s.values, y, drawstyle="steps-post")
    ax.set_xlabel("Pseudoaligned (%)"); ax.set_ylabel("Cumulative fraction"); ax.set_title("CDF of pseudoaligned %")
    fig.tight_layout(); fig.savefig(Path(OUTDIR, "cdf_pseudoaligned.png"), dpi=200); pdf.savefig(fig); plt.close(fig)

# 10) Kallisto processed vs pseudoaligned reads
if {"kallisto_n_processed","kallisto_n_pseudoaligned"}.issubset(df.columns):
    fig, ax = plt.subplots(figsize=(5.5,4))
    scatter(ax, df["kallisto_n_processed"], df["kallisto_n_pseudoaligned"],
            "Kallisto: processed vs pseudoaligned reads", "n_processed", "n_pseudoaligned",
            add_fit=True, annotate=False, samples=df["sample"])
    fig.tight_layout(); fig.savefig(Path(OUTDIR, "scatter_kallisto_counts.png"), dpi=200); pdf.savefig(fig); plt.close(fig)

pdf.close()
print(f"[OK] Plots saved to {OUTDIR} and consolidated PDF qc_plots.pdf")


In [ ]:
# Count duplicates by column "CDR3b" (as-is and with simple normalization)

import pandas as pd

# --- Edit path if needed ---
PATH = r'/content/vdjdb_trait_onlyB (1).csv'  # keep the r'' for spaces/parentheses

df = pd.read_csv(PATH, dtype=str)
col = 'CDR3b'

# --- As-is (exact string) ---
total_rows = len(df)
unique_vals = df[col].nunique(dropna=True)
dup_rows_count = int(df.duplicated(subset=[col], keep=False).sum())           # rows that belong to duplicated CDR3b values
dup_values_count = int((df[col].value_counts(dropna=True) > 1).sum())        # number of distinct CDR3b values that are duplicated

print(f"[AS-IS] total rows: {total_rows}")
print(f"[AS-IS] unique CDR3b: {unique_vals}")
print(f"[AS-IS] rows in duplicated CDR3b groups: {dup_rows_count}")
print(f"[AS-IS] distinct duplicated CDR3b values: {dup_values_count}")

# --- Normalized (uppercase + strip + remove inner spaces) ---
df['CDR3b_norm'] = (
    df[col].astype(str)
           .str.strip()
           .str.upper()
           .str.replace(r'\s+', '', regex=True)
)

unique_vals_norm = df['CDR3b_norm'].nunique(dropna=True)
dup_rows_count_norm = int(df.duplicated(subset=['CDR3b_norm'], keep=False).sum())
dup_values_count_norm = int((df['CDR3b_norm'].value_counts(dropna=True) > 1).sum())

print(f"\n[NORMALIZED] unique CDR3b: {unique_vals_norm}")
print(f"[NORMALIZED] rows in duplicated CDR3b groups: {dup_rows_count_norm}")
print(f"[NORMALIZED] distinct duplicated CDR3b values: {dup_values_count_norm}")

# --- (Optional) show top duplicated values ---
top_dups = (
    df['CDR3b_norm'].value_counts(dropna=True)
      .reset_index()
      .rename(columns={'index': 'CDR3b_norm', 'CDR3b_norm': 'count'})
)
top_dups = top_dups[top_dups['count'] > 1].sort_values('count', ascending=False)
top_dups.head(20)


In [ ]:
# ================================
# Merge TRAIT + VDJdb (slim) by TCRβ CDR3
# and compare Epitope_gene/species vs antigen.gene/species
# ================================

# If files are in Drive, uncomment:
# from google.colab import drive
# drive.mount('/content/drive')

import pandas as pd
import numpy as np
import re

# -------- Edit your paths here --------
TRAIT_XLSX = '20250312-TRAIT_search_download.xlsx'  # your TRAIT export
VDJDB_TSV  = 'vdjdb.slim.txt'                       # your vdjdb slim tsv
# --------------------------------------

# Utils
def find_col(df, options, required=True, label=""):
    for c in options:
        if c in df.columns:
            return c
    if required:
        raise ValueError(
            f"Could not find expected column for {label or options}.\n"
            f"Looked for any of: {options}\n"
            f"Available columns:\n{list(df.columns)}"
        )
    return None

def norm_cdr3(s):
    # Uppercase, strip whitespace, remove inner spaces; keep only letters (A-Z) and '*'
    s = s.astype(str).str.upper().str.strip().str.replace(r"\s+", "", regex=True)
    s = s.str.replace(r"[^A-Z\*]", "", regex=True)
    return s

def norm_text(s):
    # Lowercase, trim, collapse spaces
    s = s.astype(str).fillna("").str.strip().str.replace(r"\s+", " ", regex=True).str.lower()
    return s

# 1) Load data
trait = pd.read_excel(TRAIT_XLSX)
vdj   = pd.read_csv(VDJDB_TSV, sep="\t", dtype=str)

# 2) Pick columns (robust to naming)
trait_cdr3b_col = find_col(
    trait,
    options=['CDR3β','CDR3b','CDR3_beta','CDR3B','CDR3β.aa','CDR3b.aa','CDR3beta'],
    label='TRAIT CDR3β'
)
trait_gene_col  = find_col(trait, options=['Epitope_gene'], label='TRAIT Epitope_gene')
trait_spec_col  = find_col(trait, options=['Epitope_species'], label='TRAIT Epitope_species')

vdj_gene_col    = find_col(vdj,   options=['gene'], label='VDJdb gene')
vdj_cdr3_col    = find_col(vdj,   options=['cdr3'], label='VDJdb cdr3')
vdj_ag_gene_col = find_col(vdj,   options=['antigen.gene'], label='VDJdb antigen.gene')
vdj_ag_spec_col = find_col(vdj,   options=['antigen.species'], label='VDJdb antigen.species')

# 3) Keep TRB only in VDJdb
vdj_trb = vdj[vdj[vdj_gene_col].astype(str).str.upper().str.startswith('TRB')].copy()

# 4) Build trimmed frames
t_keep = trait[[trait_cdr3b_col, trait_gene_col, trait_spec_col]].copy()
t_keep.columns = ['cdr3b', 'Epitope_gene', 'Epitope_species']

v_keep = vdj_trb[[vdj_cdr3_col, vdj_ag_gene_col, vdj_ag_spec_col]].copy()
v_keep.columns = ['cdr3b', 'antigen.gene', 'antigen.species']

# 5) Drop missing CDR3β and normalize
t_keep = t_keep.dropna(subset=['cdr3b'])
v_keep = v_keep.dropna(subset=['cdr3b'])

t_keep['cdr3b_norm'] = norm_cdr3(t_keep['cdr3b'])
v_keep['cdr3b_norm'] = norm_cdr3(v_keep['cdr3b'])

t_keep['Epitope_gene_norm']    = norm_text(t_keep['Epitope_gene'])
t_keep['Epitope_species_norm'] = norm_text(t_keep['Epitope_species'])

v_keep['antigen.gene_norm']    = norm_text(v_keep['antigen.gene'])
v_keep['antigen.species_norm'] = norm_text(v_keep['antigen.species'])

# Optional de-dup to reduce many-to-many blowups:
t_keep = t_keep.drop_duplicates(subset=['cdr3b_norm', 'Epitope_gene_norm', 'Epitope_species_norm'])
v_keep = v_keep.drop_duplicates(subset=['cdr3b_norm', 'antigen.gene_norm', 'antigen.species_norm'])

# 6) Merge on normalized β CDR3
merged = pd.merge(
    t_keep, v_keep,
    on='cdr3b_norm', how='inner', suffixes=('_trait', '_vdjdb')
)

# 7) Compare Epitope_gene/species vs antigen.gene/species
merged['gene_match']    = (merged['Epitope_gene_norm']    == merged['antigen.gene_norm'])
merged['species_match'] = (merged['Epitope_species_norm'] == merged['antigen.species_norm'])
merged['both_match']    = merged['gene_match'] & merged['species_match']

# 8) Report
n = len(merged)
n_gene    = int(merged['gene_match'].sum())
n_species = int(merged['species_match'].sum())
n_both    = int(merged['both_match'].sum())

print(f"Total overlaps on CDR3β: {n}")
print(f"Epitope_gene matches (TRAIT vs VDJdb): {n_gene}/{n} ({(100*n_gene/n if n else 0):.1f}%)")
print(f"Epitope_species matches (TRAIT vs VDJdb): {n_species}/{n} ({(100*n_species/n if n else 0):.1f}%)")
print(f"Both gene & species match: {n_both}/{n} ({(100*n_both/n if n else 0):.1f}%)")

# 9) Inspect mismatches quickly
print("\n--- Example gene mismatches (up to 10) ---")
display(merged[~merged['gene_match']].head(10)[
    ['cdr3b_trait','cdr3b_vdjdb','Epitope_gene','antigen.gene']
])

print("\n--- Example species mismatches (up to 10) ---")
display(merged[~merged['species_match']].head(10)[
    ['cdr3b_trait','cdr3b_vdjdb','Epitope_species','antigen.species']
])

# 10) Save outputs
# Compact human review table
out_review = merged[[
    'cdr3b_trait','cdr3b_vdjdb',
    'Epitope_gene','antigen.gene','gene_match',
    'Epitope_species','antigen.species','species_match',
    'both_match'
]].copy()

out_review.to_csv('/content/merged_TRAIT_VDJdb_by_CDR3b_review.csv', index=False)
merged.to_csv('/content/merged_TRAIT_VDJdb_by_CDR3b_full.csv', index=False)

print("\nSaved:")
print(" - /content/merged_TRAIT_VDJdb_by_CDR3b_review.csv")
print(" - /content/merged_TRAIT_VDJdb_by_CDR3b_full.csv")


In [ ]:
# =========================================
# Merge TRAIT + VDJdb (slim) + McPAS by TCRβ CDR3
# Keep only: CDR3β, Epitope_gene, Epitope_species
# McPAS mapping per your decision:
#   Epitope_gene    <- Antigen.protein
#   Epitope_species <- Pathology   (note: disease context)
# =========================================

# If files are on Drive:
# from google.colab import drive
# drive.mount('/content/drive')

import pandas as pd
import numpy as np
import re

# --------- Edit your paths here ----------
TRAIT_XLSX = '20250312-TRAIT_search_download.xlsx'
VDJDB_TSV  = 'vdjdb.slim.txt'
MCPAS_CSV  = 'McPAS-TCR.csv'
# -----------------------------------------

# ---------- Helpers ----------
def find_col(df, options, required=True, label=""):
    for c in options:
        if c in df.columns:
            return c
    if required:
        raise ValueError(
            f"Missing expected column for {label or options}. "
            f"Tried: {options}\nAvailable: {list(df.columns)}"
        )
    return None

def norm_cdr3(s):
    # Uppercase AA, trim, remove spaces, keep A-Z and '*'
    s = s.astype(str).str.upper().str.strip().str.replace(r"\s+", "", regex=True)
    s = s.str.replace(r"[^A-Z\*]", "", regex=True)
    return s

def norm_text(s):
    # Lowercase text normalization, collapse spaces
    return (s.astype(str).fillna("")
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
            .str.lower())

def build_frame(df, src, cdr3_col, gene_col, species_col):
    out = df[[cdr3_col, gene_col, species_col]].copy()
    out.columns = ['cdr3b_raw', 'Epitope_gene_raw', 'Epitope_species_raw']
    out = out.dropna(subset=['cdr3b_raw'])
    out['cdr3b_norm']          = norm_cdr3(out['cdr3b_raw'])
    out['Epitope_gene_norm']   = norm_text(out['Epitope_gene_raw'])
    out['Epitope_species_norm']= norm_text(out['Epitope_species_raw'])
    out['source'] = src
    # De-dup to reduce many-to-many explosions
    out = out.drop_duplicates(subset=['cdr3b_norm','Epitope_gene_norm','Epitope_species_norm'])
    return out

def safe_eq(a, b):
    # equality that ignores NaNs (counts only when both present)
    a = a.fillna("")
    b = b.fillna("")
    return (a == b) & (a != "") & (b != "")

def choose_consensus(row, cols_in_priority):
    for c in cols_in_priority:
        val = row.get(c, None)
        if pd.notna(val) and str(val).strip() != "":
            return val
    return np.nan

# ---------- Load files ----------
trait = pd.read_excel(TRAIT_XLSX, dtype=str)
vdj   = pd.read_csv(VDJDB_TSV, sep='\t', dtype=str)
mcpas = pd.read_csv(MCPAS_CSV, dtype=str, low_memory=False)

# ---------- Pick columns (robust names) ----------

# TRAIT: expect beta CDR3 and epitope fields
trait_cdr3b_col = find_col(trait, ['CDR3β','CDR3b','CDR3_beta','CDR3B','CDR3β.aa','CDR3b.aa','CDR3beta'], label='TRAIT CDR3β')
trait_gene_col  = find_col(trait, ['Epitope_gene'], label='TRAIT Epitope_gene')
trait_spec_col  = find_col(trait, ['Epitope_species'], label='TRAIT Epitope_species')

# VDJdb slim: gene (TRA/TRB), cdr3, antigen.gene/species
vdj_gene_col    = find_col(vdj,   ['gene'], label='VDJdb gene')
vdj_cdr3_col    = find_col(vdj,   ['cdr3'], label='VDJdb cdr3')
vdj_ag_gene_col = find_col(vdj,   ['antigen.gene'], label='VDJdb antigen.gene')
vdj_ag_spec_col = find_col(vdj,   ['antigen.species'], label='VDJdb antigen.species')

# McPAS: beta AA cdr3 + Antigen.protein + Pathology
mcpas_cdr3b_col = find_col(mcpas, ['CDR3.beta.aa'], label='McPAS CDR3.beta.aa')
mcpas_gene_col  = find_col(mcpas, ['Antigen.protein'], label='McPAS Antigen.protein')
mcpas_spec_col  = find_col(mcpas, ['Pathology'], label='McPAS Pathology')

# ---------- Filter to β only and build simplified frames ----------
# TRAIT: assumed β already in selected column
t_df = build_frame(trait, 'TRAIT', trait_cdr3b_col, trait_gene_col, trait_spec_col)

# VDJdb: keep only TRB
vdj_trb = vdj[vdj[vdj_gene_col].astype(str).str.upper().str.startswith('TRB')].copy()
v_df = build_frame(vdj_trb, 'VDJdb', vdj_cdr3_col, vdj_ag_gene_col, vdj_ag_spec_col)

# McPAS: keep rows where β exists
m_df = build_frame(mcpas, 'McPAS', mcpas_cdr3b_col, mcpas_gene_col, mcpas_spec_col)

# ---------- Merge all three (outer on normalized CDR3β) ----------
# Prepare suffixed versions to avoid column collisions
t_df = t_df.add_suffix('_trait'); t_df = t_df.rename(columns={'cdr3b_norm_trait':'cdr3b_norm'})
v_df = v_df.add_suffix('_vdjdb'); v_df = v_df.rename(columns={'cdr3b_norm_vdjdb':'cdr3b_norm'})
m_df = m_df.add_suffix('_mcpas'); m_df = m_df.rename(columns={'cdr3b_norm_mcpas':'cdr3b_norm'})

merged = t_df.merge(v_df, on='cdr3b_norm', how='outer').merge(m_df, on='cdr3b_norm', how='outer')

# ---------- Pairwise match flags (normalized) ----------
# Gene matches
merged['match_gene_TRAIT_VDJdb']  = safe_eq(merged['Epitope_gene_norm_trait'],   merged['Epitope_gene_norm_vdjdb'])
merged['match_gene_TRAIT_McPAS']  = safe_eq(merged['Epitope_gene_norm_trait'],   merged['Epitope_gene_norm_mcpas'])
merged['match_gene_VDJdb_McPAS']  = safe_eq(merged['Epitope_gene_norm_vdjdb'],   merged['Epitope_gene_norm_mcpas'])
# Species matches
merged['match_spec_TRAIT_VDJdb']  = safe_eq(merged['Epitope_species_norm_trait'], merged['Epitope_species_norm_vdjdb'])
merged['match_spec_TRAIT_McPAS']  = safe_eq(merged['Epitope_species_norm_trait'], merged['Epitope_species_norm_mcpas'])
merged['match_spec_VDJdb_McPAS']  = safe_eq(merged['Epitope_species_norm_vdjdb'], merged['Epitope_species_norm_mcpas'])

# ---------- Presence indicators ----------
merged['has_TRAIT'] = merged['cdr3b_raw_trait'].notna()
merged['has_VDJdb'] = merged['cdr3b_raw_vdjdb'].notna()
merged['has_McPAS'] = merged['cdr3b_raw_mcpas'].notna()

# ---------- Simple consensus (priority: TRAIT -> VDJdb -> McPAS) ----------
merged['CDR3b_any'] = merged[['cdr3b_raw_trait','cdr3b_raw_vdjdb','cdr3b_raw_mcpas']].bfill(axis=1).iloc[:,0]

merged['Epitope_gene_consensus'] = merged.apply(
    lambda r: choose_consensus(r, ['Epitope_gene_raw_trait','Epitope_gene_raw_vdjdb','Epitope_gene_raw_mcpas']),
    axis=1
)
merged['Epitope_species_consensus'] = merged.apply(
    lambda r: choose_consensus(r, ['Epitope_species_raw_trait','Epitope_species_raw_vdjdb','Epitope_species_raw_mcpas']),
    axis=1
)

# ---------- Quick summaries ----------
n_all = len(merged)
n_trait = int(merged['has_TRAIT'].sum())
n_vdjdb = int(merged['has_VDJdb'].sum())
n_mcpas = int(merged['has_McPAS'].sum())
n_all_three = int((merged['has_TRAIT'] & merged['has_VDJdb'] & merged['has_McPAS']).sum())

print(f"Total unique CDR3β (union): {n_all}")
print(f"Present in TRAIT: {n_trait} | VDJdb: {n_vdjdb} | McPAS: {n_mcpas}")
print(f"Present in all three: {n_all_three}")

# Where at least two sources agree (gene/species)
merged['gene_any_two_agree'] = (
    merged['match_gene_TRAIT_VDJdb'] |
    merged['match_gene_TRAIT_McPAS'] |
    merged['match_gene_VDJdb_McPAS']
)
merged['spec_any_two_agree'] = (
    merged['match_spec_TRAIT_VDJdb'] |
    merged['match_spec_TRAIT_McPAS'] |
    merged['match_spec_VDJdb_McPAS']
)

print(f"Gene: any-two-agree rows: {int(merged['gene_any_two_agree'].sum())} / {n_all}")
print(f"Species: any-two-agree rows: {int(merged['spec_any_two_agree'].sum())} / {n_all}")

# ---------- Save outputs ----------
# Full merged with all columns
merged.to_csv('/content/merged_TRAIT_VDJdb_McPAS_full.csv', index=False)

# Compact review: one row per CDR3β with source values + matches + consensus
review_cols = [
    'cdr3b_norm','CDR3b_any',
    'cdr3b_raw_trait','Epitope_gene_raw_trait','Epitope_species_raw_trait',
    'cdr3b_raw_vdjdb','Epitope_gene_raw_vdjdb','Epitope_species_raw_vdjdb',
    'cdr3b_raw_mcpas','Epitope_gene_raw_mcpas','Epitope_species_raw_mcpas',
    'match_gene_TRAIT_VDJdb','match_gene_TRAIT_McPAS','match_gene_VDJdb_McPAS',
    'match_spec_TRAIT_VDJdb','match_spec_TRAIT_McPAS','match_spec_VDJdb_McPAS',
    'Epitope_gene_consensus','Epitope_species_consensus',
    'has_TRAIT','has_VDJdb','has_McPAS'
]
merged[review_cols].to_csv('/content/merged_TRAIT_VDJdb_McPAS_review.csv', index=False)

# Focused mismatches to inspect quickly
mism_gene = merged[~merged['gene_any_two_agree']]
mism_spec = merged[~merged['spec_any_two_agree']]
mism_gene.to_csv('/content/mismatches_gene_only.csv', index=False)
mism_spec.to_csv('/content/mismatches_species_only.csv', index=False)

print("\nSaved:")
print(" - /content/merged_TRAIT_VDJdb_McPAS_full.csv")
print(" - /content/merged_TRAIT_VDJdb_McPAS_review.csv")
print(" - /content/mismatches_gene_only.csv")
print(" - /content/mismatches_species_only.csv")


In [ ]:
mism_gene

In [ ]:
# =========================================================
# Merge TRAIT + VDJdb (slim) + McPAS by TCRβ CDR3 (union)
# Output columns:
#   cdrsb,
#   Epitope_gene_trait, Epitope_gene_vdjdb, Epitope_gene_McPAS,
#   Epitope_species_trait, Epitope_species_vdjdb, Epitope_species_McPAS
#
# Notes:
# - VDJdb: filter to TRB only (gene startswith 'TRB')
# - McPAS: Epitope_gene <- Antigen.protein; Epitope_species <- Pathology (per your choice)
# - Aggregates multiple values per CDR3β using " | " (unique, case-insensitive)
# =========================================================

# If files are in Drive, uncomment:
# from google.colab import drive
# drive.mount('/content/drive')

import pandas as pd
import numpy as np
import re

# -------- Edit your paths here --------
TRAIT_XLSX = '/content/20250312-TRAIT_search_download.xlsx'
VDJDB_TSV  = '/content/vdjdb.slim.txt'
MCPAS_CSV  = '/content/McPAS-TCR.csv'
# --------------------------------------

def find_col(df, options, required=True, label=""):
    for c in options:
        if c in df.columns:
            return c
    if required:
        raise ValueError(
            f"Missing expected column for {label or options}. "
            f"Tried: {options}\nAvailable: {list(df.columns)}"
        )
    return None

def norm_cdr3(s):
    # Uppercase AA, trim, remove spaces, keep only letters and '*'
    s = s.astype(str).str.upper().str.strip().str.replace(r"\s+", "", regex=True)
    s = s.str.replace(r"[^A-Z\*]", "", regex=True)
    return s

def uniq_join(values, sep=" | "):
    # Join unique (case-insensitive) non-empty strings, keep original case of first occurrence
    seen = {}
    for v in values:
        if v is None:
            continue
        vstr = str(v).strip()
        if not vstr:
            continue
        key = vstr.lower()
        if key not in seen:
            seen[key] = vstr
    if not seen:
        return np.nan
    return sep.join(seen.values())

def build_beta_frame(df, src_name, cdr3_col, gene_col, species_col):
    out = df[[cdr3_col, gene_col, species_col]].copy()
    out.columns = ['cdr3b_raw', 'Epitope_gene_raw', 'Epitope_species_raw']
    out = out.dropna(subset=['cdr3b_raw'])
    out['cdr3b_norm']           = norm_cdr3(out['cdr3b_raw'])
    # Group to one row per normalized CDR3β; aggregate values per dataset
    agg = (out
           .groupby('cdr3b_norm', as_index=False)
           .agg(cdr3b_example=('cdr3b_raw', 'first'),
                Epitope_gene_col=('Epitope_gene_raw', uniq_join),
                Epitope_species_col=('Epitope_species_raw', uniq_join)))
    # Tag columns with source
    agg = agg.rename(columns={
        'cdr3b_example': f'cdr3b_example_{src_name}',
        'Epitope_gene_col': f'Epitope_gene_{src_name}',
        'Epitope_species_col': f'Epitope_species_{src_name}',
    })
    return agg

# -------- Load files --------
trait = pd.read_excel(TRAIT_XLSX, dtype=str)
vdj   = pd.read_csv(VDJDB_TSV, sep='\t', dtype=str)
mcpas = pd.read_csv(MCPAS_CSV, dtype=str, low_memory=False)

# -------- Pick columns (robust to naming) --------
# TRAIT: expect beta CDR3 and epitope fields
trait_cdr3b_col = find_col(trait, ['CDR3β','CDR3b','CDR3_beta','CDR3B','CDR3β.aa','CDR3b.aa','CDR3beta'], label='TRAIT CDR3β')
trait_gene_col  = find_col(trait, ['Epitope_gene'], label='TRAIT Epitope_gene')
trait_spec_col  = find_col(trait, ['Epitope_species'], label='TRAIT Epitope_species')

# VDJdb slim
vdj_gene_col    = find_col(vdj,   ['gene'], label='VDJdb gene')
vdj_cdr3_col    = find_col(vdj,   ['cdr3'], label='VDJdb cdr3')
vdj_ag_gene_col = find_col(vdj,   ['antigen.gene'], label='VDJdb antigen.gene')
vdj_ag_spec_col = find_col(vdj,   ['antigen.species'], label='VDJdb antigen.species')

# McPAS
mcpas_cdr3b_col = find_col(mcpas, ['CDR3.beta.aa'], label='McPAS CDR3.beta.aa')
mcpas_gene_col  = find_col(mcpas, ['Antigen.protein'], label='McPAS Antigen.protein')
mcpas_spec_col  = find_col(mcpas, ['Pathology'], label='McPAS Pathology')

# -------- Build per-dataset beta frames --------
# TRAIT (already β)
t_df = build_beta_frame(trait, 'trait', trait_cdr3b_col, trait_gene_col, trait_spec_col)

# VDJdb: keep TRB only
vdj_trb = vdj[vdj[vdj_gene_col].astype(str).str.upper().str.startswith('TRB')].copy()
v_df = build_beta_frame(vdj_trb, 'vdjdb', vdj_cdr3_col, vdj_ag_gene_col, vdj_ag_spec_col)

# McPAS: map as requested (Epitope_gene <- Antigen.protein, Epitope_species <- Pathology)
m_df = build_beta_frame(mcpas, 'McPAS', mcpas_cdr3b_col, mcpas_gene_col, mcpas_spec_col)

# -------- Outer merge on normalized CDR3β (union) --------
merged = t_df.merge(v_df, on='cdr3b_norm', how='outer').merge(m_df, on='cdr3b_norm', how='outer')

# Choose a representative raw CDR3β to show (take first available)
merged['cdrsb'] = merged[['cdr3b_example_trait','cdr3b_example_vdjdb','cdr3b_example_McPAS']].bfill(axis=1).iloc[:,0]

# -------- Final column order --------
final_cols = [
    'cdrsb',
    'Epitope_gene_trait', 'Epitope_gene_vdjdb', 'Epitope_gene_McPAS',
    'Epitope_species_trait', 'Epitope_species_vdjdb', 'Epitope_species_McPAS'
]
final = merged[['cdrsb',
                'Epitope_gene_trait','Epitope_gene_vdjdb','Epitope_gene_McPAS',
                'Epitope_species_trait','Epitope_species_vdjdb','Epitope_species_McPAS']].copy()

# Optional: sort by cdrsb
final = final.sort_values('cdrsb', kind='stable').reset_index(drop=True)

# -------- Save --------
final.to_csv('/content/tcrb_union_TRAIT_VDJdb_McPAS.csv', index=False)
print("Saved: /content/tcrb_union_TRAIT_VDJdb_McPAS.csv")
print(f"Rows (unique CDR3β union): {len(final)}")
print(final.head(10))


In [ ]:
# ============================================
# Fix for KeyError('value') + robust lists
# Works on: /content/tcrb_union_TRAIT_VDJdb_McPAS.csv
# Outputs:
#  - epitope_genes_per_source.csv
#  - epitope_species_per_source.csv
#  - epitope_genes_all_unique.csv
#  - epitope_species_all_unique.csv
#  - epitope_genes_loose_clusters.csv
#  - epitope_species_loose_clusters.csv
# ============================================

import pandas as pd
import numpy as np
import re

PATH = '/content/tcrb_union_TRAIT_VDJdb_McPAS.csv'
df = pd.read_csv(PATH, dtype=str)

required = [
    'cdrsb',
    'Epitope_gene_trait','Epitope_gene_vdjdb','Epitope_gene_McPAS',
    'Epitope_species_trait','Epitope_species_vdjdb','Epitope_species_McPAS'
]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing expected columns: {missing}")

# ---------- Helpers ----------
def value_counts_df(series: pd.Series, source_name: str) -> pd.DataFrame:
    s = series.dropna().astype(str).str.strip()
    s = s[s != '']
    vc = s.value_counts(dropna=False).reset_index(name='count')
    vc.columns = ['value', 'count']
    vc['source'] = source_name
    return vc

def norm_loose(s: pd.Series) -> pd.Series:
    # Lowercase, trim, collapse spaces; remove common separators to cluster variants
    s = s.fillna('').astype(str).str.lower().str.strip()
    s = s.str.replace(r'\s+', ' ', regex=True)
    s = s.str.replace(r'[-_\/\.\:\;]+', '', regex=True)
    return s

def cluster_variants(series: pd.Series, source_name: str) -> pd.DataFrame:
    raw = series.dropna().astype(str).str.strip()
    raw = raw[raw != '']
    tmp = pd.DataFrame({'raw': raw, 'loose': norm_loose(raw)})
    grp = (tmp.groupby('loose', dropna=False)
              .agg(total_count=('raw','size'),
                   variants=('raw', lambda x: sorted(set(x))))
              .reset_index())
    grp['n_variants'] = grp['variants'].apply(len)
    grp['source'] = source_name
    # keep only clusters with >1 variant (likely standardization targets)
    return grp[grp['n_variants'] > 1].sort_values(
        ['n_variants','total_count'], ascending=[False, False]
    )

# ---------- Per-source lists + counts ----------
genes_trait = value_counts_df(df['Epitope_gene_trait'], 'trait')
genes_vdjdb = value_counts_df(df['Epitope_gene_vdjdb'], 'vdjdb')
genes_mcpas = value_counts_df(df['Epitope_gene_McPAS'], 'McPAS')
genes_all = pd.concat([genes_trait, genes_vdjdb, genes_mcpas], ignore_index=True)

spec_trait = value_counts_df(df['Epitope_species_trait'], 'trait')
spec_vdjdb = value_counts_df(df['Epitope_species_vdjdb'], 'vdjdb')
spec_mcpas = value_counts_df(df['Epitope_species_McPAS'], 'McPAS')
spec_all = pd.concat([spec_trait, spec_vdjdb, spec_mcpas], ignore_index=True)

# ---------- Combined unique lists ----------
genes_union = pd.DataFrame(sorted(genes_all['value'].unique()), columns=['Epitope_gene_all_unique'])
spec_union  = pd.DataFrame(sorted(spec_all['value'].unique()),  columns=['Epitope_species_all_unique'])

# ---------- Save per-source + combined ----------
genes_all[['source','value','count']].to_csv('/content/epitope_genes_per_source.csv', index=False)
spec_all [['source','value','count']].to_csv('/content/epitope_species_per_source.csv', index=False)
genes_union.to_csv('/content/epitope_genes_all_unique.csv', index=False)
spec_union.to_csv('/content/epitope_species_all_unique.csv', index=False)

print("Saved:")
print(" - /content/epitope_genes_per_source.csv")
print(" - /content/epitope_species_per_source.csv")
print(" - /content/epitope_genes_all_unique.csv")
print(" - /content/epitope_species_all_unique.csv")

# ---------- “Loose” clusters to flag variants (e.g., e1 vs e-1) ----------
genes_loose = pd.concat([
    cluster_variants(df['Epitope_gene_trait'], 'trait'),
    cluster_variants(df['Epitope_gene_vdjdb'], 'vdjdb'),
    cluster_variants(df['Epitope_gene_McPAS'], 'McPAS')
], ignore_index=True)

spec_loose = pd.concat([
    cluster_variants(df['Epitope_species_trait'], 'trait'),
    cluster_variants(df['Epitope_species_vdjdb'], 'vdjdb'),
    cluster_variants(df['Epitope_species_McPAS'], 'McPAS')
], ignore_index=True)

genes_loose.to_csv('/content/epitope_genes_loose_clusters.csv', index=False)
spec_loose.to_csv('/content/epitope_species_loose_clusters.csv', index=False)

print("Saved loose clusters:")
print(" - /content/epitope_genes_loose_clusters.csv")
print(" - /content/epitope_species_loose_clusters.csv")

# Quick peek
print("\nTop 10 gene clusters with multiple variants:")
display(genes_loose.head(10))
print("\nTop 10 species clusters with multiple variants:")
display(spec_loose.head(10))


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
from matplotlib.patches import FancyArrowPatch

# ==========================================
# 1. הגדרות עיצוב ל-Nature
# ==========================================
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial']
plt.rcParams['font.size'] = 12
plt.rcParams['axes.linewidth'] = 1
plt.rcParams['xtick.major.width'] = 1
plt.rcParams['ytick.major.width'] = 1

def plot_schematic(ax, shift_type='young'):
    # יצירת דאטה דמה
    x = np.linspace(0, 100, 500)

    # 1. התפלגות הרקע (Global Distribution) - קבועה
    # נניח ממוצע גיל 50 בערך לאוכלוסייה הכללית
    mu_global = 50
    sigma_global = 15
    y_global = stats.norm.pdf(x, mu_global, sigma_global)

    # 2. התפלגות ה-TCR הספציפי
    if shift_type == 'young':
        mu_tcr = 30 # TCR שמופיע אצל צעירים
        color_tcr = '#0072B2' # כחול
        label_tcr = 'Young-associated TCR'
        score_text = 'Negative Score (-)'
    else:
        mu_tcr = 70 # TCR שמופיע אצל מבוגרים
        color_tcr = '#D55E00' # אדום/כתום
        label_tcr = 'Old-associated TCR'
        score_text = 'Positive Score (+)'

    sigma_tcr = 10 # בדרך כלל התפלגות ספציפית היא צרה יותר, אבל לא חייב
    y_tcr = stats.norm.pdf(x, mu_tcr, sigma_tcr)

    # ==========================================
    # ציור הגרפים
    # ==========================================

    # א. התפלגות הרקע (אפור)
    ax.fill_between(x, y_global, color='grey', alpha=0.2)
    ax.plot(x, y_global, color='grey', lw=2, linestyle='--', label='Global Distribution')

    # ב. התפלגות ה-TCR (צבעוני)
    ax.fill_between(x, y_tcr, color=color_tcr, alpha=0.3)
    ax.plot(x, y_tcr, color=color_tcr, lw=2.5, label=label_tcr)

    # ג. החץ שמסמל את ה-Wasserstein Distance (המרחק בין הממוצעים)
    # אנו מציירים חץ בין השיאים של העקומות
    arrow = FancyArrowPatch((mu_global, max(y_global)*0.6), (mu_tcr, max(y_global)*0.6),
                            arrowstyle='<|-|>', mutation_scale=20, color='black', lw=1.5)
    ax.add_patch(arrow)

    # כיתוב מעל החץ
    ax.text((mu_global + mu_tcr)/2, max(y_global)*0.65, 'Wasserstein\nDistance',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

    # ד. התוצאה (Score)
    ax.text((mu_global + mu_tcr)/2, max(y_global)*0.2, score_text,
            ha='center', va='center', fontsize=11, fontweight='bold',
            bbox=dict(facecolor='white', edgecolor=color_tcr, boxstyle='round,pad=0.5'))

    # עיצוב צירים
    ax.set_xlabel('Age (years)')
    ax.set_ylabel('Density / Frequency')
    ax.set_yticks([]) # לא צריך מספרים בציר ה-Y בסכמה
    ax.set_xlim(0, 100)
    ax.set_ylim(0, max(y_tcr)*1.2)

    # הסרת מסגרות מיותרות
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)

    ax.legend(frameon=False, loc='upper right', fontsize=9)

# ==========================================
# יצירת הפיגר
# ==========================================
fig, ax = plt.subplots(figsize=(6, 4))
# אתה יכול לשנות ל-'old' כדי לראות את הדוגמה השנייה
plot_schematic(ax, shift_type='young')

plt.tight_layout()
plt.savefig('Wasserstein_Schematic.png', dpi=300)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
from matplotlib.patches import FancyArrowPatch

# הגדרות עיצוב ל-Nature
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans'] # Added DejaVu as fallback
plt.rcParams['font.size'] = 12
plt.rcParams['axes.linewidth'] = 1
plt.rcParams['xtick.major.width'] = 1
plt.rcParams['ytick.major.width'] = 1

def plot_schematic(ax, shift_type='young'):
    # יצירת דאטה דמה
    x = np.linspace(0, 100, 500)

    # 1. התפלגות הרקע (Global Distribution)
    mu_global = 50
    sigma_global = 15
    y_global = stats.norm.pdf(x, mu_global, sigma_global)

    # 2. התפלגות ה-TCR הספציפי
    if shift_type == 'young':
        mu_tcr = 30
        color_tcr = '#0072B2' # כחול
        label_tcr = 'Young-associated TCR'
        score_text = 'Negative Score (-)'
    else:
        mu_tcr = 70
        color_tcr = '#D55E00' # אדום
        label_tcr = 'Old-associated TCR'
        score_text = 'Positive Score (+)'

    sigma_tcr = 10
    y_tcr = stats.norm.pdf(x, mu_tcr, sigma_tcr)

    # ציור הגרפים
    ax.fill_between(x, y_global, color='grey', alpha=0.2)
    ax.plot(x, y_global, color='grey', lw=2, linestyle='--', label='Global Distribution')

    ax.fill_between(x, y_tcr, color=color_tcr, alpha=0.3)
    ax.plot(x, y_tcr, color=color_tcr, lw=2.5, label=label_tcr)

    # החץ שמסמל את המרחק
    arrow = FancyArrowPatch((mu_global, max(y_global)*0.6), (mu_tcr, max(y_global)*0.6),
                            arrowstyle='<|-|>', mutation_scale=20, color='black', lw=1.5)
    ax.add_patch(arrow)

    # כיתוב מעל החץ
    ax.text((mu_global + mu_tcr)/2, max(y_global)*0.65, 'Wasserstein\nDistance',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

    # התוצאה (Score)
    ax.text((mu_global + mu_tcr)/2, max(y_global)*0.2, score_text,
            ha='center', va='center', fontsize=11, fontweight='bold',
            bbox=dict(facecolor='white', edgecolor=color_tcr, boxstyle='round,pad=0.5'))

    # עיצוב צירים
    ax.set_xlabel('Age (years)')
    ax.set_ylabel('Density / Frequency')
    ax.set_yticks([])
    ax.set_xlim(0, 100)
    ax.set_ylim(0, max(y_tcr)*1.2)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)

    ax.legend(frameon=False, loc='upper right', fontsize=9)

# יצירת הפיגר
fig, ax = plt.subplots(figsize=(6, 4), dpi=150)
plot_schematic(ax, shift_type='young')
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
from matplotlib.patches import FancyArrowPatch

# הגדרות עיצוב ל-Nature
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans'] # Added DejaVu as fallback
plt.rcParams['font.size'] = 12
plt.rcParams['axes.linewidth'] = 1
plt.rcParams['xtick.major.width'] = 1
plt.rcParams['ytick.major.width'] = 1

def plot_schematic(ax, shift_type='young'):
    # יצירת דאטה דמה
    x = np.linspace(0, 100, 500)

    # 1. התפלגות הרקע (Global Distribution)
    mu_global = 50
    sigma_global = 15
    y_global = stats.norm.pdf(x, mu_global, sigma_global)

    # 2. התפלגות ה-TCR הספציפי
    if shift_type == 'young':
        mu_tcr = 30
        color_tcr = '#0072B2' # כחול
        label_tcr = 'Young-associated TCR'
        score_text = 'Negative Score (-)'
    else:
        mu_tcr = 70
        color_tcr = '#D55E00' # אדום
        label_tcr = 'Old-associated TCR'
        score_text = 'Positive Score (+)'

    sigma_tcr = 10
    y_tcr = stats.norm.pdf(x, mu_tcr, sigma_tcr)

    # ציור הגרפים
    ax.fill_between(x, y_global, color='grey', alpha=0.2)
    ax.plot(x, y_global, color='grey', lw=2, linestyle='--', label='Global Distribution')

    ax.fill_between(x, y_tcr, color=color_tcr, alpha=0.3)
    ax.plot(x, y_tcr, color=color_tcr, lw=2.5, label=label_tcr)

    # החץ שמסמל את המרחק
    arrow = FancyArrowPatch((mu_global, max(y_global)*0.6), (mu_tcr, max(y_global)*0.6),
                            arrowstyle='<|-|>', mutation_scale=20, color='black', lw=1.5)
    ax.add_patch(arrow)

    # כיתוב מעל החץ
    ax.text((mu_global + mu_tcr)/2, max(y_global)*0.65, 'Wasserstein\nDistance',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

    # התוצאה (Score)
    ax.text((mu_global + mu_tcr)/2, max(y_global)*0.2, score_text,
            ha='center', va='center', fontsize=11, fontweight='bold',
            bbox=dict(facecolor='white', edgecolor=color_tcr, boxstyle='round,pad=0.5'))

    # עיצוב צירים
    ax.set_xlabel('Age (years)')
    ax.set_ylabel('Density / Frequency')
    ax.set_yticks([])
    ax.set_xlim(0, 100)
    ax.set_ylim(0, max(y_tcr)*1.2)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['left'].set_visible(False)

    ax.legend(frameon=False, loc='upper right', fontsize=9)

# יצירת הפיגר
fig, ax = plt.subplots(figsize=(6, 4), dpi=150)
plot_schematic(ax, shift_type='')
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Rectangle, FancyArrowPatch
import matplotlib.patheffects as path_effects

# ==========================================
# 1. הגדרות עיצוב (Nature Style)
# ==========================================
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 10
plt.rcParams['axes.linewidth'] = 0.8

def plot_bifurcation_schematic(ax):
    # --- חלק א: ה-Input (רצף ומטריצה) ---
    # נצייר סתם רצף דמה
    sequence = "C A S S L G Q A Y E Q Y F"
    aa_list = sequence.split()

    # מיקום התחלתי בצד שמאל
    start_x = 0.05
    start_y = 0.6

    # כיתוב הרצף
    for i, char in enumerate(aa_list[:6]): # רק ה-6 הראשונים לדוגמה
        ax.text(start_x + i*0.04, start_y + 0.15, char,
                fontsize=9, fontweight='bold', ha='center', color='#333')

    # ציור "מטריצת One-Hot" סכמטית מתחת לרצף
    # זה מראה שמדובר בקידוד מתמטי של הרצף
    np.random.seed(42)
    grid_w = 0.04
    grid_h = 0.02
    for col in range(6):
        for row in range(5): # 5 שורות דמה
            is_active = np.random.random() > 0.7 # חלק מהריבועים צבועים
            color = '#555' if is_active else '#eee'
            rect = Rectangle((start_x + col*grid_w - grid_w/2, start_y - row*grid_h),
                             grid_w*0.9, grid_h*0.9, facecolor=color, edgecolor='none')
            ax.add_patch(rect)

    ax.text(start_x + 2.5*grid_w, start_y - 6*grid_h, "Amino-Acid\nEncoding",
            fontsize=7, ha='center', va='top', color='#555')

    # --- חלק ב: החץ (Classifier) ---
    arrow = FancyArrowPatch((0.35, 0.5), (0.55, 0.5),
                            arrowstyle='-|>', mutation_scale=20, color='black', lw=1.5)
    ax.add_patch(arrow)
    ax.text(0.45, 0.52, "Sequence\nClassifier", ha='center', va='bottom', fontsize=8)

    # --- חלק ג: ה-Bifurcation (העננים) ---
    # יצירת דאטה מלאכותי לשני הקלאסטרים
    np.random.seed(10)

    # קלאסטר צעירים (כחול) - מרוכז ורחוק שמאלה
    n_points = 60
    young_x = np.random.normal(0.65, 0.03, n_points)
    young_y = np.random.normal(0.4, 0.08, n_points)

    # קלאסטר מבוגרים (אדום) - קצת יותר מפוזר, ימינה ולמטה
    old_x = np.random.normal(0.85, 0.04, n_points)
    old_y = np.random.normal(0.6, 0.08, n_points)

    # ציור הנקודות
    ax.scatter(young_x, young_y, c='#0072B2', s=15, alpha=0.7, edgecolors='none', label='Young Motifs')
    ax.scatter(old_x, old_y, c='#D55E00', s=15, alpha=0.7, edgecolors='none', label='Old Motifs')

    # קו הפרדה מרוסק (Decision Boundary)
    ax.plot([0.73, 0.78], [0.2, 0.8], color='gray', linestyle='--', linewidth=1)

    # הוספת הציון AUC
    txt = ax.text(0.78, 0.85, "AUC ≈ 0.92", ha='center', fontsize=9, fontweight='bold', color='#333')
    txt.set_path_effects([path_effects.withStroke(linewidth=2, foreground='white')])

    # תוויות לקלאסטרים
    ax.text(0.63, 0.25, "Young-associated\nSequences", ha='center', fontsize=7, color='#0072B2', fontweight='bold')
    ax.text(0.88, 0.75, "Old-associated\nSequences", ha='center', fontsize=7, color='#D55E00', fontweight='bold')

    # כותרת תחתונה
    ax.text(0.5, 0.05, "Distinct Sequence Architecture", ha='center', fontsize=10, fontweight='bold')

    # ניקוי הצירים
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')

# ==========================================
# יצירת התמונה
# ==========================================
fig, ax = plt.subplots(figsize=(5, 3), dpi=300)
plot_bifurcation_schematic(ax)
plt.tight_layout()
plt.show()

In [ ]:
spec_loose

In [ ]:
genes_loose

In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

# Load
df = pd.read_csv("here.csv")

hp_cols = [
    "n_hidden_layers","hidden_width","activation","dropout",
    "batch_size","lr","weight_decay","residual","scheduler"
]

# Ensure mae is numeric
df["mae"] = pd.to_numeric(df["mae"], errors="coerce")

# Coerce numeric HPs
num_cols = ["n_hidden_layers","hidden_width","dropout",
            "batch_size","lr","weight_decay"]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# Clean: drop NaN/inf rows
df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=["mae"] + num_cols)

# Separate
X = df[hp_cols].copy()
y = df["mae"]

# Identify cats after coercion
num_cols = [c for c in hp_cols if pd.api.types.is_numeric_dtype(X[c])]
cat_cols = [c for c in hp_cols if c not in num_cols]

# 1) Feature importances
pre = ColumnTransformer([
    ("num","passthrough",num_cols),
    ("cat",OneHotEncoder(handle_unknown="ignore", sparse_output=False),cat_cols),
])

rf = RandomForestRegressor(n_estimators=500, random_state=42, n_jobs=-1)
pipe = Pipeline([("pre", pre), ("rf", rf)])
pipe.fit(X, y)

# Map importances
feat_names = []
if num_cols: feat_names += num_cols
if cat_cols:
    ohe = pipe.named_steps["pre"].named_transformers_["cat"]
    feat_names += list(ohe.get_feature_names_out(cat_cols))

imps = pd.Series(pipe.named_steps["rf"].feature_importances_, index=feat_names)

def base_name(feat):
    for c in cat_cols:
        if feat.startswith(c + "_"): return c
    return feat

imp_by_hp = imps.groupby(base_name).sum().sort_values(ascending=False)

print("\n=== Hyperparameter influence on MAE (RF importances) ===")
print(imp_by_hp.to_string())

# 2) Sweet spot analysis
BEST_FRAC = 0.10
k = max(3, int(np.ceil(BEST_FRAC * len(df))))
best = df.nsmallest(k, "mae")

def describe_numeric(series):
    q10, q50, q90 = series.quantile([0.10,0.50,0.90]).round(6)
    return f"{float(q10)} – {float(q90)} (median {float(q50)})"

def top_cats(series, n=3):
    vc = series.astype(str).value_counts()
    return ", ".join([f"{lab} ({cnt})" for lab,cnt in vc.head(n).items()])

print(f"\n=== Sweet spots (top {BEST_FRAC:.0%} configs) ===")
for c in hp_cols:
    # Only float/int dtypes go here
    if pd.api.types.is_numeric_dtype(df[c]) and not pd.api.types.is_bool_dtype(df[c]):
        print(f"{c:>15}: {describe_numeric(best[c])}")
    else:
        print(f"{c:>15}: {top_cats(best[c])}")

# 3) TL;DR
print("\n=== TL;DR ===")
print(f"- Best {BEST_FRAC:.0%} configs median MAE: {best['mae'].median():.3f} "
      f"(vs rest median {df.drop(best.index)['mae'].median():.3f})")
print("- Most influential hyperparameters:", ", ".join(imp_by_hp.index[:5]))
print("- Use the numeric 10–90% ranges and the top categorical values above as your default sweet spots.")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

BEST_FRAC = 0.10
k = max(3, int(np.ceil(BEST_FRAC * len(df))))
best = df.nsmallest(k, "mae")

sns.set_style("whitegrid")
plt.rcParams["figure.dpi"] = 120

for c in hp_cols:
    plt.figure(figsize=(6,4))
    if pd.api.types.is_numeric_dtype(df[c]) and not pd.api.types.is_bool_dtype(df[c]):
        # Plot histograms for all vs top 10%
        sns.histplot(df[c], bins=20, color="gray", alpha=0.4, label="all")
        sns.histplot(best[c], bins=20, color="teal", alpha=0.6, label="top 10%")
        plt.xlabel(c); plt.ylabel("count")
        plt.title(f"{c}: full vs top {BEST_FRAC:.0%}")
        plt.legend()
    else:
        # Category frequency bar chart
        all_counts = df[c].value_counts(normalize=True).rename("all")
        top_counts = best[c].value_counts(normalize=True).rename("top 10%")
        comp = pd.concat([all_counts, top_counts], axis=1).fillna(0)
        comp.plot(kind="bar", color=["gray","teal"], alpha=0.7)
        plt.title(f"{c}: category frequencies")
        plt.ylabel("fraction")
        plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()


In [ ]:
# Show TRUE cv_mae per exact hyperparameter combo (no binning, no normalization here).
# If multiple rows share the same combo, we average those rows only.

import itertools, numpy as np, pandas as pd, seaborn as sns, matplotlib.pyplot as plt

# ========= knobs =========
PATH = "grid_results_gates.csv"
HP_COLS = ["n_hidden_layers","hidden_width","activation","dropout",
           "batch_size","lr","weight_decay","l0_lambda"]
METRIC = "cv_mae"

MAX_LEVELS_PER_AXIS = 30   # cap to keep plots readable (top by frequency)
ANNOTATE_LIMIT = 25 * 25   # only annotate when cells <= this
CMAP = "magma_r"           # lower MAE -> darker with magma_r
VMIN, VMAX = None, None    # set both to numbers to fix scale; else computed from data
RANDOM_STATE = 42
# ========================

np.random.seed(RANDOM_STATE)
sns.set_theme(context="talk")
plt.rcParams["figure.dpi"] = 140

# ---------- load ----------
df = pd.read_csv(PATH)
df.columns = [c.strip() for c in df.columns]
assert METRIC in df.columns, f"{METRIC=} not found!"
df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=[METRIC]).copy()
df[METRIC] = pd.to_numeric(df[METRIC], errors="coerce")
df = df.dropna(subset=[METRIC]).copy()

# keep only available HP columns
HP_COLS = [c for c in HP_COLS if c in df.columns]
if not HP_COLS:
    raise ValueError("No matching hyperparameter columns found.")

# global color scale if not given
if VMIN is None or VMAX is None:
    vmin = df[METRIC].quantile(0.02)
    vmax = df[METRIC].quantile(0.98)
    if not np.isfinite(vmin): vmin = df[METRIC].min()
    if not np.isfinite(vmax): vmax = df[METRIC].max()
    pad = 0.02 * (vmax - vmin) if np.isfinite(vmin) and np.isfinite(vmax) else 0.0
    VMIN, VMAX = float(vmin - pad), float(vmax + pad)

# helpers
def natural_order(vals):
    """Sort labels numerically when they look like numbers/intervals."""
    s = pd.Series(list(map(str, vals)), dtype="object")
    import re
    def key(x):
        nums = re.findall(r'[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?', x)
        if len(nums) >= 2:
            try: return (float(nums[0]) + float(nums[1]))/2
            except: pass
        if len(nums) == 1:
            try: return float(nums[0])
            except: pass
        return np.inf
    return list(s.iloc[np.argsort(s.map(key).values)])

def fmt_from_range(vmin, vmax):
    rng = vmax - vmin
    if rng >= 50:   return ".1f"
    if rng >= 5:    return ".2f"
    if rng >= 0.5:  return ".3f"
    if rng >= 0.05: return ".4f"
    return ".5f"

fmt = fmt_from_range(VMIN, VMAX)

# ---------- pairwise heatmaps (EXACT combos, no binning) ----------
pairs = list(itertools.combinations(HP_COLS, 2))
print(f"Rendering {len(pairs)} heatmaps with global color scale [{VMIN:.4g}, {VMAX:.4g}]…")

for a, b in pairs:
    # reduce levels if too many (by frequency of exact values)
    work = df[[a, b, METRIC]].dropna().copy()
    # convert to string labels so pivot is 1 cell per exact value
    work[a] = work[a].astype(str)
    work[b] = work[b].astype(str)

    # keep most frequent exact values to avoid giant, unreadable plots
    if work[a].nunique() > MAX_LEVELS_PER_AXIS:
        keep_a = work[a].value_counts().index[:MAX_LEVELS_PER_AXIS]
        work = work[work[a].isin(keep_a)]
    if work[b].nunique() > MAX_LEVELS_PER_AXIS:
        keep_b = work[b].value_counts().index[:MAX_LEVELS_PER_AXIS]
        work = work[work[b].isin(keep_b)]

    if work.empty:
        plt.figure(figsize=(6, 4)); plt.title(f"{b} × {a} → {METRIC} (no data)"); plt.axis("off"); plt.show()
        continue

    piv = work.pivot_table(index=b, columns=a, values=METRIC, aggfunc="mean")

    # natural axis ordering
    piv = piv.reindex(index=natural_order(piv.index), columns=natural_order(piv.columns))

    # figure size based on grid
    h, w = piv.shape
    fig_w = max(7.0, min(22.0, 0.55 * w + 2.5))
    fig_h = max(6.0, min(22.0, 0.55 * h + 2.5))

    plt.figure(figsize=(fig_w, fig_h))
    ax = sns.heatmap(
        piv,
        cmap=CMAP,
        vmin=VMIN, vmax=VMAX,
        annot=(piv.size <= ANNOTATE_LIMIT),
        fmt=fmt,
        linewidths=0.4, linecolor="white",
        cbar_kws={"label": f"mean {METRIC}", "shrink": 0.85, "pad": 0.02},
        mask=piv.isna()
    )
    ax.set_xlabel(a); ax.set_ylabel(b)
    ax.set_title(f"{b} × {a} → mean {METRIC} (exact combos)", pad=10)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha="right", fontsize=10)
    ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=10)
    plt.tight_layout(pad=1.1); plt.show()


In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

# ========= knobs =========
PATH = "grid_results_gates.csv"
METRIC_COL = "cv_mae"
HP_COLS = [
    "n_hidden_layers", "hidden_width", "activation", "dropout",
    "batch_size", "lr", "weight_decay", "l0_lambda"
]
BEST_FRAC = 0.10  # top X% configs treated as "sweet spot"
RANDOM_STATE = 42
N_TREES = 500
# ========================

# Load
df = pd.read_csv(PATH)
df.columns = [c.strip() for c in df.columns]

# Ensure metric is numeric
df[METRIC_COL] = pd.to_numeric(df[METRIC_COL], errors="coerce")

# Coerce numeric HPs
num_hp_guess = ["n_hidden_layers","hidden_width","dropout",
                "batch_size","lr","weight_decay","l0_lambda"]
for c in num_hp_guess:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# Clean: drop NaN/inf rows
present_num_cols = [c for c in num_hp_guess if c in df.columns]
df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=[METRIC_COL] + present_num_cols)

# Separate features/target
X = df[HP_COLS].copy()
y = df[METRIC_COL].copy()

# Identify cats after coercion
num_cols = [c for c in HP_COLS if c in X.columns and pd.api.types.is_numeric_dtype(X[c])]
cat_cols = [c for c in HP_COLS if c in X.columns and c not in num_cols]

# 1) Feature importances (Random Forest on OHE+numeric)
# Handle sklearn version differences for OneHotEncoder arg name
try:
    ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:  # older sklearn
    ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

pre = ColumnTransformer(
    transformers=[
        ("num", "passthrough", num_cols),
        ("cat", ohe, cat_cols),
    ],
    remainder="drop",
)

rf = RandomForestRegressor(
    n_estimators=N_TREES, random_state=RANDOM_STATE, n_jobs=-1
)

pipe = Pipeline([("pre", pre), ("rf", rf)])
pipe.fit(X, y)

# Map importances back to HPs
feat_names = []
if num_cols:
    feat_names += num_cols
if cat_cols:
    ohe_fitted = pipe.named_steps["pre"].named_transformers_["cat"]
    feat_names += list(ohe_fitted.get_feature_names_out(cat_cols))

imps = pd.Series(pipe.named_steps["rf"].feature_importances_, index=feat_names)

def base_name(feat):
    for c in cat_cols:
        if feat.startswith(c + "_"):
            return c
    return feat

imp_by_hp = imps.groupby(base_name).sum().sort_values(ascending=False)

print("\n=== Hyperparameter influence on MAE (RF importances) ===")
print(imp_by_hp.to_string())

# 2) Sweet spot analysis (top BEST_FRAC by lowest MAE)
k = max(3, int(np.ceil(BEST_FRAC * len(df))))
best = df.nsmallest(k, METRIC_COL)

def describe_numeric(series):
    q10, q50, q90 = series.quantile([0.10, 0.50, 0.90]).round(6)
    return f"{float(q10)} – {float(q90)} (median {float(q50)})"

def top_cats(series, n=3):
    vc = series.astype(str).value_counts()
    return ", ".join([f"{lab} ({cnt})" for lab, cnt in vc.head(n).items()])

print(f"\n=== Sweet spots (top {BEST_FRAC:.0%} configs by lowest {METRIC_COL}) ===")
for c in HP_COLS:
    if c not in df.columns:
        continue
    if pd.api.types.is_numeric_dtype(df[c]) and not pd.api.types.is_bool_dtype(df[c]):
        print(f"{c:>15}: {describe_numeric(best[c])}")
    else:
        print(f"{c:>15}: {top_cats(best[c])}")

# 3) TL;DR
rest = df.drop(best.index)
print("\n=== TL;DR ===")
print(f"- Best {BEST_FRAC:.0%} configs median {METRIC_COL}: {best[METRIC_COL].median():.3f} "
      f"(vs rest median {rest[METRIC_COL].median():.3f})")
print("- Most influential hyperparameters:", ", ".join(imp_by_hp.index[:5]))
print("- Use the numeric 10–90% ranges and the top categorical values above as your default sweet spots.")


In [ ]:
import itertools, numpy as np, pandas as pd, seaborn as sns, matplotlib.pyplot as plt

# ========= knobs =========
PATH = "here.csv"
HP_COLS = ["n_hidden_layers","hidden_width","activation","dropout",
           "batch_size","lr","weight_decay","residual","scheduler"]
METRIC = "mae"
NUM_BINS = 12        # quantile bins when numeric has many unique values
MAX_CAT = 20         # cap categories for 1D/2D plots
CMAP = "magma_r"     # lower MAE → darker (good) with magma_r
RANDOM_STATE = 42
# ========================

np.random.seed(RANDOM_STATE)
sns.set_context("talk"); plt.rcParams["figure.dpi"] = 120

# 0) Load + basic clean
df = pd.read_csv(PATH)
df = df.replace([np.inf,-np.inf], np.nan).dropna(subset=[METRIC]).copy()
df[METRIC] = pd.to_numeric(df[METRIC], errors="coerce")
df = df.dropna(subset=[METRIC])

if "residual" in df.columns:
    df_filt = df[df["residual"].astype(str).str.lower().isin(["true","1"])].copy()
    df_filt = df[df["scheduler"].astype(str).str.lower().isin(["cosine","1"])].copy()
else:
    df_filt = df.copy()

df = df_filt.copy()
# identify numeric vs categorical as read
num_cols = [c for c in HP_COLS if c in df.columns and pd.api.types.is_numeric_dtype(df[c])]
cat_cols = [c for c in HP_COLS if c in df.columns and c not in num_cols]

# --- helpers ---
def bin_numeric(s, q=NUM_BINS):
    s = pd.to_numeric(s, errors="coerce")
    # constant / too few unique → just treat as categories (strings)
    if s.nunique(dropna=True) <= 2:
        return s.astype(str)
    try:
        b = pd.qcut(s, q=min(q, max(1, s.notna().sum()-1)), duplicates="drop")
        return b.astype(str)
    except Exception:
        lo, hi = s.min(), s.max()
        if not np.isfinite(lo) or not np.isfinite(hi) or lo == hi:
            return s.astype(str)
        edges = np.linspace(lo, hi, num=min(q+1, max(3, s.nunique(dropna=True))))
        edges = np.unique(edges)
        return pd.cut(s, bins=edges, include_lowest=True, duplicates="drop").astype(str)

def cap_top_k(s, k=MAX_CAT):
    s = s.astype(str)
    keep = s.value_counts().index[:k]
    return s.where(s.isin(keep), other="other")

# -----------------------------
# 1) ONE-DIMENSIONAL: mean MAE per value
# -----------------------------
print("Rendering 1D: mean MAE per hyperparameter value…")
for col in [c for c in HP_COLS if c in df.columns]:
    # choose representation
    if pd.api.types.is_numeric_dtype(df[col]):
        uniq = df[col].dropna().unique()
        x = df[col] if len(uniq) <= MAX_CAT else bin_numeric(df[col])
        x_name = f"{col}" if len(uniq) <= MAX_CAT else f"{col} (binned)"
        d = pd.DataFrame({x_name: x, METRIC: df[METRIC]}).dropna()
        # aggregate
        ag = d.groupby(x_name, dropna=False)[METRIC].mean().reset_index()
        # try to sort numeric-like labels
        try:
            mid = ag[x_name].astype(str).str.extract(r'([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)')[0].astype(float)
            ag = ag.iloc[np.argsort(mid.fillna(np.inf).values)]
        except Exception:
            ag = ag.sort_values(METRIC, ascending=True)
        # plot
        plt.figure(figsize=(7.5,4))
        plt.plot(range(len(ag)), ag[METRIC], marker="o")
        plt.xticks(range(len(ag)), ag[x_name].astype(str), rotation=30, ha="right")
        plt.ylabel(f"mean {METRIC}"); plt.xlabel(x_name)
        plt.title(f"{col}: average {METRIC} per value")
        plt.grid(axis="y", alpha=0.25); plt.tight_layout(); plt.show()
    else:
        x = cap_top_k(df[col])
        d = pd.DataFrame({col: x, METRIC: df[METRIC]}).dropna()
        ag = d.groupby(col, dropna=False)[METRIC].mean().sort_values().reset_index()
        plt.figure(figsize=(7.5,4))
        sns.barplot(data=ag, x=col, y=METRIC, order=ag[col].tolist(), alpha=0.9)
        plt.xticks(rotation=30, ha="right")
        plt.ylabel(f"mean {METRIC}"); plt.xlabel(col)
        plt.title(f"{col}: average {METRIC} per value")
        plt.grid(axis="y", alpha=0.25); plt.tight_layout(); plt.show()

# -----------------------------
# 2) TWO-DIMENSIONAL after filtering residual==true
# -----------------------------
if "residual" in df.columns:
    df_filt = df[df["residual"].astype(str).str.lower().isin(["true","1"])].copy()
else:
    df_filt = df.copy()

keep_cols = [c for c in HP_COLS if c in df_filt.columns and c != "residual"]
num_cols_f = [c for c in keep_cols if pd.api.types.is_numeric_dtype(df_filt[c])]
cat_cols_f = [c for c in keep_cols if c not in num_cols_f]

print(f"\nFiltered to residual==true: {len(df_filt)}/{len(df)} rows remain.")
global_min, global_max = df_filt[METRIC].min(), df_filt[METRIC].max()

pairs = list(itertools.combinations(keep_cols, 2))
print(f"Rendering {len(pairs)} 2D heatmaps (fixed color scale {global_min:.3g}–{global_max:.3g})…")

for a, b in pairs:
    A, B = df_filt[a], df_filt[b]
    a_is_num, b_is_num = pd.api.types.is_numeric_dtype(A), pd.api.types.is_numeric_dtype(B)

    if a_is_num and b_is_num:
        A2 = bin_numeric(A)
        B2 = bin_numeric(B)
    elif a_is_num and not b_is_num:
        A2 = bin_numeric(A)
        B2 = cap_top_k(B)
    elif not a_is_num and b_is_num:
        A2 = cap_top_k(A)
        B2 = bin_numeric(B)
    else:
        A2 = cap_top_k(A)
        B2 = cap_top_k(B)

    work = pd.DataFrame({a: A2, b: B2, METRIC: df_filt[METRIC]}).dropna(subset=[a, b, METRIC])
    if work.empty:
        plt.figure(figsize=(6,4)); plt.title(f"{b} × {a} → {METRIC} (no data)"); plt.axis("off"); plt.show()
        continue

    piv = work.pivot_table(index=b, columns=a, values=METRIC, aggfunc="mean")

    # keep top categories if too many
    if piv.shape[0] > MAX_CAT:
        keep = work[b].value_counts().index[:MAX_CAT]
        piv = piv.loc[piv.index.intersection(keep)]
    if piv.shape[1] > MAX_CAT:
        keep = work[a].value_counts().index[:MAX_CAT]
        piv = piv.loc[:, piv.columns.intersection(keep)]

    # sort labels "naturally" when possible
    def try_sort(vals):
        try:
            tmp = pd.Series(vals)
            mid = tmp.astype(str).str.extract(r'([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)')[0].astype(float)
            order = np.argsort(mid.fillna(np.inf).values)
            return tmp.iloc[order]
        except Exception:
            return vals
    piv = piv.reindex(index=try_sort(piv.index), columns=try_sort(piv.columns))

    fig = plt.figure(figsize=(max(6, 0.55*len(piv.columns)+2), max(5, 0.5*len(piv.index)+2)))
    ax = sns.heatmap(
        piv, cmap=CMAP, annot=piv.size <= 140, fmt=".3g",
        cbar_kws={"label": f"mean {METRIC}"},
        linewidths=0.3, linecolor="white",
        vmin=7.5, vmax=8.5    # <<< fixed scale across all plots
    )
    ax.set_xlabel(a); ax.set_ylabel(b)
    ax.set_title(f"{b} × {a} → mean {METRIC} (residual=true)")

    # mark best (lowest MAE) cell
    #vals = piv.values
    #if np.isfinite(vals).any():
        #iy, ix = np.unravel_index(np.nanargmin(vals), vals.shape)
        #ax.scatter([ix+0.5], [iy+0.5], s=160, facecolors='none', edgecolors='black', linewidths=1.6)

    plt.tight_layout(); plt.show()


In [ ]:
import pandas as pd

df = pd.read_csv("per_fold_stage1.csv")

In [ ]:
print(df.columns.tolist())


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1) Load CSV properly (force comma delimiter)
df = pd.read_csv("per_fold_stage1.csv", sep=",", engine="python")

# 2) Strip spaces from headers
df.columns = [c.strip() for c in df.columns]

# 3) Focus on the columns you care about
keep = [
    "n_hidden_layers","hidden_width","activation","dropout",
    "batch_size","lr","weight_decay","residual","scheduler","mae"
]
df = df[keep]

# 4) Convert obvious numerics
for c in ["n_hidden_layers","hidden_width","dropout",
          "batch_size","lr","weight_decay","mae"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# 5) Plot each hyperparameter vs MAE
sns.set_style("whitegrid"); plt.rcParams["figure.dpi"] = 120

for col in [c for c in keep if c != "mae"]:
    plt.figure(figsize=(6,4))
    if pd.api.types.is_numeric_dtype(df[col]):
        sns.scatterplot(data=df, x=col, y="mae", alpha=0.6)
    else:
        order = df.groupby(col)["mae"].mean().sort_values().index
        sns.boxplot(data=df, x=col, y="mae", order=order)
        sns.stripplot(data=df, x=col, y="mae", order=order,
                      color="black", alpha=0.3, jitter=True)
        plt.xticks(rotation=30, ha="right")
    plt.title(f"{col} vs MAE")
    plt.tight_layout()
    plt.show()


In [ ]:
# --- COLAB MERGE: TRAIT + VDJdb(slim) + McPAS-TCR ----------------------------
import pandas as pd
import numpy as np
import re

# ====== 0) INPUT FILES (edit these if names/paths differ) ====================
TRAIT_PATH = "/content/20250312-TRAIT_search_download.xlsx"
VDJDB_SLIM_PATH = "/content/vdjdb.slim.txt"          # your slim text file (tab-delimited)
MCPAS_PATH = "/content/McPAS-TCR.csv"

# ====== 1) helpers ===========================================================
def clean_str(x):
    if pd.isna(x): return None
    s = str(x).strip()
    return s if s != "" else None

def clean_cdr3(x):
    if pd.isna(x): return None
    s = re.sub(r'[^A-Za-z]', '', str(x)).upper()
    return s if s else None

def norm_species(x):
    s = clean_str(x)
    if not s: return None
    s = s.replace("HomoSapiens","Homo sapiens").replace("MusMusculus","Mus musculus")
    return s

def norm_chain(x):
    s = clean_str(x)
    if not s: return None
    s = s.upper()
    if s in ["TRB","B","BETA","TCRB","TR_B","TR-BETA"]: return "TRB"
    if s in ["TRA","A","ALPHA","TCRA","TR_A","TR-ALPHA"]: return "TRA"
    return s

def norm_gene(g):
    g = clean_str(g)
    if not g: return None
    g = g.upper()
    g = re.sub(r'\s+', '', g)
    return g

def norm_mhc(x):
    s = clean_str(x)
    if not s: return None
    return s.upper().replace("HLA-A2","HLA-A*02:01").replace("HLA-B8","HLA-B*08:01")

def to_float(x, default=np.nan):
    try:
        return float(x)
    except:
        return default

def pick_first(df, names):
    for n in names:
        if n in df.columns:
            return df[n]
    return pd.Series([None]*len(df))

TARGET = ["CDR3b","Species","Epitope","Epitope_species","Epitope_gene",
          "TRBV","TRBJ","MHC_A","MHC_B","MHC_class","PMID","Category","Source"]

# ====== 2) VDJdb(slim) =======================================================
# Your example shows columns like: gene, cdr3, species, antigen.epitope, antigen.gene, antigen.species,
# v.segm, j.segm, mhc.a, mhc.b, mhc.class, reference.id, ...
vdj = pd.read_csv(VDJDB_SLIM_PATH, sep="\t", dtype=str)

# keep only beta chain from VDJdb:
# VDJdb has 'gene' column with 'TRA'/'TRB' -> filter to TRB; if absent, we keep all.
if "gene" in vdj.columns:
    vdj = vdj[vdj["gene"].str.upper().eq("TRB")]

vdj_std = pd.DataFrame({
    "CDR3b":          vdj.get("cdr3"),
    "Species":        vdj.get("species"),
    "Epitope":        vdj.get("antigen.epitope"),
    "Epitope_species":vdj.get("antigen.species"),
    "Epitope_gene":   vdj.get("antigen.gene"),
    "TRBV":           vdj.get("v.segm"),
    "TRBJ":           vdj.get("j.segm"),
    "MHC_A":          vdj.get("mhc.a"),
    "MHC_B":          vdj.get("mhc.b"),
    "MHC_class":      vdj.get("mhc.class"),
    "PMID":           vdj.get("reference.id"),
    "Category":       vdj.get("meta") if "meta" in vdj.columns else None
})
vdj_std["Source"] = "VDJdb"

# clean VDJdb
for c in ["CDR3b","TRBV","TRBJ","MHC_A","MHC_B","MHC_class","Epitope","Epitope_gene","Epitope_species","PMID","Category"]:
    vdj_std[c] = vdj_std[c].map(clean_str)
vdj_std["CDR3b"] = vdj_std["CDR3b"].map(clean_cdr3)
vdj_std["Species"] = vdj_std["Species"].map(norm_species)
vdj_std["TRBV"] = vdj_std["TRBV"].map(norm_gene)
vdj_std["TRBJ"] = vdj_std["TRBJ"].map(norm_gene)
vdj_std["MHC_A"] = vdj_std["MHC_A"].map(norm_mhc)
vdj_std["MHC_B"] = vdj_std["MHC_B"].map(norm_mhc)

# ====== 3) TRAIT (.xlsx) =====================================================
trait = pd.read_excel(TRAIT_PATH, dtype=str)
# make flexible mapping in case headers differ slightly
trait_std = pd.DataFrame({
    "CDR3b":           pick_first(trait, ["CDR3b","cdr3b","CDR3","cdr3"]),
    "Species":         pick_first(trait, ["Species","species","Organism"]),
    "Epitope":         pick_first(trait, ["Epitope","epitope","Peptide"]),
    "Epitope_species": pick_first(trait, ["Epitope_species","epitope_species","Antigen_species","Antigen.species"]),
    "Epitope_gene":    pick_first(trait, ["Epitope_gene","epitope_gene","Antigen_gene","Antigen.gene"]),
    "TRBV":            pick_first(trait, ["TRBV","V","v_call","v.segm","TRBV_gene"]),
    "TRBJ":            pick_first(trait, ["TRBJ","J","j_call","j.segm","TRBJ_gene"]),
    "MHC_A":           pick_first(trait, ["MHC_A","HLA_A","mhc.a","HLA.class.I"]),
    "MHC_B":           pick_first(trait, ["MHC_B","HLA_B","mhc.b","HLA.class.II"]),
    "MHC_class":       pick_first(trait, ["MHC_class","mhc.class","MHC"]),
    "PMID":            pick_first(trait, ["PMID","reference","reference.id"]),
    "Category":        pick_first(trait, ["Category","Disease","Pathology","meta"]),
})
trait_std["Source"] = "TRAIT"

for c in ["CDR3b","TRBV","TRBJ","MHC_A","MHC_B","MHC_class","Epitope","Epitope_gene","Epitope_species","PMID","Category","Species"]:
    trait_std[c] = trait_std[c].map(clean_str)
trait_std["CDR3b"] = trait_std["CDR3b"].map(clean_cdr3)
trait_std["Species"] = trait_std["Species"].map(norm_species)
trait_std["TRBV"] = trait_std["TRBV"].map(norm_gene)
trait_std["TRBJ"] = trait_std["TRBJ"].map(norm_gene)
trait_std["MHC_A"] = trait_std["MHC_A"].map(norm_mhc)
trait_std["MHC_B"] = trait_std["MHC_B"].map(norm_mhc)

# ====== 4) McPAS (.csv) ======================================================
mcpas = pd.read_csv(MCPAS_PATH, dtype=str)
mcpas_std = pd.DataFrame({
    "CDR3b":           pick_first(mcpas, ["CDR3.beta.aa","cdr3b","cdr3","CDR3b"]),
    "Species":         pick_first(mcpas, ["Species","species","Organism"]),
    "Epitope":         pick_first(mcpas, ["Epitope.peptide","Epitope","peptide"]),
    "Epitope_species": pick_first(mcpas, ["Antigen.species","Epitope_species","epitope_species"]),
    "Epitope_gene":    pick_first(mcpas, ["Antigen.gene","Epitope_gene","epitope_gene"]),
    "TRBV":            pick_first(mcpas, ["V","TRBV","v_call"]),
    "TRBJ":            pick_first(mcpas, ["J","TRBJ","j_call"]),
    "MHC_A":           pick_first(mcpas, ["HLA.class.I","MHC_A","mhc.a"]),
    "MHC_B":           pick_first(mcpas, ["HLA.class.II","MHC_B","mhc.b"]),
    "MHC_class":       pick_first(mcpas, ["MHC.class","MHC_class","mhc.class"]),
    "PMID":            pick_first(mcpas, ["PMID","reference.id"]),
    "Category":        pick_first(mcpas, ["Pathology","Disease","Category"]),
})
mcpas_std["Source"] = "McPAS"

for c in ["CDR3b","TRBV","TRBJ","MHC_A","MHC_B","MHC_class","Epitope","Epitope_gene","Epitope_species","PMID","Category","Species"]:
    mcpas_std[c] = mcpas_std[c].map(clean_str)
mcpas_std["CDR3b"] = mcpas_std["CDR3b"].map(clean_cdr3)
mcpas_std["Species"] = mcpas_std["Species"].map(norm_species)
mcpas_std["TRBV"] = mcpas_std["TRBV"].map(norm_gene)
mcpas_std["TRBJ"] = mcpas_std["TRBJ"].map(norm_gene)
mcpas_std["MHC_A"] = mcpas_std["MHC_A"].map(norm_mhc)
mcpas_std["MHC_B"] = mcpas_std["MHC_B"].map(norm_mhc)

# ====== 5) combine + quality/dedup ===========================================
dfs = [vdj_std, trait_std, mcpas_std]
combined = pd.concat(dfs, ignore_index=True)[TARGET]

# drop obvious empties / invalid cdr3 lengths
combined["cdr3_len"] = combined["CDR3b"].map(lambda s: len(s) if isinstance(s,str) else np.nan)
combined = combined.dropna(subset=["CDR3b"])
combined = combined[(combined["cdr3_len"]>=5) & (combined["cdr3_len"]<=30)].drop(columns=["cdr3_len"])

# choose a conservative dedup key; adjust if you want stricter/looser
dedup_key = ["CDR3b","TRBV","TRBJ","Epitope","MHC_A","MHC_B","MHC_class"]
combined = combined.drop_duplicates(subset=dedup_key, keep="first")

print("Rows per source:\n", combined["Source"].value_counts(dropna=False))
print("Final shape:", combined.shape)

# ====== 6) save ==============================================================
combined.to_csv("/content/TCR_databases_merged.csv", index=False)
combined.to_parquet("/content/TCR_databases_merged.parquet", index=False)
print("Wrote /content/TCR_databases_merged.{csv,parquet}")


In [ ]:
combined

In [ ]:
# ==== COLAB: Merge TRAIT + VDJdb(slim) + McPAS into one clean table (β chain) ====
import pandas as pd
import numpy as np
import re

# ---------- 0) INPUT FILES (edit paths if needed) ----------
TRAIT_PATH      = "/content/20250312-TRAIT_search_download.xlsx"  # your TRAIT export
VDJDB_SLIM_PATH = "/content/vdjdb.slim.txt"                           # tab-delimited VDJdb slim/plain
MCPAS_PATH      = "/content/McPAS-TCR.csv"                        # McPAS CSV

# ---------- 1) helpers ----------
def clean_str(x):
    if pd.isna(x): return None
    s = str(x).strip()
    return s if s else None

def clean_cdr3(x):
    if pd.isna(x): return None
    s = re.sub(r'[^A-Za-z]', '', str(x)).upper()
    return s if s else None

def norm_species(x):
    s = clean_str(x)
    if not s: return None
    # normalize common variants
    s = s.replace("HomoSapiens", "Homo sapiens").replace("MusMusculus", "Mus musculus")
    return s

def norm_gene(g):
    g = clean_str(g)
    if not g: return None
    return re.sub(r"\s+", "", g.upper())

def norm_mhc(x):
    s = clean_str(x)
    if not s: return None
    return s.upper()

def infer_mhc_class(hla):
    # robust to NaN/float/None and common human/mouse notations
    if pd.isna(hla):
        return None
    h = str(hla).strip().upper()
    if not h:
        return None
    # Human class II (and shorthand)
    if h.startswith(("HLA-D", "DR", "DQ", "DP")):
        return "MHCII"
    # Human class I (and shorthand with missing 'HLA-')
    if h.startswith(("HLA-A", "HLA-B", "HLA-C", "A*", "B*", "C*")):
        return "MHCI"
    # Mouse: H-2 class I (K, D, L) vs class II (I-A, I-E)
    if h.startswith(("H-2", "H2")):
        if any(tok in h for tok in ("IA", "I-A", "IE", "I-E")):
            return "MHCII"
        return "MHCI"
    return None

TARGET = [
    "CDR3b","Species","Epitope","Epitope_species","Epitope_gene",
    "TRBV","TRBJ","MHC_A","MHC_B","MHC_class","PMID","Category","Source"
]

# ---------- 2) VDJdb (slim/plain text) ----------
vdj = pd.read_csv(VDJDB_SLIM_PATH, sep="\t", dtype=str)
# keep only beta chain from VDJdb (your file has 'gene' with TRA/TRB)
if "gene" in vdj.columns:
    vdj = vdj[vdj["gene"].str.upper() == "TRB"]

vdjdb_std = pd.DataFrame({
    "CDR3b":           vdj.get("cdr3"),
    "Species":         vdj.get("species"),
    "Epitope":         vdj.get("antigen.epitope"),
    "Epitope_species": vdj.get("antigen.species"),
    "Epitope_gene":    vdj.get("antigen.gene"),
    "TRBV":            vdj.get("v.segm"),
    "TRBJ":            vdj.get("j.segm"),
    "MHC_A":           vdj.get("mhc.a"),
    "MHC_B":           vdj.get("mhc.b"),
    "MHC_class":       vdj.get("mhc.class"),
    "PMID":            vdj.get("reference.id"),
    "Category":        None,
})
vdjdb_std["Source"] = "VDJdb"

# cleaning
for c in ["CDR3b","TRBV","TRBJ","MHC_A","MHC_B","MHC_class","Epitope",
          "Epitope_gene","Epitope_species","PMID","Category","Species"]:
    vdjdb_std[c] = vdjdb_std[c].apply(clean_str)
vdjdb_std["CDR3b"]   = vdjdb_std["CDR3b"].apply(clean_cdr3)
vdjdb_std["Species"] = vdjdb_std["Species"].apply(norm_species)
vdjdb_std["TRBV"]    = vdjdb_std["TRBV"].apply(norm_gene)
vdjdb_std["TRBJ"]    = vdjdb_std["TRBJ"].apply(norm_gene)
vdjdb_std["MHC_A"]   = vdjdb_std["MHC_A"].apply(norm_mhc)
vdjdb_std["MHC_B"]   = vdjdb_std["MHC_B"].apply(norm_mhc)

# ---------- 3) TRAIT (.xlsx) — exact headers you provided ----------
# TRAIT columns you showed:
# ID, TCR_name, CDR3α, TRAV, TRAJ, CDR3β, TRBV, TRBJ, Species, Sequencing_methods, MHC_A, MHC_B, MHC_class,
# Epitope, Epitope_gene, Epitope_species, Binding, Identification_methods, Verification_methods, KD_μM,
# Affinity_method, Structure, Structure_method, Donor, Tissue, Category, PMID, Year
trait = pd.read_excel(TRAIT_PATH, dtype=str)

trait_std = pd.DataFrame({
    "CDR3b":           trait["CDR3β"],
    "Species":         trait["Species"],
    "Epitope":         trait["Epitope"],
    "Epitope_species": trait["Epitope_species"],
    "Epitope_gene":    trait["Epitope_gene"],
    "TRBV":            trait["TRBV"],
    "TRBJ":            trait["TRBJ"],
    "MHC_A":           trait["MHC_A"],
    "MHC_B":           trait["MHC_B"],
    "MHC_class":       trait["MHC_class"],
    "PMID":            trait["PMID"],
    "Category":        trait["Category"],
})
trait_std["Source"] = "TRAIT"

for c in ["CDR3b","TRBV","TRBJ","MHC_A","MHC_B","MHC_class","Epitope",
          "Epitope_gene","Epitope_species","PMID","Category","Species"]:
    trait_std[c] = trait_std[c].apply(clean_str)
trait_std["CDR3b"]   = trait_std["CDR3b"].apply(clean_cdr3)
trait_std["Species"] = trait_std["Species"].apply(norm_species)
trait_std["TRBV"]    = trait_std["TRBV"].apply(norm_gene)
trait_std["TRBJ"]    = trait_std["TRBJ"].apply(norm_gene)
trait_std["MHC_A"]   = trait_std["MHC_A"].apply(norm_mhc)
trait_std["MHC_B"]   = trait_std["MHC_B"].apply(norm_mhc)

# ---------- 4) McPAS (.csv) — exact headers you provided ----------
# McPAS headers you showed include: CDR3.beta.aa, Species, Category/Pathology, Antigen.protein, Epitope.peptide, MHC, TRBV, TRBJ, PubMed.ID, ...
mcpas = pd.read_csv(MCPAS_PATH, dtype=str)
mcpas_mhc = mcpas.get("MHC")  # single MHC field

mcpas_std = pd.DataFrame({
    "CDR3b":           mcpas.get("CDR3.beta.aa"),
    "Species":         mcpas.get("Species"),
    "Epitope":         mcpas.get("Epitope.peptide"),
    "Epitope_species": None,                         # McPAS typically lacks explicit species per epitope
    "Epitope_gene":    mcpas.get("Antigen.protein"), # closest available field
    "TRBV":            mcpas.get("TRBV"),
    "TRBJ":            mcpas.get("TRBJ"),
    "MHC_A":           mcpas_mhc.apply(clean_str),
    "MHC_B":           None,
    "MHC_class":       mcpas_mhc.apply(infer_mhc_class),
    "PMID":            mcpas.get("PubMed.ID"),
    "Category":        (mcpas.get("Category") if "Category" in mcpas.columns else pd.Series([None]*len(mcpas))).combine_first(
                        mcpas.get("Pathology") if "Pathology" in mcpas.columns else pd.Series([None]*len(mcpas))
                      ),
})
mcpas_std["Source"] = "McPAS"

for c in ["CDR3b","TRBV","TRBJ","Epitope","Epitope_gene","PMID","Category","Species","MHC_A"]:
    mcpas_std[c] = mcpas_std[c].apply(clean_str)
mcpas_std["CDR3b"]   = mcpas_std["CDR3b"].apply(clean_cdr3)
mcpas_std["Species"] = mcpas_std["Species"].apply(norm_species)
mcpas_std["TRBV"]    = mcpas_std["TRBV"].apply(norm_gene)
mcpas_std["TRBJ"]    = mcpas_std["TRBJ"].apply(norm_gene)
mcpas_std["MHC_A"]   = mcpas_std["MHC_A"].apply(norm_mhc)

# ---------- 5) combine + QC + de-dup ----------
dfs = [vdjdb_std, trait_std, mcpas_std]
combined = pd.concat(dfs, ignore_index=True)[TARGET]

# drop rows without usable CDR3b; enforce reasonable CDR3 length
combined = combined.dropna(subset=["CDR3b"])
combined["cdr3_len"] = combined["CDR3b"].apply(lambda s: len(s) if isinstance(s, str) else np.nan)
combined = combined[(combined["cdr3_len"] >= 5) & (combined["cdr3_len"] <= 30)].drop(columns=["cdr3_len"])

# conservative duplicate key: same sequence + V/J + epitope + MHC
dedup_key = ["CDR3b","TRBV","TRBJ","Epitope","MHC_A","MHC_B","MHC_class"]
combined = combined.drop_duplicates(subset=dedup_key, keep="first")

print("Counts per Source:\n", combined["Source"].value_counts(dropna=False))
print("Final shape:", combined.shape)

# ---------- 6) save ----------
combined.to_csv("/content/TCR_databases_merged.csv", index=False)
combined.to_parquet("/content/TCR_databases_merged.parquet", index=False)
print("Wrote /content/TCR_databases_merged.{csv,parquet}")


In [ ]:
combined

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load the data
data = {
    "file": [
        "train_tcrs_across_samples_10K_cloness.csv",
        "hashed_substring_aggregated_counts.csv",
        "louvain_aggregated_counts.csv",
        "label_propagation_aggregated_counts.csv",
        "dbscan_aggregated_counts.csv",
        "connected_components_aggregated_counts.csv",
    ],
    "method": [
        "TCR Weight", "Hashed Substring", "Louvain",
        "Label Propagation", "DBSCAN", "Connected Components"
    ],
    "mae": [9.800, 10.895, 11.311, 10.039, 11.909, 11.309]
}

df = pd.DataFrame(data)

# Plot settings
fig, ax = plt.subplots(figsize=(10, 6))

# Bar positions
x = np.arange(len(df['method']))
width = 0.5

# Create the MAE bar plot
bar_mae = ax.bar(x, df['mae'], width, color='blue', alpha=0.8, label='MAE')

# Add titles and labels
ax.set_title('Mean Absolute Error (MAE) by Method', fontsize=14)
ax.set_xlabel('Method', fontsize=12)
ax.set_ylabel('MAE', fontsize=12)
ax.set_xticks(x)
ax.set_xticklabels(df['method'], rotation=30, ha='right', fontsize=10)
ax.legend()

# Add value annotations
for rect in bar_mae:
    height = rect.get_height()
    ax.annotate(f'{height:.2f}',
                xy=(rect.get_x() + rect.get_width() / 2, height),
                xytext=(0, 3),  # Offset text
                textcoords="offset points",
                ha='center', va='bottom', fontsize=9)

# Show the plot
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Example data
data = {
    'method': [
        'Hashed Substrings Clustering',
        'Louvain Clustering',
        'Label Propagation Clustering',
        'DBSCAN Clustering',
        'Connected Components Clustering',
        'TCR Weight'
    ],
    'mae': [10.89, 11.31, 10.04, 11.91, 11.31, 10.535],  # Example MAE values
}

# Convert to DataFrame
df = pd.DataFrame(data)

# Define colors for each method
colors = {
    'Hashed Substrings Clustering': '#4682B4',  # Blue
    'Louvain Clustering': '#32CD32',          # Green
    'Label Propagation Clustering': '#FFA500', # Orange
    'DBSCAN Clustering': '#8A2BE2',            # Purple
    'Connected Components Clustering': '#FF6347',  # Red
    'TCR Weight': 'gray',                   # Gold
}

# Sort data by MAE values
df = df.sort_values(by='mae', ascending=True)

# Enhanced professional-style plot for MAE
fig, ax = plt.subplots(figsize=(12, 8))

# Define x positions and bar width
x_positions = np.arange(len(df['method']))
bar_width = 0.6

# Plot MAE values with improved aesthetics
bars = ax.bar(
    x_positions,
    df['mae'],
    width=bar_width,
    color=[colors[method] for method in df['method']],
    alpha=0.85
)

# Add titles and labels
ax.set_title('Mean Absolute Error (MAE) by Clustering Method', fontsize=18, weight='bold')
ax.set_xlabel('Clustering Method', fontsize=14, weight='bold')
ax.set_ylabel('MAE', fontsize=14, weight='bold')
ax.set_xticks(x_positions)
ax.set_xticklabels(df['method'], rotation=45, ha='right', fontsize=12, weight='bold')

# Annotate the values on the bars with a more professional look
for bar in bars:
    height = bar.get_height()
    ax.annotate(f'{height:.2f}',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 5),  # Offset text
                textcoords="offset points",
                ha='center', va='bottom', fontsize=10, weight='bold', color='black')

# Add gridlines for a polished look
ax.yaxis.grid(True, linestyle='--', alpha=0.7)
ax.set_axisbelow(True)

# Tight layout for better spacing
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Data for plotting
data = {
    "File": [
        "consistency_area_top_train_tcrs_across_samples.csv",
        "consistency_area_top_train_tcrs_across_samples.csv",
        "publicity_top_train_tcrs_across_samples.csv",
        "publicity_top_train_tcrs_across_samples.csv",
        "average_count_top_train_tcrs_across_samples.csv",
        "average_count_top_train_tcrs_across_samples.csv",
        "variance_score_top_train_tcrs_across_samples.csv",
        "variance_score_top_train_tcrs_across_samples.csv"
    ],
    "Feature_Size": [1000, 1500, 1000, 1500, 1000, 1500, 1000, 1500],
    "MAE": [
        10.87993909, 10.07271369, 11.12405509, 11.72507547,
        11.64762664, 11.64762664, 13.08059808, 13.08059808
    ],
    "Method": [
        "TCR Weight", "TCR Weight", "Publicity", "Publicity",
        "Average", "Average", "Variance", "Variance"
    ]
}

# Convert to DataFrame
df = pd.DataFrame(data)

# Plot
plt.figure(figsize=(12, 6))
sns.barplot(data=df, x="Feature_Size", y="MAE", hue="Method", palette="viridis")
plt.title("Comparison of MAE by Feature Size and Method", fontsize=16, weight="bold")
plt.ylabel("Mean Absolute Error (MAE)", fontsize=12)
plt.xlabel("Feature Size", fontsize=12)
plt.ylim(0, max(df["MAE"]) + 2)

# Add labels
for index, row in df.iterrows():
    plt.text(
        x=row["Feature_Size"] + (index // 2) * 0.3 - 0.15, y=row["MAE"] + 0.2,
        s=f"{row['MAE']:.2f}",
        ha="center", fontsize=10
    )

plt.legend(title="Method")
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load the correlation files
file_abundance = 'correlation_age_abundance_kmers.csv'
file_unique = 'correlation_age_unique_kmers.csv'

df_abundance = pd.read_csv(file_abundance)
df_unique = pd.read_csv(file_unique)

# Define a function to plot the correlations
def plot_correlations(df, title, output_file):
    plt.figure(figsize=(12, 6))
    plt.scatter(df['Pearson Correlation'], df['Spearman Correlation'], alpha=0.6, label='k-mers', color='orange')
    plt.axhline(0, color='black', linestyle='--', linewidth=0.8, alpha=0.7)
    plt.axvline(0, color='black', linestyle='--', linewidth=0.8, alpha=0.7)
    plt.title(title, fontsize=16, weight='bold')
    plt.xlabel('Pearson Correlation', fontsize=14)
    plt.ylabel('Spearman Correlation', fontsize=14)
    plt.grid(alpha=0.4)
    plt.legend(fontsize=12)
    plt.tight_layout()
    #plt.savefig(output_file)
    plt.show()

# Plot for abundance k-mers
plot_correlations(df_abundance, 'Abundance k-mers Correlation', '/mnt/data/abundance_kmers_correlation_plot.png')

# Plot for unique k-mers
plot_correlations(df_unique, 'Unique k-mers Correlation', '/mnt/data/unique_kmers_correlation_plot.png')


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# File paths for correlation analysis
correlation_files = {
    "Abundance k-mers": "correlation_age_abundance_kmers.csv",
    "Unique k-mers": "correlation_age_unique_kmers.csv"
}

# Initialize a figure for distribution plots
fig, axes = plt.subplots(2, len(correlation_files), figsize=(14, 10), sharey=True)
fig.suptitle("Distributions of Correlations for k-mers", fontsize=16, weight='bold')

# Loop through files and plot
for idx, (title, file_path) in enumerate(correlation_files.items()):
    # Load the data
    data = pd.read_csv(file_path)

    # Plot Pearson Correlation distribution
    sns.histplot(data["Pearson Correlation"], bins=30, kde=True, ax=axes[0, idx], color="blue", alpha=0.7)
    axes[0, idx].set_title(f"{title} - Pearson", fontsize=14, weight='bold')
    axes[0, idx].set_xlabel("Pearson Correlation", fontsize=12)
    axes[0, idx].set_ylabel("Density", fontsize=12)
    axes[0, idx].set_xlim(-1, 1)

    # Plot Spearman Correlation distribution
    sns.histplot(data["Spearman Correlation"], bins=30, kde=True, ax=axes[1, idx], color="green", alpha=0.7)
    axes[1, idx].set_title(f"{title} - Spearman", fontsize=14, weight='bold')
    axes[1, idx].set_xlabel("Spearman Correlation", fontsize=12)
    axes[1, idx].set_ylabel("Density", fontsize=12)
    axes[1, idx].set_xlim(-1, 1)

# Adjust layout and display the plots
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load the data
file_3mers = "sample_kmers_count.csv"
file_4mers = "sample_4mers_count.csv"

data_3mers = pd.read_csv(file_3mers)
data_4mers = pd.read_csv(file_4mers)

# Extract age and counts
age_3mers = data_3mers["Age"]
unique_3mers_count = data_3mers["Sample_Count"]

age_4mers = data_4mers["Age"]
unique_4mers_count = data_4mers["Unique_4mers_Count"]

# Create the plot
fig, ax1 = plt.subplots(figsize=(12, 8))

# Plot unique 3-mers
color_3mers = "blue"
ax1.set_xlabel("Age", fontsize=14, weight='bold')
ax1.set_ylabel("Unique 3-mers Count", fontsize=14, weight='bold', color=color_3mers)
sns.scatterplot(x=age_3mers, y=unique_3mers_count, ax=ax1, color=color_3mers, alpha=0.7, label="Unique 3-mers")
sns.regplot(x=age_3mers, y=unique_3mers_count, ax=ax1, scatter=False, color=color_3mers, label="3-mers Trendline")
ax1.tick_params(axis='y', labelcolor=color_3mers)

# Create a second y-axis for unique 4-mers
ax2 = ax1.twinx()
color_4mers = "orange"
ax2.set_ylabel("Unique 4-mers Count", fontsize=14, weight='bold', color=color_4mers)
sns.scatterplot(x=age_4mers, y=unique_4mers_count, ax=ax2, color=color_4mers, alpha=0.7, label="Unique 4-mers")
sns.regplot(x=age_4mers, y=unique_4mers_count, ax=ax2, scatter=False, color=color_4mers, label="4-mers Trendline")
ax2.tick_params(axis='y', labelcolor=color_4mers)

# Title and legend
plt.title("Unique 3-mers and 4-mers Count vs Age", fontsize=16, weight='bold')
fig.tight_layout()

# Show plot
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load the data
file_3mers = "sample_kmers_count.csv"
file_4mers = "sample_4mers_count.csv"

data_3mers = pd.read_csv(file_3mers)
data_4mers = pd.read_csv(file_4mers)

# Extract columns for plotting
age_3mers = data_3mers["Age"]
unique_3mers_count = data_3mers["Sample_Count"]

age_4mers = data_4mers["Age"]
unique_4mers_count = data_4mers["Unique_4mers_Count"]

# Plot for Unique 3-mers Count vs Age
plt.figure(figsize=(10, 6))
sns.scatterplot(x=age_3mers, y=unique_3mers_count, color="blue", alpha=0.7, label="Unique 3-mers")
sns.regplot(x=age_3mers, y=unique_3mers_count, scatter=False, color="blue", label="3-mers Trendline")
plt.title("Unique 3-mers Count vs Age", fontsize=16, weight="bold")
plt.xlabel("Age", fontsize=14)
plt.ylabel("Unique 3-mers Count", fontsize=14)
plt.legend(fontsize=12)
plt.grid(alpha=0.5)
plt.tight_layout()
plt.show()

# Plot for Unique 4-mers Count vs Age
plt.figure(figsize=(10, 6))
sns.scatterplot(x=age_4mers, y=unique_4mers_count, color="orange", alpha=0.7, label="Unique 4-mers")
sns.regplot(x=age_4mers, y=unique_4mers_count, scatter=False, color="orange", label="4-mers Trendline")
plt.title("Unique 4-mers Count vs Age", fontsize=16, weight="bold")
plt.xlabel("Age", fontsize=14)
plt.ylabel("Unique 4-mers Count", fontsize=14)
plt.legend(fontsize=12)
plt.grid(alpha=0.5)
plt.tight_layout()
plt.show()


In [ ]:
# Replot with separate y-scales for the unique 3-mers and 4-mers counts

fig, ax1 = plt.subplots(figsize=(12, 8))

# Plot Unique 3-mers Count
color1 = 'tab:blue'
ax1.set_xlabel('Age', fontsize=14)
ax1.set_ylabel('Unique 3-mers Count', color=color1, fontsize=14)
ax1.scatter(unique_3mers_count_data['Age'], unique_3mers_count_data['Sample_Count'],
            alpha=0.6, label='Unique 3-mers Count', color=color1)
ax1.tick_params(axis='y', labelcolor=color1)
ax1.legend(loc='upper left', fontsize=12)

# Add secondary y-axis for Unique 4-mers Count
ax2 = ax1.twinx()
color2 = 'tab:orange'
ax2.set_ylabel('Unique 4-mers Count', color=color2, fontsize=14)
ax2.scatter(unique_4mers_count_data['Age'], unique_4mers_count_data['Unique_4mers_Count'],
            alpha=0.6, label='Unique 4-mers Count', color=color2)
ax2.tick_params(axis='y', labelcolor=color2)
ax2.legend(loc='upper right', fontsize=12)

# Add title
plt.title('Unique 3-mers and 4-mers Count vs Age', fontsize=16, weight='bold')

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, spearmanr, kendalltau

# Step 1: Load the merged dataset
merged_file_path = 'merged_graph_metrics_with_metadata.csv'
merged_df = pd.read_csv(merged_file_path)

# Step 2: Identify numeric columns for correlation
numeric_columns = merged_df.select_dtypes(include=['float64', 'int64']).columns
metrics_columns = [col for col in numeric_columns if col not in ['Age']]

# Step 3: Clean the data by removing NaN or Inf values
merged_df_cleaned = merged_df.replace([np.inf, -np.inf], np.nan).dropna(subset=metrics_columns + ['Age'])

# Step 4: Scatter plots for all metrics against Age in a single figure
fig, axes = plt.subplots(nrows=5, ncols=2, figsize=(15, 20))  # Adjust grid size as needed
axes = axes.flatten()
for i, metric in enumerate(metrics_columns):
    sns.scatterplot(data=merged_df_cleaned, x=metric, y='Age', alpha=0.7, ax=axes[i])
    axes[i].set_title(f'{metric} vs Age', fontsize=10)
    axes[i].set_xlabel(metric)
    axes[i].set_ylabel('Age')
plt.tight_layout()
plt.show()

# Step 5: Calculate correlations and p-values for each metric
results = []
for metric in metrics_columns:
    pearson_corr, pearson_p = pearsonr(merged_df_cleaned[metric], merged_df_cleaned['Age'])
    spearman_corr, spearman_p = spearmanr(merged_df_cleaned[metric], merged_df_cleaned['Age'])
    kendall_corr, kendall_p = kendalltau(merged_df_cleaned[metric], merged_df_cleaned['Age'])
    results.append({
        'Metric': metric,
        'Pearson_Corr': pearson_corr,
        'Pearson_P': pearson_p,
        'Spearman_Corr': spearman_corr,
        'Spearman_P': spearman_p,
        'Kendall_Corr': kendall_corr,
        'Kendall_P': kendall_p
    })

# Convert results to a DataFrame
correlation_results = pd.DataFrame(results)

# Step 6: Plot correlations for each method in a combined vertical bar plot
methods = ['Pearson', 'Spearman', 'Kendall']

# Plot correlation coefficients
plt.figure(figsize=(10, 15))
for i, method in enumerate(methods):
    data = correlation_results.sort_values(by=f'{method}_Corr', key=abs, ascending=False)
    plt.barh(data['Metric'], data[f'{method}_Corr'], label=f'{method} Correlation', alpha=0.7)
plt.axvline(x=0, color='gray', linestyle='--')
plt.title('Correlation Coefficients with Age')
plt.xlabel('Correlation Coefficient')
plt.ylabel('Metrics')
plt.legend()
plt.tight_layout()
plt.show()

# Plot p-values
plt.figure(figsize=(10, 15))
for i, method in enumerate(methods):
    data = correlation_results.sort_values(by=f'{method}_P', ascending=True)
    plt.barh(data['Metric'], data[f'{method}_P'], label=f'{method} P-Value', alpha=0.7)
plt.axvline(x=0.05, color='red', linestyle='--', label='p = 0.05')
plt.title('P-Values of Correlations with Age')
plt.xlabel('P-Value')
plt.ylabel('Metrics')
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, spearmanr, kendalltau

# Step 1: Load the merged dataset
merged_file_path = 'merged_graph_metrics_with_metadata.csv'
merged_df = pd.read_csv(merged_file_path)

# Step 2: Identify numeric columns for correlation
numeric_columns = merged_df.select_dtypes(include=['float64', 'int64']).columns
metrics_columns = [col for col in numeric_columns if col not in ['Age']]

# Step 3: Clean the data by removing NaN or Inf values
merged_df_cleaned = merged_df.replace([np.inf, -np.inf], np.nan).dropna(subset=metrics_columns + ['Age'])

# Step 4: Divide scatter plots into two figures
scatter_groups = [metrics_columns[:6], metrics_columns[6:]]

for idx, group in enumerate(scatter_groups):
    n_metrics = len(group)
    n_cols = 3
    n_rows = (n_metrics + n_cols - 1) // n_cols

    fig, axes = plt.subplots(nrows=n_rows, ncols=n_cols, figsize=(15, n_rows * 5))
    axes = axes.flatten()

    for i, metric in enumerate(group):
        sns.scatterplot(data=merged_df_cleaned, x=metric, y='Age', alpha=0.7, ax=axes[i])
        axes[i].set_title(f'{metric} vs Age', fontsize=10)
        axes[i].set_xlabel(metric)
        axes[i].set_ylabel('Age')

    for j in range(i + 1, len(axes)):
        axes[j].set_visible(False)

    plt.tight_layout()
    plt.suptitle(f'Scatter Plots', fontsize=16, y=1.02)
    plt.show()

# Step 5: Calculate correlations and p-values for each metric
results = []
for metric in metrics_columns:
    pearson_corr, pearson_p = pearsonr(merged_df_cleaned[metric], merged_df_cleaned['Age'])
    spearman_corr, spearman_p = spearmanr(merged_df_cleaned[metric], merged_df_cleaned['Age'])
    kendall_corr, kendall_p = kendalltau(merged_df_cleaned[metric], merged_df_cleaned['Age'])
    results.append({
        'Metric': metric,
        'Pearson_Corr': pearson_corr,
        'Pearson_P': pearson_p,
        'Spearman_Corr': spearman_corr,
        'Spearman_P': spearman_p,
        'Kendall_Corr': kendall_corr,
        'Kendall_P': kendall_p
    })

# Convert results to a DataFrame
correlation_results = pd.DataFrame(results)

# Step 6: Plot correlation coefficients and p-values
methods = ['Pearson', 'Spearman', 'Kendall']
fig, axes = plt.subplots(nrows=3, ncols=1, figsize=(10, 18))

for i, method in enumerate(methods):
    # Correlation coefficients
    data_corr = correlation_results.sort_values(by=f'{method}_Corr', key=abs, ascending=False)
    sns.barplot(
        data=data_corr, x=f'{method}_Corr', y='Metric', palette='viridis', ax=axes[i]
    )
    for j, val in enumerate(data_corr[f'{method}_P']):
        axes[i].text(
            x=data_corr[f'{method}_Corr'].iloc[j],
            y=j,
            s=f"p={val:.1e}",
            va='center',
            ha='left' if data_corr[f'{method}_Corr'].iloc[j] > 0 else 'right',
            color='black'
        )
    axes[i].axvline(x=0, color='gray', linestyle='--')
    axes[i].set_title(f'{method} Correlations with Age', fontsize=14)
    axes[i].set_xlabel('Correlation Coefficient')
    axes[i].set_ylabel('Metric')

plt.tight_layout()
plt.show()


In [ ]:
# Plot correlation coefficients and p-values with p-values on the side
methods = ['Pearson', 'Spearman', 'Kendall']
fig, axes = plt.subplots(nrows=3, ncols=1, figsize=(12, 18))

for i, method in enumerate(methods):
    # Correlation coefficients
    data_corr = correlation_results.sort_values(by=f'{method}_Corr', key=abs, ascending=False)
    sns.barplot(
        data=data_corr, x=f'{method}_Corr', y='Metric', palette='viridis', ax=axes[i]
    )

    for j, val in enumerate(data_corr[f'{method}_P']):
        # Add p-value annotations on the side
        axes[i].text(
            x=max(data_corr[f'{method}_Corr']) + 0.02,  # Adjust to place text outside the bar area
            y=j,
            s=f"p = {val:.1e}",
            va='center',
            fontsize=10,
            color='black',
            alpha=0.9
        )
    axes[i].axvline(x=0, color='gray', linestyle='--')
    axes[i].set_title(f'{method} Correlations with Age', fontsize=16)
    axes[i].set_xlabel('Correlation Coefficient')
    axes[i].set_ylabel('Metric')

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Create the DataFrame manually based on the provided data
data = {
    "file": [
        "tcrWeight_area_top_train_tcrs_across_samples.csv",
        "tcrWeight_area_top_train_tcrs_across_samples.csv",
        "tcrWeight_area_top_train_tcrs_across_samples.csv",
        "publicity_top_train_tcrs_across_samples.csv",
        "publicity_top_train_tcrs_across_samples.csv",
        "publicity_top_train_tcrs_across_samples.csv",
        "average_count_top_train_tcrs_across_samples.csv",
        "average_count_top_train_tcrs_across_samples.csv",
        "average_count_top_train_tcrs_across_samples.csv",
        "variance_score_top_train_tcrs_across_samples.csv",
        "variance_score_top_train_tcrs_across_samples.csv",
        "variance_score_top_train_tcrs_across_samples.csv",
    ],
    "feature_size": [1000, 2500, 5000, 1000, 2500, 5000, 1000, 2500, 5000, 1000, 2500, 5000],
    "rmse": [
        12.08205011,
        12.01472772,
        12.01472772,
        14.92401039,
        13.26151399,
        13.26151399,
        13.3624506,
        12.90387376,
        12.90387376,
        15.32167983,
        14.2619534,
        14.2619534,
    ],
    "mae": [
        9.531766887,
        9.556514609,
        9.556514609,
        12.04684638,
        10.66947607,
        10.66947607,
        10.63602788,
        10.25913987,
        10.25913987,
        12.34967906,
        11.46739258,
        11.46739258,
    ],
}

df = pd.DataFrame(data)

# Rename 2500 to 1500
df['feature_size'] = df['feature_size'].replace({2500: 1500})

# Extract the first word of the file name for method names
df['method'] = df['file'].apply(lambda x: x.split('_')[0])

# Filter only the feature sizes 1000 and 1500
filtered_df = df[df['feature_size'].isin([1000, 1500])]

# Plot MAE for each method
plt.figure(figsize=(10, 6))
for method in filtered_df['method'].unique():
    method_data = filtered_df[filtered_df['method'] == method]
    plt.plot(method_data['feature_size'], method_data['mae'], marker='o', label=method.capitalize())

plt.title("MAE vs Feature Size for Different Methods")
plt.xlabel("Feature Size")
plt.ylabel("Mean Absolute Error (MAE)")
plt.xticks([1000, 1500])
plt.grid(True)
plt.legend(title="Method")
plt.tight_layout()
plt.show()


In [ ]:
# Create a bar plot for MAE by feature size and method
plt.figure(figsize=(10, 6))
for method in filtered_df['method'].unique():
    method_data = filtered_df[filtered_df['method'] == method]
    plt.bar(
        method_data['feature_size'].astype(str) + f" ({method.capitalize()})",
        method_data['mae'],
        label=method.capitalize()
    )

plt.title("MAE vs Feature Size for Different Methods")
plt.xlabel("Feature Size (Method)")
plt.ylabel("Mean Absolute Error (MAE)")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.xticks(rotation=45, ha='right')
plt.legend(title="Method", loc='upper right')
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Data for plotting
data = {
    "File": [
        "consistency_area_top_train_tcrs_across_samples.csv",
        "consistency_area_top_train_tcrs_across_samples.csv",
        "publicity_top_train_tcrs_across_samples.csv",
        "publicity_top_train_tcrs_across_samples.csv",
        "average_count_top_train_tcrs_across_samples.csv",
        "average_count_top_train_tcrs_across_samples.csv",
        "variance_score_top_train_tcrs_across_samples.csv",
        "variance_score_top_train_tcrs_across_samples.csv"
    ],
    "Feature_Size": [1000, 1500, 1000, 1500, 1000, 1500, 1000, 1500],
    "MAE": [
        10.87993909, 10.07271369, 11.12405509, 11.72507547,
        11.64762664, 11.64762664, 13.08059808, 13.08059808
    ],
    "Method": [
        "TCR Weight", "TCR Weight", "Publicity", "Publicity",
        "Average", "Average", "Variance", "Variance"
    ]
}

# Convert to DataFrame
df = pd.DataFrame(data)

# Plot
plt.figure(figsize=(12, 6))
sns.barplot(data=df, x="Feature_Size", y="MAE", hue="Method", palette="viridis")
plt.title("Comparison of MAE by Feature Size and Method", fontsize=16, weight="bold")
plt.ylabel("Mean Absolute Error (MAE)", fontsize=12)
plt.xlabel("Feature Size", fontsize=12)
plt.ylim(0, max(df["MAE"]) + 2)

# Add labels
for index, row in df.iterrows():
    plt.text(
        x=row["Feature_Size"] + (index // 2) * 0.3 - 0.15, y=row["MAE"] + 0.2,
        s=f"{row['MAE']:.2f}",
        ha="center", fontsize=10
    )

plt.legend(title="Method")
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Data for plotting
data = {
    "File": [
        "consistency_area_top_train_tcrs_across_samples.csv",
        "consistency_area_top_train_tcrs_across_samples.csv",
        "publicity_top_train_tcrs_across_samples.csv",
        "publicity_top_train_tcrs_across_samples.csv",
        "average_count_top_train_tcrs_across_samples.csv",
        "average_count_top_train_tcrs_across_samples.csv",
        "variance_score_top_train_tcrs_across_samples.csv",
        "variance_score_top_train_tcrs_across_samples.csv"
    ],
    "Feature_Size": [1000, 1500, 1000, 1500, 1000, 1500, 1000, 1500],
    "MAE": [
        10.87993909, 10.07271369, 11.12405509, 11.72507547,
        11.64762664, 11.64762664, 13.08059808, 13.08059808
    ],
    "Method": [
        "TCR Weight", "TCR Weight", "Publicity", "Publicity",
        "Average", "Average", "Variance", "Variance"
    ]
}

df = pd.DataFrame(data)

# Define colors for methods
colors = {
    'TCR Weight': '#2a9d8f',  # Teal
    'Publicity': '#264653',  # Dark Blue
    'Average': '#e76f51',  # Orange
    'Variance': '#f4a261'  # Soft Orange
}

# Unique feature sizes and methods
feature_sizes = sorted(df['Feature_Size'].unique())
methods = df['Method'].unique()
bar_positions = np.arange(len(feature_sizes))

# Set width for bars
width = 0.2

# Create the plot
plt.figure(figsize=(10, 6))

for i, method in enumerate(methods):
    method_data = df[df['Method'] == method]
    plt.bar(
        bar_positions + i * width - width * (len(methods) / 2),  # Adjust bar positions
        method_data['MAE'],
        width=width,
        label=method,
        color=colors[method]
    )
    # Annotate the bars
    for j, mae in enumerate(method_data['MAE']):
        plt.text(
            bar_positions[j] + i * width - width * (len(methods) / 2),
            mae + 0.1,
            f"{mae:.2f}",
            ha='center',
            va='bottom',
            fontsize=9
        )

# Customizations
plt.xticks(bar_positions, [str(size) for size in feature_sizes], fontsize=10)
plt.xlabel("Feature Size", fontsize=12)
plt.ylabel("Mean Absolute Error (MAE)", fontsize=12)
plt.title("Comparison of MAE by Feature Size and Method", fontsize=14)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.legend(title="Method", fontsize=10, loc='right')
plt.tight_layout()

# Display the plot
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Data for plotting
data = {
    "File": [
        "consistency_area_top_train_tcrs_across_samples.csv",
        "consistency_area_top_train_tcrs_across_samples.csv",
        "consistency_area_top_train_tcrs_across_samples.csv"
    ],
    "Feature_Size": [1000, 1000, 1000],
    "Split": ["33-67", "50-50", "75-25"],
    "MAE": [10.423824615596617, 10.137606891923589, 10.87993909]
}

df = pd.DataFrame(data)

# Define colors for splits
colors = {
    '33-67': '#2a9d8f',  # Teal
    '50-50': '#264653',  # Dark Blue
    '75-25': '#e76f51'   # Orange
}

# Bar positions and width
splits = df['Split'].unique()
bar_positions = np.arange(len(splits))
width = 0.4

# Create the plot
plt.figure(figsize=(8, 5))

for i, split in enumerate(splits):
    split_data = df[df['Split'] == split]
    plt.bar(
        bar_positions[i],
        split_data['MAE'].values[0],
        width=width,
        label=f"Split {split}",
        color=colors[split]
    )
    # Annotate the bars
    plt.text(
        bar_positions[i],
        split_data['MAE'].values[0] + 0.1,
        f"{split_data['MAE'].values[0]:.2f}",
        ha='center',
        va='bottom',
        fontsize=9
    )

# Customizations
plt.xticks(bar_positions, splits, fontsize=10)
plt.xlabel("Split Ratio", fontsize=12)
plt.ylabel("Mean Absolute Error (MAE)", fontsize=12)
plt.title("Comparison of MAE for Different Splits (Consistency, 1000 Features)", fontsize=14)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.legend(title="Split", fontsize=10, loc='right')
plt.tight_layout()

# Display the plot
plt.show()


In [ ]:
import pandas as pd
import networkx as nx
from networkx.algorithms.community import louvain_communities, label_propagation_communities
from sklearn.cluster import DBSCAN
import numpy as np

# Step 1: Load the Data
file_path = '/mnt/data/train_tcrs_across_samples_10K_cloness.csv'  # Adjust path as needed
data = pd.read_csv(file_path)

# Drop non-TCR columns
tcr_columns = [col for col in data.columns if col not in ['sample name', 'Biological Sex', 'Age']]
tcr_data = data[tcr_columns]

# Step 2: Construct the Graph
# Create an undirected graph
graph = nx.Graph()

# Add nodes (TCRs)
graph.add_nodes_from(tcr_columns)

# Add edges based on co-occurrence across samples
for index, row in tcr_data.iterrows():
    present_tcrs = row[row > 0].index  # Only consider TCRs with non-zero values
    graph.add_edges_from([(tcr1, tcr2) for tcr1, tcr2 in combinations(present_tcrs, 2)])

print(f"Graph constructed with {graph.number_of_nodes()} nodes and {graph.number_of_edges()} edges.")

# Step 3: Apply Clustering Methods

# 1. **Louvain Method**
louvain_clusters = louvain_communities(graph)
print("\nLouvain Communities:")
for i, community in enumerate(louvain_clusters):
    print(f"Community {i + 1}: {len(community)} TCRs")

# 2. **Label Propagation**
label_propagation_clusters = list(label_propagation_communities(graph))
print("\nLabel Propagation Communities:")
for i, community in enumerate(label_propagation_clusters):
    print(f"Community {i + 1}: {len(community)} TCRs")

# 3. **DBSCAN**
# Convert graph adjacency matrix to distance matrix (1 - adjacency)
adj_matrix = nx.to_numpy_array(graph)
distance_matrix = 1 - adj_matrix  # Convert similarity to distance

# Apply DBSCAN
dbscan = DBSCAN(eps=0.5, min_samples=2, metric="precomputed")
dbscan_labels = dbscan.fit_predict(distance_matrix)

# Extract DBSCAN clusters
dbscan_clusters = {}
for tcr, label in zip(graph.nodes, dbscan_labels):
    if label not in dbscan_clusters:
        dbscan_clusters[label] = []
    dbscan_clusters[label].append(tcr)

print("\nDBSCAN Clusters:")
for label, cluster in dbscan_clusters.items():
    cluster_type = "Noise" if label == -1 else f"Cluster {label + 1}"
    print(f"{cluster_type}: {len(cluster)} TCRs")

# 4. **Connected Components**
connected_components_clusters = list(nx.connected_components(graph))
print("\nConnected Components:")
for i, component in enumerate(connected_components_clusters):
    print(f"Component {i + 1}: {len(component)} TCRs")

# Step 4: Save or Output Results
# Save clusters for each method
with open("louvain_clusters.txt", "w") as f:
    for i, community in enumerate(louvain_clusters):
        f.write(f"Community {i + 1}: {', '.join(community)}\n")

with open("label_propagation_clusters.txt", "w") as f:
    for i, community in enumerate(label_propagation_clusters):
        f.write(f"Community {i + 1}: {', '.join(community)}\n")

with open("dbscan_clusters.txt", "w") as f:
    for label, cluster in dbscan_clusters.items():
        cluster_type = "Noise" if label == -1 else f"Cluster {label + 1}"
        f.write(f"{cluster_type}: {', '.join(cluster)}\n")

with open("connected_components.txt", "w") as f:
    for i, component in enumerate(connected_components_clusters):
        f.write(f"Component {i + 1}: {', '.join(component)}\n")


In [ ]:
import os
import csv
from itertools import combinations
from collections import defaultdict
import pandas as pd
import networkx as nx
from networkx.algorithms.community import louvain_communities, label_propagation_communities
from tqdm import tqdm

# Define file paths
input_file_path = 'train_tcrs_across_samples_10K_cloness.csv'
output_dir = os.path.dirname(input_file_path)

# Output file paths for clusters
hashed_substring_clusters_file = os.path.join(output_dir, "hashed_substring_clusters.txt")
louvain_clusters_file = os.path.join(output_dir, "louvain_clusters.txt")
label_propagation_clusters_file = os.path.join(output_dir, "label_propagation_clusters.txt")
dbscan_clusters_file = os.path.join(output_dir, "dbscan_clusters.txt")
connected_components_clusters_file = os.path.join(output_dir, "connected_components_clusters.txt")

# Step 1: Load Data and Select TCR Columns
data = pd.read_csv(input_file_path)
columns_to_exclude = ['sample name', 'Biological Sex', 'Age']
tcr_columns = [col for col in data.columns if col not in columns_to_exclude]  # Select only TCR columns
sequences = tcr_columns  # TCR sequences are the column names
print(sequences)
# Step 2: Generate Hashed Substrings and Build Graph
def generate_hashed_substrings(sequence):
    for substring in combinations(sequence, len(sequence) - 1):
        yield hash(substring), sequence

# Group sequences by length
sequences_by_length = defaultdict(list)
for seq in sequences:
    if isinstance(seq, str) and len(seq) > 1:
        sequences_by_length[len(seq)].append(seq)

graph = nx.Graph()
edge_list = []

for length, seq_group in sorted(sequences_by_length.items(), reverse=True):
    print(f"Processing sequences of length {length}...")
    substring_map = defaultdict(list)

    # Map sequences by their hashed substrings
    for seq in seq_group:
        for hashed_substring, sequence in generate_hashed_substrings(seq):
            substring_map[hashed_substring].append(sequence)

    # Generate edges from hashed substring groups
    for group in substring_map.values():
        if len(group) > 1:
            edge_list.extend(combinations(group, 2))

# Add edges to the graph
graph.add_edges_from(edge_list)
print(f"Graph constructed with {graph.number_of_nodes()} nodes and {graph.number_of_edges()} edges.")

# Step 3: Hashed Substring Clustering (Cliques and Components)
# Find cliques
print("Finding cliques...")
cliques = list(nx.find_cliques(graph))
with open(hashed_substring_clusters_file, "w") as f:
    for i, clique in enumerate(cliques):
        f.write(f"Clique {i + 1}: {', '.join(clique)}\n")

# Step 4: Apply Clustering Methods
# 1. Louvain Method
print("Applying Louvain Method...")
louvain_clusters = louvain_communities(graph)
with open(louvain_clusters_file, "w") as f:
    for i, community in enumerate(louvain_clusters):
        f.write(f"Community {i + 1}: {', '.join(community)}\n")

# 2. Label Propagation
print("Applying Label Propagation...")
label_propagation_clusters = list(label_propagation_communities(graph))
with open(label_propagation_clusters_file, "w") as f:
    for i, community in enumerate(label_propagation_clusters):
        f.write(f"Community {i + 1}: {', '.join(community)}\n")

# 3. DBSCAN
print("Applying DBSCAN...")
adj_matrix = nx.to_numpy_array(graph)
distance_matrix = 1 - adj_matrix
from sklearn.cluster import DBSCAN

dbscan = DBSCAN(eps=0.5, min_samples=2, metric="precomputed")
dbscan_labels = dbscan.fit_predict(distance_matrix)
dbscan_clusters = defaultdict(list)
for node, label in zip(graph.nodes, dbscan_labels):
    dbscan_clusters[label].append(node)

with open(dbscan_clusters_file, "w") as f:
    for label, cluster in dbscan_clusters.items():
        cluster_type = "Noise" if label == -1 else f"Cluster {label + 1}"
        f.write(f"{cluster_type}: {', '.join(cluster)}\n")

# 4. Connected Components
print("Finding Connected Components...")
connected_components = list(nx.connected_components(graph))
with open(connected_components_clusters_file, "w") as f:
    for i, component in enumerate(connected_components):
        f.write(f"Component {i + 1}: {', '.join(component)}\n")

print("Clustering completed and results saved.")


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

# Define file paths
output_dir = ''  # Replace with your directory
hashed_substring_clusters_file = os.path.join(output_dir, "hashed_substring_clusters.txt")
louvain_clusters_file = os.path.join(output_dir, "louvain_clusters.txt")
label_propagation_clusters_file = os.path.join(output_dir, "label_propagation_clusters.txt")
dbscan_clusters_file = os.path.join(output_dir, "dbscan_clusters.txt")
connected_components_clusters_file = os.path.join(output_dir, "connected_components_clusters.txt")

# Define a function to parse cluster files
def parse_cluster_file(file_path):
    with open(file_path) as f:
        clusters = f.readlines()
    cluster_sizes = [len(line.split(":")[1].split(", ")) for line in clusters]
    return cluster_sizes

# 1. Analyze Hashed Substring Clustering
print("\nAnalyzing Hashed Substring Clustering...")
hashed_clique_sizes = parse_cluster_file(hashed_substring_clusters_file)
print(f"Total number of cliques: {len(hashed_clique_sizes)}")
print(f"Average clique size: {np.mean(hashed_clique_sizes):.2f}")
print(f"Largest clique size: {max(hashed_clique_sizes)}")

# 2. Analyze Louvain Clustering
print("\nAnalyzing Louvain Clustering...")
louvain_community_sizes = parse_cluster_file(louvain_clusters_file)
print(f"Total number of communities: {len(louvain_community_sizes)}")
print(f"Average community size: {np.mean(louvain_community_sizes):.2f}")
print(f"Largest community size: {max(louvain_community_sizes)}")

# 3. Analyze Label Propagation Clustering
print("\nAnalyzing Label Propagation Clustering...")
label_cluster_sizes = parse_cluster_file(label_propagation_clusters_file)
print(f"Total number of label clusters: {len(label_cluster_sizes)}")
print(f"Average cluster size: {np.mean(label_cluster_sizes):.2f}")
print(f"Largest label cluster size: {max(label_cluster_sizes)}")

# 4. Analyze DBSCAN Clustering
print("\nAnalyzing DBSCAN Clustering...")
with open(dbscan_clusters_file) as f:
    dbscan_data = f.readlines()

dbscan_cluster_sizes = [len(line.split(":")[1].split(", ")) for line in dbscan_data if "Noise" not in line]
noise_size = sum(len(line.split(":")[1].split(", ")) for line in dbscan_data if "Noise" in line)
print(f"Total number of DBSCAN clusters: {len(dbscan_cluster_sizes)}")
print(f"Average cluster size: {np.mean(dbscan_cluster_sizes):.2f}")
print(f"Largest DBSCAN cluster size: {max(dbscan_cluster_sizes)}")
print(f"Number of noise sequences: {noise_size}")

# 5. Analyze Connected Components Clustering
print("\nAnalyzing Connected Components Clustering...")
connected_component_sizes = parse_cluster_file(connected_components_clusters_file)
print(f"Total number of connected components: {len(connected_component_sizes)}")
print(f"Average component size: {np.mean(connected_component_sizes):.2f}")
print(f"Largest connected component size: {max(connected_component_sizes)}")

# Visualization of Cluster Size Distributions
plt.figure(figsize=(12, 8))
plt.hist(hashed_clique_sizes, bins=30, alpha=0.6, label='Hashed Substrings', color='blue')
plt.hist(louvain_community_sizes, bins=30, alpha=0.6, label='Louvain', color='green')
plt.hist(label_cluster_sizes, bins=30, alpha=0.6, label='Label Propagation', color='orange')
plt.hist(dbscan_cluster_sizes, bins=30, alpha=0.6, label='DBSCAN', color='purple')
plt.hist(connected_component_sizes, bins=30, alpha=0.6, label='Connected Components', color='red')
plt.xlabel("Cluster Size")
plt.ylabel("Frequency")
plt.title("Cluster Size Distribution by Method")
plt.legend()
plt.show()


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

# Define file paths
output_dir = ''  # Replace with your directory
hashed_substring_clusters_file = os.path.join(output_dir, "hashed_substring_clusters.txt")
louvain_clusters_file = os.path.join(output_dir, "louvain_clusters.txt")
label_propagation_clusters_file = os.path.join(output_dir, "label_propagation_clusters.txt")
dbscan_clusters_file = os.path.join(output_dir, "dbscan_clusters.txt")
connected_components_clusters_file = os.path.join(output_dir, "connected_components_clusters.txt")

# Define a function to parse cluster files
def parse_cluster_file(file_path):
    with open(file_path) as f:
        clusters = f.readlines()
    cluster_sizes = [len(line.split(":")[1].split(", ")) for line in clusters]
    return cluster_sizes

# Load cluster sizes for each method
hashed_clique_sizes = parse_cluster_file(hashed_substring_clusters_file)
louvain_community_sizes = parse_cluster_file(louvain_clusters_file)
label_cluster_sizes = parse_cluster_file(label_propagation_clusters_file)

with open(dbscan_clusters_file) as f:
    dbscan_data = f.readlines()
dbscan_cluster_sizes = [len(line.split(":")[1].split(", ")) for line in dbscan_data if "Noise" not in line]

connected_component_sizes = parse_cluster_file(connected_components_clusters_file)

# Function to generate random scatter plot positions
def random_scatter_positions(n_points, x_range, y_range):
    x = np.random.uniform(x_range[0], x_range[1], n_points)
    y = np.random.uniform(y_range[0], y_range[1], n_points)
    return x, y

# Plot for Each Method
plt.figure(figsize=(15, 10))

# 1. Hashed Substrings
plt.subplot(2, 3, 1)
x, y = random_scatter_positions(len(hashed_clique_sizes), (0, 1), (0, 1))
plt.scatter(x, y, s=hashed_clique_sizes, alpha=0.6, color='blue')
plt.title("Hashed Substrings Clustering")
plt.axis('off')

# 2. Louvain
plt.subplot(2, 3, 2)
x, y = random_scatter_positions(len(louvain_community_sizes), (0, 1), (0, 1))
plt.scatter(x, y, s=louvain_community_sizes, alpha=0.6, color='green')
plt.title("Louvain Clustering")
plt.axis('off')

# 3. Label Propagation
plt.subplot(2, 3, 3)
x, y = random_scatter_positions(len(label_cluster_sizes), (0, 1), (0, 1))
plt.scatter(x, y, s=label_cluster_sizes, alpha=0.6, color='orange')
plt.title("Label Propagation Clustering")
plt.axis('off')

# 4. DBSCAN
plt.subplot(2, 3, 4)
x, y = random_scatter_positions(len(dbscan_cluster_sizes), (0, 1), (0, 1))
plt.scatter(x, y, s=dbscan_cluster_sizes, alpha=0.6, color='purple')
plt.title("DBSCAN Clustering")
plt.axis('off')

# 5. Connected Components
plt.subplot(2, 3, 5)
x, y = random_scatter_positions(len(connected_component_sizes), (0, 1), (0, 1))
plt.scatter(x, y, s=connected_component_sizes, alpha=0.6, color='red')
plt.title("Connected Components Clustering")
plt.axis('off')

# Layout adjustment
plt.tight_layout()
plt.show()


up to here code for new analysis

In [ ]:
import pandas as pd
import numpy as np
from scipy.spatial.distance import pdist, squareform
from scipy.cluster.hierarchy import linkage, dendrogram
import matplotlib.pyplot as plt

# Step 1: Load the dataset and extract 'publicity_top' sequences
file_path = "final_tcr_selection_adjusted_distribution_with_random_groups.csv"
df = pd.read_csv(file_path)

# Filter for the 'publicity_top' group
publicity_top_df = df[df['selection_group'] == 'publicity_top']
tcr_sequences = publicity_top_df['amino_acid'].dropna().unique().tolist()

# Step 2: Calculate pairwise Hamming distances
def hamming_distance(seq1, seq2):
    if len(seq1) != len(seq2):
        return float('inf')  # Assign infinite distance for sequences of different lengths
    return sum(el1 != el2 for el1, el2 in zip(seq1, seq2))

# Compute pairwise distances, ignoring sequences of different lengths
valid_sequences = [seq for seq in tcr_sequences if len(seq) == len(tcr_sequences[0])]
distance_matrix = np.array([
    [hamming_distance(seq1, seq2) for seq2 in valid_sequences] for seq1 in valid_sequences
])

# Step 3: Perform hierarchical clustering
linked = linkage(squareform(distance_matrix), method='average')

# Step 4: Plot the dendrogram
plt.figure(figsize=(35, 10))
dendrogram(linked, labels=valid_sequences, leaf_rotation=90, leaf_font_size=10)
plt.title("TCR Sequence Similarity Tree: Publicity Top Group")
plt.xlabel("TCR Sequences")
plt.ylabel("Distance (Hamming)")
plt.tight_layout()

# Save the dendrogram
output_dendrogram = "publicity_top_tcr_similarity_tree.png"
plt.savefig(output_dendrogram)
plt.show()

print(f"Dendrogram saved to {output_dendrogram}")


In [ ]:
# Step 4: Plot the dendrogram
plt.figure(figsize=(35, 10))
dendrogram(linked, labels=valid_sequences, leaf_rotation=90, leaf_font_size=10)
plt.title("TCR Sequence Similarity Tree: Publicity Top Group")
plt.xlabel("TCR Sequences")
plt.ylabel("Distance (Hamming)")
plt.tight_layout()

# Save the dendrogram
output_dendrogram = "publicity_top_tcr_similarity_tree.png"
plt.savefig(output_dendrogram)
plt.show()

In [ ]:
pip install logomaker

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import logomaker

# File paths
file_path = "final_tcr_selection_adjusted_distribution_with_random_groups.csv"
output_dir = "logos_by_selection_group"

# Create output directory
os.makedirs(output_dir, exist_ok=True)

# Load the dataset
df = pd.read_csv(file_path)

# Drop rows with missing or invalid sequences
df = df.dropna(subset=['amino_acid'])

# Function to calculate amino acid frequencies for each position
def calculate_position_frequencies(sequences):
    max_length = max(len(seq) for seq in sequences)
    amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
    position_counts = [{aa: 0 for aa in amino_acids} for _ in range(max_length)]

    for seq in sequences:
        for i, aa in enumerate(seq):
            if aa in amino_acids:
                position_counts[i][aa] += 1

    # Convert counts to frequencies
    position_frequencies = []
    for counts in position_counts:
        total = sum(counts.values())
        if total > 0:
            frequencies = {aa: counts[aa] / total for aa in amino_acids}
        else:
            frequencies = {aa: 0 for aa in amino_acids}
        position_frequencies.append(frequencies)

    return pd.DataFrame(position_frequencies).fillna(0)

# Generate logos for each selection group
groups = df['selection_group'].unique()

for group in groups:
    print(f"Generating logo for selection group: {group}...")

    # Get sequences for the group
    group_sequences = df[df['selection_group'] == group]['amino_acid'].dropna().tolist()

    # Calculate position frequencies
    position_frequencies = calculate_position_frequencies(group_sequences)

    # Create the sequence logo
    plt.figure(figsize=(12, 6))
    logo = logomaker.Logo(position_frequencies, color_scheme='chemistry')
    plt.title(f"Sequence Logo: {group}")
    plt.xlabel("Position")
    plt.ylabel("Frequency")
    plt.tight_layout()

    # Save the logo
    output_file = os.path.join(output_dir, f"sequence_logo_{group}.png")
    plt.show()
    plt.close()
    print(f"Logo saved: {output_file}")

print(f"All logos have been saved in the directory: {output_dir}")


In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import logomaker

# File paths
file_path = "final_tcr_selection_adjusted_distribution_with_random_groups.csv"
output_dir = "logos_by_selection_group_and_length"

# Create output directory
os.makedirs(output_dir, exist_ok=True)

# Load the dataset
df = pd.read_csv(file_path)

# Drop rows with missing or invalid sequences
df = df.dropna(subset=['amino_acid'])

# Add a new column for sequence lengths
df['length'] = df['amino_acid'].apply(len)

# Filter for the desired groups and lengths
selected_groups = ['consistency_area_top', 'publicity_top', 'random_selection_1']
df = df[df['selection_group'].isin(selected_groups)]
df = df[(df['length'] >= 12) & (df['length'] <= 17)]

# Function to calculate amino acid frequencies for each position
def calculate_position_frequencies(sequences):
    max_length = max(len(seq) for seq in sequences)
    amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
    position_counts = [{aa: 0 for aa in amino_acids} for _ in range(max_length)]

    for seq in sequences:
        for i, aa in enumerate(seq):
            if aa in amino_acids:
                position_counts[i][aa] += 1

    # Convert counts to frequencies
    position_frequencies = []
    for counts in position_counts:
        total = sum(counts.values())
        if total > 0:
            frequencies = {aa: counts[aa] / total for aa in amino_acids}
        else:
            frequencies = {aa: 0 for aa in amino_acids}
        position_frequencies.append(frequencies)

    return pd.DataFrame(position_frequencies).fillna(0)

# Generate logos for each selection group and length
for group in selected_groups:
    group_df = df[df['selection_group'] == group]
    for length in range(12, 18):  # From length 12 to 17
        length_sequences = group_df[group_df['length'] == length]['amino_acid'].dropna().tolist()
        if len(length_sequences) == 0:
            print(f"No sequences for group '{group}' and length {length}")
            continue

        print(f"Generating logo for group '{group}', length {length}...")

        # Calculate position frequencies
        position_frequencies = calculate_position_frequencies(length_sequences)

        # Create the sequence logo
        plt.figure(figsize=(12, 6))
        logo = logomaker.Logo(position_frequencies, color_scheme='chemistry')
        plt.title(f"Sequence Logo: {group} (Length {length})")
        plt.xlabel("Position")
        plt.ylabel("Frequency")
        plt.tight_layout()

        # Save the logo
        output_file = os.path.join(output_dir, f"sequence_logo_{group}_length_{length}.png")
        plt.show()
        plt.close()
        print(f"Logo saved: {output_file}")

print(f"All logos have been saved in the directory: {output_dir}")


In [ ]:
# Filter data for the three selected groups
selected_groups = ['consistency_area_top', 'publicity_top', 'random_selection_1']
filtered_df = df[df['selection_group'].isin(selected_groups)]

# Calculate the sequence lengths
filtered_df['length'] = filtered_df['amino_acid'].apply(len)

# Plot the length distribution for each group
plt.figure(figsize=(12, 8))
for group in selected_groups:
    group_lengths = filtered_df[filtered_df['selection_group'] == group]['length']
    plt.hist(
        group_lengths,
        bins=range(group_lengths.min(), group_lengths.max() + 1),
        alpha=0.7,
        edgecolor='black',
        label=group
    )

# Customize the plot
plt.title("TCR Sequence Length Distribution by Group")
plt.xlabel("TCR Length")
plt.ylabel("Frequency")
plt.legend(title="Selection Group")
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()

# Display the plot
plt.show()


In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import logomaker

# File paths
file_path = "final_tcr_selection_adjusted_distribution_with_random_groups.csv"
output_dir = "logos_by_length_and_group"

# Create output directory
os.makedirs(output_dir, exist_ok=True)

# Load the dataset
df = pd.read_csv(file_path)

# Drop rows with missing or invalid sequences
df = df.dropna(subset=['amino_acid'])

# Add a new column for sequence lengths
df['length'] = df['amino_acid'].apply(len)

# Filter for the desired groups and lengths
selected_groups = ['consistency_area_top', 'publicity_top', 'random_selection_1']
df = df[df['selection_group'].isin(selected_groups)]
df = df[(df['length'] >= 12) & (df['length'] <= 17)]

# Function to calculate amino acid frequencies for each position
def calculate_position_frequencies(sequences):
    max_length = max(len(seq) for seq in sequences)
    amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
    position_counts = [{aa: 0 for aa in amino_acids} for _ in range(max_length)]

    for seq in sequences:
        for i, aa in enumerate(seq):
            if aa in amino_acids:
                position_counts[i][aa] += 1

    # Convert counts to frequencies
    position_frequencies = []
    for counts in position_counts:
        total = sum(counts.values())
        if total > 0:
            frequencies = {aa: counts[aa] / total for aa in amino_acids}
        else:
            frequencies = {aa: 0 for aa in amino_acids}
        position_frequencies.append(frequencies)

    return pd.DataFrame(position_frequencies).fillna(0)

# Generate logos for each length and selection group
for length in range(12, 18):  # From length 12 to 17
    length_df = df[df['length'] == length]
    if length_df.empty:
        print(f"No sequences for length {length}")
        continue

    print(f"Generating logos for length {length}...")

    for group in selected_groups:
        group_sequences = length_df[length_df['selection_group'] == group]['amino_acid'].dropna().tolist()
        if len(group_sequences) == 0:
            print(f"No sequences for group '{group}' at length {length}")
            continue

        # Calculate position frequencies
        position_frequencies = calculate_position_frequencies(group_sequences)

        # Create the sequence logo
        plt.figure(figsize=(12, 6))
        logo = logomaker.Logo(position_frequencies, color_scheme='chemistry')
        plt.title(f"Sequence Logo: Length {length} ({group})")
        plt.xlabel("Position")
        plt.ylabel("Frequency")
        plt.tight_layout()

        # Save the logo
        output_file = os.path.join(output_dir, f"sequence_logo_length_{length}_{group}.png")
        plt.show()
        plt.close()
        print(f"Logo saved: {output_file}")

print(f"All logos have been saved in the directory: {output_dir}")


In [ ]:
!pip install tcrdist3
!pip install parmap


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tcrdist.pgen import OlgaModel
from tcrdist.adpt_funcs import _valid_cdr3

# Initialize OLGA model for beta chains
olga_beta = OlgaModel(chain_folder="human_T_beta", recomb_type="VDJ")

# Load the data
file_path = "final_tcr_selection_adjusted_distribution_with_random_groups.csv"
df = pd.read_csv(file_path)

# Validate and compute Pgen for each sequence
df['valid'] = df['amino_acid'].apply(_valid_cdr3)  # Mark valid CDR3 sequences
valid_df = df[df['valid']].copy()  # Filter valid sequences

# Compute Pgen values
print("Computing Pgen values...")
valid_df['pgen'] = valid_df['amino_acid'].apply(olga_beta.compute_aa_cdr3_pgen)

# Group data by 'selection_group' and plot distributions
selection_groups = valid_df['selection_group'].unique()

# Initialize the plot
plt.figure(figsize=(12, 8))

# Plot the distribution for each group
for group in selection_groups:
    group_data = valid_df[valid_df['selection_group'] == group]['pgen']
    sns.kdeplot(
        group_data,
        label=f"Group: {group}",
        bw_adjust=0.5  # Adjust bandwidth for smoother curves
    )

# Customize the plot
plt.title("Pgen Distribution by Selection Group")
plt.xlabel("Pgen")
plt.ylabel("Density")
plt.legend()
plt.tight_layout()

# Save the plot
plt.savefig("pgen_distribution_by_selection_group.png")
plt.show()

# Save results to a CSV file for further analysis
valid_df[['amino_acid', 'selection_group', 'pgen']].to_csv("pgen_results_by_group.csv", index=False)

print("Pgen computation completed and results saved.")


In [ ]:
plt.figure(figsize=(12, 8))
sns.boxplot(data=valid_df, x='selection_group', y='pgen')
plt.title("Pgen Distribution by Selection Group")
plt.xlabel("Selection Group")
plt.ylabel("Pgen")
plt.xticks(rotation=45, ha='right')  # Rotate x-axis labels for readability
plt.tight_layout()
plt.savefig("pgen_boxplot_by_selection_group.png")
plt.show()


In [ ]:
plt.figure(figsize=(12, 8))
sns.violinplot(data=valid_df, x='selection_group', y='pgen', inner="quartile")
plt.title("Pgen Distribution by Selection Group")
plt.xlabel("Selection Group")
plt.ylabel("Pgen")
plt.xticks(rotation=45, ha='right')  # Rotate x-axis labels for readability
plt.tight_layout()
plt.savefig("pgen_violinplot_by_selection_group.png")
plt.show()


In [ ]:
plt.figure(figsize=(12, 8))
sns.scatterplot(data=valid_df, x='average_count', y='pgen', hue='selection_group', alpha=0.7)
plt.title("Pgen vs Average Count by Selection Group")
plt.xlabel("Average Count")
plt.ylabel("Pgen")
plt.legend(title="Selection Group", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig("pgen_vs_average_count_scatterplot.png")
plt.show()


In [ ]:
# Filter out the specified groups
filtered_df = valid_df[~valid_df['selection_group'].isin(['consistency_area2_top', 'explosion_area_top'])]

# Pivot table for heatmap
heatmap_data = filtered_df.pivot_table(index='selection_group', values='pgen', aggfunc='mean')

# Create heatmap
plt.figure(figsize=(10, 6))
sns.heatmap(heatmap_data, annot=True, fmt=".2e", cmap="Blues", cbar_kws={'label': 'Mean Pgen'})
plt.title("Mean Pgen by Selection Group (Excluding Specific Groups)")
plt.xlabel("Mean Pgen")
plt.ylabel("Selection Group")
plt.tight_layout()
plt.savefig("pgen_heatmap_by_selection_group_filtered.png")
plt.show()


In [ ]:
sns.pairplot(data=valid_df, vars=['pgen', 'average_count', 'sample_appearance_count', 'consistency_area', 'variance_score'], hue='selection_group')
plt.savefig("pairplot_pgen_and_metrics.png")
plt.show()


In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(data=valid_df, x='pgen', hue='selection_group', kde=True, alpha=0.7, bins=30)
plt.title("Histogram of Pgen Across Selection Groups")
plt.xlabel("Pgen")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig("pgen_histogram_all_groups.png")
plt.show()


In [ ]:
median_pgen = valid_df.groupby('selection_group')['pgen'].median().reset_index()

plt.figure(figsize=(12, 8))
sns.barplot(data=median_pgen, x='selection_group', y='pgen', palette='viridis')
plt.title("Median Pgen by Selection Group")
plt.xlabel("Selection Group")
plt.ylabel("Median Pgen")
plt.xticks(rotation=45, ha='right')  # Rotate x-axis labels for readability
plt.tight_layout()
plt.savefig("pgen_median_by_selection_group.png")
plt.show()


In [ ]:
!pip install bio

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from Bio.SeqUtils.ProtParam import ProteinAnalysis
import numpy as np

# Function to load data and extract sequences
def load_and_calculate_frequencies(file_name, file_type):
    if file_type == 'xlsx':
        df = pd.read_excel(file_name)
    elif file_type == 'csv':
        df = pd.read_csv(file_name)

    # Assumes the sequence column is labeled 'TCR'
    return df['TCR']

# Load sequences from each file
high_freq_sequences = load_and_calculate_frequencies('co_occurance_high.xlsx', 'xlsx')
low_freq_sequences = load_and_calculate_frequencies('co_occurance_low.xlsx', 'xlsx')
negative_corr_sequences = load_and_calculate_frequencies('negative_corr_group.csv', 'csv')
positive_corr_sequences = load_and_calculate_frequencies('positive_corr_group.csv', 'csv')

# Define function to calculate properties for a given sequence
def calculate_properties(sequence):
    analysis = ProteinAnalysis(sequence)
    hydropathy = analysis.gravy()
    isoelectric_point = analysis.isoelectric_point()
    molecular_weight = analysis.molecular_weight()
    aliphatic_index = analysis.molecular_weight() / len(sequence) if len(sequence) > 0 else 0
    return hydropathy, isoelectric_point, molecular_weight, aliphatic_index

# Function to normalize data using min-max scaling
def normalize_data(values):
    min_val = np.min(values)
    max_val = np.max(values)
    return [(val - min_val) / (max_val - min_val) if max_val > min_val else val for val in values]

# Function to calculate and plot normalized properties for each TCR set, organized into subplots
def calculate_and_plot_properties_subplot_normalized(sequences_dict):
    properties = ['Hydropathy', 'Isoelectric Point', 'Molecular Weight', 'Aliphatic Index']
    fig, axes = plt.subplots(len(properties), 1, figsize=(10, 20))  # 4 rows for 4 properties

    for i, prop in enumerate(properties):
        for label, sequences in sequences_dict.items():
            results = []
            for seq in sequences:
                try:
                    hydropathy, isoelectric_point, molecular_weight, aliphatic_index = calculate_properties(seq)
                    if prop == 'Hydropathy':
                        results.append(hydropathy)
                    elif prop == 'Isoelectric Point':
                        results.append(isoelectric_point)
                    elif prop == 'Molecular Weight':
                        results.append(molecular_weight)
                    elif prop == 'Aliphatic Index':
                        results.append(aliphatic_index)
                except Exception as e:
                    continue  # Skip problematic sequences

            # Normalize the results
            normalized_results = normalize_data(results)

            # Plot each normalized property in a single subplot
            axes[i].hist(normalized_results, bins=20, alpha=0.5, label=label, edgecolor='black')
            axes[i].set_title(f'{prop} Distribution (Normalized)')
            axes[i].set_xlabel(f'{prop} (Normalized)')
            axes[i].set_ylabel('Frequency')
            axes[i].legend()

    plt.tight_layout()
    plt.show()

# Dictionary of sequences grouped by label
sequences_dict = {
    'High Frequency': high_freq_sequences,
    'Low Frequency': low_freq_sequences,
    'Negative Correlation': negative_corr_sequences,
    'Positive Correlation': positive_corr_sequences
}

# Run property calculation and normalized plotting
calculate_and_plot_properties_subplot_normalized(sequences_dict)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind

# Function to calculate amino acid frequencies per position
def calculate_amino_acid_frequencies(tcr_series, max_len=20):
    amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
    frequency_dict = {aa: [0] * max_len for aa in amino_acids}

    for seq in tcr_series:
        for pos, aa in enumerate(seq[:max_len]):  # Limit sequences to max_len
            if aa in frequency_dict:
                frequency_dict[aa][pos] += 1
    return frequency_dict

# Amino acid groups based on properties
amino_acid_groups = {
    'Positively Charged (Basic)': ['R', 'K'],
    'Negatively Charged (Acidic)': ['D', 'E'],
    'Charged Amino Acids': ['H'],
    'Polar, Uncharged': ['S', 'T', 'C', 'N', 'Q'],
    'Polar, Charged': ['Y'],
    'Aliphatic': ['G', 'A', 'V', 'L', 'I'],
    'Aromatic': ['F', 'W'],
    'Special Amino Acids': ['P', 'M', 'C'],
    'Amino Acids with Sulfur': ['C', 'M'],
    'Amino Acids with Amines': ['H'],
    'Amino Acids with Amides': ['N', 'Q'],
    'Imidazole-Containing Amino Acids': ['H'],
    'Hydrophobic Amino Acids': ['V', 'L', 'I', 'F', 'W', 'M']
}

# Function to sum frequencies for each amino acid group
def sum_group_frequencies(frequencies, group_definitions):
    group_totals = {}
    for group_name, amino_acids in group_definitions.items():
        group_totals[group_name] = [sum(frequencies[aa]) for aa in amino_acids if aa in frequencies]
    return group_totals

# Function to load files and calculate frequencies
def load_and_calculate_frequencies(file, file_type='csv'):
    if file_type == 'csv':
        df = pd.read_csv(file)
    elif file_type == 'xlsx':
        df = pd.read_excel(file)
    return calculate_amino_acid_frequencies(df['TCR'])

# Load and calculate frequencies for each file
co_occurrence_high_freq = load_and_calculate_frequencies('co_occurance_high.xlsx', file_type='xlsx')
co_occurrence_low_freq = load_and_calculate_frequencies('co_occurance_low.xlsx', file_type='xlsx')
negative_corr_freq = load_and_calculate_frequencies('negative_corr_group.csv', file_type='csv')
positive_corr_freq = load_and_calculate_frequencies('positive_corr_group.csv', file_type='csv')

# Sum frequencies for each amino acid group
co_occ_high_totals = sum_group_frequencies(co_occurrence_high_freq, amino_acid_groups)
co_occ_low_totals = sum_group_frequencies(co_occurrence_low_freq, amino_acid_groups)
neg_corr_totals = sum_group_frequencies(negative_corr_freq, amino_acid_groups)
pos_corr_totals = sum_group_frequencies(positive_corr_freq, amino_acid_groups)

# Plot each amino acid group in a separate subplot with left and right comparisons
for group_name in amino_acid_groups.keys():
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    x = range(len(co_occ_high_totals[group_name]))
    width = 0.35

    # Left Plot: Co-occurrence Comparison with swapped colors
    axes[0].bar(x, co_occ_low_totals[group_name], width=width, color='orange', label='Low Co-occurrence', align='center')
    axes[0].bar([p + width for p in x], co_occ_high_totals[group_name], width=width, color='purple', label='High Co-occurrence', align='center')
    axes[0].set_title(f'{group_name} Amino Acids - Co-occurrence')
    axes[0].set_xlabel('Amino Acids')
    axes[0].set_ylabel('Total Count')
    axes[0].set_xticks([p + width/2 for p in x])
    axes[0].set_xticklabels(amino_acid_groups[group_name])
    axes[0].legend()

    # Right Plot: Correlation Comparison
    axes[1].bar(x, neg_corr_totals[group_name], width=width, color='lightblue', label='Negative Correlation', align='center')
    axes[1].bar([p + width for p in x], pos_corr_totals[group_name], width=width, color='green', label='Positive Correlation', align='center')
    axes[1].set_title(f'{group_name} Amino Acids - Correlation')
    axes[1].set_xlabel('Amino Acids')
    axes[1].set_ylabel('Total Count')
    axes[1].set_xticks([p + width/2 for p in x])
    axes[1].set_xticklabels(amino_acid_groups[group_name])
    axes[1].legend()

    plt.tight_layout()
    plt.show()

    # Statistical significance testing for each comparison
    t_stat_co_occ, p_val_co_occ = ttest_ind(co_occ_high_totals[group_name], co_occ_low_totals[group_name])
    t_stat_corr, p_val_corr = ttest_ind(neg_corr_totals[group_name], pos_corr_totals[group_name])

    print(f'{group_name} Amino Acids - Co-occurrence Comparison: T-Statistic={t_stat_co_occ}, P-Value={p_val_co_occ}')
    print(f'{group_name} Amino Acids - Correlation Comparison: T-Statistic={t_stat_corr}, P-Value={p_val_corr}')


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind

# Load data from files
co_occurance_high = pd.read_excel('co_occurance_high.xlsx')['TCR']
co_occurance_low = pd.read_excel('co_occurance_low.xlsx')['TCR']
positive_corr_group = pd.read_csv('positive_corr_group.csv')['TCR']
negative_corr_group = pd.read_csv('negative_corr_group.csv')['TCR']

# Function to calculate amino acid frequencies per position
def calculate_amino_acid_frequencies(group, max_len=20):
    amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
    frequency_dict = {pos: {aa: 0 for aa in amino_acids} for pos in range(max_len)}

    for seq in group:
        for pos, aa in enumerate(seq[:max_len]):  # Limit to max_len
            if aa in frequency_dict[pos]:
                frequency_dict[pos][aa] += 1

    return frequency_dict

# Function to plot amino acid frequencies by position with two side-by-side subplots
def plot_position_frequencies(freq_high, freq_low, label_high, label_low, freq_pos, freq_neg, label_pos, label_neg):
    amino_acids = list(freq_high[0].keys())  # Get amino acids list from the first position

    for pos in freq_high:
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        width = 0.35
        x = range(len(amino_acids))

        # Left Plot: Co-occurrence Comparison with swapped colors
        axes[0].bar(x, [freq_low[pos][aa] for aa in amino_acids], width=width, color='orange', label=label_low, align='center')
        axes[0].bar([p + width for p in x], [freq_high[pos][aa] for aa in amino_acids], width=width, color='purple', label=label_high, align='center')
        axes[0].set_title(f'{label_high} vs {label_low} - Amino Acid Frequencies at Position {pos}')
        axes[0].set_xlabel('Amino Acids')
        axes[0].set_ylabel('Frequency')
        axes[0].set_xticks([p + width / 2 for p in x])
        axes[0].set_xticklabels(amino_acids)
        axes[0].legend()

        # Right Plot: Correlation Comparison
        axes[1].bar(x, [freq_neg[pos][aa] for aa in amino_acids], width=width, color='lightblue', label=label_neg, align='center')
        axes[1].bar([p + width for p in x], [freq_pos[pos][aa] for aa in amino_acids], width=width, color='green', label=label_pos, align='center')
        axes[1].set_title(f'{label_pos} vs {label_neg} - Amino Acid Frequencies at Position {pos}')
        axes[1].set_xlabel('Amino Acids')
        axes[1].set_ylabel('Frequency')
        axes[1].set_xticks([p + width / 2 for p in x])
        axes[1].set_xticklabels(amino_acids)
        axes[1].legend()

        plt.tight_layout()
        plt.show()

# Calculate amino acid frequencies for each group
co_occurrence_high_freq = calculate_amino_acid_frequencies(co_occurance_high)
co_occurrence_low_freq = calculate_amino_acid_frequencies(co_occurance_low)
positive_corr_freq = calculate_amino_acid_frequencies(positive_corr_group)
negative_corr_freq = calculate_amino_acid_frequencies(negative_corr_group)

# Plot side-by-side comparisons for each position across co-occurrence and correlation groups
plot_position_frequencies(
    co_occurrence_high_freq, co_occurrence_low_freq, 'High Co-occurrence', 'Low Co-occurrence',
    positive_corr_freq, negative_corr_freq, 'Positive Correlation', 'Negative Correlation'
)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load the main data file for consistency and publicity groups
file_path = 'final_tcr_selection_adjusted_distribution_with_random_groups.csv'
df = pd.read_csv(file_path)

# Define groups of amino acids
boring_aa = set('GAVILMP')  # "Boring" amino acids
interesting_aa = set('HCRSTYDEKW')  # "Interesting" amino acids

# Define the groups directly from the data
co_occurance_high = pd.read_excel('co_occurance_high.xlsx')['TCR']
co_occurance_low = pd.read_excel('co_occurance_low.xlsx')['TCR']
consistency_group = df[df['selection_group'] == 'consistency_area_top']['amino_acid'].dropna()
publicity_group = df[df['selection_group'] == 'publicity_top']['amino_acid'].dropna()

# Function to calculate amino acid frequencies per position
def calculate_amino_acid_frequencies(group, max_len=20):
    amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
    frequency_dict = {pos: {aa: 0 for aa in amino_acids} for pos in range(max_len)}

    for seq in group:
        for pos, aa in enumerate(seq[:max_len]):  # Limit to max_len
            if aa in frequency_dict[pos]:
                frequency_dict[pos][aa] += 1

    return frequency_dict

# Function to plot amino acid frequencies by position with two side-by-side subplots
def plot_position_frequencies(freq_high, freq_low, label_high, label_low, freq_publicity, freq_consistency, label_publicity, label_consistency):
    amino_acids = list(freq_high[0].keys())  # Get amino acids list from the first position

    for pos in freq_high:
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        width = 0.35
        x = range(len(amino_acids))

        # Left Plot: Co-occurrence Comparison
        axes[0].bar(x, [freq_low[pos][aa] for aa in amino_acids], width=width, color='orange', label=label_low, align='center')
        axes[0].bar([p + width for p in x], [freq_high[pos][aa] for aa in amino_acids], width=width, color='purple', label=label_high, align='center')
        axes[0].set_title(f'{label_high} vs {label_low} - Amino Acid Frequencies at Position {pos}')
        axes[0].set_xlabel('Amino Acids')
        axes[0].set_ylabel('Frequency')
        axes[0].set_xticks([p + width / 2 for p in x])
        axes[0].set_xticklabels(amino_acids)
        axes[0].legend()

        # Right Plot: Publicity vs TCR Weight Comparison (switched positions)
        axes[1].bar(x, [freq_consistency[pos][aa] for aa in amino_acids], width=width, color='green', label=label_consistency, align='center')
        axes[1].bar([p + width for p in x], [freq_publicity[pos][aa] for aa in amino_acids], width=width, color='lightblue', label=label_publicity, align='center')
        axes[1].set_title(f'{label_publicity} vs {label_consistency} - Amino Acid Frequencies at Position {pos}')
        axes[1].set_xlabel('Amino Acids')
        axes[1].set_ylabel('Frequency')
        axes[1].set_xticks([p + width / 2 for p in x])
        axes[1].set_xticklabels(amino_acids)
        axes[1].legend()

        plt.tight_layout()
        plt.show()

# Calculate amino acid frequencies for each group
co_occurrence_high_freq = calculate_amino_acid_frequencies(co_occurance_high)
co_occurrence_low_freq = calculate_amino_acid_frequencies(co_occurance_low)
consistency_freq = calculate_amino_acid_frequencies(consistency_group)
publicity_freq = calculate_amino_acid_frequencies(publicity_group)

# Plot side-by-side comparisons for each position across co-occurrence and consistency/publicity groups
plot_position_frequencies(
    co_occurrence_high_freq, co_occurrence_low_freq, 'old', 'young',
    publicity_freq, consistency_freq, 'Publicity', 'TCR Weight'
)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load data from files
co_occurance_high = pd.read_excel('co_occurance_high.xlsx')['TCR']
co_occurance_low = pd.read_excel('co_occurance_low.xlsx')['TCR']
positive_corr_group = pd.read_csv('positive_corr_group.csv')['TCR']
negative_corr_group = pd.read_csv('negative_corr_group.csv')['TCR']

# Function to calculate amino acid frequencies per position
def calculate_amino_acid_frequencies(group, max_len=20):
    amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
    frequency_dict = {aa: [0] * max_len for aa in amino_acids}

    for seq in group:
        for pos, aa in enumerate(seq[:max_len]):  # Limit to max_len positions
            if aa in frequency_dict:
                frequency_dict[aa][pos] += 1

    return frequency_dict

# Calculate frequencies for each group
co_occurance_high_freqs = calculate_amino_acid_frequencies(co_occurance_high)
co_occurance_low_freqs = calculate_amino_acid_frequencies(co_occurance_low)
positive_corr_freqs = calculate_amino_acid_frequencies(positive_corr_group)
negative_corr_freqs = calculate_amino_acid_frequencies(negative_corr_group)

# Plot frequencies for each amino acid across all positions
amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
max_len = 20

for aa in amino_acids:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

    # Left Plot: Co-occurrence Comparison
    axes[0].plot(range(max_len), co_occurance_low_freqs[aa], marker='o', label='Low Co-occurrence', color='orange')
    axes[0].plot(range(max_len), co_occurance_high_freqs[aa], marker='o', label='High Co-occurrence', color='purple')
    axes[0].set_title(f'{aa} Amino Acid - Co-occurrence')
    axes[0].set_xlabel('Position')
    axes[0].set_ylabel('Frequency')
    axes[0].legend()
    axes[0].grid(True)

    # Right Plot: Correlation Comparison
    axes[1].plot(range(max_len), negative_corr_freqs[aa], marker='o', label='Negative Correlation', color='lightblue')
    axes[1].plot(range(max_len), positive_corr_freqs[aa], marker='o', label='Positive Correlation', color='green')
    axes[1].set_title(f'{aa} Amino Acid - Correlation')
    axes[1].set_xlabel('Position')
    axes[1].legend()
    axes[1].grid(True)

    plt.suptitle(f'Frequency of Amino Acid {aa} Across All Positions')
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])  # Adjust layout to fit title
    plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind

# Load data from files
co_occurance_high = pd.read_excel('co_occurance_high.xlsx')['TCR']
co_occurance_low = pd.read_excel('co_occurance_low.xlsx')['TCR']
positive_corr_group = pd.read_csv('positive_corr_group.csv')['TCR']
negative_corr_group = pd.read_csv('negative_corr_group.csv')['TCR']

# Function to calculate amino acid frequencies per position
def calculate_amino_acid_frequencies(group, max_len=20):
    amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
    frequency_dict = {pos: {aa: 0 for aa in amino_acids} for pos in range(max_len)}

    for seq in group:
        for pos, aa in enumerate(seq[:max_len]):  # Limit to max_len
            if aa in frequency_dict[pos]:
                frequency_dict[pos][aa] += 1

    return frequency_dict

# Function to plot amino acid frequencies by position
def plot_position_frequencies(positive_freq, negative_freq, label_positive, label_negative):
    amino_acids = list(positive_freq[0].keys())  # Get amino acids list from the first position

    for pos in positive_freq:
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        width = 0.35
        x = range(len(amino_acids))

        # Left Plot: Co-occurrence Comparison with swapped colors
        axes[0].bar(x, [negative_freq[pos][aa] for aa in amino_acids], width=width, color='orange', label='Low Co-occurrence', align='center')
        axes[0].bar([p + width for p in x], [positive_freq[pos][aa] for aa in amino_acids], width=width, color='purple', label='High Co-occurrence', align='center')
        axes[0].set_title(f'Co-occurrence Amino Acid Frequencies at Position {pos}')
        axes[0].set_xlabel('Amino Acids')
        axes[0].set_ylabel('Frequency')
        axes[0].set_xticks([p + width / 2 for p in x])
        axes[0].set_xticklabels(amino_acids)
        axes[0].legend()

        # Right Plot: Correlation Comparison
        axes[1].bar(x, [negative_freq[pos][aa] for aa in amino_acids], width=width, color='lightblue', label='Negative Correlation', align='center')
        axes[1].bar([p + width for p in x], [positive_freq[pos][aa] for aa in amino_acids], width=width, color='green', label='Positive Correlation', align='center')
        axes[1].set_title(f'Correlation Amino Acid Frequencies at Position {pos}')
        axes[1].set_xlabel('Amino Acids')
        axes[1].set_ylabel('Frequency')
        axes[1].set_xticks([p + width / 2 for p in x])
        axes[1].set_xticklabels(amino_acids)
        axes[1].legend()

        plt.tight_layout()
        plt.show()

# Process each pair of groups and plot frequencies for each position
group_pairs = [
    (calculate_amino_acid_frequencies(co_occurance_high), calculate_amino_acid_frequencies(co_occurance_low), 'High Co-occurrence', 'Low Co-occurrence'),
    (calculate_amino_acid_frequencies(positive_corr_group), calculate_amino_acid_frequencies(negative_corr_group), 'Positive Correlation', 'Negative Correlation')
]

for positive_frequencies, negative_frequencies, label_positive, label_negative in group_pairs:
    plot_position_frequencies(positive_frequencies, negative_frequencies, label_positive, label_negative)


co - occurance score

In [ ]:
import pandas as pd
from scipy.stats import ttest_1samp, ks_2samp, mannwhitneyu, levene, skew
from statsmodels.stats.multitest import multipletests

# Step 1: Load the metadata file to extract the age list
metadata_file_path = 'Matched_File_Data.xlsx'  # Update with your actual file path
metadata_df = pd.read_excel(metadata_file_path)

# Extract the full list of ages from the metadata
age_list = metadata_df['Age'].tolist()

# Calculate population mean age
population_mean_age = sum(age_list) / len(age_list)

# Step 2: Load the TCR dataset
tcr_file_path = 'tcr_age_lists.csv'  # Update with your file path
tcr_df = pd.read_csv(tcr_file_path)

# Step 3: Add statistical tests
def tcr_statistics(row, population_ages):
    tcr_ages = eval(row['Ages'])  # Convert string representation of list to actual list

    # Perform statistical tests
    t_stat, ttest_p = ttest_1samp(tcr_ages, popmean=population_mean_age)
    ks_stat, ks_p = ks_2samp(tcr_ages, population_ages)
    _, mwu_p = mannwhitneyu(tcr_ages, population_ages, alternative='two-sided')
    _, levene_p = levene(tcr_ages, population_ages)
    tcr_mean = sum(tcr_ages) / len(tcr_ages)
    tcr_skew = skew(tcr_ages)

    return pd.Series({
        'mean_age': tcr_mean,
        't_stat': t_stat,
        'ttest_p_value': ttest_p,
        'ks_p_value': ks_p,
        'mwu_p_value': mwu_p,
        'levene_p_value': levene_p,
        'skewness': tcr_skew
    })

# Apply the statistical tests
stats_df = tcr_df.apply(tcr_statistics, axis=1, population_ages=age_list)

# Merge the results back into the original DataFrame
tcr_df = pd.concat([tcr_df, stats_df], axis=1)

# Step 4: Classify TCRs based on mean age
def classify_age_group(row, population_mean):
    if row['mean_age'] < population_mean - 5:  # Younger threshold
        return 'Younger'
    elif row['mean_age'] > population_mean + 5:  # Older threshold
        return 'Older'
    else:
        return 'Neutral'

tcr_df['age_group'] = tcr_df.apply(classify_age_group, axis=1, population_mean=population_mean_age)

# Step 5: Adjust p-values for multiple comparisons (optional)
for col in ['ttest_p_value', 'ks_p_value', 'mwu_p_value', 'levene_p_value']:
    adjusted_p_values = multipletests(tcr_df[col], method='fdr_bh')[1]
    tcr_df[f'adjusted_{col}'] = adjusted_p_values

# Step 6: Save the full DataFrame with all scores
output_file = 'tcr_with_all_scores.csv'
tcr_df.to_csv(output_file, index=False)

print(f"Full TCR data with scores saved to '{output_file}'.")

# Step 7: Visualization (optional)
import matplotlib.pyplot as plt

# Scatterplot: Mean Age vs. p-value (t-test)
plt.scatter(tcr_df['mean_age'], tcr_df['ttest_p_value'], alpha=0.5, label='t-test p-values')
plt.axhline(0.05, color='red', linestyle='dashed', label='Significance Threshold (p=0.05)')
plt.xlabel('Mean Age')
plt.ylabel('p-value')
plt.title('Mean Age vs. p-value (t-test) for TCRs')
plt.legend()
plt.show()


In [ ]:
# Define the significance threshold for adjusted p-values
significance_threshold = 0.05

# Create a dictionary to store counts for each adjusted column
adjusted_significant_counts = {}

# Loop through each adjusted p-value column to count significant TCRs
for col in ['adjusted_ttest_p_value', 'adjusted_ks_p_value', 'adjusted_mwu_p_value', 'adjusted_levene_p_value']:
    adjusted_significant_counts[col] = {
        'Total Significant': (tcr_df[col] < significance_threshold).sum(),
        'Younger': tcr_df[(tcr_df[col] < significance_threshold) & (tcr_df['age_group'] == 'Younger')].shape[0],
        'Older': tcr_df[(tcr_df[col] < significance_threshold) & (tcr_df['age_group'] == 'Older')].shape[0],
        'Neutral': tcr_df[(tcr_df[col] < significance_threshold) & (tcr_df['age_group'] == 'Neutral')].shape[0],
    }

# Print the counts for each adjusted p-value column
print("Number of significant TCRs by adjusted p-value column:")
for test, counts in adjusted_significant_counts.items():
    print(f"{test}:")
    print(f"  Total Significant: {counts['Total Significant']}")
    print(f"  Younger: {counts['Younger']}")
    print(f"  Older: {counts['Older']}")
    print(f"  Neutral: {counts['Neutral']}")


160K TCRs ranking are relation


In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import wasserstein_distance
import ast

# Load the uploaded file
file_path = "updated_tcr_age_lists_with_scores.csv"
df = pd.read_csv(file_path)

# Ensure 'Ages' column is properly parsed into lists
df["Ages"] = df["Ages"].apply(ast.literal_eval)

# Flatten all ages into a single list (TCR-weighted global distribution)
global_ages = [age for sublist in df["Ages"] for age in sublist]
global_mean = np.mean(global_ages)

# Define scoring functions
def mean_age(ages):
    return np.mean(ages) if len(ages) > 0 else 0

def squared_mean_age(ages):
    return np.mean(np.square(ages)) if len(ages) > 0 else 0

def signed_wasserstein(ages, global_ages, global_mean):
    if len(ages) == 0:
        return 0
    dist = wasserstein_distance(ages, global_ages)
    sign = np.sign(np.mean(ages) - global_mean)
    return sign * dist

# Apply scoring functions
df["mean_age"] = df["Ages"].apply(mean_age)
df["squared_mean_age"] = df["Ages"].apply(squared_mean_age)
df["signed_wasserstein"] = df["Ages"].apply(lambda x: signed_wasserstein(x, global_ages, global_mean))



In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations

# Load your file
df = pd.read_csv("updated_tcr_age_lists_with_scored_output.csv")

# Normalize scores
score_cols = ["tcr_score", "mean_age", "squared_mean_age", "signed_wasserstein"]
for col in score_cols:
    df[f"{col}_norm"] = (df[col] - df[col].min()) / (df[col].max() - df[col].min())

# Correlation matrix
corr = df[score_cols].corr()

# Plot 1: Correlation heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Between Scoring Methods")
plt.tight_layout()
plt.show()

# Plot 2: Pairwise plot of normalized scores
sns.pairplot(df[[f"{col}_norm" for col in score_cols]])
plt.suptitle("Pairwise Comparison of Normalized Scores", y=1.02)
plt.show()

# Plot 3: Score distributions
plt.figure(figsize=(12, 6))
for i, col in enumerate(score_cols):
    plt.subplot(2, 2, i + 1)
    sns.histplot(df[col], kde=True, bins=50)
    plt.title(f"Distribution of {col}")
plt.tight_layout()
plt.show()

# Top-K and Bottom-K overlap analysis
k_values = [100, 500, 1000, 5000, 10000]
overlap_results = []
for k in k_values:
    for a, b in combinations(score_cols, 2):
        top_a = set(df.nlargest(k, a).index)
        top_b = set(df.nlargest(k, b).index)
        bottom_a = set(df.nsmallest(k, a).index)
        bottom_b = set(df.nsmallest(k, b).index)
        overlap_results.append({
            "k": k,
            "method_a": a,
            "method_b": b,
            "top_overlap": len(top_a & top_b) / k,
            "bottom_overlap": len(bottom_a & bottom_b) / k
        })

# Save or view overlap results
overlap_df = pd.DataFrame(overlap_results)
overlap_df.to_csv("tcr_score_overlap_comparison.csv", index=False)
print("Overlap analysis saved.")

# Optional: Display top of the table
print(overlap_df.head())


In [ ]:
overlap_df

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from itertools import combinations

# Load your data
df = pd.read_csv("updated_tcr_age_lists_with_scored_output.csv")

# Define methods and K values
scoring_cols = ["tcr_score", "mean_age", "squared_mean_age", "signed_wasserstein"]
ks = [100, 500, 1000]

# Create ranking dictionaries
ranked = {col: df.sort_values(by=col, ascending=False)["TCR"].tolist() for col in scoring_cols}
bottom_ranked = {col: df.sort_values(by=col, ascending=True)["TCR"].tolist() for col in scoring_cols}

# Function to compute overlap
def compute_overlap_matrix(ranking_dict, k):
    overlap_matrix = pd.DataFrame(index=scoring_cols, columns=scoring_cols, dtype=float)
    for a, b in combinations(scoring_cols, 2):
        set_a, set_b = set(ranking_dict[a][:k]), set(ranking_dict[b][:k])
        overlap = len(set_a & set_b) / k
        overlap_matrix.at[a, b] = overlap
        overlap_matrix.at[b, a] = overlap
    for m in scoring_cols:
        overlap_matrix.at[m, m] = 1.0
    return overlap_matrix

# Plotting
for k in ks:
    top_matrix = compute_overlap_matrix(ranked, k)
    bottom_matrix = compute_overlap_matrix(bottom_ranked, k)

    plt.figure(figsize=(6, 5))
    sns.heatmap(top_matrix, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1)
    plt.title(f"Top-{k} Overlap")
    plt.tight_layout()
    #plt.savefig(f"top_{k}_overlap_heatmap.png")
    plt.show()
    plt.close()

    plt.figure(figsize=(6, 5))
    sns.heatmap(bottom_matrix, annot=True, fmt=".2f", cmap="Oranges", vmin=0, vmax=1)
    plt.title(f"Bottom-{k} Overlap")
    plt.tight_layout()
    plt.show()
    #plt.savefig(f"bottom_{k}_overlap_heatmap.png")
    plt.close()


In [ ]:
# Re-import necessary packages after code execution environment was reset
import pandas as pd
import matplotlib.pyplot as plt
import ast

# Load the file
df = pd.read_csv("updated_tcr_age_lists_with_scored_output.csv")
df["Ages"] = df["Ages"].apply(ast.literal_eval)

# Scoring methods to consider
score_columns = ["tcr_score", "mean_age", "squared_mean_age", "signed_wasserstein"]

# Group sizes
top_k = [50, 100]

# Dictionary to hold distributions
distribution_data = {}

for method in score_columns:
    for k in top_k:
        top_k_ages = df.nlargest(k, method)["Ages"].explode().astype(int).tolist()
        bottom_k_ages = df.nsmallest(k, method)["Ages"].explode().astype(int).tolist()
        distribution_data[f"{method}_top_{k}"] = top_k_ages
        distribution_data[f"{method}_bottom_{k}"] = bottom_k_ages

# Plotting
n_plots = len(distribution_data)
n_cols = 4
n_rows = (n_plots + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
axes = axes.flatten()

for i, (key, ages) in enumerate(distribution_data.items()):
    ax = axes[i]
    ax.hist(ages, bins=range(0, 101, 5), color='skyblue', edgecolor='black')
    ax.set_title(f"Age Distribution - {key.replace('_', ' ').title()}")
    ax.set_xlabel("Age")
    ax.set_ylabel("Frequency")
    ax.grid(True)

# Hide any unused subplots
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ast

top_ks = [50, 100, 500]
scoring_cols = ["tcr_score", "mean_age", "squared_mean_age", "signed_wasserstein"]

# Function to aggregate age distributions
def get_age_distributions(df, method, k, ascending=False):
    selected = df.sort_values(by=method, ascending=ascending).head(k)
    return [age for ages in selected["Ages"] for age in ages]

# Prepare data for plotting
plot_data = []
for method in scoring_cols:
    for k in top_ks:
        top_ages = get_age_distributions(df, method, k, ascending=False)
        bottom_ages = get_age_distributions(df, method, k, ascending=True)
        plot_data.append((f"{method} Top-{k}", top_ages))
        plot_data.append((f"{method} Bottom-{k}", bottom_ages))

# Plot age distributions for top/bottom K TCRs for each method
n_plots = len(plot_data)
fig, axes = plt.subplots(n_plots, 1, figsize=(10, 4 * n_plots), sharex=True)

for ax, (title, ages) in zip(axes, plot_data):
    sns.histplot(ages, bins=range(0, 101, 5), kde=False, ax=ax, color='skyblue')
    ax.set_title(f"Age Distribution: {title}")
    ax.set_ylabel("TCR Count")
    ax.set_xlim(0, 100)

plt.xlabel("Age")
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import gaussian_kde
import numpy as np

# Load your file
#df = pd.read_csv("updated_tcr_age_lists_with_scored_output.csv")
#df["Ages"] = df["Ages"].apply(eval)

# Define scoring methods
methods = ['tcr_score', 'mean_age', 'squared_mean_age', 'signed_wasserstein']

# Store top 50 age vectors
top_50_ages = {}
for method in methods:
    top_tcrs = df.nlargest(1000, method)
    combined_ages = [age for sublist in top_tcrs["Ages"] for age in sublist]
    top_50_ages[method] = combined_ages

# KDE Plot
plt.figure(figsize=(12, 6))
for method, ages in top_50_ages.items():
    if len(ages) > 1:
        kde = gaussian_kde(ages)
        x = np.linspace(0, 100, 200)
        y = kde(x)
        plt.plot(x, y, label=method)
plt.title("Smoothed Age Distributions (KDE) - Top 50 TCRs by Method")
plt.xlabel("Age")
plt.ylabel("Density")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# Mean + STD Plot
means = [np.mean(ages) for ages in top_50_ages.values()]
stds = [np.std(ages) for ages in top_50_ages.values()]

plt.figure(figsize=(10, 5))
plt.bar(methods, means, yerr=stds, capsize=5, color='lightblue')
plt.ylabel("Mean Age (± STD)")
plt.title("Mean Age with Standard Deviation - Top 50 TCRs by Method")
plt.grid(True, axis='y')
plt.tight_layout()
plt.show()

# Heatmap of Deciles
decile_cols = [str(i) for i in range(1, 11)]
heatmap_data = pd.DataFrame(index=methods, columns=decile_cols)

for method in methods:
    top_tcrs = df.nlargest(50, method)
    heatmap_data.loc[method] = top_tcrs[decile_cols].mean()

heatmap_data = heatmap_data.astype(float)

plt.figure(figsize=(10, 6))
sns.heatmap(heatmap_data, annot=True, cmap='coolwarm', fmt=".2f", cbar_kws={'label': 'Avg % in Decile'})
plt.title("Decile Distribution - Top 50 TCRs by Method")
plt.xlabel("Age Group (Deciles)")
plt.ylabel("Scoring Method")
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import zscore

# Load your CSV
#df = pd.read_csv("updated_tcr_age_lists_full_with_deciles.csv")

# Define methods
methods = ["tcr_score", "mean_age", "squared_mean_age"]

# Calculate Z-scores
for method in methods:
    df[f"{method}_z"] = zscore(df[method])

# KDE plot of Z-scores
plt.figure(figsize=(14, 8))
for method in methods:
    sns.kdeplot(df[f"{method}_z"], label=method, fill=True, alpha=0.4)

plt.axvline(-2, color='blue', linestyle='--', label='Z = -2 (Younger-biased)')
plt.axvline(2, color='red', linestyle='--', label='Z = +2 (Older-biased)')
plt.title("Distribution of Z-Scores for Each Age Scoring Method")
plt.xlabel("Z-score")
plt.ylabel("Density")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# Threshold for significance
threshold = 2

# Count how many TCRs are statistically significant (|Z| > 2)
for method in methods:
    sig_count = df[df[f"{method}_z"].abs() > threshold].shape[0]
    print(f"{method}: {sig_count} significant TCRs (|Z| > 2)")



In [ ]:
import pandas as pd
import numpy as np
from sklearn.mixture import GaussianMixture
import matplotlib.pyplot as plt
import seaborn as sns

# Load the data
df = pd.read_csv("updated_tcr_age_lists_with_scored_output.csv")

# Ensure 'signed_wasserstein' column exists
if "signed_wasserstein" in df.columns:
    # Fit GMM with 2 components
    gmm = GaussianMixture(n_components=2, random_state=42)
    signed_vals = df["signed_wasserstein"].values.reshape(-1, 1)
    gmm.fit(signed_vals)

    # Predict probabilities
    probs = gmm.predict_proba(signed_vals)

    # Determine which component corresponds to older (positive) and younger (negative)
    means = gmm.means_.flatten()
    old_idx = np.argmax(means)
    young_idx = np.argmin(means)

    # Assign probabilities to dataframe
    df["old_prob"] = probs[:, old_idx]
    df["young_prob"] = probs[:, young_idx]

    # Sort top TCRs with highest probabilities
    top_old = df.sort_values(by="old_prob", ascending=False).head(50)
    top_young = df.sort_values(by="young_prob", ascending=False).head(50)


In [ ]:
df = pd.read_csv("merged_vdjdb_significant_tcrs_test_across_samples.csv")
df.shape

In [ ]:
# Set threshold
min_tcr_count = 75

# Filter each k-mer group
filtered_kmers = {
    k: {kmer for kmer, tcrs in kmer_to_tcrs[k].items() if len(tcrs) >= min_tcr_count}
    for k in [3, 4, 5]
}

# Print how many survived
for k in [3, 4, 5]:
    print(f"{k}-mers kept (≥{min_tcr_count} TCRs): {len(filtered_kmers[k])}")


In [ ]:
# Save filtered k-mers as a single list
all_filtered_kmers = list(set().union(*filtered_kmers.values()))
pd.DataFrame({'kmer': all_filtered_kmers}).to_csv('filtered_kmers_list.csv', index=False)

In [ ]:
import pandas as pd
from collections import Counter
import matplotlib.pyplot as plt

# Load TCRs
df = pd.read_csv('significant_tcrs_signed_wasserstein.csv')
tcrs = df['TCR'].dropna().unique().tolist()

print(f"✅ Loaded {len(tcrs):,} unique significant TCRs")

# K-mer extraction function
def extract_kmers(seq, k):
    return [seq[i:i+k] for i in range(len(seq) - k + 1)]

# Collect all k-mers
kmer_lengths = [3, 4, 5]
kmer_counts = {k: Counter() for k in kmer_lengths}

for tcr in tcrs:
    for k in kmer_lengths:
        kmers = extract_kmers(tcr, k)
        kmer_counts[k].update(kmers)

# Summary statistics
for k in kmer_lengths:
    print(f"\n🔹 {k}-mers:")
    print(f"- Total occurrences: {sum(kmer_counts[k].values()):,}")
    print(f"- Unique k-mers: {len(kmer_counts[k])}")
    print(f"- Top 10 frequent {k}-mers:")
    print(kmer_counts[k].most_common(10))

# Plot distribution of unique counts
for k in kmer_lengths:
    freqs = list(kmer_counts[k].values())
    plt.figure(figsize=(8, 4))
    plt.hist(freqs, bins=50, color='skyblue', edgecolor='black')
    plt.title(f"Distribution of {k}-mer Frequencies")
    plt.xlabel("Count")
    plt.ylabel("Number of k-mers")
    plt.grid(True)
    plt.tight_layout()
    plt.show()


In [ ]:
from collections import defaultdict
import pandas as pd

# Reload TCRs (if needed)
df = pd.read_csv('significant_tcrs_signed_wasserstein.csv')
tcrs = df['TCR'].dropna().unique().tolist()

# Build: kmer → set of TCRs it appears in
kmer_to_tcrs = {k: defaultdict(set) for k in [3, 4, 5]}

for tcr in tcrs:
    for k in [3, 4, 5]:
        kmers = set([tcr[i:i+k] for i in range(len(tcr) - k + 1)])
        for kmer in kmers:
            kmer_to_tcrs[k][kmer].add(tcr)

# Count: how many k-mers appear in exactly n TCRs (for n = 1 to 20)
summary = {}
for k in [3, 4, 5]:
    freq_counter = defaultdict(int)
    for kmer, tcr_set in kmer_to_tcrs[k].items():
        count = len(tcr_set)
        if count <= 20:
            freq_counter[count] += 1
    summary[k] = freq_counter

# Display summary as DataFrame
summary_df = pd.DataFrame.from_dict(summary, orient='index').T.fillna(0).astype(int)
summary_df.index.name = 'n_TCRs'
summary_df.columns = ['3-mer', '4-mer', '5-mer']
print(summary_df)


In [ ]:
# Assuming summary_df is already defined
summary_df_sorted = summary_df.sort_index()
print(summary_df_sorted)


In [ ]:
import pandas as pd
import os
from collections import Counter
from tqdm import tqdm

# K-mer lengths
kmer_lengths = [3, 4, 5]

# Input files
train_file = "merged_vdjdb_significant_tcrs_train_across_samples.csv"
test_file = "merged_vdjdb_significant_tcrs_test_across_samples.csv"

# Output directory
output_dir = "kmer_outputs"
os.makedirs(output_dir, exist_ok=True)

# Extract k-mers
def extract_kmers(seq, ks):
    all_kmers = []
    for k in ks:
        if len(seq) >= k:
            all_kmers.extend([seq[i:i+k] for i in range(len(seq) - k + 1)])
    return all_kmers

# Process dataset
def process_kmers(file_path, ks, output_name):
    df = pd.read_csv(file_path)
    df.set_index("sample name", inplace=True)

    all_kmers = set()
    sample_kmer_counts = {}

    for sample_name, row in tqdm(df.iterrows(), total=len(df), desc=f"Processing {output_name}"):
        kmer_counter = Counter()
        for tcr_seq, count in row.items():
            if pd.isna(count) or count == 0:
                continue
            try:
                kmers = extract_kmers(tcr_seq, ks)
                kmer_counter.update({kmer: count for kmer in kmers})
                all_kmers.update(kmers)
            except:
                continue

        sample_kmer_counts[sample_name] = kmer_counter

    all_kmers = sorted(all_kmers)
    df_kmer = pd.DataFrame(index=sample_kmer_counts.keys(), columns=all_kmers).fillna(0)

    for sample, kmer_counts in sample_kmer_counts.items():
        for kmer, freq in kmer_counts.items():
            df_kmer.at[sample, kmer] = freq

    df_kmer.reset_index(inplace=True)
    df_kmer.rename(columns={"index": "sample name"}, inplace=True)
    df_kmer.to_csv(os.path.join(output_dir, f"{output_name}_kmers.csv"), index=False)
    print(f"Saved {output_name} k-mer matrix with shape: {df_kmer.shape}")

# Run
process_kmers(train_file, kmer_lengths, "train")
process_kmers(test_file, kmer_lengths, "test")


In [ ]:
train_kmers = pd.read_csv("kmer_outputs/train_kmers.csv")
test_kmers = pd.read_csv("kmer_outputs/test_kmers.csv")

print("Train shape:", train_kmers.shape)
print("Test shape:", test_kmers.shape)

# Top 20 k-mers
top_kmers = train_kmers.drop(columns="sample name").sum().sort_values(ascending=True).head(100)
print("Top 20 most frequent k-mers:\n", top_kmers)


In [ ]:
train_kmers

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load k-mer matrix (train)
train_kmers = pd.read_csv("kmer_outputs/train_kmers.csv")
X_train = train_kmers.drop(columns='sample name')

# Compute variance for each k-mer
variances = X_train.var()

# Plot histogram of variances
plt.figure(figsize=(10, 6))
plt.hist(variances, bins=10000, color='skyblue', edgecolor='black')
plt.title("Distribution of K-mer Variance Across Samples (Train Set)")
plt.xlabel("Variance")
plt.ylabel("Number of K-mers")
plt.xlim(0, 10)
plt.grid(True)
plt.tight_layout()
plt.axvline(x=1.0, color='red', linestyle='--', label='Variance Threshold = 1.0')
plt.legend()
plt.show()


In [ ]:
import pandas as pd

# Load the training k-mer matrix
train_kmers = pd.read_csv("kmer_outputs/train_kmers.csv")
X_train = train_kmers.drop(columns='sample name')

# Calculate variance
variances = X_train.var()

# Count k-mers by variance range
low_variance_count = (variances <= 1.0).sum()
high_variance_count = (variances > 1.0).sum()

print(f"Number of k-mers with variance ≤ 1.0: {low_variance_count}")
print(f"Number of k-mers with variance > 1.0: {high_variance_count}")


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Assuming df is already loaded and has the necessary columns from both methods
# Extract sets of significant TCRs for each method
significant_sets = {
    "tcr_score": set(df[df["tcr_score_z"].abs() > 2]["TCR"]),
    "mean_age": set(df[df["mean_age_z"].abs() > 2]["TCR"]),
    "squared_mean_age": set(df[df["squared_mean_age_z"].abs() > 2]["TCR"]),
    "signed_wasserstein_gmm": set(df[df["significant"]]["TCR"]),
}

# Create an empty DataFrame to store overlaps
methods = list(significant_sets.keys())
overlap_matrix = pd.DataFrame(index=methods, columns=methods, dtype=float)

# Calculate Jaccard similarity (intersection / union)
for m1 in methods:
    for m2 in methods:
        inter = len(significant_sets[m1] & significant_sets[m2])
        union = len(significant_sets[m1] | significant_sets[m2])
        overlap_matrix.loc[m1, m2] = inter / union if union > 0 else 0

# Plot the heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(overlap_matrix.astype(float), annot=True, fmt=".2f", cmap="Blues", square=True)
plt.title("Jaccard Overlap of Statistically Significant TCRs Across Methods")
plt.tight_layout()
plt.show()


In [ ]:
# Step 1: Assign each TCR to its GMM component
df["gmm_component"] = gmm.predict(signed_vals)

# Step 2: Compute mean and std for each component
component_stats = df.groupby("gmm_component")["signed_wasserstein"].agg(["mean", "std"])

# Step 3: Compute Z-score relative to component
def calc_z(row):
    mu = component_stats.loc[row["gmm_component"], "mean"]
    std = component_stats.loc[row["gmm_component"], "std"]
    return (row["signed_wasserstein"] - mu) / std if std > 0 else 0

df["component_zscore"] = df.apply(calc_z, axis=1)

# Step 4: Mark statistically significant TCRs (p < 0.05 ≈ |Z| > 1.96)
df["significant"] = df["component_zscore"].abs() > 2

# Optional: Save or preview significant TCRs
significant_tcrs = df[df["significant"]]
print("Number of significant TCRs:", len(significant_tcrs))


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import zscore
from sklearn.mixture import GaussianMixture

# === Load your CSV file ===
df = pd.read_csv("updated_tcr_age_lists_with_scored_output.csv")

# === Z-SCORE ANALYSIS for: tcr_score, mean_age, squared_mean_age ===

methods = ["tcr_score", "mean_age", "squared_mean_age"]

# Calculate Z-scores
for method in methods:
    df[f"{method}_z"] = zscore(df[method])

# KDE plot of Z-scores
plt.figure(figsize=(14, 8))
for method in methods:
    sns.kdeplot(df[f"{method}_z"], label=method, fill=True, alpha=0.4)

plt.axvline(-2, color='blue', linestyle='--', label='Z = -2 (Younger-biased)')
plt.axvline(2, color='red', linestyle='--', label='Z = +2 (Older-biased)')
plt.title("Distribution of Z-Scores for Each Age Scoring Method")
plt.xlabel("Z-score")
plt.ylabel("Density")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# Count significant TCRs for each method
threshold = 2
print("Significant TCR counts by method (|Z| > 2):")
for method in methods:
    sig_count = df[df[f"{method}_z"].abs() > threshold].shape[0]
    print(f" - {method}: {sig_count} TCRs")

# === GMM-BASED SIGNIFICANCE for signed_wasserstein ===

if "signed_wasserstein" in df.columns:
    # Step 1: Fit GMM with 2 components
    gmm = GaussianMixture(n_components=2, random_state=42)
    signed_vals = df["signed_wasserstein"].values.reshape(-1, 1)
    gmm.fit(signed_vals)

    # Step 2: Get probabilities for each GMM component
    probs = gmm.predict_proba(signed_vals)
    means = gmm.means_.flatten()
    old_idx = np.argmax(means)
    young_idx = np.argmin(means)

    # Step 3: Assign GMM probabilities to dataframe
    df["old_prob"] = probs[:, old_idx]
    df["young_prob"] = probs[:, young_idx]

    # Step 4: Assign each TCR to a GMM component
    df["gmm_component"] = gmm.predict(signed_vals)

    # Step 5: Compute mean and std for each component
    component_stats = df.groupby("gmm_component")["signed_wasserstein"].agg(["mean", "std"])

    # Step 6: Compute component-wise Z-score
    def calc_z(row):
        mu = component_stats.loc[row["gmm_component"], "mean"]
        std = component_stats.loc[row["gmm_component"], "std"]
        return (row["signed_wasserstein"] - mu) / std if std > 0 else 0

    df["component_zscore"] = df.apply(calc_z, axis=1)

    # Step 7: Mark significant TCRs (|Z| > 1.96 ≈ p < 0.05)
    df["signed_wasserstein_significant"] = df["component_zscore"].abs() > 1.96

    # Step 8: Extract top TCRs
    top_old = df.sort_values(by="old_prob", ascending=False).head(50)
    top_young = df.sort_values(by="young_prob", ascending=False).head(50)

    print(f"\nSigned-Wasserstein (GMM-based): {df['signed_wasserstein_significant'].sum()} significant TCRs")

# === Final: Save all Z-scores and significance flags ===
df.to_csv("updated_tcr_age_lists_with_all_significance.csv.gz", index=False)
print("\n✅ Saved full DataFrame with Z-scores and significance flags to: updated_tcr_age_lists_with_all_significance.csv.gz")


In [ ]:
import pandas as pd
from scipy.stats import norm

# Load your data
#df = pd.read_csv("updated_tcr_age_lists_with_all_significance.csv.gz")

# Compute two-tailed p-values from z-scores
df["p_value"] = 2 * norm.sf(abs(df["component_zscore"]))

# Filter for TCRs with p < 0.1
filtered_df = df[df["p_value"] < 0.05]

# Save the list of relevant TCRs
#filtered_df.to_csv("tcrs_p_below_0.1.csv", index=False)


In [ ]:
filtered_df

In [ ]:
# === Save only significant TCRs by signed_wasserstein ===
significant_sw = df[df["signed_wasserstein_significant"] == True]

# Save to CSV
significant_sw.to_csv("significant_tcrs_signed_wasserstein.csv", index=False)
print(f"\n💾 Saved {len(significant_sw)} significant TCRs to: significant_tcrs_signed_wasserstein.csv")


In [ ]:
df

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
import seaborn as sns

# === Load your TCR DataFrame ===
df = pd.read_csv("updated_tcr_age_lists_with_all_significance.csv.gz")

# === Define scoring methods and their z-score columns ===
methods = {
    "tcr_score": "tcr_score_z",
    "mean_age": "mean_age_z",
    "squared_mean_age": "squared_mean_age_z",
    "signed_wasserstein": "component_zscore"
}

# === One-hot encode TCR sequences ===
def one_hot_encode_tcrs(tcr_list, max_len=None):
    aa_vocab = list("ACDEFGHIKLMNPQRSTVWY") + ["X"]  # 21 amino acids
    aa_to_idx = {aa: i for i, aa in enumerate(aa_vocab)}

    if max_len is None:
        max_len = max(len(seq) for seq in tcr_list)

    one_hot = np.zeros((len(tcr_list), max_len * len(aa_vocab)))

    for i, seq in enumerate(tcr_list):
        for j, aa in enumerate(seq[:max_len]):
            idx = aa_to_idx.get(aa, aa_to_idx["X"])
            one_hot[i, j * len(aa_vocab) + idx] = 1

    return one_hot

# === Classification Function ===
def classify_by_onehot(method_name, z_col, threshold=2.0):
    print(f"\n🔬 {method_name} (|Z| > {threshold})")

    # Select TCRs above and below threshold
    top = df[df[z_col] > threshold]["TCR"]
    bottom = df[df[z_col] < -threshold]["TCR"]

    print(f"  Top TCRs: {len(top)} | Bottom TCRs: {len(bottom)}")

    if len(top) < 10 or len(bottom) < 10:
        print("  ⚠️ Too few TCRs for reliable classification. Skipping.")
        return

    # Create labeled DataFrame
    top_df = pd.DataFrame({"TCR": top, "group": "top"})
    bottom_df = pd.DataFrame({"TCR": bottom, "group": "bottom"})
    data = pd.concat([top_df, bottom_df])

    # One-hot encode
    X = one_hot_encode_tcrs(data["TCR"])
    y = LabelEncoder().fit_transform(data["group"])

    # Split and train
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, random_state=42, stratify=y
    )

    model = XGBClassifier(use_label_encoder=False, eval_metric="logloss", random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    # Evaluation
    print(classification_report(y_test, y_pred, target_names=["bottom", "top"]))
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["bottom", "top"], yticklabels=["bottom", "top"])
    plt.title(f"Confusion Matrix - {method_name} (|Z| > {threshold})")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.show()

# === Run for different thresholds ===
z_thresholds = [1.96, 2.5, 3.0]  # Feel free to modify this list

for thresh in z_thresholds:
    print(f"\n=============================\nRunning for threshold: |Z| > {thresh}\n=============================")
    for method, zscore_col in methods.items():
        classify_by_onehot(method, zscore_col, threshold=thresh)


In [ ]:
# 📦 Install missing packages (if needed)
!pip install seaborn --quiet

# 📚 Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_selection import VarianceThreshold

# 📂 Load dataset (upload to Colab or mount Drive first)
file_path = "merged_unique_kmers_with_metadata.csv"  # adjust if using Drive

df = pd.read_csv(file_path)

# 🧼 Drop metadata columns and keep only k-mer features
meta_cols = ['sample name', 'Age', 'Biological Sex']
kmer_cols = [col for col in df.columns if col not in meta_cols]
df_kmers = df[kmer_cols]

print(f"✅ Total k-mer features: {df_kmers.shape[1]}")
print(f"✅ Total samples: {df_kmers.shape[0]}")

# ============================================================================
# 🔬 1. Variance Analysis
# ============================================================================

variances = df_kmers.var()
plt.figure(figsize=(10, 4))
sns.histplot(variances, bins=50, color='skyblue')
plt.title("Distribution of K-mer Feature Variance")
plt.xlabel("Variance")
plt.ylabel("Number of Features")
plt.grid(True)
plt.tight_layout()
plt.show()

low_var_threshold = 1e-5
low_variance_count = (variances < low_var_threshold).sum()
print(f"📉 K-mers with variance < {low_var_threshold}: {low_variance_count}")

# ============================================================================
# 🟨 2. Sparsity Analysis
# ============================================================================

nonzero_ratio = (df_kmers != 0).sum(axis=0) / df_kmers.shape[0]
sparsity = 1 - nonzero_ratio

plt.figure(figsize=(10, 4))
sns.histplot(sparsity, bins=50, color='salmon')
plt.title("Sparsity of K-mer Features")
plt.xlabel("Sparsity (1 - Non-zero Frequency)")
plt.ylabel("Number of Features")
plt.grid(True)
plt.tight_layout()
plt.show()

# Report on sparse features
sparsity_threshold = 0.95
high_sparse_count = (sparsity > sparsity_threshold).sum()
print(f"🥶 K-mers appearing in <5% of samples: {high_sparse_count}")

# ============================================================================
# 🧠 3. Correlation Heatmap (top 100 most variable features)
# ============================================================================

# Select top 100 most variable features for visualization
top_var_features = variances.sort_values(ascending=False).head(100).index
corr_matrix = df_kmers[top_var_features].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, cmap='coolwarm', center=0, linewidths=0.1)
plt.title("Correlation Heatmap (Top 100 Most Variable K-mers)")
plt.tight_layout()
plt.show()

# Optional: Number of highly correlated pairs
upper_triangle = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
high_corr_pairs = (upper_triangle.abs() > 0.95).sum().sum()
print(f"🔗 Highly correlated feature pairs (ρ > 0.95): {high_corr_pairs}")


In [ ]:
# 1. Load and Inspect Data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load feature importance table
importance_df = pd.read_csv("importance_df.csv")  # Or use the DataFrame you already have
print("Head of file:\n", importance_df.head())
print("\nColumns:", importance_df.columns.tolist())
print("\nDescribe:\n", importance_df.describe())

# If your importance column is just "importance" (not "scaled_importance"), you can rename it:
if "scaled_importance" not in importance_df.columns and "importance" in importance_df.columns:
    importance_df = importance_df.rename(columns={"importance": "scaled_importance"})

# 2. Basic Stats
print("\nTotal features:", len(importance_df))
print("Features with nonzero importance:", (importance_df['scaled_importance'] > 0).sum())
print("Features with zero importance:", (importance_df['scaled_importance'] == 0).sum())

# 3. Sort and Cumulative Importance
importance_df_sorted = importance_df.sort_values('scaled_importance', ascending=False).reset_index(drop=True)
importance_df_sorted['cumulative_importance'] = importance_df_sorted['scaled_importance'].cumsum()

# 4. Plots

# Histogram of importance
plt.figure(figsize=(10,5))
sns.histplot(importance_df['scaled_importance'], bins=40, color='royalblue')
plt.title('Histogram of Feature Importances')
plt.xlabel('Scaled Importance')
plt.ylabel('Number of Features')
plt.tight_layout()
plt.show()

# Cumulative plot
plt.figure(figsize=(10,5))
plt.plot(importance_df_sorted.index, importance_df_sorted['cumulative_importance'], marker='o')
plt.title('Cumulative Feature Importance')
plt.xlabel('Number of Features (sorted)')
plt.ylabel('Cumulative Importance')
plt.axhline(0.8 * importance_df_sorted['scaled_importance'].sum(), color='red', linestyle='--', label='80%')
plt.axhline(0.95 * importance_df_sorted['scaled_importance'].sum(), color='orange', linestyle='--', label='95%')
plt.legend()
plt.tight_layout()
plt.show()

# 5. Top Features Table
print("\nTop 20 Features:\n", importance_df_sorted[['feature','scaled_importance']].head(20))

# 6. How many features are needed for 80% / 95% of importance?
cumulative_sum = importance_df_sorted['cumulative_importance']
total_importance = cumulative_sum.iloc[-1]
n_80 = np.argmax(cumulative_sum >= 0.8 * total_importance) + 1
n_95 = np.argmax(cumulative_sum >= 0.95 * total_importance) + 1
print(f"\nFeatures to reach 80% cumulative importance: {n_80}")
print(f"Features to reach 95% cumulative importance: {n_95}")

# 7. Where is the biggest drop in feature importance? (elbow)
diffs = np.diff(importance_df_sorted['scaled_importance'].values)
biggest_drop_idx = np.argmax(np.abs(diffs))
print(f"\nBiggest single drop in importance: between features {biggest_drop_idx} and {biggest_drop_idx+1}")

# 8. Tail: How many features are <0.001 importance?
tail_count = (importance_df_sorted['scaled_importance'] < 0.001).sum()
print(f"Features with scaled importance <0.001: {tail_count}")

# 9. Optional: Boxplot for feature importance
plt.figure(figsize=(7,2))
sns.boxplot(x=importance_df_sorted['scaled_importance'], color='lightblue')
plt.title("Boxplot of Feature Importances")
plt.tight_layout()
plt.show()

# 10. Recommendations for Downstream Analysis
print("\n=== Actionable Recommendations ===")
print(f"- Consider keeping only the top {n_80}–{n_95} features (covering 80–95% cumulative importance).")
print(f"- {tail_count} features have negligible contribution (<0.001 importance); can likely be dropped.")
print(f"- Check the top 20 for biological meaning or motif clustering.")
print(f"- Combine this with Lasso/H2O results for robust intersection.")
print(f"- Consider retraining your model on just the top {n_80}–{n_95} features for a compact model.")

# 11. (Optional) Save top 80% and 95% feature lists
# importance_df_sorted.iloc[:n_80][['feature']].to_csv("top_80pct_features_xgb.csv", index=False)
# importance_df_sorted.iloc[:n_95][['feature']].to_csv("top_95pct_features_xgb.csv", index=False)

print(f"\n(If uncommented) Saved top 80% features to top_80pct_features_xgb.csv and top 95% to top_95pct_features_xgb.csv.")


In [ ]:
# 1. Load and Inspect Data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load variable importance table
varimp = pd.read_csv("full_feature_importance.csv")
print("Head of file:\n", varimp.head())
print("\nColumns:", varimp.columns.tolist())
print("\nDescribe:\n", varimp.describe())

# 2. Basic Stats
print("\nTotal features:", len(varimp))
print("Features with nonzero importance:", (varimp['scaled_importance'] > 0).sum())
print("Features with zero importance:", (varimp['scaled_importance'] == 0).sum())

# 3. Sort and Cumulative Importance
varimp_sorted = varimp.sort_values('scaled_importance', ascending=False).reset_index(drop=True)
varimp_sorted['cumulative_importance'] = varimp_sorted['scaled_importance'].cumsum()

# 4. Plots

# Histogram of importance
plt.figure(figsize=(10,5))
sns.histplot(varimp['scaled_importance'], bins=40, color='royalblue')
plt.title('Histogram of Scaled Feature Importances')
plt.xlabel('Scaled Importance')
plt.ylabel('Number of Features')
plt.tight_layout()
plt.show()

# Cumulative plot
plt.figure(figsize=(10,5))
plt.plot(varimp_sorted.index, varimp_sorted['cumulative_importance'], marker='o')
plt.title('Cumulative Feature Importance')
plt.xlabel('Number of Features (sorted)')
plt.ylabel('Cumulative Importance')
plt.axhline(0.8, color='red', linestyle='--', label='80%')
plt.axhline(0.95, color='orange', linestyle='--', label='95%')
plt.legend()
plt.tight_layout()
plt.show()

# 5. Top Features Table
print("\nTop 20 Features:\n", varimp_sorted[['variable','scaled_importance']].head(20))

# 6. How many features are needed for 80% / 95% of importance?
n_80 = np.argmax(varimp_sorted['cumulative_importance'] >= 0.8) + 1
n_95 = np.argmax(varimp_sorted['cumulative_importance'] >= 0.95) + 1
print(f"\nFeatures to reach 80% cumulative importance: {n_80}")
print(f"Features to reach 95% cumulative importance: {n_95}")

# 7. Where is the biggest drop in feature importance? (elbow)
diffs = np.diff(varimp_sorted['scaled_importance'].values)
biggest_drop_idx = np.argmax(np.abs(diffs))
print(f"\nBiggest single drop in importance: between features {biggest_drop_idx} and {biggest_drop_idx+1}")

# 8. Tail: How many features are <0.001 importance?
tail_count = (varimp_sorted['scaled_importance'] < 0.001).sum()
print(f"Features with scaled importance <0.001: {tail_count}")

# 9. Optional: Boxplot for feature importance
plt.figure(figsize=(7,2))
sns.boxplot(x=varimp_sorted['scaled_importance'], color='lightblue')
plt.title("Boxplot of Feature Importances")
plt.tight_layout()
plt.show()

# 10. Recommendations for Downstream Analysis
print("\n=== Actionable Recommendations ===")
print(f"- Consider keeping only the top {n_80}–{n_95} features (covering 80–95% cumulative importance).")
print(f"- {tail_count} features have negligible contribution (<0.001 importance); can likely be dropped.")
print(f"- Check the top 20 for biological meaning or motif clustering.")
print(f"- Combine this with Lasso/XGBoost results for robust intersection.")
print(f"- Consider retraining your model on just the top {n_80}–{n_95} features for a compact model.")

# 11. (Optional) Save top 80% and 95% feature lists
#varimp_sorted.iloc[:n_80][['variable']].to_csv("top_80pct_features.csv", index=False)
#varimp_sorted.iloc[:n_95][['variable']].to_csv("top_95pct_features.csv", index=False)

print(f"\nSaved top 80% features to top_80pct_features.csv and top 95% to top_95pct_features.csv.")


In [ ]:
# Install dependencies (if not already)
!pip install xgboost scikit-learn seaborn

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor

# === Load dataset ===
df = pd.read_csv("merged_unique_kmers_with_metadata.csv")

# === Drop metadata and extract features/target ===
drop_cols = ["sample name", "Biological Sex"]
target = "Age"
X = df.drop(columns=drop_cols + [target])
y = df[target]

# === Split data ===
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# === Train model ===
xgb_model = XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.1, n_jobs=-1, random_state=42)
xgb_model.fit(X_train, y_train)

# === Feature importance ===
importance_df = pd.DataFrame({
    "feature": X.columns,
    "importance": xgb_model.feature_importances_
}).sort_values(by="importance", ascending=False)

# === Plot ===
plt.figure(figsize=(8, 10))
sns.barplot(data=importance_df.head(30), x="importance", y="feature", palette="viridis")
plt.title("Top 30 k-mer Importances (XGBoost)")
plt.tight_layout()
plt.show()


In [ ]:
importance_df

In [ ]:
importance_df.to_csv("importance_df.csv", index=False)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Load importance data
importance_df = pd.read_csv("importance_df.csv")  # Replace with actual path if needed

# Sort for cumulative plot
importance_df_sorted = importance_df.sort_values("importance", ascending=False).reset_index(drop=True)
importance_df_sorted["cumulative_importance"] = importance_df_sorted["importance"].cumsum()

# Plot 1: Histogram of Importance Values
plt.figure(figsize=(8, 5))
sns.histplot(importance_df["importance"], bins=50, kde=True, color='teal')
plt.title("Distribution of XGBoost Feature Importances")
plt.xlabel("Importance")
plt.ylabel("Number of Features")
plt.grid(True, axis='y')
plt.tight_layout()
plt.show()

# Plot 2: Cumulative Importance Plot
plt.figure(figsize=(8, 5))
plt.plot(importance_df_sorted.index, importance_df_sorted["cumulative_importance"], marker='o')
plt.axhline(y=0.8, color='r', linestyle='--', label="80% cumulative importance")
plt.title("Cumulative Feature Importance")
plt.xlabel("Top N Features")
plt.ylabel("Cumulative Importance")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

# Plot 3: Top 20 Feature Names with Importance Values
print("\n🧠 Top 20 Most Important Features:")
print(importance_df_sorted.head(20).to_string(index=False))

# Plot 4: KDE with Rugplot
plt.figure(figsize=(8, 5))
sns.kdeplot(importance_df["importance"], fill=True, color='skyblue')
sns.rugplot(importance_df["importance"], height=0.05)
plt.title("Kernel Density of Feature Importance")
plt.xlabel("Importance")
plt.ylabel("Density")
plt.tight_layout()
plt.show()

# Plot 5: Log-Scaled Importance Histogram
plt.figure(figsize=(8, 5))
sns.histplot(np.log1p(importance_df["importance"]), bins=50, color='purple')
plt.title("Log-Scaled Feature Importance Histogram")
plt.xlabel("log(1 + Importance)")
plt.ylabel("Number of Features")
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc, classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
import seaborn as sns

# Load data
df = pd.read_csv("updated_tcr_age_lists_with_all_significance.csv.gz")

# Define thresholds
z_col = "component_zscore"
age_threshold = 1.96
non_age_min, non_age_max = -0.5, 0.5

# Create groups
age_related = df[df[z_col].abs() > age_threshold]
non_age_related = df[(df[z_col] > non_age_min) & (df[z_col] < non_age_max)]
sample_size = min(len(age_related), len(non_age_related))
old = df[df[z_col] > age_threshold]
young = df[df[z_col] < -age_threshold]
sample_size_old_young = min(len(old), len(young))

print(f"Age-related TCRs: {len(age_related)}")
print(f"Non-age-related TCRs: {len(non_age_related)}")
print(f"Old: {len(old)}, Young: {len(young)}")

# One-hot encoding helper (same as before)
def one_hot_encode_tcrs(tcr_list, max_len=None):
    aa_vocab = list("ACDEFGHIKLMNPQRSTVWY") + ["X"]
    aa_to_idx = {aa: i for i, aa in enumerate(aa_vocab)}
    max_len = max_len or max(len(seq) for seq in tcr_list)
    one_hot = np.zeros((len(tcr_list), max_len * len(aa_vocab)))
    for i, seq in enumerate(tcr_list):
        for j, aa in enumerate(seq[:max_len]):
            idx = aa_to_idx.get(aa, aa_to_idx["X"])
            one_hot[i, j * len(aa_vocab) + idx] = 1
    return one_hot

# Classification/ROC function for two groups
from sklearn.metrics import roc_curve, auc

def classify_and_roc_accum(group1_df, group2_df, label1, label2, n_seeds=5):
    combined_df = pd.concat([
        pd.DataFrame({'TCR': group1_df['TCR'], 'group': label1}),
        pd.DataFrame({'TCR': group2_df['TCR'], 'group': label2})
    ])
    X = one_hot_encode_tcrs(combined_df["TCR"])
    y = LabelEncoder().fit_transform(combined_df["group"])

    # For accumulation:
    all_y_true, all_y_proba = [], []

    aucs = []
    plt.figure(figsize=(8, 6))
    for seed in range(n_seeds):
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.25, stratify=y, random_state=seed
        )
        model = XGBClassifier(use_label_encoder=False, eval_metric="logloss", random_state=seed)
        model.fit(X_train, y_train)
        y_proba = model.predict_proba(X_test)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, y_proba)
        roc_auc = auc(fpr, tpr)
        aucs.append(roc_auc)
        # Faint, uniform color for all seeds
        plt.plot(fpr, tpr, color='gray', lw=1.2, alpha=0.30)
        # For accumulation
        all_y_true.extend(y_test)
        all_y_proba.extend(y_proba)

    # Plot accumulated ROC
    fpr_acc, tpr_acc, _ = roc_curve(all_y_true, all_y_proba)
    auc_acc = auc(fpr_acc, tpr_acc)
    plt.plot(fpr_acc, tpr_acc, color='blue', lw=3, label=f'Accumulated ROC (AUC={auc_acc:.2f})')
    plt.plot([0, 1], [0, 1], 'k--', lw=1)
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'ROC: {label1} vs {label2}')
    plt.legend(loc='lower right')
    plt.tight_layout()
    plt.show()
    print(f"Mean AUC over {n_seeds} seeds: {np.mean(aucs):.3f}")
    print(f"Accumulated ROC AUC: {auc_acc:.3f}\n")



# --- Non1 vs Non2 (two random samples from non_age_related) ---
non1 = non_age_related.sample(n=sample_size, random_state=1)
non2 = non_age_related.drop(non1.index).sample(n=sample_size, random_state=2)
print("\n===== Non1 vs Non2 =====")
classify_and_roc(non1, non2, "non1", "non2", n_seeds=5)

# --- Old vs Young ---
old_sample = old.sample(n=sample_size_old_young, random_state=1)
young_sample = young.sample(n=sample_size_old_young, random_state=2)
print("\n===== Old vs Young =====")
classify_and_roc(old_sample, young_sample, "old", "young", n_seeds=5)

# --- Age-related vs 5 different non-age-related groups (show ROC for each non-group and mean) ---
n_groups = 1
non_groups = []
non_age_remaining = non_age_related.copy()
for i in range(n_groups):
    chosen = non_age_remaining.sample(n=sample_size, random_state=100+i)
    non_groups.append(chosen)
    non_age_remaining = non_age_remaining.drop(chosen.index)

for i, nongroup in enumerate(non_groups, 1):
    print(f"\n===== Age-related vs non{i} =====")
    classify_and_roc(age_related, nongroup, "age-related", f"non{i}", n_seeds=5)


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
import matplotlib.pyplot as plt

# ---------------------------
# Load data and group creation
# ---------------------------
df = pd.read_csv("updated_tcr_age_lists_with_all_significance.csv.gz")

z_col = "component_zscore"
age_threshold = 1.96
non_age_min, non_age_max = -0.5, 0.5

age_related = df[df[z_col].abs() > age_threshold]
non_age_related = df[(df[z_col] > non_age_min) & (df[z_col] < non_age_max)]
sample_size = min(len(age_related), len(non_age_related))
old = df[df[z_col] > age_threshold]
young = df[df[z_col] < -age_threshold]
sample_size_old_young = min(len(old), len(young))

print(f"Age-related TCRs: {len(age_related)}")
print(f"Non-age-related TCRs: {len(non_age_related)}")
print(f"Old: {len(old)}, Young: {len(young)}")

# ---------------------------------------
# One-hot encoding helper
# ---------------------------------------
def one_hot_encode_tcrs(tcr_list, max_len=None):
    aa_vocab = list("ACDEFGHIKLMNPQRSTVWY") + ["X"]
    aa_to_idx = {aa: i for i, aa in enumerate(aa_vocab)}
    max_len = max_len or max(len(seq) for seq in tcr_list)
    one_hot = np.zeros((len(tcr_list), max_len * len(aa_vocab)))
    for i, seq in enumerate(tcr_list):
        for j, aa in enumerate(seq[:max_len]):
            idx = aa_to_idx.get(aa, aa_to_idx["X"])
            one_hot[i, j * len(aa_vocab) + idx] = 1
    return one_hot

# ---------------------------------------
# ROC plotting & results collection
# ---------------------------------------
def classify_and_roc_accum(group1_df, group2_df, label1, label2, n_seeds=5, show_plot=True):
    combined_df = pd.concat([
        pd.DataFrame({'TCR': group1_df['TCR'], 'group': label1}),
        pd.DataFrame({'TCR': group2_df['TCR'], 'group': label2})
    ])
    X = one_hot_encode_tcrs(combined_df["TCR"])
    y = LabelEncoder().fit_transform(combined_df["group"])

    all_y_true, all_y_proba = [], []
    aucs = []
    if show_plot:
        plt.figure(figsize=(9, 7))
    colors = plt.cm.tab10.colors

    for seed in range(n_seeds):
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.25, stratify=y, random_state=seed
        )
        model = XGBClassifier(use_label_encoder=False, eval_metric="logloss", random_state=seed)
        model.fit(X_train, y_train)
        y_proba = model.predict_proba(X_test)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, y_proba)
        roc_auc = auc(fpr, tpr)
        aucs.append(roc_auc)
        if show_plot:
            plt.plot(fpr, tpr,
                     color=colors[seed % len(colors)], lw=1.5, alpha=0.85,
                     label=f'Seed {seed+1} (AUC={roc_auc:.2f})')
        all_y_true.extend(y_test)
        all_y_proba.extend(y_proba)

    # Accumulated ROC
    fpr_acc, tpr_acc, _ = roc_curve(all_y_true, all_y_proba)
    auc_acc = auc(fpr_acc, tpr_acc)
    if show_plot:
        plt.plot(fpr_acc, tpr_acc, color='dodgerblue', lw=2, linestyle='--', alpha=0.7,
                 label=f'Accumulated ROC (AUC={auc_acc:.2f})')
        plt.plot([0, 1], [0, 1], 'k--', lw=1, label='No Skill')
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title(f'ROC: {label1} vs {label2}\n({n_seeds} seeds + Accumulated)')
        plt.legend(loc='lower right')
        plt.tight_layout()
        plt.show()
    print(f"Mean AUC over {n_seeds} seeds: {np.mean(aucs):.3f}")
    print(f"Accumulated ROC AUC: {auc_acc:.3f}\n")
    return fpr_acc, tpr_acc, auc_acc, f'{label1} vs {label2}'

# ---------------------------------------
# Run all three comparisons and store ROC data
# ---------------------------------------

# Non1 vs Non2
non1 = non_age_related.sample(n=sample_size, random_state=1)
non2 = non_age_related.drop(non1.index).sample(n=sample_size, random_state=2)
print("\n===== Non1 vs Non2 =====")
fpr1, tpr1, auc1, label1 = classify_and_roc_accum(non1, non2, "non1", "non2", n_seeds=5, show_plot=True)

# Old vs Young
old_sample = old.sample(n=sample_size_old_young, random_state=1)
young_sample = young.sample(n=sample_size_old_young, random_state=2)
print("\n===== Old vs Young =====")
fpr2, tpr2, auc2, label2 = classify_and_roc_accum(old_sample, young_sample, "old", "young", n_seeds=5, show_plot=True)

# Age-related vs Non-age-related (use only one random sample for non)
n_groups = 1
non_groups = []
non_age_remaining = non_age_related.copy()
for i in range(n_groups):
    chosen = non_age_remaining.sample(n=sample_size, random_state=100+i)
    non_groups.append(chosen)
    non_age_remaining = non_age_remaining.drop(chosen.index)

for i, nongroup in enumerate(non_groups, 1):
    print(f"\n===== Age-related vs non{i} =====")
    fpr3, tpr3, auc3, label3 = classify_and_roc_accum(age_related, nongroup, "age-related", f"non{i}", n_seeds=5, show_plot=True)

# ---------------------------------------
# Plot all three accumulated ROC curves together
# ---------------------------------------
plt.figure(figsize=(9, 7))
plt.plot(fpr1, tpr1, lw=2, color='red', linestyle='-', label=f'{label1} (AUC={auc1:.2f})')
plt.plot(fpr2, tpr2, lw=2, color='green', linestyle='-', label=f'{label2} (AUC={auc2:.2f})')
plt.plot(fpr3, tpr3, lw=2, color='blue', linestyle='-', label=f'{label3} (AUC={auc3:.2f})')
plt.plot([0, 1], [0, 1], 'k--', lw=1, label='No Skill')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Accumulated ROC Curves: All Comparisons')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()


In [ ]:
# Article-style palette: colorblind-friendly, distinct
SEED_COLORS = [
    '#1b9e77', '#d95f02', '#7570b3', '#e7298a', '#66a61e'  # green, orange, purple, pink, olive
]

roc_results = [res1, res2, res3]
roc_titles = ['non1 vs non2', 'old vs young', 'age-related vs non1']

fig, axs = plt.subplots(1, 3, figsize=(20, 6))
for plot_idx, (res, ax, title) in enumerate(zip(roc_results, axs, roc_titles)):
    # Plot seeds
    for i in range(5):
        fpr = res['fpr_list'][i]
        tpr = res['tpr_list'][i]
        auc_i = res['aucs'][i]
        ax.plot(fpr, tpr, color=SEED_COLORS[i], lw=1.8, label=f'Seed {i+1} (AUC={auc_i:.2f})')
    # Accumulated ROC
    ax.plot(res['fpr_acc'], res['tpr_acc'], color='dodgerblue', lw=2.8, linestyle='--',
            label=f'Accumulated ROC (AUC={res["auc_acc"]:.2f})')
    # No Skill
    ax.plot([0, 1], [0, 1], 'k--', lw=1, label='No Skill')
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(title)
    ax.legend(loc='lower right', fontsize=11)

plt.suptitle('ROC Curves: All Comparisons (5 seeds + Accumulated)', fontsize=16)
plt.tight_layout(rect=[0, 0.03, 1, 0.97])
plt.show()


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, auc, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
import seaborn as sns

# ARTICLE COLORS
ARTICLE_COLORS = ['#1b9e77', '#d95f02', '#7570b3']  # green, orange, purple

# --- Data & groups ---
df = pd.read_csv("updated_tcr_age_lists_with_all_significance.csv.gz")
z_col = "component_zscore"
age_threshold = 1.96
non_age_min, non_age_max = -0.5, 0.5
age_related = df[df[z_col].abs() > age_threshold]
non_age_related = df[(df[z_col] > non_age_min) & (df[z_col] < non_age_max)]
sample_size = min(len(age_related), len(non_age_related))
old = df[df[z_col] > age_threshold]
young = df[df[z_col] < -age_threshold]
sample_size_old_young = min(len(old), len(young))

# --- One-hot encoding ---
def one_hot_encode_tcrs(tcr_list, max_len=None):
    aa_vocab = list("ACDEFGHIKLMNPQRSTVWY") + ["X"]
    aa_to_idx = {aa: i for i, aa in enumerate(aa_vocab)}
    max_len = max_len or max(len(seq) for seq in tcr_list)
    one_hot = np.zeros((len(tcr_list), max_len * len(aa_vocab)))
    for i, seq in enumerate(tcr_list):
        for j, aa in enumerate(seq[:max_len]):
            idx = aa_to_idx.get(aa, aa_to_idx["X"])
            one_hot[i, j * len(aa_vocab) + idx] = 1
    return one_hot

# --- Classification, ROC and CM ---
def classify_roc_cm(group1_df, group2_df, label1, label2, n_seeds=5):
    combined_df = pd.concat([
        pd.DataFrame({'TCR': group1_df['TCR'], 'group': label1}),
        pd.DataFrame({'TCR': group2_df['TCR'], 'group': label2})
    ])
    X = one_hot_encode_tcrs(combined_df["TCR"])
    y = LabelEncoder().fit_transform(combined_df["group"])
    y_true_accum, y_pred_accum, y_proba_accum = [], [], []
    fpr_list, tpr_list, aucs, cms = [], [], [], []
    for seed in range(n_seeds):
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.25, stratify=y, random_state=seed
        )
        model = XGBClassifier(use_label_encoder=False, eval_metric="logloss", random_state=seed)
        model.fit(X_train, y_train)
        y_proba = model.predict_proba(X_test)[:, 1]
        y_pred = model.predict(X_test)
        fpr, tpr, _ = roc_curve(y_test, y_proba)
        roc_auc = auc(fpr, tpr)
        aucs.append(roc_auc)
        fpr_list.append(fpr)
        tpr_list.append(tpr)
        cms.append(confusion_matrix(y_test, y_pred))
        # For accum
        y_true_accum.extend(y_test)
        y_pred_accum.extend(y_pred)
        y_proba_accum.extend(y_proba)
    # Accumulated ROC
    fpr_acc, tpr_acc, _ = roc_curve(y_true_accum, y_proba_accum)
    auc_acc = auc(fpr_acc, tpr_acc)
    # Averaged confusion matrix
    cm_mean = np.mean(np.stack(cms), axis=0)
    return {
        'fpr_acc': fpr_acc, 'tpr_acc': tpr_acc, 'auc_acc': auc_acc,
        'aucs': aucs, 'cms': cms, 'cm_mean': cm_mean,
        'fpr_list': fpr_list, 'tpr_list': tpr_list
    }

# --- Run all comparisons ---
non1 = non_age_related.sample(n=sample_size, random_state=1)
non2 = non_age_related.drop(non1.index).sample(n=sample_size, random_state=2)
res1 = classify_roc_cm(non1, non2, "non1", "non2", n_seeds=5)

old_sample = old.sample(n=sample_size_old_young, random_state=1)
young_sample = young.sample(n=sample_size_old_young, random_state=2)
res2 = classify_roc_cm(old_sample, young_sample, "old", "young", n_seeds=5)

non_group = non_age_related.sample(n=sample_size, random_state=100)
res3 = classify_roc_cm(age_related, non_group, "age-related", "non1", n_seeds=5)

# For labels in all plots
LABELS = [
    f'non1 vs non2 (AUC={res1["auc_acc"]:.2f})',
    f'old vs young (AUC={res2["auc_acc"]:.2f})',
    f'age-related vs non1 (AUC={res3["auc_acc"]:.2f})'
]

# ==============================
# 1. ALL ROC on one plot
# ==============================
plt.figure(figsize=(9, 7))
for i, (res, color, label) in enumerate(zip([res1, res2, res3], ARTICLE_COLORS, LABELS)):
    plt.plot(res['fpr_acc'], res['tpr_acc'], lw=2.5, color=color, label=label)
plt.plot([0, 1], [0, 1], 'k--', lw=1, label='No Skill')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Accumulated ROC Curves: All Comparisons')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

# ==============================
# 2. Three ROC subplots
# ==============================
fig, axs = plt.subplots(1, 3, figsize=(16, 5))
for i, (res, color, label) in enumerate(zip([res1, res2, res3], ARTICLE_COLORS, LABELS)):
    axs[i].plot(res['fpr_acc'], res['tpr_acc'], lw=2.5, color=color, label=label)
    axs[i].plot([0, 1], [0, 1], 'k--', lw=1)
    axs[i].set_xlabel('False Positive Rate')
    axs[i].set_ylabel('True Positive Rate')
    axs[i].set_title(label)
    axs[i].legend(loc='lower right')
plt.tight_layout()
plt.show()

# ==============================
# 3. Three mean confusion matrix subplots
# ==============================
fig, axs = plt.subplots(1, 3, figsize=(15, 5))
cm_titles = ['non1 vs non2', 'old vs young', 'age-related vs non1']
for i, (res, title) in enumerate(zip([res1, res2, res3], cm_titles)):
    sns.heatmap(res['cm_mean'], annot=True, fmt='.1f', cmap='Blues', ax=axs[i], cbar=False,
                xticklabels=['Neg', 'Pos'], yticklabels=['Neg', 'Pos'])
    axs[i].set_title(f'Avg. Confusion Matrix\n{title}')
    axs[i].set_xlabel('Predicted')
    axs[i].set_ylabel('Actual')
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# --- Confusion matrices ---
cm1 = np.array([[943.2, 932.8],
                [911.0, 965.0]])

cm2 = np.array([[777.0, 149.8],
                [139.6, 786.6]])

cm3 = np.array([[1072.4, 803.6],
                [601.2, 1274.8]])

# --- Shared normalization for 1st and 3rd only ---
shared_values = np.concatenate([cm1.ravel(), cm3.ravel()])
vmin, vmax = shared_values.min(), shared_values.max()

# --- Plot ---
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
titles = [
    "Avg. Confusion Matrix\nnon1 vs non2",
    "Avg. Confusion Matrix\nold vs young ",
    "Avg. Confusion Matrix\nage-related vs non1"
]

# Apply normalization only to 1st and 3rd
sns.heatmap(cm1, annot=True, fmt=".1f", cmap="Blues",
            vmin=vmin, vmax=vmax, cbar=False, ax=axes[0],
             annot_kws={"size": 12})
sns.heatmap(cm2, annot=True, fmt=".1f", cmap="Blues",
            cbar=False, ax=axes[1],
             annot_kws={"size": 12})  # own independent scale
sns.heatmap(cm3, annot=True, fmt=".1f", cmap="Blues",
            vmin=vmin, vmax=vmax, cbar=False, ax=axes[2], annot_kws={"size": 12})

# --- Styling ---
for ax, title in zip(axes, titles):
    ax.set_title(title, fontsize=12)
    ax.set_xlabel("Predicted", fontsize=11)
    ax.set_ylabel("Actual", fontsize=11)
    ax.set_xticklabels(["Neg", "Pos"], fontsize=10)
    ax.set_yticklabels(["Neg", "Pos"], rotation=0, fontsize=10)

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

# Simulate fake data
n_seeds = 5
n = 100  # Samples per seed
rng = np.random.RandomState(42)

all_y_true = []
all_y_proba = []
aucs = []

plt.figure(figsize=(9, 7))
for seed in range(n_seeds):
    y_true = rng.randint(0, 2, n)
    # Simulate a classifier with variable noise
    y_score = 0.5 * y_true + rng.normal(0, 0.3 + seed*0.1, n)
    y_score = (y_score - y_score.min()) / (y_score.max() - y_score.min())  # Normalize

    fpr, tpr, _ = roc_curve(y_true, y_score)
    roc_auc = auc(fpr, tpr)
    aucs.append(roc_auc)
    plt.plot(fpr, tpr, lw=1.5, alpha=0.6, label=f"Seed {seed+1} (AUC={roc_auc:.2f})")

    all_y_true.extend(y_true)
    all_y_proba.extend(y_score)

# Accumulated ROC
fpr_acc, tpr_acc, _ = roc_curve(all_y_true, all_y_proba)
auc_acc = auc(fpr_acc, tpr_acc)
plt.plot(fpr_acc, tpr_acc, color='blue', lw=3, label=f'Accumulated ROC (AUC={auc_acc:.2f})')

plt.plot([0, 1], [0, 1], 'k--', lw=1, label='No Skill')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC with Fake Data\n(5 Seeds + Accumulated)')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
import seaborn as sns

# Load data
df = pd.read_csv("updated_tcr_age_lists_with_all_significance.csv.gz")

# Define thresholds
z_col = "component_zscore"
age_threshold = 1.96
non_age_min, non_age_max = -0.5, 0.5

# Define age and non-age groups
age_related = df[df[z_col].abs() > age_threshold]
non_age_related = df[(df[z_col] > non_age_min) & (df[z_col] < non_age_max)]

sample_size = len(age_related)
print(f"Age-related TCRs: {sample_size}")

# Helper: one-hot encoding
def one_hot_encode_tcrs(tcr_list, max_len=None):
    aa_vocab = list("ACDEFGHIKLMNPQRSTVWY") + ["X"]
    aa_to_idx = {aa: i for i, aa in enumerate(aa_vocab)}
    max_len = max_len or max(len(seq) for seq in tcr_list)
    one_hot = np.zeros((len(tcr_list), max_len * len(aa_vocab)))
    for i, seq in enumerate(tcr_list):
        for j, aa in enumerate(seq[:max_len]):
            idx = aa_to_idx.get(aa, aa_to_idx["X"])
            one_hot[i, j * len(aa_vocab) + idx] = 1
    return one_hot

# Classification function
def classify_groups(group1_df, group2_df, label1, label2):
    combined_df = pd.concat([
        pd.DataFrame({'TCR': group1_df['TCR'], 'group': label1}),
        pd.DataFrame({'TCR': group2_df['TCR'], 'group': label2})
    ])
    X = one_hot_encode_tcrs(combined_df["TCR"])
    y = LabelEncoder().fit_transform(combined_df["group"])
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, stratify=y, random_state=42
    )
    model = XGBClassifier(use_label_encoder=False, eval_metric="logloss", random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    report = classification_report(y_test, y_pred, target_names=[label1, label2])
    cm = confusion_matrix(y_test, y_pred)
    print(f"\nClassification: {label1} vs {label2}\n")
    print(report)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=[label1, label2], yticklabels=[label1, label2])
    plt.title(f"Confusion Matrix: {label1} vs {label2}")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.show()

# === Step 1: Create 5 random non-age-related groups ===
non_groups = []
non_age_remaining = non_age_related.copy()
rng = np.random.RandomState(42)  # For reproducibility

for i in range(5):
    chosen = non_age_remaining.sample(n=sample_size, random_state=100+i)
    non_groups.append(chosen)
    non_age_remaining = non_age_remaining.drop(chosen.index)

# === Step 2: Age-related vs each non group ===
for i, nongroup in enumerate(non_groups, 1):
    print(f"\n===== Age-related vs non{i} =====")
    classify_groups(age_related, nongroup, "age-related", f"non{i}")

# === Step 3: Each non group vs each other ===
for i in range(5):
    for j in range(i+1, 5):
        print(f"\n===== non{i+1} vs non{j+1} =====")
        classify_groups(non_groups[i], non_groups[j], f"non{i+1}", f"non{j+1}")


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
import seaborn as sns

# Load data
df = pd.read_csv("updated_tcr_age_lists_with_all_significance.csv.gz")

# Define groups
z_col = "component_zscore"
age_threshold = 1.96
non_age_min, non_age_max = -0.5, 0.5

age_related = df[df[z_col].abs() > age_threshold]
non_age_related = df[(df[z_col] > non_age_min) & (df[z_col] < non_age_max)]

sample_size = len(age_related)
group_labels = ['age-related'] + [f'non{i+1}' for i in range(5)]
groups = [age_related]
non_age_remaining = non_age_related.copy()

rng = np.random.RandomState(42)
for i in range(5):
    chosen = non_age_remaining.sample(n=sample_size, random_state=100+i)
    groups.append(chosen)
    non_age_remaining = non_age_remaining.drop(chosen.index)

# One-hot encoding
def one_hot_encode_tcrs(tcr_list, max_len=None):
    aa_vocab = list("ACDEFGHIKLMNPQRSTVWY") + ["X"]
    aa_to_idx = {aa: i for i, aa in enumerate(aa_vocab)}
    max_len = max_len or max(len(seq) for seq in tcr_list)
    one_hot = np.zeros((len(tcr_list), max_len * len(aa_vocab)))
    for i, seq in enumerate(tcr_list):
        for j, aa in enumerate(seq[:max_len]):
            idx = aa_to_idx.get(aa, aa_to_idx["X"])
            one_hot[i, j * len(aa_vocab) + idx] = 1
    return one_hot

# Pairwise accuracy calculation
def pairwise_accuracy(df1, df2, label1, label2):
    combined_df = pd.concat([
        pd.DataFrame({'TCR': df1['TCR'], 'group': label1}),
        pd.DataFrame({'TCR': df2['TCR'], 'group': label2})
    ])
    X = one_hot_encode_tcrs(combined_df["TCR"])
    y = LabelEncoder().fit_transform(combined_df["group"])
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, stratify=y, random_state=42
    )
    model = XGBClassifier(use_label_encoder=False, eval_metric="logloss", random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return accuracy_score(y_test, y_pred)

# Build accuracy matrix
n = len(groups)
acc_matrix = np.zeros((n, n))

for i in range(n):
    for j in range(n):
        if i == j:
            acc_matrix[i, j] = 1.0
        else:
            print(f"Classifying: {group_labels[i]} vs {group_labels[j]}")
            acc = pairwise_accuracy(groups[i], groups[j], group_labels[i], group_labels[j])
            acc_matrix[i, j] = acc

# Plot heatmap
plt.figure(figsize=(8,6))
sns.heatmap(acc_matrix, annot=True, fmt=".2f", cmap="viridis", xticklabels=group_labels, yticklabels=group_labels)
plt.title("Pairwise TCR Group Classification Accuracy")
plt.xlabel("Group (comparison)")
plt.ylabel("Group (reference)")
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
import seaborn as sns

# Load data
df = pd.read_csv("updated_tcr_age_lists_with_all_significance.csv.gz")

z_col = "component_zscore"
age_pos = df[df[z_col] > 1.96]
age_neg = df[df[z_col] < -1.96]
non_age = df[(df[z_col] > -0.5) & (df[z_col] < 0.5)]

# Downsample so pos/neg have the same size
size = min(len(age_pos), len(age_neg))
age_pos = age_pos.sample(n=size, random_state=1)
age_neg = age_neg.sample(n=size, random_state=2)

print(f"Positive age-related: {len(age_pos)}, Negative age-related: {len(age_neg)}")

# 5 random non-age groups of same size
non_groups = []
non_age_remaining = non_age.copy()
rng = np.random.RandomState(42)
for i in range(5):
    chosen = non_age_remaining.sample(n=size, random_state=100+i)
    non_groups.append(chosen)
    non_age_remaining = non_age_remaining.drop(chosen.index)

group_labels = ['positive', 'negative'] + [f'non{i+1}' for i in range(5)]
groups = [age_pos, age_neg] + non_groups

# One-hot encoding
def one_hot_encode_tcrs(tcr_list, max_len=None):
    aa_vocab = list("ACDEFGHIKLMNPQRSTVWY") + ["X"]
    aa_to_idx = {aa: i for i, aa in enumerate(aa_vocab)}
    max_len = max_len or max(len(seq) for seq in tcr_list)
    one_hot = np.zeros((len(tcr_list), max_len * len(aa_vocab)))
    for i, seq in enumerate(tcr_list):
        for j, aa in enumerate(seq[:max_len]):
            idx = aa_to_idx.get(aa, aa_to_idx["X"])
            one_hot[i, j * len(aa_vocab) + idx] = 1
    return one_hot

# Pairwise accuracy calculation
def pairwise_accuracy(df1, df2, label1, label2):
    combined_df = pd.concat([
        pd.DataFrame({'TCR': df1['TCR'], 'group': label1}),
        pd.DataFrame({'TCR': df2['TCR'], 'group': label2})
    ])
    X = one_hot_encode_tcrs(combined_df["TCR"])
    y = LabelEncoder().fit_transform(combined_df["group"])
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.25, stratify=y, random_state=42
    )
    model = XGBClassifier(use_label_encoder=False, eval_metric="logloss", random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    return accuracy_score(y_test, y_pred)

# Build accuracy matrix
n = len(groups)
acc_matrix = np.zeros((n, n))

for i in range(n):
    for j in range(n):
        if i == j:
            acc_matrix[i, j] = 1.0
        else:
            print(f"Classifying: {group_labels[i]} vs {group_labels[j]}")
            acc = pairwise_accuracy(groups[i], groups[j], group_labels[i], group_labels[j])
            acc_matrix[i, j] = acc

# Plot heatmap
plt.figure(figsize=(9,7))
sns.heatmap(acc_matrix, annot=True, fmt=".2f", cmap="magma", xticklabels=group_labels, yticklabels=group_labels)
plt.title("Pairwise TCR Group Classification Accuracy")
plt.xlabel("Group (comparison)")
plt.ylabel("Group (reference)")
plt.tight_layout()
plt.show()


In [ ]:


# Plot heatmap
plt.figure(figsize=(9,7))
sns.heatmap(acc_matrix, annot=True, fmt=".2f", cmap="viridis", xticklabels=group_labels, yticklabels=group_labels)
plt.title("Pairwise TCR Group Classification Accuracy")
plt.xlabel("Group (comparison)")
plt.ylabel("Group (reference)")
plt.tight_layout()
plt.show()

In [ ]:
# Install H2O in Colab (run first)
!pip install -f http://h2o-release.s3.amazonaws.com/h2o/latest_stable_Py.html h2o

import h2o
from h2o.estimators.gbm import H2OGradientBoostingEstimator
from h2o.estimators.random_forest import H2ORandomForestEstimator
from h2o.estimators.xgboost import H2OXGBoostEstimator  # May require GPU or h2o >=3.32
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
# --- Install and Import ---
import h2o
from h2o.automl import H2OAutoML
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# --- Helper: One-hot encode TCRs ---
def one_hot_encode_tcrs(tcr_list, max_len=None):
    aa_vocab = list("ACDEFGHIKLMNPQRSTVWY") + ["X"]
    aa_to_idx = {aa: i for i, aa in enumerate(aa_vocab)}
    max_len = max_len or max(len(seq) for seq in tcr_list)
    one_hot = np.zeros((len(tcr_list), max_len * len(aa_vocab)))
    for i, seq in enumerate(tcr_list):
        for j, aa in enumerate(seq[:max_len]):
            idx = aa_to_idx.get(aa, aa_to_idx["X"])
            one_hot[i, j * len(aa_vocab) + idx] = 1
    return one_hot

# --- Load data ---
df = pd.read_csv("updated_tcr_age_lists_with_all_significance.csv.gz")
z_col = "component_zscore"
age_threshold = 1.96
non_age_min, non_age_max = -0.5, 0.5

old_related = df[df[z_col] > age_threshold]
young_related = df[df[z_col] < -age_threshold]
age_related = df[df[z_col].abs() > age_threshold]
non_age_related = df[(df[z_col] > non_age_min) & (df[z_col] < non_age_max)]

# === Define both comparisons ===
comparisons = [
    # (label, group1_df, group2_df, group1_name, group2_name, output_csv)
    ("old_vs_young", old_related, young_related, "old", "young", "h2o_test_predictions_old_vs_young.csv"),
    ("age_vs_non", age_related, non_age_related.sample(n=len(age_related), random_state=42), "age", "non", "h2o_test_predictions_age_vs_non.csv"),
]

# --- H2O setup ---
h2o.init(max_mem_size="12G", nthreads=-1)

for label, g1_df, g2_df, g1_name, g2_name, output_csv in comparisons:
    print(f"\n========== {g1_name} vs {g2_name} ==========")
    g1_data = pd.DataFrame({'TCR': g1_df['TCR'], 'group': g1_name})
    g2_data = pd.DataFrame({'TCR': g2_df['TCR'], 'group': g2_name})
    data = pd.concat([g1_data, g2_data], ignore_index=True)
    data['group'] = data['group'].astype("category")

    # One-hot encoding
    X = one_hot_encode_tcrs(data['TCR'])
    X = pd.DataFrame(X, columns=[f"pos{i}_{aa}" for i in range(X.shape[1]//21) for aa in list("ACDEFGHIKLMNPQRSTVWY") + ["X"]])
    full_data = pd.concat([data.reset_index(drop=True), X], axis=1)

    # Train/test split
    train, test = train_test_split(full_data, test_size=0.25, stratify=full_data['group'], random_state=42)
    hf_train = h2o.H2OFrame(train)
    hf_test = h2o.H2OFrame(test)
    features = [c for c in hf_train.columns if c not in ["group", "TCR"]]
    target = "group"

    # --- AutoML for classification ---
    aml = H2OAutoML(
        max_models=30,            # You can increase if you have time
        max_runtime_secs=1800,    # 30 minutes max
        seed=42,
        sort_metric="AUC",
        balance_classes=True
    )
    aml.train(x=features, y=target, training_frame=hf_train, leaderboard_frame=hf_test)

    # Get test predictions
    preds = aml.leader.predict(hf_test).as_data_frame()
    test_out = test.copy()
    test_out["pred_label"] = preds["predict"].values
    # Add probability columns for each class
    prob_cols = [c for c in preds.columns if c != "predict"]
    for c in prob_cols:
        test_out[f"prob_{c}"] = preds[c].values

    # Save everything to CSV
    test_out.to_csv(output_csv, index=False)
    print(f"✅ Test predictions saved to: {output_csv}")

# --- Optionally, shutdown H2O ---
# h2o.shutdown(prompt=False)


In [ ]:
# --- Imports ---
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_curve,
    auc
)

# --- Plot settings ---
sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

# --- Helper function to analyze a result file ---
def analyze_predictions(file_path, positive_class):
    print(f"\n🔍 Analyzing: {file_path}")
    df = pd.read_csv(file_path)

    # --- Confusion Matrix ---
    cm = confusion_matrix(df['group'], df['pred_label'], labels=df['group'].unique())
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=df['group'].unique())
    disp.plot(cmap="Blues")
    plt.title(f"Confusion Matrix ({file_path})")
    plt.show()

    # --- ROC Curve ---
    y_true = (df['group'] == positive_class).astype(int)
    y_score = df[f"prob_{positive_class}"]
    fpr, tpr, _ = roc_curve(y_true, y_score)
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.2f}")
    plt.plot([0, 1], [0, 1], 'k--')
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(f"ROC Curve ({file_path})")
    plt.legend()
    plt.show()

    # --- Probability Histogram ---
    sns.histplot(data=df, x=f"prob_{positive_class}", hue="group", bins=20, kde=True, stat="density")
    plt.title(f"Probability Distribution: {positive_class} class")
    plt.xlabel(f"Probability of {positive_class}")
    plt.show()

    # --- Confidence vs Accuracy ---
    prob_cols = [col for col in df.columns if col.startswith("prob_")]
    df["confidence"] = df[prob_cols].max(axis=1)
    df["correct"] = df["group"] == df["pred_label"]
    df["confidence_bucket"] = pd.cut(df["confidence"], bins=[0,0.6,0.7,0.8,0.9,1.0])

    bucket_acc = df.groupby("confidence_bucket")["correct"].mean().reset_index()
    sns.barplot(x="confidence_bucket", y="correct", data=bucket_acc)
    plt.ylabel("Accuracy")
    plt.title(f"Confidence vs Accuracy ({file_path})")
    plt.xticks(rotation=45)
    plt.show()

    # --- Top confident misclassifications ---
    misclassified = df[df["group"] != df["pred_label"]].copy()
    misclassified["confidence"] = df[prob_cols].max(axis=1)
    print("\nTop 10 confident misclassifications:")
    print(misclassified.sort_values("confidence", ascending=False)[["TCR", "group", "pred_label", "confidence"]].head(10))


# --- Analyze both results ---
analyze_predictions("h2o_test_predictions_old_vs_young.csv", positive_class="old")
analyze_predictions("h2o_test_predictions_age_vs_non.csv", positive_class="age")

# --- Optional: Compare across tasks ---
df1 = pd.read_csv("h2o_test_predictions_old_vs_young.csv")
df1["task"] = "old_vs_young"
df2 = pd.read_csv("h2o_test_predictions_age_vs_non.csv")
df2["task"] = "age_vs_non"
df_combined = pd.concat([df1, df2])

if "prob_old" in df_combined.columns:
    sns.boxplot(data=df_combined, x="task", y="prob_old")
    plt.title("Comparison of 'old' Probabilities Across Tasks")
    plt.show()


In [ ]:
# --- Imports ---
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    roc_curve, auc, accuracy_score
)

# --- Plot settings ---
sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)

# --- Confusion Matrix Plot ---
def plot_confusion_matrix(df, file_label, positive_class):
    labels = df['group'].unique()
    cm = confusion_matrix(df['group'], df['pred_label'], labels=labels)
    acc = accuracy_score(df['group'], df['pred_label'])

    fig, ax = plt.subplots()
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
    disp.plot(cmap="Blues", ax=ax, values_format='d', colorbar=True)
    plt.grid(False)
    plt.title(f"Confusion Matrix ({file_label})\nAccuracy: {acc:.3f}")
    plt.tight_layout()
    plt.show()
    return acc

# --- ROC Curve Plot ---
def plot_roc_curve(df, positive_class, label):
    y_true = (df['group'] == positive_class).astype(int)
    y_score = df[f"prob_{positive_class}"]
    fpr, tpr, _ = roc_curve(y_true, y_score)
    auc_score = auc(fpr, tpr)
    return fpr, tpr, auc_score, label

# --- Confidence vs Accuracy Plot ---
def confidence_accuracy_plot(df, title):
    prob_cols = [col for col in df.columns if col.startswith("prob_")]
    df["confidence"] = df[prob_cols].max(axis=1)
    df["correct"] = df["group"] == df["pred_label"]
    df["confidence_bucket"] = pd.cut(df["confidence"], bins=[0,0.6,0.7,0.8,0.9,1.0])

    acc_by_conf = df.groupby("confidence_bucket")["correct"].mean().reset_index()
    sns.barplot(x="confidence_bucket", y="correct", data=acc_by_conf)
    plt.ylabel("Accuracy")
    plt.title(f"{title}: Confidence vs Accuracy")
    plt.xticks(rotation=45)
    plt.ylim(0, 1)
    plt.show()

# --- Individual File Analysis ---
def analyze_file(file_path, positive_class, label):
    df = pd.read_csv(file_path)
    print(f"\n🔍 Analyzing: {file_path}")

    acc = plot_confusion_matrix(df, label, positive_class)

    # ROC Curve
    fpr, tpr, auc_score, _ = plot_roc_curve(df, positive_class, label)
    plt.plot(fpr, tpr, label=f"{label} (AUC = {auc_score:.3f})")

    # Probability Histogram
    sns.histplot(data=df, x=f"prob_{positive_class}", hue="group", bins=20, kde=True, stat="density")
    plt.title(f"{label}: Probability Distribution for '{positive_class}'")
    plt.xlabel(f"Probability of {positive_class}")
    plt.show()

    # Confidence vs Accuracy
    confidence_accuracy_plot(df, title=label)

    return df, acc, auc_score

# --- Run analysis on both files ---
df1, acc1, auc1 = analyze_file("h2o_test_predictions_old_vs_young.csv", positive_class="old", label="Old vs Young")
df2, acc2, auc2 = analyze_file("h2o_test_predictions_age_vs_non.csv", positive_class="age", label="Age vs Non-Age")

# --- Combined ROC Plot ---
fpr1, tpr1, _, _ = plot_roc_curve(df1, "old", "Old vs Young")
fpr2, tpr2, _, _ = plot_roc_curve(df2, "age", "Age vs Non-Age")
plt.plot(fpr1, tpr1, label=f"Old vs Young (AUC = {auc1:.3f})")
plt.plot(fpr2, tpr2, label=f"Age vs Non (AUC = {auc2:.3f})")
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend()
plt.show()

# --- Combined Accuracy Table ---
summary = pd.DataFrame({
    "Task": ["Old vs Young", "Age vs Non-Age"],
    "Accuracy": [acc1, acc2],
    "AUC": [auc1, auc2]
})
print("\n📋 Accuracy and AUC Summary:")
print(summary)

# --- Combined Confidence vs Accuracy ---
df1["task"] = "Old vs Young"
df2["task"] = "Age vs Non-Age"
combined = pd.concat([df1, df2])
combined["confidence"] = combined[[c for c in combined.columns if c.startswith("prob_")]].max(axis=1)
combined["correct"] = combined["group"] == combined["pred_label"]
combined["confidence_bucket"] = pd.cut(combined["confidence"], bins=[0,0.6,0.7,0.8,0.9,1.0])

bucket_grouped = combined.groupby(["task", "confidence_bucket"])["correct"].mean().reset_index()
sns.barplot(data=bucket_grouped, x="confidence_bucket", y="correct", hue="task")
plt.title("Confidence vs Accuracy Across Tasks")
plt.ylabel("Accuracy")
plt.xticks(rotation=45)
plt.ylim(0, 1)
plt.show()


In [ ]:

# Plot heatmap
plt.figure(figsize=(9,7))
sns.heatmap(acc_matrix, annot=True, fmt=".2f", cmap="magma", xticklabels=group_labels, yticklabels=group_labels)
plt.title("Pairwise TCR Group Classification Accuracy")
plt.xlabel("Group (comparison)")
plt.ylabel("Group (reference)")
plt.tight_layout()
plt.show()pip install logomaker

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import logomaker

# === Padding helper ===
def pad_sequences(series, pad_char="X"):
    max_len = series.map(len).max()
    return series.map(lambda x: x + pad_char * (max_len - len(x)))

# === Clean and pad ===
def clean_tcrs(tcr_series):
    return (
        tcr_series.dropna()
        .astype(str)
        .map(str.strip)
        .loc[lambda x: x != ""]
    )

younger_tcrs = clean_tcrs(df[df["component_zscore"] < -1.96]["TCR"])
older_tcrs = clean_tcrs(df[df["component_zscore"] > 1.96]["TCR"])

# Pad to equal length
younger_tcrs_padded = pad_sequences(younger_tcrs)
older_tcrs_padded = pad_sequences(older_tcrs)

# === Function to create logo dataframe ===
def create_logo_df(tcr_series):
    tcr_df = pd.DataFrame([list(seq) for seq in tcr_series])
    counts_df = tcr_df.apply(lambda col: col.value_counts(normalize=True)).fillna(0)
    return counts_df.T  # logomaker expects positions as rows

# === Create logos ===
fig, axes = plt.subplots(2, 1, figsize=(16, 10))

# Younger group
younger_logo = create_logo_df(younger_tcrs_padded)
logomaker.Logo(younger_logo, ax=axes[0])
axes[0].set_title(f"Younger-biased TCRs (n={len(younger_tcrs_padded)})")
axes[0].set_ylabel("Frequency")
axes[0].set_xlabel("Position")

# Older group
older_logo = create_logo_df(older_tcrs_padded)
logomaker.Logo(older_logo, ax=axes[1])
axes[1].set_title(f"Older-biased TCRs (n={len(older_tcrs_padded)})")
axes[1].set_ylabel("Frequency")
axes[1].set_xlabel("Position")

plt.tight_layout()
plt.show()


In [ ]:
# === Define color scheme for amino acids ===
amino_acid_colors = {
    'A': 'gray',   # Ala - nonpolar
    'C': 'gold',   # Cys - polar
    'D': 'red',    # Asp - acidic
    'E': 'red',    # Glu - acidic
    'F': 'purple', # Phe - nonpolar aromatic
    'G': 'lightgray', # Gly - nonpolar
    'H': 'blue',   # His - basic
    'I': 'gray',   # Ile - nonpolar
    'K': 'blue',   # Lys - basic
    'L': 'gray',   # Leu - nonpolar
    'M': 'gray',   # Met - nonpolar
    'N': 'green',  # Asn - polar
    'P': 'orange', # Pro - nonpolar
    'Q': 'green',  # Gln - polar
    'R': 'blue',   # Arg - basic
    'S': 'green',  # Ser - polar
    'T': 'green',  # Thr - polar
    'V': 'gray',   # Val - nonpolar
    'W': 'purple', # Trp - aromatic
    'Y': 'purple', # Tyr - aromatic
    'X': 'white'   # Padding character
}

# === Create logos with color ===
fig, axes = plt.subplots(2, 1, figsize=(16, 8))

# Younger group
younger_logo = create_logo_df(younger_tcrs_padded)
logomaker.Logo(younger_logo, ax=axes[0], color_scheme=amino_acid_colors)
axes[0].set_title(f"Younger-biased TCRs (n={len(younger_tcrs_padded)})")
axes[0].set_ylabel("Frequency")
axes[0].set_xlabel("Position")

# Older group
older_logo = create_logo_df(older_tcrs_padded)
logomaker.Logo(older_logo, ax=axes[1], color_scheme=amino_acid_colors)
axes[1].set_title(f"Older-biased TCRs (n={len(older_tcrs_padded)})")
axes[1].set_ylabel("Frequency")
axes[1].set_xlabel("Position")

plt.tight_layout()
plt.show()


In [ ]:
# Get the maximum CDR3 length from both groups
max_len = max(younger_tcrs.map(len).max(), older_tcrs.map(len).max())

# Pad to same max length
def pad_to_length(series, length, pad_char="X"):
    return series.map(lambda x: x + pad_char * (length - len(x)))

younger_tcrs_padded = pad_to_length(younger_tcrs, max_len)
older_tcrs_padded = pad_to_length(older_tcrs, max_len)


In [ ]:
# === Create logos with color ===
fig, axes = plt.subplots(2, 1, figsize=(16, 8))

# Younger group
younger_logo = create_logo_df(younger_tcrs_padded)
logomaker.Logo(younger_logo, ax=axes[0], color_scheme=amino_acid_colors)
axes[0].set_title(f"Younger-biased TCRs (n={len(younger_tcrs_padded)})")
axes[0].set_ylabel("Frequency")
axes[0].set_xlabel("Position")

# Older group
older_logo = create_logo_df(older_tcrs_padded)
logomaker.Logo(older_logo, ax=axes[1], color_scheme=amino_acid_colors)
axes[1].set_title(f"Older-biased TCRs (n={len(older_tcrs_padded)})")
axes[1].set_ylabel("Frequency")
axes[1].set_xlabel("Position")

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(younger_tcrs.map(len), bins=range(5, max_len + 2), alpha=0.6, label="Younger", density=True)
plt.hist(older_tcrs.map(len), bins=range(5, max_len + 2), alpha=0.6, label="Older", density=True)
plt.title("CDR3 Length Distribution")
plt.xlabel("CDR3 Length")
plt.ylabel("Density")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
from collections import Counter

def get_kmer_freqs(tcr_series, k=3):
    kmers = []
    for seq in tcr_series:
        kmers += [seq[i:i+k] for i in range(len(seq) - k + 1)]
    return pd.Series(Counter(kmers)).sort_values(ascending=False)

# Compute k-mer frequencies
younger_kmers = get_kmer_freqs(younger_tcrs, k=3)
older_kmers = get_kmer_freqs(older_tcrs, k=3)

# Merge into a dataframe
kmer_df = pd.DataFrame({
    'younger': younger_kmers,
    'older': older_kmers
}).fillna(0)

# Normalize by total counts
kmer_df = kmer_df / kmer_df.sum()

# Plot top differing kmers
kmer_df['diff'] = kmer_df['younger'] - kmer_df['older']
top_kmers = kmer_df.reindex(kmer_df['diff'].abs().sort_values(ascending=False).index[:20])

# Plot
top_kmers[['younger', 'older']].plot(kind='bar', figsize=(14, 6), rot=45)
plt.title("Top Differentiating 3-mers Between Groups")
plt.ylabel("Normalized Frequency")
plt.tight_layout()
plt.show()


In [ ]:
def aa_frequency(series):
    aa_counts = Counter("".join(series))
    total = sum(aa_counts.values())
    return pd.Series({aa: count / total for aa, count in aa_counts.items()})

# Compute frequencies
younger_aa_freq = aa_frequency(younger_tcrs)
older_aa_freq = aa_frequency(older_tcrs)

# Combine and plot
aa_df = pd.DataFrame({'Younger': younger_aa_freq, 'Older': older_aa_freq}).fillna(0)
aa_df.plot(kind='bar', figsize=(14, 6))
plt.title("Amino Acid Usage Frequency")
plt.ylabel("Frequency")
plt.xlabel("Amino Acid")
plt.tight_layout()
plt.show()


In [ ]:
lengths = [13, 14, 15]

for length in lengths:
    # Filter by exact length
    y_subset = younger_tcrs[younger_tcrs.map(len) == length]
    o_subset = older_tcrs[older_tcrs.map(len) == length]

    # Pad (no change needed since all are same length)
    y_padded = pad_to_length(y_subset, length)
    o_padded = pad_to_length(o_subset, length)

    # Create logo matrices
    y_logo = create_logo_df(y_padded)
    o_logo = create_logo_df(o_padded)

    # === Plot ===
    fig, axes = plt.subplots(2, 1, figsize=(16, 8))

    logomaker.Logo(y_logo, ax=axes[0], color_scheme=amino_acid_colors)
    axes[0].set_title(f"Younger-biased TCRs (Length = {length}, n={len(y_padded)})")
    axes[0].set_ylabel("Frequency")
    axes[0].set_xlabel("Position")

    logomaker.Logo(o_logo, ax=axes[1], color_scheme=amino_acid_colors)
    axes[1].set_title(f"Older-biased TCRs (Length = {length}, n={len(o_padded)})")
    axes[1].set_ylabel("Frequency")
    axes[1].set_xlabel("Position")

    plt.tight_layout()
    plt.show()


In [ ]:
from collections import Counter
from scipy.stats import entropy

# Define relevant scales
hydro_scale = {
    'A': 1.8, 'C': 2.5, 'D': -3.5, 'E': -3.5, 'F': 2.8,
    'G': -0.4, 'H': -3.2, 'I': 4.5, 'K': -3.9, 'L': 3.8,
    'M': 1.9, 'N': -3.5, 'P': -1.6, 'Q': -3.5, 'R': -4.5,
    'S': -0.8, 'T': -0.7, 'V': 4.2, 'W': -0.9, 'Y': -1.3
}

positive = {'K', 'R', 'H'}
negative = {'D', 'E'}
neutral = set(hydro_scale.keys()) - positive - negative

# Helper functions
def aa_frequency(series):
    all_aas = "".join(series)
    total = len(all_aas)
    freqs = Counter(all_aas)
    return pd.Series({aa: freqs.get(aa, 0)/total for aa in sorted(hydro_scale.keys())})

def compute_hydro_score(series):
    return series.map(lambda seq: np.mean([hydro_scale.get(aa, 0) for aa in seq]))

def charge_count(seq):
    pos = sum(1 for aa in seq if aa in positive)
    neg = sum(1 for aa in seq if aa in negative)
    neu = sum(1 for aa in seq if aa in neutral)
    total = max(len(seq), 1)
    return pd.Series({'Positive': pos / total, 'Negative': neg / total, 'Neutral': neu / total})

def compute_entropy(series):
    df = pd.DataFrame([list(s) for s in series])
    return df.apply(lambda col: entropy(col.value_counts(normalize=True)), axis=0)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from scipy.stats import entropy

# Define Kyte-Doolittle hydrophobicity scale
hydro_scale = {
    'A': 1.8, 'C': 2.5, 'D': -3.5, 'E': -3.5, 'F': 2.8,
    'G': -0.4, 'H': -3.2, 'I': 4.5, 'K': -3.9, 'L': 3.8,
    'M': 1.9, 'N': -3.5, 'P': -1.6, 'Q': -3.5, 'R': -4.5,
    'S': -0.8, 'T': -0.7, 'V': 4.2, 'W': -0.9, 'Y': -1.3
}

# Charge groups
positive = {'K', 'R', 'H'}
negative = {'D', 'E'}
neutral = set(hydro_scale.keys()) - positive - negative

# === Helper Functions ===

def aa_frequency(series):
    all_aas = "".join(series)
    total = len(all_aas)
    freqs = Counter(all_aas)
    return pd.Series({aa: freqs.get(aa, 0)/total for aa in sorted(hydro_scale.keys())})

def compute_hydro_score(series):
    return series.map(lambda seq: np.mean([hydro_scale.get(aa, 0) for aa in seq]))

def charge_count(seq):
    pos = sum(1 for aa in seq if aa in positive)
    neg = sum(1 for aa in seq if aa in negative)
    neu = sum(1 for aa in seq if aa in neutral)
    total = max(len(seq), 1)
    return {'Positive': pos / total, 'Negative': neg / total, 'Neutral': neu / total}

def compute_entropy(series):
    df = pd.DataFrame([list(s) for s in series])
    return df.apply(lambda col: entropy(col.value_counts(normalize=True)), axis=0)

# === Main Loop for Lengths 13, 14, 15 ===

lengths = [13, 14, 15]

for length in lengths:
    print(f"\n=== Analyzing CDR3 Length {length} ===")

    # Subset data
    y_subset = younger_tcrs[younger_tcrs.map(len) == length]
    o_subset = older_tcrs[older_tcrs.map(len) == length]

    if y_subset.empty or o_subset.empty:
        print(f"Skipped length {length} due to insufficient data.")
        continue

    # --- 1. Amino Acid Frequency ---
    y_freq = aa_frequency(y_subset)
    o_freq = aa_frequency(o_subset)

    freq_df = pd.DataFrame({'Younger': y_freq, 'Older': o_freq}).fillna(0)
    freq_df.plot(kind='bar', figsize=(14, 5))
    plt.title(f"Amino Acid Frequency (CDR3 Length {length})")
    plt.ylabel("Normalized Frequency")
    plt.xlabel("Amino Acid")
    plt.tight_layout()
    plt.show()

    # --- 2. Hydrophobicity Distribution ---
    y_hydro = compute_hydro_score(y_subset)
    o_hydro = compute_hydro_score(o_subset)

    plt.figure(figsize=(10, 5))
    plt.hist(y_hydro, bins=30, alpha=0.6, label='Younger', density=True)
    plt.hist(o_hydro, bins=30, alpha=0.6, label='Older', density=True)
    plt.title(f"Hydrophobicity Score Distribution (CDR3 Length {length})")
    plt.xlabel("Average Hydrophobicity")
    plt.ylabel("Density")
    plt.legend()
    plt.tight_layout()
    plt.show()

    # --- 3. Charge Composition ---
    y_charge = pd.DataFrame([charge_count(seq) for seq in y_subset])
    o_charge = pd.DataFrame([charge_count(seq) for seq in o_subset])

    charge_df = pd.DataFrame({
        'Younger': y_charge.mean(),
        'Older': o_charge.mean()
    })

    charge_df.plot(kind='bar', figsize=(8, 4))
    plt.title(f"Normalized Charge Composition (CDR3 Length {length})")
    plt.ylabel("Fraction of Sequence")
    plt.tight_layout()
    plt.show()

    # --- 4. Entropy per Position ---
    y_entropy = compute_entropy(y_subset)
    o_entropy = compute_entropy(o_subset)

    plt.figure(figsize=(14, 5))
    plt.plot(y_entropy, label='Younger')
    plt.plot(o_entropy, label='Older')
    plt.title(f"Shannon Entropy per Position (CDR3 Length {length})")
    plt.xlabel("Position")
    plt.ylabel("Entropy")
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

# === Define parameter grid ===
param_grid = {
    'max_depth': [3, 5, 10, 15],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 5],
    'criterion': ['gini', 'entropy']
}

# === GridSearch with 5-fold CV ===
dt = DecisionTreeClassifier(random_state=42)
grid = GridSearchCV(dt, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid.fit(X_train, y_train)

# === Evaluate best model ===
best_dt = grid.best_estimator_
y_pred = best_dt.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f"\n✅ Best Decision Tree Parameters: {grid.best_params_}")
print(f"✅ Accuracy on Test Set: {acc:.3f}")
print(classification_report(y_test, y_pred))


In [ ]:
# Limit tree depth just for visualization clarity (optional)
viz_depth = min(grid.best_params_['max_depth'], 5)

plt.figure(figsize=(20, 10))
plot_tree(
    best_dt,
    feature_names=feature_names,
    class_names=["Younger", "Older"],
    filled=True,
    rounded=True,
    max_depth=viz_depth,
    fontsize=10
)
plt.title("Best Decision Tree (Tuned with GridSearchCV)")
plt.show()


In [ ]:
# Kyte-Doolittle scale
hydro_scale = {
    'A': 1.8, 'C': 2.5, 'D': -3.5, 'E': -3.5, 'F': 2.8,
    'G': -0.4, 'H': -3.2, 'I': 4.5, 'K': -3.9, 'L': 3.8,
    'M': 1.9, 'N': -3.5, 'P': -1.6, 'Q': -3.5, 'R': -4.5,
    'S': -0.8, 'T': -0.7, 'V': 4.2, 'W': -0.9, 'Y': -1.3
}

def compute_hydro_score(series):
    return series.map(lambda seq: np.mean([hydro_scale.get(aa, 0) for aa in seq]))

younger_hydro = compute_hydro_score(younger_tcrs)
older_hydro = compute_hydro_score(older_tcrs)

# Plot
plt.figure(figsize=(10, 5))
plt.hist(younger_hydro, bins=30, alpha=0.6, label='Younger', density=True)
plt.hist(older_hydro, bins=30, alpha=0.6, label='Older', density=True)
plt.title("Kyte-Doolittle Hydrophobicity Score per TCR")
plt.xlabel("Average Hydrophobicity")
plt.ylabel("Density")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
from scipy.stats import entropy

def compute_entropy(series):
    padded = pad_to_length(series, max_len)
    df = pd.DataFrame([list(s) for s in padded])
    return df.apply(lambda col: entropy(col.value_counts(normalize=True)), axis=0)

younger_entropy = compute_entropy(younger_tcrs)
older_entropy = compute_entropy(older_tcrs)

# Plot
plt.figure(figsize=(14, 5))
plt.plot(younger_entropy, label='Younger')
plt.plot(older_entropy, label='Older')
plt.title("Shannon Entropy per Position")
plt.xlabel("Position")
plt.ylabel("Entropy")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Raw TCRs from both groups
raw_younger = df[df["component_zscore"] < -1.96]["TCR"]
raw_older = df[df["component_zscore"] > 1.96]["TCR"]

# Define function to find non-clean entries
def find_non_clean(series):
    return series[
        series.isna() |                          # NaN values
        (series.astype(str).str.strip() == "")  # Empty after stripping
    ]

# Find problematic TCRs
non_clean_younger = find_non_clean(raw_younger)
non_clean_older = find_non_clean(raw_older)

# Print results
print("❗ Non-clean younger-biased TCRs:")
print(non_clean_younger)
print(f"Total: {len(non_clean_younger)}\n")

print("❗ Non-clean older-biased TCRs:")
print(non_clean_older)
print(f"Total: {len(non_clean_older)}")


In [ ]:
older_group

In [ ]:
import pandas as pd

# Load the files
df_main = pd.read_csv("updated_tcr_age_lists_with_all_significance.csv.gz")
df_vdj = pd.read_csv("vdjdb_trait_onlyB.csv")

# Rename for consistency
df_vdj = df_vdj.rename(columns={"CDR3b": "TCR"})

# Merge on 'TCR'
merged = pd.merge(df_main, df_vdj, on="TCR", how="inner")

# Save merged output
merged.to_csv("merged_tcr_vdjdb.csv", index=False)
print(f"Merged {len(merged)} rows saved to 'merged_tcr_vdjdb.csv'")


In [ ]:
df_vdj

In [ ]:
merged

In [ ]:
import pandas as pd

# Load and clean data
df = pd.read_csv("merged_tcr_vdjdb.csv")

In [ ]:
df = df.drop_duplicates()

In [ ]:
df

In [ ]:
df.to_csv("merged_overlap_tcrs_wasserstein.csv", index=False)

In [ ]:
df

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load and clean data
df = pd.read_csv("merged_overlap_tcrs_wasserstein.csv")
df = df.dropna(subset=["Epitope_species", "Epitope_gene"])

# Normalize species names
species_map = {
    'DENV1': 'DENV',
    'DENV3/4': 'DENV',
    'HIV-1': 'HIV',
    'Homo Sapiens': None,
    'HomoSapiens': None,
    'SARS-CoV': 'SARS',
    'SARS-CoV-2': 'SARS'
}
df["Epitope_species"] = df["Epitope_species"].replace(species_map)
df = df[df["Epitope_species"].notna()]

# Count per (species, gene)
counts = df.groupby(["Epitope_species", "Epitope_gene"]).size().reset_index(name="count")

# Normalize counts within each species
counts["proportion"] = counts.groupby("Epitope_species")["count"].transform(lambda x: x / x.sum())

# Pivot: index=species, columns=gene
pivot = counts.pivot(index="Epitope_species", columns="Epitope_gene", values="proportion").fillna(0)

# Sort species by total TCR count for consistent order
species_order = df["Epitope_species"].value_counts().index
pivot = pivot.loc[species_order.intersection(pivot.index)]

# Plot
pivot.plot(kind="barh", stacked=True, figsize=(12, max(6, 0.4 * len(pivot))))
plt.xlabel("Proportion of Epitope Genes within Species")
plt.ylabel("Epitope_species")
plt.title("Gene Usage per Epitope Species (Normalized)")
plt.legend(title="Epitope_gene", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load and clean
df = pd.read_csv("merged_overlap_tcrs_wasserstein.csv")
df = df.dropna(subset=["Epitope_species", "Epitope_gene"])

# Normalize species names
species_map = {
    'DENV1': 'DENV',
    'DENV3/4': 'DENV',
    'HIV-1': 'HIV',
    'Homo Sapiens': None,
    'HomoSapiens': None,
    'SARS-CoV': 'SARS',
    'SARS-CoV-2': 'SARS'
}
df["Epitope_species"] = df["Epitope_species"].replace(species_map)
df = df[df["Epitope_species"].notna()]

# Count occurrences
counts = df.groupby(["Epitope_species", "Epitope_gene"]).size().reset_index(name="count")

# Filter species with at least 2 genes
gene_counts_per_species = counts.groupby("Epitope_species")["Epitope_gene"].nunique()
valid_species = gene_counts_per_species[gene_counts_per_species > 1].index
counts = counts[counts["Epitope_species"].isin(valid_species)]

# Keep top 20 genes only
top_genes = counts["Epitope_gene"].value_counts().head(20).index
counts = counts[counts["Epitope_gene"].isin(top_genes)]

# Pivot raw + normalized
raw_pivot = counts.pivot(index="Epitope_species", columns="Epitope_gene", values="count").fillna(0)
norm_pivot = raw_pivot.div(raw_pivot.sum(axis=1), axis=0)

# Sort species
raw_pivot = raw_pivot.loc[raw_pivot.sum(axis=1).sort_values(ascending=False).index]
norm_pivot = norm_pivot.loc[raw_pivot.index]

# Plot
fig, axs = plt.subplots(2, 1, figsize=(14, 0.6 * len(norm_pivot) + 6), sharex=True,
                        gridspec_kw={"height_ratios": [2, 1]})

# Normalized
norm_pivot.plot(kind="barh", stacked=True, ax=axs[0], width=0.8, legend=False)
axs[0].set_title("Normalized Gene Usage per Epitope Species (Filtered)")
axs[0].set_ylabel("Epitope_species")
axs[0].grid(True)

# Raw
raw_pivot.plot(kind="barh", stacked=True, ax=axs[1], width=0.8, legend=True)
axs[1].set_title("Raw Gene Counts per Epitope Species (Filtered)")
axs[1].set_ylabel("Epitope_species")
axs[1].set_xlabel("TCR Count")
axs[1].grid(True)
axs[1].legend(title="Epitope_gene", bbox_to_anchor=(1.05, 1), loc="upper left", fontsize='small')

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load and clean
df = pd.read_csv("merged_overlap_tcrs_wasserstein.csv")
df = df.dropna(subset=["Epitope_species", "Epitope_gene"])

# Normalize species names
species_map = {
    'DENV1': 'DENV', 'DENV3/4': 'DENV', 'HIV-1': 'HIV',
    'Homo Sapiens': None, 'HomoSapiens': None,
    'SARS-CoV': 'SARS', 'SARS-CoV-2': 'SARS'
}
df["Epitope_species"] = df["Epitope_species"].replace(species_map)
df = df[df["Epitope_species"].notna()]

# Group counts
counts = df.groupby(["Epitope_species", "Epitope_gene"]).size().reset_index(name="count")

# Limit to top 6 species by total count
top_species = counts.groupby("Epitope_species")["count"].sum().sort_values(ascending=False).head(6).index
counts = counts[counts["Epitope_species"].isin(top_species)]

# Plot setup
n_species = len(top_species)
fig, axes = plt.subplots(n_species, 2, subplot_kw=dict(polar=True), figsize=(14, 2.8 * n_species))
if n_species == 1:  # handle special case of 1 species
    axes = np.expand_dims(axes, axis=0)

# Radar plot function
def radar(ax, labels, values, title):
    angles = np.linspace(0, 2 * np.pi, len(labels), endpoint=False).tolist()
    values += values[:1]
    angles += angles[:1]
    ax.plot(angles, values, marker='o')
    ax.fill(angles, values, alpha=0.25)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(labels, fontsize=8)
    ax.set_title(title, pad=12)
    ax.set_yticklabels([])

# Build plots
for i, species in enumerate(top_species):
    subset = counts[counts["Epitope_species"] == species].sort_values("count", ascending=False)
    genes = subset["Epitope_gene"].tolist()
    raw = subset["count"].tolist()
    norm = (subset["count"] / sum(raw)).tolist()

    # Limit to top 8 genes
    if len(genes) > 8:
        genes = genes[:8]
        raw = raw[:8]
        norm = norm[:8]

    radar(axes[i, 0], genes, norm + [norm[0]], f"{species} (Normalized)")
    radar(axes[i, 1], genes, raw + [raw[0]], f"{species} (Raw Counts)")

# Layout
plt.suptitle("Radar Plots of Epitope_gene Composition per Epitope_species", fontsize=16)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# --- Gene normalization dictionary ---
gene_norm_dict = {
    # CMV
    "IE1": "IE1",
    "IE-1": "IE1",
    "pp65": "pp65",
    "pp50": "pp50",
    "IE2": "IE2",
    "UL29/28": "UL29/28",
    "UL40": "UL40",
    # InfluenzaA
    "M": "M",
    "Flu-MP": "Flu-MP",
    "NP": "NP",
    "HA": "HA",
    "M1": "M1",
    "PA": "PA",
    "PB1": "PB1",
    "PB": "PB",
    "NS2": "NS2",
    "NEF": "NEF",
    "M1-F5L": "M1-F5L",
    "M1-G4E": "M1-G4E",
    # EBV
    "EBNA-3B": "EBNA3B",
    "EBNA3B": "EBNA3B",
    "EBNA3A": "EBNA3A",
    "EBNA-3A": "EBNA3A",
    "EBNA4": "EBNA4",
    "BMLF1": "BMLF1",
    "BZLF1": "BZLF1",
    "LMP2A": "LMP2A",
    "BRLF1": "BRLF1",
    "EMNA-3A": "EBNA3A",
    "EBNA1": "EBNA1",
    "LMP1": "LMP1",
    "EBNA-6": "EBNA6",
    "EBNA6": "EBNA6",
    # SARS
    "Spike": "Spike",
    "ORF1ab": "ORF1ab",
    "Nucleocapsid": "Nucleocapsid",
    "Nucleocapsid  ": "Nucleocapsid",
    "ORF3": "ORF3",
    "NSP3": "NSP3",
    "Matrix": "Matrix",
    "ORF7a": "ORF7a",
    "RNP": "RNP",
    "Envelope": "Envelope",
    "ORF9b": "ORF9b",
    "ORF7b": "ORF7b",
    "ORF8": "ORF8",
    "ORF14": "ORF14",
    "ORF10": "ORF10",
    "ORF6": "ORF6",
    # HIV
    "Gag": "Gag",
    "Gag-protein": "Gag",
    "gp160": "GP160",
    "GP160": "GP160",
    "Nef": "Nef",
    "Pol": "Pol",
    "POL": "Pol",
    "Vif": "Vif",
    "Vpr": "Vpr",
    "VPR": "Vpr",
    "RT": "RT",
    "GAG": "Gag",
    # HCV
    "NS3": "NS3",
    "NS5B": "NS5B",
    "CORE": "CORE",
    "NS4B": "NS4B",
    "POL": "POL",
    # YFV
    "NS4B": "NS4B",
    # DENV
    "NS3": "NS3",
    # Mtb
    "Rv1518": "Rv1518",
    "Rv1734c": "Rv1734c",
    # Influenza B
    "NS1": "NS1",
    "NP": "NP",
}

# Load and clean
df = pd.read_csv("merged_overlap_tcrs_wasserstein.csv")
df = df.dropna(subset=["Epitope_species", "Epitope_gene"])

# Normalize species names
species_map = {
    'DENV1': 'DENV', 'DENV3/4': 'DENV', 'HIV-1': 'HIV',
    'Homo Sapiens': None, 'HomoSapiens': None,
    'SARS-CoV': 'SARS', 'SARS-CoV-2': 'SARS'
}
df["Epitope_species"] = df["Epitope_species"].replace(species_map)
df = df[df["Epitope_species"].notna()]

# --- Normalize gene names ---
df["Epitope_gene_norm"] = df["Epitope_gene"].map(gene_norm_dict).fillna(df["Epitope_gene"])

# Group counts using normalized gene names
counts = df.groupby(["Epitope_species", "Epitope_gene_norm"]).size().reset_index(name="count")

# Select top species by total count
top_species = counts.groupby("Epitope_species")["count"].sum().sort_values(ascending=False).index
counts = counts[counts["Epitope_species"].isin(top_species)]

# Plot setup
n_species = len(top_species)
fig, axes = plt.subplots(n_species, 2, figsize=(14, 3 * n_species), sharex=False)
if n_species == 1:
    axes = axes.reshape(1, 2)

for i, species in enumerate(top_species):
    subset = counts[counts["Epitope_species"] == species].sort_values("count", ascending=True)
    genes = subset["Epitope_gene_norm"].tolist()
    raw_counts = subset["count"].tolist()
    norm_counts = (subset["count"] / subset["count"].sum()).tolist()

    # Left: normalized proportions
    axes[i, 0].barh(genes, norm_counts, color='skyblue')
    axes[i, 0].set_title(f"{species} (Normalized Proportions)")
    axes[i, 0].set_xlabel("Proportion")
    axes[i, 0].set_xlim(0, max(norm_counts) * 1.1 if norm_counts else 1)

    # Right: raw counts
    axes[i, 1].barh(genes, raw_counts, color='salmon')
    axes[i, 1].set_title(f"{species} (Raw Counts)")
    axes[i, 1].set_xlabel("TCR Count")
    axes[i, 1].set_xlim(0,5000)

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# --- Gene normalization dictionary ---
gene_norm_dict = {
    # CMV
    "IE1": "IE1",
    "IE-1": "IE1",
    "pp65": "pp65",
    "pp50": "pp50",
    "IE2": "IE2",
    "UL29/28": "UL29/28",
    "UL40": "UL40",
    # InfluenzaA
    "M": "M",
    "Flu-MP": "Flu-MP",
    "NP": "NP",
    "HA": "HA",
    "M1": "M1",
    "PA": "PA",
    "PB1": "PB1",
    "PB": "PB",
    "NS2": "NS2",
    "NEF": "NEF",
    "M1-F5L": "M1-F5L",
    "M1-G4E": "M1-G4E",
    # EBV
    "EBNA-3B": "EBNA3B",
    "EBNA3B": "EBNA3B",
    "EBNA3A": "EBNA3A",
    "EBNA-3A": "EBNA3A",
    "EBNA4": "EBNA4",
    "BMLF1": "BMLF1",
    "BZLF1": "BZLF1",
    "LMP2A": "LMP2A",
    "BRLF1": "BRLF1",
    "EMNA-3A": "EBNA3A",
    "EBNA1": "EBNA1",
    "LMP1": "LMP1",
    "EBNA-6": "EBNA6",
    "EBNA6": "EBNA6",
    # SARS
    "Spike": "Spike",
    "ORF1ab": "ORF1ab",
    "Nucleocapsid": "Nucleocapsid",
    "Nucleocapsid  ": "Nucleocapsid",
    "ORF3": "ORF3",
    "NSP3": "NSP3",
    "Matrix": "Matrix",
    "ORF7a": "ORF7a",
    "RNP": "RNP",
    "Envelope": "Envelope",
    "ORF9b": "ORF9b",
    "ORF7b": "ORF7b",
    "ORF8": "ORF8",
    "ORF14": "ORF14",
    "ORF10": "ORF10",
    "ORF6": "ORF6",
    # HIV
    "Gag": "Gag",
    "Gag-protein": "Gag",
    "gp160": "GP160",
    "GP160": "GP160",
    "Nef": "Nef",
    "Pol": "Pol",
    "POL": "Pol",
    "Vif": "Vif",
    "Vpr": "Vpr",
    "VPR": "Vpr",
    "RT": "RT",
    "GAG": "Gag",
    # HCV
    "NS3": "NS3",
    "NS5B": "NS5B",
    "CORE": "CORE",
    "NS4B": "NS4B",
    "POL": "POL",
    # YFV
    "NS4B": "NS4B",
    # DENV
    "NS3": "NS3",
    # Mtb
    "Rv1518": "Rv1518",
    "Rv1734c": "Rv1734c",
    # Influenza B
    "NS1": "NS1",
    "NP": "NP",
}
# Load and clean
df = pd.read_csv("merged_overlap_tcrs_wasserstein.csv")
df = df.dropna(subset=["Epitope_species", "Epitope_gene"])

# Normalize species names
species_map = {
    'DENV1': 'DENV', 'DENV3/4': 'DENV', 'HIV-1': 'HIV',
    'Homo Sapiens': None, 'HomoSapiens': None,
    'SARS-CoV': 'SARS', 'SARS-CoV-2': 'SARS'
}
df["Epitope_species"] = df["Epitope_species"].replace(species_map)
df = df[df["Epitope_species"].notna()]

# Normalize gene names
df["Epitope_gene_norm"] = df["Epitope_gene"].map(gene_norm_dict).fillna(df["Epitope_gene"])

# Group counts using normalized gene names
counts = df.groupby(["Epitope_species", "Epitope_gene_norm"]).size().reset_index(name="count")

# Sort species by total count
species_by_count = counts.groupby("Epitope_species")["count"].sum().sort_values(ascending=False)
top5_species = species_by_count.index[:5]
other_species = species_by_count.index[5:]

# === Big horizontal plot for top 5 species ===
fig, axes = plt.subplots(5, 2, figsize=(14, 16), sharex=False)
for i, species in enumerate(top5_species):
    subset = counts[counts["Epitope_species"] == species].sort_values("count", ascending=True)
    genes = subset["Epitope_gene_norm"].tolist()
    raw_counts = subset["count"].tolist()
    norm_counts = (subset["count"] / subset["count"].sum()).tolist()

    # Normalized
    axes[i, 0].barh(genes, norm_counts, color='skyblue')
    axes[i, 0].set_title(f"{species} (Normalized Proportions)")
    axes[i, 0].set_xlabel("Proportion")
    axes[i, 0].set_xlim(0, max(norm_counts) * 1.1 if norm_counts else 1)

    # Raw
    axes[i, 1].barh(genes, raw_counts, color='salmon')
    axes[i, 1].set_title(f"{species} (Raw Counts)")
    axes[i, 1].set_xlabel("TCR Count")
    axes[i, 1].set_xlim(0, max(raw_counts) * 1.1 if raw_counts else 1)

plt.tight_layout()
plt.show()

# === Packed plot for other species ===
n_other = len(other_species)
ncol = 4
nrow = int(np.ceil(n_other / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(16, 3*nrow), sharex=False, sharey=False)
axes = axes.flatten()

for i, species in enumerate(other_species):
    ax = axes[i]
    subset = counts[counts["Epitope_species"] == species].sort_values("count", ascending=True)
    genes = subset["Epitope_gene_norm"].tolist()
    norm_counts = (subset["count"] / subset["count"].sum()).tolist()
    ax.barh(genes, norm_counts, color='skyblue')
    ax.set_title(species, fontsize=10)
    ax.set_xlabel("Proportion")
    ax.set_xlim(0, max(norm_counts) * 1.1 if norm_counts else 1)
    ax.tick_params(axis='y', labelsize=8)
for ax in axes[n_other:]:
    ax.axis('off')

fig.suptitle("Other Species: Normalized Gene Proportions", fontsize=16)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# (Assume gene_norm_dict, df loading, normalization as before...)

# Group counts using normalized gene names
counts = df.groupby(["Epitope_species", "Epitope_gene_norm"]).size().reset_index(name="count")

# Sort species by total count
species_by_count = counts.groupby("Epitope_species")["count"].sum().sort_values(ascending=False)
top5_species = species_by_count.index[:5]
other_species = species_by_count.index[5:]

# === Packed plot for other species: NORMALIZED only ===
n_other = len(other_species)
ncol = 4
nrow = int(np.ceil(n_other / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(16, 3 * nrow), sharex=False, sharey=False)
axes = axes.flatten()

for i, species in enumerate(other_species):
    ax = axes[i]
    subset = counts[counts["Epitope_species"] == species].sort_values("count", ascending=True)
    genes = subset["Epitope_gene_norm"].tolist()
    norm_counts = (subset["count"] / subset["count"].sum()).tolist()
    ax.barh(genes, norm_counts, color='skyblue')
    ax.set_title(species, fontsize=10)
    ax.set_xlabel("Proportion")
    ax.set_xlim(0, max(norm_counts) * 1.1 if norm_counts else 1)
    ax.tick_params(axis='y', labelsize=8)
for ax in axes[n_other:]:
    ax.axis('off')

fig.suptitle("Other Species: Normalized Gene Proportions", fontsize=16)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

# === Packed plot for other species: RAW only ===
fig, axes = plt.subplots(nrow, ncol, figsize=(16, 3 * nrow), sharex=False, sharey=False)
axes = axes.flatten()

for i, species in enumerate(other_species):
    ax = axes[i]
    subset = counts[counts["Epitope_species"] == species].sort_values("count", ascending=True)
    genes = subset["Epitope_gene_norm"].tolist()
    raw_counts = subset["count"].tolist()
    ax.barh(genes, raw_counts, color='salmon')
    ax.set_title(species, fontsize=10)
    ax.set_xlabel("TCR Count")
    ax.set_xlim(0, 1000)
    ax.tick_params(axis='y', labelsize=8)
for ax in axes[n_other:]:
    ax.axis('off')

fig.suptitle("Other Species: Raw Gene Counts", fontsize=16)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load and clean
df = pd.read_csv("merged_overlap_tcrs_wasserstein.csv")
df = df.dropna(subset=["Epitope_species", "Epitope_gene"])

# Normalize species names
species_map = {
    'DENV1': 'DENV', 'DENV3/4': 'DENV', 'HIV-1': 'HIV',
    'Homo Sapiens': None, 'HomoSapiens': None,
    'SARS-CoV': 'SARS', 'SARS-CoV-2': 'SARS'
}
df["Epitope_species"] = df["Epitope_species"].replace(species_map)
df = df[df["Epitope_species"].notna()]

# Group counts
counts = df.groupby(["Epitope_species", "Epitope_gene"]).size().reset_index(name="count")

# Select top species by total count
top_species = counts.groupby("Epitope_species")["count"].sum().sort_values(ascending=False).index
counts = counts[counts["Epitope_species"].isin(top_species)]

# Plot setup
n_species = len(top_species)
fig, axes = plt.subplots(n_species, 2, figsize=(14, 3 * n_species), sharex=False)
if n_species == 1:
    axes = axes.reshape(1, 2)

for i, species in enumerate(top_species):
    subset = counts[counts["Epitope_species"] == species].sort_values("count", ascending=True)
    genes = subset["Epitope_gene"].tolist()
    raw_counts = subset["count"].tolist()
    norm_counts = (subset["count"] / subset["count"].sum()).tolist()

    # Left: normalized proportions
    axes[i, 0].barh(genes, norm_counts, color='skyblue')
    axes[i, 0].set_title(f"{species} (Normalized Proportions)")
    axes[i, 0].set_xlabel("Proportion")
    axes[i, 0].set_xlim(0, max(norm_counts) * 1.1)

    # Right: raw counts
    axes[i, 1].barh(genes, raw_counts, color='salmon')
    axes[i, 1].set_title(f"{species} (Raw Counts)")
    axes[i, 1].set_xlabel("TCR Count")
    axes[i, 1].set_xlim(0, 3000)

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

def plot_species_gene_distribution(df, title, xlim=3000, top_species=None):
    # Clean
    df = df.dropna(subset=["Epitope_species", "Epitope_gene"])

    # Normalize species names (same mapping as before)
    species_map = {
        'DENV1': 'DENV', 'DENV3/4': 'DENV', 'HIV-1': 'HIV',
        'Homo Sapiens': None, 'HomoSapiens': None,
        'SARS-CoV': 'SARS', 'SARS-CoV-2': 'SARS'
    }
    df["Epitope_species"] = df["Epitope_species"].replace(species_map)
    df = df[df["Epitope_species"].notna()]

    # Count
    counts = df.groupby(["Epitope_species", "Epitope_gene"]).size().reset_index(name="count")

    # Optionally filter top species for clarity
    if top_species is not None:
        counts = counts[counts["Epitope_species"].isin(top_species)]

    species_order = counts.groupby("Epitope_species")["count"].sum().sort_values(ascending=False).index
    top_species = species_order if top_species is None else top_species

    n_species = len(top_species)
    fig, axes = plt.subplots(n_species, 2, figsize=(14, 3 * n_species), sharex=False)
    if n_species == 1:
        axes = axes.reshape(1, 2)

    for i, species in enumerate(top_species):
        subset = counts[counts["Epitope_species"] == species].sort_values("count", ascending=True)
        genes = subset["Epitope_gene"].tolist()
        raw_counts = subset["count"].tolist()
        norm_counts = (subset["count"] / subset["count"].sum()).tolist()

        # Normalized plot
        axes[i, 0].barh(genes, norm_counts, color='skyblue')
        axes[i, 0].set_title(f"{species} (Normalized Proportions)")
        axes[i, 0].set_xlabel("Proportion")
        axes[i, 0].set_xlim(0, 1)  # Proportions max at 1
        axes[i, 0].grid(True)

        # Raw counts plot
        axes[i, 1].barh(genes, raw_counts, color='salmon')
        axes[i, 1].set_title(f"{species} (Raw Counts)")
        axes[i, 1].set_xlabel("TCR Count")
        axes[i, 1].set_xlim(0, xlim)
        axes[i, 1].grid(True)

    plt.suptitle(title, fontsize=16)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load full dataset
df = pd.read_csv("merged_overlap_tcrs_wasserstein.csv")
df = df.dropna(subset=["Epitope_species", "Epitope_gene"])

species_map = {
    'DENV1': 'DENV', 'DENV3/4': 'DENV', 'HIV-1': 'HIV',
    'Homo Sapiens': None, 'HomoSapiens': None,
    'SARS-CoV': 'SARS', 'SARS-CoV-2': 'SARS'
}
df["Epitope_species"] = df["Epitope_species"].replace(species_map)
df = df[df["Epitope_species"].notna()]

# Assign groups based on z-score
z_thresh = 1.96
df["Group"] = "All"
df.loc[df["component_zscore"] < -z_thresh, "Group"] = "Younger"
df.loc[df["component_zscore"] > z_thresh, "Group"] = "Older"

# Function to plot per species
def plot_species_gene_comparison(species, df):
    subset = df[df["Epitope_species"] == species]

    # Pivot for counts: index=Epitope_gene, columns=Group
    counts = subset.groupby(["Epitope_gene", "Group"]).size().unstack(fill_value=0)

    # Normalize counts per group
    counts_norm = counts.div(counts.sum(axis=0), axis=1)

    # Plot normalized
    ax = counts_norm.plot(kind="bar", figsize=(12,6))
    ax.set_title(f"Normalized Epitope_gene counts for {species} (All vs Younger vs Older)")
    ax.set_ylabel("Proportion")
    ax.set_xlabel("Epitope_gene")
    plt.xticks(rotation=45, ha='right')
    plt.legend(title="Group")
    plt.tight_layout()
    plt.show()

    # Plot raw counts
    ax2 = counts.plot(kind="bar", figsize=(12,6))
    ax2.set_title(f"Raw Epitope_gene counts for {species} (All vs Younger vs Older)")
    ax2.set_ylabel("Count")
    ax2.set_xlabel("Epitope_gene")
    plt.xticks(rotation=45, ha='right')
    plt.legend(title="Group")
    plt.tight_layout()
    plt.show()

# Example usage: plot for top 3 species by total count
top_species = df.groupby("Epitope_species").size().sort_values(ascending=False).head(3).index

for sp in top_species:
    plot_species_gene_comparison(sp, df)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load full dataset
df = pd.read_csv("merged_overlap_tcrs_wasserstein.csv")
df = df.dropna(subset=["Epitope_species", "Epitope_gene"])

species_map = {
    'DENV1': 'DENV', 'DENV3/4': 'DENV', 'HIV-1': 'HIV',
    'Homo Sapiens': None, 'HomoSapiens': None,
    'SARS-CoV': 'SARS', 'SARS-CoV-2': 'SARS'
}
df["Epitope_species"] = df["Epitope_species"].replace(species_map)
df = df[df["Epitope_species"].notna()]

# Your gene normalization dictionary
gene_norm_dict = {
    # CMV
    "IE1": "IE1",
    "IE-1": "IE1",
    "pp65": "pp65",
    "pp50": "pp50",
    "IE2": "IE2",
    "UL29/28": "UL29/28",
    "UL40": "UL40",
    # InfluenzaA
    "M": "M",
    "Flu-MP": "Flu-MP",
    "NP": "NP",
    "HA": "HA",
    "M1": "M1",
    "PA": "PA",
    "PB1": "PB1",
    "PB": "PB",
    "NS2": "NS2",
    "NEF": "NEF",
    "M1-F5L": "M1-F5L",
    "M1-G4E": "M1-G4E",
    # EBV
    "EBNA-3B": "EBNA3B",
    "EBNA3B": "EBNA3B",
    "EBNA3A": "EBNA3A",
    "EBNA-3A": "EBNA3A",
    "EBNA4": "EBNA4",
    "BMLF1": "BMLF1",
    "BZLF1": "BZLF1",
    "LMP2A": "LMP2A",
    "BRLF1": "BRLF1",
    "EMNA-3A": "EBNA3A",
    "EBNA1": "EBNA1",
    "LMP1": "LMP1",
    "EBNA-6": "EBNA6",
    "EBNA6": "EBNA6",
    # SARS
    "Spike": "Spike",
    "ORF1ab": "ORF1ab",
    "Nucleocapsid": "Nucleocapsid",
    "Nucleocapsid  ": "Nucleocapsid",
    "ORF3": "ORF3",
    "NSP3": "NSP3",
    "Matrix": "Matrix",
    "ORF7a": "ORF7a",
    "RNP": "RNP",
    "Envelope": "Envelope",
    "ORF9b": "ORF9b",
    "ORF7b": "ORF7b",
    "ORF8": "ORF8",
    "ORF14": "ORF14",
    "ORF10": "ORF10",
    "ORF6": "ORF6",
    # HIV
    "Gag": "Gag",
    "Gag-protein": "Gag",
    "gp160": "GP160",
    "GP160": "GP160",
    "Nef": "Nef",
    "Pol": "Pol",
    "POL": "Pol",
    "Vif": "Vif",
    "Vpr": "Vpr",
    "VPR": "Vpr",
    "RT": "RT",
    "GAG": "Gag",
    # HCV
    "NS3": "NS3",
    "NS5B": "NS5B",
    "CORE": "CORE",
    "NS4B": "NS4B",
    "POL": "POL",
    # YFV
    "NS4B": "NS4B",
    # DENV
    "NS3": "NS3",
    # Mtb
    "Rv1518": "Rv1518",
    "Rv1734c": "Rv1734c",
    # Influenza B
    "NS1": "NS1",
    "NP": "NP",
}

# Apply gene normalization
df['Epitope_gene_norm'] = df['Epitope_gene'].map(gene_norm_dict).fillna(df['Epitope_gene'])

# Assign groups based on z-score
z_thresh = 1.96
df["Group"] = "All"
df.loc[df["component_zscore"] < -z_thresh, "Group"] = "Younger"
df.loc[df["component_zscore"] > z_thresh, "Group"] = "Older"

# Function to plot per species with normalized gene names
def plot_species_gene_comparison(species, df):
    subset = df[df["Epitope_species"] == species]

    # Pivot for counts: index=Epitope_gene_norm, columns=Group
    counts = subset.groupby(["Epitope_gene_norm", "Group"]).size().unstack(fill_value=0)

    # Normalize counts per group
    counts_norm = counts.div(counts.sum(axis=0), axis=1)

    # Plot normalized
    ax = counts_norm.plot(kind="bar", figsize=(12,6))
    ax.set_title(f"Normalized Epitope_gene counts for {species} (All vs Younger vs Older)")
    ax.set_ylabel("Proportion")
    ax.set_xlabel("Epitope_gene")
    plt.xticks(rotation=45, ha='right')
    plt.legend(title="Group")
    plt.tight_layout()
    plt.show()

    # Plot raw counts
    ax2 = counts.plot(kind="bar", figsize=(12,6))
    ax2.set_title(f"Raw Epitope_gene counts for {species} (All vs Younger vs Older)")
    ax2.set_ylabel("Count")
    ax2.set_xlabel("Epitope_gene")
    plt.xticks(rotation=45, ha='right')
    plt.legend(title="Group")
    plt.tight_layout()
    plt.show()

# Example usage: plot for top 3 species by total count
top_species = df.groupby("Epitope_species").size().sort_values(ascending=False).head(3).index

for sp in top_species:
    plot_species_gene_comparison(sp, df)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load and clean data
df = pd.read_csv("merged_overlap_tcrs_wasserstein.csv")
df = df.dropna(subset=["Epitope_species", "Epitope_gene"])

species_map = {
    'DENV1': 'DENV', 'DENV3/4': 'DENV', 'HIV-1': 'HIV',
    'Homo Sapiens': None, 'HomoSapiens': None,
    'SARS-CoV': 'SARS', 'SARS-CoV-2': 'SARS'
}
df["Epitope_species"] = df["Epitope_species"].replace(species_map)
df = df[df["Epitope_species"].notna()]


df['Epitope_gene_norm'] = df['Epitope_gene'].map(gene_norm_dict).fillna(df['Epitope_gene'])

# Thresholds to use
thresholds = [1.64, 1.96, 2.5, 3.0]

def plot_combined_species(df, species, thresholds):
    fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=True)

    normalized_dfs = []
    raw_dfs = []

    for thresh in thresholds:
        temp_df = df[df["Epitope_species"] == species].copy()
        temp_df["Group"] = "All"
        temp_df.loc[temp_df["component_zscore"] < -thresh, "Group"] = "Younger"
        temp_df.loc[temp_df["component_zscore"] > thresh, "Group"] = "Older"

        counts = temp_df.groupby(["Epitope_gene_norm", "Group"]).size().unstack(fill_value=0)
        counts_norm = counts.div(counts.sum(axis=0), axis=1)

        # Append with multi-level column for threshold
        counts_norm.columns = pd.MultiIndex.from_product([[f"Thresh {thresh}"], counts_norm.columns])
        counts.columns = pd.MultiIndex.from_product([[f"Thresh {thresh}"], counts.columns])

        normalized_dfs.append(counts_norm)
        raw_dfs.append(counts)

    # Concatenate all thresholds horizontally
    norm_concat = pd.concat(normalized_dfs, axis=1).fillna(0)
    raw_concat = pd.concat(raw_dfs, axis=1).fillna(0)

    # Plot normalized
    norm_concat.plot(kind='bar', ax=axes[0], width=0.8)
    axes[0].set_title(f"{species} - Normalized Epitope_gene counts across thresholds")
    axes[0].set_ylabel("Proportion")
    axes[0].legend(title="Threshold and Group", bbox_to_anchor=(1.05, 1), loc='upper left')
    axes[0].tick_params(axis='x', rotation=45)
    axes[0].grid(True)

    # Plot raw counts
    raw_concat.plot(kind='bar', ax=axes[1], width=0.8)
    axes[1].set_title(f"{species} - Raw Epitope_gene counts across thresholds")
    axes[1].set_ylabel("Count")
    axes[1].legend(title="Threshold and Group", bbox_to_anchor=(1.05, 1), loc='upper left')
    axes[1].tick_params(axis='x', rotation=45)
    axes[1].grid(True)

    plt.tight_layout()
    plt.show()

# Select top 3 species by total count
top_species = df.groupby("Epitope_species").size().sort_values(ascending=False).head(3).index

for sp in top_species:
    plot_combined_species(df, sp, thresholds)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency
import numpy as np

# Load and preprocess as before
df = pd.read_csv("merged_overlap_tcrs_wasserstein.csv")
df = df.dropna(subset=["Epitope_species", "Epitope_gene"])

species_map = {
    'DENV1': 'DENV', 'DENV3/4': 'DENV', 'HIV-1': 'HIV',
    'Homo Sapiens': None, 'HomoSapiens': None,
    'SARS-CoV': 'SARS', 'SARS-CoV-2': 'SARS'
}
df["Epitope_species"] = df["Epitope_species"].replace(species_map)
df = df[df["Epitope_species"].notna()]

df['Epitope_gene_norm'] = df['Epitope_gene'].map(gene_norm_dict).fillna(df['Epitope_gene'])

threshold = 1.28
alpha = 0.05

# Assign group
df_thr = df[(df["component_zscore"] < -threshold) | (df["component_zscore"] > threshold)].copy()
df_thr["Group"] = df_thr["component_zscore"].apply(lambda z: "Younger" if z < -threshold else "Older")

# Count how many unique entries in each group (for downsampling)
n_young = (df_thr["Group"] == "Younger").sum()
n_old = (df_thr["Group"] == "Older").sum()

print(f"Original group sizes: Younger = {n_young}, Older = {n_old}")

# Downsample the larger group
min_n = min(n_young, n_old)
younger_idx = df_thr[df_thr["Group"] == "Younger"].sample(min_n, random_state=1).index
older_idx = df_thr[df_thr["Group"] == "Older"].sample(min_n, random_state=1).index
df_balanced = df_thr.loc[younger_idx.union(older_idx)]

# Group by gene and group
counts = df_balanced.groupby(["Epitope_gene_norm", "Group"]).size().unstack(fill_value=0)
counts = counts[["Younger", "Older"]]

# Plot actual counts (now unbiased for group size)
ax = counts.plot(kind="bar", figsize=(18, 7), width=0.8)
plt.title(f"Balanced gene counts (Younger vs Older, z-score {threshold})")
plt.ylabel("Count")
plt.xlabel("Gene")
plt.xticks(rotation=45, ha="right")
plt.legend(title="Group")
plt.tight_layout()
plt.show()

# Print significant genes
print("\nSignificant genes after balancing (p < 0.05):")
found_any = False
for gene in counts.index:
    c = counts.loc[gene]
    table = [
        [c["Younger"], c["Older"]],
        [counts["Younger"].sum() - c["Younger"], counts["Older"].sum() - c["Older"]]
    ]
    try:
        chi2, p, _, _ = chi2_contingency(table)
        if p < alpha:
            found_any = True
            print(f"Gene: {gene:20} | Younger: {c['Younger']:3} | Older: {c['Older']:3} | p = {p:.3e}")
    except Exception:
        continue
if not found_any:
    print("No significant genes found.")



In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency
import numpy as np

# Load and preprocess as before
df = pd.read_csv("merged_overlap_tcrs_wasserstein.csv")
df = df.dropna(subset=["Epitope_species", "Epitope_gene"])

species_map = {
    'DENV1': 'DENV', 'DENV3/4': 'DENV', 'HIV-1': 'HIV',
    'Homo Sapiens': None, 'HomoSapiens': None,
    'SARS-CoV': 'SARS', 'SARS-CoV-2': 'SARS'
}
df["Epitope_species"] = df["Epitope_species"].replace(species_map)
df = df[df["Epitope_species"].notna()]

df['Epitope_gene_norm'] = df['Epitope_gene'].map(gene_norm_dict).fillna(df['Epitope_gene'])

threshold = 1.64
alpha = 0.05

# Assign group
df_thr = df[(df["component_zscore"] < -threshold) | (df["component_zscore"] > threshold)].copy()
df_thr["Group"] = df_thr["component_zscore"].apply(lambda z: "Younger" if z < -threshold else "Older")

# Count how many unique entries in each group (for downsampling)
n_young = (df_thr["Group"] == "Younger").sum()
n_old = (df_thr["Group"] == "Older").sum()

print(f"Original group sizes: Younger = {n_young}, Older = {n_old}")

# Downsample the larger group
min_n = min(n_young, n_old)
younger_idx = df_thr[df_thr["Group"] == "Younger"].sample(min_n, random_state=1).index
older_idx = df_thr[df_thr["Group"] == "Older"].sample(min_n, random_state=1).index
df_balanced = df_thr.loc[younger_idx.union(older_idx)]

# Group by gene and group
counts = df_balanced.groupby(["Epitope_gene_norm", "Group"]).size().unstack(fill_value=0)
counts = counts[["Younger", "Older"]]

# Plot actual counts (now unbiased for group size)
ax = counts.plot(kind="bar", figsize=(18, 7), width=0.8)
plt.title(f"Balanced gene counts (Younger vs Older, z-score {threshold})")
plt.ylabel("Count")
plt.xlabel("Gene")
plt.xticks(rotation=45, ha="right")
plt.legend(title="Group")
plt.tight_layout()
plt.show()

# Print significant genes
print("\nSignificant genes after balancing (p < 0.05):")
found_any = False
for gene in counts.index:
    c = counts.loc[gene]
    table = [
        [c["Younger"], c["Older"]],
        [counts["Younger"].sum() - c["Younger"], counts["Older"].sum() - c["Older"]]
    ]
    try:
        chi2, p, _, _ = chi2_contingency(table)
        if p < alpha:
            found_any = True
            print(f"Gene: {gene:20} | Younger: {c['Younger']:3} | Older: {c['Older']:3} | p = {p:.3e}")
    except Exception:
        continue
if not found_any:
    print("No significant genes found.")



In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency
import numpy as np

# Load and preprocess as before
df = pd.read_csv("merged_overlap_tcrs_wasserstein.csv")
df = df.dropna(subset=["Epitope_species", "Epitope_gene"])

species_map = {
    'DENV1': 'DENV', 'DENV3/4': 'DENV', 'HIV-1': 'HIV',
    'Homo Sapiens': None, 'HomoSapiens': None,
    'SARS-CoV': 'SARS', 'SARS-CoV-2': 'SARS'
}
df["Epitope_species"] = df["Epitope_species"].replace(species_map)
df = df[df["Epitope_species"].notna()]

df['Epitope_gene_norm'] = df['Epitope_gene'].map(gene_norm_dict).fillna(df['Epitope_gene'])

threshold = 1.96
alpha = 0.05

# Assign group
df_thr = df[(df["component_zscore"] < -threshold) | (df["component_zscore"] > threshold)].copy()
df_thr["Group"] = df_thr["component_zscore"].apply(lambda z: "Younger" if z < -threshold else "Older")

# Count how many unique entries in each group (for downsampling)
n_young = (df_thr["Group"] == "Younger").sum()
n_old = (df_thr["Group"] == "Older").sum()

print(f"Original group sizes: Younger = {n_young}, Older = {n_old}")

# Downsample the larger group
min_n = min(n_young, n_old)
younger_idx = df_thr[df_thr["Group"] == "Younger"].sample(min_n, random_state=1).index
older_idx = df_thr[df_thr["Group"] == "Older"].sample(min_n, random_state=1).index
df_balanced = df_thr.loc[younger_idx.union(older_idx)]

# Group by gene and group
counts = df_balanced.groupby(["Epitope_gene_norm", "Group"]).size().unstack(fill_value=0)
counts = counts[["Younger", "Older"]]

# Plot actual counts (now unbiased for group size)
ax = counts.plot(kind="bar", figsize=(18, 7), width=0.8)
plt.title(f"Balanced gene counts (Younger vs Older, z-score {threshold})")
plt.ylabel("Count")
plt.xlabel("Gene")
plt.xticks(rotation=45, ha="right")
plt.legend(title="Group")
plt.tight_layout()
plt.show()

# Print significant genes
print("\nSignificant genes after balancing (p < 0.05):")
found_any = False
for gene in counts.index:
    c = counts.loc[gene]
    table = [
        [c["Younger"], c["Older"]],
        [counts["Younger"].sum() - c["Younger"], counts["Older"].sum() - c["Older"]]
    ]
    try:
        chi2, p, _, _ = chi2_contingency(table)
        if p < alpha:
            found_any = True
            print(f"Gene: {gene:20} | Younger: {c['Younger']:3} | Older: {c['Older']:3} | p = {p:.3e}")
    except Exception:
        continue
if not found_any:
    print("No significant genes found.")



In [ ]:
# Only include species with enough data points (e.g., at least 10 TCRs)
min_n = 50
species_counts = df['Epitope_species'].value_counts()
valid_species = species_counts[species_counts >= min_n].index
df_plot = df[df['Epitope_species'].isin(valid_species)]

# Sort species by mean signed_wasserstein (ascending: youngest to oldest)
means = df_plot.groupby('Epitope_species')['signed_wasserstein'].mean().sort_values()
sorted_species = means.index.tolist()
df_plot['Epitope_species'] = pd.Categorical(df_plot['Epitope_species'], categories=sorted_species, ordered=True)

# Plot: horizontal boxplot
plt.figure(figsize=(max(8, len(sorted_species)*0.5), 8))
ax = plt.subplot(111)
df_plot.boxplot(column='signed_wasserstein', by='Epitope_species', vert=False, ax=ax, showfliers=False)
plt.xlabel('Signed Wasserstein (Age Association)')
plt.ylabel('Species')
plt.title('Age Association (Signed Wasserstein) per Species')
plt.suptitle('')
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# --- Load your data as before ---
df = pd.read_csv("merged_overlap_tcrs_wasserstein.csv")
df = df.dropna(subset=["Epitope_species", "Epitope_gene", "signed_wasserstein"])

# --- Your clusterings dictionary here ---
clusterings = {
    "PathogenFamily": {
        "CMV": "Herpesvirus", "EBV": "Herpesvirus", "MCMV": "Herpesvirus",
        "InfluenzaA": "Orthomyxovirus", "Influenza B": "Orthomyxovirus",
        "SARS": "Coronavirus", "HCoV-HKU1": "Coronavirus",
        "HIV": "Retrovirus", "HTLV-1": "Retrovirus", "SIV": "Retrovirus",
        "DENV": "Flavivirus", "DENV2": "Flavivirus", "YFV": "Flavivirus", "HCV": "Flavivirus",
        "Mtb": "Bacteria", "M.tuberculosis": "Bacteria", "E.Coli": "Bacteria",
        "Wheat": "Plant/Food", "TriticumAestivum": "Plant/Food", "Selaginella Moellendorffii": "Plant/Food",
        "MusMusculus": "Mouse/Control", "Synthetic": "Synthetic/Control",
        "MCPyV": "Polyomavirus", "HPV-16": "Papillomavirus", "AdV": "Adenovirus",
        "RSV": "Paramyxovirus", "LCMV": "Other", "RotavirusA": "Other", "CoxsackievirusB": "Other",
        "Streptomyceskanamyceticus": "Bacteria", "Salmonella Moellendorffii": "Bacteria",
        "Trypanosoma cruzi": "Parasite"
    },
    "AcuteChronic": {
        "CMV": "Chronic", "EBV": "Chronic", "HIV": "Chronic", "HCV": "Chronic", "Mtb": "Chronic",
        "M.tuberculosis": "Chronic", "MCMV": "Chronic",
        "InfluenzaA": "Acute", "Influenza B": "Acute", "DENV": "Acute", "DENV2": "Acute", "YFV": "Acute",
        "SARS": "Acute", "HCoV-HKU1": "Acute", "RSV": "Acute", "LCMV": "Acute", "RotavirusA": "Acute",
        "Synthetic": "Control", "MusMusculus": "Control", "Wheat": "Control", "TriticumAestivum": "Control",
        "Selaginella Moellendorffii": "Control", "E.Coli": "Control", "Salmonella Moellendorffii": "Control",
        "Streptomyceskanamyceticus": "Control", "Trypanosoma cruzi": "Control", "AdV": "Acute",
        "HTLV-1": "Chronic", "SIV": "Chronic", "MCPyV": "Chronic", "HPV-16": "Chronic", "CoxsackievirusB": "Acute"
    },
    "VaccineStatus": {
        "InfluenzaA": "Vaccine", "Influenza B": "Vaccine", "Mtb": "Vaccine", "M.tuberculosis": "Vaccine",
        "DENV": "Vaccine", "DENV2": "Vaccine", "YFV": "Vaccine",
        "CMV": "Natural", "EBV": "Natural", "HIV": "Natural", "HCV": "Natural", "MCMV": "Natural", "SARS": "Natural",
        "HCoV-HKU1": "Natural", "RSV": "Natural", "LCMV": "Natural", "RotavirusA": "Natural", "HTLV-1": "Natural",
        "SIV": "Natural", "MCPyV": "Natural", "HPV-16": "Natural", "E.Coli": "Natural", "Salmonella Moellendorffii": "Natural",
        "Streptomyceskanamyceticus": "Natural", "MusMusculus": "Control", "Synthetic": "Control", "Wheat": "Control",
        "TriticumAestivum": "Control", "Selaginella Moellendorffii": "Control", "AdV": "Vaccine", "Trypanosoma cruzi": "Natural",
        "CoxsackievirusB": "Natural"
    },
    "Route": {
        "InfluenzaA": "Respiratory", "Influenza B": "Respiratory", "SARS": "Respiratory", "HCoV-HKU1": "Respiratory", "RSV": "Respiratory",
        "CMV": "Blood/Other", "EBV": "Blood/Other", "DENV": "Blood/Other", "DENV2": "Blood/Other", "YFV": "Blood/Other",
        "HIV": "Blood/Other", "HCV": "Blood/Other", "HTLV-1": "Blood/Other", "SIV": "Blood/Other", "RotavirusA": "GI",
        "Mtb": "Respiratory", "M.tuberculosis": "Respiratory", "MCMV": "Blood/Other", "MCPyV": "Other", "HPV-16": "Other",
        "LCMV": "Other", "Synthetic": "Control", "MusMusculus": "Control", "Wheat": "Control", "TriticumAestivum": "Control",
        "Selaginella Moellendorffii": "Control", "E.Coli": "GI", "Salmonella Moellendorffii": "GI", "Streptomyceskanamyceticus": "Other",
        "AdV": "Respiratory", "Trypanosoma cruzi": "Blood/Other", "CoxsackievirusB": "Respiratory"
    },
    "Zoonotic": {
        "InfluenzaA": "Zoonotic", "SARS": "Zoonotic", "YFV": "Zoonotic", "DENV": "Zoonotic", "DENV2": "Zoonotic",
        "EBV": "Human", "CMV": "Human", "Influenza B": "Human", "Mtb": "Human", "M.tuberculosis": "Human", "HIV": "Human",
        "HCV": "Human", "HTLV-1": "Human", "SIV": "Zoonotic", "RotavirusA": "Human", "MusMusculus": "Control", "Synthetic": "Control",
        "MCMV": "Zoonotic", "HCoV-HKU1": "Human", "RSV": "Human", "LCMV": "Zoonotic", "MCPyV": "Human", "HPV-16": "Human",
        "Wheat": "Control", "TriticumAestivum": "Control", "Selaginella Moellendorffii": "Control", "E.Coli": "Human",
        "Salmonella Moellendorffii": "Control", "Streptomyceskanamyceticus": "Control", "AdV": "Human", "Trypanosoma cruzi": "Zoonotic",
        "CoxsackievirusB": "Human"
    },
    "ExposureTiming": {
        "Mtb": "Childhood", "M.tuberculosis": "Childhood", "EBV": "Childhood", "InfluenzaA": "Childhood", "DENV": "Childhood",
        "DENV2": "Childhood", "YFV": "Childhood", "RotavirusA": "Childhood",
        "CMV": "Adulthood", "HIV": "Adulthood", "HCV": "Adulthood", "HTLV-1": "Adulthood", "Influenza B": "Childhood",
        "SARS": "Adulthood", "HCoV-HKU1": "Childhood", "RSV": "Childhood", "MCMV": "Adulthood", "MCPyV": "Adulthood",
        "HPV-16": "Adulthood", "SIV": "Adulthood", "LCMV": "Adulthood", "Synthetic": "Control", "MusMusculus": "Control",
        "Wheat": "Control", "TriticumAestivum": "Control", "Selaginella Moellendorffii": "Control", "E.Coli": "Control",
        "Salmonella Moellendorffii": "Control", "Streptomyceskanamyceticus": "Control", "AdV": "Childhood",
        "Trypanosoma cruzi": "Adulthood", "CoxsackievirusB": "Childhood"
    }
}

# --- Loop through all clusterings and plot ---
min_n = 10  # Minimum points to show a cluster

for clustering_name, clustering_map in clusterings.items():
    df[clustering_name] = df['Epitope_species'].map(clustering_map).fillna('Other')
    cluster_counts = df[clustering_name].value_counts()
    valid_clusters = cluster_counts[cluster_counts >= min_n].index
    df_plot = df[df[clustering_name].isin(valid_clusters)]
    means = df_plot.groupby(clustering_name)['signed_wasserstein'].mean().sort_values()
    sorted_clusters = means.index.tolist()
    df_plot[clustering_name] = pd.Categorical(df_plot[clustering_name], categories=sorted_clusters, ordered=True)

    plt.figure(figsize=(max(8, len(sorted_clusters)*0.5), 8))
    ax = plt.subplot(111)
    df_plot.boxplot(column='signed_wasserstein', by=clustering_name, vert=True, ax=ax, showfliers=False)
    plt.xlabel('Signed Wasserstein (Age Association)')
    plt.ylabel(clustering_name)
    plt.title(f'Age Association by {clustering_name}')
    plt.suptitle('')
    plt.tight_layout()
    plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# ... (your data loading and clusterings code) ...

min_n = 10  # Minimum points to show a cluster

for clustering_name, clustering_map in clusterings.items():
    df[clustering_name] = df['Epitope_species'].map(clustering_map).fillna('Other')
    cluster_counts = df[clustering_name].value_counts()
    valid_clusters = cluster_counts[cluster_counts >= min_n].index
    df_plot = df[df[clustering_name].isin(valid_clusters)]
    means = df_plot.groupby(clustering_name)['signed_wasserstein'].mean().sort_values()
    sorted_clusters = means.index.tolist()
    df_plot[clustering_name] = pd.Categorical(df_plot[clustering_name], categories=sorted_clusters, ordered=True)

    # Get group sizes for x-tick labels
    value_counts = df_plot[clustering_name].value_counts().reindex(sorted_clusters).fillna(0).astype(int)
    xtick_labels = [f"{cat}\n(n={value_counts[cat]})" for cat in sorted_clusters]

    plt.figure(figsize=(max(8, len(sorted_clusters)), 7))
    ax = plt.subplot(111)
    df_plot.boxplot(column='signed_wasserstein', by=clustering_name, vert=True, ax=ax, showfliers=False)
    ax.set_xticklabels(xtick_labels, rotation=45, ha='right', fontsize=10)
    plt.xlabel(clustering_name)
    plt.ylabel('Signed Wasserstein (Age Association)')
    plt.title(f'Age Association by {clustering_name}')
    plt.suptitle('')
    plt.tight_layout()
    plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import itertools

# Load data
df = pd.read_csv("merged_overlap_tcrs_wasserstein.csv")
df = df.dropna(subset=["Epitope_species", "Epitope_gene", "signed_wasserstein"])

# Helper to merge GI + Other for "Route"
def collapse_gi_other(val):
    return "GI / Other" if val in ["GI", "Other"] else val

# Labels to drop
remove_labels = {
    "PathogenFamily": {"Mouse/Control", "Synthetic/Control", "Other"},
    "AcuteChronic": {"Control", "Other"},
    "VaccineStatus": {"Control", "Other"},
    "Route": {"Control"},
    "Zoonotic": {"Control", "Other"},
    "ExposureTiming": {"Control", "Other"}
}

# Minimum number of sequences to show a group
min_n = 10

# Define your clusterings dictionary here
# clusterings = { "YourLabel": mapping_dict, ... }

for clustering_name, clustering_map in clusterings.items():
    df[clustering_name] = df["Epitope_species"].map(clustering_map).fillna("Other")
    if clustering_name == "Route":
        df[clustering_name] = df[clustering_name].apply(collapse_gi_other)

    # Filter unwanted labels
    drop_set = remove_labels.get(clustering_name, set())
    df_plot = df[~df[clustering_name].isin(drop_set)].copy()
    cluster_counts = df_plot[clustering_name].value_counts()
    valid_clusters = cluster_counts[cluster_counts >= min_n].index
    df_plot = df_plot[df_plot[clustering_name].isin(valid_clusters)].copy()

    # Sort by median for plotting
    medians = df_plot.groupby(clustering_name)["signed_wasserstein"].median().sort_values()
    ordered = medians.index.tolist()
    df_plot[clustering_name] = pd.Categorical(df_plot[clustering_name], categories=ordered, ordered=True)

    # Compute p-values between all pairs
    all_groups = df_plot[clustering_name].cat.categories
    pval_dict = {}
    for g1, g2 in itertools.combinations(all_groups, 2):
        vals1 = df_plot[df_plot[clustering_name] == g1]["signed_wasserstein"]
        vals2 = df_plot[df_plot[clustering_name] == g2]["signed_wasserstein"]
        stat, p = mannwhitneyu(vals1, vals2, alternative='two-sided')
        pval_dict[(g1, g2)] = p

    # Sort comparisons by significance
    sorted_pvals = sorted(pval_dict.items(), key=lambda x: x[1])

    # Create boxplot
    plt.figure(figsize=(max(9, len(ordered) * 1.4), 6))
    ax = sns.boxplot(data=df_plot, x=clustering_name, y="signed_wasserstein", showfliers=False)
    xticks = [f"{cat}\n(n={cluster_counts[cat]})" for cat in ordered]
    ax.set_xticklabels(xticks, rotation=30, ha='right')

    # Build formatted legend text with highlights for significant ones
    lines = []
    for (g1, g2), p in sorted_pvals:
        star = " **" if p < 0.05 else ""
        lines.append(f"{g1} vs {g2}: p={p:.2e}{star}")
    legend_text = "\n".join(lines)

    # Draw legend box with background highlight
    props = dict(boxstyle='round', facecolor='lightyellow', alpha=0.9)
    ax.text(1.02, 1, legend_text, transform=ax.transAxes, fontsize=8,
            verticalalignment='top', bbox=props)

    # Labels and layout
    plt.ylabel("Signed Wasserstein (Age Association)")
    plt.xlabel(clustering_name)
    plt.title(f"Age Association by {clustering_name}")
    plt.tight_layout()
    plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import itertools

# Load data
df = pd.read_csv("merged_overlap_tcrs_wasserstein.csv")
df = df.dropna(subset=["Epitope_species", "Epitope_gene", "signed_wasserstein"])

# Collapse GI + Other for Route
def collapse_gi_other(val):
    return "GI / Other" if val in ["GI", "Other"] else val

# Labels to drop
remove_labels = {
    "PathogenFamily": {"Mouse/Control", "Synthetic/Control", "Other"},
    "AcuteChronic": {"Control", "Other"},
    "VaccineStatus": {"Control", "Other"},
    "Route": {"Control"},
    "Zoonotic": {"Control", "Other"},
    "ExposureTiming": {"Control", "Other"}
}

# Category descriptions
category_descriptions = {
    "PathogenFamily": {
        "Herpesvirus": "Herpesvirus\n(CMV, EBV)",
        "Orthomyxovirus": "Orthomyxovirus\n(Flu A/B)",
        "Coronavirus": "Coronavirus\n(SARS, HKU1)",
        "Retrovirus": "Retrovirus\n(HIV, HTLV-1)",
        "Flavivirus": "Flavivirus\n(DENV, YFV, HCV)",
        "Bacteria": "Bacteria\n(Mtb, E.Coli)",
        "Parasite": "Parasite\n(T. cruzi)",
        "Plant/Food": "Plant/Food\n(Wheat, Selaginella)",
        "Papillomavirus": "Papillomavirus\n(HPV-16)",
        "Polyomavirus": "Polyomavirus\n(MCPyV)",
        "Adenovirus": "Adenovirus\n(AdV)",
        "Paramyxovirus": "Paramyxovirus\n(RSV)"
    },
    "AcuteChronic": {
        "Acute": "Acute\n(Flu, RSV, SARS)",
        "Chronic": "Chronic\n(CMV, HIV, HCV)"
    },
    "VaccineStatus": {
        "Vaccine": "Vaccine\n(Flu, YFV, Mtb)",
        "Natural": "Natural\n(HIV, HCV, CMV)"
    },
    "Route": {
        "Respiratory": "Respiratory\n(Flu, RSV, SARS)",
        "GI / Other": "GI / Other\n(E.Coli, Rotavirus)",
        "Blood/Other": "Bloodborne\n(HIV, HCV, YFV)"
    },
    "Zoonotic": {
        "Zoonotic": "Zoonotic\n(SARS, DENV, YFV)",
        "Human": "Human-restricted\n(CMV, EBV, HIV)"
    },
    "ExposureTiming": {
        "Childhood": "Childhood\n(EBV, Mtb, Rotavirus)",
        "Adulthood": "Adulthood\n(HIV, CMV, HPV)"
    }
}

# Clusterings dictionary (fully paste the actual values)
clusterings = {
    "PathogenFamily": {
        "CMV": "Herpesvirus", "EBV": "Herpesvirus", "MCMV": "Herpesvirus",
        "InfluenzaA": "Orthomyxovirus", "Influenza B": "Orthomyxovirus",
        "SARS": "Coronavirus", "HCoV-HKU1": "Coronavirus",
        "HIV": "Retrovirus", "HTLV-1": "Retrovirus", "SIV": "Retrovirus",
        "DENV": "Flavivirus", "DENV2": "Flavivirus", "YFV": "Flavivirus", "HCV": "Flavivirus",
        "Mtb": "Bacteria", "M.tuberculosis": "Bacteria", "E.Coli": "Bacteria",
        "Wheat": "Plant/Food", "TriticumAestivum": "Plant/Food", "Selaginella Moellendorffii": "Plant/Food",
        "MusMusculus": "Mouse/Control", "Synthetic": "Synthetic/Control",
        "MCPyV": "Polyomavirus", "HPV-16": "Papillomavirus", "AdV": "Adenovirus",
        "RSV": "Paramyxovirus", "LCMV": "Other", "RotavirusA": "Other", "CoxsackievirusB": "Other",
        "Streptomyceskanamyceticus": "Bacteria", "Salmonella Moellendorffii": "Bacteria",
        "Trypanosoma cruzi": "Parasite"
    },
    "AcuteChronic": {
        "CMV": "Chronic", "EBV": "Chronic", "HIV": "Chronic", "HCV": "Chronic", "Mtb": "Chronic",
        "M.tuberculosis": "Chronic", "MCMV": "Chronic",
        "InfluenzaA": "Acute", "Influenza B": "Acute", "DENV": "Acute", "DENV2": "Acute", "YFV": "Acute",
        "SARS": "Acute", "HCoV-HKU1": "Acute", "RSV": "Acute", "LCMV": "Acute", "RotavirusA": "Acute",
        "Synthetic": "Control", "MusMusculus": "Control", "Wheat": "Control", "TriticumAestivum": "Control",
        "Selaginella Moellendorffii": "Control", "E.Coli": "Control", "Salmonella Moellendorffii": "Control",
        "Streptomyceskanamyceticus": "Control", "Trypanosoma cruzi": "Control", "AdV": "Acute",
        "HTLV-1": "Chronic", "SIV": "Chronic", "MCPyV": "Chronic", "HPV-16": "Chronic", "CoxsackievirusB": "Acute"
    },
    "VaccineStatus": {
        "InfluenzaA": "Vaccine", "Influenza B": "Vaccine", "Mtb": "Vaccine", "M.tuberculosis": "Vaccine",
        "DENV": "Vaccine", "DENV2": "Vaccine", "YFV": "Vaccine",
        "CMV": "Natural", "EBV": "Natural", "HIV": "Natural", "HCV": "Natural", "MCMV": "Natural", "SARS": "Natural",
        "HCoV-HKU1": "Natural", "RSV": "Natural", "LCMV": "Natural", "RotavirusA": "Natural", "HTLV-1": "Natural",
        "SIV": "Natural", "MCPyV": "Natural", "HPV-16": "Natural", "E.Coli": "Natural", "Salmonella Moellendorffii": "Natural",
        "Streptomyceskanamyceticus": "Natural", "MusMusculus": "Control", "Synthetic": "Control", "Wheat": "Control",
        "TriticumAestivum": "Control", "Selaginella Moellendorffii": "Control", "AdV": "Vaccine", "Trypanosoma cruzi": "Natural",
        "CoxsackievirusB": "Natural"
    },
    "Route": {
        "InfluenzaA": "Respiratory", "Influenza B": "Respiratory", "SARS": "Respiratory", "HCoV-HKU1": "Respiratory", "RSV": "Respiratory",
        "CMV": "Blood/Other", "EBV": "Blood/Other", "DENV": "Blood/Other", "DENV2": "Blood/Other", "YFV": "Blood/Other",
        "HIV": "Blood/Other", "HCV": "Blood/Other", "HTLV-1": "Blood/Other", "SIV": "Blood/Other", "RotavirusA": "GI",
        "Mtb": "Respiratory", "M.tuberculosis": "Respiratory", "MCMV": "Blood/Other", "MCPyV": "Other", "HPV-16": "Other",
        "LCMV": "Other", "Synthetic": "Control", "MusMusculus": "Control", "Wheat": "Control", "TriticumAestivum": "Control",
        "Selaginella Moellendorffii": "Control", "E.Coli": "GI", "Salmonella Moellendorffii": "GI", "Streptomyceskanamyceticus": "Other",
        "AdV": "Respiratory", "Trypanosoma cruzi": "Blood/Other", "CoxsackievirusB": "Respiratory"
    },
    "Zoonotic": {
        "InfluenzaA": "Zoonotic", "SARS": "Zoonotic", "YFV": "Zoonotic", "DENV": "Zoonotic", "DENV2": "Zoonotic",
        "EBV": "Human", "CMV": "Human", "Influenza B": "Human", "Mtb": "Human", "M.tuberculosis": "Human", "HIV": "Human",
        "HCV": "Human", "HTLV-1": "Human", "SIV": "Zoonotic", "RotavirusA": "Human", "MusMusculus": "Control", "Synthetic": "Control",
        "MCMV": "Zoonotic", "HCoV-HKU1": "Human", "RSV": "Human", "LCMV": "Zoonotic", "MCPyV": "Human", "HPV-16": "Human",
        "Wheat": "Control", "TriticumAestivum": "Control", "Selaginella Moellendorffii": "Control", "E.Coli": "Human",
        "Salmonella Moellendorffii": "Control", "Streptomyceskanamyceticus": "Control", "AdV": "Human", "Trypanosoma cruzi": "Zoonotic",
        "CoxsackievirusB": "Human"
    },
    "ExposureTiming": {
        "Mtb": "Childhood", "M.tuberculosis": "Childhood", "EBV": "Childhood", "InfluenzaA": "Childhood", "DENV": "Childhood",
        "DENV2": "Childhood", "YFV": "Childhood", "RotavirusA": "Childhood",
        "CMV": "Adulthood", "HIV": "Adulthood", "HCV": "Adulthood", "HTLV-1": "Adulthood", "Influenza B": "Childhood",
        "SARS": "Adulthood", "HCoV-HKU1": "Childhood", "RSV": "Childhood", "MCMV": "Adulthood", "MCPyV": "Adulthood",
        "HPV-16": "Adulthood", "SIV": "Adulthood", "LCMV": "Adulthood", "Synthetic": "Control", "MusMusculus": "Control",
        "Wheat": "Control", "TriticumAestivum": "Control", "Selaginella Moellendorffii": "Control", "E.Coli": "Control",
        "Salmonella Moellendorffii": "Control", "Streptomyceskanamyceticus": "Control", "AdV": "Childhood",
        "Trypanosoma cruzi": "Adulthood", "CoxsackievirusB": "Childhood"
    }
}

# Plot loop
plots = {}
for clustering_name in clusterings:
    clustering_map = clusterings[clustering_name]
    df[clustering_name] = df["Epitope_species"].map(clustering_map).fillna("Other")

    if clustering_name == "Route":
        df[clustering_name] = df[clustering_name].apply(collapse_gi_other)

    drop_set = remove_labels.get(clustering_name, set())
    df_plot = df[~df[clustering_name].isin(drop_set)]
    cluster_counts = df_plot[clustering_name].value_counts()
    valid_clusters = cluster_counts[cluster_counts >= 10].index
    df_plot = df_plot[df_plot[clustering_name].isin(valid_clusters)]

    medians = df_plot.groupby(clustering_name)["signed_wasserstein"].median().sort_values()
    ordered = medians.index.tolist()
    df_plot[clustering_name] = pd.Categorical(df_plot[clustering_name], categories=ordered, ordered=True)

    # Compute p-values
    pvals = []
    for g1, g2 in itertools.combinations(ordered, 2):
        vals1 = df_plot[df_plot[clustering_name] == g1]["signed_wasserstein"]
        vals2 = df_plot[df_plot[clustering_name] == g2]["signed_wasserstein"]
        stat, p = mannwhitneyu(vals1, vals2, alternative='two-sided')
        pvals.append(((g1, g2), p))
    pvals = sorted(pvals, key=lambda x: x[1])

    sig_lines = [f"{g1} vs {g2}: p={p:.2e}" for (g1, g2), p in pvals]
    colors = ['#ffdddd' if p < 0.05 else 'none' for (_, p) in pvals]

    # Plot
    fig, ax = plt.subplots(figsize=(max(10, len(ordered) * 1.5), 6))
    sns.boxplot(data=df_plot, x=clustering_name, y="signed_wasserstein", ax=ax, showfliers=False)

    xticks = [f"{category_descriptions.get(clustering_name, {}).get(cat, cat)}\n(n={cluster_counts[cat]})" for cat in ordered]
    ax.set_xticklabels(xticks, rotation=30, ha='right')
    ax.set_ylabel("Signed Wasserstein (Age Association)")
    ax.set_title(f"Age Association by {clustering_name}")

    # Annotate significance
    if True:
        for i, (text, bg_color) in enumerate(zip(sig_lines, colors)):
            fig.text(1.01, 0.95 - i * 0.035, text, fontsize=8, bbox=dict(facecolor=bg_color, edgecolor='gray'))
    else:
        # Legend in upper right corner inside the plot
        handles = [plt.Line2D([0], [0], color='gray', marker='s', markersize=10, markerfacecolor=bg) for bg in colors]
        ax.legend(handles, sig_lines, loc='upper right', fontsize=8, title='p-values', title_fontsize=9, frameon=False)

    plt.tight_layout()
    plots[clustering_name] = fig
    plt.close()


In [ ]:
from IPython.display import display

for name, fig in plots.items():
    print(f"Plot: {name}")
    display(fig)


In [ ]:
# ... [your existing imports and data loading code] ...

# Define significance thresholds
def significance_marker(p):
    if p < 0.001:
        return '***'
    elif p < 0.01:
        return '**'
    elif p < 0.05:
        return '*'
    else:
        return ''

for clustering_name, clustering_map in clusterings.items():
    df[clustering_name] = df["Epitope_species"].map(clustering_map).fillna("Other")
    if clustering_name == "Route":
        df[clustering_name] = df[clustering_name].apply(collapse_gi_other)

    drop_set = remove_labels.get(clustering_name, set())
    df_plot = df[~df[clustering_name].isin(drop_set)].copy()
    cluster_counts = df_plot[clustering_name].value_counts()
    valid_clusters = cluster_counts[cluster_counts >= min_n].index
    df_plot = df_plot[df_plot[clustering_name].isin(valid_clusters)].copy()

    medians = df_plot.groupby(clustering_name)["signed_wasserstein"].median().sort_values()
    ordered = medians.index.tolist()
    df_plot[clustering_name] = pd.Categorical(df_plot[clustering_name], categories=ordered, ordered=True)

    all_groups = df_plot[clustering_name].cat.categories
    pval_dict = {}
    for g1, g2 in itertools.combinations(all_groups, 2):
        vals1 = df_plot[df_plot[clustering_name] == g1]["signed_wasserstein"]
        vals2 = df_plot[df_plot[clustering_name] == g2]["signed_wasserstein"]
        stat, p = mannwhitneyu(vals1, vals2, alternative='two-sided')
        pval_dict[(g1, g2)] = p

    # Sort p-values for legend
    sorted_pvals = sorted(pval_dict.items(), key=lambda x: x[1])
    legend_text = "\n".join([
        f"{g1} vs {g2}: p={pval:.2e} {significance_marker(pval)}"
        for (g1, g2), pval in sorted_pvals
    ])

    # Create boxplot
    plt.figure(figsize=(max(9, len(ordered) * 1.4), 6))
    ax = sns.boxplot(data=df_plot, x=clustering_name, y="signed_wasserstein", showfliers=False)
    xticks = [f"{cat}\n(n={cluster_counts[cat]})" for cat in ordered]
    ax.set_xticklabels(xticks, rotation=30, ha='right')

    # Add legend box
    props = dict(boxstyle='round', facecolor='white', alpha=0.8)
    ax.text(1.02, 1, legend_text, transform=ax.transAxes, fontsize=8,
            verticalalignment='top', bbox=props)

    plt.ylabel("Signed Wasserstein (Age Association)")
    plt.xlabel(clustering_name)
    plt.title(f"Age Association by {clustering_name}")
    plt.tight_layout()
    plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import itertools

# Load and clean
df = pd.read_csv("merged_overlap_tcrs_wasserstein.csv")
df = df.dropna(subset=["Epitope_species", "Epitope_gene", "signed_wasserstein"])

# Normalize species names
species_map = {
    'DENV1': 'DENV', 'DENV3/4': 'DENV', 'HIV-1': 'HIV',
    'Homo Sapiens': None, 'HomoSapiens': None,
    'SARS-CoV': 'SARS', 'SARS-CoV-2': 'SARS'
}
df["Epitope_species"] = df["Epitope_species"].replace(species_map)
df = df[df["Epitope_species"].notna()]

# Gene normalization
gene_norm_dict = {
    "IE1": "IE1", "IE-1": "IE1", "pp65": "pp65", "pp50": "pp50", "IE2": "IE2", "UL29/28": "UL29/28", "UL40": "UL40",
    "M": "M", "Flu-MP": "Flu-MP", "NP": "NP", "HA": "HA", "M1": "M1", "PA": "PA", "PB1": "PB1", "PB": "PB",
    "NS2": "NS2", "NEF": "NEF", "M1-F5L": "M1-F5L", "M1-G4E": "M1-G4E",
    "EBNA-3B": "EBNA3B", "EBNA3B": "EBNA3B", "EBNA3A": "EBNA3A", "EBNA-3A": "EBNA3A", "EBNA4": "EBNA4",
    "BMLF1": "BMLF1", "BZLF1": "BZLF1", "LMP2A": "LMP2A", "BRLF1": "BRLF1", "EMNA-3A": "EBNA3A",
    "EBNA1": "EBNA1", "LMP1": "LMP1", "EBNA-6": "EBNA6", "EBNA6": "EBNA6",
    "Spike": "Spike", "ORF1ab": "ORF1ab", "Nucleocapsid": "Nucleocapsid", "Nucleocapsid  ": "Nucleocapsid",
    "ORF3": "ORF3", "NSP3": "NSP3", "Matrix": "Matrix", "ORF7a": "ORF7a", "RNP": "RNP", "Envelope": "Envelope",
    "ORF9b": "ORF9b", "ORF7b": "ORF7b", "ORF8": "ORF8", "ORF14": "ORF14", "ORF10": "ORF10", "ORF6": "ORF6",
    "Gag": "Gag", "Gag-protein": "Gag", "gp160": "GP160", "GP160": "GP160", "Nef": "Nef", "Pol": "Pol", "POL": "Pol",
    "Vif": "Vif", "Vpr": "Vpr", "VPR": "Vpr", "RT": "RT", "GAG": "Gag",
    "NS3": "NS3", "NS5B": "NS5B", "CORE": "CORE", "NS4B": "NS4B",
    "Rv1518": "Rv1518", "Rv1734c": "Rv1734c",
    "NS1": "NS1",
}

# Normalize gene column
df["Epitope_gene_norm"] = df["Epitope_gene"].map(gene_norm_dict).fillna(df["Epitope_gene"])

# Get top 6 species by total counts (optional: change n)
top_species = df["Epitope_species"].value_counts().nlargest(50).index.tolist()

for species in top_species:
    df_sp = df[df["Epitope_species"] == species].copy()

    # Drop genes with fewer than 10 examples
    gene_counts = df_sp["Epitope_gene_norm"].value_counts()
    valid_genes = gene_counts[gene_counts >= 10].index
    df_sp = df_sp[df_sp["Epitope_gene_norm"].isin(valid_genes)]
    ordered_genes = df_sp.groupby("Epitope_gene_norm")["signed_wasserstein"].median().sort_values().index.tolist()
    df_sp["Epitope_gene_norm"] = pd.Categorical(df_sp["Epitope_gene_norm"], categories=ordered_genes, ordered=True)

    # Compute p-values between all gene pairs
    pval_dict = {}
    for g1, g2 in itertools.combinations(ordered_genes, 2):
        vals1 = df_sp[df_sp["Epitope_gene_norm"] == g1]["signed_wasserstein"]
        vals2 = df_sp[df_sp["Epitope_gene_norm"] == g2]["signed_wasserstein"]
        if len(vals1) > 0 and len(vals2) > 0:
            stat, p = mannwhitneyu(vals1, vals2, alternative='two-sided')
            pval_dict[(g1, g2)] = p

    sorted_pvals = sorted(pval_dict.items(), key=lambda x: x[1])
    legend_entries = [f"{a} vs {b}: p={p:.2e}" for (a, b), p in sorted_pvals]
    legend_colors = ['#ffeeee' if p < 0.05 else 'none' for (_, p) in sorted_pvals]

    # Plot
        # Plot setup
    fig, ax = plt.subplots(figsize=(max(10, len(ordered_genes) * 1.2), 6))

    # Colored boxplot
    palette = sns.color_palette("Set2", len(ordered_genes))
    sns.boxplot(data=df_sp, x="Epitope_gene_norm", y="signed_wasserstein",
                ax=ax, showfliers=False, palette=palette)

    # Optional: Overlay dots for density (adds insight)
    #sns.stripplot(data=df_sp, x="Epitope_gene_norm", y="signed_wasserstein",
    #              ax=ax, color='black', alpha=0.1, jitter=0.25, size=3)

    # Titles and labels
    ax.set_title(f"{species} - Gene-level Age Association", fontsize=14)
    ax.set_ylabel("Signed Wasserstein", fontsize=12)
    ax.set_xlabel("Gene", fontsize=12)

    # Sample sizes under each tick
    gene_n = df_sp["Epitope_gene_norm"].value_counts().reindex(ordered_genes).fillna(0).astype(int)
    xticks = [f"{g}\n(n={gene_n[g]})" for g in ordered_genes]
    ax.set_xticklabels(xticks, rotation=30, ha='right', fontsize=10)

    # Draw right-side legend with colored background for significant comparisons
    for i, (text, p) in enumerate(zip(legend_entries, [p for (_, p) in sorted_pvals])):
        bg_color = "#fca5a5" if p < 0.05 else "none"  # stronger red for sig, none for rest
        fig.text(1.01, 0.95 - i * 0.035, text,
                 fontsize=8,
                 bbox=dict(facecolor=bg_color, edgecolor='gray', boxstyle='round,pad=0.3'))

    plt.tight_layout()
    plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load and clean your data (assume gene_norm_dict etc. already defined)
df = pd.read_csv("merged_overlap_tcrs_wasserstein.csv")
df = df.dropna(subset=["Epitope_species", "Epitope_gene", "signed_wasserstein"])
df['Epitope_gene_norm'] = df['Epitope_gene'].map(gene_norm_dict).fillna(df['Epitope_gene'])

min_gene_n = 50  # Only plot genes with at least this many TCRs in a species
min_species_n = 2  # Only plot species with at least this many qualifying genes

species_list = sorted(df['Epitope_species'].unique())

for species in species_list:
    df_sp = df[df['Epitope_species'] == species].copy()
    gene_counts = df_sp['Epitope_gene_norm'].value_counts()
    valid_genes = gene_counts[gene_counts >= min_gene_n].index
    df_sp_plot = df_sp[df_sp['Epitope_gene_norm'].isin(valid_genes)]
    if df_sp_plot['Epitope_gene_norm'].nunique() < min_species_n:
        continue  # Skip if too few genes

    # Sort genes by mean signed_wasserstein
    means = df_sp_plot.groupby('Epitope_gene_norm')['signed_wasserstein'].mean().sort_values()
    sorted_genes = means.index.tolist()
    df_sp_plot['Epitope_gene_norm'] = pd.Categorical(df_sp_plot['Epitope_gene_norm'], categories=sorted_genes, ordered=True)

    plt.figure(figsize=(max(8, len(sorted_genes)), 6))
    ax = sns.violinplot(
        x='Epitope_gene_norm',
        y='signed_wasserstein',
        data=df_sp_plot,
        order=sorted_genes,
        cut=0,
        inner='box'
    )
    plt.ylabel('Signed Wasserstein (Age Association)')
    plt.xlabel('Gene')
    plt.title(f'{species}: Age Association (Signed Wasserstein) by Gene')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Assuming df is already loaded and cleaned

species_counts = df['Epitope_species'].value_counts().sort_values(ascending=True)  # ascending for horizontal plot

plt.figure(figsize=(10, max(6, len(species_counts) * 0.4)))
species_counts.plot(kind='barh')
plt.xlabel('Number of TCRs')
plt.ylabel('Species')
plt.title('TCR Count per Species')
plt.tight_layout()
plt.xscale('log')
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load data, normalize gene names as before...
# (include your gene_norm_dict, species_map, and DataFrame preprocessing)

threshold = 1.28  # Use your chosen threshold

# Select genes at the threshold, aggregate across all species
df_thr = df[(df["component_zscore"] < -threshold) | (df["component_zscore"] > threshold)].copy()
df_thr["Group"] = df_thr["component_zscore"].apply(lambda z: "Younger" if z < -threshold else "Older")

# Group by normalized gene only
counts = df_thr.groupby(["Epitope_gene_norm", "Group"]).size().unstack(fill_value=0)
counts = counts[["Younger", "Older"]]  # ensure order

# Optional: sort by sum or by difference
counts = counts.loc[counts.sum(axis=1).sort_values(ascending=False).index]

# Plot
ax = counts.plot(kind="bar", figsize=(18, 7), width=0.8)
plt.title(f"Total counts for all genes (Younger vs Older) | Z-score {threshold}")
plt.ylabel("Count")
plt.xlabel("Gene")
plt.xticks(rotation=45, ha="right")
plt.legend(title="Group")
plt.tight_layout()
plt.yscale("log")
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency

df = pd.read_csv("merged_overlap_tcrs_wasserstein.csv")
df = df.dropna(subset=["Epitope_species", "Epitope_gene"])

species_map = {
    'DENV1': 'DENV', 'DENV3/4': 'DENV', 'HIV-1': 'HIV',
    'Homo Sapiens': None, 'HomoSapiens': None,
    'SARS-CoV': 'SARS', 'SARS-CoV-2': 'SARS'
}
df["Epitope_species"] = df["Epitope_species"].replace(species_map)
df = df[df["Epitope_species"].notna()]

df['Epitope_gene_norm'] = df['Epitope_gene'].map(gene_norm_dict).fillna(df['Epitope_gene'])

thresholds = [1.28, 1.64, 1.96]  # Only these three
alpha = 0.25
report_alpha = 0.25

def analyze_species_all_thresholds(df, species, thresholds, alpha=0.05, report_alpha=0.1):
    gene_pvals = {thresh: {} for thresh in thresholds}
    counts_dict = {}

    temp_df = df[df["Epitope_species"] == species].copy()
    if temp_df.shape[0] == 0:
        return

    for thresh in thresholds:
        th_df = temp_df[(temp_df["component_zscore"] < -thresh) | (temp_df["component_zscore"] > thresh)].copy()
        if th_df.shape[0] == 0:
            counts_dict[thresh] = None
            continue
        th_df["Group"] = th_df["component_zscore"].apply(lambda z: "Younger" if z < -thresh else "Older")
        counts = th_df.groupby(["Epitope_gene_norm", "Group"]).size().unstack(fill_value=0)
        for group in ["Younger", "Older"]:
            if group not in counts.columns:
                counts[group] = 0
        counts = counts[["Younger", "Older"]]
        counts_dict[thresh] = counts

        for gene in counts.index:
            c = counts.loc[gene]
            table = [
                [c["Younger"], c["Older"]],
                [counts["Younger"].sum() - c["Younger"], counts["Older"].sum() - c["Older"]]
            ]
            try:
                chi2, p, _, _ = chi2_contingency(table)
                gene_pvals[thresh][gene] = p
            except Exception:
                continue

    if not any([p < report_alpha for pv in gene_pvals.values() for p in pv.values()]):
        return

    # Make split-plot (3 subplots now)
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    for idx, thresh in enumerate(thresholds):
        counts = counts_dict[thresh]
        ax = axes[idx]
        if counts is not None and counts.shape[0] > 0:
            counts_norm = counts.div(counts.sum(axis=0), axis=1)
            counts_norm.plot(kind='bar', width=0.8, ax=ax, legend=False)
            ax.set_title(f"{species} | Z-score {thresh}")
            ax.set_ylabel("Proportion")
            ax.set_xlabel("Epitope_gene")
            ax.grid(True)
            ax.tick_params(axis='x', rotation=45)
        else:
            ax.set_title(f"{species} | Z-score {thresh} (no data)")
            ax.axis('off')
    plt.suptitle(f"{species} - Normalized Epitope_gene counts across thresholds")
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

    # Print significant genes for each threshold
    for thresh in thresholds:
        print(f"\nSignificant differences in {species} (threshold={thresh}):")
        found_any = False
        for gene, p in gene_pvals[thresh].items():
            counts = counts_dict[thresh]
            c = counts.loc[gene]
            if p < alpha:
                found_any = True
                print(f"Gene: {gene:20} | Younger: {c['Younger']:3} | Older: {c['Older']:3} | p = {p:.3e}")
        if not found_any:
            print("No significant genes found.")

for sp in df["Epitope_species"].unique():
    analyze_species_all_thresholds(df, sp, thresholds, alpha=alpha, report_alpha=report_alpha)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import chi2_contingency

df = pd.read_csv("merged_overlap_tcrs_wasserstein.csv")
df = df.dropna(subset=["Epitope_species", "Epitope_gene"])

species_map = {
    'DENV1': 'DENV', 'DENV3/4': 'DENV', 'HIV-1': 'HIV',
    'Homo Sapiens': None, 'HomoSapiens': None,
    'SARS-CoV': 'SARS', 'SARS-CoV-2': 'SARS'
}
df["Epitope_species"] = df["Epitope_species"].replace(species_map)
df = df[df["Epitope_species"].notna()]

df['Epitope_gene_norm'] = df['Epitope_gene'].map(gene_norm_dict).fillna(df['Epitope_gene'])

threshold = 1.64
alpha = 0.05
report_alpha = 0.1  # threshold for plotting/reporting

def analyze_species(df, species, threshold, alpha=0.05, report_alpha=0.1):
    temp_df = df[df["Epitope_species"] == species].copy()
    temp_df = temp_df[(temp_df["component_zscore"] < -threshold) | (temp_df["component_zscore"] > threshold)]
    temp_df["Group"] = temp_df["component_zscore"].apply(lambda z: "Younger" if z < -threshold else "Older")

    counts = temp_df.groupby(["Epitope_gene_norm", "Group"]).size().unstack(fill_value=0)
    for group in ["Younger", "Older"]:
        if group not in counts.columns:
            counts[group] = 0
    counts = counts[["Younger", "Older"]]

    if counts.shape[0] == 0:
        return  # nothing to plot or test

    # Compute all p-values first
    pvals = {}
    for gene in counts.index:
        c = counts.loc[gene]
        table = [
            [c["Younger"], c["Older"]],
            [counts["Younger"].sum() - c["Younger"], counts["Older"].sum() - c["Older"]]
        ]
        try:
            chi2, p, _, _ = chi2_contingency(table)
            pvals[gene] = p
        except Exception:
            continue

    # Only plot if at least one gene has p < report_alpha
    if not any([p < report_alpha for p in pvals.values()]):
        return

    # Plot normalized counts
    counts_norm = counts.div(counts.sum(axis=0), axis=1)
    counts_norm.plot(kind='bar', width=0.8, figsize=(12, 5))
    plt.title(f"{species} | Z-score {threshold} | Normalized Epitope_gene counts")
    plt.ylabel("Proportion")
    plt.legend(title="Group")
    plt.xticks(rotation=45)
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    print(f"\nSignificant differences in {species} (threshold={threshold}):")
    found_any = False
    for gene, p in pvals.items():
        c = counts.loc[gene]
        if p < alpha:
            found_any = True
            print(f"Gene: {gene:20} | Younger: {c['Younger']:3} | Older: {c['Older']:3} | p = {p:.3e}")
    if not found_any:
        print("No genes with p < 0.05, but at least one with p < 0.1.")

# Run for all species in the data
for sp in df["Epitope_species"].unique():
    analyze_species(df, sp, threshold, alpha=alpha, report_alpha=report_alpha)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load and clean data
df = pd.read_csv("merged_overlap_tcrs_wasserstein.csv")
df = df.dropna(subset=["Epitope_species", "Epitope_gene"])

species_map = {
    'DENV1': 'DENV', 'DENV3/4': 'DENV', 'HIV-1': 'HIV',
    'Homo Sapiens': None, 'HomoSapiens': None,
    'SARS-CoV': 'SARS', 'SARS-CoV-2': 'SARS'
}
df["Epitope_species"] = df["Epitope_species"].replace(species_map)
df = df[df["Epitope_species"].notna()]

# Your gene normalization dictionary (insert full dict here)
gene_norm_dict = {
    # CMV
    "IE1": "IE1",
    "IE-1": "IE1",
    "pp65": "pp65",
    "pp50": "pp50",
    "IE2": "IE2",
    "UL29/28": "UL29/28",
    "UL40": "UL40",
    # InfluenzaA
    "M": "M",
    "Flu-MP": "Flu-MP",
    "NP": "NP",
    "HA": "HA",
    "M1": "M1",
    "PA": "PA",
    "PB1": "PB1",
    "PB": "PB",
    "NS2": "NS2",
    "NEF": "NEF",
    "M1-F5L": "M1-F5L",
    "M1-G4E": "M1-G4E",
    # EBV
    "EBNA-3B": "EBNA3B",
    "EBNA3B": "EBNA3B",
    "EBNA3A": "EBNA3A",
    "EBNA-3A": "EBNA3A",
    "EBNA4": "EBNA4",
    "BMLF1": "BMLF1",
    "BZLF1": "BZLF1",
    "LMP2A": "LMP2A",
    "BRLF1": "BRLF1",
    "EMNA-3A": "EBNA3A",
    "EBNA1": "EBNA1",
    "LMP1": "LMP1",
    "EBNA-6": "EBNA6",
    "EBNA6": "EBNA6",
    # SARS
    "Spike": "Spike",
    "ORF1ab": "ORF1ab",
    "Nucleocapsid": "Nucleocapsid",
    "Nucleocapsid  ": "Nucleocapsid",
    "ORF3": "ORF3",
    "NSP3": "NSP3",
    "Matrix": "Matrix",
    "ORF7a": "ORF7a",
    "RNP": "RNP",
    "Envelope": "Envelope",
    "ORF9b": "ORF9b",
    "ORF7b": "ORF7b",
    "ORF8": "ORF8",
    "ORF14": "ORF14",
    "ORF10": "ORF10",
    "ORF6": "ORF6",
    # HIV
    "Gag": "Gag",
    "Gag-protein": "Gag",
    "gp160": "GP160",
    "GP160": "GP160",
    "Nef": "Nef",
    "Pol": "Pol",
    "POL": "Pol",
    "Vif": "Vif",
    "Vpr": "Vpr",
    "VPR": "Vpr",
    "RT": "RT",
    "GAG": "Gag",
    # HCV
    "NS3": "NS3",
    "NS5B": "NS5B",
    "CORE": "CORE",
    "NS4B": "NS4B",
    "POL": "POL",
    # YFV
    "NS4B": "NS4B",
    # DENV
    "NS3": "NS3",
    # Mtb
    "Rv1518": "Rv1518",
    "Rv1734c": "Rv1734c",
    # Influenza B
    "NS1": "NS1",
    "NP": "NP",
}

df['Epitope_gene_norm'] = df['Epitope_gene'].map(gene_norm_dict).fillna(df['Epitope_gene'])

thresholds = [1.96, 1.64, 1.28, 1]

def plot_threshold_pair(df, species, thresh, ax_norm, ax_raw):
    temp_df = df[df["Epitope_species"] == species].copy()
    temp_df["Group"] = "All"
    temp_df.loc[temp_df["component_zscore"] < -thresh, "Group"] = "Younger"
    temp_df.loc[temp_df["component_zscore"] > thresh, "Group"] = "Older"

    counts = temp_df.groupby(["Epitope_gene_norm", "Group"]).size().unstack(fill_value=0)
    counts_norm = counts.div(counts.sum(axis=0), axis=1)

    counts_norm.plot(kind="bar", ax=ax_norm, legend=False)
    ax_norm.set_title(f"Norm. counts (Z>{thresh}) - {species}")
    ax_norm.set_ylabel("Proportion")
    ax_norm.tick_params(axis='x', rotation=45)

    counts.plot(kind="bar", ax=ax_raw, legend=False)
    ax_raw.set_title(f"Raw counts (Z>{thresh}) - {species}")
    ax_raw.set_ylabel("Count")
    ax_raw.tick_params(axis='x', rotation=45)

for species in df["Epitope_species"].unique()[:20]:  # top 3 species or change as needed
    fig, axs = plt.subplots(len(thresholds), 2, figsize=(14, 4*len(thresholds)))
    fig.suptitle(f"{species} Epitope Gene Distributions Across Thresholds", fontsize=16)
    for i, thresh in enumerate(thresholds):
        plot_threshold_pair(df, species, thresh, axs[i, 0], axs[i, 1])
    # Add legend to bottom right plot
    handles, labels = axs[-1, 1].get_legend_handles_labels()
    fig.legend(handles, labels, title="Group", bbox_to_anchor=(1.02, 0.5), loc='center left')
    plt.tight_layout(rect=[0, 0, 0.9, 0.96])
    plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load and clean
df = pd.read_csv("merged_overlap_tcrs_wasserstein.csv")
df = df.dropna(subset=["component_zscore", "Epitope_species"])

species_map = {
    'DENV1': 'DENV', 'DENV3/4': 'DENV', 'HIV-1': 'HIV',
    'Homo Sapiens': None, 'HomoSapiens': None,
    'SARS-CoV': 'SARS', 'SARS-CoV-2': 'SARS'
}
df["Epitope_species"] = df["Epitope_species"].replace(species_map)
df = df[df["Epitope_species"].notna()]

thresholds = [1, 1.28, 1.64, 1.96]

for thresh in thresholds:
    temp_df = df.copy()
    temp_df["Group"] = "All"
    temp_df.loc[temp_df["component_zscore"] < -thresh, "Group"] = "Younger"
    temp_df.loc[temp_df["component_zscore"] > thresh, "Group"] = "Older"
    temp_df = temp_df[temp_df["Group"].isin(["Younger", "Older"])]

    # Count per species per group
    counts = temp_df.groupby(["Epitope_species", "Group"]).size().unstack(fill_value=0)

    # Normalize counts by even-ing out group sizes
    younger_total = counts["Younger"].sum()
    older_total = counts["Older"].sum()

    if younger_total > older_total:
        scale_factor = younger_total / older_total
        counts["Older"] = counts["Older"] * scale_factor
    else:
        scale_factor = older_total / younger_total
        counts["Younger"] = counts["Younger"] * scale_factor

    # Plot grouped bar chart
    ax = counts.plot(kind="bar", figsize=(12, 6))
    ax.set_title(f"Epitope_species Counts (Normalized) at Z-threshold {thresh}")
    ax.set_ylabel("Normalized TCR Count")
    ax.set_xlabel("Epitope_species")
    ax.legend(title="Group")
    plt.yscale("log")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import fisher_exact

# Load and clean
df = pd.read_csv("merged_overlap_tcrs_wasserstein.csv")
df = df.dropna(subset=["component_zscore", "Epitope_species"])

species_map = {
    'DENV1': 'DENV', 'DENV3/4': 'DENV', 'HIV-1': 'HIV',
    'Homo Sapiens': None, 'HomoSapiens': None,
    'SARS-CoV': 'SARS', 'SARS-CoV-2': 'SARS'
}
df["Epitope_species"] = df["Epitope_species"].replace(species_map)
df = df[df["Epitope_species"].notna()]

# Biological clustering for each species as in your plot
cluster_map = {
    "CMV": "Herpesvirus", "EBV": "Herpesvirus", "MCMV": "Herpesvirus",
    "InfluenzaA": "Orthomyxovirus", "Influenza B": "Orthomyxovirus",
    "SARS": "Coronavirus", "HCoV-HKU1": "Coronavirus",
    "HIV": "Retrovirus", "HTLV-1": "Retrovirus", "SIV": "Retrovirus",
    "DENV": "Flavivirus", "DENV2": "Flavivirus", "YFV": "Flavivirus", "HCV": "Flavivirus",
    "Mtb": "Bacteria", "M.tuberculosis": "Bacteria", "E.Coli": "Bacteria",
    "Wheat": "Plant/Food", "TriticumAestivum": "Plant/Food", "Selaginella Moellendorffii": "Plant/Food",
    "MusMusculus": "Mouse/Control", "Synthetic": "Synthetic/Control",
    "MCPyV": "Polyomavirus", "HPV-16": "Papillomavirus", "AdV": "Adenovirus",
    "RSV": "Paramyxovirus", "LCMV": "Other", "RotavirusA": "Other", "CoxsackievirusB": "Other",
    "Streptomyceskanamyceticus": "Bacteria", "Salmonella Moellendorffii": "Bacteria",
    "Trypanosoma cruzi": "Parasite"
}
df["Biology_Cluster"] = df["Epitope_species"].map(cluster_map).fillna("Other")

thresholds = [1, 1.28, 1.64, 1.96]
alpha = 0.05

for thresh in thresholds:
    df["Group"] = "All"
    df.loc[df["component_zscore"] < -thresh, "Group"] = "Younger"
    df.loc[df["component_zscore"] > thresh, "Group"] = "Older"
    df_sub = df[df["Group"].isin(["Younger", "Older"])]

    # Count per cluster per group
    clust_counts = df_sub.groupby(["Biology_Cluster", "Group"]).size().unstack(fill_value=0)

    # Normalize by even-ing out group sizes
    younger_total = clust_counts["Younger"].sum()
    older_total = clust_counts["Older"].sum()
    if younger_total > older_total:
        clust_counts["Older"] = clust_counts["Older"] * (younger_total / older_total)
    else:
        clust_counts["Younger"] = clust_counts["Younger"] * (older_total / younger_total)

    # Fisher's Exact Test for each cluster
    print(f"\nZ-threshold {thresh} — Significant Fisher's p-values (p < {alpha}):")
    any_sig = False
    for cluster in clust_counts.index:
        cluster_older = clust_counts.loc[cluster, "Older"]
        cluster_younger = clust_counts.loc[cluster, "Younger"]
        other_older = clust_counts["Older"].sum() - cluster_older
        other_younger = clust_counts["Younger"].sum() - cluster_younger
        contingency = [[cluster_older, cluster_younger],
                       [other_older, other_younger]]
        _, p = fisher_exact(contingency)
        if p < alpha:
            any_sig = True
            print(f"{cluster:<20}{cluster_older:10.1f}{cluster_younger:12.1f}{p:14.3g}")
    if not any_sig:
        print("No significant differences found for this threshold.")

    # Plot
    ax = clust_counts.plot(kind="bar", figsize=(10, 5))
    ax.set_title(f"TCR Count by Pathogen Cluster (Normalized, Z>{thresh})")
    ax.set_ylabel("Normalized TCR Count")
    ax.set_xlabel("Biological Cluster")
    ax.legend(title="Group")
    plt.yscale("log")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    #plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import fisher_exact

# Load and clean
df = pd.read_csv("merged_overlap_tcrs_wasserstein.csv")
df = df.dropna(subset=["component_zscore", "Epitope_species"])

species_map = {
    'DENV1': 'DENV', 'DENV3/4': 'DENV', 'HIV-1': 'HIV',
    'Homo Sapiens': None, 'HomoSapiens': None,
    'SARS-CoV': 'SARS', 'SARS-CoV-2': 'SARS'
}
df["Epitope_species"] = df["Epitope_species"].replace(species_map)
df = df[df["Epitope_species"].notna()]

thresholds = [1, 1.28, 1.64, 1.96]
alpha = 0.05  # significance threshold

for thresh in thresholds:
    temp_df = df.copy()
    temp_df["Group"] = "All"
    temp_df.loc[temp_df["component_zscore"] < -thresh, "Group"] = "Younger"
    temp_df.loc[temp_df["component_zscore"] > thresh, "Group"] = "Older"
    temp_df = temp_df[temp_df["Group"].isin(["Younger", "Older"])]

    # Count per species per group
    counts = temp_df.groupby(["Epitope_species", "Group"]).size().unstack(fill_value=0)

    # Normalize counts by evening out group sizes
    younger_total = counts["Younger"].sum()
    older_total = counts["Older"].sum()

    if younger_total > older_total:
        scale_factor = younger_total / older_total
        counts["Older"] = counts["Older"] * scale_factor
    else:
        scale_factor = older_total / younger_total
        counts["Younger"] = counts["Younger"] * scale_factor

    # Calculate and print only significant p-values
    print(f"\nZ-threshold {thresh} — Significant Fisher's Exact p-values (p < {alpha}):")
    any_sig = False
    for species in counts.index:
        species_older = counts.loc[species, "Older"]
        species_younger = counts.loc[species, "Younger"]
        other_older = counts["Older"].sum() - species_older
        other_younger = counts["Younger"].sum() - species_younger
        contingency = [[species_older, species_younger],
                       [other_older, other_younger]]
        _, p = fisher_exact(contingency)
        if p < alpha:
            any_sig = True
            print(f"{species:<25}{species_older:10.1f}{species_younger:12.1f}{p:14.3g}")
    if not any_sig:
        print("No significant differences found for this threshold.")

    # (Optional) Plot grouped bar chart
    ax = counts.plot(kind="bar", figsize=(12, 6))
    ax.set_title(f"Epitope_species Counts (Normalized) at Z-threshold {thresh}")
    ax.set_ylabel("Normalized TCR Count")
    ax.set_xlabel("Epitope_species")
    ax.legend(title="Group")
    plt.yscale("log")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    #plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import fisher_exact

# Load and clean data
df = pd.read_csv("merged_overlap_tcrs_wasserstein.csv")
df = df.dropna(subset=["component_zscore", "Epitope_species"])

species_map = {
    'DENV1': 'DENV', 'DENV3/4': 'DENV', 'HIV-1': 'HIV',
    'Homo Sapiens': None, 'HomoSapiens': None,
    'SARS-CoV': 'SARS', 'SARS-CoV-2': 'SARS'
}
df["Epitope_species"] = df["Epitope_species"].replace(species_map)
df = df[df["Epitope_species"].notna()]

thresholds = [1, 1.28, 1.64, 1.96]
alpha = 0.05

# Define all clusterings
clusterings = {
    "PathogenFamily": {
        "CMV": "Herpesvirus", "EBV": "Herpesvirus", "MCMV": "Herpesvirus",
        "InfluenzaA": "Orthomyxovirus", "Influenza B": "Orthomyxovirus",
        "SARS": "Coronavirus", "HCoV-HKU1": "Coronavirus",
        "HIV": "Retrovirus", "HTLV-1": "Retrovirus", "SIV": "Retrovirus",
        "DENV": "Flavivirus", "DENV2": "Flavivirus", "YFV": "Flavivirus", "HCV": "Flavivirus",
        "Mtb": "Bacteria", "M.tuberculosis": "Bacteria", "E.Coli": "Bacteria",
        "Wheat": "Plant/Food", "TriticumAestivum": "Plant/Food", "Selaginella Moellendorffii": "Plant/Food",
        "MusMusculus": "Mouse/Control", "Synthetic": "Synthetic/Control",
        "MCPyV": "Polyomavirus", "HPV-16": "Papillomavirus", "AdV": "Adenovirus",
        "RSV": "Paramyxovirus", "LCMV": "Other", "RotavirusA": "Other", "CoxsackievirusB": "Other",
        "Streptomyceskanamyceticus": "Bacteria", "Salmonella Moellendorffii": "Bacteria",
        "Trypanosoma cruzi": "Parasite"
    },
    "AcuteChronic": {
        "CMV": "Chronic", "EBV": "Chronic", "HIV": "Chronic", "HCV": "Chronic", "Mtb": "Chronic",
        "M.tuberculosis": "Chronic", "MCMV": "Chronic",
        "InfluenzaA": "Acute", "Influenza B": "Acute", "DENV": "Acute", "DENV2": "Acute", "YFV": "Acute",
        "SARS": "Acute", "HCoV-HKU1": "Acute", "RSV": "Acute", "LCMV": "Acute", "RotavirusA": "Acute",
        "Synthetic": "Control", "MusMusculus": "Control", "Wheat": "Control", "TriticumAestivum": "Control",
        "Selaginella Moellendorffii": "Control", "E.Coli": "Control", "Salmonella Moellendorffii": "Control",
        "Streptomyceskanamyceticus": "Control", "Trypanosoma cruzi": "Control", "AdV": "Acute",
        "HTLV-1": "Chronic", "SIV": "Chronic", "MCPyV": "Chronic", "HPV-16": "Chronic", "CoxsackievirusB": "Acute"
    },
    "VaccineStatus": {
        "InfluenzaA": "Vaccine", "Influenza B": "Vaccine", "Mtb": "Vaccine", "M.tuberculosis": "Vaccine",
        "DENV": "Vaccine", "DENV2": "Vaccine", "YFV": "Vaccine",
        "CMV": "Natural", "EBV": "Natural", "HIV": "Natural", "HCV": "Natural", "MCMV": "Natural", "SARS": "Natural",
        "HCoV-HKU1": "Natural", "RSV": "Natural", "LCMV": "Natural", "RotavirusA": "Natural", "HTLV-1": "Natural",
        "SIV": "Natural", "MCPyV": "Natural", "HPV-16": "Natural", "E.Coli": "Natural", "Salmonella Moellendorffii": "Natural",
        "Streptomyceskanamyceticus": "Natural", "MusMusculus": "Control", "Synthetic": "Control", "Wheat": "Control",
        "TriticumAestivum": "Control", "Selaginella Moellendorffii": "Control", "AdV": "Vaccine", "Trypanosoma cruzi": "Natural",
        "CoxsackievirusB": "Natural"
    },
    "Route": {
        "InfluenzaA": "Respiratory", "Influenza B": "Respiratory", "SARS": "Respiratory", "HCoV-HKU1": "Respiratory", "RSV": "Respiratory",
        "CMV": "Blood/Other", "EBV": "Blood/Other", "DENV": "Blood/Other", "DENV2": "Blood/Other", "YFV": "Blood/Other",
        "HIV": "Blood/Other", "HCV": "Blood/Other", "HTLV-1": "Blood/Other", "SIV": "Blood/Other", "RotavirusA": "GI",
        "Mtb": "Respiratory", "M.tuberculosis": "Respiratory", "MCMV": "Blood/Other", "MCPyV": "Other", "HPV-16": "Other",
        "LCMV": "Other", "Synthetic": "Control", "MusMusculus": "Control", "Wheat": "Control", "TriticumAestivum": "Control",
        "Selaginella Moellendorffii": "Control", "E.Coli": "GI", "Salmonella Moellendorffii": "GI", "Streptomyceskanamyceticus": "Other",
        "AdV": "Respiratory", "Trypanosoma cruzi": "Blood/Other", "CoxsackievirusB": "Respiratory"
    },
    "Zoonotic": {
        "InfluenzaA": "Zoonotic", "SARS": "Zoonotic", "YFV": "Zoonotic", "DENV": "Zoonotic", "DENV2": "Zoonotic",
        "EBV": "Human", "CMV": "Human", "Influenza B": "Human", "Mtb": "Human", "M.tuberculosis": "Human", "HIV": "Human",
        "HCV": "Human", "HTLV-1": "Human", "SIV": "Zoonotic", "RotavirusA": "Human", "MusMusculus": "Control", "Synthetic": "Control",
        "MCMV": "Zoonotic", "HCoV-HKU1": "Human", "RSV": "Human", "LCMV": "Zoonotic", "MCPyV": "Human", "HPV-16": "Human",
        "Wheat": "Control", "TriticumAestivum": "Control", "Selaginella Moellendorffii": "Control", "E.Coli": "Human",
        "Salmonella Moellendorffii": "Control", "Streptomyceskanamyceticus": "Control", "AdV": "Human", "Trypanosoma cruzi": "Zoonotic",
        "CoxsackievirusB": "Human"
    },
    "ExposureTiming": {
        "Mtb": "Childhood", "M.tuberculosis": "Childhood", "EBV": "Childhood", "InfluenzaA": "Childhood", "DENV": "Childhood",
        "DENV2": "Childhood", "YFV": "Childhood", "RotavirusA": "Childhood",
        "CMV": "Adulthood", "HIV": "Adulthood", "HCV": "Adulthood", "HTLV-1": "Adulthood", "Influenza B": "Childhood",
        "SARS": "Adulthood", "HCoV-HKU1": "Childhood", "RSV": "Childhood", "MCMV": "Adulthood", "MCPyV": "Adulthood",
        "HPV-16": "Adulthood", "SIV": "Adulthood", "LCMV": "Adulthood", "Synthetic": "Control", "MusMusculus": "Control",
        "Wheat": "Control", "TriticumAestivum": "Control", "Selaginella Moellendorffii": "Control", "E.Coli": "Control",
        "Salmonella Moellendorffii": "Control", "Streptomyceskanamyceticus": "Control", "AdV": "Childhood",
        "Trypanosoma cruzi": "Adulthood", "CoxsackievirusB": "Childhood"
    }
}

for clust_type, mapping in clusterings.items():
    print(f"\n===== {clust_type} =====")
    df[clust_type] = df["Epitope_species"].map(mapping).fillna("Other")
    for thresh in thresholds:
        df["Group"] = "All"
        df.loc[df["component_zscore"] < -thresh, "Group"] = "Younger"
        df.loc[df["component_zscore"] > thresh, "Group"] = "Older"
        df_sub = df[df["Group"].isin(["Younger", "Older"])]
        # Count per cluster per group
        clust_counts = df_sub.groupby([clust_type, "Group"]).size().unstack(fill_value=0)
        # Normalize
        younger_total = clust_counts["Younger"].sum()
        older_total = clust_counts["Older"].sum()
        if younger_total > older_total:
            clust_counts["Older"] = clust_counts["Older"] * (younger_total / older_total)
        else:
            clust_counts["Younger"] = clust_counts["Younger"] * (older_total / younger_total)
        # Fisher's Exact Test for each cluster
        print(f"\nZ-threshold {thresh} — Significant Fisher's p-values (p < {alpha}):")
        any_sig = False
        for cluster in clust_counts.index:
            cluster_older = clust_counts.loc[cluster, "Older"]
            cluster_younger = clust_counts.loc[cluster, "Younger"]
            other_older = clust_counts["Older"].sum() - cluster_older
            other_younger = clust_counts["Younger"].sum() - cluster_younger
            contingency = [[cluster_older, cluster_younger],
                           [other_older, other_younger]]
            _, p = fisher_exact(contingency)
            if p < alpha:
                any_sig = True
                print(f"{cluster:<20}{cluster_older:10.1f}{cluster_younger:12.1f}{p:14.3g}")
        if not any_sig:
            print("No significant differences found for this threshold.")
        # Plot
        ax = clust_counts.plot(kind="bar", figsize=(10, 5))
        ax.set_title(f"{clust_type} - Normalized TCR Count (Z>{thresh})")
        ax.set_ylabel("Normalized TCR Count")
        ax.set_xlabel(clust_type)
        ax.legend(title="Group")
        plt.yscale("log")
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        #plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import fisher_exact

# Load and clean
df = pd.read_csv("merged_overlap_tcrs_wasserstein.csv")
df = df.dropna(subset=["component_zscore", "Epitope_species", "Epitope_gene"])

species_map = {
    'DENV1': 'DENV', 'DENV3/4': 'DENV', 'HIV-1': 'HIV',
    'Homo Sapiens': None, 'HomoSapiens': None,
    'SARS-CoV': 'SARS', 'SARS-CoV-2': 'SARS'
}
df["Epitope_species"] = df["Epitope_species"].replace(species_map)
df = df[df["Epitope_species"].notna()]

# Normalize gene names
gene_norm_dict = {
    # ... (as you defined above)
}
df['Epitope_gene_norm'] = df['Epitope_gene'].map(gene_norm_dict).fillna(df['Epitope_gene'])

thresholds = [1, 1.28, 1.64, 1.96]
alpha = 0.05  # significance threshold

for thresh in thresholds:
    print(f"\n===== Z-threshold {thresh} =====")
    temp_df = df.copy()
    temp_df["Group"] = "All"
    temp_df.loc[temp_df["component_zscore"] < -thresh, "Group"] = "Younger"
    temp_df.loc[temp_df["component_zscore"] > thresh, "Group"] = "Older"
    temp_df = temp_df[temp_df["Group"].isin(["Younger", "Older"])]
    for species in temp_df["Epitope_species"].unique():
        sp_df = temp_df[temp_df["Epitope_species"] == species].copy()
        if sp_df.empty:
            continue

        # Count per gene per group
        counts = sp_df.groupby(["Epitope_gene_norm", "Group"]).size().unstack(fill_value=0)

        # Normalize group sizes for the species
        younger_total = counts["Younger"].sum()
        older_total = counts["Older"].sum()
        if younger_total > older_total:
            counts["Older"] = counts["Older"] * (younger_total / older_total)
        else:
            counts["Younger"] = counts["Younger"] * (older_total / younger_total)

        # Fisher's exact test per gene in this species
        sig_genes = []
        for gene in counts.index:
            gene_older = counts.loc[gene, "Older"]
            gene_younger = counts.loc[gene, "Younger"]
            other_older = counts["Older"].sum() - gene_older
            other_younger = counts["Younger"].sum() - gene_younger
            contingency = [[gene_older, gene_younger], [other_older, other_younger]]
            _, p = fisher_exact(contingency)
            if p < alpha:
                sig_genes.append((gene, gene_older, gene_younger, p))

        # Print table if significant genes found
        if sig_genes:
            print(f"\nSpecies: {species} — Significant genes at Z>{thresh}")
            print(f"{'Gene':<25}{'Older':>10}{'Younger':>12}{'p-value':>14}")
            for gene, older, younger, p in sig_genes:
                print(f"{gene:<25}{older:10.1f}{younger:12.1f}{p:14.3g}")

        # Bar plot for this species (all genes)
        ax = counts.plot(kind="bar", figsize=(10, 4))
        ax.set_title(f"{species} — Genes (Normalized, Z>{thresh})")
        ax.set_ylabel("Normalized TCR Count")
        ax.set_xlabel("Epitope Gene")
        ax.legend(title="Group")
        plt.xticks(rotation=45, ha="right")
        plt.tight_layout()
        plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load and clean
df = pd.read_csv("merged_overlap_tcrs_wasserstein.csv")
df = df.dropna(subset=["component_zscore", "Epitope_species"])

species_map = {
    'DENV1': 'DENV', 'DENV3/4': 'DENV', 'HIV-1': 'HIV',
    'Homo Sapiens': None, 'HomoSapiens': None,
    'SARS-CoV': 'SARS', 'SARS-CoV-2': 'SARS'
}
df["Epitope_species"] = df["Epitope_species"].replace(species_map)
df = df[df["Epitope_species"].notna()]

thresh = 1.28

df["Group"] = "All"
df.loc[df["component_zscore"] < -thresh, "Group"] = "Younger"
df.loc[df["component_zscore"] > thresh, "Group"] = "Older"
df_filtered = df[df["Group"].isin(["Younger", "Older"])]

# Count per species per group
counts = df_filtered.groupby(["Epitope_species", "Group"]).size().unstack(fill_value=0)

# Even out counts by scaling smaller group
younger_total = counts["Younger"].sum()
older_total = counts["Older"].sum()

if younger_total > older_total:
    scale_factor = younger_total / older_total
    counts["Older"] = counts["Older"] * scale_factor
else:
    scale_factor = older_total / younger_total
    counts["Younger"] = counts["Younger"] * scale_factor

# Plot
ax = counts.plot(kind="bar", figsize=(12, 6))
ax.set_title(f"Epitope_species Counts (Normalized) at Z-threshold {thresh}")
ax.set_ylabel("Normalized TCR Count")
ax.set_xlabel("Epitope_species")
ax.legend(title="Group")
plt.yscale("log")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load and clean
df = pd.read_csv("merged_overlap_tcrs_wasserstein.csv")
df = df.dropna(subset=["component_zscore"])

thresholds = np.arange(1.0, 1.97, 0.05)  # 1.00 to 1.95

younger_counts = []
older_counts = []

for thresh in thresholds:
    df["Group"] = "All"
    df.loc[df["component_zscore"] < -thresh, "Group"] = "Younger"
    df.loc[df["component_zscore"] > thresh, "Group"] = "Older"
    younger_counts.append((df["Group"] == "Younger").sum())
    older_counts.append((df["Group"] == "Older").sum())

# Plotting
plt.figure(figsize=(10, 6))
plt.plot(thresholds, younger_counts, label='Younger TCRs', marker='o')
plt.plot(thresholds, older_counts, label='Older TCRs', marker='s')

plt.xlabel("Z-score Threshold")
plt.ylabel("Number of TCRs")
plt.title("Number of Biased TCRs vs Z-score Threshold")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd

# Load data
df = pd.read_csv("merged_overlap_tcrs_wasserstein.csv")

# Drop rows with missing required columns
df = df.dropna(subset=["Epitope_species", "Epitope_gene"])

# Normalize species names (optional, keep consistent with earlier)
species_map = {
    'DENV1': 'DENV', 'DENV3/4': 'DENV', 'HIV-1': 'HIV',
    'Homo Sapiens': None, 'HomoSapiens': None,
    'SARS-CoV': 'SARS', 'SARS-CoV-2': 'SARS'
}
df["Epitope_species"] = df["Epitope_species"].replace(species_map)
df = df[df["Epitope_species"].notna()]

# Normalization dictionary (your dictionary)
gene_norm_dict = {
    # CMV
    "IE1": "IE1",
    "IE-1": "IE1",
    "pp65": "pp65",
    "pp50": "pp50",
    "IE2": "IE2",
    "UL29/28": "UL29/28",
    "UL40": "UL40",

    # InfluenzaA
    "M": "M",
    "Flu-MP": "Flu-MP",
    "NP": "NP",
    "HA": "HA",
    "M1": "M1",
    "PA": "PA",
    "PB1": "PB1",
    "PB": "PB",
    "NS2": "NS2",
    "NEF": "NEF",
    "M1-F5L": "M1-F5L",
    "M1-G4E": "M1-G4E",

    # EBV
    "EBNA-3B": "EBNA3B",
    "EBNA3B": "EBNA3B",
    "EBNA3A": "EBNA3A",
    "EBNA-3A": "EBNA3A",
    "EBNA4": "EBNA4",
    "BMLF1": "BMLF1",
    "BZLF1": "BZLF1",
    "LMP2A": "LMP2A",
    "BRLF1": "BRLF1",
    "EMNA-3A": "EBNA3A",
    "EBNA1": "EBNA1",
    "LMP1": "LMP1",
    "EBNA-6": "EBNA6",
    "EBNA6": "EBNA6",

    # SARS
    "Spike": "Spike",
    "ORF1ab": "ORF1ab",
    "Nucleocapsid": "Nucleocapsid",
    "Nucleocapsid  ": "Nucleocapsid",
    "ORF3": "ORF3",
    "NSP3": "NSP3",
    "Matrix": "Matrix",
    "ORF7a": "ORF7a",
    "RNP": "RNP",
    "Envelope": "Envelope",
    "ORF9b": "ORF9b",
    "ORF7b": "ORF7b",
    "ORF8": "ORF8",
    "ORF14": "ORF14",
    "ORF10": "ORF10",
    "ORF6": "ORF6",

    # HIV
    "Gag": "Gag",
    "Gag-protein": "Gag",
    "gp160": "GP160",
    "GP160": "GP160",
    "Nef": "Nef",
    "Pol": "Pol",
    "POL": "Pol",
    "Vif": "Vif",
    "Vpr": "Vpr",
    "VPR": "Vpr",
    "RT": "RT",
    "GAG": "Gag",

    # HCV
    "NS3": "NS3",
    "NS5B": "NS5B",
    "CORE": "CORE",
    "NS4B": "NS4B",
    "POL": "POL",

    # YFV
    "NS4B": "NS4B",

    # DENV
    "NS3": "NS3",

    # Mtb
    "Rv1518": "Rv1518",
    "Rv1734c": "Rv1734c",

    # Influenza B
    "NS1": "NS1",
    "NP": "NP",
}

# Apply normalization
df['Epitope_gene_norm'] = df['Epitope_gene'].map(gene_norm_dict).fillna(df['Epitope_gene'])

# Aggregate counts by species and normalized gene
agg_counts = df.groupby(['Epitope_species', 'Epitope_gene_norm']).size().reset_index(name='count')

# Get first 10 species to print as practice
species_list = agg_counts['Epitope_species'].unique()[:10]

for species in species_list:
    subset = agg_counts[agg_counts['Epitope_species'] == species].sort_values('count', ascending=False)
    gene_list = [f"{row.Epitope_gene_norm} ({row.count})" for row in subset.itertuples()]
    total_tcr = subset['count'].sum()
    print(f"\nSpecies: {species} — Total TCRs: {total_tcr} — Genes: {len(gene_list)}")
    print(", ".join(gene_list))


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load the data
df = pd.read_csv("merged_overlap_tcrs_wasserstein.csv")

# Subset groups by z-score
younger_group = df[df["component_zscore"] < -1.96]
older_group = df[df["component_zscore"] > 1.96]

# Count Epitope_species (normalized by group size)
younger_counts = (younger_group["Epitope_species"].value_counts(normalize=True)
                  .rename("Younger (normalized)"))
older_counts = (older_group["Epitope_species"].value_counts(normalize=True)
                .rename("Older (normalized)"))

# Combine
comparison_df = pd.concat([younger_counts, older_counts], axis=1).fillna(0)

# Plot
plt.figure(figsize=(10, 12))
comparison_df.plot(kind='barh', stacked=False)
plt.xlabel("Proportion")
plt.title("Normalized Epitope_species Frequencies in Younger vs Older-biased TCRs")
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load the data
df = pd.read_csv("merged_overlap_tcrs_wasserstein.csv")

# Subset groups by z-score
younger_group = df[df["component_zscore"] < -1.96]
older_group = df[df["component_zscore"] > 1.96]

# Count Epitope_gene (normalized by group size)
younger_counts = (younger_group["Epitope_gene"].value_counts(normalize=True)
                  .rename("Younger (normalized)"))
older_counts = (older_group["Epitope_gene"].value_counts(normalize=True)
                .rename("Older (normalized)"))

# Combine
comparison_df = pd.concat([younger_counts, older_counts], axis=1).fillna(0)

# Plot
plt.figure(figsize=(10, 12))
comparison_df.plot(kind='barh', figsize=(5, 8), stacked=False)
plt.xlabel("Proportion")
plt.title("Normalized Epitope_gene Frequencies in Younger vs Older-biased TCRs")
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load your dataset
df = pd.read_csv("merged_overlap_tcrs_wasserstein.csv")

# Drop rows with missing species/gene
df = df.dropna(subset=['Epitope_species', 'Epitope_gene'])

# Normalize species labels
species_map = {
    'DENV1': 'DENV',
    'DENV3/4': 'DENV',
    'HIV-1': 'HIV',
    'Homo Sapiens': None,
    'HomoSapiens': None,
    'SARS-CoV': 'SARS',
    'SARS-CoV-2': 'SARS'
}
df['Epitope_species'] = df['Epitope_species'].replace(species_map)
df = df[df['Epitope_species'].notna()]

# Assign groups based on z-score
df['Group'] = pd.NA
df.loc[df['component_zscore'] < -1.96, 'Group'] = 'Younger'
df.loc[df['component_zscore'] > 1.96, 'Group'] = 'Older'
df = df[df['Group'].notna()]  # keep only defined groups

# Get unique species
species_list = df['Epitope_species'].unique()

# Plot per species (normalized proportions)
for species in species_list:
    subset = df[df['Epitope_species'] == species]
    gene_counts = subset.groupby(['Epitope_gene', 'Group']).size().unstack(fill_value=0)

    # Normalize each group column to proportion
    gene_props = gene_counts.div(gene_counts.sum(axis=0), axis=1)

    gene_props.plot(
        kind='barh',
        figsize=(8, max(4, 0.4 * len(gene_props))),
        title=f"{species} Epitope_gene (Normalized) in Younger vs Older-biased TCRs"
    )
    plt.xlabel("Proportion")
    plt.ylabel("Epitope_gene")
    plt.grid(True)
    plt.tight_layout()
    plt.show()


In [ ]:
import pandas as pd

# Load the Excel file
file_path = 'Matched_File_Data.xlsx'  # Path to your file
df = pd.read_excel(file_path)

# Divide the 'age' column into 5 groups (quintiles) and include the ranges in output
age_groups, bins = pd.qcut(df['Age'], q=5, labels=[1, 2, 3, 4, 5], retbins=True)

# Print the ranges (bins) for each age group
for i in range(len(bins) - 1):
    print(f"Group {i + 1}: {bins[i]} to {bins[i + 1]}")

# Save the groups into the dataframe
df['age_group'] = age_groups


print("Age groups created and file saved.")


In [ ]:
import pandas as pd

# Step 1: Load the Excel file containing the full dataset to calculate bin ranges
file_path = 'Matched_File_Data.xlsx'  # Path to your file
df = pd.read_excel(file_path)

# Step 2: Divide the 'age' column into 10 groups (deciles) and get the bin ranges
age_groups, bins = pd.qcut(df['Age'], q=10, labels=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10], retbins=True)

# Print the calculated bin ranges
for i in range(len(bins) - 1):
    print(f"Group {i + 1}: {bins[i]} to {bins[i + 1]}")

# Step 3: Load the entire TCR-Ages list CSV file (no row limit)
tcr_age_file_path = 'tcr_age_lists.csv'
df_tcr = pd.read_csv(tcr_age_file_path)  # Loading the full dataset

# Step 4: Function to calculate the percentage of ages in each decile based on the new bins
def calculate_age_group_percentages(ages_list, bins):
    ages_list = eval(ages_list)  # Convert string representation of list to actual list
    total_count = len(ages_list)

    # Count how many ages fall into each bin using the calculated bins
    counts_per_bin = [0] * (len(bins) - 1)  # Initialize with zeros for each bin

    for age in ages_list:
        for i in range(len(bins) - 1):
            if bins[i] <= age < bins[i + 1]:
                counts_per_bin[i] += 1
                break
            # Special case for the last bin (include the upper boundary)
            elif i == len(bins) - 2 and bins[i + 1] <= age <= bins[i + 1]:
                counts_per_bin[i] += 1

    # Convert counts to percentages
    percentages_per_bin = [(count / total_count) * 100 for count in counts_per_bin]

    # Ensure the percentages sum exactly to 100% by verifying the sum
    total_percentage = sum(percentages_per_bin)

    # Round small floating-point errors (e.g., 99.999% or 100.001%)
    if abs(total_percentage - 100) > 1e-6:
        print(f"Error: Percentages sum to {total_percentage}% for ages {ages_list}")

    return percentages_per_bin + [total_count]

# Step 5: Apply the function to all rows and create new columns for each decile (10 bins)
df_tcr[['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', 'n_elements']] = df_tcr['Ages'].apply(lambda ages: pd.Series(calculate_age_group_percentages(ages, bins)))

# Step 6: Add a new column to sum the percentages (just for verification)
df_tcr['sum_percentages'] = df_tcr[['1', '2', '3', '4', '5', '6', '7', '8', '9', '10']].sum(axis=1)

# Step 7: Save the updated DataFrame with the sum of percentages to a CSV file
df_tcr.to_csv('updated_tcr_age_lists_full_with_deciles.csv', index=False)

print("CSV with correct decile percentages saved as 'updated_tcr_age_lists_full_with_deciles.csv'.")


In [ ]:
df_tcr_sampled.to_csv('updated_tcr_age_lists_sampled_with_sum.csv', index=False)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Step 1: Load the updated CSV file with deciles
df_tcr = pd.read_csv('updated_tcr_age_lists_full_with_deciles.csv')

# Step 2: Calculate the average for each decile column (1 to 10)
averages = df_tcr[['1', '2', '3', '4', '5', '6', '7', '8', '9', '10']].mean()

# Step 3: Select the top 50 and top 500 rows based on the highest '1' and '10' decile values
top_50_highest_1 = df_tcr.nlargest(50, '1')
top_500_highest_1 = df_tcr.nlargest(500, '1')

top_50_highest_10 = df_tcr.nlargest(50, '10')
top_500_highest_10 = df_tcr.nlargest(500, '10')

10, 6))
plt.fill_between(top_500_highest_10_averages.index, top_500_highest_10_averages.values, color='lightyellow', alpha=0.5, label='Top 500 Highest "10" Decile')
plt.plot(top_500_high# Step 4: Create an area plot of the average percentages across the deciles
plt.figure(figsize=(10, 6))

# Area plot for the average across all rows
plt.fill_between(averages.index, averages.values, color='skyblue', alpha=0.5, label='Average % in Age Groups')
plt.plot(averages.index, averages.values, color='blue', label='Average % Line')

plt.title('Average Percentage of Ages in Each Decile Group')
plt.xlabel('Age Group (Deciles)')
plt.ylabel('Average Percentage (%)')
plt.legend()
plt.grid(True)
plt.show()

# Step 5: Plot the average of top 50 rows with the highest '1' decile value
top_50_highest_1_averages = top_50_highest_1[['1', '2', '3', '4', '5', '6', '7', '8', '9', '10']].mean()

plt.figure(figsize=(10, 6))
plt.fill_between(top_50_highest_1_averages.index, top_50_highest_1_averages.values, color='lightgreen', alpha=0.5, label='Top 50 Highest "1" Decile')
plt.plot(top_50_highest_1_averages.index, top_50_highest_1_averages.values, color='green', label='Top 50 Highest "1" Decile Line')
plt.title('Average Percentage of Ages (Top 50 Highest "1" Decile)')
plt.xlabel('Age Group (Deciles)')
plt.ylabel('Average Percentage (%)')
plt.legend()
plt.grid(True)
plt.show()

# Step 6: Plot the average of top 500 rows with the highest '1' decile value
top_500_highest_1_averages = top_500_highest_1[['1', '2', '3', '4', '5', '6', '7', '8', '9', '10']].mean()

plt.figure(figsize=(10, 6))
plt.fill_between(top_500_highest_1_averages.index, top_500_highest_1_averages.values, color='lightcoral', alpha=0.5, label='Top 500 Highest "1" Decile')
plt.plot(top_500_highest_1_averages.index, top_500_highest_1_averages.values, color='red', label='Top 500 Highest "1" Decile Line')
plt.title('Average Percentage of Ages (Top 500 Highest "1" Decile)')
plt.xlabel('Age Group (Deciles)')
plt.ylabel('Average Percentage (%)')
plt.legend()
plt.grid(True)
plt.show()

# Step 7: Plot the average of top 50 rows with the highest '10' decile value
top_50_highest_10_averages = top_50_highest_10[['1', '2', '3', '4', '5', '6', '7', '8', '9', '10']].mean()

plt.figure(figsize=(10, 6))
plt.fill_between(top_50_highest_10_averages.index, top_50_highest_10_averages.values, color='lightblue', alpha=0.5, label='Top 50 Highest "10" Decile')
plt.plot(top_50_highest_10_averages.index, top_50_highest_10_averages.values, color='blue', label='Top 50 Highest "10" Decile Line')
plt.title('Average Percentage of Ages (Top 50 Highest "10" Decile)')
plt.xlabel('Age Group (Deciles)')
plt.ylabel('Average Percentage (%)')
plt.legend()
plt.grid(True)
plt.show()

# Step 8: Plot the average of top 500 rows with the highest '10' decile value
top_500_highest_10_averages = top_500_highest_10[['1', '2', '3', '4', '5', '6', '7', '8', '9', '10']].mean()

plt.figure(figsize=(est_10_averages.index, top_500_highest_10_averages.values, color='orange', label='Top 500 Highest "10" Decile Line')
plt.title('Average Percentage of Ages (Top 500 Highest "10" Decile)')
plt.xlabel('Age Group (Deciles)')
plt.ylabel('Average Percentage (%)')
plt.legend()
plt.grid(True)
plt.show()

# Step 9: Additional plot ideas

# Plot the distribution of '1' decile (youngest group) across all rows
plt.figure(figsize=(10, 6))
plt.hist(df_tcr['1'], bins=20, color='lightgreen', alpha=0.7, edgecolor='black')
plt.title('Distribution of "1" Decile (Youngest Group)')
plt.xlabel('Percentage in "1" Decile')
plt.ylabel('Frequency')
plt.grid(True)
plt.show()

# Plot the distribution of '10' decile (oldest group) across all rows
plt.figure(figsize=(10, 6))
plt.hist(df_tcr['10'], bins=20, color='lightblue', alpha=0.7, edgecolor='black')
plt.title('Distribution of "10" Decile (Oldest Group)')
plt.xlabel('Percentage in "10" Decile')
plt.ylabel('Frequency')
plt.grid(True)
plt.show()

# Scatter plot of '1' vs '10' decile to see the correlation between youngest and oldest group percentages
plt.figure(figsize=(10, 6))
plt.scatter(df_tcr['1'], df_tcr['10'], alpha=0.5, color='purple')
plt.title('Scatter Plot of "1" Decile (Youngest) vs "10" Decile (Oldest)')
plt.xlabel('Percentage in "1" Decile (Youngest)')
plt.ylabel('Percentage in "10" Decile (Oldest)')
plt.grid(True)
plt.show()

# Correlation heatmap for all decile columns
import seaborn as sns

plt.figure(figsize=(12, 8))
sns.heatmap(df_tcr[['1', '2', '3', '4', '5', '6', '7', '8', '9', '10']].corr(), annot=True, cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Heatmap of Decile Percentages')
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Step 1: Load the updated CSV file with deciles
df_tcr = pd.read_csv('updated_tcr_age_lists_full_with_deciles.csv')

# Function to calculate the TCR score based on the percentage distribution across deciles
def calculate_tcr_score(row):
    # Weights corresponding to deciles 1 to 10
    weights = list(range(1, 11))  # [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

    # Extract percentages for deciles 1 to 10
    percentages = row[['1', '2', '3', '4', '5', '6', '7', '8', '9', '10']].values

    # Calculate the weighted sum (percentage * weight)
    score = sum(percentages * weights)

    return score

# Step 8: Apply the function to calculate the score for each TCR
df_tcr['tcr_score'] = df_tcr.apply(calculate_tcr_score, axis=1)

# Step 9: Save the updated DataFrame with the TCR scores to a CSV file
output_file_path = 'updated_tcr_age_lists_with_scores.csv'
df_tcr.to_csv(output_file_path, index=False)

print(f"CSV with TCR scores saved as '{output_file_path}'.")


In [ ]:
df_tcr

In [ ]:
# Step 1: Sort the dataframe by TCR scores
df_tcr_sorted = df_tcr.sort_values(by='tcr_score', ascending=False)

# Step 2: Select the top 50 and bottom 50 rows based on TCR scores
top_1000_highest_scores = df_tcr_sorted.head(1000)
bottom_1000_lowest_scores = df_tcr_sorted.tail(1000)

newwwwwwwwwwwwwwwwwwwwwwwwwwwwwwwwwwwwwwwwwwwwwwwww

> ⛔ **DEAD END — לא בשימוש. אל תריץ.**
>
> התא הבא הוא מתכון חלופי ל‑`merged_overlap_tcrs_wasserstein.csv` ש**מעולם לא ייצר**
> את הקובץ שעל הדיסק. המתכון האמיתי הוא cells 183 → 186 → 187 → 190.
>
> ראיה: מתכון זה ממזג עם `left_on='TCR', right_on='CDR3b'` ולכן היה מייצר עמודת
> `CDR3b`, שאינה קיימת בקובץ בפועל. בנוסף הוא קורא את הגרסה בת 9 העמודות של
> `significant_tcrs_signed_wasserstein.csv`, בעוד שלקובץ בפועל 26 עמודות בצד השמאלי.
>
> ---
>
> **DEAD END — NOT USED. DO NOT RUN.**
>
> The cell below is an alternative recipe for `merged_overlap_tcrs_wasserstein.csv` that
> never produced the file on disk. The real chain is cells 183 → 186 → 187 → 190.
> This recipe merges with `right_on='CDR3b'` and would therefore emit a `CDR3b` column,
> which the actual file does not have; it also reads the 9-column build of
> `significant_tcrs_signed_wasserstein.csv` whereas the real file carries 26 left-side columns.


In [ ]:
import pandas as pd

# Load your data
tcr_df = pd.read_csv("significant_tcrs_signed_wasserstein.csv")
vdjdb_df = pd.read_csv("vdjdb_trait_onlyB.csv")

# Merge datasets on matching TCRs (assuming 'TCR' column in your file matches 'cdr3b' in VDJdb)
merged_df = pd.merge(tcr_df, vdjdb_df, left_on='TCR', right_on='CDR3b', how='inner')

# Save the merged results
merged_df.to_csv("merged_overlap_tcrs_wasserstein.csv", index=False)


In [ ]:
merged_df

In [ ]:
merged_df

In [ ]:
merged_df.to_csv("merged_overlap_tcrs.csv", index=False)


In [ ]:
import pandas as pd

# Load your file
df = pd.read_csv("merged_overlap_tcrs.csv")

# Sort dataframe by 'tcr_score'
sorted_df = df.sort_values(by='tcr_score', ascending=False)

# Select top 250 and bottom 250 rows
top250 = sorted_df.head(250)
bottom250 = sorted_df.tail(250)

# Concatenate top and bottom rows into a single DataFrame
combined_df = pd.concat([top250, bottom250])

# Sort the combined DataFrame again by 'tcr_score'
final_sorted_df = combined_df.sort_values(by='tcr_score', ascending=False)

# Save to CSV
final_sorted_df.to_csv("top_bottom_250_sorted_by_score.csv", index=False)


In [ ]:
import pandas as pd

# Load your file
df = pd.read_csv("merged_overlap_tcrs.csv")

# Sort dataframe by 'tcr_score'
sorted_df = df.sort_values(by='tcr_score', ascending=False)

# Select top 250 and bottom 250 rows
top250 = sorted_df.head(500)
bottom250 = sorted_df.tail(500)

# Concatenate top and bottom rows into a single DataFrame
combined_df = pd.concat([top250, bottom250])

# Sort the combined DataFrame again by 'tcr_score'
final_sorted_df = combined_df.sort_values(by='tcr_score', ascending=False)

# Save to CSV
final_sorted_df.to_csv("top_bottom_500_sorted_by_score.csv", index=False)


In [ ]:
import pandas as pd

# Load your file
df = pd.read_csv("merged_overlap_tcrs.csv")

# Sort dataframe by 'tcr_score'
sorted_df = df.sort_values(by='tcr_score', ascending=False)

# Select top 250 and bottom 250 rows
top250 = sorted_df.head(500)
bottom250 = sorted_df.tail(500)

# Concatenate top and bottom rows into a single DataFrame
combined_df = pd.concat([top250, bottom250])

# Sort the combined DataFrame again by 'tcr_score'
final_sorted_df = combined_df.sort_values(by='tcr_score', ascending=False)

# Save to CSV
final_sorted_df.to_csv("top_bottom_500_sorted_by_score.csv", index=False)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load the data
df = pd.read_csv("top_bottom_250_sorted_by_score.csv")

# Split into top and bottom 250
top_group = df.head(250)
bottom_group = df.tail(250)

# Count Epitope_species occurrences
top_counts = top_group['Epitope_species'].value_counts().rename("Top 250")
bottom_counts = bottom_group['Epitope_species'].value_counts().rename("Bottom 250")

# Combine counts
comparison_df = pd.concat([top_counts, bottom_counts], axis=1).fillna(0).astype(int)

# Plot
plt.figure(figsize=(10, 12))
comparison_df.plot(kind='barh', stacked=False)
plt.xlabel("Count")
plt.title("Epitope_species Counts in Top and Bottom 250 TCRs")
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load the data
df = pd.read_csv("top_bottom_500_sorted_by_score.csv")

# Split into top and bottom 250
top_group = df.head(500)
bottom_group = df.tail(500)

# Count Epitope_species occurrences
top_counts = top_group['Epitope_species'].value_counts().rename("Top 500")
bottom_counts = bottom_group['Epitope_species'].value_counts().rename("Bottom 500")

# Combine counts
comparison_df = pd.concat([top_counts, bottom_counts], axis=1).fillna(0).astype(int)

# Plot
plt.figure(figsize=(10, 12))
comparison_df.plot(kind='barh', stacked=False)
plt.xlabel("Count")
plt.title("Epitope_species Counts in Top and Bottom 500 TCRs")
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load the data
df = pd.read_csv("top_bottom_250_sorted_by_score.csv")

# Split into top and bottom 250
top_group = df.head(250)
bottom_group = df.tail(250)

# Count Epitope_species occurrences
top_counts = top_group['Epitope_gene'].value_counts().rename("Top 250")
bottom_counts = bottom_group['Epitope_gene'].value_counts().rename("Bottom 250")

# Combine counts
comparison_df = pd.concat([top_counts, bottom_counts], axis=1).fillna(0).astype(int)

# Plot
plt.figure(figsize=(10, 12))
comparison_df.plot(kind='barh', figsize=(5, 8), stacked=False)
plt.xlabel("Count")
plt.title("Epitope_gene Counts in Top and Bottom 250 TCRs")
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load the data
df = pd.read_csv("top_bottom_250_sorted_by_score.csv")

# Split into top and bottom 250
top_group = df.head(250)
bottom_group = df.tail(250)

# Count Epitope_species occurrences
top_counts = top_group['Epitope_species'].value_counts().rename("Top 250")
bottom_counts = bottom_group['Epitope_species'].value_counts().rename("Bottom 250")

# Combine counts
comparison_df = pd.concat([top_counts, bottom_counts], axis=1).fillna(0).astype(int)

# Plot with log scale
plt.figure(figsize=(10, 12))
ax = comparison_df.plot(kind='barh', stacked=False)
ax.set_xscale('log')
plt.xlabel("Log-scaled Count")
plt.title("Epitope_species Counts in Top and Bottom 250 TCRs (Log Scale)")
plt.grid(True, which='both', axis='x')
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import plotly.graph_objects as go

# Load the data
df = pd.read_csv("top_bottom_250_sorted_by_score.csv")

# Create group column
df['Group'] = ['Top 250'] * 250 + ['Bottom 250'] * 250

# Drop missing entries
df = df.dropna(subset=['Epitope_species', 'Epitope_gene'])

# Function to create Sankey plot
def create_sankey(df_subset, title):
    # Group and count source-target links
    links = df_subset.groupby(['Epitope_species', 'Epitope_gene']).size().reset_index(name='Count')

    # Create label list
    species = links['Epitope_species'].unique().tolist()
    genes = links['Epitope_gene'].unique().tolist()
    labels = species + genes

    # Map labels to indices
    label_to_index = {label: i for i, label in enumerate(labels)}
    links['source'] = links['Epitope_species'].map(label_to_index)
    links['target'] = links['Epitope_gene'].map(lambda x: label_to_index[x] + len(species))  # Offset

    # Create Sankey diagram
    fig = go.Figure(data=[go.Sankey(
        node=dict(
            pad=15,
            thickness=20,
            line=dict(color="black", width=0.5),
            label=labels
        ),
        link=dict(
            source=links['source'],
            target=links['target'],
            value=links['Count']
        )
    )])

    fig.update_layout(
        title_text=title,
        font_size=12,
        width=800,
        height=900
    )
    fig.show()

# Create and show plots
create_sankey(df[df['Group'] == 'Top 250'], "Epitope_species to Epitope_gene (Top 250)")
create_sankey(df[df['Group'] == 'Bottom 250'], "Epitope_species to Epitope_gene (Bottom 250)")


In [ ]:
import pandas as pd
import plotly.graph_objects as go

# Load the data
df = pd.read_csv("top_bottom_250_sorted_by_score.csv")

# Create 'Group' column
df['Group'] = ['Top 250'] * 250 + ['Bottom 250'] * 250

# Drop rows with missing values
df = df.dropna(subset=['Epitope_species', 'Epitope_gene'])

# ========================
# Clean Epitope_species values
# ========================
def clean_species(x):
    x = x.strip()
    if x in ["DENV1", "DENV3/4", "DENV"]:
        return "DENV"
    elif x in ["HIV", "HIV-1"]:
        return "HIV"
    elif x in ["SARS-CoV", "SARS-CoV-2"]:
        return "SARS-CoV-2"
    elif x in ["Homo Sapiens", "HomoSapiens"]:
        return None
    else:
        return x

df['Epitope_species'] = df['Epitope_species'].apply(clean_species)

# Drop rows with removed species
df = df.dropna(subset=['Epitope_species'])

# ========================
# Sankey Function
# ========================
def create_sankey(df_subset, title):
    # Group and count source-target links
    links = df_subset.groupby(['Epitope_species', 'Epitope_gene']).size().reset_index(name='Count')

    # Create label list
    species = links['Epitope_species'].unique().tolist()
    genes = links['Epitope_gene'].unique().tolist()
    labels = species + genes

    # Map labels to indices
    label_to_index = {label: i for i, label in enumerate(labels)}
    links['source'] = links['Epitope_species'].map(label_to_index)
    links['target'] = links['Epitope_gene'].map(lambda x: label_to_index[x] + len(species))  # Offset

    # Create Sankey diagram
    fig = go.Figure(data=[go.Sankey(
        node=dict(
            pad=15,
            thickness=20,
            line=dict(color="black", width=0.5),
            label=labels
        ),
        link=dict(
            source=links['source'],
            target=links['target'],
            value=links['Count']
        )
    )])

    fig.update_layout(
        title_text=title,
        font_size=12,
        width=1000,
        height=900
    )
    fig.show()

# ========================
# Plot the two groups
# ========================
create_sankey(df[df['Group'] == 'Top 250'], "Epitope_species to Epitope_gene (Top 250)")
create_sankey(df[df['Group'] == 'Bottom 250'], "Epitope_species to Epitope_gene (Bottom 250)")


In [ ]:
import pandas as pd
import plotly.graph_objects as go

# Load the cleaned data
df = pd.read_csv("top_bottom_250_sorted_by_score.csv")
df['Group'] = ['Top 250'] * 250 + ['Bottom 250'] * 250

# Drop missing entries
df = df.dropna(subset=['Epitope_species', 'Epitope_gene'])

# Standardize Epitope_species
df['Epitope_species'] = df['Epitope_species'].replace({
    'HIV-1': 'HIV',
    'Homo Sapiens': None,
    'HomoSapiens': None,
    'SARS-CoV': 'SARS-CoV-2',
    'DENV1': 'DENV',
    'DENV3/4': 'DENV',
    'Mtb': 'M.tuberculosis'
})

# Drop rows where species is None
df = df.dropna(subset=['Epitope_species'])

# Standardize Epitope_gene
df['Epitope_gene'] = df['Epitope_gene'].replace({
    'IE1': 'IE-1',
    'EBNA-3B': 'EBNA3B',
    'EBNA-3A': 'EBNA3A',
    'EMNA-3A': 'EBNA3A',  # Typo fix
    'Flu-MP': 'M1',
    'EBNA4': 'EBNA3B'
})

# Function to create Sankey plot
def create_sankey(df_subset, title):
    links = df_subset.groupby(['Epitope_species', 'Epitope_gene']).size().reset_index(name='Count')

    # Create label list
    species = links['Epitope_species'].unique().tolist()
    genes = links['Epitope_gene'].unique().tolist()
    labels = species + genes

    # Map to indices
    label_to_index = {label: i for i, label in enumerate(labels)}
    links['source'] = links['Epitope_species'].map(label_to_index)
    links['target'] = links['Epitope_gene'].map(lambda x: label_to_index[x])

    # Sankey plot
    fig = go.Figure(data=[go.Sankey(
        node=dict(
            pad=15,
            thickness=20,
            line=dict(color="black", width=0.5),
            label=labels
        ),
        link=dict(
            source=links['source'],
            target=links['target'],
            value=links['Count']
        )
    )])

    fig.update_layout(
        title_text=title,
        font_size=12,
        width=1100,
        height=900
    )
    fig.show()

# Plot for Top 250
create_sankey(df[df['Group'] == 'Top 250'], "Epitope_species → Epitope_gene (Top 250)")

# Plot for Bottom 250
create_sankey(df[df['Group'] == 'Bottom 250'], "Epitope_species → Epitope_gene (Bottom 250)")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load the cleaned file (re-uploaded earlier)
df = pd.read_csv("top_bottom_250_sorted_by_score.csv")

# Assign group labels
df['Group'] = ['Top 250'] * 250 + ['Bottom 250'] * 250

# Drop missing values
df = df.dropna(subset=['Epitope_species', 'Epitope_gene'])

# Normalize known label variations
species_map = {
    'DENV1': 'DENV',
    'DENV3/4': 'DENV',
    'HIV-1': 'HIV',
    'Homo Sapiens': None,
    'HomoSapiens': None,
    'SARS-CoV': 'SARS',
    'SARS-CoV-2': 'SARS'
}
df['Epitope_species'] = df['Epitope_species'].replace(species_map)
df = df[df['Epitope_species'].notna()]

# Get all unique species
species_list = df['Epitope_species'].unique()

# Generate plots per species
for species in species_list:
    subset = df[df['Epitope_species'] == species]
    gene_counts = subset.groupby(['Epitope_gene', 'Group']).size().unstack(fill_value=0)

    gene_counts.plot(kind='barh', figsize=(8, max(4, 0.4 * len(gene_counts))), title=f"{species} Epitope_gene Counts in Top vs Bottom 250")
    plt.xlabel("Count")
    plt.ylabel("Epitope_gene")
    plt.grid(True)
    plt.tight_layout()
    plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load the cleaned file (re-uploaded earlier)
df = pd.read_csv("top_bottom_500_sorted_by_score.csv")

# Assign group labels
df['Group'] = ['Top 500'] * 500 + ['Bottom 500'] * 500

# Drop missing values
df = df.dropna(subset=['Epitope_species', 'Epitope_gene'])

# Normalize known label variations
species_map = {
    'DENV1': 'DENV',
    'DENV3/4': 'DENV',
    'HIV-1': 'HIV',
    'Homo Sapiens': None,
    'HomoSapiens': None,
    'SARS-CoV': 'SARS',
    'SARS-CoV-2': 'SARS'
}
df['Epitope_species'] = df['Epitope_species'].replace(species_map)
df = df[df['Epitope_species'].notna()]

# Get all unique species
species_list = df['Epitope_species'].unique()

# Generate plots per species
for species in species_list:
    subset = df[df['Epitope_species'] == species]
    gene_counts = subset.groupby(['Epitope_gene', 'Group']).size().unstack(fill_value=0)

    gene_counts.plot(kind='barh', figsize=(8, max(4, 0.4 * len(gene_counts))), title=f"{species} Epitope_gene Counts in Top vs Bottom 500")
    plt.xlabel("Count")
    plt.ylabel("Epitope_gene")
    plt.grid(True)
    plt.tight_layout()
    plt.show()


In [ ]:
pip install logomaker


In [ ]:
import pandas as pd
import logomaker
import matplotlib.pyplot as plt

# Load your CSV file
df = pd.read_csv("top_bottom_250_sorted_by_score.csv")

# Extract top and bottom 250 Epitope sequences
top_250 = df.head(250)['Epitope'].dropna().astype(str).tolist()
bottom_250 = df.tail(250)['Epitope'].dropna().astype(str).tolist()

# Pad sequences to the same length
max_len = max(max(len(seq) for seq in top_250), max(len(seq) for seq in bottom_250))
top_padded = [seq.ljust(max_len, '-') for seq in top_250]
bottom_padded = [seq.ljust(max_len, '-') for seq in bottom_250]

# Function to plot logo
def plot_logo(sequences, title):
    counts_df = logomaker.alignment_to_matrix(sequences)
    plt.figure(figsize=(max_len, 3))
    logomaker.Logo(counts_df, color_scheme='classic')
    plt.title(title)
    plt.xlabel("Position")
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()

# Plot sequence logos
plot_logo(top_padded, "Epitope Logo - Top 250 TCRs")
plot_logo(bottom_padded, "Epitope Logo - Bottom 250 TCRs")


In [ ]:
import pandas as pd

# Load both CSV files
top5k_df = pd.read_csv("top5k_tcrs_by_score.csv")
bottom5k_df = pd.read_csv("bottom5k_tcrs_by_score.csv")

# Extract sequences
top_seqs = top5k_df['TCR'].dropna().unique().tolist()
bottom_seqs = bottom5k_df['TCR'].dropna().unique().tolist()

# Combine and label
combined_df = pd.DataFrame({
    'TCR': top_seqs + bottom_seqs,
    'group': ['top'] * len(top_seqs) + ['bottom'] * len(bottom_seqs)
})

# Remove duplicates, keep the first appearance
combined_df = combined_df.drop_duplicates(subset='TCR')

# Save
combined_df.to_csv("tcrbert_input_top_bottom.csv", index=False)


In [ ]:
pip install torch transformers logomaker


In [ ]:
# Step 1: Clone the repository
!git clone https://github.com/wukevin/tcr-bert.git
%cd tcr-bert

# Step 2: Install required packages (if needed)
!pip install transformers
!pip install logomaker


In [ ]:
from transformers import BertModel, BertTokenizer
import torch
import pandas as pd

# Load pretrained model & tokenizer
model_name = "wukevin/tcr-bert"
tokenizer = BertTokenizer.from_pretrained(model_name, do_lower_case=False)
model = BertModel.from_pretrained(model_name)
model.eval()

# Utility to embed a single CDR3
def embed_cdr3(seq):
    tokens = seq.split()  # expects spaced input like "C A S S P ..."
    tokens = ["[CLS]"] + tokens + ["[SEP]"]
    input_ids = torch.tensor([tokenizer.convert_tokens_to_ids(tokens)])
    mask = torch.ones_like(input_ids)
    with torch.no_grad():
        outputs = model(input_ids, attention_mask=mask)
    # Use the CLS token embedding as sequence embedding
    return outputs.last_hidden_state[0][0].numpy()

# Load your TCR list
df = pd.read_csv("tcrbert_input_top_bottom.csv")
embeddings = df['TCR'].apply(lambda seq: embed_cdr3(" ".join(seq)))

# Expand to DataFrame
embed_df = pd.DataFrame(embeddings.tolist(), columns=[f"dim{i}" for i in range(embeddings.iloc[0].shape[0])])
embed_df['TCR'] = df['TCR']
embed_df['group'] = df['group']

# Save embeddings
embed_df.to_csv("tcrbert_embeddings.csv", index=False)
print("✅ Embeddings generated and saved as tcrbert_embeddings.csv")


In [ ]:
import pandas as pd
from sklearn.decomposition import PCA
import umap
import matplotlib.pyplot as plt
import seaborn as sns

# Load embeddings
df = pd.read_csv("tcrbert_embeddings.csv")

# Select embedding dimensions
embedding_cols = [col for col in df.columns if col.startswith("dim")]
X = df[embedding_cols]
y = df['group']

# ---------- PCA ----------
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

plt.figure(figsize=(8, 6))
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=y, palette="Set2", s=60, alpha=0.8)
plt.title("PCA of TCRBERT Embeddings")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend(title="Group")
plt.tight_layout()
plt.show()

# ---------- UMAP ----------
reducer = umap.UMAP(n_components=2, random_state=42)
X_umap = reducer.fit_transform(X)

plt.figure(figsize=(8, 6))
sns.scatterplot(x=X_umap[:, 0], y=X_umap[:, 1], hue=y, palette="Set2", s=60, alpha=0.8)
plt.title("UMAP of TCRBERT Embeddings")
plt.xlabel("UMAP1")
plt.ylabel("UMAP2")
plt.legend(title="Group")
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import umap
import matplotlib.pyplot as plt
import seaborn as sns

# Load the top and bottom files
top_df = pd.read_csv("top5k_tcrs_by_score.csv")[['TCR', 'tcr_score']]
bottom_df = pd.read_csv("bottom5k_tcrs_by_score.csv")[['TCR', 'tcr_score']]

# Label the groups
top_df['group'] = 'top'
bottom_df['group'] = 'bottom'

# Merge and sort by score
merged_df = pd.concat([top_df, bottom_df])
merged_df.sort_values(by='tcr_score', ascending=False, inplace=True)

# Load the embeddings
embed_df = pd.read_csv("tcrbert_embeddings.csv")

# Merge embeddings with labels
full_df = pd.merge(embed_df, merged_df, on='TCR')

# Subset sizes
subset_sizes = [10, 100, 500, 1000]
embedding_cols = [col for col in embed_df.columns if col.startswith("dim")]

# Create plots
for size in subset_sizes:
    subset_top = full_df[full_df['group_y'] == 'top'].head(size)
    subset_bottom = full_df[full_df['group_y'] == 'bottom'].head(size)
    subset = pd.concat([subset_top, subset_bottom])
    X = subset[embedding_cols]
    y = subset['group_y']

    # PCA
    pca = PCA(n_components=2)
    X_pca = pca.fit_transform(X)
    plt.figure(figsize=(6, 5))
    sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=y)
    plt.title(f"PCA - Top & Bottom {size}")
    plt.tight_layout()
    plt.show()

    # UMAP
    reducer = umap.UMAP(n_components=2, random_state=42)
    X_umap = reducer.fit_transform(X)
    plt.figure(figsize=(6, 5))
    sns.scatterplot(x=X_umap[:, 0], y=X_umap[:, 1], hue=y)
    plt.title(f"UMAP - Top & Bottom {size}")
    plt.tight_layout()
    plt.show()

    # TSNE
    perplexity = min(30, max(5, len(X) // 2))
    tsne = TSNE(n_components=2, random_state=42, perplexity=perplexity)
    X_tsne = tsne.fit_transform(X)

    plt.figure(figsize=(6, 5))
    sns.scatterplot(x=X_tsne[:, 0], y=X_tsne[:, 1], hue=y)
    plt.title(f"t-SNE - Top & Bottom {size}")
    plt.tight_layout()
    plt.show()


In [ ]:
df

In [ ]:
# Step 1: Install ESM
!pip install fair-esm

# Step 2: Imports
import torch
import pandas as pd
from esm import pretrained
from tqdm.notebook import tqdm

# Step 3: Load sequences
top5k = pd.read_csv("top5k_tcrs_by_score.csv")
bottom5k = pd.read_csv("bottom5k_tcrs_by_score.csv")

# Combine and label
top5k["group"] = "top"
bottom5k["group"] = "bottom"
df = pd.concat([top5k, bottom5k], ignore_index=True)
df = df.dropna(subset=["TCR"])
sequences = df["TCR"].tolist()

# Step 4: Load ESM model
model, alphabet = pretrained.esm1b_t33_650M_UR50S()
batch_converter = alphabet.get_batch_converter()
model.eval().cuda()  # Move to GPU

# Step 5: Prepare batches
batch_size = 64
all_embeddings = []
all_labels = []

with torch.no_grad():
    for i in tqdm(range(0, len(sequences), batch_size)):
        batch_seqs = sequences[i:i+batch_size]
        batch_labels = [f"seq_{j}" for j in range(i, i + len(batch_seqs))]
        batch_data = list(zip(batch_labels, batch_seqs))

        _, _, batch_tokens = batch_converter(batch_data)
        batch_tokens = batch_tokens.cuda()

        results = model(batch_tokens, repr_layers=[33], return_contacts=False)
        token_reps = results["representations"][33]

        # Use mean of per-token representations (excluding padding, CLS, and EOS)
        for j, (label, seq) in enumerate(batch_data):
            tokens_len = len(seq)
            embedding = token_reps[j, 1:1+tokens_len].mean(0).cpu().numpy()
            all_embeddings.append(embedding)
            all_labels.append(label)

# Step 6: Create DataFrame with embeddings
embedding_df = pd.DataFrame(all_embeddings)
embedding_df["TCR"] = sequences
embedding_df["group"] = df["group"].values

# Step 7: Save to CSV
embedding_df.to_csv("esm_embeddings_top_bottom.csv", index=False)
print("✅ Saved: esm_embeddings_top_bottom.csv")


In [ ]:
# Step 1: Imports (after installation)
import torch
import pandas as pd
from esm import pretrained
from tqdm.notebook import tqdm

# Step 2: Load and clean sequences
top5k = pd.read_csv("top5k_tcrs_by_score.csv")
bottom5k = pd.read_csv("bottom5k_tcrs_by_score.csv")

# Remove sequences with '*'
top5k = top5k[~top5k['TCR'].str.contains(r'\*', na=False)].reset_index(drop=True)
bottom5k = bottom5k[~bottom5k['TCR'].str.contains(r'\*', na=False)].reset_index(drop=True)

# Label and combine
top5k["group"] = "top"
bottom5k["group"] = "bottom"
df = pd.concat([top5k, bottom5k], ignore_index=True)
df = df.dropna(subset=["TCR"])
sequences = df["TCR"].tolist()

# Step 3: Load ESM model
model, alphabet = pretrained.esm1b_t33_650M_UR50S()
batch_converter = alphabet.get_batch_converter()
model.eval().cuda()  # Move model to GPU

# Step 4: Generate embeddings
batch_size = 64
all_embeddings = []
all_labels = []

with torch.no_grad():
    for i in tqdm(range(0, len(sequences), batch_size)):
        batch_seqs = sequences[i:i + batch_size]
        batch_labels = [f"seq_{j}" for j in range(i, i + len(batch_seqs))]
        batch_data = list(zip(batch_labels, batch_seqs))

        try:
            _, _, batch_tokens = batch_converter(batch_data)
        except KeyError as e:
            print(f"Skipping batch {i} due to token error: {e}")
            continue

        batch_tokens = batch_tokens.cuda()
        results = model(batch_tokens, repr_layers=[33], return_contacts=False)
        token_reps = results["representations"][33]

        for j, (label, seq) in enumerate(batch_data):
            tokens_len = len(seq)
            embedding = token_reps[j, 1:1 + tokens_len].mean(0).cpu().numpy()
            all_embeddings.append(embedding)
            all_labels.append(label)

# Step 5: Create DataFrame with embeddings
embedding_df = pd.DataFrame(all_embeddings)
embedding_df["TCR"] = sequences[:len(embedding_df)]
embedding_df["group"] = df["group"].values[:len(embedding_df)]

# Step 6: Save
embedding_df.to_csv("esm_embeddings_top_bottom.csv", index=False)
print("✅ Saved: esm_embeddings_top_bottom.csv")


In [ ]:
embedding_df

In [ ]:
# Re-import required libraries after code execution state reset
import pandas as pd
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import umap
import matplotlib.pyplot as plt
import seaborn as sns

# Reload the file
file_path = "esm_embeddings_top_bottom.csv"
embedding_df = pd.read_csv(file_path)

# Extract features and labels
features = embedding_df.drop(columns=["TCR", "group"])
labels = embedding_df["group"]

# Apply PCA
pca = PCA(n_components=2, random_state=42)
pca_result = pca.fit_transform(features)

# Apply t-SNE
tsne = TSNE(n_components=2, random_state=42, perplexity=30)
tsne_result = tsne.fit_transform(features)

# Apply UMAP
umap_result = umap.UMAP(n_components=2, random_state=42).fit_transform(features)

# Create plots
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# PCA plot
sns.scatterplot(x=pca_result[:, 0], y=pca_result[:, 1], hue=labels, ax=axes[0])
axes[0].set_title("PCA")

# t-SNE plot
sns.scatterplot(x=tsne_result[:, 0], y=tsne_result[:, 1], hue=labels, ax=axes[1])
axes[1].set_title("t-SNE")

# UMAP plot
sns.scatterplot(x=umap_result[:, 0], y=umap_result[:, 1], hue=labels, ax=axes[2])
axes[2].set_title("UMAP")

plt.tight_layout()
plt.show()


In [ ]:
# 📦 Required libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# ✅ Step 1: Load the embeddings
df = pd.read_csv("esm_embeddings_top_bottom.csv")

# ✅ Step 2: Prepare features and labels
X = df.drop(columns=["TCR", "group"])
y = LabelEncoder().fit_transform(df["group"])  # 'bottom' = 0, 'top' = 1

# ✅ Step 3: Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# ✅ Step 4: Train XGBoost
model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
model.fit(X_train, y_train)

# ✅ Step 5: Predict and evaluate
y_pred = model.predict(X_test)

# 🧾 Classification report
print(classification_report(y_test, y_pred, target_names=["bottom", "top"]))

# 📊 Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["bottom", "top"], yticklabels=["bottom", "top"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Load full datasets
top5k = pd.read_csv("top5k_tcrs_by_score.csv")
bottom5k = pd.read_csv("bottom5k_tcrs_by_score.csv")

# Load ESM embeddings
embeddings = pd.read_csv("esm_embeddings_top_bottom.csv")

# Function to filter by top-N tcr_score and run XGBoost
def run_classification(N):
    top_N = top5k.head(N).copy()         # Top N highest scoring TCRs
    bottom_N = bottom5k.tail(N).copy()   # Bottom N lowest scoring TCRs

    # Add group labels
    top_N['group'] = 'top'
    bottom_N['group'] = 'bottom'

    # Combine
    df_filtered = pd.concat([top_N, bottom_N])

    # Filter embeddings based on TCRs present
    subset = embeddings[embeddings['TCR'].isin(df_filtered['TCR'])].copy()
    X = subset.drop(columns=['TCR', 'group'])
    y = LabelEncoder().fit_transform(subset['group'])

    # Train/test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

    # Train model
    model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Report
    print(f"\n🧪 Results for top {N} vs bottom {N} (total {2*N} samples):")
    print(classification_report(y_test, y_pred, target_names=["bottom", "top"]))

    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["bottom", "top"], yticklabels=["bottom", "top"])
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(f"Confusion Matrix - Top {N} vs Bottom {N}")
    plt.show()

# Run for different sizes
for N in [2500, 1000, 500]:
    run_classification(N)


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Load original TCR datasets
top5k = pd.read_csv("top5k_tcrs_by_score.csv")
bottom5k = pd.read_csv("bottom5k_tcrs_by_score.csv")

# All amino acids (standard 20) + placeholder for padding
amino_acids = list("ACDEFGHIKLMNPQRSTVWY")
aa_to_int = {aa: i for i, aa in enumerate(amino_acids)}
PAD_IDX = len(amino_acids)  # index for padding

def one_hot_encode_sequences(sequences, max_len=None):
    if max_len is None:
        max_len = max(len(seq) for seq in sequences)

    encoded = np.zeros((len(sequences), max_len, len(amino_acids) + 1), dtype=int)

    for i, seq in enumerate(sequences):
        for j, aa in enumerate(seq):
            encoded[i, j, aa_to_int.get(aa, PAD_IDX)] = 1
        for j in range(len(seq), max_len):  # padding
            encoded[i, j, PAD_IDX] = 1

    return encoded.reshape(len(sequences), -1)  # Flatten

# Main function
def run_classification_onehot(N):
    # Select top and bottom N
    top_N = top5k.head(N).copy()
    bottom_N = bottom5k.tail(N).copy()
    top_N['group'] = 'top'
    bottom_N['group'] = 'bottom'
    df_filtered = pd.concat([top_N, bottom_N])

    sequences = df_filtered['TCR'].tolist()
    labels = df_filtered['group'].tolist()

    # One-hot encode with padding
    X = one_hot_encode_sequences(sequences)
    y = LabelEncoder().fit_transform(labels)

    # Train/test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

    # Train model
    model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Report
    print(f"\n🧪 Results for top {N} vs bottom {N} (One-hot):")
    print(classification_report(y_test, y_pred, target_names=["bottom", "top"]))

    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["bottom", "top"], yticklabels=["bottom", "top"])
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(f"Confusion Matrix - One-hot (Top {N} vs Bottom {N})")
    plt.show()

# Run for different sizes
for N in [2500, 1000, 500]:
    run_classification_onehot(N)


In [ ]:
!pip install --upgrade sympy umap-learn --quiet


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import OneHotEncoder

# Load sequence data
top5k = pd.read_csv("top5k_tcrs_by_score.csv")
bottom5k = pd.read_csv("bottom5k_tcrs_by_score.csv")

# Add group labels
top5k["group"] = "top"
bottom5k["group"] = "bottom"

# Combine all
full_df = pd.concat([top5k, bottom5k], ignore_index=True)
full_df = full_df.dropna(subset=["TCR"])  # drop missing

# Function to one-hot encode sequences (padded)
def one_hot_encode_seqs(seqs, max_len=20):
    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    aa_list = list("ACDEFGHIKLMNPQRSTVWY") + ["X", "-"]  # '-' = padding
    encoder.fit(np.array(aa_list).reshape(-1, 1))

    padded = [list(seq.ljust(max_len, '-'))[:max_len] for seq in seqs]
    onehot = np.stack([encoder.transform(np.array(s).reshape(-1, 1)).flatten() for s in padded])
    return onehot

# Plotting function (PCA and t-SNE only)
def plot_dr(X, labels, title_prefix):
    pca = PCA(n_components=2, random_state=42).fit_transform(X)
    tsne = TSNE(n_components=2, random_state=42, perplexity=30).fit_transform(X)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, data, name in zip(axes, [pca, tsne], ["PCA", "t-SNE"]):
        sns.scatterplot(x=data[:, 0], y=data[:, 1], hue=labels, ax=ax, s=20)
        ax.set_title(f"{title_prefix} - {name}")
        ax.set_xticks([])
        ax.set_yticks([])
    plt.tight_layout()
    plt.show()

# Run for each group size
for N in [5000, 2500, 1000, 500]:
    top_N = top5k.head(N)
    bottom_N = bottom5k.tail(N)
    df_subset = pd.concat([top_N, bottom_N])
    feats = one_hot_encode_seqs(df_subset["TCR"])
    labels = df_subset["group"].values
    plot_dr(feats, labels, f"Top {N} vs Bottom {N}")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import OneHotEncoder

# Load sequence data
top5k = pd.read_csv("top5k_tcrs_by_score.csv")
bottom5k = pd.read_csv("bottom5k_tcrs_by_score.csv")

# Add group labels
top5k["group"] = "top"
bottom5k["group"] = "bottom"

# Combine all
full_df = pd.concat([top5k, bottom5k], ignore_index=True)
full_df = full_df.dropna(subset=["TCR"])  # drop missing

# Function to one-hot encode sequences (padded)
def one_hot_encode_seqs(seqs, max_len=20):
    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    aa_list = list("ACDEFGHIKLMNPQRSTVWY") + ["X", "-"]  # '-' = padding
    encoder.fit(np.array(aa_list).reshape(-1, 1))

    padded = [list(seq.ljust(max_len, '-'))[:max_len] for seq in seqs]
    onehot = np.stack([encoder.transform(np.array(s).reshape(-1, 1)).flatten() for s in padded])
    return onehot

# Plotting function (PCA and t-SNE only)
def plot_dr(X, labels, title_prefix):
    pca = PCA(n_components=2, random_state=42).fit_transform(X)
    tsne = TSNE(n_components=2, random_state=42, perplexity=30).fit_transform(X)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, data, name in zip(axes, [pca, tsne], ["PCA", "t-SNE"]):
        sns.scatterplot(x=data[:, 0], y=data[:, 1], hue=labels, ax=ax, s=20)
        ax.set_title(f"{title_prefix} - {name}")
        ax.set_xticks([])
        ax.set_yticks([])
    plt.tight_layout()
    plt.show()

# Run for each group size
for N in [5000, 2500, 1000, 500]:
    top_N = top5k.head(N)
    bottom_N = bottom5k.tail(N)
    df_subset = pd.concat([top_N, bottom_N])
    feats = one_hot_encode_seqs(df_subset["TCR"])
    labels = df_subset["group"].values
    plot_dr(feats, labels, f"Top {N} vs Bottom {N}")


In [ ]:
from sklearn.cluster import KMeans

# 1. Encode and reduce dimensionality
feats = one_hot_encode_seqs(full_df["TCR"])
pca = PCA(n_components=2, random_state=42).fit_transform(feats)

# 2. KMeans clustering on PCA space
kmeans = KMeans(n_clusters=3, random_state=42)
cluster_labels = kmeans.fit_predict(pca)

# 3. Add cluster labels to DataFrame
full_df["cluster"] = cluster_labels

# 4. Plot clusters
plt.figure(figsize=(7, 6))
sns.scatterplot(x=pca[:, 0], y=pca[:, 1], hue=cluster_labels, palette="Set1", s=20)
plt.title("TCR Clusters in PCA Space")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend(title="Cluster")
plt.tight_layout()
plt.show()

# 5. Analyze cluster composition
print(full_df.groupby(["cluster", "group"]).size().unstack(fill_value=0))


In [ ]:
# -------------------------------
# 1. Install dependencies
# -------------------------------
!pip install pandas scikit-learn matplotlib seaborn

# -------------------------------
# 2. Import libraries
# -------------------------------
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.preprocessing import OneHotEncoder

# -------------------------------
# 3. Upload data
# -------------------------------
from google.colab import files
uploaded = files.upload()  # Upload: top5k_tcrs_by_score.csv and bottom5k_tcrs_by_score.csv

# -------------------------------
# 4. Load and preprocess
# -------------------------------
top5k = pd.read_csv("top5k_tcrs_by_score.csv")
bottom5k = pd.read_csv("bottom5k_tcrs_by_score.csv")

top5k["group"] = "top"
bottom5k["group"] = "bottom"

full_df = pd.concat([top5k, bottom5k], ignore_index=True)
full_df = full_df.dropna(subset=["TCR"])
sequences = full_df["TCR"].tolist()

# -------------------------------
# 5. One-hot encode sequences
# -------------------------------
aa_list = list("ACDEFGHIKLMNPQRSTVWY") + ["X", "-"]
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
encoder.fit(np.array(aa_list).reshape(-1, 1))

max_len = 20
padded = [list(seq.ljust(max_len, '-'))[:max_len] for seq in sequences]
encoded = np.stack([encoder.transform(np.array(s).reshape(-1, 1)).flatten() for s in padded])

# -------------------------------
# 6. PCA and KMeans clustering
# -------------------------------
pca = PCA(n_components=2, random_state=42).fit_transform(encoded)
kmeans = KMeans(n_clusters=3, random_state=42)
cluster_labels = kmeans.fit_predict(pca)
full_df["cluster"] = cluster_labels

# -------------------------------
# 7. Sequence Logo Plotting
# -------------------------------
def compute_aa_frequency_matrix(sequences, max_len=20):
    counts = pd.DataFrame(0, index=aa_list, columns=range(max_len))
    for seq in sequences:
        seq = seq.ljust(max_len, '-')[:max_len]
        for i, aa in enumerate(seq):
            if aa in counts.index:
                counts.at[aa, i] += 1
    freq_matrix = counts.div(counts.sum(axis=0), axis=1)
    return freq_matrix

# Prepare frequency matrices per cluster
logo_data = {}
for c in sorted(full_df["cluster"].unique()):
    seqs = full_df[full_df["cluster"] == c]["TCR"].tolist()
    logo_data[c] = compute_aa_frequency_matrix(seqs)

# Plot sequence logos
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
for idx, (cluster_id, freq_matrix) in enumerate(logo_data.items()):
    ax = axes[idx]
    sns.heatmap(freq_matrix, cmap="viridis", ax=ax, cbar=False)
    ax.set_title(f"Cluster {cluster_id} Sequence Logo")
    ax.set_xlabel("Position")
    ax.set_ylabel("Amino Acid")
plt.tight_layout()
plt.show()


In [ ]:
pip install logomaker

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import logomaker

# Amino acids + padding/unknown symbols
aa_list = list("ACDEFGHIKLMNPQRSTVWY") + ["X", "-"]
max_len = 20

# Frequency matrix
def compute_aa_freq_matrix(sequences, aa_list=aa_list, max_len=max_len):
    counts = pd.DataFrame(0, index=aa_list, columns=np.arange(max_len))
    for seq in sequences:
        seq_padded = seq.ljust(max_len, '-')[:max_len]
        for pos, aa in enumerate(seq_padded):
            if aa in counts.index:
                counts.at[aa, pos] += 1
    return counts.div(counts.sum(axis=0), axis=1)

# Plot logos for each cluster
for cluster_id in sorted(full_df["cluster"].unique()):
    seqs = full_df[full_df["cluster"] == cluster_id]["TCR"].dropna().tolist()
    if not seqs:
        continue

    freq_matrix = compute_aa_freq_matrix(seqs)
    freq_matrix = freq_matrix.T  # logomaker expects positions as rows

    # Plot
    plt.figure(figsize=(12, 3))
    logo = logomaker.Logo(freq_matrix,
                          color_scheme='classic',  # try 'chemistry', 'classic', or 'black'
                          vpad=0.1,
                          width=1.0)

    plt.title(f"Cluster {cluster_id} Sequence Logo (n={len(seqs)})")
    plt.xlabel("Position")
    plt.ylabel("Frequency")
    plt.xticks(ticks=np.arange(0, max_len), labels=np.arange(1, max_len+1))
    plt.tight_layout()
    plt.show()


cvc


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# ------------------ Load Data ------------------ #
file_path = "top_bottom_tcrs_with_embeddings.csv"
df = pd.read_csv(file_path)

# ------------------ Classification Function ------------------ #
def run_classification_embeddings(N):
    # Filter top N and bottom N
    top_df = df[df['Group'] == 'Top'].head(N).copy()
    bottom_df = df[df['Group'] == 'Bottom'].tail(N).copy()
    df_filtered = pd.concat([top_df, bottom_df])

    # Extract embeddings and labels
    X = df_filtered.drop(columns=['Group', 'Sequences']).values
    y = LabelEncoder().fit_transform(df_filtered['Group'])

    # Train/test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=42)

    # Train model
    model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
    model.fit(X_train, y_train)

    # Predict and report
    y_pred = model.predict(X_test)

    print(f"\n🧪 Results for top {N} vs bottom {N} (Embeddings):")
    print(classification_report(y_test, y_pred, target_names=["Bottom", "Top"]))

    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["Bottom", "Top"], yticklabels=["Bottom", "Top"])
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(f"Confusion Matrix - Embeddings (Top {N} vs Bottom {N})")
    plt.tight_layout()
    plt.show()

# ------------------ Run for Multiple N ------------------ #
for N in [2500, 1000, 500]:
    run_classification_embeddings(N)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# Load CVC embeddings
file_path = "top_bottom_tcrs_with_embeddings.csv"
df = pd.read_csv(file_path)

# DR and plotting function
def plot_embeddings(X, labels, title_prefix):
    pca = PCA(n_components=2, random_state=42).fit_transform(X)
    tsne = TSNE(n_components=2, random_state=42, perplexity=30).fit_transform(X)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, data, name in zip(axes, [pca, tsne], ["PCA", "t-SNE"]):
        sns.scatterplot(x=data[:, 0], y=data[:, 1], hue=labels, ax=ax, s=20)
        ax.set_title(f"{title_prefix} - {name}")
        ax.set_xticks([])
        ax.set_yticks([])
    plt.tight_layout()
    plt.show()

# Run for each N
for N in [5000, 2500, 1000, 500]:
    top_N = df[df["Group"] == "Top"].head(N)
    bottom_N = df[df["Group"] == "Bottom"].tail(N)
    df_subset = pd.concat([top_N, bottom_N])

    # Drop label and sequence
    X = df_subset.drop(columns=["Group", "Sequences"]).values
    labels = df_subset["Group"].values

    plot_embeddings(X, labels, f"CVC Embeddings: Top {N} vs Bottom {N}")


In [ ]:
import pandas as pd

# Load the file
df = pd.read_csv("updated_tcr_age_lists_with_scores.csv")  # Replace with your actual filename if different

# Sort by tcr_score
sorted_df = df.sort_values(by="tcr_score", ascending=False)

# Get top 5
top5 = sorted_df.head(5000)

# Get bottom 5
bottom5 = sorted_df.tail(5000)

# Save to CSV
top5.to_csv("top5k_tcrs_by_score.csv", index=False)
bottom5.to_csv("bottom5k_tcrs_by_score.csv", index=False)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load the files
top5k = pd.read_csv("top5k_tcrs_by_score.csv")
bottom5k = pd.read_csv("bottom5k_tcrs_by_score.csv")

# Helper function to extract age values from string representation of lists
def extract_ages(df, count):
    ages = []
    for row in df.head(count)["Ages"]:
        row_ages = eval(row) if isinstance(row, str) else row
        ages.extend(row_ages)
    return ages

# Define the sample sizes to plot
sample_sizes = [10, 100, 1000, 5000]

# Plotting
for size in sample_sizes:
    top_ages = extract_ages(top5k, size)
    bottom_ages = extract_ages(bottom5k, size)

    plt.figure(figsize=(10, 4))
    plt.hist(top_ages, bins=range(0, 101, 5), alpha=0.6, label=f'Top {size}', density=True)
    plt.hist(bottom_ages, bins=range(0, 101, 5), alpha=0.6, label=f'Bottom {size}', density=True)
    plt.title(f'Distribution of Ages for Top and Bottom {size} TCRs by Score')
    plt.xlabel('Age')
    plt.ylabel('Density')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


In [ ]:
# Helper to extract ages
def extract_ages(df, counts):
    grouped_ages = {}
    for count in counts:
        ages = []
        for row in df.head(count)["Ages"]:
            row_ages = eval(row) if isinstance(row, str) else row
            ages.extend(row_ages)
        grouped_ages[count] = ages
    return grouped_ages

group_sizes = [10, 100, 1000, 5000]

top_ages_grouped = extract_ages(top5k, group_sizes)
bottom_ages_grouped = extract_ages(bottom5k, group_sizes)

# Plot Top (one color, different transparencies)
plt.figure(figsize=(10, 5))
for i, count in enumerate(group_sizes):
    plt.hist(top_ages_grouped[count], bins=range(0, 101, 5),
             alpha=0.2 + 0.2 * i, label=f"Top {count}", density=True, color='blue')
plt.title("Age Distribution for Top TCRs by Score")
plt.xlabel("Age")
plt.ylabel("Density")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# Plot Bottom (one color, different transparencies)
plt.figure(figsize=(10, 5))
for i, count in enumerate(group_sizes):
    plt.hist(bottom_ages_grouped[count], bins=range(0, 101, 5),
             alpha=0.2 + 0.2 * i, label=f"Bottom {count}", density=True, color='red')
plt.title("Age Distribution for Bottom TCRs by Score")
plt.xlabel("Age")
plt.ylabel("Density")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


# Define group sizes
group_sizes = [100, 1000, 5000]
amino_acids = 'ACDEFGHIKLMNPQRSTVWY'

# Function to calculate amino acid frequencies per position
def calculate_amino_acid_frequencies(tcr_list, max_len=20):
    freq_dict = {pos: {aa: 0 for aa in amino_acids} for pos in range(max_len)}
    for seq in tcr_list:
        for pos, aa in enumerate(seq[:max_len]):
            if aa in freq_dict[pos]:
                freq_dict[pos][aa] += 1
    return freq_dict

# Plot function
def plot_frequencies(freq_top, freq_bottom, group_size):
    for pos in range(20):
        plt.figure(figsize=(12, 5))
        x = range(len(amino_acids))
        width = 0.35
        plt.bar([p - width/2 for p in x],
                [freq_top[pos][aa] for aa in amino_acids],
                width=width, label='Top', color='blue', alpha=0.6)
        plt.bar([p + width/2 for p in x],
                [freq_bottom[pos][aa] for aa in amino_acids],
                width=width, label='Bottom', color='red', alpha=0.6)
        plt.title(f"Amino Acid Frequencies at Position {pos} (Top vs Bottom {group_size})")
        plt.xticks(x, amino_acids)
        plt.xlabel("Amino Acid")
        plt.ylabel("Frequency")
        plt.legend()
        plt.tight_layout()
        plt.show()

# Process and plot for each group size
for size in group_sizes:
    top_group = top5k.head(size)["TCR"].tolist()
    bottom_group = bottom5k.head(size)["TCR"].tolist()
    freq_top = calculate_amino_acid_frequencies(top_group)
    freq_bottom = calculate_amino_acid_frequencies(bottom_group)
    plot_frequencies(freq_top, freq_bottom, size)


newwwwwwwwwwwwwwwwwwwwwwwwwwwwwwwwwwwwwwwwwwwwwwwwwwwwwwww

In [ ]:
import pandas as pd

# Get top 10 and bottom 10 from each
top_10 = top5k.head(10)
bottom_10 = bottom5k.head(10)


In [ ]:
top_10['TCR']

In [ ]:
bottom_10['TCR']

In [ ]:
bottom_1000_lowest_scores['TCR'].

In [ ]:
train_df = top_1000_highest_scores['TCR'].to_list()
test_df = bottom_1000_lowest_scores['TCR'].to_list()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind


positive_corr_group = top_1000_highest_scores['TCR'].to_list()
negative_corr_group = bottom_1000_lowest_scores['TCR'].to_list()

# Function to calculate amino acid frequencies per position
def calculate_amino_acid_frequencies(group):
    amino_acids = 'ACDEFGHIKLMNPQRSTVWY'  # standard amino acids
    max_len = 20  # Get the max length of TCR sequences
    frequency_dict = {pos: {aa: 0 for aa in amino_acids} for pos in range(max_len)}

    for seq in group:
        for pos, aa in enumerate(seq):
            if pos in frequency_dict and aa in frequency_dict[pos]:
                frequency_dict[pos][aa] += 1

    return frequency_dict

# Calculate frequencies for both groups
positive_frequencies = calculate_amino_acid_frequencies(positive_corr_group)
negative_frequencies = calculate_amino_acid_frequencies(negative_corr_group)

# Plot frequencies for each position with bars side by side
for pos in positive_frequencies:
    plt.figure(figsize=(12, 6))
    width = 0.35  # Width of the bars

    amino_acids = list(positive_frequencies[pos].keys())
    x = range(len(amino_acids))

    plt.bar(x, [positive_frequencies[pos][aa] for aa in amino_acids], width=width, label='Positive Correlation', align='center')
    plt.bar([p + width for p in x], [negative_frequencies[pos][aa] for aa in amino_acids], width=width, label='Negative Correlation', align='center')

    plt.xlabel('Amino Acid')
    plt.ylabel('Frequency')
    plt.title(f'Amino Acid Frequencies at Position {pos}')
    plt.xticks([p + width/2 for p in x], amino_acids)
    plt.legend()
    plt.show()

# Function to summarize frequencies across all positions
def summarize_frequencies(frequencies):
    summary = {aa: 0 for aa in 'ACDEFGHIKLMNPQRSTVWY'}
    for pos in frequencies:
        for aa in frequencies[pos]:
            summary[aa] += frequencies[pos][aa]
    return summary

# Summarize frequencies
positive_summary = summarize_frequencies(positive_frequencies)
negative_summary = summarize_frequencies(negative_frequencies)

# Plot summary of frequencies across all positions with bars side by side
plt.figure(figsize=(12, 6))
width = 0.35  # Width of the bars

amino_acids = list(positive_summary.keys())
x = range(len(amino_acids))

plt.bar(x, [positive_summary[aa] for aa in amino_acids], width=width, label='Positive Correlation', align='center')
plt.bar([p + width for p in x], [negative_summary[aa] for aa in amino_acids], width=width, label='Negative Correlation', align='center')

plt.xlabel('Amino Acid')
plt.ylabel('Total Frequency')
plt.title('Total Amino Acid Frequencies Across All Positions')
plt.xticks([p + width/2 for p in x], amino_acids)
plt.legend()
plt.show()

# Calculate CDR3 lengths for each group
positive_lengths = positive_corr_group['TCR'].apply(len)
negative_lengths = negative_corr_group['TCR'].apply(len)

# Plot the CDR3 length distributions
plt.figure(figsize=(12, 6))
plt.hist(positive_lengths, bins=30, alpha=0.5, label='Positive Correlation')
plt.hist(negative_lengths, bins=30, alpha=0.5, label='Negative Correlation')
plt.xlabel('CDR3 Length')
plt.ylabel('Frequency')
plt.title('CDR3 Length Distribution Comparison')
plt.legend()
plt.show()

# Perform statistical test for CDR3 length differences
t_stat, p_value = ttest_ind(positive_lengths, negative_lengths)
print(f'T-Statistic: {t_stat}, P-Value: {p_value}')


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind


# Plot each amino acid's frequency across all positions for both groups
amino_acids = 'ACDEFGHIKLMNPQRSTVWY'

for aa in amino_acids:
    plt.figure(figsize=(12, 6))
    positions = range(len(positive_frequencies[aa]))

    plt.plot(positions, positive_frequencies[aa], marker='o', label='Positive Correlation', color='blue')
    plt.plot(positions, negative_frequencies[aa], marker='o', label='Negative Correlation', color='red')

    plt.xlabel('Position')
    plt.ylabel('Frequency')
    plt.title(f'Frequency of Amino Acid {aa} Across All Positions')
    plt.legend()
    plt.grid(True)
    plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind

# Summing the total counts across all positions for each amino acid
positive_totals = {aa: sum(positive_frequencies[aa]) for aa in positive_frequencies}
negative_totals = {aa: sum(negative_frequencies[aa]) for aa in negative_frequencies}

# Plotting the total counts for both groups
amino_acids = list(positive_totals.keys())
x = range(len(amino_acids))
width = 0.35  # Width of the bars

plt.figure(figsize=(14, 7))

plt.bar(x, [positive_totals[aa] for aa in amino_acids], width=width, label='Positive Correlation', align='center')
plt.bar([p + width for p in x], [negative_totals[aa] for aa in amino_acids], width=width, label='Negative Correlation', align='center')

plt.xlabel('Amino Acid')
plt.ylabel('Total Count')
plt.title('Total Count of Each Amino Acid Across All Positions')
plt.xticks([p + width/2 for p in x], amino_acids)
plt.legend()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind

# Function to calculate amino acid frequencies per position
def calculate_amino_acid_frequencies(group):
    amino_acids = 'ACDEFGHIKLMNPQRSTVWY'  # standard amino acids
    max_len =20  # Get the max length of TCR sequences
    frequency_dict = {aa: [0] * max_len for aa in amino_acids}

    for seq in group:
        for pos, aa in enumerate(seq):
            if aa in frequency_dict:
                frequency_dict[aa][pos] += 1

    return frequency_dict

# Calculate frequencies for both groups
positive_frequencies = calculate_amino_acid_frequencies(positive_corr_group)
negative_frequencies = calculate_amino_acid_frequencies(negative_corr_group)

# Define amino acid groups
amino_acid_groups = {
    'Positively Charged (Basic)': ['R', 'K'],
    'Negatively Charged (Acidic)': ['D', 'E'],
    'Charged Amino Acids': ['H'],
    'Polar, Uncharged': ['S', 'T', 'C', 'N', 'Q'],
    'Polar, Charged': ['Y'],
    'Aliphatic': ['G', 'A', 'V', 'L', 'I'],
    'Aromatic': ['F', 'W'],
    'Special Amino Acids': ['P', 'M', 'C'],
    'Amino Acids with Sulfur': ['C', 'M'],
    'Amino Acids with Amines': ['H'],
    'Amino Acids with Amides': ['N', 'Q'],
    'Imidazole-Containing Amino Acids': ['H'],
    'Hydrophobic Amino Acids': ['V', 'L', 'I', 'F', 'W', 'M']
}

# Sum the total counts for each group
def sum_group_frequencies(frequencies, group_definitions):
    group_totals = {}
    for group_name, amino_acids in group_definitions.items():
        group_totals[group_name] = [sum(frequencies[aa]) for aa in amino_acids if aa in frequencies]
    return group_totals

positive_group_totals = sum_group_frequencies(positive_frequencies, amino_acid_groups)
negative_group_totals = sum_group_frequencies(negative_frequencies, amino_acid_groups)

# Plot each amino acid group in a separate plot and calculate significance
for group_name in amino_acid_groups.keys():
    plt.figure(figsize=(10, 6))
    x = range(len(positive_group_totals[group_name]))
    width = 0.35  # Width of the bars

    plt.bar(x, positive_group_totals[group_name], width=width, label='Positive Correlation', align='center')
    plt.bar([p + width for p in x], negative_group_totals[group_name], width=width, label='Negative Correlation', align='center')

    plt.xlabel('Amino Acids')
    plt.ylabel('Total Count')
    plt.title(f'Total Count of {group_name} Amino Acids')
    plt.xticks([p + width/2 for p in x], amino_acid_groups[group_name])
    plt.legend()
    plt.tight_layout()
    plt.show()

    # Calculate statistical significance
    t_stat, p_value = ttest_ind(positive_group_totals[group_name], negative_group_totals[group_name])
    print(f'Group: {group_name}, T-Statistic: {t_stat}, P-Value: {p_value}')


In [ ]:
import seaborn as sns

# Step 1: Sort the dataframe by TCR scores
df_tcr_sorted = df_tcr.sort_values(by='tcr_score', ascending=False)

# Step 2: Select the top 50 and bottom 50 rows based on TCR scores
top_50_highest_scores = df_tcr_sorted.head(50)
bottom_50_lowest_scores = df_tcr_sorted.tail(50)

# Step 3: Calculate the average percentages across deciles for the highest and lowest scored TCRs
top_50_highest_scores_averages = top_50_highest_scores[['1', '2', '3', '4', '5', '6', '7', '8', '9', '10']].mean()
bottom_50_lowest_scores_averages = bottom_50_lowest_scores[['1', '2', '3', '4', '5', '6', '7', '8', '9', '10']].mean()

# Step 4: Plot the average percentages for highest scores
plt.figure(figsize=(10, 6))
plt.fill_between(top_50_highest_scores_averages.index, top_50_highest_scores_averages.values, color='lightcoral', alpha=0.5, label='Top 50 Highest Scores')
plt.plot(top_50_highest_scores_averages.index, top_50_highest_scores_averages.values, color='red', label='Top 50 Highest Scores Line')
plt.title('Average Percentage of Ages (Top 50 Highest Scores)')
plt.xlabel('Age Group (Deciles)')
plt.ylabel('Average Percentage (%)')
plt.legend()
plt.grid(True)
plt.show()

# Step 5: Plot the average percentages for lowest scores
plt.figure(figsize=(10, 6))
plt.fill_between(bottom_50_lowest_scores_averages.index, bottom_50_lowest_scores_averages.values, color='lightblue', alpha=0.5, label='Bottom 50 Lowest Scores')
plt.plot(bottom_50_lowest_scores_averages.index, bottom_50_lowest_scores_averages.values, color='blue', label='Bottom 50 Lowest Scores Line')
plt.title('Average Percentage of Ages (Bottom 50 Lowest Scores)')
plt.xlabel('Age Group (Deciles)')
plt.ylabel('Average Percentage (%)')
plt.legend()
plt.grid(True)
plt.show()

# Additional plot ideas:

# 1. Plot the distribution of scores
plt.figure(figsize=(10, 6))
plt.hist(df_tcr['tcr_score'], bins=30, color='purple', alpha=0.7, edgecolor='black')
plt.title('Distribution of TCR Scores')
plt.xlabel('TCR Score')
plt.ylabel('Frequency')
plt.grid(True)
plt.show()

# 2. Scatter plot of TCR scores against a specific decile, e.g., decile 1 (youngest group)
plt.figure(figsize=(10, 6))
plt.scatter(df_tcr['1'], df_tcr['tcr_score'], alpha=0.5, color='green')
plt.title('Scatter Plot of Decile 1 Percentage vs TCR Score')
plt.xlabel('Percentage in Decile 1 (Youngest Group)')
plt.ylabel('TCR Score')
plt.grid(True)
plt.show()

# 3. Box plot of TCR scores per decile
plt.figure(figsize=(10, 6))
sns.boxplot(data=df_tcr[['1', '2', '3', '4', '5', '6', '7', '8', '9', '10']])
plt.title('Box Plot of Decile Percentages Across All TCRs')
plt.xlabel('Age Group (Deciles)')
plt.ylabel('Percentage')
plt.grid(True)
plt.show()

# 4. Heatmap of correlation between TCR score and age deciles
plt.figure(figsize=(10, 6))
sns.heatmap(df_tcr.corr(), annot=True, cmap='coolwarm')
plt.title('Correlation Heatmap Between TCR Score and Age Deciles')
plt.show()


In [ ]:
import pandas as pd
import numpy as np

# Load the train and test CSV files
train_df = top_1000_highest_scores['TCR']
test_df = bottom_1000_lowest_scores['TCR']


# Combine the datasets temporarily to determine the maximum sequence length
combined_df = pd.concat([train_df, test_df])

# Define the amino acids and padding character
amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
padding_char = 'X'  # 'X' is often used as a padding character in bioinformatics

# One-hot encode a single sequence
def one_hot_encode_sequence(seq, max_length, amino_acids, padding_char):
    # Pad the sequence with the padding character to the max_length
    seq = seq.ljust(max_length, padding_char)
    # Create a one-hot encoded matrix of shape (max_length, number of amino acids)
    one_hot_matrix = np.zeros((max_length, len(amino_acids)), dtype=int)
    for i, char in enumerate(seq):
        if char in amino_acids:
            one_hot_matrix[i, amino_acids.index(char)] = 1
        else:
            # Assign padding character to a column of zeros since it's not a valid amino acid
            pass  # The matrix is already initialized to zeros
    return one_hot_matrix

# Determine the maximum length among all sequences in both train and test sets
max_length = combined_df['TCR'].apply(len).max()

# Function to encode and save the dataset
def encode_and_save(df, output_file_path, max_length):
    encoded_seqs = np.zeros((len(df), max_length * len(amino_acids)), dtype=int)

    for idx, seq in enumerate(df['TCR']):
        # One-hot encode each sequence and flatten the matrix to a 1D array
        encoded_seq = one_hot_encode_sequence(seq, max_length, amino_acids, padding_char).flatten()
        encoded_seqs[idx, :] = encoded_seq

    # Create a DataFrame for the encoded data
    encoded_df = pd.DataFrame(encoded_seqs,
                              columns=[f'pos_{i}_{aa}' for i in range(max_length) for aa in amino_acids])

    # Add the target column (positive or negative correlation)
    encoded_df['Target'] = df['Pearson Correlation with Age'].apply(lambda x: 1 if x > 0 else 0).values

    # Save the encoded data to a new CSV file
    encoded_df.to_csv(output_file_path, index=False)
    print(f"Encoding complete. Data saved to '{output_file_path}'")

# Encode and save the train dataset
encode_and_save(train_df, 'encoded_tcr_train.csv', max_length)

# Encode and save the test dataset
encode_and_save(test_df, 'encoded_tcr_test.csv', max_length)


rest of the code

In [ ]:
import pandas as pd
import numpy as np

# Function to filter out rows with NaN in the primary target
def filter_nan_targets(data):
    return data[~data['target'].apply(lambda x: np.isnan(eval(x, {"nan": np.nan})[1]))]

# Function to flatten the embeddings
def flatten_embeddings(embedding_column):
    return np.array([np.array(eval(embedding, {"nan": np.nan})).flatten() for embedding in embedding_column])

# Load the CSV file
file_path = 'embeddings_layers5_hidden256_dropout0.0_lr0.001_weightdecay0.0.csv'
data = pd.read_csv(file_path)

# Filter out rows with NaN in the primary target (age)
data_filtered = filter_nan_targets(data)

# Flatten the embeddings
embeddings = flatten_embeddings(data_filtered['embedding'])

# Extract the primary target (age)
targets = np.array([eval(target, {"nan": np.nan})[1] for target in data_filtered['target']])

# Extract the first target (biological sex: Male = 0, Female = 1)
biological_sex = np.array([eval(target, {"nan": np.nan})[0] for target in data_filtered['target']])

# Combine embeddings (features), age, and biological sex into one DataFrame
features_df = pd.DataFrame(embeddings, columns=[f'feature_{i}' for i in range(embeddings.shape[1])])
features_df['age'] = targets
features_df['biological_sex'] = biological_sex

# Split the data into two tables by biological sex
male_df = features_df[features_df['biological_sex'] == 0].drop(columns=['biological_sex'])
female_df = features_df[features_df['biological_sex'] == 1].drop(columns=['biological_sex'])

# Save the two tables to separate CSV files
male_output_file = 'male_combined_features_and_target.csv'
female_output_file = 'female_combined_features_and_target.csv'
male_df.to_csv(male_output_file, index=False)
female_df.to_csv(female_output_file, index=False)

print(f"Male table saved to {male_output_file}")
print(f"Female table saved to {female_output_file}")


In [ ]:
female_df

In [ ]:
male_df

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load the uploaded CSV file
file_path = 'xgboost_results_age_combined.csv'
results_df_age = pd.read_csv(file_path)

# Sorting the DataFrame by MAE within each vector type
results_df_age_sorted = results_df_age.sort_values(by=['vector_type', 'mae'])

# Reset the index after sorting
results_df_age_sorted['index'] = results_df_age_sorted.groupby('vector_type').cumcount()

# Create a figure
plt.figure(figsize=(10, 6))

# Plot the sorted MAE values for each combination by index, grouped by vector type
for vector_type in results_df_age_sorted['vector_type'].unique():
    subset = results_df_age_sorted[results_df_age_sorted['vector_type'] == vector_type]
    plt.plot(subset['index'], subset['mae'], marker='o', label=vector_type)

# Adding plot details
plt.title('Sorted MAE by Combination Index for Different Methods')
plt.xlabel('Combination Index')
plt.ylabel('Mean Absolute Error (MAE)')
plt.legend(title="Method")
plt.grid(True)
plt.show()


In [ ]:
# Dropping any rows with 'mae' values above 17
results_df_filtered = results_df_age[results_df_age['mae'] <= 17]

# Sorting the DataFrame by MAE within each vector type
results_df_filtered_sorted = results_df_filtered.sort_values(by=['vector_type', 'mae'])

# Reset the index after sorting
results_df_filtered_sorted['index'] = results_df_filtered_sorted.groupby('vector_type').cumcount()

# Create a figure for the sorted MAE values
plt.figure(figsize=(10, 6))

# Plot the sorted MAE values for each combination by index, grouped by vector type
for vector_type in results_df_filtered_sorted['vector_type'].unique():
    subset = results_df_filtered_sorted[results_df_filtered_sorted['vector_type'] == vector_type]
    plt.plot(subset['index'], subset['mae'], marker='o', label=vector_type)

# Adding plot details
plt.title('Sorted MAE by Combination Index for Different Methods')
plt.xlabel('Combination Index')
plt.ylabel('Mean Absolute Error (MAE)')
plt.legend(title="Method")
plt.grid(True)
plt.show()


In [ ]:
pip install logomaker

In [ ]:
import pandas as pd

# Load the statistical results
stat_results_file = 'train_tcr_correlation_ttest_results.csv'  # Adjust path as necessary
stat_results_df = pd.read_csv(stat_results_file)

# Define significance thresholds
significance_threshold_negative = 0.05
significance_threshold_positive = 0.5

# Filter for significant TCRs based on Pearson Correlation with Age
positive_corr_group = stat_results_df[
    (stat_results_df['p-value with Age'] < significance_threshold_positive) &
    (stat_results_df['Pearson Correlation with Age'] > 0)
]

negative_corr_group = stat_results_df[
    (stat_results_df['p-value with Age'] < significance_threshold_negative) &
    (stat_results_df['Pearson Correlation with Age'] < 0)
]

# Downsample the larger group to balance the classes
min_size = min(len(positive_corr_group), len(negative_corr_group))
positive_corr_group = positive_corr_group.sample(min_size, random_state=42)
negative_corr_group = negative_corr_group.sample(min_size, random_state=42)

# Pad the sequences with 'O' to the maximum sequence length
max_len = max(positive_corr_group['TCR'].apply(len).max(), negative_corr_group['TCR'].apply(len).max())
positive_corr_group['Padded_TCR'] = positive_corr_group['TCR'].apply(lambda x: x.ljust(max_len, 'O'))
negative_corr_group['Padded_TCR'] = negative_corr_group['TCR'].apply(lambda x: x.ljust(max_len, 'O'))

# Save the padded sequences to files
positive_corr_group[['Padded_TCR']].to_csv('positive_padded_tcrs.csv', index=False)
negative_corr_group[['Padded_TCR']].to_csv('negative_padded_tcrs.csv', index=False)

print("Padded sequences saved to 'positive_padded_tcrs.csv' and 'negative_padded_tcrs.csv'.")


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind

# Load the statistical results
stat_results_file = 'train_tcr_correlation_ttest_results.csv'  # Adjust path as necessary
stat_results_df = pd.read_csv(stat_results_file)

# Define significance thresholds
significance_threshold_negative = 0.05
significance_threshold_positive = 0.5

# Filter for significant TCRs based on Pearson Correlation with Age
positive_corr_group = stat_results_df[
    (stat_results_df['p-value with Age'] < significance_threshold_positive) &
    (stat_results_df['Pearson Correlation with Age'] > 0)
]

negative_corr_group = stat_results_df[
    (stat_results_df['p-value with Age'] < significance_threshold_negative) &
    (stat_results_df['Pearson Correlation with Age'] < 0)
]

# Downsample the larger group to balance the classes
min_size = min(len(positive_corr_group), len(negative_corr_group))
positive_corr_group = positive_corr_group.sample(min_size, random_state=42)
negative_corr_group = negative_corr_group.sample(min_size, random_state=42)

# Function to calculate amino acid frequencies per position
def calculate_amino_acid_frequencies(group):
    amino_acids = 'ACDEFGHIKLMNPQRSTVWY'  # standard amino acids
    max_len = max(group['TCR'].apply(len))  # Get the max length of TCR sequences
    frequency_dict = {pos: {aa: 0 for aa in amino_acids} for pos in range(max_len)}

    for seq in group['TCR']:
        for pos, aa in enumerate(seq):
            if pos in frequency_dict and aa in frequency_dict[pos]:
                frequency_dict[pos][aa] += 1

    return frequency_dict

# Calculate frequencies for both groups
positive_frequencies = calculate_amino_acid_frequencies(positive_corr_group)
negative_frequencies = calculate_amino_acid_frequencies(negative_corr_group)

# Plot frequencies for each position with bars side by side
for pos in positive_frequencies:
    plt.figure(figsize=(12, 6))
    width = 0.35  # Width of the bars

    amino_acids = list(positive_frequencies[pos].keys())
    x = range(len(amino_acids))

    plt.bar(x, [positive_frequencies[pos][aa] for aa in amino_acids], width=width, label='Positive Correlation', align='center')
    plt.bar([p + width for p in x], [negative_frequencies[pos][aa] for aa in amino_acids], width=width, label='Negative Correlation', align='center')

    plt.xlabel('Amino Acid')
    plt.ylabel('Frequency')
    plt.title(f'Amino Acid Frequencies at Position {pos}')
    plt.xticks([p + width/2 for p in x], amino_acids)
    plt.legend()
    plt.show()

# Function to summarize frequencies across all positions
def summarize_frequencies(frequencies):
    summary = {aa: 0 for aa in 'ACDEFGHIKLMNPQRSTVWY'}
    for pos in frequencies:
        for aa in frequencies[pos]:
            summary[aa] += frequencies[pos][aa]
    return summary

# Summarize frequencies
positive_summary = summarize_frequencies(positive_frequencies)
negative_summary = summarize_frequencies(negative_frequencies)

# Plot summary of frequencies across all positions with bars side by side
plt.figure(figsize=(12, 6))
width = 0.35  # Width of the bars

amino_acids = list(positive_summary.keys())
x = range(len(amino_acids))

plt.bar(x, [positive_summary[aa] for aa in amino_acids], width=width, label='Positive Correlation', align='center')
plt.bar([p + width for p in x], [negative_summary[aa] for aa in amino_acids], width=width, label='Negative Correlation', align='center')

plt.xlabel('Amino Acid')
plt.ylabel('Total Frequency')
plt.title('Total Amino Acid Frequencies Across All Positions')
plt.xticks([p + width/2 for p in x], amino_acids)
plt.legend()
plt.show()

# Calculate CDR3 lengths for each group
positive_lengths = positive_corr_group['TCR'].apply(len)
negative_lengths = negative_corr_group['TCR'].apply(len)

# Plot the CDR3 length distributions
plt.figure(figsize=(12, 6))
plt.hist(positive_lengths, bins=30, alpha=0.5, label='Positive Correlation')
plt.hist(negative_lengths, bins=30, alpha=0.5, label='Negative Correlation')
plt.xlabel('CDR3 Length')
plt.ylabel('Frequency')
plt.title('CDR3 Length Distribution Comparison')
plt.legend()
plt.show()

# Perform statistical test for CDR3 length differences
t_stat, p_value = ttest_ind(positive_lengths, negative_lengths)
print(f'T-Statistic: {t_stat}, P-Value: {p_value}')


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind

# Load the statistical results
stat_results_file = 'train_tcr_correlation_ttest_results.csv'  # Adjust path as necessary
stat_results_df = pd.read_csv(stat_results_file)

# Define significance thresholds
significance_threshold_negative = 0.05
significance_threshold_positive = 0.5
# Filter for significant TCRs based on Pearson Correlation with Age
positive_corr_group = stat_results_df[
    (stat_results_df['p-value with Age'] < significance_threshold_positive) &
    (stat_results_df['Pearson Correlation with Age'] > 0)
]

negative_corr_group = stat_results_df[
    (stat_results_df['p-value with Age'] < significance_threshold_negative) &
    (stat_results_df['Pearson Correlation with Age'] < 0)
]

# Downsample the larger group to balance the classes
min_size = min(len(positive_corr_group), len(negative_corr_group))
positive_corr_group = positive_corr_group.sample(min_size, random_state=42)
negative_corr_group = negative_corr_group.sample(min_size, random_state=42)

# Function to calculate amino acid frequencies per position
def calculate_amino_acid_frequencies(group):
    amino_acids = 'ACDEFGHIKLMNPQRSTVWY'  # standard amino acids
    max_len = max(group['TCR'].apply(len))  # Get the max length of TCR sequences
    frequency_dict = {aa: [0] * max_len for aa in amino_acids}

    for seq in group['TCR']:
        for pos, aa in enumerate(seq):
            if aa in frequency_dict:
                frequency_dict[aa][pos] += 1

    return frequency_dict

# Calculate frequencies for both groups
positive_frequencies = calculate_amino_acid_frequencies(positive_corr_group)
negative_frequencies = calculate_amino_acid_frequencies(negative_corr_group)

# Plot each amino acid's frequency across all positions for both groups
amino_acids = 'ACDEFGHIKLMNPQRSTVWY'

for aa in amino_acids:
    plt.figure(figsize=(12, 6))
    positions = range(len(positive_frequencies[aa]))

    plt.plot(positions, positive_frequencies[aa], marker='o', label='Positive Correlation', color='blue')
    plt.plot(positions, negative_frequencies[aa], marker='o', label='Negative Correlation', color='red')

    plt.xlabel('Position')
    plt.ylabel('Frequency')
    plt.title(f'Frequency of Amino Acid {aa} Across All Positions')
    plt.legend()
    plt.grid(True)
    plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind

# Load the statistical results
stat_results_file = 'train_tcr_correlation_ttest_results.csv'  # Adjust path as necessary
stat_results_df = pd.read_csv(stat_results_file)

# Define significance thresholds
significance_threshold_negative = 0.05
significance_threshold_positive = 0.5

# Filter for significant TCRs based on Pearson Correlation with Age
positive_corr_group = stat_results_df[
    (stat_results_df['p-value with Age'] < significance_threshold_positive) &
    (stat_results_df['Pearson Correlation with Age'] > 0)
]

negative_corr_group = stat_results_df[
    (stat_results_df['p-value with Age'] < significance_threshold_negative) &
    (stat_results_df['Pearson Correlation with Age'] < 0)
]

# Downsample the larger group to balance the classes
min_size = min(len(positive_corr_group), len(negative_corr_group))
positive_corr_group = positive_corr_group.sample(min_size, random_state=42)
negative_corr_group = negative_corr_group.sample(min_size, random_state=42)


In [ ]:
positive_corr_group.to_csv('positive_corr_group.csv')
negative_corr_group.to_csv('negative_corr_group.csv')

In [ ]:
negative_corr_group

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind

# Load the statistical results
stat_results_file = 'train_tcr_correlation_ttest_results.csv'  # Adjust path as necessary
stat_results_df = pd.read_csv(stat_results_file)

# Define significance thresholds
significance_threshold_negative = 0.05
significance_threshold_positive = 0.5

# Filter for significant TCRs based on Pearson Correlation with Age
positive_corr_group = stat_results_df[
    (stat_results_df['p-value with Age'] < significance_threshold_positive) &
    (stat_results_df['Pearson Correlation with Age'] > 0)
]

negative_corr_group = stat_results_df[
    (stat_results_df['p-value with Age'] < significance_threshold_negative) &
    (stat_results_df['Pearson Correlation with Age'] < 0)
]

# Downsample the larger group to balance the classes
min_size = min(len(positive_corr_group), len(negative_corr_group))
positive_corr_group = positive_corr_group.sample(min_size, random_state=42)
negative_corr_group = negative_corr_group.sample(min_size, random_state=42)

# Function to calculate amino acid frequencies per position
def calculate_amino_acid_frequencies(group):
    amino_acids = 'ACDEFGHIKLMNPQRSTVWY'  # standard amino acids
    max_len = max(group['TCR'].apply(len))  # Get the max length of TCR sequences
    frequency_dict = {aa: [0] * max_len for aa in amino_acids}

    for seq in group['TCR']:
        for pos, aa in enumerate(seq):
            if aa in frequency_dict:
                frequency_dict[aa][pos] += 1

    return frequency_dict

# Calculate frequencies for both groups
positive_frequencies = calculate_amino_acid_frequencies(positive_corr_group)
negative_frequencies = calculate_amino_acid_frequencies(negative_corr_group)

# Summing the total counts across all positions for each amino acid
positive_totals = {aa: sum(positive_frequencies[aa]) for aa in positive_frequencies}
negative_totals = {aa: sum(negative_frequencies[aa]) for aa in negative_frequencies}

# Plotting the total counts for both groups
amino_acids = list(positive_totals.keys())
x = range(len(amino_acids))
width = 0.35  # Width of the bars

plt.figure(figsize=(14, 7))

plt.bar(x, [positive_totals[aa] for aa in amino_acids], width=width, label='Positive Correlation', align='center')
plt.bar([p + width for p in x], [negative_totals[aa] for aa in amino_acids], width=width, label='Negative Correlation', align='center')

plt.xlabel('Amino Acid')
plt.ylabel('Total Count')
plt.title('Total Count of Each Amino Acid Across All Positions')
plt.xticks([p + width/2 for p in x], amino_acids)
plt.legend()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind

# Load the statistical results
stat_results_file = 'train_tcr_correlation_ttest_results.csv'  # Adjust path as necessary
stat_results_df = pd.read_csv(stat_results_file)

# Define significance thresholds
significance_threshold_negative = 0.05
significance_threshold_positive = 0.5

# Filter for significant TCRs based on Pearson Correlation with Age
positive_corr_group = stat_results_df[
    (stat_results_df['p-value with Age'] < significance_threshold_positive) &
    (stat_results_df['Pearson Correlation with Age'] > 0)
]

negative_corr_group = stat_results_df[
    (stat_results_df['p-value with Age'] < significance_threshold_negative) &
    (stat_results_df['Pearson Correlation with Age'] < 0)
]

# Downsample the larger group to balance the classes
min_size = min(len(positive_corr_group), len(negative_corr_group))
positive_corr_group = positive_corr_group.sample(min_size, random_state=42)
negative_corr_group = negative_corr_group.sample(min_size, random_state=42)

# Function to calculate amino acid frequencies per position
def calculate_amino_acid_frequencies(group):
    amino_acids = 'ACDEFGHIKLMNPQRSTVWY'  # standard amino acids
    max_len = max(group['TCR'].apply(len))  # Get the max length of TCR sequences
    frequency_dict = {aa: [0] * max_len for aa in amino_acids}

    for seq in group['TCR']:
        for pos, aa in enumerate(seq):
            if aa in frequency_dict:
                frequency_dict[aa][pos] += 1

    return frequency_dict

# Calculate frequencies for both groups
positive_frequencies = calculate_amino_acid_frequencies(positive_corr_group)
negative_frequencies = calculate_amino_acid_frequencies(negative_corr_group)

# Define amino acid groups
amino_acid_groups = {
    'Positively Charged (Basic)': ['R', 'K'],
    'Negatively Charged (Acidic)': ['D', 'E'],
    'Charged Amino Acids': ['H'],
    'Polar, Uncharged': ['S', 'T', 'C', 'N', 'Q'],
    'Polar, Charged': ['Y'],
    'Aliphatic': ['G', 'A', 'V', 'L', 'I'],
    'Aromatic': ['F', 'W'],
    'Special Amino Acids': ['P', 'M', 'C'],
    'Amino Acids with Sulfur': ['C', 'M'],
    'Amino Acids with Amines': ['H'],
    'Amino Acids with Amides': ['N', 'Q'],
    'Imidazole-Containing Amino Acids': ['H'],
    'Hydrophobic Amino Acids': ['V', 'L', 'I', 'F', 'W', 'M']
}

# Sum the total counts for each group
def sum_group_frequencies(frequencies, group_definitions):
    group_totals = {}
    for group_name, amino_acids in group_definitions.items():
        group_totals[group_name] = [sum(frequencies[aa]) for aa in amino_acids if aa in frequencies]
    return group_totals

positive_group_totals = sum_group_frequencies(positive_frequencies, amino_acid_groups)
negative_group_totals = sum_group_frequencies(negative_frequencies, amino_acid_groups)

# Plot each amino acid group in a separate plot and calculate significance
for group_name in amino_acid_groups.keys():
    plt.figure(figsize=(10, 6))
    x = range(len(positive_group_totals[group_name]))
    width = 0.35  # Width of the bars

    plt.bar(x, positive_group_totals[group_name], width=width, label='Positive Correlation', align='center')
    plt.bar([p + width for p in x], negative_group_totals[group_name], width=width, label='Negative Correlation', align='center')

    plt.xlabel('Amino Acids')
    plt.ylabel('Total Count')
    plt.title(f'Total Count of {group_name} Amino Acids')
    plt.xticks([p + width/2 for p in x], amino_acid_groups[group_name])
    plt.legend()
    plt.tight_layout()
    plt.show()

    # Calculate statistical significance
    t_stat, p_value = ttest_ind(positive_group_totals[group_name], negative_group_totals[group_name])
    print(f'Group: {group_name}, T-Statistic: {t_stat}, P-Value: {p_value}')


In [ ]:
import pandas as pd
import numpy as np

# Load the train and test CSV files
train_df = pd.read_csv('train_tcr_correlation_ttest_results.csv')
test_df = pd.read_csv('test_tcr_correlation_ttest_results.csv')

# Combine the datasets temporarily to determine the maximum sequence length
combined_df = pd.concat([train_df, test_df])

# Define the amino acids and padding character
amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
padding_char = 'X'  # 'X' is often used as a padding character in bioinformatics

# One-hot encode a single sequence
def one_hot_encode_sequence(seq, max_length, amino_acids, padding_char):
    # Pad the sequence with the padding character to the max_length
    seq = seq.ljust(max_length, padding_char)
    # Create a one-hot encoded matrix of shape (max_length, number of amino acids)
    one_hot_matrix = np.zeros((max_length, len(amino_acids)), dtype=int)
    for i, char in enumerate(seq):
        if char in amino_acids:
            one_hot_matrix[i, amino_acids.index(char)] = 1
        else:
            # Assign padding character to a column of zeros since it's not a valid amino acid
            pass  # The matrix is already initialized to zeros
    return one_hot_matrix

# Determine the maximum length among all sequences in both train and test sets
max_length = combined_df['TCR'].apply(len).max()

# Function to encode and save the dataset
def encode_and_save(df, output_file_path, max_length):
    encoded_seqs = np.zeros((len(df), max_length * len(amino_acids)), dtype=int)

    for idx, seq in enumerate(df['TCR']):
        # One-hot encode each sequence and flatten the matrix to a 1D array
        encoded_seq = one_hot_encode_sequence(seq, max_length, amino_acids, padding_char).flatten()
        encoded_seqs[idx, :] = encoded_seq

    # Create a DataFrame for the encoded data
    encoded_df = pd.DataFrame(encoded_seqs,
                              columns=[f'pos_{i}_{aa}' for i in range(max_length) for aa in amino_acids])

    # Add the target column (positive or negative correlation)
    encoded_df['Target'] = df['Pearson Correlation with Age'].apply(lambda x: 1 if x > 0 else 0).values

    # Save the encoded data to a new CSV file
    encoded_df.to_csv(output_file_path, index=False)
    print(f"Encoding complete. Data saved to '{output_file_path}'")

# Encode and save the train dataset
encode_and_save(train_df, 'encoded_tcr_train.csv', max_length)

# Encode and save the test dataset
encode_and_save(test_df, 'encoded_tcr_test.csv', max_length)


In [ ]:
import pandas as pd
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.utils import resample

# Load the encoded training and testing data
train_df = pd.read_csv('encoded_tcr_train.csv')
test_df = pd.read_csv('encoded_tcr_test.csv')

# Load the original data to filter by p-value
train_original = pd.read_csv('train_tcr_correlation_ttest_results.csv')
test_original = pd.read_csv('test_tcr_correlation_ttest_results.csv')

# Define significance thresholds
significance_threshold_negative = 0.05
significance_threshold_positive = 0.5

# Filter the training data for significant TCRs
train_positive_indices = train_original[
    (train_original['p-value with Age'] < significance_threshold_positive) &
    (train_original['Pearson Correlation with Age'] > 0)
].index

train_negative_indices = train_original[
    (train_original['p-value with Age'] < significance_threshold_negative) &
    (train_original['Pearson Correlation with Age'] < 0)
].index

train_significant_indices = train_positive_indices.union(train_negative_indices)
train_df_significant = train_df.iloc[train_significant_indices]

# Filter the test data for significant TCRs
test_positive_indices = test_original[
    (test_original['p-value with Age'] < significance_threshold_positive) &
    (test_original['Pearson Correlation with Age'] > 0)
].index

test_negative_indices = test_original[
    (test_original['p-value with Age'] < significance_threshold_negative) &
    (test_original['Pearson Correlation with Age'] < 0)
].index

test_significant_indices = test_positive_indices.union(test_negative_indices)
test_df_significant = test_df.iloc[test_significant_indices]

# Separate features and target in the filtered data
X_train = train_df_significant.drop(columns=['Target'])
y_train = train_df_significant['Target']

X_test = test_df_significant.drop(columns=['Target'])
y_test = test_df_significant['Target']

# Combine X_train and y_train for downsampling
train_combined = pd.concat([X_train, y_train], axis=1)

# Separate majority and minority classes
majority_class = train_combined[train_combined['Target'] == 0]
minority_class = train_combined[train_combined['Target'] == 1]

# Downsample the majority class
majority_class_downsampled = resample(majority_class,
                                      replace=False,  # sample without replacement
                                      n_samples=len(minority_class),  # match minority class size
                                      random_state=42)  # for reproducibility

# Combine the downsampled majority class with the minority class
downsampled_train = pd.concat([majority_class_downsampled, minority_class])

# Separate features and target from the downsampled data
X_train_downsampled = downsampled_train.drop(columns=['Target'])
y_train_downsampled = downsampled_train['Target']

# Initialize the XGBoost classifier without class weighting (as data is now balanced)
xgb_model_downsampled = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss')

# Train the model on the downsampled data
xgb_model_downsampled.fit(X_train_downsampled, y_train_downsampled)

# Predict on the test set
y_pred_downsampled = xgb_model_downsampled.predict(X_test)

# Evaluate the model
accuracy_downsampled = accuracy_score(y_test, y_pred_downsampled)
print(f"Accuracy: {accuracy_downsampled:.4f}")

# Print classification report
print("Classification Report (Downsampled):")
print(classification_report(y_test, y_pred_downsampled))

# Print confusion matrix
print("Confusion Matrix (Downsampled):")
print(confusion_matrix(y_test, y_pred_downsampled))


In [ ]:
from sklearn.model_selection import GridSearchCV

# Define the parameter grid
param_grid = {
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'n_estimators': [100, 200, 500],
    'min_child_weight': [1, 5, 10],
    'gamma': [0, 0.1, 0.3],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

# Initialize the XGBoost classifier with scale_pos_weight
xgb_model = xgb.XGBClassifier(scale_pos_weight=scale_pos_weight, use_label_encoder=False, eval_metric='logloss')

# Initialize GridSearchCV
grid_search = GridSearchCV(estimator=xgb_model, param_grid=param_grid, scoring='f1_weighted', cv=3, verbose=1, n_jobs=-1)

# Fit the model
grid_search.fit(X_train, y_train)

# Best parameters
print("Best parameters found: ", grid_search.best_params_)

# Best model
best_model = grid_search.best_estimator_

# Predict on the test set
y_pred = best_model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")
print("Classification Report:")
print(classification_report(y_test, y_pred))
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load the data
data = pd.read_csv('xgboost_results.csv')

# Function to plot boxplots for individual variables
def plot_boxplot(data, x_col, y_col, title):
    plt.figure(figsize=(10, 6))
    sns.boxplot(x=x_col, y=y_col, data=data)
    plt.title(title)
    plt.show()

# Function to plot scatter plots for pairwise comparisons
def plot_pairwise(data, x_col, y_col, hue_col, title):
    plt.figure(figsize=(10, 6))
    sns.scatterplot(x=x_col, y=y_col, hue=hue_col, data=data)
    plt.title(title)
    plt.show()

# Function to plot heatmaps for combinations of variables
def plot_heatmap(data, index_col, columns_col, values_col, title):
    pivot_table = data.pivot_table(index=index_col, columns=columns_col, values=values_col, aggfunc='mean')
    plt.figure(figsize=(10, 8))
    sns.heatmap(pivot_table, annot=True, cmap="YlGnBu", fmt=".2f")
    plt.title(title)
    plt.show()

# 1. Boxplot for each variable to see distribution of accuracy
plot_boxplot(data, 'layers', 'accuracy', 'Accuracy Distribution by Layers')
plot_boxplot(data, 'hidden_units', 'accuracy', 'Accuracy Distribution by Hidden Units')
plot_boxplot(data, 'learning_rate', 'accuracy', 'Accuracy Distribution by Learning Rate')
plot_boxplot(data, 'fc_layer_size', 'accuracy', 'Accuracy Distribution by FC Layer Size')
plot_boxplot(data, 'vector_type', 'accuracy', 'Accuracy Distribution by Vector Type')

# 2. Pairwise scatter plots to see interaction between variables
plot_pairwise(data, 'learning_rate', 'accuracy', 'hidden_units', 'Learning Rate vs. Accuracy by Hidden Units')
plot_pairwise(data, 'layers', 'accuracy', 'vector_type', 'Layers vs. Accuracy by Vector Type')
plot_pairwise(data, 'fc_layer_size', 'accuracy', 'vector_type', 'FC Layer Size vs. Accuracy by Vector Type')

# 3. Heatmaps for combinations of variables
plot_heatmap(data, 'hidden_units', 'learning_rate', 'accuracy', 'Heatmap: Hidden Units vs. Learning Rate')
plot_heatmap(data, 'layers', 'fc_layer_size', 'accuracy', 'Heatmap: Layers vs. FC Layer Size')


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Load the data
data = pd.read_csv('xgboost_results.csv')

# Separate the data by vector type (dummy and frequency)
dummy_data = data[data['vector_type'] == 'dummy']
frequency_data = data[data['vector_type'] == 'frequency']

# Separate the data by FC layer size (with and without FC layer)
with_fc_data = data[data['fc_layer_size'].notna()]
without_fc_data = data[data['fc_layer_size'].isna()]

# Function to create a pivot table for heatmaps
def create_pivot(df, index, columns, values):
    pivot_table = df.pivot_table(index=index, columns=columns, values=values, aggfunc=np.mean)
    return pivot_table

# Function to plot heatmaps
def plot_heatmap(pivot_table, title, cmap="YlGnBu"):
    plt.figure(figsize=(10, 8))
    sns.heatmap(pivot_table, annot=True, cmap=cmap, fmt=".2f")
    plt.title(title)
    plt.show()

# Function to create scatter plots
def scatter_plot(data, x_col, y_col, hue_col, title):
    plt.figure(figsize=(10, 6))
    sns.scatterplot(x=x_col, y=y_col, hue=hue_col, data=data)
    plt.title(title)
    plt.show()

# Function to create line plots
def line_plot(data, x_col, y_col, hue_col, title):
    plt.figure(figsize=(10, 6))
    sns.lineplot(x=x_col, y=y_col, hue=hue_col, data=data, marker="o")
    plt.title(title)
    plt.show()

# Heatmaps for Dummy Vectors with and without FC layer
pivot_dummy_with_fc = create_pivot(with_fc_data[with_fc_data['vector_type'] == 'dummy'], 'hidden_units', 'learning_rate', 'accuracy')
plot_heatmap(pivot_dummy_with_fc, 'Heatmap: Accuracy for Dummy Vectors with FC Layer')

pivot_dummy_without_fc = create_pivot(without_fc_data[without_fc_data['vector_type'] == 'dummy'], 'hidden_units', 'learning_rate', 'accuracy')
plot_heatmap(pivot_dummy_without_fc, 'Heatmap: Accuracy for Dummy Vectors without FC Layer')

# Heatmaps for Frequency Vectors with and without FC layer
pivot_frequency_with_fc = create_pivot(with_fc_data[with_fc_data['vector_type'] == 'frequency'], 'hidden_units', 'learning_rate', 'accuracy')
plot_heatmap(pivot_frequency_with_fc, 'Heatmap: Accuracy for Frequency Vectors with FC Layer')

pivot_frequency_without_fc = create_pivot(without_fc_data[without_fc_data['vector_type'] == 'frequency'], 'hidden_units', 'learning_rate', 'accuracy')
plot_heatmap(pivot_frequency_without_fc, 'Heatmap: Accuracy for Frequency Vectors without FC Layer')

# Scatter plot: Learning rate vs. Accuracy with hidden_units as hue (with and without FC)
scatter_plot(with_fc_data, 'learning_rate', 'accuracy', 'hidden_units', 'Scatter Plot: Learning Rate vs. Accuracy with FC Layer')
scatter_plot(without_fc_data, 'learning_rate', 'accuracy', 'hidden_units', 'Scatter Plot: Learning Rate vs. Accuracy without FC Layer')

# Line plot: Accuracy over layers with vector_type as hue (with and without FC)
line_plot(with_fc_data, 'layers', 'accuracy', 'vector_type', 'Line Plot: Accuracy Over Layers with FC Layer')
line_plot(without_fc_data, 'layers', 'accuracy', 'vector_type', 'Line Plot: Accuracy Over Layers without FC Layer')


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
import pandas as pd

# Load the merged diversity scores with age file
diversity_df = pd.read_csv('merged_diversity_scores_with_age.csv')

# Calculate the correlation of each diversity score with Age
age_correlations = []
for column in diversity_df.columns:
    if column not in ['sample', 'Biological Sex', 'Age']:
        corr, p_value = pearsonr(diversity_df['Age'], diversity_df[column])
        age_correlations.append({'Feature': column, 'Pearson Correlation': corr, 'p-value': p_value})

# Convert to DataFrame and filter for significance
age_correlations_df = pd.DataFrame(age_correlations)
significant_age_correlations_df = age_correlations_df[age_correlations_df['p-value'] < 0.05]

# Calculate the difference between Biological Sex and each diversity score
# Convert 'Biological Sex' to numerical (Male=0, Female=1)
diversity_df['Biological Sex Numeric'] = diversity_df['Biological Sex'].apply(lambda x: 1 if x == 'Female' else 0)

sex_correlations = []
for column in diversity_df.columns:
    if column not in ['sample', 'Biological Sex', 'Age', 'Biological Sex Numeric']:
        corr, p_value = pearsonr(diversity_df['Biological Sex Numeric'], diversity_df[column])
        sex_correlations.append({'Feature': column, 'Pearson Correlation': -corr, 'p-value': p_value})  # Inverting correlation

# Convert to DataFrame and filter for significance
sex_correlations_df = pd.DataFrame(sex_correlations)
significant_sex_correlations_df = sex_correlations_df[sex_correlations_df['p-value'] < 0.05]

# Separated Bar Plots
plt.figure(figsize=(10, 8))
sns.barplot(x='Pearson Correlation', y='Feature', data=age_correlations_df, palette='coolwarm')
plt.title('Correlation of Diversity Scores with Age')
plt.xlabel('Pearson Correlation')
plt.ylabel('Feature')
for index, row in age_correlations_df.iterrows():
    significance = "*" if row['p-value'] < 0.05 else ""
    plt.text(row['Pearson Correlation'], index, f"{significance}p-value: {row['p-value']:.2e}", color='black', ha="left", va="center")
plt.show()

plt.figure(figsize=(10, 8))
sns.barplot(x='Pearson Correlation', y='Feature', data=sex_correlations_df, palette='coolwarm')
plt.title('Correlation of Diversity Scores with Biological Sex (Positive for Male)')
plt.xlabel('Pearson Correlation')
plt.ylabel('Feature')
for index, row in sex_correlations_df.iterrows():
    significance = "*" if row['p-value'] < 0.05 else ""
    plt.text(row['Pearson Correlation'], index, f"{significance}p-value: {row['p-value']:.2e}", color='black', ha="left", va="center")
plt.show()

# Scatter Plot with Annotations
plt.figure(figsize=(10, 8))
sns.scatterplot(x='Pearson Correlation', y='Feature', data=age_correlations_df, color='blue', label='Age Correlation')
sns.scatterplot(x='Pearson Correlation', y='Feature', data=sex_correlations_df, color='orange', label='Biological Sex Correlation')
plt.title('Scatter Plot of Correlations with Age and Biological Sex')
plt.xlabel('Pearson Correlation')
plt.ylabel('Feature')
plt.axvline(0, color='black', linestyle='--')
plt.legend()
plt.grid(True)
plt.show()

# Heatmap (correlation matrix for additional insight)
combined_corr = pd.merge(age_correlations_df[['Feature', 'Pearson Correlation']],
                         sex_correlations_df[['Feature', 'Pearson Correlation']],
                         on='Feature',
                         suffixes=('_Age', '_Sex'))

plt.figure(figsize=(8, 10))
sns.heatmap(combined_corr.set_index('Feature').corr(), annot=True, cmap='coolwarm')
plt.title('Correlation Matrix between Age and Biological Sex Correlations')
plt.show()


In [ ]:
import pandas as pd
from scipy.stats import pearsonr
import numpy as np

# Load the data
unique_kmers_sex_df = pd.read_csv('tests_sex_unique_kmers.csv')
abundance_kmers_sex_df = pd.read_csv('tests_sex_abundance_kmers.csv')
correlation_age_abundance_kmers_df = pd.read_csv('correlation_age_abundance_kmers.csv')
correlation_age_unique_kmers_df = pd.read_csv('correlation_age_unique_kmers.csv')

# Filter to only include numeric columns
unique_kmers_sex_df = unique_kmers_sex_df.select_dtypes(include=[np.number])
abundance_kmers_sex_df = abundance_kmers_sex_df.select_dtypes(include=[np.number])
correlation_age_abundance_kmers_df = correlation_age_abundance_kmers_df.select_dtypes(include=[np.number])
correlation_age_unique_kmers_df = correlation_age_unique_kmers_df.select_dtypes(include=[np.number])

# Initialize lists to store the correlation results
unique_kmers_corr = []
abundance_kmers_corr = []

# Calculate correlations between all combinations of 'Unique Kmers Sex' and 'Correlation Age Unique Kmers'
for column1 in unique_kmers_sex_df.columns:
    for column2 in correlation_age_unique_kmers_df.columns:
        corr, p_value = pearsonr(unique_kmers_sex_df[column1], correlation_age_unique_kmers_df[column2])
        unique_kmers_corr.append({'Sex Metric': column1, 'Age Metric': column2, 'Correlation': corr, 'p-value': p_value})

# Calculate correlations between all combinations of 'Abundance Kmers Sex' and 'Correlation Age Abundance Kmers'
for column1 in abundance_kmers_sex_df.columns:
    for column2 in correlation_age_abundance_kmers_df.columns:
        corr, p_value = pearsonr(abundance_kmers_sex_df[column1], correlation_age_abundance_kmers_df[column2])
        abundance_kmers_corr.append({'Sex Metric': column1, 'Age Metric': column2, 'Correlation': corr, 'p-value': p_value})

# Convert results to DataFrames
unique_kmers_corr_df = pd.DataFrame(unique_kmers_corr)
abundance_kmers_corr_df = pd.DataFrame(abundance_kmers_corr)

# Save the correlation results to CSV files
unique_kmers_corr_df.to_csv('unique_kmers_correlation_results_all_combinations.csv', index=False)
abundance_kmers_corr_df.to_csv('abundance_kmers_correlation_results_all_combinations.csv', index=False)

# Display the first few rows of each result
unique_kmers_corr_df.head(), abundance_kmers_corr_df.head()


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Load the data
correlation_age_unique_kmers_df = pd.read_csv('correlation_age_unique_kmers.csv')
tests_sex_unique_kmers_df = pd.read_csv('tests_sex_unique_kmers.csv')

# Sort by the strongest correlation (absolute value) and take the top 100
top_100_kmers = correlation_age_unique_kmers_df.loc[correlation_age_unique_kmers_df['Pearson Correlation'].abs().nlargest(500).index]

# Sort the top 100 k-mers by Pearson Correlation
top_100_kmers_sorted = top_100_kmers.sort_values(by='Pearson Correlation', ascending=False)

# Plot the top 100 correlations from Correlation Age Unique Kmers with correlation on Y-axis
plt.figure(figsize=(12, 8))
sns.barplot(y='Pearson Correlation', x=top_100_kmers_sorted.index, data=top_100_kmers_sorted, palette='viridis')
plt.title('Top 100 Strongest Correlations in Correlation Age Unique Kmers')
plt.ylabel('Pearson Correlation')
plt.xlabel('k-mer')
plt.xticks(rotation=90)
plt.show()

# Look up the corresponding t-statistic values for the same k-mers
top_100_kmer_names = top_100_kmers_sorted.index
t_stat_values = tests_sex_unique_kmers_df.loc[top_100_kmer_names, 't-statistic']

# Plot the t-statistic values for these k-mers with t-statistic on Y-axis
plt.figure(figsize=(12, 8))
sns.barplot(y=t_stat_values, x=top_100_kmer_names, palette='viridis')
plt.title('t-statistic Values for Top 100 k-mers in Tests Sex Unique Kmers')
plt.ylabel('t-statistic')
plt.xlabel('k-mer')
plt.xticks(rotation=90)
plt.show()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

# Assuming the data is already in DataFrame format
# Example data based on your provided sample
unique_kmers_corr_df = pd.DataFrame({
    'Sex Metric': ['t-statistic', 't-statistic', 't p-value', 't p-value', 'Mann-Whitney U'],
    'Age Metric': ['Pearson Correlation', 'Spearman Correlation', 'Pearson Correlation', 'Spearman Correlation', 'Pearson Correlation'],
    'Correlation': [0.749367, 0.744150, 0.556847, 0.555262, 0.774798],
    'p-value': [0.0, 0.0, 0.0, 0.0, 0.0]
})

abundance_kmers_corr_df = pd.DataFrame({
    'Sex Metric': ['t-statistic', 't-statistic', 't p-value', 't p-value', 'Mann-Whitney U'],
    'Age Metric': ['Pearson Correlation', 'Spearman Correlation', 'Pearson Correlation', 'Spearman Correlation', 'Pearson Correlation'],
    'Correlation': [0.245808, -0.019115, 0.025236, -0.015698, 0.000835],
    'p-value': [2.081977e-110, 8.734848e-02, 2.399633e-02, 1.603258e-01, 9.404929e-01]
})

# 1. Heatmap for Unique Kmers Correlations
plt.figure(figsize=(10, 6))
sns.heatmap(unique_kmers_corr_df.pivot(index="Sex Metric", columns="Age Metric", values="Correlation"), annot=True, cmap='coolwarm', center=0)
plt.title('Heatmap of Correlations Between Unique Kmers Sex Metrics and Age Metrics')
plt.show()

# 2. Heatmap for Abundance Kmers Correlations
plt.figure(figsize=(10, 6))
sns.heatmap(abundance_kmers_corr_df.pivot(index="Sex Metric", columns="Age Metric", values="Correlation"), annot=True, cmap='coolwarm', center=0)
plt.title('Heatmap of Correlations Between Abundance Kmers Sex Metrics and Age Metrics')
plt.show()

# 3. Bar Plot for Unique Kmers Correlations
plt.figure(figsize=(12, 8))
sns.barplot(x='Correlation', y='Sex Metric', hue='Age Metric', data=unique_kmers_corr_df, palette='viridis')
plt.title('Bar Plot of Correlations Between Unique Kmers Sex Metrics and Age Metrics')
plt.show()

# 4. Bar Plot for Abundance Kmers Correlations
plt.figure(figsize=(12, 8))
sns.barplot(x='Correlation', y='Sex Metric', hue='Age Metric', data=abundance_kmers_corr_df, palette='viridis')
plt.title('Bar Plot of Correlations Between Abundance Kmers Sex Metrics and Age Metrics')
plt.show()

# 5. Scatter Plot for Unique Kmers vs Abundance Kmers (example of correlation with Pearson)
plt.figure(figsize=(8, 6))
sns.scatterplot(x=unique_kmers_corr_df['Correlation'], y=abundance_kmers_corr_df['Correlation'], hue=unique_kmers_corr_df['Sex Metric'])
plt.title('Scatter Plot of Correlations (Unique Kmers vs Abundance Kmers)')
plt.xlabel('Unique Kmers Correlation')
plt.ylabel('Abundance Kmers Correlation')
plt.axhline(0, color='black', linestyle='--')
plt.axvline(0, color='black', linestyle='--')
plt.grid(True)
plt.show()


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Load the results data
file_path = 'tcr_correlation_ttest_results.csv'  # Replace with your local file path
results_df = pd.read_csv(file_path)

# Filter the 25 strongest correlations with Age (absolute value of Pearson Correlation)
strongest_correlations = results_df[['TCR', 'Pearson Correlation with Age']].copy()
strongest_correlations['abs_corr'] = strongest_correlations['Pearson Correlation with Age'].abs()
strongest_25 = strongest_correlations.nlargest(25, 'abs_corr')

# Plot the 25 strongest correlations
plt.figure(figsize=(10, 8))
sns.barplot(x='Pearson Correlation with Age', y='TCR', data=strongest_25.sort_values(by='Pearson Correlation with Age', ascending=False), palette='viridis')
plt.title('Top 25 Strongest Correlations with Age')
plt.xlabel('Pearson Correlation with Age')
plt.ylabel('TCR')
plt.show()


In [ ]:
# Find the 25 TCRs with the strongest "correlation" to biological sex (using the absolute value of T-Statistic)
strongest_sex_correlations = results_df[['TCR', 'T-Statistic with Biological Sex']].copy()
strongest_sex_correlations['abs_t_stat'] = strongest_sex_correlations['T-Statistic with Biological Sex'].abs()
strongest_25_sex = strongest_sex_correlations.nlargest(100, 'abs_t_stat')

# Plot the T-Statistic for Biological Sex for these top 25 TCRs
plt.figure(figsize=(10, 8))
sns.barplot(x='T-Statistic with Biological Sex', y='TCR', data=strongest_25_sex.sort_values(by='T-Statistic with Biological Sex', ascending=False), palette='viridis')
plt.title('Top 25 Strongest T-Statistics with Biological Sex')
plt.xlabel('T-Statistic with Biological Sex')
plt.ylabel('TCR')
plt.show()

# Now plot the Pearson Correlation with Age for the same 25 TCRs (no sorting)
top_25_tcrs_sex = strongest_25_sex['TCR']
age_correlations_for_sex_tcrs = results_df[results_df['TCR'].isin(top_25_tcrs_sex)][['TCR', 'Pearson Correlation with Age', 'p-value with Age']]

plt.figure(figsize=(10, 8))
sns.barplot(x='Pearson Correlation with Age', y='TCR', data=age_correlations_for_sex_tcrs, palette='viridis')
plt.title('Pearson Correlation with Age for Top 25 TCRs (Based on Biological Sex T-Statistic)')
plt.xlabel('Pearson Correlation with Age')
plt.ylabel('TCR')
plt.show()


In [ ]:
# Calculate the correlation between "Pearson Correlation with Age" and "T-Statistic with Biological Sex"

# Extract the relevant columns
correlation_columns = results_df[['Pearson Correlation with Age', 'T-Statistic with Biological Sex']]

# Calculate the correlation matrix
correlation_matrix = correlation_columns.corr()

# Display the correlation matrix
correlation_matrix


# new section

In [ ]:
!pip install ruptures

In [ ]:
import pandas as pd
import numpy as np
from kneed import KneeLocator
import matplotlib.pyplot as plt

# Load the data
file_path = 'subset123_tcr_counts_and_cloness.csv'
data = pd.read_csv(file_path)

# Extract the 'consistency_area' column and sort it
consistency_area = data['consistency_area'].sort_values(ascending=False).reset_index(drop=True)

# Find the elbow point
x = np.arange(1, len(consistency_area) + 1)
y = consistency_area.values

kneedle = KneeLocator(x, y, curve='convex', direction='decreasing')

# Plot the data with the knee point
plt.figure(figsize=(10, 6))
plt.plot(x, y, label='Consistency Area')
plt.axvline(x=kneedle.elbow, color='r', linestyle='--', label='Elbow Point')
plt.xlabel('Index')
plt.ylabel('Consistency Area')
plt.title('Elbow Point Detection')
plt.legend()
plt.grid(True)
plt.show()

# Get the elbow index
elbow_index = kneedle.elbow
# Get the index slightly after the elbow point
slightly_after_elbow_index = elbow_index + 1 if elbow_index + 1 < len(x) else elbow_index

(elbow_index, slightly_after_elbow_index)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from kneed import KneeLocator
import ruptures as rpt

# Load the data
file_path = 'subset123_tcr_counts_and_cloness.csv'
data = pd.read_csv(file_path)

# Extract the 'consistency_area' column and sort it
consistency_area = data['consistency_area'].sort_values(ascending=False).reset_index(drop=True)

# Find the elbow point
x = np.arange(1, len(consistency_area) + 1)
y = consistency_area.values

kneedle = KneeLocator(x, y, curve='convex', direction='decreasing')
elbow_index = kneedle.elbow

# Calculate the first derivative (slope)
dy = np.gradient(y)

# Calculate the second derivative (change in slope)
d2y = np.gradient(dy)

# Find the point where the second derivative is maximized
change_point_index_slope = np.argmax(d2y)

# Apply change point detection using the Pelt method
model = rpt.Pelt(model="rbf").fit(y)
breakpoints = model.predict(pen=10)
change_point_indices_pelt = breakpoints[:-1]  # Remove the last point, which is the end of the series

# Plot the data with the elbow point, slope change point, and Pelt method change points
plt.figure(figsize=(10, 6))
plt.plot(x, y, label='Consistency Area')

# Plot the elbow point
plt.axvline(x=elbow_index, color='r', linestyle='--', label='Elbow Point')

# Plot the slope change point
plt.axvline(x=change_point_index_slope, color='g', linestyle='--', label='Slope Change Point')

# Plot the Pelt method change points
for bp in change_point_indices_pelt:
    plt.axvline(x=bp, color='b', linestyle='--', label='Pelt Change Point' if bp == change_point_indices_pelt[0] else "")

plt.xlabel('Index')
plt.ylabel('Consistency Area')
plt.title('Elbow Point and Change Point Detection')
plt.legend()
plt.grid(True)
plt.show()

(elbow_index, change_point_index_slope, change_point_indices_pelt)


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load the data
file_path = 'subset123_tcr_counts_and_cloness.csv'
data = pd.read_csv(file_path)

# Extract the 'consistency_area' column and sort it
consistency_area = data['consistency_area'].sort_values(ascending=False).reset_index(drop=True)

# Calculate the first derivative (slope)
x = np.arange(1, len(consistency_area) + 1)
y = consistency_area.values
dy = np.gradient(y)

# Find the indices where the slope is closest to -1
target_slope = -1
slope_diff = np.abs(dy - target_slope)
closest_slope_indices = np.argsort(slope_diff)[:5]

# Plot the data with the points where the slope is closest to -1
plt.figure(figsize=(10, 6))
plt.plot(x, y, label='Consistency Area')
for idx in closest_slope_indices:
    plt.axvline(x=idx, color='r', linestyle='--', label=f'Closest to Slope -1 (Index {idx})')
plt.xlabel('Index')
plt.ylabel('Consistency Area')
plt.title('Top 5 Points where Slope is Closest to -1')
plt.legend()
plt.grid(True)
plt.show()

closest_slope_indices



In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from math import pi
import os

# Define the path where the results are stored
results_path = './'  # Adjust this to your actual path if different

# Function to process the 'file' column to extract the group names
def process_file_column(file_column):
    return file_column.apply(lambda x: 'consistency1' if 'consistency_area_top' in x
                             else ('consistency2' if 'consistency_area2_top' in x
                                   else x.split('_')[0]))

# Load the data for age classification
age_df = pd.read_csv(os.path.join(results_path, 'model_results_binary_age_classification.csv'))
age_df['file'] = process_file_column(age_df['file'])

# Load the data for sex classification
sex_df = pd.read_csv(os.path.join(results_path, 'model_results_binary_sex_classification.csv'))
sex_df['file'] = process_file_column(sex_df['file'])

# Ensure the 'accuracy' column is numeric
age_df['accuracy'] = pd.to_numeric(age_df['accuracy'], errors='coerce')
sex_df['accuracy'] = pd.to_numeric(sex_df['accuracy'], errors='coerce')

# Drop rows with NaN values in 'accuracy'
age_df = age_df.dropna(subset=['accuracy'])
sex_df = sex_df.dropna(subset=['accuracy'])

# Define a color palette for different methods
color_map = {
    'average': '#1f77b4',   # A professional blue shade
    'variance': '#1f77b4',  # Same blue for variance
    'consistency1': '#2ca02c',  # A professional green shade
    'consistency2': '#2ca02c',  # Same green for consistency
    'explosion': '#17becf',  # A different shade of blue-green
    'publicity': '#7f7f7f',  # A medium grey
    'random': '#ff7f0e'  # A professional orange
}

# Separate the plots for Age and Sex classification
def plot_classification_results(df, title):
    # Sort the DataFrame by accuracy for plotting
    df_sorted = df.sort_values(by='accuracy', ascending=False)

    # 1. Line Plot
    plt.figure(figsize=(12, 6))
    for group in df['file'].unique():
        group_data = df[df['file'] == group]
        plt.plot(group_data['original_num_features'], group_data['accuracy'], marker='o', label=group)
    plt.xlabel('Number of Features')
    plt.ylabel('Accuracy')
    plt.title(f'Accuracy vs Number of Features ({title})')
    plt.legend()
    plt.grid(True)
    plt.show()

    # 2. Box Plot
    plt.figure(figsize=(12, 6))
    sns.boxplot(x='original_num_features', y='accuracy', hue='file', data=df)
    plt.xlabel('Number of Features')
    plt.ylabel('Accuracy')
    plt.title(f'Box Plot of Accuracy for Different Numbers of Features ({title})')
    plt.legend()
    plt.grid(True)
    plt.show()

    # 3. Bar Plot
    plt.figure(figsize=(12, 6))
    sns.barplot(x='original_num_features', y='accuracy', hue='file', data=df_sorted)
    plt.xlabel('Number of Features')
    plt.ylabel('Accuracy')
    plt.title(f'Bar Plot of Accuracy for Different Numbers of Features ({title})')
    plt.legend()
    plt.grid(True)
    plt.show()

    # 4. Scatter Plot
    plt.figure(figsize=(12, 6))
    for group in df['file'].unique():
        group_data = df[df['file'] == group]
        plt.scatter(group_data['original_num_features'], group_data['accuracy'], label=group, alpha=0.7)
    plt.xlabel('Number of Features')
    plt.ylabel('Accuracy')
    plt.title(f'Scatter Plot of Accuracy vs Number of Features ({title})')
    plt.legend()
    plt.grid(True)
    plt.show()


    # 6. Heatmap
    pivot_table = df.pivot_table(values='accuracy', index='file', columns='original_num_features')
    plt.figure(figsize=(12, 6))
    sns.heatmap(pivot_table, annot=True, cmap="YlGnBu")
    plt.title(f'Heatmap of Accuracy Across Different Features ({title})')
    plt.ylabel('File')
    plt.xlabel('Number of Features')
    plt.show()

    # 7. Point Plot
    plt.figure(figsize=(12, 6))
    sns.pointplot(x='original_num_features', y='accuracy', hue='file', data=df)
    plt.xlabel('Number of Features')
    plt.ylabel('Accuracy')
    plt.title(f'Point Plot of Accuracy Across Different Features ({title})')
    plt.legend()
    plt.grid(True)
    plt.show()

    # 8. Violin Plot
    plt.figure(figsize=(12, 6))
    sns.violinplot(x='original_num_features', y='accuracy', hue='file', data=df, split=True)
    plt.xlabel('Number of Features')
    plt.ylabel('Accuracy')
    plt.title(f'Violin Plot of Accuracy Across Different Features ({title})')
    plt.legend()
    plt.grid(True)
    plt.show()

# Plotting for Age Classification
plot_classification_results(age_df, 'Age Classification')

# Plotting for Sex Classification
plot_classification_results(sex_df, 'Sex Classification')


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

# Define the path where the results are stored
results_path = './'  # Adjust this to your actual path if different

# Function to process the 'file' column to extract the group names
def process_file_column(file_column):
    return file_column.apply(lambda x: 'consistency1' if 'consistency_area_top' in x
                             else ('consistency2' if 'consistency_area2_top' in x
                                   else x.split('_')[0]))

# Load the data for age classification
age_df = pd.read_csv(os.path.join(results_path, 'model_results_binary_age_classification.csv'))
age_df['file'] = process_file_column(age_df['file'])

# Load the data for sex classification
sex_df = pd.read_csv(os.path.join(results_path, 'model_results_binary_sex_classification.csv'))
sex_df['file'] = process_file_column(sex_df['file'])

# Ensure the 'accuracy' column is numeric
age_df['accuracy'] = pd.to_numeric(age_df['accuracy'], errors='coerce')
sex_df['accuracy'] = pd.to_numeric(sex_df['accuracy'], errors='coerce')

# Drop rows with NaN values in 'accuracy'
age_df = age_df.dropna(subset=['accuracy'])
sex_df = sex_df.dropna(subset=['accuracy'])

# Define a color palette for different methods
color_map = {
    'average': '#dbde18',   # A professional blue shade
    'variance': '#ff7f0e',  # Orange
    'consistency1': '#0248f7',  # A professional green shade
    'consistency2': '#0332a6',  # Slightly different shade of green
    'explosion': '#2999e3',  # Red
    'publicity': '#39c72c'  # A medium grey
}

# Plotting function for Line Plot
def plot_line_results(df, title):
    # Sort the DataFrame by file and original_num_features for consistency in plotting
    df = df.sort_values(by=['file', 'original_num_features'])

    # Get unique feature counts and map them to categorical positions
    unique_features = sorted(df['original_num_features'].unique())
    feature_pos_map = {feature: i for i, feature in enumerate(unique_features)}

    # Line Plot
    plt.figure(figsize=(12, 6))
    for group in df['file'].unique():
        group_data = df[df['file'] == group]
        plt.plot([feature_pos_map[feature] for feature in group_data['original_num_features']],
                 group_data['accuracy'], marker='o', label=group, color=color_map[group])

    plt.xticks(range(len(unique_features)), unique_features)
    plt.xlabel('Number of Features')
    plt.ylabel('Accuracy')
    plt.title(f'Accuracy vs Number of Features ({title})')
    plt.legend()
    plt.grid(True)
    plt.show()

# Plotting for Age Classification
plot_line_results(age_df, 'Age Classification')

# Plotting for Sex Classification
plot_line_results(sex_df, 'Sex Classification')


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from math import pi
import os

# Define the path where the results are stored
results_path = './'  # Adjust this to your actual path if different

# Function to process the 'file' column to extract the group names
def process_file_column(file_column):
    return file_column.apply(lambda x: 'consistency1' if 'consistency_area_top' in x
                             else ('consistency2' if 'consistency_area2_top' in x
                                   else x.split('_')[0]))

# Load the data for age classification
age_df = pd.read_csv(os.path.join(results_path, 'model_results_binary_age_classification.csv'))
age_df['file'] = process_file_column(age_df['file'])

# Load the data for sex classification
sex_df = pd.read_csv(os.path.join(results_path, 'model_results_binary_sex_classification.csv'))
sex_df['file'] = process_file_column(sex_df['file'])

# Ensure the 'accuracy' column is numeric
age_df['accuracy'] = pd.to_numeric(age_df['accuracy'], errors='coerce')
sex_df['accuracy'] = pd.to_numeric(sex_df['accuracy'], errors='coerce')

# Drop rows with NaN values in 'accuracy'
age_df = age_df.dropna(subset=['accuracy'])
sex_df = sex_df.dropna(subset=['accuracy'])

# Define a color palette for different methods with close shades for consistency groups
color_map = {
    'average': '#2ca02c',   # A professional blue shade
    'variance': '#ff7f0e',  # Orange
    'consistency1': '#303866',  # Green
    'consistency2': '#8691cf',  # Same green for consistency
    'explosion': '#d62728',  # Red
    'publicity': '#9467bd',  # Purple
    'random': '#8c564b'  # Brown
}

# Modify the color slightly for consistency2 to make it close but distinct
color_map['consistency2'] = '#28a02c'  # Slightly different green

# Define the order of groups
group_order = ['consistency1', 'consistency2', 'explosion', 'publicity', 'average', 'variance']

# Plotting function
def plot_classification_results(df, title):
    # Ensure correct order of groups
    df['file'] = pd.Categorical(df['file'], categories=group_order, ordered=True)

    # Sort the DataFrame by accuracy for plotting
    df_sorted = df.sort_values(by=['file', 'original_num_features'])

    # 1. Line Plot
    plt.figure(figsize=(12, 6))
    for group in group_order:
        group_data = df_sorted[df_sorted['file'] == group]
        if not group_data.empty:
            plt.plot(group_data['original_num_features'], group_data['accuracy'], marker='o', label=group, color=color_map[group])
    plt.xlabel('Number of Features')
    plt.ylabel('Accuracy')
    plt.title(f'Accuracy vs Number of Features ({title})')
    plt.legend()
    plt.grid(True)
    plt.show()

    # 2. Box Plot
    plt.figure(figsize=(12, 6))
    sns.boxplot(x='original_num_features', y='accuracy', hue='file', data=df_sorted, palette=color_map)
    plt.xlabel('Number of Features')
    plt.ylabel('Accuracy')
    plt.title(f'Box Plot of Accuracy for Different Numbers of Features ({title})')
    plt.legend()
    plt.grid(True)
    plt.show()

    # 3. Bar Plot
    plt.figure(figsize=(12, 6))
    sns.barplot(x='original_num_features', y='accuracy', hue='file', data=df_sorted, palette=color_map)
    plt.xlabel('Number of Features')
    plt.ylabel('Accuracy')
    plt.title(f'Bar Plot of Accuracy for Different Numbers of Features ({title})')
    plt.legend()
    plt.grid(True)
    plt.show()

    # 4. Scatter Plot
    plt.figure(figsize=(12, 6))
    for group in group_order:
        group_data = df_sorted[df_sorted['file'] == group]
        if not group_data.empty:
            plt.scatter(group_data['original_num_features'], group_data['accuracy'], label=group, alpha=0.7, color=color_map[group])
    plt.xlabel('Number of Features')
    plt.ylabel('Accuracy')
    plt.title(f'Scatter Plot of Accuracy vs Number of Features ({title})')
    plt.legend()
    plt.grid(True)
    plt.show()

    # 5. Heatmap
    pivot_table = df.pivot_table(values='accuracy', index='file', columns='original_num_features')
    plt.figure(figsize=(12, 6))
    sns.heatmap(pivot_table, annot=True, cmap="YlGnBu")
    plt.title(f'Heatmap of Accuracy Across Different Features ({title})')
    plt.ylabel('File')
    plt.xlabel('Number of Features')
    plt.show()

    # 6. Point Plot
    plt.figure(figsize=(12, 6))
    sns.pointplot(x='original_num_features', y='accuracy', hue='file', data=df_sorted, palette=color_map)
    plt.xlabel('Number of Features')
    plt.ylabel('Accuracy')
    plt.title(f'Point Plot of Accuracy Across Different Features ({title})')
    plt.legend()
    plt.grid(True)
    plt.show()

    # 7. Violin Plot
    plt.figure(figsize=(12, 6))
    sns.violinplot(x='original_num_features', y='accuracy', hue='file', data=df_sorted, split=True, palette=color_map)
    plt.xlabel('Number of Features')
    plt.ylabel('Accuracy')
    plt.title(f'Violin Plot of Accuracy Across Different Features ({title})')
    plt.legend()
    plt.grid(True)
    plt.show()

    # 8. Area Plot
    plt.figure(figsize=(12, 6))
    for group in group_order:
        group_data = df_sorted[df_sorted['file'] == group]
        if not group_data.empty:
            plt.fill_between(group_data['original_num_features'], group_data['accuracy'], label=group, alpha=0.4, color=color_map[group])
    plt.xlabel('Number of Features')
    plt.ylabel('Accuracy')
    plt.title(f'Area Plot of Accuracy Across Different Features ({title})')
    plt.legend()
    plt.grid(True)
    plt.show()

    # 9. Strip Plot
    plt.figure(figsize=(12, 6))
    sns.stripplot(x='original_num_features', y='accuracy', hue='file', data=df_sorted, jitter=True, palette=color_map)
    plt.xlabel('Number of Features')
    plt.ylabel('Accuracy')
    plt.title(f'Strip Plot of Accuracy Across Different Features ({title})')
    plt.legend()
    plt.grid(True)
    plt.show()

    # 10. Density Plot
    plt.figure(figsize=(12, 6))
    for group in group_order:
        group_data = df_sorted[df_sorted['file'] == group]
        if not group_data.empty:
            sns.kdeplot(group_data['accuracy'], label=group, fill=True, alpha=0.3, color=color_map[group])
    plt.xlabel('Accuracy')
    plt.ylabel('Density')
    plt.title(f'Density Plot of Accuracy ({title})')
    plt.legend()
    plt.grid(True)
    plt.show()

    # 11. Swarm Plot
    plt.figure(figsize=(12, 6))
    sns.swarmplot(x='original_num_features', y='accuracy', hue='file', data=df_sorted, palette=color_map)
    plt.xlabel('Number of Features')
    plt.ylabel('Accuracy')
    plt.title(f'Swarm Plot of Accuracy Across Different Features ({title})')
    plt.legend()
    plt.grid(True)
    plt.show()

    # 12. Histogram
    plt.figure(figsize=(12, 6))
    for group in group_order:
        group_data = df_sorted[df_sorted['file'] == group]
        if not group_data.empty:
            plt.hist(group_data['accuracy'], bins=20, alpha=0.5, label=group, color=color_map[group])
    plt.xlabel('Accuracy')
    plt.ylabel('Frequency')
    plt.title(f'Histogram of Accuracy ({title})')
    plt.legend()
    plt.grid(True)
    plt.show()

    # 13. Facet Grid
    g = sns.FacetGrid(df_sorted, col="file", col_wrap=4, height=4, palette=color_map)
    g.map(sns.histplot, "accuracy")
    g.set_axis_labels('Accuracy', 'Count')
    g.fig.suptitle(f'Facet Grid of Accuracy Across Different Features ({title})')
    g.fig.tight_layout(rect=[0, 0, 1, 0.95])
    plt.show()

    # 14. Bubble Plot
    plt.figure(figsize=(12, 6))
    for group in group_order:
        group_data = df_sorted[df_sorted['file'] == group]
        if not group_data.empty:
            plt.scatter(group_data['original_num_features'], group_data['accuracy'],
                        s=group_data['accuracy'] * 200, alpha=0.5, label=group, color=color_map[group])
    plt.xlabel('Number of Features')
    plt.ylabel('Accuracy')
    plt.title(f'Bubble Plot of Accuracy ({title})')
    plt.legend()
    plt.grid(True)
    plt.show()

# Plotting for Age Classification
plot_classification_results(age_df, 'Age Classification')

# Plotting for Sex Classification
plot_classification_results(sex_df, 'Sex Classification')


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from math import pi

# Define the path where the results are stored
results_path = './'  # Adjust this to your actual path if different

# Function to process the 'file' column to extract the group names
def process_file_column(file_column):
    return file_column.apply(lambda x: 'consistency1' if 'consistency_area_top' in x
                             else ('consistency2' if 'consistency_area2_top' in x
                                   else x.split('_')[0]))

# Load the data for age classification
age_df = pd.read_csv(os.path.join(results_path, 'model_results_binary_age_classification.csv'))
age_df['file'] = process_file_column(age_df['file'])

# Ensure the 'original_num_features' and 'accuracy' columns are numeric
age_df['accuracy'] = pd.to_numeric(age_df['accuracy'], errors='coerce')
age_df['original_num_features'] = pd.to_numeric(age_df['original_num_features'], errors='coerce')

# Drop rows with NaN values in 'accuracy' or 'original_num_features'
age_df = age_df.dropna(subset=['accuracy', 'original_num_features'])

# Debugging: Check if any non-numeric data slipped through
print("Data types after conversion:")
print(age_df.dtypes)
print("\nData preview after cleaning:")
print(age_df.head())

# Radar Chart
plt.figure(figsize=(8, 8))
ax = plt.subplot(111, polar=True)

# Get the list of unique 'original_num_features' for the angles
unique_features = sorted(age_df['original_num_features'].unique())
num_vars = len(unique_features)
angles = [n / float(num_vars) * 2 * pi for n in range(num_vars)]
angles += angles[:1]  # Complete the circle

# Iterate over each unique group in the file column
for group in age_df['file'].unique():
    # Group data by the original number of features and compute the mean accuracy
    try:
        group_data = age_df[age_df['file'] == group].groupby('original_num_features').mean().reindex(unique_features).reset_index()
    except Exception as e:
        print(f"Error processing group {group}: {e}")
        continue

    print(f"Group: {group}")
    print(group_data)

    # Ensure that both original_num_features and accuracy are lists
    values = group_data['accuracy'].tolist()
    if len(values) == 0:
        print(f"No data available for group {group}. Skipping.")
        continue

    values += values[:1]  # Repeat the first value to close the circle
    angles = [n / float(len(values)) * 2 * pi for n in range(len(values))]
    angles += angles[:1]  # Repeat the first angle to close the circle

    # Plotting each group on the radar chart
    ax.plot(angles, values, linewidth=1, linestyle='solid', label=group)
    ax.fill(angles, values, alpha=0.1)

# Setting x-ticks to the feature numbers and closing the circle
plt.xticks(angles[:-1], [str(int(num)) for num in unique_features])

# Set title and legend
ax.set_title('Radar Chart of Accuracy (Age Classification)')
plt.legend(loc='upper right', bbox_to_anchor=(1.1, 1.1))

# Display the plot
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from math import pi
import os

# Define the path where the results are stored
results_path = './'  # Adjust this to your actual path if different

# Function to process the 'file' column to extract the group names
def process_file_column(file_column):
    return file_column.apply(lambda x: 'consistency1' if 'consistency_area_top' in x
                             else ('consistency2' if 'consistency_area2_top' in x
                                   else x.split('_')[0]))

# Load the data for age classification
age_df = pd.read_csv(os.path.join(results_path, 'model_results_binary_age_classification.csv'))
age_df['file'] = process_file_column(age_df['file'])

# Ensure the 'original_num_features' and 'accuracy' columns are numeric
age_df['accuracy'] = pd.to_numeric(age_df['accuracy'], errors='coerce')
age_df['original_num_features'] = pd.to_numeric(age_df['original_num_features'], errors='coerce')

# Drop rows with NaN values in 'accuracy' or 'original_num_features'
age_df = age_df.dropna(subset=['accuracy', 'original_num_features'])

# Define a distinct color map for each group

# Define a color palette for different methods
color_map = {
    'average': '#dbde18',   # A professional blue shade
    'variance': '#ff7f0e',  # Orange
    'consistency1': '#0248f7',  # A professional green shade
    'consistency2': '#0332a6',  # Slightly different shade of green
    'explosion': '#2999e3',  # Red
    'publicity': '#39c72c'  # A medium grey
}


# Radar Chart
plt.figure(figsize=(8, 8))
ax = plt.subplot(111, polar=True)

# Get the list of unique 'original_num_features' for the angles
unique_features = sorted(age_df['original_num_features'].unique())
num_vars = len(unique_features)
angles = [n / float(num_vars) * 2 * pi for n in range(num_vars)]
angles += angles[:1]  # Complete the circle

# Iterate over each unique group in the file column
for group in age_df['file'].unique():
    # Filter data for the group
    group_data = age_df[age_df['file'] == group]

    # Values for the radar chart
    values = group_data.set_index('original_num_features').reindex(unique_features)['accuracy'].fillna(0).tolist()
    values += values[:1]  # Repeat the first value to close the circle

    # Plotting each group on the radar chart
    ax.plot(angles, values, linewidth=2, linestyle='solid', label=group, color=color_map[group])  # Thicker line
    ax.fill(angles, values, alpha=0.1, color=color_map[group])  # Stronger fill color

# Set the range for the radial axis (0.5 to 1)
ax.set_ylim(0.5, 1)

# Set the radial ticks
ax.set_yticks([0.5, 0.6, 0.7, 0.8, 0.9, 1.0])
ax.set_yticklabels(['0.5', '0.6', '0.7', '0.8', '0.9', '1.0'], color="grey", size=15)

# Setting x-ticks to the feature numbers and closing the circle
plt.xticks(angles[:-1], [str(int(num)) for num in unique_features])

# Set title and legend
ax.set_title('classify group age Accuracy by Number of Features')
ax.legend(loc='upper right', bbox_to_anchor=(1.1, 1.1))

# Display the plot
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Load the data
file_path = 'tcrs123.xlsx'
df = pd.read_excel(file_path)

# Extract the top 25 and bottom 25 rows based on sample_counts
df['min_count'] = df['sample_counts'].apply(lambda x: min(eval(x)))
df_sorted = df.sort_values(by='min_count')
top_25 = df_sorted.head(25)
bottom_25 = df_sorted.tail(25)

# Define a function to extract and prepare the data for plotting
def prepare_data(df):
    all_values = []
    for index, row in df.iterrows():
        sample_counts = eval(row['sample_counts'])
        all_values.extend(sample_counts)
    return all_values

# Prepare data for top 25 and bottom 25
top_25_data = prepare_data(top_25)
bottom_25_data = prepare_data(bottom_25)

# Plot 1: Density Plot
plt.figure(figsize=(10, 6))
sns.kdeplot(top_25_data, label='Top 25', color='blue')
sns.kdeplot(bottom_25_data, label='Bottom 25', color='red')
plt.xlabel('Clonality Count')
plt.ylabel('Density')
plt.title('Density Plot of Clonality Counts')
plt.legend()
plt.grid(True)
plt.show()

# Plot 2: Box Plot
plt.figure(figsize=(10, 6))
plt.boxplot([top_25_data, bottom_25_data], labels=['Top 25', 'Bottom 25'])
plt.yscale('log')
plt.ylabel('Clonality Count')
plt.title('Box Plot of Clonality Counts')
plt.grid(True)
plt.show()

# Plot 3: Violin Plot
plt.figure(figsize=(10, 6))
sns.violinplot(data=[top_25_data, bottom_25_data], scale='width')
plt.yscale('log')
plt.xticks([0, 1], ['Top 25', 'Bottom 25'])
plt.ylabel('Clonality Count')
plt.title('Violin Plot of Clonality Counts')
plt.grid(True)
plt.show()

# Plot 4: CDF Plot
plt.figure(figsize=(10, 6))
top_25_sorted = np.sort(top_25_data)
bottom_25_sorted = np.sort(bottom_25_data)
plt.plot(top_25_sorted, np.arange(1, len(top_25_sorted) + 1) / len(top_25_sorted), label='Top 25')
plt.plot(bottom_25_sorted, np.arange(1, len(bottom_25_sorted) + 1) / len(bottom_25_sorted), label='Bottom 25')
plt.xlabel('Clonality Count')
plt.ylabel('CDF')
plt.title('CDF Plot of Clonality Counts')
plt.legend()
plt.grid(True)
plt.show()

# Plot 5: Scatter Plot
plt.figure(figsize=(10, 6))
plt.scatter(range(len(top_25_data)), top_25_data, label='Top 25', alpha=0.5)
plt.scatter(range(len(bottom_25_data)), bottom_25_data, label='Bottom 25', alpha=0.5)
plt.yscale('log')
plt.xlabel('Index')
plt.ylabel('Clonality Count')
plt.title('Scatter Plot of Clonality Counts')
plt.legend()
plt.grid(True)
plt.show()

# Non-log versions of plots
# Box Plot (Non-log)
plt.figure(figsize=(10, 6))
plt.boxplot([top_25_data, bottom_25_data], labels=['Top 25', 'Bottom 25'])
plt.ylabel('Clonality Count')
plt.title('Box Plot of Clonality Counts (Non-log)')
plt.grid(True)
plt.show()

# Violin Plot (Non-log)
plt.figure(figsize=(10, 6))
sns.violinplot(data=[top_25_data, bottom_25_data], scale='width')
plt.xticks([0, 1], ['Top 25', 'Bottom 25'])
plt.ylabel('Clonality Count')
plt.title('Violin Plot of Clonality Counts (Non-log)')
plt.grid(True)
plt.show()

# CDF Plot (Non-log)
plt.figure(figsize=(10, 6))
plt.plot(top_25_sorted, np.arange(1, len(top_25_sorted) + 1) / len(top_25_sorted), label='Top 25')
plt.plot(bottom_25_sorted, np.arange(1, len(bottom_25_sorted) + 1) / len(bottom_25_sorted), label='Bottom 25')
plt.xlabel('Clonality Count')
plt.ylabel('CDF')
plt.title('CDF Plot of Clonality Counts (Non-log)')
plt.legend()
plt.grid(True)
plt.show()

# Scatter Plot (Non-log)
plt.figure(figsize=(10, 6))
plt.scatter(range(len(top_25_data)), top_25_data, label='Top 25', alpha=0.5)
plt.scatter(range(len(bottom_25_data)), bottom_25_data, label='Bottom 25', alpha=0.5)
plt.xlabel('Index')
plt.ylabel('Clonality Count')
plt.title('Scatter Plot of Clonality Counts (Non-log)')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Load the data
file_path = 'tcrs123.xlsx'
df = pd.read_excel(file_path)

# Extract the top 25 and bottom 25 rows based on sample_counts
df['min_count'] = df['sample_counts'].apply(lambda x: min(eval(x)))
df_sorted = df.sort_values(by='min_count')
top_25 = df_sorted.head(25)
bottom_25 = df_sorted.tail(25)

# Define a function to extract and prepare the data for plotting
def prepare_data(df):
    all_values = []
    for index, row in df.iterrows():
        sample_counts = eval(row['sample_counts'])
        all_values.extend(sample_counts)
    return all_values

# Prepare data for top 25 and bottom 25
top_25_data = prepare_data(top_25)
bottom_25_data = prepare_data(bottom_25)

# Plot 1: Stacked Bar Plot
plt.figure(figsize=(10, 6))
bins = np.logspace(np.log10(min(top_25_data + bottom_25_data)), np.log10(max(top_25_data + bottom_25_data)), 50)
plt.hist([top_25_data, bottom_25_data], bins=bins, stacked=True, label=['Top 25', 'Bottom 25'])
plt.xscale('log')
plt.xlabel('Clonality Count')
plt.ylabel('Frequency')
plt.title('Stacked Bar Plot of Clonality Counts')
plt.legend()
plt.grid(True)
plt.show()

# Plot 2: Heatmap
plt.figure(figsize=(10, 6))
heatmap_data = np.array([top_25_data + bottom_25_data]).T
sns.heatmap(heatmap_data, cmap='viridis', cbar_kws={'label': 'Clonality Count'})
plt.title('Heatmap of Clonality Counts')
plt.ylabel('Sample Index')
plt.xlabel('Clonality Count')
plt.grid(False)
plt.show()

# Plot 3: Histogram with KDE Overlay
plt.figure(figsize=(10, 6))
sns.histplot(top_25_data, kde=True, color='blue', label='Top 25', bins=50)
sns.histplot(bottom_25_data, kde=True, color='red', label='Bottom 25', bins=50)
plt.xlabel('Clonality Count')
plt.ylabel('Density')
plt.title('Histogram with KDE Overlay')
plt.legend()
plt.grid(True)
plt.show()

# Plot 4: Ridgeline Plot
plt.figure(figsize=(10, 6))
sns.violinplot(data=[top_25_data, bottom_25_data], scale='width')
plt.yscale('log')
plt.xticks([0, 1], ['Top 25', 'Bottom 25'])
plt.ylabel('Clonality Count')
plt.title('Ridgeline Plot of Clonality Counts')
plt.grid(True)
plt.show()

# Plot 5: Swarm Plot
plt.figure(figsize=(10, 6))
sns.swarmplot(data=[top_25_data, bottom_25_data], palette=['blue', 'red'], size=2)
plt.yscale('log')
plt.xticks([0, 1], ['Top 25', 'Bottom 25'])
plt.ylabel('Clonality Count')
plt.title('Swarm Plot of Clonality Counts')
plt.grid(True)
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial import distance

# Load the CSV file into a DataFrame
file_path = 'final_tcr_selection_adjusted_distribution_with_random_groups.csv'
df = pd.read_csv(file_path)

# Extract relevant columns
df = df[['amino_acid', 'selection_group']]

# Dynamically generate a list of groups including all random groups
unique_groups = df['selection_group'].unique()
groups = {group: df[df['selection_group'] == group]['amino_acid'].dropna() for group in unique_groups}

# Process and remove sequences containing '*' for each group
for group_name in groups:
    groups[group_name] = groups[group_name][~groups[group_name].str.contains('\*', na=False)].tolist()

def amino_acid_presence_at_position(group_seqs, position):
    amino_acids = 'ACDEFGHIKLMNPQRSTVWY'  # List of all 20 standard amino acids
    presence_counts = {aa: 0 for aa in amino_acids}
    for seq in group_seqs:
        if len(seq) >= position:
            aa = seq[position - 1]
            presence_counts[aa] += 1
    return presence_counts

# Define groups of interest
target_groups = ['consistency_area_top', 'consistency_area2_top', 'average_count_top', 'variance_score_top']
random_groups = [f'random_selection_{i}' for i in range(1, 6)]

# Initialize a dictionary to hold distances
distances = {tg: {rg: [] for rg in random_groups} for tg in target_groups}

# Loop through positions 1 to 20
for position in range(1, 21):
    aa_presence = {group: amino_acid_presence_at_position(seqs, position) for group, seqs in groups.items()}
    data = pd.DataFrame.from_dict(aa_presence, orient='index').fillna(0)

    # Calculate distances from each target group to each random group
    for tg in target_groups:
        for rg in random_groups:
            if tg in data.index and rg in data.index:
                distance_value = distance.euclidean(data.loc[tg], data.loc[rg])
                distances[tg][rg].append(distance_value)

# Plotting the distances
plt.figure(figsize=(14, 8))

# Define colors for different groups
colors = {
    'consistency_area_top': 'red',
    'consistency_area2_top': 'orange',
    'average_count_top': 'blue',
    'variance_score_top': 'green'
}

# Plot distances for each target group
for tg in target_groups:
    for rg in random_groups:
        plt.plot(range(1, 21), distances[tg][rg], marker='o', label=f'{tg} to {rg}', color=colors[tg], alpha=0.6)

plt.xlabel('Position', fontsize=14)
plt.ylabel('Euclidean Distance', fontsize=14)
plt.title('Euclidean Distance from Target Groups to Random Groups Across Positions', fontsize=18)
plt.legend(loc='upper right', bbox_to_anchor=(1.2, 1))

# Set the x-axis grid with a major interval of 1
plt.xticks(range(1, 21))
plt.grid(True, which='both', axis='x', linestyle='--', linewidth=0.5)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial import distance

# Load the CSV file into a DataFrame
file_path = 'final_tcr_selection_adjusted_distribution_with_random_groups.csv'
df = pd.read_csv(file_path)

# Extract relevant columns
df = df[['amino_acid', 'selection_group']]

# Function to process the 'file' column to extract the group names
def process_file_column(file_column):
    return file_column.apply(lambda x: 'consistency1' if 'consistency_area_top' in x
                             else ('consistency2' if 'consistency_area2_top' in x
                                   else x.split('_')[0]))

# Apply the function to the 'selection_group' column
df['selection_group'] = process_file_column(df['selection_group'])

# Dynamically generate a list of groups including all random groups
unique_groups = df['selection_group'].unique()
groups = {group: df[df['selection_group'] == group]['amino_acid'].dropna() for group in unique_groups}

# Process and remove sequences containing '*' for each group
for group_name in groups:
    groups[group_name] = groups[group_name][~groups[group_name].str.contains('\*', na=False)].tolist()

def amino_acid_presence_at_position(group_seqs, position):
    amino_acids = 'ACDEFGHIKLMNPQRSTVWY'  # List of all 20 standard amino acids
    presence_counts = {aa: 0 for aa in amino_acids}
    for seq in group_seqs:
        if len(seq) >= position:
            aa = seq[position - 1]
            presence_counts[aa] += 1
    return presence_counts

# Define groups of interest
target_groups = ['consistency1', 'consistency2', 'average', 'variance']
random_groups = [f'random_selection_{i}' for i in range(1, 6)]

# Initialize a dictionary to hold distances
distances = {tg: {rg: [] for rg in random_groups} for tg in target_groups}

# Loop through positions 1 to 20
for position in range(1, 21):
    aa_presence = {group: amino_acid_presence_at_position(seqs, position) for group, seqs in groups.items()}
    data = pd.DataFrame.from_dict(aa_presence, orient='index').fillna(0)

    # Calculate distances from each target group to each random group
    for tg in target_groups:
        for rg in random_groups:
            if tg in data.index and rg in data.index:
                distance_value = distance.euclidean(data.loc[tg], data.loc[rg])
                distances[tg][rg].append(distance_value)

# Plotting the distances
plt.figure(figsize=(14, 8))

# Define colors for different groups
colors = {
    'consistency1': 'red',
    'consistency2': 'orange',
    'average': '#dbde18',
    'variance': 'green'
}

# Plot distances for each target group
for tg in target_groups:
    for rg in random_groups:
        y_values = distances[tg][rg]
        if len(y_values) == 20:
            plt.plot(range(1, 21), y_values, marker='o', label=f'{tg} to {rg}', color=colors[tg], alpha=0.6)

plt.xlabel('Position', fontsize=14)
plt.ylabel('Euclidean Distance', fontsize=14)
plt.title('Euclidean Distance from Target Groups to Random Groups Across Positions', fontsize=18)
plt.legend(loc='upper right', bbox_to_anchor=(1.2, 1))
plt.xticks(range(1, 21))
plt.grid(True, which='both', axis='x', linestyle='--', linewidth=0.5)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Load the data
file_path = 'tcrs123.xlsx'
df = pd.read_excel(file_path)

# Extract the top 25 and bottom 25 rows based on 'consistency_area'
df_sorted = df.sort_values(by='consistency_area')
top_25 = df_sorted.head(25)
bottom_25 = df_sorted.tail(25)

# Prepare the data for plots
def prepare_plot_data(df):
    data = []
    labels = df['amino_acid']
    for _, row in df.iterrows():
        sample_counts = eval(row['sample_counts'])
        data.append(sample_counts)
    return data, labels

# Prepare data for top 25 and bottom 25
top_25_data, top_25_labels = prepare_plot_data(top_25)
bottom_25_data, bottom_25_labels = prepare_plot_data(bottom_25)

# Box Plots
fig, axs = plt.subplots(2, 1, figsize=(15, 14))
axs[0].boxplot(top_25_data, labels=top_25_labels)
axs[0].set_yscale('log')
axs[0].set_ylabel('Clonality Count')
axs[0].set_title('Box Plot of Clonality Counts - Top 25 TCR Sequences')
axs[0].tick_params(axis='x', rotation=90)
axs[0].grid(True)

axs[1].boxplot(bottom_25_data, labels=bottom_25_labels)
axs[1].set_yscale('log')
axs[1].set_ylabel('Clonality Count')
axs[1].set_title('Box Plot of Clonality Counts - Bottom 25 TCR Sequences')
axs[1].tick_params(axis='x', rotation=90)
axs[1].grid(True)

plt.tight_layout()
plt.show()

# Sorted Counts
fig, axs = plt.subplots(2, 1, figsize=(12, 12))
for i, sample in enumerate(top_25_data):
    sample_sorted = np.sort(sample)
    axs[0].plot(sample_sorted, label=top_25_labels.iloc[i])
axs[0].set_xlabel('Sorted Index')
axs[0].set_ylabel('Counts')
axs[0].set_yscale('log')
axs[0].set_title('Sorted Counts - Top 25 TCR Sequences')
axs[0].grid(True)
axs[0].legend(loc='upper left', bbox_to_anchor=(1, 1))

for i, sample in enumerate(bottom_25_data):
    sample_sorted = np.sort(sample)
    axs[1].plot(sample_sorted, label=bottom_25_labels.iloc[i])
axs[1].set_xlabel('Sorted Index')
axs[1].set_ylabel('Counts')
axs[1].set_yscale('log')
axs[1].set_title('Sorted Counts - Bottom 25 TCR Sequences')
axs[1].grid(True)
axs[1].legend(loc='upper left', bbox_to_anchor=(1, 1))

plt.tight_layout()
plt.show()

# Additional Plot 1: Density Plot
fig, axs = plt.subplots(2, 1, figsize=(12, 12))
sns.kdeplot([item for sublist in top_25_data for item in sublist], ax=axs[0], color='blue', label='Top 25')
axs[0].set_title('Density Plot - Top 25 TCR Sequences')
axs[0].set_xlabel('Clonality Count')
axs[0].set_ylabel('Density')
axs[0].legend()
axs[0].grid(True)

sns.kdeplot([item for sublist in bottom_25_data for item in sublist], ax=axs[1], color='red', label='Bottom 25')
axs[1].set_title('Density Plot - Bottom 25 TCR Sequences')
axs[1].set_xlabel('Clonality Count')
axs[1].set_ylabel('Density')
axs[1].legend()
axs[1].grid(True)

plt.tight_layout()
plt.show()

# Additional Plot 2: Violin Plot
fig, axs = plt.subplots(2, 1, figsize=(15, 14))
sns.violinplot(data=[top_25_data], ax=axs[0], scale='width')
axs[0].set_yscale('log')
axs[0].set_title('Violin Plot - Top 25 TCR Sequences')
axs[0].set_xticks(range(25))
axs[0].set_xticklabels(top_25_labels, rotation=90)
axs[0].set_ylabel('Clonality Count')
axs[0].grid(True)

sns.violinplot(data=[bottom_25_data], ax=axs[1], scale='width')
axs[1].set_yscale('log')
axs[1].set_title('Violin Plot - Bottom 25 TCR Sequences')
axs[1].set_xticks(range(25))
axs[1].set_xticklabels(bottom_25_labels, rotation=90)
axs[1].set_ylabel('Clonality Count')
axs[1].grid(True)

plt.tight_layout()
plt.show()

# Additional Plot 3: Scatter Plot
fig, axs = plt.subplots(2, 1, figsize=(15, 14))
axs[0].scatter(range(len([item for sublist in top_25_data for item in sublist])),
               [item for sublist in top_25_data for item in sublist],
               label='Top 25', alpha=0.5)
axs[0].set_yscale('log')
axs[0].set_xlabel('Index')
axs[0].set_ylabel('Clonality Count')
axs[0].set_title('Scatter Plot - Top 25 TCR Sequences')
axs[0].legend()
axs[0].grid(True)

axs[1].scatter(range(len([item for sublist in bottom_25_data for item in sublist])),
               [item for sublist in bottom_25_data for item in sublist],
               label='Bottom 25', alpha=0.5, color='red')
axs[1].set_yscale('log')
axs[1].set_xlabel('Index')
axs[1].set_ylabel('Clonality Count')
axs[1].set_title('Scatter Plot - Bottom 25 TCR Sequences')
axs[1].legend()
axs[1].grid(True)

plt.tight_layout()
plt.show()

# Additional Plot 4: CDF Plot
fig, axs = plt.subplots(2, 1, figsize=(12, 12))
top_25_sorted = [np.sort(sample) for sample in top_25_data]
bottom_25_sorted = [np.sort(sample) for sample in bottom_25_data]
axs[0].plot(np.arange(len(top_25_sorted[0])), top_25_sorted[0], label='Top 25')
axs[0].set_title('CDF Plot - Top 25 TCR Sequences')
axs[0].set_xlabel('Clonality Count')
axs[0].set_ylabel('CDF')
axs[0].grid(True)

axs[1].plot(np.arange(len(bottom_25_sorted[0])), bottom_25_sorted[0], label='Bottom 25', color='red')
axs[1].set_title('CDF Plot - Bottom 25 TCR Sequences')
axs[1].set_xlabel('Clonality Count')
axs[1].set_ylabel('CDF')
axs[1].grid(True)

plt.tight_layout()
plt.show()

# Additional Plot 5: Mean Plot
fig, axs = plt.subplots(2, 1, figsize=(12, 12))
top_25_means = [np.mean(sample) for sample in top_25_data]
bottom_25_means = [np.mean(sample) for sample in bottom_25_data]

axs[0].bar(top_25_labels, top_25_means)
axs[0].set_yscale('log')
axs[0].set_title('Mean Plot - Top 25 TCR Sequences')
axs[0].set_xlabel('TCR Sequences')
axs[0].set_ylabel('Mean Clonality Count')
axs[0].tick_params(axis='x', rotation=90)
axs[0].grid(True)

axs[1].bar(bottom_25_labels, bottom_25_means, color='red')
axs[1].set_yscale('log')
axs[1].set_title('Mean Plot - Bottom 25 TCR Sequences')
axs[1].set_xlabel('TCR Sequences')
axs[1].set_ylabel('Mean Clonality Count')
axs[1].tick_params(axis='x', rotation=90)
axs[1].grid(True)

plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Load the data
file_path = 'tcrs123.xlsx'
df = pd.read_excel(file_path)

# Extract the top 25 and bottom 25 rows based on 'consistency_area'
df_sorted = df.sort_values(by='consistency_area')
top_25 = df_sorted.tail(25)  # Highest values
bottom_25 = df_sorted.head(25)  # Lowest values

# Prepare the data for plots
def prepare_plot_data(df):
    data = []
    labels = df['amino_acid']
    for _, row in df.iterrows():
        sample_counts = eval(row['sample_counts'])
        data.append(sample_counts)
    return data, labels

# Prepare data for top 25 and bottom 25
top_25_data, top_25_labels = prepare_plot_data(top_25)
bottom_25_data, bottom_25_labels = prepare_plot_data(bottom_25)



# Plot 9: Pair Plot of Clonality Counts (for numerical comparison)
top_25_pairplot_data = pd.DataFrame(top_25_data_expanded, columns=top_25_labels)
sns.pairplot(top_25_pairplot_data)
plt.suptitle('Pair Plot - Top 25 TCR Sequences')
plt.show()

bottom_25_pairplot_data = pd.DataFrame(bottom_25_data_expanded, columns=bottom_25_labels)
sns.pairplot(bottom_25_pairplot_data)
plt.suptitle('Pair Plot - Bottom 25 TCR Sequences')
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Load the data
file_path = 'tcrs123.xlsx'
df = pd.read_excel(file_path)

# Extract the top 25 and bottom 25 rows based on 'consistency_area'
df_sorted = df.sort_values(by='consistency_area')
top_25 = df_sorted.tail(25)  # Highest values
bottom_25 = df_sorted.head(25)  # Lowest values

# Prepare the data for plots
def prepare_plot_data(df):
    data = []
    labels = df['amino_acid']
    for _, row in df.iterrows():
        sample_counts = eval(row['sample_counts'])
        data.append(sample_counts)
    return data, labels

# Prepare data for top 25 and bottom 25
top_25_data, top_25_labels = prepare_plot_data(top_25)
bottom_25_data, bottom_25_labels = prepare_plot_data(bottom_25)

# Box Plots
fig, axs = plt.subplots(2, 1, figsize=(15, 14))
axs[0].boxplot(top_25_data, labels=top_25_labels)
axs[0].set_yscale('log')
axs[0].set_ylim(0, 10**2)
axs[0].set_ylabel('Clonality Count')
axs[0].set_title('Box Plot of Clonality Counts - Top 25 TCR Sequences')
axs[0].tick_params(axis='x', rotation=90)
axs[0].grid(True)

axs[1].boxplot(bottom_25_data, labels=bottom_25_labels)
axs[1].set_yscale('log')
axs[1].set_ylim(0, 10**2)
axs[1].set_ylabel('Clonality Count')
axs[1].set_title('Box Plot of Clonality Counts - Bottom 25 TCR Sequences')
axs[1].tick_params(axis='x', rotation=90)
axs[1].grid(True)

plt.tight_layout()
plt.show()

# Sorted Counts
fig, axs = plt.subplots(2, 1, figsize=(12, 12))
for i, sample in enumerate(top_25_data):
    sample_sorted = np.sort(sample)
    axs[0].plot(sample_sorted, label=top_25_labels.iloc[i])
axs[0].set_xlabel('Sorted Index')
axs[0].set_ylabel('Counts')
axs[0].set_yscale('log')
axs[0].set_ylim(1, 10**3)
axs[0].set_title('Sorted Counts - Top 25 TCR Sequences')
axs[0].grid(True)
axs[0].legend(loc='upper left', bbox_to_anchor=(1, 1))

for i, sample in enumerate(bottom_25_data):
    sample_sorted = np.sort(sample)
    axs[1].plot(sample_sorted, label=bottom_25_labels.iloc[i])
axs[1].set_xlabel('Sorted Index')
axs[1].set_ylabel('Counts')
axs[1].set_yscale('log')
axs[1].set_ylim(1, 10**3)
axs[1].set_title('Sorted Counts - Bottom 25 TCR Sequences')
axs[1].grid(True)
axs[1].legend(loc='upper left', bbox_to_anchor=(1, 1))

plt.tight_layout()
plt.show()

# Additional Plot 1: Density Plot
fig, axs = plt.subplots(2, 1, figsize=(12, 12))
sns.kdeplot([item for sublist in top_25_data for item in sublist], ax=axs[0], color='blue', label='Top 25')
axs[0].set_title('Density Plot - Top 25 TCR Sequences')
axs[0].set_xlabel('Clonality Count')
axs[0].set_ylabel('Density')
axs[0].legend()
axs[0].grid(True)

sns.kdeplot([item for sublist in bottom_25_data for item in sublist], ax=axs[1], color='red', label='Bottom 25')
axs[1].set_title('Density Plot - Bottom 25 TCR Sequences')
axs[1].set_xlabel('Clonality Count')
axs[1].set_ylabel('Density')
axs[1].legend()
axs[1].grid(True)

plt.tight_layout()
plt.show()

# Additional Plot 2: Violin Plot
fig, axs = plt.subplots(2, 1, figsize=(15, 14))
sns.violinplot(data=[top_25_data], ax=axs[0], scale='width')
axs[0].set_yscale('log')
axs[0].set_title('Violin Plot - Top 25 TCR Sequences')
axs[0].set_xticks(range(25))
axs[0].set_xticklabels(top_25_labels, rotation=90)
axs[0].set_ylabel('Clonality Count')
axs[0].grid(True)

sns.violinplot(data=[bottom_25_data], ax=axs[1], scale='width')
axs[1].set_yscale('log')
axs[1].set_title('Violin Plot - Bottom 25 TCR Sequences')
axs[1].set_xticks(range(25))
axs[1].set_xticklabels(bottom_25_labels, rotation=90)
axs[1].set_ylabel('Clonality Count')
axs[1].grid(True)

plt.tight_layout()
plt.show()

# Additional Plot 3: Scatter Plot
fig, axs = plt.subplots(2, 1, figsize=(15, 14))
axs[0].scatter(range(len([item for sublist in top_25_data for item in sublist])),
               [item for sublist in top_25_data for item in sublist],
               label='Top 25', alpha=0.5)
axs[0].set_yscale('log')
axs[0].set_xlabel('Index')
axs[0].set_ylabel('Clonality Count')
axs[0].set_title('Scatter Plot - Top 25 TCR Sequences')
axs[0].legend()
axs[0].grid(True)

axs[1].scatter(range(len([item for sublist in bottom_25_data for item in sublist])),
               [item for sublist in bottom_25_data for item in sublist],
               label='Bottom 25', alpha=0.5, color='red')
axs[1].set_yscale('log')
axs[1].set_xlabel('Index')
axs[1].set_ylabel('Clonality Count')
axs[1].set_title('Scatter Plot - Bottom 25 TCR Sequences')
axs[1].legend()
axs[1].grid(True)

plt.tight_layout()
plt.show()

# Additional Plot 4: CDF Plot
fig, axs = plt.subplots(2, 1, figsize=(12, 12))
top_25_sorted = [np.sort(sample) for sample in top_25_data]
bottom_25_sorted = [np.sort(sample) for sample in bottom_25_data]
axs[0].plot(np.arange(len(top_25_sorted[0])), top_25_sorted[0], label='Top 25')
axs[0].set_ylim(1, max([max(sample) for sample in top_25_sorted]))
axs[0].set_title('CDF Plot - Top 25 TCR Sequences')
axs[0].set_xlabel('Clonality Count')
axs[0].set_ylabel('CDF')
axs[0].grid(True)

axs[1].plot(np.arange(len(bottom_25_sorted[0])), bottom_25_sorted[0], label='Bottom 25', color='red')
axs[1].set_ylim(1, max([max(sample) for sample in bottom_25_sorted]))
axs[1].set_title('CDF Plot - Bottom 25 TCR Sequences')
axs[1].set_xlabel('Clonality Count')
axs[1].set_ylabel('CDF')
axs[1].grid(True)

plt.tight_layout()
plt.show()

# Additional Plot 5: Mean Plot
fig, axs = plt.subplots(2, 1, figsize=(12, 12))
top_25_means = [np.mean(sample) for sample in top_25_data]
bottom_25_means = [np.mean(sample) for sample in bottom_25_data]

axs[0].bar(top_25_labels, top_25_means)
axs[0].set_yscale('log')
axs[0].set_ylim(1, 10**2)
axs[0].set_title('Mean Plot - Top 25 TCR Sequences')
axs[0].set_xlabel('TCR Sequences')
axs[0].set_ylabel('Mean Clonality Count')
axs[0].tick_params(axis='x', rotation=90)
axs[0].grid(True)

axs[1].bar(bottom_25_labels, bottom_25_means, color='red')
axs[1].set_yscale('log')
axs[1].set_ylim(1, 10**2)
axs[1].set_title('Mean Plot - Bottom 25 TCR Sequences')
axs[1].set_xlabel('TCR Sequences')
axs[1].set_ylabel('Mean Clonality Count')
axs[1].tick_params(axis='x', rotation=90)
axs[1].grid(True)

plt.tight_layout()
plt.show()

# Plot 6: Violin Plot
fig, axs = plt.subplots(1, 2, figsize=(20, 10))

# Top 25
top_25_data_expanded = [eval(counts) for counts in top_25['sample_counts']]
sns.violinplot(data=top_25_data_expanded, ax=axs[0], scale='width')
axs[0].set_yscale('log')
axs[0].set_ylim(1, 10**10)
axs[0].set_title('Violin Plot - Top 25 TCR Sequences')
axs[0].set_xlabel('TCR Sequences')
axs[0].set_ylabel('Clonality Count')
axs[0].tick_params(axis='x', rotation=90)
axs[0].grid(True)

# Bottom 25
bottom_25_data_expanded = [eval(counts) for counts in bottom_25['sample_counts']]
sns.violinplot(data=bottom_25_data_expanded, ax=axs[1], scale='width')
axs[1].set_yscale('log')
axs[1].set_ylim(1, 10**10)
axs[1].set_title('Violin Plot - Bottom 25 TCR Sequences')
axs[1].set_xlabel('TCR Sequences')
axs[1].set_ylabel('Clonality Count')
axs[1].tick_params(axis='x', rotation=90)
axs[1].grid(True)

plt.tight_layout()
plt.show()

# Plot 7: Bar Plot with Error Bars
fig, axs = plt.subplots(1, 2, figsize=(20, 10))

# Top 25
top_25_means = [np.mean(eval(counts)) for counts in top_25['sample_counts']]
top_25_stds = [np.std(eval(counts)) for counts in top_25['sample_counts']]
axs[0].bar(top_25_labels, top_25_means, yerr=top_25_stds, capsize=5, color='blue')
axs[0].set_yscale('log')
axs[0].set_ylim(1, 10**2)
axs[0].set_title('Bar Plot with Error Bars - Top 25 TCR Sequences')
axs[0].set_xlabel('TCR Sequences')
axs[0].set_ylabel('Mean Clonality Count ± STD')
axs[0].tick_params(axis='x', rotation=90)
axs[0].grid(True)

# Bottom 25
bottom_25_means = [np.mean(eval(counts)) for counts in bottom_25['sample_counts']]
bottom_25_stds = [np.std(eval(counts)) for counts in bottom_25['sample_counts']]
axs[1].bar(bottom_25_labels, bottom_25_means, yerr=bottom_25_stds, capsize=5, color='red')
axs[1].set_yscale('log')
axs[1].set_ylim(1, 10**2)
axs[1].set_title('Bar Plot with Error Bars - Bottom 25 TCR Sequences')
axs[1].set_xlabel('TCR Sequences')
axs[1].set_ylabel('Mean Clonality Count ± STD')
axs[1].tick_params(axis='x', rotation=90)
axs[1].grid(True)

plt.tight_layout()
plt.show()


# Plot 9: Pair Plot of Clonality Counts (for numerical comparison)
top_25_pairplot_data = pd.DataFrame(top_25_data_expanded, columns=top_25_labels)
sns.pairplot(top_25_pairplot_data)
plt.suptitle('Pair Plot - Top 25 TCR Sequences')
plt.show()

bottom_25_pairplot_data = pd.DataFrame(bottom_25_data_expanded, columns=bottom_25_labels)
sns.pairplot(bottom_25_pairplot_data)
plt.suptitle('Pair Plot - Bottom 25 TCR Sequences')
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import os

# Define the path where the results are stored
results_path = './'  # Adjust this to your actual path if different

# Define the groups you compared
groups_to_compare = [
    'average_count_top',
    'explosion_area_top',
    'consistency_area_top',
    'consistency_area2_top',
    'variance_score_top',
    'publicity_top',
    'random_selection_2'
]

# Initialize a list to hold accuracy results
accuracy_results = []

# Load the test performance results for each group comparison
for group in groups_to_compare:
    # Construct the file path for the test performance CSV
    file_path = os.path.join(results_path, f'random1_vs_{group}_test_performance_without_threshold.csv')

    # Read the CSV file into a DataFrame
    try:
        test_performance_df = pd.read_csv(file_path)

        # Extract the accuracy
        accuracy = test_performance_df['accuracy'].values[0]
        # Modify group names as specified
        if group == 'consistency_area_top':
            group_name = 'consistency1'
        elif group == 'consistency_area2_top':
            group_name = 'consistency2'
        else:
            group_name = group.split('_')[0]
        accuracy_results.append((group_name, accuracy))
    except FileNotFoundError:
        print(f"File not found: {file_path}")
        accuracy_results.append((group, None))

# Convert results to a DataFrame for easier plotting
accuracy_df = pd.DataFrame(accuracy_results, columns=['Group', 'Accuracy']).dropna()
accuracy_df = accuracy_df.sort_values(by='Accuracy', ascending=False)

# Separate the publicity group from others
publicity_df = accuracy_df[accuracy_df['Group'] == 'publicity']
other_df = accuracy_df[accuracy_df['Group'] != 'publicity']

# Define a more professional color palette
color_map = {
    'average': '#1f77b4',   # A professional blue shade
    'variance': '#1f77b4',  # Same blue for consistency
    'consistency1': '#2ca02c',  # A professional green shade
    'consistency2': '#2ca02c',  # Same green for consistency
    'explosion': '#17becf',  # A different shade of blue-green
    'publicity': '#7f7f7f',  # A medium grey
    'random': '#ff7f0e'  # A professional orange
}

# Generate colors for each bar based on the group
colors = [color_map[group] for group in other_df['Group']]

# Setting up the figure and GridSpec layout
fig = plt.figure(figsize=(14, 6))
gs = gridspec.GridSpec(1, 2, width_ratios=[8, 1], wspace=0.1)

# Plotting the accuracy results for other groups
ax1 = fig.add_subplot(gs[0])
ax1.bar(other_df['Group'], other_df['Accuracy'], color=colors)
ax1.set_xlabel('Group')
ax1.set_ylabel('Accuracy')
ax1.set_title('Classification Accuracy: Random Selection 1 vs Other Groups')
ax1.set_xticks(range(len(other_df)))
ax1.set_xticklabels(other_df['Group'], rotation=45, ha='right')
ax1.set_ylim(0.5, 1)  # Set y-axis limits from 0.5 to 1
ax1.grid(axis='y', linestyle='--', alpha=0.7)

# Add a legend including the random group
legend_handles = [
    plt.Rectangle((0, 0), 1, 1, color='#1f77b4', alpha=0.7, label='Average/Variance'),
    plt.Rectangle((0, 0), 1, 1, color='#2ca02c', alpha=0.7, label='Consistency1/2'),
    plt.Rectangle((0, 0), 1, 1, color='#17becf', alpha=0.7, label='Explosion'),
    plt.Rectangle((0, 0), 1, 1, color='#7f7f7f', alpha=0.7, label='Publicity'),
    plt.Rectangle((0, 0), 1, 1, color='#ff7f0e', alpha=0.7, label='Random')
]
ax1.legend(handles=legend_handles)

# Plotting the accuracy results for the publicity group in a separate plot
ax2 = fig.add_subplot(gs[1])
ax2.bar(publicity_df['Group'], publicity_df['Accuracy'], color='#7f7f7f')
ax2.set_xlabel('Group')
#ax2.set_ylabel('Accuracy')  # Uncomment if you want the ylabel
#ax2.set_title('Publicity Group')  # Uncomment if you want a separate title
ax2.set_xticks(range(len(publicity_df)))
ax2.set_xticklabels(publicity_df['Group'], rotation=45, ha='right')
ax2.set_ylim(0.5, 1)  # Set y-axis limits from 0.5 to 1
ax2.grid(axis='y', linestyle='--', alpha=0.7)

# Adjust layout and show the plots
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

# Define the path where the results are stored
results_path = './'  # Adjust this to your actual path if different

# Define the groups you compared
groups_to_compare = [
    'average_count_top',
    'explosion_area_top',
    'consistency_area_top',
    'consistency_area2_top',
    'variance_score_top',
    'publicity_top',
    'random_selection_2'
]

# Initialize a list to hold accuracy results
accuracy_results = []

# Load the test performance results for each group comparison
for group in groups_to_compare:
    # Construct the file path for the test performance CSV
    file_path = os.path.join(results_path, f'random1_vs_{group}_test_performance_without_threshold.csv')

    # Read the CSV file into a DataFrame
    try:
        test_performance_df = pd.read_csv(file_path)

        # Extract the accuracy
        accuracy = test_performance_df['accuracy'].values[0]
        accuracy_results.append((group, accuracy))
    except FileNotFoundError:
        print(f"File not found: {file_path}")
        accuracy_results.append((group, None))

# Convert results to a DataFrame for easier plotting
accuracy_df = pd.DataFrame(accuracy_results, columns=['Group', 'Accuracy']).dropna()
accuracy_df = accuracy_df.sort_values(by='Accuracy', ascending=False)

# Plotting the accuracy results
plt.figure(figsize=(10, 6))
plt.bar(accuracy_df['Group'], accuracy_df['Accuracy'], color='skyblue')
plt.xlabel('Group')
plt.ylabel('Accuracy')
plt.title('Classification Accuracy: Random Selection 1 vs Other Groups')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

# Load and prepare data
df = pd.read_csv('final_tcr_selection_adjusted_distribution_with_random_groups.csv')

# Exclude the 'publicity_top' group
df = df[df['selection_group'] != 'publicity_top']

unique_groups = df['selection_group'].unique()
groups = {group: df[df['selection_group'] == group]['amino_acid'].dropna() for group in unique_groups}

for group_name in groups:
    groups[group_name] = groups[group_name][~groups[group_name].str.contains('\*', na=False)].tolist()

def amino_acid_presence_at_position(group_seqs, position):
    amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
    presence_counts = {aa: 0 for aa in amino_acids}
    for seq in group_seqs:
        if len(seq) >= position:
            aa = seq[position - 1]
            presence_counts[aa] += 1
    return presence_counts

# Setting up the figure
fig, axes = plt.subplots(5, 4, figsize=(20, 25))
#fig.suptitle('2D PCA of Amino Acid Counts at Positions 1 to 17', fontsize=16, y=0.92)

# Flatten axes for easy iteration
axes = axes.flatten()

# Loop through positions 1 to 17
for position in range(1, 18):
    aa_presence = {group: amino_acid_presence_at_position(seqs, position) for group, seqs in groups.items()}
    data = pd.DataFrame.from_dict(aa_presence, orient='index').fillna(0)

    # Standardizing the data
    scaler = StandardScaler()
    data_scaled = scaler.fit_transform(data)

    # Performing PCA to reduce to two dimensions
    pca = PCA(n_components=2)
    principal_components = pca.fit_transform(data_scaled)

    # Create a DataFrame with the PCA results
    pca_df = pd.DataFrame(data=principal_components, columns=['Principal Component 1', 'Principal Component 2'])
    pca_df['Group'] = data.index

    # Colors for the groups
    predefined_colors = {
        'average_count_top': 'red',
        'explosion_area_top': 'blue',
        'consistency_area_top': 'green',
        'consistency_area2_top': 'purple',
        'variance_score_top': 'orange',
        'publicity_top': 'cyan'
    }
    default_random_color = 'grey'  # Default color for random or undefined groups

    # Add subplot for the current position
    ax = axes[position - 1]
    for idx, row in pca_df.iterrows():
        group_color = predefined_colors.get(row['Group'], default_random_color)
        ax.scatter(row['Principal Component 1'], row['Principal Component 2'], color=group_color, s=500, label=row['Group'])

    # Simplifying each subplot
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(f'Position {position}')
    if position == 17:  # Add legend only to the last subplot
        handles, labels = ax.get_legend_handles_labels()
        by_label = dict(zip(labels, handles))
        fig.legend(by_label.values(), by_label.keys(), loc='upper right', bbox_to_anchor=(1.15, 1))

plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()


In [ ]:
import pandas as pd
import numpy as np

# Load the CSV file
df = pd.read_csv('f_final_tcr_selection_adjusted_distribution_with_random_groups.csv')

# Define the amino acids and padding character
amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
padding_char = 'X'  # 'X' is often used as a padding character in bioinformatics

# One-hot encode a single sequence
def one_hot_encode_sequence(seq, max_length, amino_acids, padding_char):
    # Pad the sequence with the padding character to the max_length
    seq = seq.ljust(max_length, padding_char)
    # Create a one-hot encoded matrix of shape (max_length, number of amino acids)
    one_hot_matrix = np.zeros((max_length, len(amino_acids)), dtype=int)
    for i, char in enumerate(seq):
        if char in amino_acids:
            one_hot_matrix[i, amino_acids.index(char)] = 1
        else:
            # Assign padding character to a column of zeros since it's not a valid amino acid
            pass  # The matrix is already initialized to zeros
    return one_hot_matrix

# Determine the maximum length among all sequences
max_length = df['amino_acid'].apply(len).max()

# Initialize a DataFrame to hold the encoded sequences
encoded_data = pd.DataFrame()

# Encode sequences from each selection group
for group in df['selection_group'].unique():
    group_sequences = df[df['selection_group'] == group]['amino_acid']

    # Initialize a matrix to hold the encoded sequences for the current group
    group_encoded_seqs = np.zeros((len(group_sequences), max_length * len(amino_acids)), dtype=int)

    for idx, seq in enumerate(group_sequences):
        # One-hot encode each sequence and flatten the matrix to a 1D array
        encoded_seq = one_hot_encode_sequence(seq, max_length, amino_acids, padding_char).flatten()
        group_encoded_seqs[idx, :] = encoded_seq

    # Create a DataFrame for the current group
    group_encoded_df = pd.DataFrame(group_encoded_seqs,
                                    columns=[f'pos_{i}_{aa}' for i in range(max_length) for aa in amino_acids])
    group_encoded_df['Group'] = group  # Add a column indicating the group of each sequence

    # Append the current group's encoded data to the main DataFrame
    encoded_data = pd.concat([encoded_data, group_encoded_df], ignore_index=True)

# Save the encoded data to a new CSV file
output_file_path = 'encoded_tcr_sequences_without_threshold.csv'
encoded_data.to_csv(output_file_path, index=False)

print(f"Encoding complete. Data saved to '{output_file_path}'")


In [ ]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # Required for 3D plotting

# Load the CSV file
df = pd.read_csv('/mnt/data/final_tcr_selection_adjusted_distribution_with_random_groups.csv')

# Define the amino acids and padding character
amino_acids = 'ACDEFGHIKLMNPQRSTVWY'
padding_char = 'X'  # 'X' is often used as a padding character in bioinformatics

# One-hot encode a single sequence
def one_hot_encode_sequence(seq, max_length, amino_acids, padding_char):
    # Pad the sequence with the padding character to the max_length
    seq = seq.ljust(max_length, padding_char)
    # Create a one-hot encoded matrix of shape (max_length, number of amino acids)
    one_hot_matrix = np.zeros((max_length, len(amino_acids)), dtype=int)
    for i, char in enumerate(seq):
        if char in amino_acids:
            one_hot_matrix[i, amino_acids.index(char)] = 1
        else:
            # Assign padding character to a column of zeros since it's not a valid amino acid
            pass  # The matrix is already initialized to zeros
    return one_hot_matrix

# Determine the maximum length among all sequences
max_length = df['amino_acid'].apply(len).max()

# Initialize a DataFrame to hold the encoded sequences
encoded_data = pd.DataFrame()

# Encode sequences from each selection group
for group in df['selection_group'].unique():
    group_sequences = df[df['selection_group'] == group]['amino_acid']

    # Initialize a matrix to hold the encoded sequences for the current group
    group_encoded_seqs = np.zeros((len(group_sequences), max_length * len(amino_acids)), dtype=int)

    for idx, seq in enumerate(group_sequences):
        # One-hot encode each sequence and flatten the matrix to a 1D array
        encoded_seq = one_hot_encode_sequence(seq, max_length, amino_acids, padding_char).flatten()
        group_encoded_seqs[idx, :] = encoded_seq

    # Create a DataFrame for the current group
    group_encoded_df = pd.DataFrame(group_encoded_seqs,
                                    columns=[f'pos_{i}_{aa}' for i in range(max_length) for aa in amino_acids])
    group_encoded_df['Group'] = group  # Add a column indicating the group of each sequence

    # Append the current group's encoded data to the main DataFrame
    encoded_data = pd.concat([encoded_data, group_encoded_df], ignore_index=True)

# One-hot encoding complete

# Standardizing the data
scaler = StandardScaler()
data_scaled = scaler.fit_transform(encoded_data.drop(columns=['Group']))

# Performing PCA to reduce to three dimensions
pca = PCA(n_components=3)
principal_components = pca.fit_transform(data_scaled)

# Create a DataFrame with the PCA results
pca_df = pd.DataFrame(data=principal_components, columns=['Principal Component 1', 'Principal Component 2', 'Principal Component 3'])
pca_df['Group'] = encoded_data['Group']

# Colors for the groups
predefined_colors = {
    'average_count_top': 'red',
    'explosion_area_top': 'blue',
    'consistency_area_top': 'green',
    'consistency_area2_top': 'purple',
    'variance_score_top': 'orange',
    'publicity_top': 'cyan'
}
default_random_color = 'grey'  # Default color for random or undefined groups

# Plotting the PCA results in 3D
fig = plt.figure(figsize=(12, 10))
ax = fig.add_subplot(111, projection='3d')
for idx, row in pca_df.iterrows():
    group_color = predefined_colors.get(row['Group'], default_random_color)
    ax.scatter(row['Principal Component 1'], row['Principal Component 2'], row['Principal Component 3'],
               color=group_color, s=50, label=row['Group'] if row['Group'] not in [h.get_label() for h in ax.legend().legendHandles] else "")

ax.set_title('3D PCA of Amino Acid Counts at Position 5')
ax.set_xlabel('Principal Component 1')
ax.set_ylabel('Principal Component 2')
ax.set_zlabel('Principal Component 3')
ax.legend()
plt.show()


In [ ]:
encoded_data

In [ ]:
import pandas as pd
import itertools
import matplotlib.pyplot as plt
from matplotlib_venn import venn3
from collections import defaultdict
import os
import seaborn as sns
from scipy.spatial.distance import pdist, squareform
from sklearn.manifold import MDS
from scipy.cluster.hierarchy import dendrogram, linkage

# Define file paths
file_path = 'final_tcr_selection_adjusted_distribution_with_random_groups.csv'

output_folder = ""  # Specify your output folder here
# Load the CSV file
df = pd.read_csv(file_path)

# Extract relevant columns
df = df[['amino_acid', 'selection_group']]

# Define the groups to include (first random selection and top five metric groups)
selected_groups = ['random_selection_1', 'average_count_top', 'explosion_area_top', 'consistency_area_top', 'consistency_area2_top', 'variance_score_top', 'publicity_top']

# Filter to include only the selected groups
df = df[df['selection_group'].isin(selected_groups)]

# Create a dictionary to hold sets of amino acids for each group
group_dict = defaultdict(set)

# Populate the dictionary with amino acids for each selection group
for group in df['selection_group'].unique():
    group_dict[group] = set(df[df['selection_group'] == group]['amino_acid'])

# Function to create Venn diagrams for all combinations of groups and save them
def plot_and_save_venn_diagrams(group_dict):
    groups = list(group_dict.keys())
    if len(groups) < 3:
        print("Not enough groups for Venn diagram. At least 3 groups are required.")
        return

    # Generate all 3-group combinations
    combinations = list(itertools.combinations(groups, 3))

    for combo in combinations:
        plt.figure(figsize=(10, 8))  # Adjust the size as needed
        venn3([group_dict[combo[0]], group_dict[combo[1]], group_dict[combo[2]]], set_labels=(combo[0], combo[1], combo[2]))
        plt.title(f'Overlap between selection groups: {", ".join(combo)}')
        file_name = f"{'_'.join(combo)}.png"
        plt.savefig(os.path.join(output_folder, file_name))  # Save the figure
        plt.show()

# Plot and save Venn diagrams for all combinations of 3 groups
plot_and_save_venn_diagrams(group_dict)

# Calculate the Jaccard distance between each pair of groups
group_names = list(group_dict.keys())
num_groups = len(group_names)
distance_matrix = pd.DataFrame(index=group_names, columns=group_names)

for i, group1 in enumerate(group_names):
    for j, group2 in enumerate(group_names):
        if i == j:
            distance_matrix.loc[group1, group2] = 0.0
        else:
            set1 = group_dict[group1]
            set2 = group_dict[group2]
            intersection = len(set1 & set2)
            union = len(set1 | set2)
            jaccard_distance = 1 - intersection / union
            distance_matrix.loc[group1, group2] = jaccard_distance

# Convert the distance matrix to a numerical type
distance_matrix = distance_matrix.astype(float)

# Plot the distance matrix
plt.figure(figsize=(12, 10))
sns.heatmap(distance_matrix, annot=True, cmap='viridis', xticklabels=group_names, yticklabels=group_names)
plt.title('Jaccard Distance Matrix Between Selection Groups')
plt.show()

# Save the distance matrix to a CSV file
distance_matrix.to_csv(os.path.join(output_folder, 'distance_matrix.csv'))

# Load the distance matrix for further analysis
distance_matrix = pd.read_csv(os.path.join(output_folder, 'distance_matrix.csv'), index_col=0)

# Ensure the matrix is numerical
distance_matrix = distance_matrix.astype(float)

# Plot the heatmap of the distance matrix
plt.figure(figsize=(12, 10))
sns.heatmap(distance_matrix, annot=True, cmap='viridis', xticklabels=True, yticklabels=True)
plt.title('Jaccard Distance Matrix Between Selection Groups')
plt.show()

# Perform MDS (Multidimensional Scaling) for 2D visualization
mds = MDS(n_components=2, dissimilarity="precomputed", random_state=42)
mds_results = mds.fit_transform(distance_matrix.values)

# Create a DataFrame for MDS results
mds_df = pd.DataFrame(mds_results, index=distance_matrix.index, columns=['MDS1', 'MDS2'])

# Plot MDS results
plt.figure(figsize=(12, 10))
sns.scatterplot(x='MDS1', y='MDS2', data=mds_df, s=100)

# Annotate points with group names
for i in range(mds_df.shape[0]):
    plt.text(mds_df['MDS1'][i], mds_df['MDS2'][i], mds_df.index[i], fontsize=12)

plt.title('MDS Visualization of Jaccard Distance Matrix')
plt.xlabel('MDS1')
plt.ylabel('MDS2')
plt.grid(True)
plt.show()

# Perform hierarchical clustering and plot dendrogram
# Generate the linkage matrix
Z = linkage(distance_matrix, 'ward')

# Plot the dendrogram
plt.figure(figsize=(12, 10))
dendrogram(Z, labels=distance_matrix.index, leaf_rotation=90, leaf_font_size=12, color_threshold=0.5)
plt.title('Hierarchical Clustering Dendrogram')
plt.xlabel('Selection Groups')
plt.ylabel('Distance')
plt.show()


In [ ]:

import pandas as pd
import itertools
import matplotlib.pyplot as plt
from matplotlib_venn import venn3
from collections import defaultdict
import os
import seaborn as sns
from scipy.spatial.distance import pdist, squareform
from sklearn.manifold import MDS
from scipy.cluster.hierarchy import dendrogram, linkage

# Define file paths
file_path = 'final_tcr_selection_adjusted_distribution_with_random_groups.csv'

output_folder = ""
# Load the CSV file
df = pd.read_csv(file_path)

# Extract relevant columns
df = df[['amino_acid', 'selection_group']]

# Define the groups to include (first random selection and top five metric groups)
selected_groups = ['random_selection_1', 'average_count_top', 'explosion_area_top', 'consistency_area_top', 'consistency_area2_top', 'variance_score_top', 'publicity_top']

# Filter to include only the selected groups
df = df[df['selection_group'].isin(selected_groups)]

# Create a dictionary to hold sets of amino acids for each group
group_dict = defaultdict(set)

# Populate the dictionary with amino acids for each selection group
for group in df['selection_group'].unique():
    group_dict[group] = set(df[df['selection_group'] == group]['amino_acid'])

# Function to create Venn diagrams for all combinations of groups and save them
def plot_and_save_venn_diagrams(group_dict):
    groups = list(group_dict.keys())
    if len(groups) < 3:
        print("Not enough groups for Venn diagram. At least 3 groups are required.")
        return

    # Generate all 3-group combinations
    combinations = list(itertools.combinations(groups, 3))

    for combo in combinations:
        plt.figure()
        venn3([group_dict[combo[0]], group_dict[combo[1]], group_dict[combo[2]]], set_labels=(combo[0], combo[1], combo[2]))
        plt.title(f'Overlap between selection groups: {", ".join(combo)}')
        file_name = f"{'_'.join(combo)}.png"
        plt.show()

# Plot and save Venn diagrams for all combinations of 3 groups
plot_and_save_venn_diagrams(group_dict)

# Calculate the Jaccard distance between each pair of groups
group_names = list(group_dict.keys())
num_groups = len(group_names)
distance_matrix = pd.DataFrame(index=group_names, columns=group_names)

for i, group1 in enumerate(group_names):
    for j, group2 in enumerate(group_names):
        if i == j:
            distance_matrix.loc[group1, group2] = 0.0
        else:
            set1 = group_dict[group1]
            set2 = group_dict[group2]
            intersection = len(set1 & set2)
            union = len(set1 | set2)
            jaccard_distance = 1 - intersection / union
            distance_matrix.loc[group1, group2] = jaccard_distance

# Convert the distance matrix to a numerical type
distance_matrix = distance_matrix.astype(float)

# Plot the distance matrix
plt.figure(figsize=(10, 8))
sns.heatmap(distance_matrix, annot=True, cmap='viridis', xticklabels=group_names, yticklabels=group_names)
plt.title('Jaccard Distance Matrix Between Selection Groups')
plt.show()

# Save the distance matrix to a CSV file
distance_matrix.to_csv(os.path.join(output_folder, 'distance_matrix.csv'))

# Load the distance matrix for further analysis
distance_matrix = pd.read_csv(os.path.join(output_folder, 'distance_matrix.csv'), index_col=0)

# Ensure the matrix is numerical
distance_matrix = distance_matrix.astype(float)

# Plot the heatmap of the distance matrix
plt.figure(figsize=(12, 10))
sns.heatmap(distance_matrix, annot=True, cmap='viridis', xticklabels=True, yticklabels=True)
plt.title('Jaccard Distance Matrix Between Selection Groups')
plt.show()

# Perform MDS (Multidimensional Scaling) for 2D visualization
mds = MDS(n_components=2, dissimilarity="precomputed", random_state=42)
mds_results = mds.fit_transform(distance_matrix.values)

# Create a DataFrame for MDS results
mds_df = pd.DataFrame(mds_results, index=distance_matrix.index, columns=['MDS1', 'MDS2'])

# Plot MDS results
plt.figure(figsize=(12, 10))
sns.scatterplot(x='MDS1', y='MDS2', data=mds_df, s=100)

# Annotate points with group names
for i in range(mds_df.shape[0]):
    plt.text(mds_df['MDS1'][i], mds_df['MDS2'][i], mds_df.index[i], fontsize=12)

plt.title('MDS Visualization of Jaccard Distance Matrix')
plt.xlabel('MDS1')
plt.ylabel('MDS2')
plt.grid(True)
plt.show()

# Perform hierarchical clustering and plot dendrogram
# Generate the linkage matrix
Z = linkage(distance_matrix, 'ward')

# Plot the dendrogram
plt.figure(figsize=(12, 10))
dendrogram(Z, labels=distance_matrix.index, leaf_rotation=90, leaf_font_size=12, color_threshold=0.5)
plt.title('Hierarchical Clustering Dendrogram')
plt.xlabel('Selection Groups')
plt.ylabel('Distance')
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np

# Load the data
file_path = 'final_tcr_selection_adjusted_distribution_with_random_groups.csv'
df = pd.read_csv(file_path)

# Exclude the 'publicity_top' group
df = df[df['selection_group'] != 'publicity_top']

# Extract the first 25 rows for each 'selection_group'
first_10_rows_each_group = df.groupby('selection_group').head(25)

# Find the global max value for the y-axis limit
max_value = 0
for index, row in first_10_rows_each_group.iterrows():
    sample_counts = eval(row['sample_counts'])  # assuming 'sample_counts' is stored as a string representation of a list
    max_value = max(max_value, max(sample_counts))

# Get unique selection groups
groups = first_10_rows_each_group['selection_group'].unique()

# Get a colormap
colors = cm.get_cmap('tab10', len(groups))

# Specify the number of rows and columns for the subplots
num_cols = 3  # Number of columns in the grid
num_rows = (len(groups) + num_cols - 1) // num_cols  # Calculate the number of rows needed

# Create a single figure with subplots for all groups in a grid layout
fig, axs = plt.subplots(num_rows, num_cols, figsize=(20, 6 * num_rows))

for i, group in enumerate(groups):
    row = i // num_cols
    col = i % num_cols
    group_data = first_10_rows_each_group[first_10_rows_each_group['selection_group'] == group]
    for index, row_data in group_data.iterrows():
        sample_counts = row_data['sample_counts']
        sample_counts_sorted = sorted(eval(sample_counts))  # assuming 'sample_counts' is stored as a string representation of a list
        axs[row, col].plot(sample_counts_sorted, color=colors(i))
    axs[row, col].set_title(f'Selection Group: {group}')
    axs[row, col].set_xlabel('Index')
    axs[row, col].set_ylabel('Value')
    axs[row, col].set_yscale('log')
    axs[row, col].set_ylim(1, max_value)  # Set the y-axis limit

# Remove empty subplots
for j in range(len(groups), num_rows * num_cols):
    fig.delaxes(axs.flatten()[j])

# Adjust layout and save the figure
plt.tight_layout()
plt.savefig('selection_groups_plot_combined.png')
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np

# Load the data
file_path = 'f_final_tcr_selection_adjusted_distribution_with_random_groups.csv'
df = pd.read_csv(file_path)

# Extract the first 25 rows for each 'selection_group'
first_10_rows_each_group = df.groupby('selection_group').head(25)

# Find the global max value for the y-axis limit
max_value = 0
for index, row in first_10_rows_each_group.iterrows():
    sample_counts = eval(row['sample_counts'])  # assuming 'sample_counts' is stored as a string representation of a list
    max_value = max(max_value, max(sample_counts))

# Get unique selection groups
groups = first_10_rows_each_group['selection_group'].unique()

# Get a colormap
colors = cm.get_cmap('tab10', len(groups))

# Specify the number of rows and columns for the subplots
num_cols = 3  # Number of columns in the grid
num_rows = (len(groups) + num_cols - 1) // num_cols  # Calculate the number of rows needed

# Create a single figure with subplots for all groups in a grid layout
fig, axs = plt.subplots(num_rows, num_cols, figsize=(20, 6 * num_rows))

for i, group in enumerate(groups):
    row = i // num_cols
    col = i % num_cols
    group_data = first_10_rows_each_group[first_10_rows_each_group['selection_group'] == group]
    for index, row_data in group_data.iterrows():
        sample_counts = row_data['sample_counts']
        sample_counts_sorted = sorted(eval(sample_counts))  # assuming 'sample_counts' is stored as a string representation of a list
        axs[row, col].plot(range(len(sample_counts_sorted)), sample_counts_sorted, marker='o', color=colors(i), markersize=2.5, linewidth=1)
    axs[row, col].set_title(f'Selection Group: {group}')
    axs[row, col].set_xlabel('Index')
    axs[row, col].set_ylabel('Value')
    axs[row, col].set_yscale('log')
    axs[row, col].set_ylim(1, max_value)  # Set the y-axis limit

# Remove empty subplots
for j in range(len(groups), num_rows * num_cols):
    fig.delaxes(axs.flatten()[j])

# Adjust layout and save the figure
plt.tight_layout()
plt.savefig('selection_groups_plot_combined.png')
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load the data
file_path = 'final_tcr_selection_adjusted_distribution_with_random_groups.csv'
df = pd.read_csv(file_path)

# Exclude the 'publicity_top' group
df = df[df['selection_group'] != 'publicity_top']

# Extract the first 25 rows for each 'selection_group'
first_10_rows_each_group = df.groupby('selection_group').head(50)

# Prepare the data for box plots
data = []
labels = []
for group in first_10_rows_each_group['selection_group'].unique():
    group_data = first_10_rows_each_group[first_10_rows_each_group['selection_group'] == group]
    all_values = []
    for index, row in group_data.iterrows():
        sample_counts = eval(row['sample_counts'])
        all_values.extend(sample_counts)
    data.append(all_values)
    labels.append(group)

# Plot box plots
plt.figure(figsize=(20, 10))
plt.boxplot(data, labels=labels)
plt.yscale('log')
plt.ylabel('Value')
plt.xticks(rotation=90)
plt.title('Box Plots of Sample Counts by Selection Group')
plt.show()


In [ ]:
import seaborn as sns

# Prepare the data for violin plots
data_for_violin = []
for group in first_10_rows_each_group['selection_group'].unique():
    group_data = first_10_rows_each_group[first_10_rows_each_group['selection_group'] == group]
    for index, row in group_data.iterrows():
        sample_counts = eval(row['sample_counts'])
        data_for_violin.extend([(group, val) for val in sample_counts])

df_violin = pd.DataFrame(data_for_violin, columns=['Selection Group', 'Value'])

# Plot violin plots
plt.figure(figsize=(20, 10))
sns.violinplot(x='Selection Group', y='Value', data=df_violin)
plt.yscale('log')
plt.title('Violin Plots of Sample Counts by Selection Group')
plt.xticks(rotation=90)
plt.show()


In [ ]:
# Plot histograms
fig, axs = plt.subplots(num_rows, num_cols, figsize=(20, 6 * num_rows))

for i, group in enumerate(groups):
    row = i // num_cols
    col = i % num_cols
    group_data = first_10_rows_each_group[first_10_rows_each_group['selection_group'] == group]
    all_values = []
    for index, row_data in group_data.iterrows():
        sample_counts = eval(row_data['sample_counts'])
        all_values.extend(sample_counts)
    axs[row, col].hist(all_values, bins=50, log=True, color=colors(i))
    axs[row, col].set_title(f'Selection Group: {group}')
    axs[row, col].set_xlabel('Value')
    axs[row, col].set_ylabel('Frequency')

plt.tight_layout()
plt.show()


In [ ]:
# Plot KDE plots
fig, axs = plt.subplots(num_rows, num_cols, figsize=(20, 6 * num_rows))

for i, group in enumerate(groups):
    row = i // num_cols
    col = i % num_cols
    group_data = first_10_rows_each_group[first_10_rows_each_group['selection_group'] == group]
    all_values = []
    for index, row_data in group_data.iterrows():
        sample_counts = eval(row_data['sample_counts'])
        all_values.extend(sample_counts)
    sns.kdeplot(all_values, ax=axs[row, col], log_scale=True, color=colors(i))
    axs[row, col].set_title(f'Selection Group: {group}')
    axs[row, col].set_xlabel('Value')
    axs[row, col].set_ylabel('Density')

plt.tight_layout()
plt.show()


In [ ]:
# Plot CDF plots
fig, axs = plt.subplots(num_rows, num_cols, figsize=(20, 6 * num_rows))

for i, group in enumerate(groups):
    row = i // num_cols
    col = i % num_cols
    group_data = first_10_rows_each_group[first_10_rows_each_group['selection_group'] == group]
    all_values = []
    for index, row_data in group_data.iterrows():
        sample_counts = eval(row_data['sample_counts'])
        all_values.extend(sample_counts)
    sorted_values = np.sort(all_values)
    cdf = np.arange(1, len(sorted_values) + 1) / len(sorted_values)
    axs[row, col].plot(sorted_values, cdf, color=colors(i))
    axs[row, col].set_title(f'Selection Group: {group}')
    axs[row, col].set_xlabel('Value')
    axs[row, col].set_ylabel('CDF')
    axs[row, col].set_xscale('log')

plt.tight_layout()
plt.show()


In [ ]:
import seaborn as sns

# Prepare the data for heatmap
heatmap_data = []
for group in first_10_rows_each_group['selection_group'].unique():
    group_data = first_10_rows_each_group[first_10_rows_each_group['selection_group'] == group]
    all_values = []
    for index, row in group_data.iterrows():
        sample_counts = eval(row['sample_counts'])
        all_values.extend(sample_counts)
    heatmap_data.append(all_values)

# Convert to DataFrame and fill NaNs for unequal lengths
heatmap_df = pd.DataFrame(heatmap_data).T
heatmap_df.fillna(0, inplace=True)
heatmap_df.columns = first_10_rows_each_group['selection_group'].unique()

# Plot heatmap
plt.figure(figsize=(20, 10))
sns.heatmap(heatmap_df, cmap='viridis', cbar=True)
plt.title('Heatmap of Sample Counts by Selection Group')
plt.xlabel('Selection Group')
plt.ylabel('Sample Index')
plt.show()


In [ ]:
# Prepare the data for pair plots
pairplot_data = []
for group in first_10_rows_each_group['selection_group'].unique():
    group_data = first_10_rows_each_group[first_10_rows_each_group['selection_group'] == group]
    all_values = []
    for index, row in group_data.iterrows():
        sample_counts = eval(row['sample_counts'])
        all_values.extend(sample_counts)
    pairplot_data.append(pd.DataFrame({group: all_values}))

# Concatenate all dataframes
pairplot_df = pd.concat(pairplot_data, axis=1)

# Plot pair plots
sns.pairplot(pairplot_df)
plt.suptitle('Pair Plots of Sample Counts by Selection Group')
plt.show()


In [ ]:
# Prepare data for radial plot
categories = first_10_rows_each_group['selection_group'].unique()
N = len(categories)

angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]  # Complete the loop

# Radial plot
fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))

for group in categories:
    group_data = first_10_rows_each_group[first_10_rows_each_group['selection_group'] == group]
    all_values = []
    for index, row in group_data.iterrows():
        sample_counts = eval(row['sample_counts'])
        all_values.extend(sample_counts)
    values = [np.mean(all_values)] * N
    values += values[:1]
    ax.plot(angles, values, linewidth=2, linestyle='solid', label=group)
    ax.fill(angles, values, alpha=0.25)

ax.set_yticklabels([])
ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories)
plt.title('Radial Plot of Average Sample Counts by Selection Group')
plt.legend(loc='upper right', bbox_to_anchor=(1.1, 1.1))
plt.show()


In [ ]:
from mpl_toolkits.mplot3d import Axes3D

# Prepare data for 3D scatter plot
fig = plt.figure(figsize=(15, 10))
ax = fig.add_subplot(111, projection='3d')

for i, group in enumerate(groups):
    group_data = first_10_rows_each_group[first_10_rows_each_group['selection_group'] == group]
    all_values = []
    for index, row_data in group_data.iterrows():
        sample_counts = eval(row_data['sample_counts'])
        all_values.extend(sample_counts)
    ax.scatter(np.arange(len(all_values)), all_values, zs=i, zdir='y', label=group, depthshade=True)

ax.set_xlabel('Index')
ax.set_ylabel('Selection Group')
ax.set_zlabel('Value')
ax.set_yscale('log')
ax.set_ylim(-1, len(groups))
ax.set_yticks(range(len(groups)))
ax.set_yticklabels(groups)
plt.title('3D Scatter Plot of Sample Counts by Selection Group')
plt.legend()
plt.show()


In [ ]:
import joypy

# Prepare data for ridgeline plot
ridgeline_data = []
for group in first_10_rows_each_group['selection_group'].unique():
    group_data = first_10_rows_each_group[first_10_rows_each_group['selection_group'] == group]
    all_values = []
    for index, row in group_data.iterrows():
        sample_counts = eval(row['sample_counts'])
        all_values.extend(sample_counts)
    ridgeline_data.append(pd.DataFrame({'Value': all_values, 'Selection Group': group}))

ridgeline_df = pd.concat(ridgeline_data)

# Plot ridgeline plot
plt.figure(figsize=(20, 10))
joypy.joyplot(ridgeline_df, by='Selection Group', column='Value', figsize=(20, 10), log_scale=True, ylim='max', overlap=2)
plt.title('Ridgeline Plot of Sample Counts by Selection Group')
plt.xlabel('Value')
plt.show()


In [ ]:
!pip install joypy

In [ ]:
import pandas as pd

# Load the data
file_path = 'explosion_area_top_test_tcrs_across_samples.csv'
df = pd.read_csv(file_path)

# Store the original column names
original_column_names = df.columns.tolist()

# Transpose the DataFrame
df_transposed = df.transpose()

# Use the first row as the header
df_transposed.columns = df_transposed.iloc[0]
df_transposed = df_transposed[1:]

df_transposed = df_transposed.reset_index().rename({'index':'sample name'}, axis = 'columns')
df_transposed



In [ ]:
import pandas as pd

# Load the data
file_path = 'explosion_area_top_test_tcrs_across_samples.csv'
df = pd.read_csv(file_path)

# Transpose the DataFrame
df_transposed = df.transpose()

# Reset index to treat 'amino_acid' as a column
df_transposed.reset_index(inplace=True)

# Use the first row as the header and drop the first row
df_transposed.columns = df_transposed.iloc[0]
df_transposed = df_transposed[1:]

# Rename the index column

df_transposed = df_transposed.rename({'amino_acid':'sample name'}, axis = 'columns')

df_transposed


In [ ]:
import pandas as pd

# Load the data
file_path = 'explosion_area_top_test_tcrs_across_samples.csv'
df = pd.read_csv(file_path)

# Store the original column names
original_column_names = df.columns.tolist()

# Transpose the DataFrame
df_transposed = df.transpose()
df_transposed

In [ ]:
import pandas as pd

# Define the file path
file_path = 'Matched_File_Data.xlsx'

# Read the Excel file
df_metadata = pd.read_excel(file_path)

# Remove the '.tsv' extension from the 'sample name' column
df_metadata['sample name'] = df_metadata['sample name'].str.replace('.tsv', '')

df_metadata

In [ ]:
import pandas as pd

# Load the data
file_path = 'explosion_area_top_test_tcrs_across_samples.csv'
df = pd.read_csv(file_path)

# Store the original column names
original_column_names = df.columns.tolist()

# Transpose the DataFrame
df_transposed = df.transpose()

# Use the first row as the header
df_transposed.columns = df_transposed.iloc[0]
df_transposed = df_transposed[1:]

# Reset index to treat 'sample name' as a column
df_transposed = df_transposed.reset_index().rename({'index': 'sample name'}, axis='columns')

# Load the metadata
metadata_file_path = 'Matched_File_Data.xlsx'
df_metadata = pd.read_excel(metadata_file_path)

# Remove the '.tsv' extension from the 'sample name' column in metadata
df_metadata['sample name'] = df_metadata['sample name'].str.replace('.tsv', '')

# Merge the transposed data with the metadata on 'sample name'
merged_df = pd.merge(df_transposed, df_metadata, on='sample name', how='inner')

# Display the resulting DataFrame
merged_df

# Optionally, save the merged DataFrame to a new CSV file
#output_file_path = 'merged_transposed_metadata.csv'
#merged_df.to_csv(output_file_path, index=False)


In [ ]:
first_10_rows_each_group

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load the CSV file into a DataFrame
file_path = 'final_tcr_selection_adjusted_distribution_with_random_groups.csv'
df = pd.read_csv(file_path)

# Define the maximum CDR3 length (adjust this based on your data)
max_cdr3_length = 20

# Function to count amino acid presence for a group at a specific position
def amino_acid_presence_at_position(group_seqs, position):
    amino_acids = 'ACDEFGHIKLMNPQRSTVWY'  # List of all 20 standard amino acids
    presence_counts = {aa: 0 for aa in amino_acids}
    for seq in group_seqs:
        if len(seq) >= position:
            aa = seq[position - 1]
            presence_counts[aa] += 1
    return presence_counts

# Dynamically generate list of groups including all random groups
unique_groups = df['selection_group'].unique()
groups = {group: df[df['selection_group'] == group]['amino_acid'].dropna() for group in unique_groups}

# Process and remove sequences containing '*' for each group
for group_name in groups:
    groups[group_name] = groups[group_name][~groups[group_name].str.contains('\*', na=False)].tolist()

# Pre-define colors for different groups
colors = {
    'average_count_top': 'red',
    'variance_score_top': 'orange',
    'consistency_area_top': 'green',
    'consistency_area2_top': 'purple',
}
random_group_color = 'black'  # Use this color for all random groups

# Groups for left and right plots
left_groups = ['average_count_top', 'variance_score_top']
right_groups = ['consistency_area_top', 'consistency_area2_top']

# Width of each bar and the spacing between groups of bars
bar_width = 0.075
group_spacing = 0.01

# Create a separate plot for each position (location in CDR3)
for position in range(1, max_cdr3_length + 1):
    # Initialize a set to track the amino acids that appear at this position
    amino_acids_present = set()

    # Count amino acid presence for each group at the specified position and collect the amino acids that are present
    aa_counts_by_group = {}
    for group_name, seqs in groups.items():
        aa_counts = amino_acid_presence_at_position(seqs, position)
        aa_counts_by_group[group_name] = aa_counts
        amino_acids_present.update([aa for aa, count in aa_counts.items() if count > 0])

    # Sort the amino acids that are present
    amino_acids_present = sorted(amino_acids_present)

    # Create a mapping from amino acids to their x positions
    aa_to_x = {aa: i for i, aa in enumerate(amino_acids_present)}

    # Calculate the base x position for each amino acid's group of bars
    base_x_positions = np.arange(len(amino_acids_present))

    # Plot the data for the left groups
    plt.figure(figsize=(20, 8))
    left_idx = 0
    for group_name in left_groups + [name for name in groups if 'random_selection' in name]:
        if group_name not in aa_counts_by_group:
            continue

        aa_counts = aa_counts_by_group[group_name]

        # Determine color for the group
        if 'random_selection' in group_name:
            color = random_group_color
        else:
            color = colors.get(group_name, 'grey')  # Default color for other groups if not specified

        # Adjust x positions for this group's bars
        x_positions = [aa_to_x[aa] + left_idx * (bar_width + group_spacing) for aa in amino_acids_present]

        plt.bar(x_positions, [aa_counts[aa] for aa in amino_acids_present], width=bar_width, label=group_name, alpha=0.5, color=color)
        left_idx += 1

    plt.title(f'Amino Acid Presence at Position {position} - Randoms and Selected')
    plt.xlabel('Amino Acid')
    plt.xticks(base_x_positions + (left_idx - 1) * (bar_width + group_spacing) / 2, amino_acids_present)
    plt.ylabel('Count')
    plt.legend(loc='upper right')
    plt.tight_layout()
    plt.show()

    # Plot the data for the right groups
    plt.figure(figsize=(20, 8))
    right_idx = 0
    for group_name in right_groups + [name for name in groups if 'random_selection' in name]:
        if group_name not in aa_counts_by_group:
            continue

        aa_counts = aa_counts_by_group[group_name]

        # Determine color for the group
        if 'random_selection' in group_name:
            color = random_group_color
        else:
            color = colors.get(group_name, 'grey')  # Default color for other groups if not specified

        # Adjust x positions for this group's bars
        x_positions = [aa_to_x[aa] + right_idx * (bar_width + group_spacing) for aa in amino_acids_present]

        plt.bar(x_positions, [aa_counts[aa] for aa in amino_acids_present], width=bar_width, label=group_name, alpha=0.5, color=color)
        right_idx += 1

    plt.title(f'Amino Acid Presence at Position {position} - Consistency Areas')
    plt.xlabel('Amino Acid')
    plt.xticks(base_x_positions + (right_idx - 1) * (bar_width + group_spacing) / 2, amino_acids_present)
    plt.ylabel('Count')
    plt.legend(loc='upper right')
    plt.tight_layout()
    plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Load the CSV file into a DataFrame
file_path = 'final_tcr_selection_adjusted_distribution_with_random_groups.csv'
df = pd.read_csv(file_path)

# Define the maximum CDR3 length (adjust this based on your data)
max_cdr3_length = 20

# Function to count amino acid presence for a group at a specific position
def amino_acid_presence_at_position(group_seqs, position):
    amino_acids = 'ACDEFGHIKLMNPQRSTVWY'  # List of all 20 standard amino acids
    presence_counts = {aa: 0 for aa in amino_acids}
    for seq in group_seqs:
        if len(seq) >= position:
            aa = seq[position - 1]
            presence_counts[aa] += 1
    return presence_counts

# Dynamically generate list of groups including all random groups
unique_groups = df['selection_group'].unique()
groups = {group: df[df['selection_group'] == group]['amino_acid'].dropna() for group in unique_groups}

# Process and remove sequences containing '*' for each group
for group_name in groups:
    groups[group_name] = groups[group_name][~groups[group_name].str.contains('\*', na=False)].tolist()

# Pre-define colors for different groups
colors = {
    'average_count_top': 'blue',
    'explosion_area_top': 'purple',
    'consistency_area_top': 'red',
    'consistency_area2_top': 'red',
    'variance_score_top': 'blue'
}
random_group_color = 'grey'  # Use this color for all random groups

# Define the desired order of groups
desired_order = [
    'consistency_area_top',
    'consistency_area2_top',
    'explosion_area_top',
    'average_count_top',
    'variance_score_top'
]

# Adjust the group names to match the order
ordered_groups = {key: groups[key] for key in desired_order if key in groups}
ordered_groups.update({key: groups[key] for key in groups if 'random_selection' in key})

# Width of each bar and the spacing between groups of bars
bar_width = 0.07
group_spacing = 0.015

# Create a separate plot for each position (location in CDR3)
for position in range(1, max_cdr3_length + 1):
    plt.figure(figsize=(20, 8))

    # Calculate the base x position for each amino acid's group of bars
    base_x_positions = np.arange(len('ACDEFGHIKLMNPQRSTVWY'))

    # Count amino acid presence for each group at the specified position and plot
    for i, (group_name, seqs) in enumerate(ordered_groups.items()):
        aa_counts = amino_acid_presence_at_position(seqs, position)

        # Determine color for the group
        if 'random_selection' in group_name:
            color = random_group_color
        else:
            color = colors.get(group_name, 'grey')  # Default color for other groups if not specified

        # Adjust x positions for this group's bars
        x_positions = base_x_positions + i * (bar_width + group_spacing)

        plt.bar(x_positions, aa_counts.values(), width=bar_width, label=group_name, alpha=0.5, color=color)

    plt.title(f'Amino Acid Presence at Position {position}')
    plt.xlabel('Amino Acid')
    plt.xticks(base_x_positions + (len(ordered_groups) - 1) * (bar_width + group_spacing) / 2, 'ACDEFGHIKLMNPQRSTVWY')
    plt.ylabel('Count')
    plt.legend(loc='upper right')
    plt.show()


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial import distance

# Load the CSV file into a DataFrame
file_path = 'final_tcr_selection_adjusted_distribution_with_random_groups.csv'
df = pd.read_csv(file_path)

# Extract relevant columns
df = df[['amino_acid', 'selection_group']]

# Dynamically generate a list of groups including all random groups
unique_groups = df['selection_group'].unique()
groups = {group: df[df['selection_group'] == group]['amino_acid'].dropna() for group in unique_groups}

# Process and remove sequences containing '*' for each group
for group_name in groups:
    groups[group_name] = groups[group_name][~groups[group_name].str.contains('\*', na=False)].tolist()

def amino_acid_presence_at_position(group_seqs, position):
    amino_acids = 'ACDEFGHIKLMNPQRSTVWY'  # List of all 20 standard amino acids
    presence_counts = {aa: 0 for aa in amino_acids}
    for seq in group_seqs:
        if len(seq) >= position:
            aa = seq[position - 1]
            presence_counts[aa] += 1
    return presence_counts

# Define groups of interest
target_groups = ['consistency_area_top', 'consistency_area2_top', 'average_count_top', 'variance_score_top']
random_groups = [f'random_selection_{i}' for i in range(1, 6)]

# Initialize a dictionary to hold distances
distances = {tg: {rg: [] for rg in random_groups} for tg in target_groups}

# Loop through positions 1 to 20
for position in range(1, 21):
    aa_presence = {group: amino_acid_presence_at_position(seqs, position) for group, seqs in groups.items()}
    data = pd.DataFrame.from_dict(aa_presence, orient='index').fillna(0)

    # Calculate distances from each target group to each random group
    for tg in target_groups:
        for rg in random_groups:
            if tg in data.index and rg in data.index:
                distance_value = distance.euclidean(data.loc[tg], data.loc[rg])
                distances[tg][rg].append(distance_value)

# Plotting the distances
plt.figure(figsize=(14, 8))

# Define colors for different groups
colors = {
    'consistency_area_top': 'red',
    'consistency_area2_top': 'orange',
    'average_count_top': 'blue',
    'variance_score_top': 'green'
}

# Plot distances for each target group
for tg in target_groups:
    for rg in random_groups:
        plt.plot(range(1, 21), distances[tg][rg], marker='o', label=f'{tg} to {rg}', color=colors[tg], alpha=0.6)

plt.xlabel('Position', fontsize=14)
plt.ylabel('Euclidean Distance', fontsize=14)
plt.title('Euclidean Distance from Target Groups to Random Groups Across Positions', fontsize=18)
plt.legend(loc='upper right', bbox_to_anchor=(1.2, 1))

# Set the x-axis grid with a major interval of 1
plt.xticks(range(1, 21))
plt.grid(True, which='both', axis='x', linestyle='--', linewidth=0.5)
plt.tight_layout()
plt.show()


In [ ]:
# Plotting the summed distance matrix using seaborn
plt.figure(figsize=(19, 12))
sns.heatmap(summed_distance_df, annot=True, fmt=".2f", cmap='coolwarm', linewidths=.5, cbar_kws={'shrink': 0.5})
plt.title('Summed Euclidean Distance Matrix Across Positions 1 to 20', fontsize=18)
plt.xlabel('Selection Groups', fontsize=14)
plt.ylabel('Selection Groups', fontsize=14)
plt.xticks(rotation=45, ha='right', fontsize=12)
plt.yticks(fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial import distance

# Load the CSV file into a DataFrame
file_path = 'final_tcr_selection_adjusted_distribution_with_random_groups.csv'
df = pd.read_csv(file_path)

# Exclude the 'publicity_top' group
df = df[df['selection_group'] != 'publicity_top']

# Extract relevant columns
df = df[['amino_acid', 'selection_group']]

# Dynamically generate list of groups including all random groups
unique_groups = df['selection_group'].unique()
groups = {group: df[df['selection_group'] == group]['amino_acid'].dropna() for group in unique_groups}

# Process and remove sequences containing '*' for each group
for group_name in groups:
    groups[group_name] = groups[group_name][~groups[group_name].str.contains('\*', na=False)].tolist()

def amino_acid_presence_at_position(group_seqs, position):
    amino_acids = 'ACDEFGHIKLMNPQRSTVWY'  # List of all 20 standard amino acids
    presence_counts = {aa: 0 for aa in amino_acids}
    for seq in group_seqs:
        if len(seq) >= position:
            aa = seq[position - 1]
            presence_counts[aa] += 1
    return presence_counts

# Initialize a matrix to hold the summed distances
summed_distances = None

# Loop through positions 1 to 20
for position in range(1, 21):
    aa_presence = {group: amino_acid_presence_at_position(seqs, position) for group, seqs in groups.items()}
    data = pd.DataFrame.from_dict(aa_presence, orient='index').fillna(0)

    # Calculate the Euclidean distances between each pair of groups
    distance_matrix = distance.squareform(distance.pdist(data, metric='euclidean'))

    # Sum the distances from each position
    if summed_distances is None:
        summed_distances = distance_matrix
    else:
        summed_distances += distance_matrix

# Convert the summed distances into a DataFrame
summed_distance_df = pd.DataFrame(summed_distances, index=data.index, columns=data.index)

# Plotting the summed distance matrix using seaborn
plt.figure(figsize=(12, 10))
sns.heatmap(summed_distance_df, annot=True, fmt=".2f", cmap='coolwarm', linewidths=.5)
plt.title('Summed Euclidean Distance Matrix Across Positions 1 to 20')
plt.show()


In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import seaborn as sns
import matplotlib.pyplot as plt


# Load the CSV file into a DataFrame
file_path = 'final_tcr_selection_adjusted_distribution_with_random_groups.csv'
df = pd.read_csv(file_path)

# Exclude the 'publicity_top' group
#df = df[df['selection_group'] != 'publicity_top']
#

# Define the maximum CDR3 length (adjust this based on your data)
max_cdr3_length = 20
amino_acids = 'ACDEFGHIKLMNPQRSTVWY'  # List of all 20 standard amino acids

# Function to count amino acid presence for a group at a specific position
def amino_acid_presence_at_position(group_seqs, position):
    presence_counts = {aa: 0 for aa in amino_acids}
    for seq in group_seqs:
        if len(seq) >= position:
            aa = seq[position - 1]
            presence_counts[aa] += 1
    return presence_counts

# Prepare groups, excluding sequences with '*'
groups = {group: df[df['selection_group'] == group]['amino_acid'].dropna() for group in df['selection_group'].unique()}
for group_name in groups:
    groups[group_name] = groups[group_name][~groups[group_name].str.contains('\*', na=False)].tolist()

# Compute amino acid presence vectors for each group at each position
group_vectors = {}
for group_name, seqs in groups.items():
    vectors = []
    for position in range(1, max_cdr3_length + 1):
        presence_counts = amino_acid_presence_at_position(seqs, position)
        vectors.append([presence_counts[aa] for aa in amino_acids])
    group_vectors[group_name] = np.array(vectors)

# Function to calculate detailed similarity scores
def detailed_similarity_score(group_vectors):
    similarity_scores = pd.DataFrame(index=group_vectors.keys(), columns=group_vectors.keys(), data=0.0)

    for group1, vectors1 in group_vectors.items():
        for group2, vectors2 in group_vectors.items():
            score = 0

            for pos in range(vectors1.shape[0]):
                for aa_index in range(vectors1.shape[1]):
                    count1 = vectors1[pos, aa_index]
                    count2 = vectors2[pos, aa_index]
                    total_count = count1 + count2

                    if total_count > 0:
                        normalized_diff = abs(count1 - count2) / total_count
                        score += normalized_diff

            normalized_score = score / (vectors1.shape[0] * vectors1.shape[1])
            similarity_scores.at[group1, group2] = 1 - normalized_score  # Invert score for similarity

    return similarity_scores

# Calculate the detailed similarity scores
detailed_similarity_scores = detailed_similarity_score(group_vectors)

# Plotting the detailed similarity matrix
sns.set_context('talk')
plt.figure(figsize=(17, 10))
heatmap = sns.heatmap(detailed_similarity_scores, annot=True, fmt=".2f", cmap='coolwarm', linewidths=.5)
plt.title('Detailed Group Similarity Matrix')
plt.xlabel('Group')
plt.ylabel('Group')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()


In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import umap

# Function to load pickle files
def load_pickle(file_path):
    with open(file_path, 'rb') as file:
        data = pickle.load(file)
    return data

# Load the embedding vectors
file_path2 = 'high_cloness_embeddings.pkl'
file_path1 = 'low_cloness_embeddings.pkl'

data1 = load_pickle(file_path1)
data2 = load_pickle(file_path2)

data1 = data1.drop(columns='Sequences')
data2 = data2.drop(columns='Sequences')
# Convert to numpy arrays if they aren't already
data1 = np.array(data1)
data2 = np.array(data2)

# Combine the data for visualization
combined_data = np.vstack((data1, data2))
labels = np.array([0] * len(data1) + [1] * len(data2))

# Function to plot t-SNE
def plot_tsne(data, labels, title, save_path):
    tsne = TSNE(n_components=2, random_state=42)
    tsne_results = tsne.fit_transform(data)

    plt.figure(figsize=(8, 6))
    plt.scatter(tsne_results[:, 0], tsne_results[:, 1], c=labels, cmap='viridis', s=5)
    plt.title(title)
    plt.colorbar()
    plt.show()
    #plt.savefig(save_path)
    #plt.close()

# Function to plot UMAP
def plot_umap(data, labels, title, save_path):
    reducer = umap.UMAP(random_state=42)
    umap_results = reducer.fit_transform(data)

    plt.figure(figsize=(8, 6))
    plt.scatter(umap_results[:, 0], umap_results[:, 1], c=labels, cmap='viridis', s=5)
    plt.title(title)
    plt.colorbar()
    plt.show()
    #plt.savefig(save_path)
    #plt.close()

# Plot and save t-SNE
plot_tsne(combined_data, labels, 't-SNE Visualization', 'tsne_visualization.png')

# Plot and save UMAP
plot_umap(combined_data, labels, 'UMAP Visualization', 'umap_visualization.png')


In [ ]:
!pip install umap-learn


In [ ]:
import pickle
import numpy as np

# Function to load pickle files
def load_pickle(file_path):
    with open(file_path, 'rb') as file:
        data = pickle.load(file)
    return data


# Convert to numpy arrays if they aren't already
data1 = np.array(data1)
data2 = np.array(data2)

# Function to find and print non-float columns
def print_non_float_columns(data):
    non_float_columns = []
    for col in range(data.shape[1]):
        if not np.issubdtype(data[:, col].dtype, np.floating):
            non_float_columns.append(col)
    return non_float_columns

non_float_columns_data1 = print_non_float_columns(data1)
non_float_columns_data2 = print_non_float_columns(data2)

print("Non-float columns in high_cloness_embeddings:", non_float_columns_data1)
print("Non-float columns in low_cloness_embeddings:", non_float_columns_data2)


In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
import umap

# Function to load pickle files
def load_pickle(file_path):
    with open(file_path, 'rb') as file:
        data = pickle.load(file)
    return data

# Load the embedding vectors
file_path1 = 'high_cloness_embeddings.pkl'
file_path2 = 'low_cloness_embeddings.pkl'

data1 = load_pickle(file_path1)
data2 = load_pickle(file_path2)

print(data1[:10])